<a href="https://colab.research.google.com/github/stilyank01-create/Media_AI/blob/main/09_transformative_weight_and_recurrent_state.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Media AI — Transformative Weight and Recurrent State

**Notebook:** `09_transformative_weight_and_recurrent_state.ipynb`  
**Stage:** Transformative weighting and recurrent state construction  
**Pipeline position:** Notebook 09

This notebook introduces the recurrent transformation stage of the Media AI modelling architecture.

It receives the validated handover produced by Notebook 08, containing the complete frozen representation model together with the trained factual, psychological, and social transformative mechanisms. These inherited components provide the current pathway-specific representations and learned candidate transformations required for recurrent state evolution.

Notebook 08 deliberately stopped before introducing **Transformative Weights**, persistent recurrent states, or recurrent state updates. Notebook 09 begins from that explicit architectural boundary.

The central purpose of this notebook is to determine how strongly each learned candidate transformation should influence the evolving factual, psychological, and social states and to define how those states propagate causally through an ordered sequence of sentences.

---

## Architectural position

At the beginning of Notebook 09, the inherited architecture contains two validated learned stages:

$$
\text{prepared input}
\rightarrow
\text{representation model}
\rightarrow
\text{candidate transformative mechanism}.
$$

For each sentence position \(t\), the representation model provides three pathway-specific current representations:

$$
r_t^{(F)}\in\mathbb{R}^{10},
\qquad
r_t^{(P)}\in\mathbb{R}^{34},
\qquad
r_t^{(S)}\in\mathbb{R}^{7}.
$$

The trained transformative mechanisms inherited from Notebook 08 operate independently within these three spaces.

For pathway \(k\in\{F,P,S\}\),

$$
c_t^{(k)}
=
T_k\!\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right),
$$

where \(T_k\) is the trained transformative mechanism, \(r_t^{(k)}\) is the current representation, \(h_{t-1}^{(k)}\) is the preceding pathway state, and \(c_t^{(k)}\) is the resulting candidate transformation.

The candidate transformation describes a learned direction and magnitude of potential change. It does not, by itself, determine how much of that change should enter the evolving state.

That distinction defines the starting point of Notebook 09.

---

## Transformative Weight

Notebook 09 introduces a pathway-specific **Transformative Weight**, denoted

$$
TW_F,\qquad TW_P,\qquad TW_S.
$$

The Transformative Weight controls the contribution of a candidate transformation to the corresponding recurrent state transition.

Conceptually, the transformation entering pathway \(k\) is therefore separated into two components:

$$
\text{candidate transformation}
\quad\text{and}\quad
\text{transformative weighting}.
$$

This distinction is fundamental to the Media AI architecture.

The candidate transformation \(c_t^{(k)}\) is produced by the trained transformative mechanism inherited from Notebook 08. The Transformative Weight determines how strongly that candidate transformation contributes to state evolution.

Accordingly,

$$
c_t^{(k)} \neq TW_k.
$$

The Transformative Weights are also distinct from the optimisation-loss coefficients used when training the transformative mechanisms in Notebook 08. Those coefficients controlled the relative contribution of factual, psychological, and social losses to the training objective; they do not represent transformation strength and are not inherited as Transformative Weights.

The exact parameterisation, dimensionality, constraints, and learning policy of the Transformative Weights are established and validated explicitly in this notebook before recurrent state updates are permitted.

---

## Recurrent state

The second new architectural component is a persistent pathway-specific state.

For the three representation spaces, the recurrent states are

$$
h_t^{(F)}\in\mathbb{R}^{10},
\qquad
h_t^{(P)}\in\mathbb{R}^{34},
\qquad
h_t^{(S)}\in\mathbb{R}^{7}.
$$

Unlike an isolated sentence representation, the recurrent state carries information forward through the ordered article sequence.

The state associated with sentence \(t\) is therefore allowed to depend on the state produced by preceding sentences, but never on future sentences.

The general recurrent structure is

$$
h_t^{(k)}
=
\mathcal{U}_k
\left(
h_{t-1}^{(k)},
c_t^{(k)},
TW_k
\right),
$$

where \(\mathcal{U}_k\) denotes the pathway-specific state-update rule.

A simple conceptual form is

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)},
$$

where \(\odot\) denotes the weighting operation defined by the eventual Transformative-Weight contract.

This expression describes the architectural principle rather than assuming in advance that the final implementation must use a particular scalar, vector, gating, or other weighting parameterisation. Those choices must be established experimentally and methodologically within this notebook.

---

## Causal sequence evolution

The recurrent mechanism operates over the canonical sentence order established by the upstream notebooks.

For an article containing the ordered sequence

$$
s_1,s_2,\ldots,s_T,
$$

state evolution must proceed strictly forward:

$$
h_0
\rightarrow
h_1
\rightarrow
h_2
\rightarrow
\cdots
\rightarrow
h_T.
$$

When processing sentence \(s_t\), the model may use the current sentence representation and the state inherited from positions earlier than \(t\).

It must not use

$$
s_{t+1},s_{t+2},\ldots,s_T
$$

to construct the state at position \(t\).

This causal restriction is essential because the recurrent state is intended to represent accumulated transformation through the article rather than a retrospectively reconstructed sequence using future information.

---

## Initial-state contract

Every recurrent sequence requires an explicit initial condition.

For the first sentence of an article, no preceding recurrent state exists. Notebook 09 therefore defines and validates an initial-state policy before recurrent execution begins.

The architectural baseline is a deterministic zero state:

$$
h_0^{(F)}=\mathbf{0}_{10},
$$

$$
h_0^{(P)}=\mathbf{0}_{34},
$$

$$
h_0^{(S)}=\mathbf{0}_{7}.
$$

This provides a neutral and reproducible starting condition without introducing information from outside the article or from future observations.

The initial state is an architectural condition rather than an observed sentence-level supervision target.

---

## Independent representation spaces

Factual, psychological, and social state evolution remains structurally distinct.

The three recurrent pathways preserve their established dimensionalities:

$$
d_F=10,
\qquad
d_P=34,
\qquad
d_S=7.
$$

A factual candidate transformation modifies only the factual recurrent state, a psychological candidate transformation modifies only the psychological recurrent state, and a social candidate transformation modifies only the social recurrent state unless a later architecture explicitly introduces and validates cross-pathway interaction.

Notebook 09 therefore begins with independent recurrent pathways:

$$
h_t^{(F)}
=
\mathcal{U}_F
\left(
h_{t-1}^{(F)},
c_t^{(F)},
TW_F
\right),
$$

$$
h_t^{(P)}
=
\mathcal{U}_P
\left(
h_{t-1}^{(P)},
c_t^{(P)},
TW_P
\right),
$$

$$
h_t^{(S)}
=
\mathcal{U}_S
\left(
h_{t-1}^{(S)},
c_t^{(S)},
TW_S
\right).
$$

This preserves the semantic separation established by the representation and transformative stages.

---

## Inherited model boundary

Notebook 09 does not retrain the upstream representation model or the transformative mechanisms merely because they participate in recurrent computation.

The validated Notebook 08 handover contains:

- **894,003 representation-model parameters**;
- **8,187 trained transformative-mechanism parameters**;
- factual, psychological, and social pathway dimensions;
- active and deferred transformative dimensions;
- the validated target and training contracts;
- explicit confirmation that Transformative Weights were not created;
- explicit confirmation that recurrent state was not created.

The inherited parameter state therefore contains

$$
894{,}003 + 8{,}187 = 902{,}190
$$

validated parameters before Notebook 09 introduces any new parameterised component.

These inherited components are restored exactly, frozen, and placed in evaluation mode before the new Notebook 09 mechanism is constructed.

This makes any new trainable parameter introduced in Notebook 09 explicitly attributable to the Transformative-Weight or recurrent-state architecture.

---

## Active and deferred transformative dimensions

Notebook 08 also established pathway-specific active and deferred transformative dimensions from the available pilot supervision.

These activation boundaries are inherited rather than rediscovered.

An active dimension has sufficient validated supervision and transformation variation to participate in the current experimental transformation contract. A deferred dimension remains part of the architectural representation space but is not treated as adequately supported for the same learned transformation behaviour under the current pilot dataset.

Deferred dimensions are therefore **not removed from the state space**.

The recurrent states retain their complete dimensions:

$$
10,\qquad34,\qquad7.
$$

Activation eligibility instead determines where learned transformative behaviour is currently supported.

This distinction prevents limited pilot supervision from being mistaken for a permanent architectural reduction.

---

## Missing supervision and zero transformation

The mask-aware distinction established upstream remains applicable throughout Notebook 09.

A genuine zero transformation represents an observed or valid state in which no transformation occurs:

$$
\Delta h_t^{(k)}=0.
$$

Missing or unavailable supervision means that the corresponding transformation cannot be established from the available evidence.

These conditions are not equivalent.

Notebook 09 must therefore preserve the principle

$$
\text{missing transformation supervision}
\neq
\text{zero transformation}.
$$

No recurrent or Transformative-Weight objective may silently convert unavailable supervision into observed zero-valued behaviour.

---

## Methodological scope

Notebook 09 is responsible for:

- restoring and validating the complete Notebook 08 handover;
- confirming the exact inherited 902,190-parameter state;
- defining the Transformative-Weight contract;
- determining the dimensionality and constraints of each Transformative Weight;
- defining the recurrent-state contract;
- establishing deterministic initial states;
- defining the causal recurrent update rule;
- preserving factual, psychological, and social pathway separation;
- preserving active and deferred transformation boundaries;
- validating recurrent forward execution under controlled conditions;
- defining an appropriate optimisation policy where learning is required;
- training only explicitly eligible Notebook 09 parameters;
- verifying that inherited representation and transformative parameters remain unchanged;
- validating causal state propagation across ordered sentences;
- and constructing a persistent, reproducible handover for the subsequent modelling stage.

The notebook does **not** reinterpret the learned candidate transformations as Transformative Weights, does not use future sentence information in recurrent updates, and does not modify inherited parameters unless a later explicitly documented architectural decision requires such behaviour.

---

## Notebook objective

The objective of Notebook 09 is to extend the validated Media AI architecture from independent candidate transformations to controlled, causal state evolution:

$$
\boxed{
\text{representation}
\rightarrow
\text{candidate transformation}
\rightarrow
\text{Transformative Weight}
\rightarrow
\text{recurrent state}
}
$$

The resulting mechanism should provide a mathematically explicit and experimentally traceable representation of how factual, psychological, and social effects accumulate and evolve as an article is processed sentence by sentence.

At the beginning of this notebook, only the first two learned stages exist.

Transformative Weights and recurrent states are introduced only after the inherited Notebook 08 handover has been restored and validated successfully. This preserves a strict architectural boundary between what has already been learned and validated and what Notebook 09 is responsible for introducing.

## Block 1 — Notebook 08 → 09 Handover Restoration, Transformative-Weight Scope and Recurrent-State Initialisation Audit

This block establishes the controlled starting state of Notebook 09.

Notebook 08 completed the representation and candidate-transformation stages of the Media AI architecture and persisted them as a self-contained, validated handover. The present block retrieves that handover directly from persistent storage, validates its internal contract, reconstructs the inherited architecture, and confirms the methodological boundary from which Notebook 09 is permitted to proceed.

No Transformative Weight, recurrent state, recurrent update, training objective, optimiser, or parameter update is introduced in this block.

The purpose is to demonstrate that the state entering Notebook 09 is exactly the state that left Notebook 08.

### Persistent handover contract

Notebook 08 produced two canonical persistent artefacts:

- `notebook_08_final_checkpoint.pt`;
- `notebook_08_final_metadata.json`.

The checkpoint contains the complete validated parameter state required by Notebook 09. The metadata provides the corresponding architectural, methodological, and provenance contract.

The handover contains two inherited learned parameter families.

The first is the complete representation model:

$$
\theta_R
=
\{
\theta_{\mathrm{backbone}},
\theta_{\mathrm{global}},
\theta_{\mathrm{psychological}},
\theta_{\mathrm{factual}},
\theta_{\mathrm{social}}
\},
$$

with

$$
|\theta_R|=894{,}003.
$$

The second is the trained transformative-mechanism state:

$$
\theta_T
=
\{
\theta_{T_F},
\theta_{T_P},
\theta_{T_S}
\},
$$

with

$$
|\theta_T|=8{,}187.
$$

The complete inherited learned state therefore contains

$$
|\theta_R|+|\theta_T|
=
894{,}003+8{,}187
=
902{,}190
$$

parameters.

This value is treated as a strict Notebook 08 → 09 handover invariant.

### Direct artefact retrieval

Notebook 09 retrieves the checkpoint and metadata through their explicit Google Drive file identifiers recorded at Notebook 08 completion.

This avoids filesystem discovery and prevents Notebook 09 from attempting to reconstruct its inherited state by searching through earlier notebook artefacts.

The persistent Notebook 08 handover is the authoritative starting point.

Retrieval therefore follows the direct relation

$$
\text{Notebook 08 final handover}
\longrightarrow
\text{Notebook 09 initial state}.
$$

Notebook 09 does not independently rediscover Notebook 03, Notebook 05, Notebook 06, or Notebook 07 supervision and checkpoint artefacts. Their validated contributions have already been incorporated into the Notebook 08 handover.

The existing Google Drive API access pattern is reused. No Google Drive filesystem mount is required.

### Checkpoint and metadata validation

Retrieval alone is insufficient to establish a valid inheritance.

Before any model module is restored, the checkpoint and metadata must reproduce the contract persisted by Notebook 08.

The validation therefore confirms:

- the expected checkpoint and metadata artefact types;
- source notebook `08_transformative_mechanism`;
- target notebook `09_transformative_weight_and_recurrent_state`;
- the representation-model parameter contract;
- the transformative-mechanism parameter contract;
- factual, psychological, and social pathway dimensions;
- active and deferred transformative dimensions;
- the Notebook 08 methodological-boundary flags;
- checkpoint identity and integrity information;
- and consistency between checkpoint and metadata.

The checkpoint must explicitly establish that candidate transformations are available while Transformative Weights and recurrent states are absent.

The inherited boundary is therefore

$$
\text{candidate transformations available}
=
\mathrm{True},
$$

while

$$
\text{Transformative Weights created}
=
\mathrm{False},
$$

and

$$
\text{recurrent state created}
=
\mathrm{False}.
$$

Any contradiction of this boundary invalidates the Notebook 09 initialisation contract.

### Representation-model restoration

The inherited representation architecture is reconstructed exactly as validated in Notebook 08.

Its deterministic architecture is

$$
\mathbb{R}^{768}
\rightarrow
\mathbb{R}^{768}
\rightarrow
\mathbb{R}^{256}
\rightarrow
\mathbb{R}^{128},
$$

followed by three pathway-specific outputs:

$$
r_t^{(F)}\in\mathbb{R}^{10},
$$

$$
r_t^{(P)}\in\mathbb{R}^{34},
$$

$$
r_t^{(S)}\in\mathbb{R}^{7}.
$$

The restored representation parameter state must reproduce the persisted checkpoint exactly.

For every inherited representation parameter tensor \(\theta_i^{(R)}\),

$$
\theta_{i,\mathrm{runtime}}^{(R)}
=
\theta_{i,\mathrm{checkpoint}}^{(R)}.
$$

The representation model is then placed in evaluation mode and all of its parameters remain frozen.

Notebook 09 does not reopen representation learning.

### Transformative-mechanism restoration

The three trained transformative mechanisms are also reconstructed from the Notebook 08 handover.

Their inherited architectures are

$$
T_F:\mathbb{R}^{20}\rightarrow\mathbb{R}^{20}\rightarrow\mathbb{R}^{10},
$$

$$
T_P:\mathbb{R}^{68}\rightarrow\mathbb{R}^{68}\rightarrow\mathbb{R}^{34},
$$

and

$$
T_S:\mathbb{R}^{14}\rightarrow\mathbb{R}^{14}\rightarrow\mathbb{R}^{7}.
$$

Each mechanism receives a current representation and a same-dimensional preceding condition:

$$
c_t^{(k)}
=
T_k
\left(
[
r_t^{(k)};
h_{t-1}^{(k)}
]
\right),
\qquad
k\in\{F,P,S\}.
$$

The restored transformative parameters must reproduce the trained Notebook 08 state exactly:

$$
\theta_{i,\mathrm{runtime}}^{(T)}
=
\theta_{i,\mathrm{checkpoint}}^{(T)}.
$$

The three mechanisms are placed in evaluation mode and remain frozen at Notebook 09 initialisation.

This is important because Notebook 09 must distinguish parameters learned previously from parameters introduced by the new recurrent architecture.

### Inherited pathway dimensions

The factual, psychological, and social spaces retain their established dimensions:

$$
d_F=10,
\qquad
d_P=34,
\qquad
d_S=7.
$$

Consequently, candidate transformations retain the contracts

$$
c_t^{(F)}\in\mathbb{R}^{10},
\qquad
c_t^{(P)}\in\mathbb{R}^{34},
\qquad
c_t^{(S)}\in\mathbb{R}^{7}.
$$

These dimensions also define the eventual recurrent-state spaces:

$$
h_t^{(F)}\in\mathbb{R}^{10},
\qquad
h_t^{(P)}\in\mathbb{R}^{34},
\qquad
h_t^{(S)}\in\mathbb{R}^{7}.
$$

However, these equations define the required dimensional contract only.

No runtime recurrent-state tensor is created in this block.

### Active and deferred transformative dimensions

Notebook 08 identified active and deferred dimensions separately for each pathway.

These indices are inherited exactly rather than recalculated from the pilot supervision.

For the factual pathway, the active dimensions are

$$
A_F=\{1,2,4,8,9\},
$$

with deferred dimensions

$$
D_F=\{0,3,5,6,7\}.
$$

For the psychological pathway,

$$
A_P=
\{1,4,11,12,14,15,16,19,21,22,23,25,26,27,30,32,33\},
$$

while the remaining validated indices are deferred.

For the social pathway,

$$
A_S=\{0,2,4\},
$$

with

$$
D_S=\{1,3,5,6\}.
$$

These activation boundaries describe the current empirical support for learned transformation behaviour. They do not alter the dimensionality of the representation, candidate-transformation, or future recurrent-state spaces.

A deferred dimension therefore remains architecturally present.

### Transformative-Weight scope boundary

The principal new parameter concept of Notebook 09 is the Transformative Weight.

At the start of this block,

$$
TW_F,\quad TW_P,\quad TW_S
$$

do not yet exist.

The block explicitly verifies their absence from the Notebook 08 checkpoint and from the restored runtime architecture.

This is a methodological requirement rather than merely an implementation detail.

The trained candidate transformation

$$
c_t^{(k)}
$$

and the future Transformative Weight

$$
TW_k
$$

represent different quantities:

$$
c_t^{(k)}\neq TW_k.
$$

Likewise, the equal pathway coefficients used in the Notebook 08 optimisation objective are loss-combination coefficients and must not be inherited or reinterpreted as Transformative Weights.

The exact mathematical form of \(TW_k\), including whether the weighting contract is scalar, vector-valued, gated, constrained, fixed, or learned, remains intentionally unresolved in Block 1.

That decision belongs to the subsequent architectural-definition stage.

### Recurrent-state scope boundary

Notebook 08 used preceding conditions to construct and train candidate transformations, but it deliberately did not create a persistent recurrent mechanism.

Block 1 therefore verifies that no inherited object is being misidentified as the Notebook 09 recurrent state.

At this stage,

$$
h_t^{(F)},\quad
h_t^{(P)},\quad
h_t^{(S)}
$$

describe future architectural state spaces only.

No recurrent transition such as

$$
h_t^{(k)}
=
\mathcal{U}_k
\left(
h_{t-1}^{(k)},
c_t^{(k)},
TW_k
\right)
$$

is executed.

Likewise, the conceptual zero initial conditions

$$
h_0^{(F)}=\mathbf{0}_{10},
\qquad
h_0^{(P)}=\mathbf{0}_{34},
\qquad
h_0^{(S)}=\mathbf{0}_{7}
$$

are audited as the intended initial-state policy but are not yet instantiated as recurrent runtime state.

This preserves a clean distinction between defining the state contract and executing state evolution.

### Parameter-freezing policy

At the completion of Block 1, all inherited learned parameters remain frozen:

$$
\nabla_{\theta_R}=0,
$$

and

$$
\nabla_{\theta_T}=0.
$$

No optimiser is created and no parameter is eligible for update.

The inherited architecture therefore forms a fixed computational foundation on top of which the new Notebook 09 mechanism can subsequently be introduced.

Any trainable parameters created later in Notebook 09 must be explicitly identified as new Notebook 09 parameters.

### Execution boundary

Block 1 is deliberately non-training and non-recurrent.

It may retrieve persistent artefacts, reconstruct modules, load validated state dictionaries, move modules to the configured execution device, freeze parameters, and validate architecture and state equality.

It does not:

- execute representation-model inference;
- execute transformative forward inference;
- calculate a loss;
- create an optimiser;
- execute backpropagation;
- update a parameter;
- create a Transformative Weight;
- create recurrent state;
- or execute a recurrent state transition.

This allows the notebook to establish a verified initial architecture before introducing any new behaviour.

### Block completion contract

Block 1 is complete only when:

- the Notebook 08 checkpoint is retrieved directly and successfully;
- the Notebook 08 metadata is retrieved directly and successfully;
- checkpoint and metadata contracts are mutually consistent;
- all 894,003 representation parameters are restored exactly;
- all 8,187 transformative parameters are restored exactly;
- the complete 902,190-parameter inherited state is validated;
- factual, psychological, and social dimensions are reproduced correctly;
- active and deferred transformative dimensions are inherited exactly;
- all inherited modules are frozen;
- all inherited modules are in evaluation mode;
- Transformative Weights remain absent;
- recurrent state remains absent;
- no recurrent update has occurred;
- no forward pass has occurred;
- and no optimisation or parameter update has occurred.

Successful completion establishes the Notebook 09 initial condition

$$
\boxed{
\text{validated frozen representation model}
+
\text{validated frozen transformative mechanisms}
}
$$

with

$$
\boxed{
\text{Transformative Weights absent}
+
\text{recurrent state absent}.
}
$$

Notebook 09 may then proceed to the explicit architectural definition of the Transformative-Weight mechanism without ambiguity about which components were inherited and which components are newly introduced.

In [51]:
# =============================================================================
# Media AI — Notebook 09
# Block 1: Notebook 08 → 09 Handover Restoration,
#          Transformative-Weight Scope and Recurrent-State Initialisation Audit
# =============================================================================

from collections import OrderedDict
from copy import deepcopy
from datetime import datetime, timezone

import hashlib
import io
import json

import numpy as np
import torch
import torch.nn as nn

from google.colab import auth
import google.auth

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09 = (
    "09_transformative_weight_and_recurrent_state"
)

NOTEBOOK_09_VERSION = "1.0"

NOTEBOOK_09_BLOCK_1 = 1

NOTEBOOK_09_BLOCK_1_NAME = (
    "Notebook 08 → 09 Handover Restoration, "
    "Transformative-Weight Scope and Recurrent-State Initialisation Audit"
)

NOTEBOOK_09_BLOCK_1_VERSION = "1.3"

BLOCK_1_EXECUTED_AT_UTC = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# =============================================================================
# Project constants
# =============================================================================

NOTEBOOK_09_RANDOM_SEED = 42

DEFAULT_DTYPE = torch.float32

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


np.random.seed(
    NOTEBOOK_09_RANDOM_SEED
)


torch.manual_seed(
    NOTEBOOK_09_RANDOM_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        NOTEBOOK_09_RANDOM_SEED
    )


torch.set_default_dtype(
    DEFAULT_DTYPE
)


# =============================================================================
# Persistent Notebook 08 → 09 handover contract
# =============================================================================

NOTEBOOK_08_TO_09_CHECKPOINT_FILENAME = (
    "notebook_08_final_checkpoint.pt"
)


NOTEBOOK_08_TO_09_METADATA_FILENAME = (
    "notebook_08_final_metadata.json"
)


NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID = (
    "1cjxowL02MQWbQ0lHCoET6QVklGVkhSsp"
)


NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID = (
    "1_A2mOaoW741tGzBrTvim6_XFsaS0eLt3"
)


# =============================================================================
# Execution boundary
# =============================================================================

NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED = False

NOTEBOOK_09_RECURRENT_STATE_CREATED = False

NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED = False

NOTEBOOK_09_LOSS_CALCULATED = False

NOTEBOOK_09_OPTIMIZER_CREATED = False

NOTEBOOK_09_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_TRAINING_EXECUTED = False


# =============================================================================
# Google authentication
# =============================================================================
#
# This reproduces the proven Media AI Colab authentication pattern used by
# Notebook 07.
#
# Authentication is performed once at notebook initialisation.
# The resulting Drive API service is reused by later Notebook 09 blocks.
#
# No Google Drive filesystem mount is used.
# =============================================================================

auth.authenticate_user()


BLOCK_1_CREDENTIALS, _ = (
    google.auth.default()
)


NOTEBOOK_09_DRIVE_SERVICE = build(
    "drive",
    "v3",
    credentials=
        BLOCK_1_CREDENTIALS,
    cache_discovery=
        False,
)


BLOCK_1_COLAB_AUTHENTICATED = True


BLOCK_1_DRIVE_SERVICE_INITIALISED = (
    NOTEBOOK_09_DRIVE_SERVICE
    is not None
)


if not BLOCK_1_DRIVE_SERVICE_INITIALISED:

    raise RuntimeError(
        "Notebook 09 could not initialise the Google Drive service."
    )


# =============================================================================
# Drive helpers
# =============================================================================

def block_1_download_drive_bytes(
    drive_service,
    file_id,
):
    """
    Download one Google Drive file and return raw bytes.
    """

    request = (
        drive_service.files()
        .get_media(
            fileId=
                file_id
        )
    )


    buffer = io.BytesIO()


    downloader = MediaIoBaseDownload(
        buffer,
        request,
    )


    complete = False


    while not complete:

        _, complete = (
            downloader.next_chunk()
        )


    buffer.seek(
        0
    )


    return buffer.read()


def block_1_sha256(
    raw_bytes,
):

    return hashlib.sha256(
        raw_bytes
    ).hexdigest()


# =============================================================================
# Retrieve checkpoint Drive metadata
# =============================================================================

BLOCK_1_CHECKPOINT_DRIVE_METADATA = (
    NOTEBOOK_09_DRIVE_SERVICE.files()
    .get(
        fileId=
            NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID,
        fields=
            "id,name,modifiedTime,parents,size,mimeType",
    )
    .execute()
)


BLOCK_1_CHECKPOINT_LOCATED = (
    BLOCK_1_CHECKPOINT_DRIVE_METADATA.get(
        "id"
    )
    ==
    NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID
)


BLOCK_1_CHECKPOINT_FILENAME_VALID = (
    BLOCK_1_CHECKPOINT_DRIVE_METADATA.get(
        "name"
    )
    ==
    NOTEBOOK_08_TO_09_CHECKPOINT_FILENAME
)


if not BLOCK_1_CHECKPOINT_LOCATED:

    raise FileNotFoundError(
        "Notebook 08 final checkpoint could not be located."
    )


if not BLOCK_1_CHECKPOINT_FILENAME_VALID:

    raise RuntimeError(
        "Notebook 08 checkpoint filename does not match "
        "the expected persistent handover contract."
    )


# =============================================================================
# Retrieve metadata Drive metadata
# =============================================================================

BLOCK_1_METADATA_DRIVE_METADATA = (
    NOTEBOOK_09_DRIVE_SERVICE.files()
    .get(
        fileId=
            NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID,
        fields=
            "id,name,modifiedTime,parents,size,mimeType",
    )
    .execute()
)


BLOCK_1_METADATA_LOCATED = (
    BLOCK_1_METADATA_DRIVE_METADATA.get(
        "id"
    )
    ==
    NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID
)


BLOCK_1_METADATA_FILENAME_VALID = (
    BLOCK_1_METADATA_DRIVE_METADATA.get(
        "name"
    )
    ==
    NOTEBOOK_08_TO_09_METADATA_FILENAME
)


if not BLOCK_1_METADATA_LOCATED:

    raise FileNotFoundError(
        "Notebook 08 final metadata could not be located."
    )


if not BLOCK_1_METADATA_FILENAME_VALID:

    raise RuntimeError(
        "Notebook 08 metadata filename does not match "
        "the expected persistent handover contract."
    )


# =============================================================================
# Download checkpoint and metadata
# =============================================================================

BLOCK_1_CHECKPOINT_BYTES = (
    block_1_download_drive_bytes(
        NOTEBOOK_09_DRIVE_SERVICE,
        NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID,
    )
)


BLOCK_1_METADATA_BYTES = (
    block_1_download_drive_bytes(
        NOTEBOOK_09_DRIVE_SERVICE,
        NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID,
    )
)


BLOCK_1_CHECKPOINT_DOWNLOADED = bool(
    BLOCK_1_CHECKPOINT_BYTES
)


BLOCK_1_METADATA_DOWNLOADED = bool(
    BLOCK_1_METADATA_BYTES
)


if not BLOCK_1_CHECKPOINT_DOWNLOADED:

    raise RuntimeError(
        "Notebook 08 final checkpoint was empty."
    )


if not BLOCK_1_METADATA_DOWNLOADED:

    raise RuntimeError(
        "Notebook 08 final metadata artefact was empty."
    )


# =============================================================================
# Parse metadata
# =============================================================================

try:

    NOTEBOOK_09_INHERITED_METADATA = json.loads(
        BLOCK_1_METADATA_BYTES.decode(
            "utf-8"
        )
    )


except (
    UnicodeDecodeError,
    json.JSONDecodeError,
) as error:

    raise RuntimeError(
        "Notebook 08 metadata could not be parsed as UTF-8 JSON."
    ) from error


BLOCK_1_METADATA_JSON_PARSED = isinstance(
    NOTEBOOK_09_INHERITED_METADATA,
    dict,
)


if not BLOCK_1_METADATA_JSON_PARSED:

    raise TypeError(
        "Notebook 08 metadata must be a JSON object."
    )


# =============================================================================
# Checkpoint integrity validation before deserialisation
# =============================================================================

BLOCK_1_CHECKPOINT_SHA256 = (
    block_1_sha256(
        BLOCK_1_CHECKPOINT_BYTES
    )
)


BLOCK_1_EXPECTED_CHECKPOINT_SHA256 = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "checkpoint_sha256"
    )
)


BLOCK_1_CHECKPOINT_SHA256_VALID = (
    isinstance(
        BLOCK_1_EXPECTED_CHECKPOINT_SHA256,
        str,
    )
    and
    BLOCK_1_CHECKPOINT_SHA256
    ==
    BLOCK_1_EXPECTED_CHECKPOINT_SHA256
)


if not BLOCK_1_CHECKPOINT_SHA256_VALID:

    raise RuntimeError(
        "Notebook 08 checkpoint SHA-256 validation failed."
    )


BLOCK_1_CHECKPOINT_BYTE_COUNT_VALID = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "checkpoint_bytes"
    )
    ==
    len(
        BLOCK_1_CHECKPOINT_BYTES
    )
)


if not BLOCK_1_CHECKPOINT_BYTE_COUNT_VALID:

    raise RuntimeError(
        "Notebook 08 checkpoint byte count does not match metadata."
    )


# =============================================================================
# Deserialize checkpoint
# =============================================================================

BLOCK_1_CHECKPOINT_BUFFER = io.BytesIO(
    BLOCK_1_CHECKPOINT_BYTES
)


try:

    NOTEBOOK_09_INHERITED_CHECKPOINT = torch.load(
        BLOCK_1_CHECKPOINT_BUFFER,
        map_location=
            "cpu",
        weights_only=
            False,
    )


except TypeError:

    BLOCK_1_CHECKPOINT_BUFFER.seek(
        0
    )


    NOTEBOOK_09_INHERITED_CHECKPOINT = torch.load(
        BLOCK_1_CHECKPOINT_BUFFER,
        map_location=
            "cpu",
    )


BLOCK_1_CHECKPOINT_PARSED = isinstance(
    NOTEBOOK_09_INHERITED_CHECKPOINT,
    dict,
)


if not BLOCK_1_CHECKPOINT_PARSED:

    raise TypeError(
        "Notebook 08 checkpoint must be dictionary-like."
    )


# =============================================================================
# Checkpoint identity contract
# =============================================================================

BLOCK_1_CHECKPOINT_TYPE_VALID = (
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "checkpoint_type"
    )
    ==
    "validated_transformative_model_handover"
)


BLOCK_1_CHECKPOINT_SOURCE_VALID = (
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "source_notebook"
    )
    ==
    "08_transformative_mechanism"
)


BLOCK_1_CHECKPOINT_TARGET_VALID = (
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "target_notebook"
    )
    ==
    "09_transformative_weight_and_recurrent_state"
)


BLOCK_1_CHECKPOINT_IDENTITY_VALID = all(
    [
        BLOCK_1_CHECKPOINT_TYPE_VALID,
        BLOCK_1_CHECKPOINT_SOURCE_VALID,
        BLOCK_1_CHECKPOINT_TARGET_VALID,
    ]
)


if not BLOCK_1_CHECKPOINT_IDENTITY_VALID:

    raise RuntimeError(
        "Notebook 08 checkpoint identity contract is invalid."
    )


# =============================================================================
# Metadata identity contract
# =============================================================================

BLOCK_1_METADATA_TYPE_VALID = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "artifact_type"
    )
    ==
    "media_ai_notebook_08_to_09_handover_metadata"
)


BLOCK_1_METADATA_SOURCE_VALID = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "source_notebook"
    )
    ==
    "08_transformative_mechanism"
)


BLOCK_1_METADATA_TARGET_VALID = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "target_notebook"
    )
    ==
    "09_transformative_weight_and_recurrent_state"
)


BLOCK_1_METADATA_CHECKPOINT_TYPE_VALID = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "checkpoint_type"
    )
    ==
    "validated_transformative_model_handover"
)


BLOCK_1_METADATA_CHECKPOINT_FILENAME_VALID = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "checkpoint_filename"
    )
    ==
    NOTEBOOK_08_TO_09_CHECKPOINT_FILENAME
)


BLOCK_1_METADATA_CHECKPOINT_FILE_ID_VALID = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "checkpoint_drive_file_id"
    )
    ==
    NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID
)


BLOCK_1_METADATA_CHECKPOINT_HASH_VALID = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "checkpoint_sha256"
    )
    ==
    BLOCK_1_CHECKPOINT_SHA256
)


BLOCK_1_NOTEBOOK_09_READY_IN_METADATA = (
    NOTEBOOK_09_INHERITED_METADATA.get(
        "notebook_09_ready"
    )
    is True
)


BLOCK_1_METADATA_IDENTITY_VALID = all(
    [
        BLOCK_1_METADATA_TYPE_VALID,
        BLOCK_1_METADATA_SOURCE_VALID,
        BLOCK_1_METADATA_TARGET_VALID,
        BLOCK_1_METADATA_CHECKPOINT_TYPE_VALID,
        BLOCK_1_METADATA_CHECKPOINT_FILENAME_VALID,
        BLOCK_1_METADATA_CHECKPOINT_FILE_ID_VALID,
        BLOCK_1_METADATA_CHECKPOINT_HASH_VALID,
        BLOCK_1_NOTEBOOK_09_READY_IN_METADATA,
    ]
)


if not BLOCK_1_METADATA_IDENTITY_VALID:

    raise RuntimeError(
        "Notebook 08 metadata identity contract is invalid."
    )


# =============================================================================
# Combined handover identity contract
# =============================================================================

BLOCK_1_HANDOVER_IDENTITY_VALID = all(
    [
        BLOCK_1_CHECKPOINT_IDENTITY_VALID,
        BLOCK_1_METADATA_IDENTITY_VALID,
    ]
)


if not BLOCK_1_HANDOVER_IDENTITY_VALID:

    raise RuntimeError(
        "Notebook 08 → Notebook 09 handover identity is invalid."
    )


# =============================================================================
# Cross-artefact contract
# =============================================================================

BLOCK_1_CROSS_ARTEFACT_VALID = all(
    [
        (
            NOTEBOOK_09_INHERITED_METADATA.get(
                "checkpoint_type"
            )
            ==
            NOTEBOOK_09_INHERITED_CHECKPOINT.get(
                "checkpoint_type"
            )
        ),

        (
            NOTEBOOK_09_INHERITED_METADATA.get(
                "source_notebook"
            )
            ==
            NOTEBOOK_09_INHERITED_CHECKPOINT.get(
                "source_notebook"
            )
        ),

        (
            NOTEBOOK_09_INHERITED_METADATA.get(
                "target_notebook"
            )
            ==
            NOTEBOOK_09_INHERITED_CHECKPOINT.get(
                "target_notebook"
            )
        ),

        (
            NOTEBOOK_09_INHERITED_METADATA.get(
                "representation",
                {},
            ).get(
                "parameter_count"
            )
            ==
            NOTEBOOK_09_INHERITED_CHECKPOINT.get(
                "parameter_counts",
                {},
            ).get(
                "representation_total"
            )
        ),

        (
            NOTEBOOK_09_INHERITED_METADATA.get(
                "transformative_mechanisms",
                {},
            ).get(
                "parameter_count"
            )
            ==
            NOTEBOOK_09_INHERITED_CHECKPOINT.get(
                "parameter_counts",
                {},
            ).get(
                "transformative_total"
            )
        ),
    ]
)


if not BLOCK_1_CROSS_ARTEFACT_VALID:

    raise RuntimeError(
        "Notebook 08 checkpoint and metadata are inconsistent."
    )


# =============================================================================
# Inherited parameter and pathway contracts
# =============================================================================
#
# Notebook 09 does not recreate Notebook 08 architecture totals or pathway
# dimensions as literals. The validated Notebook 08 checkpoint is the source
# of truth; Block 1 validates its internal accounting and cross-artefact
# consistency before restoring executable modules.
# =============================================================================

BLOCK_1_PARAMETER_COUNTS = deepcopy(
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "parameter_counts",
        {},
    )
)


BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_COUNTS = deepcopy(
    BLOCK_1_PARAMETER_COUNTS.get(
        "representation",
        {},
    )
)


BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_COUNTS = deepcopy(
    BLOCK_1_PARAMETER_COUNTS.get(
        "transformative",
        {},
    )
)


BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_TOTAL = (
    BLOCK_1_PARAMETER_COUNTS.get(
        "representation_total"
    )
)


BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_TOTAL = (
    BLOCK_1_PARAMETER_COUNTS.get(
        "transformative_total"
    )
)


BLOCK_1_INHERITED_COMBINED_PARAMETER_TOTAL = (
    BLOCK_1_PARAMETER_COUNTS.get(
        "combined_total"
    )
)


BLOCK_1_PARAMETER_CONTRACT_VALID = all(
    [
        isinstance(
            BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_COUNTS,
            dict,
        ),

        isinstance(
            BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_COUNTS,
            dict,
        ),

        bool(
            BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_COUNTS
        ),

        bool(
            BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_COUNTS
        ),

        all(
            isinstance(
                parameter_count,
                int,
            )
            and
            parameter_count > 0

            for parameter_count
            in BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_COUNTS.values()
        ),

        all(
            isinstance(
                parameter_count,
                int,
            )
            and
            parameter_count > 0

            for parameter_count
            in BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_COUNTS.values()
        ),

        (
            BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_TOTAL
            ==
            sum(
                BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_COUNTS.values()
            )
        ),

        (
            BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_TOTAL
            ==
            sum(
                BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_COUNTS.values()
            )
        ),

        (
            BLOCK_1_INHERITED_COMBINED_PARAMETER_TOTAL
            ==
            (
                BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_TOTAL
                +
                BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_TOTAL
            )
        ),
    ]
)


if not BLOCK_1_PARAMETER_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 08 inherited parameter-count contract is invalid."
    )


# =============================================================================
# Pathway dimensional contract
# =============================================================================

NOTEBOOK_09_PATHWAY_DIMS = deepcopy(
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "pathway_dimensions",
        {},
    )
)


BLOCK_1_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            NOTEBOOK_09_PATHWAY_DIMS,
            dict,
        ),

        bool(
            NOTEBOOK_09_PATHWAY_DIMS
        ),

        all(
            isinstance(
                pathway_dim,
                int,
            )
            and
            pathway_dim > 0

            for pathway_dim
            in NOTEBOOK_09_PATHWAY_DIMS.values()
        ),

        (
            set(
                NOTEBOOK_09_PATHWAY_DIMS.keys()
            )
            ==
            set(
                BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_COUNTS.keys()
            )
        ),
    ]
)


if not BLOCK_1_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Inherited pathway dimensional contract is invalid."
    )


# =============================================================================
# Active and deferred dimension contract
# =============================================================================

NOTEBOOK_09_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in (
        NOTEBOOK_09_INHERITED_CHECKPOINT.get(
            "active_dimension_indices",
            {},
        ).items()
    )
}


NOTEBOOK_09_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in (
        NOTEBOOK_09_INHERITED_CHECKPOINT.get(
            "deferred_dimension_indices",
            {},
        ).items()
    )
}


BLOCK_1_ACTIVATION_PARTITION_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in NOTEBOOK_09_PATHWAY_DIMS.items():

    active_indices = set(
        NOTEBOOK_09_ACTIVE_DIMENSION_INDICES.get(
            pathway_name,
            (),
        )
    )


    deferred_indices = set(
        NOTEBOOK_09_DEFERRED_DIMENSION_INDICES.get(
            pathway_name,
            (),
        )
    )


    expected_indices = set(
        range(
            pathway_dim
        )
    )


    BLOCK_1_ACTIVATION_PARTITION_VALID[
        pathway_name
    ] = all(
        [
            active_indices.isdisjoint(
                deferred_indices
            ),

            (
                active_indices
                |
                deferred_indices
            )
            ==
            expected_indices,

            all(
                isinstance(
                    dimension_index,
                    int,
                )
                and
                0
                <=
                dimension_index
                <
                pathway_dim

                for dimension_index
                in (
                    active_indices
                    |
                    deferred_indices
                )
            ),
        ]
    )


BLOCK_1_ALL_ACTIVATION_PARTITIONS_VALID = all(
    BLOCK_1_ACTIVATION_PARTITION_VALID.values()
)


if not BLOCK_1_ALL_ACTIVATION_PARTITIONS_VALID:

    raise RuntimeError(
        "Active and deferred transformative dimensions "
        "do not form valid pathway partitions."
    )


# =============================================================================
# Metadata ↔ checkpoint pathway validation
# =============================================================================

BLOCK_1_METADATA_TRANSFORMATIVE = deepcopy(
    NOTEBOOK_09_INHERITED_METADATA.get(
        "transformative_mechanisms",
        {},
    )
)


BLOCK_1_METADATA_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in (
        BLOCK_1_METADATA_TRANSFORMATIVE.get(
            "active_dimension_indices",
            {},
        ).items()
    )
}


BLOCK_1_METADATA_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in (
        BLOCK_1_METADATA_TRANSFORMATIVE.get(
            "deferred_dimension_indices",
            {},
        ).items()
    )
}


BLOCK_1_PATHWAY_CROSS_ARTEFACT_VALID = all(
    [
        (
            BLOCK_1_METADATA_TRANSFORMATIVE.get(
                "pathway_dimensions"
            )
            ==
            NOTEBOOK_09_PATHWAY_DIMS
        ),

        (
            BLOCK_1_METADATA_ACTIVE_DIMENSION_INDICES
            ==
            NOTEBOOK_09_ACTIVE_DIMENSION_INDICES
        ),

        (
            BLOCK_1_METADATA_DEFERRED_DIMENSION_INDICES
            ==
            NOTEBOOK_09_DEFERRED_DIMENSION_INDICES
        ),
    ]
)


if not BLOCK_1_PATHWAY_CROSS_ARTEFACT_VALID:

    raise RuntimeError(
        "Checkpoint and metadata pathway contracts disagree."
    )


# =============================================================================
# Notebook 09 inherited methodological boundary
# =============================================================================

BLOCK_1_CHECKPOINT_BOUNDARY = deepcopy(
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "methodological_boundary",
        {},
    )
)


BLOCK_1_METADATA_BOUNDARY = deepcopy(
    NOTEBOOK_09_INHERITED_METADATA.get(
        "notebook_09_boundary",
        {},
    )
)


BLOCK_1_INHERITED_BOUNDARY_VALID = all(
    [
        (
            BLOCK_1_CHECKPOINT_BOUNDARY.get(
                "candidate_transformations_available"
            )
            is True
        ),

        (
            BLOCK_1_CHECKPOINT_BOUNDARY.get(
                "candidate_transformations_are_transformative_weights"
            )
            is False
        ),

        (
            BLOCK_1_CHECKPOINT_BOUNDARY.get(
                "transformative_weights_created"
            )
            is False
        ),

        (
            BLOCK_1_CHECKPOINT_BOUNDARY.get(
                "recurrent_state_created"
            )
            is False
        ),

        (
            BLOCK_1_CHECKPOINT_BOUNDARY.get(
                "recurrent_update_executed"
            )
            is False
        ),

        (
            BLOCK_1_CHECKPOINT_BOUNDARY.get(
                "future_context_used"
            )
            is False
        ),

        (
            BLOCK_1_METADATA_BOUNDARY.get(
                "candidate_transformations_available"
            )
            is True
        ),

        (
            BLOCK_1_METADATA_BOUNDARY.get(
                "transformative_weights_available"
            )
            is False
        ),

        (
            BLOCK_1_METADATA_BOUNDARY.get(
                "recurrent_state_available"
            )
            is False
        ),

        (
            BLOCK_1_METADATA_BOUNDARY.get(
                "recurrent_update_available"
            )
            is False
        ),
    ]
)


if not BLOCK_1_INHERITED_BOUNDARY_VALID:

    raise RuntimeError(
        "Notebook 08 handover violates the required "
        "Notebook 09 starting boundary."
    )


# =============================================================================
# Persisted runtime architecture contracts
# =============================================================================

BLOCK_1_REPRESENTATION_ARCHITECTURE_CONTRACT = deepcopy(
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "representation_architecture",
        {},
    )
)


BLOCK_1_TRANSFORMATIVE_ARCHITECTURE_CONTRACT = deepcopy(
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "transformative_architecture",
        {},
    )
)


BLOCK_1_ARCHITECTURE_CONTRACTS_PRESENT = all(
    [
        isinstance(
            BLOCK_1_REPRESENTATION_ARCHITECTURE_CONTRACT,
            dict,
        ),

        isinstance(
            BLOCK_1_TRANSFORMATIVE_ARCHITECTURE_CONTRACT,
            dict,
        ),

        bool(
            BLOCK_1_REPRESENTATION_ARCHITECTURE_CONTRACT
        ),

        bool(
            BLOCK_1_TRANSFORMATIVE_ARCHITECTURE_CONTRACT
        ),
    ]
)


if not BLOCK_1_ARCHITECTURE_CONTRACTS_PRESENT:

    raise RuntimeError(
        "Notebook 08 runtime architecture contracts are unavailable."
    )


# =============================================================================
# Persisted state collections
# =============================================================================

BLOCK_1_PERSISTED_REPRESENTATION_STATE = (
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "representation_state_dict",
        {},
    )
)


BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE = (
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "transformative_state_dict",
        {},
    )
)


# =============================================================================
# Runtime architecture reconstruction helpers
# =============================================================================
#
# Notebook 08 persisted the actual runtime module repr and exact state-shape
# map. Notebook 09 reconstructs the SAME named module topology:
#
#   backbone:
#       projection
#
#   global confluent:
#       global_projection[0..3]
#
#   representation heads:
#       core.network[0..3]
#       output_layer
#
#   transformative mechanisms:
#       transform_network[0..4]
#
# Dimensions are inferred from persisted tensor shapes. Activation and dropout
# are recovered from the persisted runtime repr. No model dimension or
# parameter total is recreated as a literal.
# =============================================================================

def block_1_state_shape_contract(
    state_dict,
):

    if not isinstance(
        state_dict,
        dict,
    ):

        raise TypeError(
            "Persisted state must be dictionary-like."
        )


    return {
        key:
            list(
                tensor.shape
            )

        for (
            key,
            tensor,
        ) in state_dict.items()
    }


def block_1_architecture_state_shapes_valid(
    architecture_contract,
    persisted_state,
):

    if not isinstance(
        architecture_contract,
        dict,
    ):

        return False


    return (
        architecture_contract.get(
            "state_shapes"
        )
        ==
        block_1_state_shape_contract(
            persisted_state
        )
    )


def block_1_activation_from_repr(
    module_repr,
):

    if not isinstance(
        module_repr,
        str,
    ):

        raise TypeError(
            "Persisted module repr must be a string."
        )


    if "ReLU(" in module_repr:

        return nn.ReLU()


    if "GELU(" in module_repr:

        return nn.GELU()


    raise RuntimeError(
        "Unsupported inherited activation in persisted architecture repr."
    )


def block_1_dropout_probability_from_repr(
    module_repr,
):

    if not isinstance(
        module_repr,
        str,
    ):

        raise TypeError(
            "Persisted module repr must be a string."
        )


    marker = "Dropout(p="


    marker_index = module_repr.find(
        marker
    )


    if marker_index < 0:

        raise RuntimeError(
            "Persisted architecture repr does not expose Dropout(p=...)."
        )


    value_start = (
        marker_index
        +
        len(
            marker
        )
    )


    value_end = module_repr.find(
        ",",
        value_start,
    )


    if value_end < 0:

        value_end = module_repr.find(
            ")",
            value_start,
        )


    if value_end < 0:

        raise RuntimeError(
            "Persisted dropout probability could not be parsed."
        )


    dropout_probability = float(
        module_repr[
            value_start:
            value_end
        ]
    )


    if not (
        0.0
        <=
        dropout_probability
        <=
        1.0
    ):

        raise RuntimeError(
            "Persisted dropout probability is outside [0, 1]."
        )


    return dropout_probability


def block_1_require_state_key(
    state_dict,
    key,
    contract_name,
):

    if key not in state_dict:

        raise RuntimeError(
            f"{contract_name} is missing required persisted state key "
            f"{key!r}. Available keys: {list(state_dict.keys())}"
        )


    tensor = state_dict[
        key
    ]


    if not torch.is_tensor(
        tensor
    ):

        raise TypeError(
            f"{contract_name}.{key} is not a tensor."
        )


    return tensor


class Notebook09Backbone(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        output_dim,
    ):

        super().__init__()


        self.projection = nn.Linear(
            input_dim,
            output_dim,
        )


    def forward(
        self,
        x,
    ):

        return self.projection(
            x
        )


class Notebook09GlobalConfluent(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        output_dim,
        activation,
        dropout_probability,
    ):

        super().__init__()


        self.global_projection = nn.Sequential(
            nn.Linear(
                input_dim,
                output_dim,
            ),

            activation,

            nn.Dropout(
                dropout_probability
            ),

            nn.LayerNorm(
                output_dim
            ),
        )


    def forward(
        self,
        x,
    ):

        return self.global_projection(
            x
        )


class Notebook09RepresentationCore(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        activation,
        dropout_probability,
    ):

        super().__init__()


        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),

            activation,

            nn.Dropout(
                dropout_probability
            ),

            nn.LayerNorm(
                hidden_dim
            ),
        )


    def forward(
        self,
        x,
    ):

        return self.network(
            x
        )


class Notebook09RepresentationHead(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        activation,
        dropout_probability,
    ):

        super().__init__()


        self.core = Notebook09RepresentationCore(
            input_dim=
                input_dim,

            hidden_dim=
                hidden_dim,

            activation=
                activation,

            dropout_probability=
                dropout_probability,
        )


        self.output_layer = nn.Linear(
            hidden_dim,
            output_dim,
        )


    def forward(
        self,
        x,
    ):

        return self.output_layer(
            self.core(
                x
            )
        )


class Notebook09TransformativeMechanism(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        activation,
        dropout_probability,
    ):

        super().__init__()


        self.transform_network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),

            activation,

            nn.Dropout(
                dropout_probability
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
        )


    def forward(
        self,
        current_representation,
        preceding_state,
    ):

        combined = torch.cat(
            [
                current_representation,
                preceding_state,
            ],
            dim=
                -1,
        )


        return self.transform_network(
            combined
        )


def block_1_build_representation_module(
    module_name,
    architecture_contract,
    persisted_state,
):

    module_repr = architecture_contract.get(
        "repr"
    )


    if not block_1_architecture_state_shapes_valid(
        architecture_contract,
        persisted_state,
    ):

        raise RuntimeError(
            f"Persisted architecture/state shape contract is invalid for "
            f"representation.{module_name}."
        )


    if module_name == "backbone":

        weight = block_1_require_state_key(
            persisted_state,
            "projection.weight",
            "representation.backbone",
        )


        output_dim, input_dim = weight.shape


        module = Notebook09Backbone(
            input_dim=
                int(
                    input_dim
                ),

            output_dim=
                int(
                    output_dim
                ),
        )


    elif module_name == "global_confluent":

        projection_weight = block_1_require_state_key(
            persisted_state,
            "global_projection.0.weight",
            "representation.global_confluent",
        )


        layernorm_weight = block_1_require_state_key(
            persisted_state,
            "global_projection.3.weight",
            "representation.global_confluent",
        )


        output_dim, input_dim = projection_weight.shape


        if tuple(
            layernorm_weight.shape
        ) != (
            int(
                output_dim
            ),
        ):

            raise RuntimeError(
                "Global confluent LayerNorm shape is inconsistent with "
                "the persisted projection output dimension."
            )


        module = Notebook09GlobalConfluent(
            input_dim=
                int(
                    input_dim
                ),

            output_dim=
                int(
                    output_dim
                ),

            activation=
                block_1_activation_from_repr(
                    module_repr
                ),

            dropout_probability=
                block_1_dropout_probability_from_repr(
                    module_repr
                ),
        )


    elif module_name.endswith(
        "_head"
    ):

        core_weight = block_1_require_state_key(
            persisted_state,
            "core.network.0.weight",
            f"representation.{module_name}",
        )


        core_layernorm_weight = block_1_require_state_key(
            persisted_state,
            "core.network.3.weight",
            f"representation.{module_name}",
        )


        output_weight = block_1_require_state_key(
            persisted_state,
            "output_layer.weight",
            f"representation.{module_name}",
        )


        hidden_dim, input_dim = core_weight.shape

        output_dim, output_hidden_dim = output_weight.shape


        if int(
            output_hidden_dim
        ) != int(
            hidden_dim
        ):

            raise RuntimeError(
                f"Representation head {module_name} has inconsistent "
                "persisted hidden dimensions."
            )


        if tuple(
            core_layernorm_weight.shape
        ) != (
            int(
                hidden_dim
            ),
        ):

            raise RuntimeError(
                f"Representation head {module_name} LayerNorm shape "
                "is inconsistent with the persisted hidden dimension."
            )


        module = Notebook09RepresentationHead(
            input_dim=
                int(
                    input_dim
                ),

            hidden_dim=
                int(
                    hidden_dim
                ),

            output_dim=
                int(
                    output_dim
                ),

            activation=
                block_1_activation_from_repr(
                    module_repr
                ),

            dropout_probability=
                block_1_dropout_probability_from_repr(
                    module_repr
                ),
        )


    else:

        raise RuntimeError(
            f"Unsupported inherited representation module "
            f"{module_name!r}."
        )


    runtime_contract = {
        "class":
            module.__class__.__name__,

        "repr":
            repr(
                module
            ),

        "state_shapes":
            block_1_state_shape_contract(
                module.state_dict()
            ),
    }


    if runtime_contract[
        "state_shapes"
    ] != architecture_contract[
        "state_shapes"
    ]:

        raise RuntimeError(
            f"Reconstructed representation.{module_name} topology does not "
            "match the persisted Notebook 08 state-shape contract."
        )


    return module


def block_1_build_transformative_module(
    pathway_name,
    architecture_contract,
    persisted_state,
):

    module_repr = architecture_contract.get(
        "repr"
    )


    pathway_dim = int(
        architecture_contract.get(
            "pathway_dimension"
        )
    )


    if pathway_dim != int(
        NOTEBOOK_09_PATHWAY_DIMS[
            pathway_name
        ]
    ):

        raise RuntimeError(
            f"Transformative pathway dimension mismatch for "
            f"{pathway_name}."
        )


    if not block_1_architecture_state_shapes_valid(
        architecture_contract,
        persisted_state,
    ):

        raise RuntimeError(
            f"Persisted architecture/state shape contract is invalid for "
            f"transformative.{pathway_name}."
        )


    input_weight = block_1_require_state_key(
        persisted_state,
        "transform_network.0.weight",
        f"transformative.{pathway_name}",
    )


    layernorm_weight = block_1_require_state_key(
        persisted_state,
        "transform_network.3.weight",
        f"transformative.{pathway_name}",
    )


    output_weight = block_1_require_state_key(
        persisted_state,
        "transform_network.4.weight",
        f"transformative.{pathway_name}",
    )


    hidden_dim, input_dim = input_weight.shape

    output_dim, output_hidden_dim = output_weight.shape


    if int(
        input_dim
    ) != (
        2
        *
        pathway_dim
    ):

        raise RuntimeError(
            f"Transformative pathway {pathway_name} persisted input "
            "dimension is incompatible with current+preceding concatenation."
        )


    if int(
        output_dim
    ) != pathway_dim:

        raise RuntimeError(
            f"Transformative pathway {pathway_name} persisted output "
            "dimension does not preserve pathway dimensionality."
        )


    if int(
        output_hidden_dim
    ) != int(
        hidden_dim
    ):

        raise RuntimeError(
            f"Transformative pathway {pathway_name} persisted hidden "
            "dimensions are inconsistent."
        )


    if tuple(
        layernorm_weight.shape
    ) != (
        int(
            hidden_dim
        ),
    ):

        raise RuntimeError(
            f"Transformative pathway {pathway_name} LayerNorm shape "
            "is inconsistent with the persisted hidden dimension."
        )


    module = Notebook09TransformativeMechanism(
        input_dim=
            int(
                input_dim
            ),

        hidden_dim=
            int(
                hidden_dim
            ),

        output_dim=
            int(
                output_dim
            ),

        activation=
            block_1_activation_from_repr(
                module_repr
            ),

        dropout_probability=
            block_1_dropout_probability_from_repr(
                module_repr
            ),
    )


    runtime_state_shapes = block_1_state_shape_contract(
        module.state_dict()
    )


    if runtime_state_shapes != architecture_contract[
        "state_shapes"
    ]:

        raise RuntimeError(
            f"Reconstructed transformative.{pathway_name} topology does not "
            "match the persisted Notebook 08 state-shape contract."
        )


    return module


# =============================================================================
# Instantiate inherited architecture from persisted contracts
# =============================================================================

BLOCK_1_REPRESENTATION_MODULES = OrderedDict(
    (
        module_name,
        block_1_build_representation_module(
            module_name,
            BLOCK_1_REPRESENTATION_ARCHITECTURE_CONTRACT[
                module_name
            ],
            BLOCK_1_PERSISTED_REPRESENTATION_STATE[
                module_name
            ],
        ),
    )

    for module_name
    in BLOCK_1_PERSISTED_REPRESENTATION_STATE
)


BLOCK_1_TRANSFORMATIVE_MODULES = OrderedDict(
    (
        pathway_name,
        block_1_build_transformative_module(
            pathway_name,
            BLOCK_1_TRANSFORMATIVE_ARCHITECTURE_CONTRACT[
                pathway_name
            ],
            BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE[
                pathway_name
            ],
        ),
    )

    for pathway_name
    in BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE
)


NOTEBOOK_09_RESTORED_BACKBONE = (
    BLOCK_1_REPRESENTATION_MODULES[
        "backbone"
    ]
)


NOTEBOOK_09_RESTORED_GLOBAL_CONFLUENT = (
    BLOCK_1_REPRESENTATION_MODULES[
        "global_confluent"
    ]
)


NOTEBOOK_09_RESTORED_PSYCHOLOGICAL_HEAD = (
    BLOCK_1_REPRESENTATION_MODULES[
        "psychological_head"
    ]
)


NOTEBOOK_09_RESTORED_FACTUAL_HEAD = (
    BLOCK_1_REPRESENTATION_MODULES[
        "factual_head"
    ]
)


NOTEBOOK_09_RESTORED_SOCIAL_HEAD = (
    BLOCK_1_REPRESENTATION_MODULES[
        "social_head"
    ]
)


NOTEBOOK_09_RESTORED_FACTUAL_TRANSFORMATIVE_MECHANISM = (
    BLOCK_1_TRANSFORMATIVE_MODULES[
        "factual"
    ]
)


NOTEBOOK_09_RESTORED_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM = (
    BLOCK_1_TRANSFORMATIVE_MODULES[
        "psychological"
    ]
)


NOTEBOOK_09_RESTORED_SOCIAL_TRANSFORMATIVE_MECHANISM = (
    BLOCK_1_TRANSFORMATIVE_MODULES[
        "social"
    ]
)


BLOCK_1_PERSISTED_MODULE_CONTRACT_VALID = all(
    [
        (
            set(
                BLOCK_1_PERSISTED_REPRESENTATION_STATE.keys()
            )
            ==
            set(
                BLOCK_1_REPRESENTATION_MODULES.keys()
            )
        ),

        (
            set(
                BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE.keys()
            )
            ==
            set(
                BLOCK_1_TRANSFORMATIVE_MODULES.keys()
            )
        ),

        (
            set(
                BLOCK_1_REPRESENTATION_ARCHITECTURE_CONTRACT.keys()
            )
            ==
            set(
                BLOCK_1_REPRESENTATION_MODULES.keys()
            )
        ),

        (
            set(
                BLOCK_1_TRANSFORMATIVE_ARCHITECTURE_CONTRACT.keys()
            )
            ==
            set(
                BLOCK_1_TRANSFORMATIVE_MODULES.keys()
            )
        ),
    ]
)


if not BLOCK_1_PERSISTED_MODULE_CONTRACT_VALID:

    raise RuntimeError(
        "Persisted representation or transformative module contracts "
        "are invalid."
    )


# =============================================================================
# Canonical inherited module collections
# =============================================================================

BLOCK_1_REPRESENTATION_MODULES = OrderedDict(
    [
        (
            "backbone",
            NOTEBOOK_09_RESTORED_BACKBONE,
        ),

        (
            "global_confluent",
            NOTEBOOK_09_RESTORED_GLOBAL_CONFLUENT,
        ),

        (
            "psychological_head",
            NOTEBOOK_09_RESTORED_PSYCHOLOGICAL_HEAD,
        ),

        (
            "factual_head",
            NOTEBOOK_09_RESTORED_FACTUAL_HEAD,
        ),

        (
            "social_head",
            NOTEBOOK_09_RESTORED_SOCIAL_HEAD,
        ),
    ]
)


BLOCK_1_TRANSFORMATIVE_MODULES = OrderedDict(
    [
        (
            "factual",
            NOTEBOOK_09_RESTORED_FACTUAL_TRANSFORMATIVE_MECHANISM,
        ),

        (
            "psychological",
            NOTEBOOK_09_RESTORED_PSYCHOLOGICAL_TRANSFORMATIVE_MECHANISM,
        ),

        (
            "social",
            NOTEBOOK_09_RESTORED_SOCIAL_TRANSFORMATIVE_MECHANISM,
        ),
    ]
)


# =============================================================================
# Persisted state collections
# =============================================================================

BLOCK_1_PERSISTED_REPRESENTATION_STATE = (
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "representation_state_dict",
        {},
    )
)


BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE = (
    NOTEBOOK_09_INHERITED_CHECKPOINT.get(
        "transformative_state_dict",
        {},
    )
)


BLOCK_1_PERSISTED_MODULE_CONTRACT_VALID = all(
    [
        isinstance(
            BLOCK_1_PERSISTED_REPRESENTATION_STATE,
            dict,
        ),

        isinstance(
            BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE,
            dict,
        ),

        (
            set(
                BLOCK_1_PERSISTED_REPRESENTATION_STATE.keys()
            )
            ==
            set(
                BLOCK_1_REPRESENTATION_MODULES.keys()
            )
        ),

        (
            set(
                BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE.keys()
            )
            ==
            set(
                BLOCK_1_TRANSFORMATIVE_MODULES.keys()
            )
        ),
    ]
)


if not BLOCK_1_PERSISTED_MODULE_CONTRACT_VALID:

    raise RuntimeError(
        "Persisted representation or transformative state "
        "collections are invalid."
    )


# =============================================================================
# Validated state restoration helper
# =============================================================================

def block_1_restore_state(
    module,
    persisted_state,
    contract_name,
):

    if not isinstance(
        persisted_state,
        dict,
    ):

        raise TypeError(
            f"{contract_name} persisted state must be dictionary-like."
        )


    runtime_state = module.state_dict()


    runtime_keys = tuple(
        runtime_state.keys()
    )


    persisted_keys = tuple(
        persisted_state.keys()
    )


    if runtime_keys != persisted_keys:

        raise RuntimeError(
            f"{contract_name} state-dict keys do not reproduce the "
            "persisted Notebook 08 contract."
        )


    for key in runtime_keys:

        if tuple(
            runtime_state[
                key
            ].shape
        ) != tuple(
            persisted_state[
                key
            ].shape
        ):

            raise RuntimeError(
                f"{contract_name}.{key} shape does not reproduce "
                "the persisted Notebook 08 contract."
            )


    load_state = OrderedDict(
        (
            key,
            persisted_state[
                key
            ].detach()
            .cpu()
            .clone(),
        )

        for key
        in runtime_keys
    )


    module.load_state_dict(
        load_state,
        strict=True,
    )


    restored_exact = all(
        torch.equal(
            module.state_dict()[
                key
            ].detach()
            .cpu(),
            load_state[
                key
            ],
        )

        for key
        in runtime_keys
    )


    if not restored_exact:

        raise RuntimeError(
            f"{contract_name} state restoration was not exact."
        )


    return {
        "restoration_mode":
            "strict_persisted_contract",

        "exact_key_match":
            True,

        "tensor_count":
            len(
                load_state
            ),

        "restored_exact":
            restored_exact,
    }


# =============================================================================
# Restore representation state
# =============================================================================

BLOCK_1_REPRESENTATION_RESTORATION = {}


for (
    module_name,
    module,
) in BLOCK_1_REPRESENTATION_MODULES.items():

    BLOCK_1_REPRESENTATION_RESTORATION[
        module_name
    ] = block_1_restore_state(
        module=
            module,

        persisted_state=
            BLOCK_1_PERSISTED_REPRESENTATION_STATE[
                module_name
            ],

        contract_name=
            f"representation.{module_name}",
    )


# =============================================================================
# Restore transformative state
# =============================================================================

BLOCK_1_TRANSFORMATIVE_RESTORATION = {}


for (
    pathway_name,
    module,
) in BLOCK_1_TRANSFORMATIVE_MODULES.items():

    BLOCK_1_TRANSFORMATIVE_RESTORATION[
        pathway_name
    ] = block_1_restore_state(
        module=
            module,

        persisted_state=
            BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE[
                pathway_name
            ],

        contract_name=
            f"transformative.{pathway_name}",
    )


BLOCK_1_REPRESENTATION_RESTORATION_EXACT = all(
    result[
        "restored_exact"
    ]

    for result
    in BLOCK_1_REPRESENTATION_RESTORATION.values()
)


BLOCK_1_TRANSFORMATIVE_RESTORATION_EXACT = all(
    result[
        "restored_exact"
    ]

    for result
    in BLOCK_1_TRANSFORMATIVE_RESTORATION.values()
)


if not all(
    [
        BLOCK_1_REPRESENTATION_RESTORATION_EXACT,
        BLOCK_1_TRANSFORMATIVE_RESTORATION_EXACT,
    ]
):

    raise RuntimeError(
        "Notebook 08 inherited state restoration was not exact."
    )


# =============================================================================
# Restored parameter accounting
# =============================================================================

BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_COUNTS = {
    module_name:
        int(
            sum(
                parameter.numel()

                for parameter
                in module.parameters()
            )
        )

    for (
        module_name,
        module,
    ) in BLOCK_1_REPRESENTATION_MODULES.items()
}


BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_COUNTS = {
    pathway_name:
        int(
            sum(
                parameter.numel()

                for parameter
                in module.parameters()
            )
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_1_TRANSFORMATIVE_MODULES.items()
}


BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_TOTAL = int(
    sum(
        BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_COUNTS.values()
    )
)


BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_TOTAL = int(
    sum(
        BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_COUNTS.values()
    )
)


BLOCK_1_RESTORED_PARAMETER_TOTAL = (
    BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_TOTAL
    +
    BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_TOTAL
)


BLOCK_1_RESTORED_PARAMETER_COUNTS_VALID = all(
    [
        (
            BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_COUNTS
            ==
            BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_COUNTS
        ),

        (
            BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_COUNTS
            ==
            BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_COUNTS
        ),

        (
            BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_TOTAL
            ==
            BLOCK_1_INHERITED_REPRESENTATION_PARAMETER_TOTAL
        ),

        (
            BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_TOTAL
            ==
            BLOCK_1_INHERITED_TRANSFORMATIVE_PARAMETER_TOTAL
        ),

        (
            BLOCK_1_RESTORED_PARAMETER_TOTAL
            ==
            BLOCK_1_INHERITED_COMBINED_PARAMETER_TOTAL
        ),
    ]
)


if not BLOCK_1_RESTORED_PARAMETER_COUNTS_VALID:

    raise RuntimeError(
        "Restored Notebook 09 inherited parameter counts are invalid."
    )


# =============================================================================
# Move inherited architecture to execution device
# =============================================================================

for module in BLOCK_1_REPRESENTATION_MODULES.values():

    module.to(
        DEVICE
    )


for module in BLOCK_1_TRANSFORMATIVE_MODULES.values():

    module.to(
        DEVICE
    )


# =============================================================================
# Freeze inherited architecture
# =============================================================================

for module in BLOCK_1_REPRESENTATION_MODULES.values():

    module.eval()


    for parameter in module.parameters():

        parameter.requires_grad_(
            False
        )


for module in BLOCK_1_TRANSFORMATIVE_MODULES.values():

    module.eval()


    for parameter in module.parameters():

        parameter.requires_grad_(
            False
        )


BLOCK_1_REPRESENTATION_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_1_TRANSFORMATIVE_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_1_ALL_MODULES_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


# =============================================================================
# Device-placement validation
# =============================================================================

BLOCK_1_DEVICE_PLACEMENT_VALID = all(
    parameter.device
    ==
    DEVICE

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )

    for parameter
    in module.parameters()
)


if not BLOCK_1_DEVICE_PLACEMENT_VALID:

    raise RuntimeError(
        "One or more inherited modules were not placed "
        "on the configured execution device."
    )


# =============================================================================
# Exact post-device state validation
# =============================================================================

def block_1_runtime_matches_persisted_state(
    module,
    persisted_state,
):

    runtime_items = list(
        module.state_dict().items()
    )


    persisted_items = list(
        persisted_state.items()
    )


    if len(
        runtime_items
    ) != len(
        persisted_items
    ):

        return False


    for (
        (
            _,
            runtime_tensor,
        ),
        (
            _,
            persisted_tensor,
        ),
    ) in zip(
        runtime_items,
        persisted_items,
    ):

        if (
            tuple(
                runtime_tensor.shape
            )
            !=
            tuple(
                persisted_tensor.shape
            )
        ):

            return False


        if not torch.equal(
            runtime_tensor.detach().cpu(),
            persisted_tensor.detach().cpu(),
        ):

            return False


    return True


BLOCK_1_POST_DEVICE_REPRESENTATION_EXACT = all(
    block_1_runtime_matches_persisted_state(
        module,
        BLOCK_1_PERSISTED_REPRESENTATION_STATE[
            module_name
        ],
    )

    for (
        module_name,
        module,
    ) in BLOCK_1_REPRESENTATION_MODULES.items()
)


BLOCK_1_POST_DEVICE_TRANSFORMATIVE_EXACT = all(
    block_1_runtime_matches_persisted_state(
        module,
        BLOCK_1_PERSISTED_TRANSFORMATIVE_STATE[
            pathway_name
        ],
    )

    for (
        pathway_name,
        module,
    ) in BLOCK_1_TRANSFORMATIVE_MODULES.items()
)


if not all(
    [
        BLOCK_1_POST_DEVICE_REPRESENTATION_EXACT,
        BLOCK_1_POST_DEVICE_TRANSFORMATIVE_EXACT,
    ]
):

    raise RuntimeError(
        "Inherited parameter state changed during device placement."
    )


# =============================================================================
# Notebook 09 inherited parameter contract
# =============================================================================

NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT = (
    BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_TOTAL
)


NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT = (
    BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_TOTAL
)


NOTEBOOK_09_INHERITED_PARAMETER_COUNT = (
    BLOCK_1_RESTORED_PARAMETER_TOTAL
)


# =============================================================================
# Explicit Notebook 09 starting boundary
# =============================================================================

NOTEBOOK_09_CANDIDATE_TRANSFORMATIONS_AVAILABLE = True

NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED = False

NOTEBOOK_09_RECURRENT_STATE_CREATED = False

NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED = False


# =============================================================================
# Explicit absence audit
# =============================================================================
#
# These objects must not exist before Notebook 09 explicitly defines them.
# =============================================================================

BLOCK_1_FORBIDDEN_NEW_RUNTIME_OBJECTS = (
    "TW_F",
    "TW_P",
    "TW_S",

    "NOTEBOOK_09_TW_F",
    "NOTEBOOK_09_TW_P",
    "NOTEBOOK_09_TW_S",

    "NOTEBOOK_09_FACTUAL_RECURRENT_STATE",
    "NOTEBOOK_09_PSYCHOLOGICAL_RECURRENT_STATE",
    "NOTEBOOK_09_SOCIAL_RECURRENT_STATE",
)


BLOCK_1_NEW_ARCHITECTURAL_OBJECTS_ABSENT = all(
    object_name not in globals()

    for object_name
    in BLOCK_1_FORBIDDEN_NEW_RUNTIME_OBJECTS
)


if not BLOCK_1_NEW_ARCHITECTURAL_OBJECTS_ABSENT:

    existing_objects = [
        object_name

        for object_name
        in BLOCK_1_FORBIDDEN_NEW_RUNTIME_OBJECTS

        if object_name in globals()
    ]


    raise RuntimeError(
        "Notebook 09 Block 1 detected Transformative-Weight or "
        "recurrent-state objects before their architectural definition. "
        f"Objects: {existing_objects}"
    )


# =============================================================================
# Final Block 1 validation
# =============================================================================

NOTEBOOK_09_BLOCK_1_ERRORS = []


block_1_validation_checks = {
    "drive_service_not_initialised":
        BLOCK_1_DRIVE_SERVICE_INITIALISED,

    "checkpoint_not_located":
        BLOCK_1_CHECKPOINT_LOCATED,

    "checkpoint_filename_invalid":
        BLOCK_1_CHECKPOINT_FILENAME_VALID,

    "metadata_not_located":
        BLOCK_1_METADATA_LOCATED,

    "metadata_filename_invalid":
        BLOCK_1_METADATA_FILENAME_VALID,

    "checkpoint_not_downloaded":
        BLOCK_1_CHECKPOINT_DOWNLOADED,

    "metadata_not_downloaded":
        BLOCK_1_METADATA_DOWNLOADED,

    "metadata_not_parsed":
        BLOCK_1_METADATA_JSON_PARSED,

    "checkpoint_not_parsed":
        BLOCK_1_CHECKPOINT_PARSED,

    "checkpoint_sha256_invalid":
        BLOCK_1_CHECKPOINT_SHA256_VALID,

    "checkpoint_byte_count_invalid":
        BLOCK_1_CHECKPOINT_BYTE_COUNT_VALID,

    "checkpoint_identity_invalid":
        BLOCK_1_CHECKPOINT_IDENTITY_VALID,

    "metadata_identity_invalid":
        BLOCK_1_METADATA_IDENTITY_VALID,

    "handover_identity_invalid":
        BLOCK_1_HANDOVER_IDENTITY_VALID,

    "cross_artefact_invalid":
        BLOCK_1_CROSS_ARTEFACT_VALID,

    "parameter_contract_invalid":
        BLOCK_1_PARAMETER_CONTRACT_VALID,

    "pathway_dimensions_invalid":
        BLOCK_1_PATHWAY_DIMENSIONS_VALID,

    "activation_partition_invalid":
        BLOCK_1_ALL_ACTIVATION_PARTITIONS_VALID,

    "pathway_cross_artefact_invalid":
        BLOCK_1_PATHWAY_CROSS_ARTEFACT_VALID,

    "inherited_boundary_invalid":
        BLOCK_1_INHERITED_BOUNDARY_VALID,

    "architecture_contracts_missing":
        BLOCK_1_ARCHITECTURE_CONTRACTS_PRESENT,

    "persisted_module_contract_invalid":
        BLOCK_1_PERSISTED_MODULE_CONTRACT_VALID,

    "representation_restoration_invalid":
        BLOCK_1_REPRESENTATION_RESTORATION_EXACT,

    "transformative_restoration_invalid":
        BLOCK_1_TRANSFORMATIVE_RESTORATION_EXACT,

    "restored_parameter_counts_invalid":
        BLOCK_1_RESTORED_PARAMETER_COUNTS_VALID,

    "representation_not_frozen":
        BLOCK_1_REPRESENTATION_FROZEN,

    "transformative_not_frozen":
        BLOCK_1_TRANSFORMATIVE_FROZEN,

    "modules_not_in_eval_mode":
        BLOCK_1_ALL_MODULES_EVAL,

    "device_placement_invalid":
        BLOCK_1_DEVICE_PLACEMENT_VALID,

    "post_device_representation_invalid":
        BLOCK_1_POST_DEVICE_REPRESENTATION_EXACT,

    "post_device_transformative_invalid":
        BLOCK_1_POST_DEVICE_TRANSFORMATIVE_EXACT,

    "new_architectural_objects_present":
        BLOCK_1_NEW_ARCHITECTURAL_OBJECTS_ABSENT,
}


for (
    error_name,
    condition,
) in block_1_validation_checks.items():

    if not condition:

        NOTEBOOK_09_BLOCK_1_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation validation
# =============================================================================

prohibited_block_1_operations = {
    "representation_forward_incorrectly_executed":
        NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED,

    "transformative_forward_incorrectly_executed":
        NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED,

    "transformative_weights_incorrectly_created":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED,

    "recurrent_state_incorrectly_created":
        NOTEBOOK_09_RECURRENT_STATE_CREATED,

    "recurrent_update_incorrectly_executed":
        NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED,

    "loss_incorrectly_calculated":
        NOTEBOOK_09_LOSS_CALCULATED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BACKWARD_PASS_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in prohibited_block_1_operations.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_1_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 1 state
# =============================================================================

NOTEBOOK_09_HANDOVER_RESTORED = (
    len(
        NOTEBOOK_09_BLOCK_1_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_BLOCK_1_VALID = (
    NOTEBOOK_09_HANDOVER_RESTORED
)


if not NOTEBOOK_09_BLOCK_1_VALID:

    raise RuntimeError(
        "Notebook 09 Block 1 handover restoration and "
        "initialisation audit failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_1_ERRORS}"
    )


NOTEBOOK_09_INITIALISED = True

NOTEBOOK_09_BLOCK_1_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_1_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_1,

    "block_name":
        NOTEBOOK_09_BLOCK_1_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_1_VERSION,

    "executed_at_utc":
        BLOCK_1_EXECUTED_AT_UTC,

    "notebook":
        NOTEBOOK_09,

    "notebook_version":
        NOTEBOOK_09_VERSION,

    "checkpoint_drive_file_id":
        NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID,

    "metadata_drive_file_id":
        NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID,

    "checkpoint_sha256_valid":
        BLOCK_1_CHECKPOINT_SHA256_VALID,

    "checkpoint_identity_valid":
        BLOCK_1_CHECKPOINT_IDENTITY_VALID,

    "metadata_identity_valid":
        BLOCK_1_METADATA_IDENTITY_VALID,

    "cross_artefact_valid":
        BLOCK_1_CROSS_ARTEFACT_VALID,

    "representation_parameters":
        BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_TOTAL,

    "transformative_parameters":
        BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_TOTAL,

    "combined_parameters":
        BLOCK_1_RESTORED_PARAMETER_TOTAL,

    "pathway_dimensions":
        deepcopy(
            NOTEBOOK_09_PATHWAY_DIMS
        ),

    "representation_architecture_contract":
        deepcopy(
            BLOCK_1_REPRESENTATION_ARCHITECTURE_CONTRACT
        ),

    "transformative_architecture_contract":
        deepcopy(
            BLOCK_1_TRANSFORMATIVE_ARCHITECTURE_CONTRACT
        ),

    "active_dimension_indices":
        deepcopy(
            NOTEBOOK_09_ACTIVE_DIMENSION_INDICES
        ),

    "deferred_dimension_indices":
        deepcopy(
            NOTEBOOK_09_DEFERRED_DIMENSION_INDICES
        ),

    "candidate_transformations_available":
        NOTEBOOK_09_CANDIDATE_TRANSFORMATIONS_AVAILABLE,

    "transformative_weights_created":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED,

    "recurrent_state_created":
        NOTEBOOK_09_RECURRENT_STATE_CREATED,

    "recurrent_update_executed":
        NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED,

    "handover_restored":
        NOTEBOOK_09_HANDOVER_RESTORED,

    "block_valid":
        NOTEBOOK_09_BLOCK_1_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_1_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 1: "
    "Notebook 08 → 09 Handover Restoration and Initialisation Audit"
)

print("=" * 72)

print(
    f"Notebook                     : "
    f"{NOTEBOOK_09}"
)

print(
    f"Notebook version             : "
    f"{NOTEBOOK_09_VERSION}"
)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_1_VERSION}"
)

print(
    f"Project random seed          : "
    f"{NOTEBOOK_09_RANDOM_SEED}"
)

print(
    f"Execution device             : "
    f"{DEVICE}"
)

print("-" * 72)

print(
    "Google authentication"
)

print(
    f"Colab authenticated          : "
    f"{BLOCK_1_COLAB_AUTHENTICATED}"
)

print(
    f"Drive service initialised    : "
    f"{BLOCK_1_DRIVE_SERVICE_INITIALISED}"
)

print("-" * 72)

print(
    "Persistent Notebook 08 → 09 handover"
)

print(
    f"Checkpoint filename          : "
    f"{NOTEBOOK_08_TO_09_CHECKPOINT_FILENAME}"
)

print(
    f"Checkpoint Drive file ID     : "
    f"{NOTEBOOK_08_TO_09_CHECKPOINT_DRIVE_FILE_ID}"
)

print(
    f"Checkpoint located           : "
    f"{BLOCK_1_CHECKPOINT_LOCATED}"
)

print(
    f"Checkpoint downloaded        : "
    f"{BLOCK_1_CHECKPOINT_DOWNLOADED}"
)

print(
    f"Checkpoint SHA-256 valid     : "
    f"{BLOCK_1_CHECKPOINT_SHA256_VALID}"
)

print(
    f"Checkpoint byte count valid  : "
    f"{BLOCK_1_CHECKPOINT_BYTE_COUNT_VALID}"
)

print(
    f"Checkpoint size              : "
    f"{len(BLOCK_1_CHECKPOINT_BYTES) / (1024 ** 2):.3f} MB"
)

print(
    f"Metadata filename            : "
    f"{NOTEBOOK_08_TO_09_METADATA_FILENAME}"
)

print(
    f"Metadata Drive file ID       : "
    f"{NOTEBOOK_08_TO_09_METADATA_DRIVE_FILE_ID}"
)

print(
    f"Metadata located             : "
    f"{BLOCK_1_METADATA_LOCATED}"
)

print(
    f"Metadata downloaded          : "
    f"{BLOCK_1_METADATA_DOWNLOADED}"
)

print(
    f"Metadata JSON parsed         : "
    f"{BLOCK_1_METADATA_JSON_PARSED}"
)

print(
    f"Metadata size                : "
    f"{len(BLOCK_1_METADATA_BYTES) / 1024:.3f} KB"
)

print("-" * 72)

print(
    "Inherited architecture"
)

print(
    f"Representation parameters    : "
    f"{BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_TOTAL:,}"
)

print(
    f"Transformative parameters    : "
    f"{BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_TOTAL:,}"
)

print(
    f"Combined inherited params    : "
    f"{BLOCK_1_RESTORED_PARAMETER_TOTAL:,}"
)

print("-" * 72)

print(
    "Transformative pathway contracts"
)

for pathway_name in NOTEBOOK_09_PATHWAY_DIMS:

    print(
        f"{pathway_name:<14} dimension          : "
        f"{NOTEBOOK_09_PATHWAY_DIMS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} active dimensions  : "
        f"{list(NOTEBOOK_09_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} deferred dimensions: "
        f"{list(NOTEBOOK_09_DEFERRED_DIMENSION_INDICES[pathway_name])}"
    )

print("-" * 72)

print(
    "Restoration validation"
)

print(
    f"Checkpoint identity valid    : "
    f"{BLOCK_1_CHECKPOINT_IDENTITY_VALID}"
)

print(
    f"Metadata identity valid      : "
    f"{BLOCK_1_METADATA_IDENTITY_VALID}"
)

print(
    f"Handover identity valid      : "
    f"{BLOCK_1_HANDOVER_IDENTITY_VALID}"
)

print(
    f"Cross-artefact valid         : "
    f"{BLOCK_1_CROSS_ARTEFACT_VALID}"
)

print(
    f"Parameter counts valid       : "
    f"{BLOCK_1_PARAMETER_CONTRACT_VALID}"
)

print(
    f"Pathway contract valid       : "
    f"{BLOCK_1_PATHWAY_CROSS_ARTEFACT_VALID}"
)


print(
    f"Architecture contracts valid : "
    f"{BLOCK_1_ARCHITECTURE_CONTRACTS_PRESENT}"
)

print(
    f"Representation state exact   : "
    f"{BLOCK_1_REPRESENTATION_RESTORATION_EXACT}"
)

print(
    f"Transformative state exact   : "
    f"{BLOCK_1_TRANSFORMATIVE_RESTORATION_EXACT}"
)

print(
    f"Post-device repr. exact      : "
    f"{BLOCK_1_POST_DEVICE_REPRESENTATION_EXACT}"
)

print(
    f"Post-device transform exact  : "
    f"{BLOCK_1_POST_DEVICE_TRANSFORMATIVE_EXACT}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_1_REPRESENTATION_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_1_TRANSFORMATIVE_FROZEN}"
)

print(
    f"All inherited modules eval   : "
    f"{BLOCK_1_ALL_MODULES_EVAL}"
)

print(
    f"Device placement valid       : "
    f"{BLOCK_1_DEVICE_PLACEMENT_VALID}"
)

print("-" * 72)

print(
    "Notebook 09 execution boundary"
)

print(
    f"Candidate transformations    : "
    f"{NOTEBOOK_09_CANDIDATE_TRANSFORMATIONS_AVAILABLE}"
)

print(
    f"Transform weights created    : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED}"
)

print(
    f"Recurrent state created      : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CREATED}"
)

print(
    f"Recurrent update executed    : "
    f"{NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"Representation forward pass  : "
    f"{NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED}"
)

print(
    f"Transform forward pass       : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_LOSS_CALCULATED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Training executed            : "
    f"{NOTEBOOK_09_TRAINING_EXECUTED}"
)

print("-" * 72)

print(
    "Readiness"
)

print(
    f"Notebook 08 → 09 valid       : "
    f"{NOTEBOOK_09_HANDOVER_RESTORED}"
)

print(
    f"Notebook 09 initialised      : "
    f"{NOTEBOOK_09_INITIALISED}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_1_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_1_COMPLETE}"
)

print("=" * 72)

print(
    "Notebook 09 initialised successfully from the persistent "
    "Notebook 08 → Notebook 09 handover."
)

print(
    "The validated representation model and all trained transformative "
    "mechanisms were restored from the canonical Notebook 08 checkpoint."
)

print(
    f"All {BLOCK_1_RESTORED_REPRESENTATION_PARAMETER_TOTAL:,} "
    f"representation parameters and all "
    f"{BLOCK_1_RESTORED_TRANSFORMATIVE_PARAMETER_TOTAL:,} transformative "
    "parameters were recovered exactly."
)

print(
    f"The complete inherited {BLOCK_1_RESTORED_PARAMETER_TOTAL:,}-parameter "
    "architecture remains frozen and in evaluation mode."
)

print(
    "Pathway dimensions and active/deferred transformation boundaries "
    "were inherited directly from Notebook 08."
)

print(
    "Candidate transformations are available through the restored "
    "transformative mechanisms; no candidate tensor was materialised "
    "by Block 1."
)

print(
    "No Transformative Weight has been inherited, inferred or created."
)

print(
    "No recurrent state has been inherited or instantiated."
)

print(
    "No recurrent update has been executed."
)

print(
    "No representation or transformative forward pass, loss calculation, "
    "optimiser creation, backward pass, parameter update or training "
    "has been performed by Block 1."
)

print(
    "Block 2 may proceed to the explicit Transformative-Weight "
    "architectural contract and parameterisation policy."
)

print("=" * 72)

Media AI — Notebook 09, Block 1: Notebook 08 → 09 Handover Restoration and Initialisation Audit
Notebook                     : 09_transformative_weight_and_recurrent_state
Notebook version             : 1.0
Block version                : 1.3
Project random seed          : 42
Execution device             : cpu
------------------------------------------------------------------------
Google authentication
Colab authenticated          : True
Drive service initialised    : True
------------------------------------------------------------------------
Persistent Notebook 08 → 09 handover
Checkpoint filename          : notebook_08_final_checkpoint.pt
Checkpoint Drive file ID     : 1cjxowL02MQWbQ0lHCoET6QVklGVkhSsp
Checkpoint located           : True
Checkpoint downloaded        : True
Checkpoint SHA-256 valid     : True
Checkpoint byte count valid  : True
Checkpoint size              : 3.460 MB
Metadata filename            : notebook_08_final_metadata.json
Metadata Drive file ID       : 1_A2mO

## Block 2 — Transformative-Weight Architectural Contract and Parameterisation Policy

This block defines the architectural role, dimensional form, and admissible behaviour of the Transformative Weights used by the factual, psychological, and social recurrent pathways.

Block 1 restored the complete validated Notebook 08 handover and established the precise Notebook 09 starting state. The representation model and the three trained transformative mechanisms are available, frozen, and in evaluation mode. Candidate transformations can therefore be produced by the inherited architecture, but no Transformative Weight and no recurrent state yet exists.

The purpose of Block 2 is to define the new weighting mechanism **before any parameter object is created**.

No Transformative Weight is instantiated in this block. No recurrent state is created. No forward pass, loss calculation, optimiser construction, backpropagation, or parameter update is executed.

### Architectural role of the Transformative Weight

For each pathway \(k\in\{F,P,S\}\), the inherited transformative mechanism produces a candidate transformation

$$
c_t^{(k)}\in\mathbb{R}^{d_k},
$$

where

$$
d_F=10,\qquad d_P=34,\qquad d_S=7.
$$

The candidate transformation describes the learned direction and magnitude of a possible state change. It does not determine how strongly that change should be admitted into the recurrent state.

The Transformative Weight provides this separate control function.

The recurrent contribution of a candidate transformation is therefore written conceptually as

$$
\Delta h_t^{(k)}
=
TW_k \odot c_t^{(k)},
$$

where \(\odot\) denotes elementwise weighting under the parameterisation adopted in this notebook.

The subsequent recurrent update will take the general form

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
\Delta h_t^{(k)}.
$$

Block 2 defines the admissible form of \(TW_k\), but it does not yet execute this update.

### Why a pathway scalar is insufficient

A single scalar Transformative Weight per pathway,

$$
TW_k\in\mathbb{R},
$$

would apply one common transformation strength to every semantic dimension within that pathway.

That would imply, for example, that every psychological dimension receives the same transformation strength despite the psychological representation containing 34 semantically distinct dimensions.

Such a parameterisation would be unnecessarily restrictive.

The inherited representation spaces are explicitly multidimensional, and Notebook 08 already established dimension-specific active and deferred transformation support. The Transformative-Weight contract should therefore preserve that granularity rather than collapse it into a single pathway-level coefficient.

### Dimension-wise Transformative Weights

Notebook 09 adopts a **dimension-wise Transformative-Weight vector** for each pathway.

The three weight vectors therefore have the forms

$$
TW_F\in\mathbb{R}^{10},
$$

$$
TW_P\in\mathbb{R}^{34},
$$

and

$$
TW_S\in\mathbb{R}^{7}.
$$

For dimension \(j\) of pathway \(k\), the weighted transformation contribution is

$$
\Delta h_{t,j}^{(k)}
=
TW_{k,j}\,
c_{t,j}^{(k)}.
$$

This gives every semantic dimension its own transformation-strength parameter while preserving complete separation between the factual, psychological, and social pathways.

The resulting structural parameter space contains

$$
10+34+7=51
$$

Transformative-Weight dimensions.

The inherited 902,190 representation and transformative parameters remain outside this new parameter scope.

### Bounded weighting contract

The Transformative Weight should control the extent to which a candidate transformation enters the recurrent state without arbitrarily reversing its learned direction or amplifying it without bound.

For this reason, the effective Transformative Weights are constrained to

$$
0\leq TW_{k,j}\leq1.
$$

A weight of zero means that the corresponding candidate transformation does not enter the recurrent state:

$$
TW_{k,j}=0
\quad\Rightarrow\quad
\Delta h_{t,j}^{(k)}=0.
$$

A weight of one admits the full candidate transformation:

$$
TW_{k,j}=1
\quad\Rightarrow\quad
\Delta h_{t,j}^{(k)}
=
c_{t,j}^{(k)}.
$$

Intermediate values provide partial admission.

This preserves the sign and direction learned by the candidate-transformative mechanism while allowing Notebook 09 to learn the strength of recurrent incorporation.

### Unconstrained learnable parameters and bounded effective weights

Directly optimising parameters under hard interval constraints is unnecessarily awkward.

Instead, Notebook 09 defines an unconstrained latent parameter vector for each pathway:

$$
\alpha_F\in\mathbb{R}^{10},
$$

$$
\alpha_P\in\mathbb{R}^{34},
$$

$$
\alpha_S\in\mathbb{R}^{7}.
$$

The effective Transformative Weights are obtained through a sigmoid mapping:

$$
TW_k
=
\sigma(\alpha_k),
$$

where

$$
\sigma(x)
=
\frac{1}{1+e^{-x}}.
$$

Therefore,

$$
0<TW_{k,j}<1
$$

for all finite latent parameters.

This parameterisation provides smooth differentiability, bounded recurrent contributions, and a direct interpretation as transformation-admission strength.

The latent variables \(\alpha_k\) are the trainable parameters. The bounded vectors \(TW_k\) are their effective architectural values.

### Neutral initialisation policy

The Transformative Weights must begin from an explicit and reproducible initial condition.

Notebook 09 adopts a neutral effective weight of

$$
TW_{k,j}^{(0)}=0.5
$$

for every currently eligible dimension.

Under sigmoid parameterisation,

$$
\sigma(0)=0.5.
$$

The corresponding latent parameters therefore initialise at

$$
\alpha_{k,j}^{(0)}=0.
$$

This avoids beginning with either complete suppression or complete admission of the candidate transformation.

It also avoids imposing directional prior knowledge that is not supported by the current pilot data.

### Active and deferred dimensions

Notebook 08 established active and deferred transformative dimensions independently for the three representation spaces.

For the factual pathway,

$$
A_F=\{1,2,4,8,9\}.
$$

For the psychological pathway,

$$
A_P=
\{1,4,11,12,14,15,16,19,21,22,23,25,26,27,30,32,33\}.
$$

For the social pathway,

$$
A_S=\{0,2,4\}.
$$

Only these active dimensions currently have sufficient validated pilot support for learned transformative behaviour.

The remaining dimensions are deferred.

Deferred dimensions remain structurally present in the recurrent state, but their Transformative Weights must not become trainable merely because the recurrent architecture now exists.

The initial Notebook 09 policy is therefore

$$
TW_{k,j}=0
\qquad
\text{for }j\in D_k,
$$

where \(D_k\) denotes the deferred dimensions of pathway \(k\).

This guarantees that unsupported candidate transformations do not enter the recurrent state during the current controlled experiment.

The deferred dimensions are not deleted and are not permanently fixed by the architecture. They remain eligible for future activation when adequate supervision becomes available.

### Trainable Transformative-Weight scope

The currently trainable weight parameters therefore correspond only to active dimensions.

The factual pathway contributes

$$
|A_F|=5
$$

trainable latent parameters.

The psychological pathway contributes

$$
|A_P|=17.
$$

The social pathway contributes

$$
|A_S|=3.
$$

The current trainable Transformative-Weight scope is therefore

$$
5+17+3=25
$$

parameters.

The full structural Transformative-Weight space contains 51 dimensions, but only 25 are currently activation-eligible.

This distinction mirrors the active/deferred policy inherited from Notebook 08.

### Separation from optimisation-loss coefficients

The Transformative Weights must remain distinct from all earlier loss-combination coefficients.

Notebook 08 used equal pathway coefficients

$$
\lambda_F
=
\lambda_P
=
\lambda_S
=
\frac{1}{3}
$$

to combine factual, psychological, and social training losses.

Those coefficients controlled optimisation balance only.

They do not represent state-transition strength and are not reused as

$$
TW_F,\qquad TW_P,\qquad TW_S.
$$

The architectural relation is therefore

$$
\lambda_k \neq TW_k.
$$

Likewise, the candidate-transformation magnitude itself does not define the Transformative Weight:

$$
c_t^{(k)}\neq TW_k.
$$

### Separation from recurrent state

The Transformative Weight is also distinct from recurrent state.

The recurrent state has pathway dimensions

$$
h_t^{(F)}\in\mathbb{R}^{10},
$$

$$
h_t^{(P)}\in\mathbb{R}^{34},
$$

$$
h_t^{(S)}\in\mathbb{R}^{7}.
$$

The Transformative Weight controls how a candidate transformation contributes to state evolution but does not itself store temporal information.

Conceptually,

$$
TW_k
\neq
h_t^{(k)}.
$$

The weight is a learned model parameter; the recurrent state is a sequence-dependent runtime quantity.

### Pathway independence

Notebook 09 preserves the pathway separation established upstream.

The factual Transformative Weight affects only the factual candidate transformation:

$$
\Delta h_t^{(F)}
=
TW_F\odot c_t^{(F)}.
$$

The psychological Transformative Weight affects only the psychological candidate transformation:

$$
\Delta h_t^{(P)}
=
TW_P\odot c_t^{(P)}.
$$

The social Transformative Weight affects only the social candidate transformation:

$$
\Delta h_t^{(S)}
=
TW_S\odot c_t^{(S)}.
$$

No cross-pathway weight matrix is introduced.

Accordingly, Block 2 does not permit relations such as

$$
TW_{FP},
\qquad
TW_{FS},
\qquad
TW_{PS}.
$$

Cross-pathway interaction remains outside the present recurrent-weight contract unless introduced explicitly in a later architectural stage.

### Inherited parameter-freezing policy

The complete inherited architecture remains frozen.

The inherited representation parameters satisfy

$$
\nabla_{\theta_R}=0,
$$

while the inherited transformative parameters satisfy

$$
\nabla_{\theta_T}=0.
$$

The only future trainable parameters introduced by the Transformative-Weight stage are the active latent weight parameters

$$
\alpha_{k,j},
\qquad
j\in A_k.
$$

This creates a strict optimisation boundary between previously trained architecture and newly introduced recurrent weighting.

### No recurrent execution in Block 2

Although the Transformative-Weight contract is defined here, Block 2 does not yet execute recurrent state evolution.

In particular, the block does not instantiate or update

$$
h_t^{(F)},
\qquad
h_t^{(P)},
\qquad
h_t^{(S)}.
$$

It also does not yet evaluate

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}.
$$

The recurrent state and its deterministic initialisation are introduced only after the Transformative-Weight architecture itself has been constructed and validated.

### Block 2 completion contract

Block 2 is complete when the following architectural decisions have been established without executing training:

- Transformative Weights are dimension-wise rather than scalar;
- the full factual, psychological, and social weight spaces have dimensions 10, 34, and 7;
- the complete structural Transformative-Weight space contains 51 dimensions;
- effective weights are bounded to the interval \((0,1)\) through sigmoid parameterisation;
- latent trainable parameters initialise at zero;
- active dimensions begin with effective weight 0.5;
- deferred dimensions remain structurally present but are fixed at zero under the current pilot contract;
- only 25 active dimensions are currently eligible for Transformative-Weight learning;
- factual, psychological, and social weights remain independent;
- Transformative Weights remain distinct from candidate transformations;
- Transformative Weights remain distinct from optimisation-loss coefficients;
- Transformative Weights remain distinct from recurrent state;
- the inherited 902,190 parameters remain frozen;
- no recurrent state has been instantiated;
- no forward pass has been executed;
- no loss has been calculated;
- no optimiser has been created;
- and no parameter update has occurred.

Successful completion establishes the architectural contract

$$
\boxed{
TW_F\in[0,1]^{10},
\qquad
TW_P\in[0,1]^{34},
\qquad
TW_S\in[0,1]^{7}
}
$$

with only the inherited active dimensions currently eligible for learning.

Notebook 09 may then proceed to controlled Transformative-Weight module construction and parameter-scope validation.

In [52]:
# =============================================================================
# Media AI — Notebook 09
# Block 2: Transformative-Weight Architectural Contract
#          and Parameterisation Policy
# =============================================================================

from copy import deepcopy

import numpy as np
import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_2 = 2

NOTEBOOK_09_BLOCK_2_NAME = (
    "Transformative-Weight Architectural Contract "
    "and Parameterisation Policy"
)

NOTEBOOK_09_BLOCK_2_VERSION = "1.2"


# =============================================================================
# Required Block 1 state
# =============================================================================

BLOCK_2_REQUIRED_OBJECTS = (
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_BLOCK_1_COMPLETE",
    "NOTEBOOK_09_BLOCK_1_VALID",
    "NOTEBOOK_09_HANDOVER_RESTORED",
    "NOTEBOOK_09_PATHWAY_DIMS",
    "NOTEBOOK_09_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_DEFERRED_DIMENSION_INDICES",
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",
    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",
    "BLOCK_1_REPRESENTATION_FROZEN",
    "BLOCK_1_TRANSFORMATIVE_FROZEN",
    "BLOCK_1_ALL_MODULES_EVAL",
)


BLOCK_2_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_2_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_2_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 2 prerequisites are not initialised. "
        f"Missing: {BLOCK_2_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate Block 1 completion
# =============================================================================

BLOCK_2_BLOCK_1_READY = all(
    [
        NOTEBOOK_09_INITIALISED is True,
        NOTEBOOK_09_BLOCK_1_COMPLETE is True,
        NOTEBOOK_09_BLOCK_1_VALID is True,
        NOTEBOOK_09_HANDOVER_RESTORED is True,
    ]
)


if not BLOCK_2_BLOCK_1_READY:

    raise RuntimeError(
        "Notebook 09 Block 1 is not in a valid completed state."
    )


# =============================================================================
# Downstream-runtime snapshot for rerun-safe execution
# =============================================================================
#
# Block 2 is a contract-definition block. It must not create Transformative-
# Weight modules or recurrent-state objects itself.
#
# In an interactive notebook, however, Block 2 may legitimately be rerun after
# Block 3 or later blocks have already executed. Therefore Block 2 records any
# downstream objects that existed BEFORE this execution and later verifies that
# it did not create additional downstream objects.
# =============================================================================

BLOCK_2_DOWNSTREAM_WEIGHT_OBJECT_NAMES = (
    "TW_F",
    "TW_P",
    "TW_S",

    "NOTEBOOK_09_TW_F",
    "NOTEBOOK_09_TW_P",
    "NOTEBOOK_09_TW_S",

    "NOTEBOOK_09_ALPHA_F",
    "NOTEBOOK_09_ALPHA_P",
    "NOTEBOOK_09_ALPHA_S",

    "NOTEBOOK_09_FACTUAL_TRANSFORMATIVE_WEIGHT",
    "NOTEBOOK_09_PSYCHOLOGICAL_TRANSFORMATIVE_WEIGHT",
    "NOTEBOOK_09_SOCIAL_TRANSFORMATIVE_WEIGHT",
)


BLOCK_2_DOWNSTREAM_RECURRENT_OBJECT_NAMES = (
    "NOTEBOOK_09_FACTUAL_RECURRENT_STATE",
    "NOTEBOOK_09_PSYCHOLOGICAL_RECURRENT_STATE",
    "NOTEBOOK_09_SOCIAL_RECURRENT_STATE",
)


BLOCK_2_PREEXISTING_WEIGHT_OBJECTS = tuple(
    object_name

    for object_name
    in BLOCK_2_DOWNSTREAM_WEIGHT_OBJECT_NAMES

    if object_name in globals()
)


BLOCK_2_PREEXISTING_RECURRENT_OBJECTS = tuple(
    object_name

    for object_name
    in BLOCK_2_DOWNSTREAM_RECURRENT_OBJECT_NAMES

    if object_name in globals()
)


BLOCK_2_RERUN_DETECTED = bool(
    BLOCK_2_PREEXISTING_WEIGHT_OBJECTS
    or
    BLOCK_2_PREEXISTING_RECURRENT_OBJECTS
)


# =============================================================================
# Inherited architecture and activation contracts
# =============================================================================
#
# Notebook 09 Block 2 inherits the final Notebook 08 pathway dimensions and
# active/deferred transformation boundaries exactly as restored by Block 1.
# It does not recreate pilot-era dimension lists.
# =============================================================================

BLOCK_2_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_PATHWAY_DIMS
)


BLOCK_2_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_ACTIVE_DIMENSION_INDICES.items()
}


BLOCK_2_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_DEFERRED_DIMENSION_INDICES.items()
}


BLOCK_2_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            BLOCK_2_PATHWAY_DIMENSIONS,
            dict,
        ),

        bool(
            BLOCK_2_PATHWAY_DIMENSIONS
        ),

        (
            set(
                BLOCK_2_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                BLOCK_2_ACTIVE_DIMENSION_INDICES.keys()
            )
            ==
            set(
                BLOCK_2_DEFERRED_DIMENSION_INDICES.keys()
            )
        ),

        all(
            isinstance(
                pathway_dim,
                int,
            )
            and
            pathway_dim > 0

            for pathway_dim
            in BLOCK_2_PATHWAY_DIMENSIONS.values()
        ),
    ]
)


if not BLOCK_2_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 09 inherited pathway dimension contract is invalid."
    )


BLOCK_2_ACTIVE_DIMENSIONS_VALID = all(
    all(
        isinstance(
            dimension_index,
            int,
        )
        and
        0
        <=
        dimension_index
        <
        BLOCK_2_PATHWAY_DIMENSIONS[
            pathway_name
        ]

        for dimension_index
        in active_indices
    )

    for (
        pathway_name,
        active_indices,
    ) in BLOCK_2_ACTIVE_DIMENSION_INDICES.items()
)


BLOCK_2_DEFERRED_DIMENSIONS_VALID = all(
    all(
        isinstance(
            dimension_index,
            int,
        )
        and
        0
        <=
        dimension_index
        <
        BLOCK_2_PATHWAY_DIMENSIONS[
            pathway_name
        ]

        for dimension_index
        in deferred_indices
    )

    for (
        pathway_name,
        deferred_indices,
    ) in BLOCK_2_DEFERRED_DIMENSION_INDICES.items()
)


if not BLOCK_2_ACTIVE_DIMENSIONS_VALID:

    raise RuntimeError(
        "Inherited active transformative dimensions are invalid."
    )


if not BLOCK_2_DEFERRED_DIMENSIONS_VALID:

    raise RuntimeError(
        "Inherited deferred transformative dimensions are invalid."
    )


# =============================================================================
# Validate active/deferred partition
# =============================================================================

BLOCK_2_DIMENSION_PARTITION_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_2_PATHWAY_DIMENSIONS.items():

    active_indices = set(
        BLOCK_2_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    deferred_indices = set(
        BLOCK_2_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )


    complete_indices = set(
        range(
            pathway_dim
        )
    )


    BLOCK_2_DIMENSION_PARTITION_VALID[
        pathway_name
    ] = all(
        [
            active_indices.isdisjoint(
                deferred_indices
            ),

            (
                active_indices
                |
                deferred_indices
            )
            ==
            complete_indices,
        ]
    )


BLOCK_2_ALL_DIMENSION_PARTITIONS_VALID = all(
    BLOCK_2_DIMENSION_PARTITION_VALID.values()
)


if not BLOCK_2_ALL_DIMENSION_PARTITIONS_VALID:

    raise RuntimeError(
        "Active and deferred dimensions do not form complete "
        "non-overlapping pathway partitions."
    )


# =============================================================================
# Transformative-Weight structural dimensional contract
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DIMENSIONS = {
    pathway_name:
        pathway_dim

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_2_PATHWAY_DIMENSIONS.items()
}


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS = int(
    sum(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DIMENSIONS.values()
    )
)


BLOCK_2_STRUCTURAL_DIMENSIONS_VALID = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS
    ==
    sum(
        BLOCK_2_PATHWAY_DIMENSIONS.values()
    )
)


if not BLOCK_2_STRUCTURAL_DIMENSIONS_VALID:

    raise RuntimeError(
        "Transformative-Weight structural dimensionality is invalid."
    )


# =============================================================================
# Active trainable scope
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_COUNTS = {
    pathway_name:
        len(
            BLOCK_2_ACTIVE_DIMENSION_INDICES[
                pathway_name
            ]
        )

    for pathway_name
    in BLOCK_2_PATHWAY_DIMENSIONS
}


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_COUNTS = {
    pathway_name:
        len(
            BLOCK_2_DEFERRED_DIMENSION_INDICES[
                pathway_name
            ]
        )

    for pathway_name
    in BLOCK_2_PATHWAY_DIMENSIONS
}


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT = int(
    sum(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_COUNTS.values()
    )
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT = int(
    sum(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_COUNTS.values()
    )
)


BLOCK_2_ACTIVE_COUNTS_VALID = all(
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_COUNTS[
        pathway_name
    ]
    ==
    len(
        BLOCK_2_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )

    for pathway_name
    in BLOCK_2_PATHWAY_DIMENSIONS
)


BLOCK_2_DEFERRED_COUNTS_VALID = all(
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_COUNTS[
        pathway_name
    ]
    ==
    len(
        BLOCK_2_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )

    for pathway_name
    in BLOCK_2_PATHWAY_DIMENSIONS
)


BLOCK_2_TRAINABLE_PARAMETER_COUNT_VALID = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
    ==
    sum(
        len(
            BLOCK_2_ACTIVE_DIMENSION_INDICES[
                pathway_name
            ]
        )

        for pathway_name
        in BLOCK_2_PATHWAY_DIMENSIONS
    )
)


BLOCK_2_DEFERRED_DIMENSION_COUNT_VALID = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT
    ==
    sum(
        len(
            BLOCK_2_DEFERRED_DIMENSION_INDICES[
                pathway_name
            ]
        )

        for pathway_name
        in BLOCK_2_PATHWAY_DIMENSIONS
    )
)


BLOCK_2_TOTAL_DIMENSION_ACCOUNTING_VALID = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
    +
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT
    ==
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS
)


if not all(
    [
        BLOCK_2_ACTIVE_COUNTS_VALID,
        BLOCK_2_DEFERRED_COUNTS_VALID,
        BLOCK_2_TRAINABLE_PARAMETER_COUNT_VALID,
        BLOCK_2_DEFERRED_DIMENSION_COUNT_VALID,
        BLOCK_2_TOTAL_DIMENSION_ACCOUNTING_VALID,
    ]
):

    raise RuntimeError(
        "Transformative-Weight active/deferred parameter accounting "
        "is invalid."
    )


# =============================================================================
# Transformative-Weight parameterisation contract
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETERISATION = (
    "dimension_wise_sigmoid_gated"
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_LATENT_PARAMETER = (
    "alpha"
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_EFFECTIVE_PARAMETER = (
    "sigmoid(alpha)"
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MINIMUM = (
    0.0
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MAXIMUM = (
    1.0
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT = (
    0.0
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE = (
    float(
        torch.sigmoid(
            torch.tensor(
                0.0,
                dtype=
                    torch.float64,
            )
        ).item()
    )
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE = (
    0.0
)


BLOCK_2_INITIALISATION_VALID = all(
    [
        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT
            ==
            0.0
        ),

        np.isclose(
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE,
            0.5,
            rtol=
                0.0,
            atol=
                1e-12,
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE
            ==
            0.0
        ),
    ]
)


if not BLOCK_2_INITIALISATION_VALID:

    raise RuntimeError(
        "Transformative-Weight initialisation policy is invalid."
    )


# =============================================================================
# Effective-weight policy
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY = {
    "granularity":
        "dimension_wise",

    "pathway_independence":
        True,

    "latent_parameterisation":
        "unconstrained_real",

    "effective_mapping":
        "sigmoid",

    "effective_interval":
        (
            0.0,
            1.0,
        ),

    "active_initial_latent":
        0.0,

    "active_initial_effective":
        0.5,

    "deferred_effective":
        0.0,

    "deferred_trainable":
        False,

    "candidate_direction_preserved":
        True,

    "cross_pathway_weights":
        False,
}


BLOCK_2_WEIGHT_POLICY_VALID = all(
    [
        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY[
                "granularity"
            ]
            ==
            "dimension_wise"
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY[
                "pathway_independence"
            ]
            is True
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY[
                "latent_parameterisation"
            ]
            ==
            "unconstrained_real"
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY[
                "effective_mapping"
            ]
            ==
            "sigmoid"
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY[
                "effective_interval"
            ]
            ==
            (
                0.0,
                1.0,
            )
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY[
                "deferred_trainable"
            ]
            is False
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY[
                "candidate_direction_preserved"
            ]
            is True
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY[
                "cross_pathway_weights"
            ]
            is False
        ),
    ]
)


if not BLOCK_2_WEIGHT_POLICY_VALID:

    raise RuntimeError(
        "Transformative-Weight parameterisation policy is invalid."
    )


# =============================================================================
# Conceptual recurrent-update contract
# =============================================================================
#
# This contract is descriptive only.
#
# No recurrent state is instantiated and no recurrent update is executed.
#
# For pathway k:
#
# candidate transformation:
#
#     c_t^(k)
#
# effective Transformative Weight:
#
#     TW_k = sigmoid(alpha_k)
#
# weighted recurrent contribution:
#
#     delta_h_t^(k) = TW_k ⊙ c_t^(k)
#
# future recurrent update:
#
#     h_t^(k) = h_(t-1)^(k) + delta_h_t^(k)
#
# Deferred dimensions are explicitly gated to zero.
# =============================================================================

NOTEBOOK_09_RECURRENT_UPDATE_CONTRACT = {
    "candidate_symbol":
        "c_t",

    "transformative_weight_symbol":
        "TW",

    "latent_weight_symbol":
        "alpha",

    "weighted_contribution":
        "TW_k * candidate_transformation",

    "state_update":
        "previous_state + weighted_contribution",

    "elementwise_weighting":
        True,

    "future_context_allowed":
        False,

    "executed":
        False,
}


BLOCK_2_RECURRENT_CONTRACT_VALID = all(
    [
        (
            NOTEBOOK_09_RECURRENT_UPDATE_CONTRACT[
                "elementwise_weighting"
            ]
            is True
        ),

        (
            NOTEBOOK_09_RECURRENT_UPDATE_CONTRACT[
                "future_context_allowed"
            ]
            is False
        ),

        (
            NOTEBOOK_09_RECURRENT_UPDATE_CONTRACT[
                "executed"
            ]
            is False
        ),
    ]
)


# =============================================================================
# Architectural separation contract
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_SEPARATION_POLICY = {
    "distinct_from_candidate_transformations":
        True,

    "distinct_from_optimisation_loss_weights":
        True,

    "distinct_from_recurrent_state":
        True,

    "distinct_from_representation_parameters":
        True,

    "distinct_from_transformative_mechanism_parameters":
        True,
}


BLOCK_2_SEPARATION_POLICY_VALID = all(
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_SEPARATION_POLICY.values()
)


if not BLOCK_2_SEPARATION_POLICY_VALID:

    raise RuntimeError(
        "Transformative-Weight architectural separation contract "
        "is invalid."
    )


# =============================================================================
# Inherited architecture remains frozen
# =============================================================================

BLOCK_2_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_2_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_2_INHERITED_MODULES_STILL_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


BLOCK_2_INHERITED_PARAMETER_COUNTS_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),
    ]
)


if not all(
    [
        BLOCK_2_REPRESENTATION_STILL_FROZEN,
        BLOCK_2_TRANSFORMATIVE_STILL_FROZEN,
        BLOCK_2_INHERITED_MODULES_STILL_EVAL,
        BLOCK_2_INHERITED_PARAMETER_COUNTS_VALID,
    ]
):

    raise RuntimeError(
        "Inherited Notebook 08 architecture changed during "
        "Notebook 09 Block 2."
    )


# =============================================================================
# Explicit execution boundary
# =============================================================================

NOTEBOOK_09_BLOCK_2_TRANSFORMATIVE_WEIGHT_OBJECT_CREATED = False

NOTEBOOK_09_BLOCK_2_RECURRENT_STATE_CREATED = False

NOTEBOOK_09_BLOCK_2_RECURRENT_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_2_REPRESENTATION_FORWARD_EXECUTED = False

NOTEBOOK_09_BLOCK_2_TRANSFORM_FORWARD_EXECUTED = False

NOTEBOOK_09_BLOCK_2_WEIGHT_FORWARD_EXECUTED = False

NOTEBOOK_09_BLOCK_2_LOSS_CALCULATED = False

NOTEBOOK_09_BLOCK_2_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_2_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_2_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_2_TRAINING_EXECUTED = False


# =============================================================================
# Runtime downstream-object creation audit
# =============================================================================
#
# On a first clean run, no downstream weight/recurrent objects should exist.
# On a notebook rerun, downstream objects may already exist from later blocks.
# The invariant enforced here is therefore:
#
#     Block 2 must not CREATE any new downstream objects.
#
# This preserves the architectural boundary while making Block 2 idempotent
# under normal Jupyter/Colab cell re-execution.
# =============================================================================

BLOCK_2_WEIGHT_OBJECTS_PRESENT_AFTER = tuple(
    object_name

    for object_name
    in BLOCK_2_DOWNSTREAM_WEIGHT_OBJECT_NAMES

    if object_name in globals()
)


BLOCK_2_RECURRENT_OBJECTS_PRESENT_AFTER = tuple(
    object_name

    for object_name
    in BLOCK_2_DOWNSTREAM_RECURRENT_OBJECT_NAMES

    if object_name in globals()
)


BLOCK_2_NEW_WEIGHT_OBJECTS_CREATED = tuple(
    object_name

    for object_name
    in BLOCK_2_WEIGHT_OBJECTS_PRESENT_AFTER

    if object_name not in BLOCK_2_PREEXISTING_WEIGHT_OBJECTS
)


BLOCK_2_NEW_RECURRENT_OBJECTS_CREATED = tuple(
    object_name

    for object_name
    in BLOCK_2_RECURRENT_OBJECTS_PRESENT_AFTER

    if object_name not in BLOCK_2_PREEXISTING_RECURRENT_OBJECTS
)


BLOCK_2_WEIGHT_OBJECTS_NOT_CREATED_BY_BLOCK_2 = (
    len(
        BLOCK_2_NEW_WEIGHT_OBJECTS_CREATED
    )
    ==
    0
)


BLOCK_2_RECURRENT_OBJECTS_NOT_CREATED_BY_BLOCK_2 = (
    len(
        BLOCK_2_NEW_RECURRENT_OBJECTS_CREATED
    )
    ==
    0
)


# Retained as observational diagnostics only.
BLOCK_2_WEIGHT_OBJECTS_ABSENT = (
    len(
        BLOCK_2_WEIGHT_OBJECTS_PRESENT_AFTER
    )
    ==
    0
)


BLOCK_2_RECURRENT_OBJECTS_ABSENT = (
    len(
        BLOCK_2_RECURRENT_OBJECTS_PRESENT_AFTER
    )
    ==
    0
)


if not BLOCK_2_WEIGHT_OBJECTS_NOT_CREATED_BY_BLOCK_2:

    raise RuntimeError(
        "Notebook 09 Block 2 unexpectedly created downstream "
        "Transformative-Weight objects. "
        f"Objects: {list(BLOCK_2_NEW_WEIGHT_OBJECTS_CREATED)}"
    )


if not BLOCK_2_RECURRENT_OBJECTS_NOT_CREATED_BY_BLOCK_2:

    raise RuntimeError(
        "Notebook 09 Block 2 unexpectedly created downstream "
        "recurrent-state objects. "
        f"Objects: {list(BLOCK_2_NEW_RECURRENT_OBJECTS_CREATED)}"
    )


# =============================================================================
# Final validation
# =============================================================================

NOTEBOOK_09_BLOCK_2_ERRORS = []


BLOCK_2_VALIDATION_CHECKS = {
    "block_1_not_ready":
        BLOCK_2_BLOCK_1_READY,

    "pathway_dimensions_invalid":
        BLOCK_2_PATHWAY_DIMENSIONS_VALID,

    "active_dimensions_invalid":
        BLOCK_2_ACTIVE_DIMENSIONS_VALID,

    "deferred_dimensions_invalid":
        BLOCK_2_DEFERRED_DIMENSIONS_VALID,

    "dimension_partition_invalid":
        BLOCK_2_ALL_DIMENSION_PARTITIONS_VALID,

    "structural_dimensions_invalid":
        BLOCK_2_STRUCTURAL_DIMENSIONS_VALID,

    "active_counts_invalid":
        BLOCK_2_ACTIVE_COUNTS_VALID,

    "deferred_counts_invalid":
        BLOCK_2_DEFERRED_COUNTS_VALID,

    "trainable_parameter_count_invalid":
        BLOCK_2_TRAINABLE_PARAMETER_COUNT_VALID,

    "deferred_dimension_count_invalid":
        BLOCK_2_DEFERRED_DIMENSION_COUNT_VALID,

    "dimension_accounting_invalid":
        BLOCK_2_TOTAL_DIMENSION_ACCOUNTING_VALID,

    "initialisation_invalid":
        BLOCK_2_INITIALISATION_VALID,

    "weight_policy_invalid":
        BLOCK_2_WEIGHT_POLICY_VALID,

    "recurrent_contract_invalid":
        BLOCK_2_RECURRENT_CONTRACT_VALID,

    "separation_policy_invalid":
        BLOCK_2_SEPARATION_POLICY_VALID,

    "representation_not_frozen":
        BLOCK_2_REPRESENTATION_STILL_FROZEN,

    "transformative_mechanisms_not_frozen":
        BLOCK_2_TRANSFORMATIVE_STILL_FROZEN,

    "inherited_modules_not_eval":
        BLOCK_2_INHERITED_MODULES_STILL_EVAL,

    "inherited_parameter_counts_invalid":
        BLOCK_2_INHERITED_PARAMETER_COUNTS_VALID,

    "weight_objects_created_by_block_2":
        BLOCK_2_WEIGHT_OBJECTS_NOT_CREATED_BY_BLOCK_2,

    "recurrent_objects_created_by_block_2":
        BLOCK_2_RECURRENT_OBJECTS_NOT_CREATED_BY_BLOCK_2,
}


for (
    error_name,
    condition,
) in BLOCK_2_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_2_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation audit
# =============================================================================

BLOCK_2_PROHIBITED_OPERATIONS = {
    "transformative_weight_object_created":
        NOTEBOOK_09_BLOCK_2_TRANSFORMATIVE_WEIGHT_OBJECT_CREATED,

    "recurrent_state_created":
        NOTEBOOK_09_BLOCK_2_RECURRENT_STATE_CREATED,

    "recurrent_update_executed":
        NOTEBOOK_09_BLOCK_2_RECURRENT_UPDATE_EXECUTED,

    "representation_forward_executed":
        NOTEBOOK_09_BLOCK_2_REPRESENTATION_FORWARD_EXECUTED,

    "transform_forward_executed":
        NOTEBOOK_09_BLOCK_2_TRANSFORM_FORWARD_EXECUTED,

    "weight_forward_executed":
        NOTEBOOK_09_BLOCK_2_WEIGHT_FORWARD_EXECUTED,

    "loss_calculated":
        NOTEBOOK_09_BLOCK_2_LOSS_CALCULATED,

    "optimizer_created":
        NOTEBOOK_09_BLOCK_2_OPTIMIZER_CREATED,

    "backward_pass_executed":
        NOTEBOOK_09_BLOCK_2_BACKWARD_PASS_EXECUTED,

    "parameter_update_executed":
        NOTEBOOK_09_BLOCK_2_PARAMETER_UPDATE_EXECUTED,

    "training_executed":
        NOTEBOOK_09_BLOCK_2_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_2_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_2_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 2 state
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID = (
    len(
        NOTEBOOK_09_BLOCK_2_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ARCHITECTURE_READY = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID
)


NOTEBOOK_09_BLOCK_2_VALID = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID
)


if not NOTEBOOK_09_BLOCK_2_VALID:

    raise RuntimeError(
        "Notebook 09 Block 2 Transformative-Weight architectural "
        "contract validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_2_ERRORS}"
    )


NOTEBOOK_09_BLOCK_2_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_2_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_2,

    "block_name":
        NOTEBOOK_09_BLOCK_2_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_2_VERSION,

    "pathway_dimensions":
        deepcopy(
            BLOCK_2_PATHWAY_DIMENSIONS
        ),

    "structural_weight_dimensions":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS,

    "active_dimension_counts":
        deepcopy(
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_COUNTS
        ),

    "deferred_dimension_counts":
        deepcopy(
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_COUNTS
        ),

    "trainable_weight_parameter_count":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT,

    "parameterisation":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETERISATION,

    "active_initial_latent":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT,

    "active_initial_effective":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE,

    "deferred_effective":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE,

    "inherited_parameter_count":
        NOTEBOOK_09_INHERITED_PARAMETER_COUNT,

    "transformative_weight_objects_created":
        NOTEBOOK_09_BLOCK_2_TRANSFORMATIVE_WEIGHT_OBJECT_CREATED,

    "recurrent_state_created":
        NOTEBOOK_09_BLOCK_2_RECURRENT_STATE_CREATED,

    "recurrent_update_executed":
        NOTEBOOK_09_BLOCK_2_RECURRENT_UPDATE_EXECUTED,

    "rerun_detected":
        BLOCK_2_RERUN_DETECTED,

    "preexisting_weight_objects":
        tuple(
            BLOCK_2_PREEXISTING_WEIGHT_OBJECTS
        ),

    "preexisting_recurrent_objects":
        tuple(
            BLOCK_2_PREEXISTING_RECURRENT_OBJECTS
        ),

    "new_weight_objects_created":
        tuple(
            BLOCK_2_NEW_WEIGHT_OBJECTS_CREATED
        ),

    "new_recurrent_objects_created":
        tuple(
            BLOCK_2_NEW_RECURRENT_OBJECTS_CREATED
        ),

    "contract_valid":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID,

    "block_valid":
        NOTEBOOK_09_BLOCK_2_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_2_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 2: "
    "Transformative-Weight Architectural Contract and Parameterisation Policy"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_2_VERSION}"
)

print("-" * 72)

print(
    "Inherited architecture"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Combined inherited params    : "
    f"{NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_2_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_2_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"All inherited modules eval   : "
    f"{BLOCK_2_INHERITED_MODULES_STILL_EVAL}"
)

print("-" * 72)

print(
    "Transformative-Weight structural contract"
)

for pathway_name in BLOCK_2_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} dimension          : "
        f"{BLOCK_2_PATHWAY_DIMENSIONS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} active dimensions  : "
        f"{list(BLOCK_2_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} deferred dimensions: "
        f"{list(BLOCK_2_DEFERRED_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} trainable weights   : "
        f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_COUNTS[pathway_name]}"
    )

print("-" * 72)

print(
    f"Structural weight dimensions : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS}"
)

print(
    f"Trainable active parameters   : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT}"
)

print(
    f"Deferred dimensions           : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT}"
)

print("-" * 72)

print(
    "Parameterisation policy"
)

print(
    f"Granularity                  : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY['granularity']}"
)

print(
    f"Latent parameter             : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_LATENT_PARAMETER}"
)

print(
    f"Effective mapping            : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_EFFECTIVE_PARAMETER}"
)

print(
    f"Effective lower bound        : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MINIMUM:.1f}"
)

print(
    f"Effective upper bound        : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MAXIMUM:.1f}"
)

print(
    f"Active initial latent        : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT:.1f}"
)

print(
    f"Active initial weight        : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE:.1f}"
)

print(
    f"Deferred effective weight    : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE:.1f}"
)

print(
    f"Deferred trainable           : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY['deferred_trainable']}"
)

print(
    f"Pathways independent         : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY['pathway_independence']}"
)

print(
    f"Candidate direction preserved: "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY['candidate_direction_preserved']}"
)

print("-" * 72)

print(
    "Architectural separation"
)

print(
    f"Distinct from candidates     : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_SEPARATION_POLICY['distinct_from_candidate_transformations']}"
)

print(
    f"Distinct from loss weights   : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_SEPARATION_POLICY['distinct_from_optimisation_loss_weights']}"
)

print(
    f"Distinct from recurrent state: "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_SEPARATION_POLICY['distinct_from_recurrent_state']}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Transform weights created    : "
    f"{NOTEBOOK_09_BLOCK_2_TRANSFORMATIVE_WEIGHT_OBJECT_CREATED}"
)

print(
    f"Recurrent state created      : "
    f"{NOTEBOOK_09_BLOCK_2_RECURRENT_STATE_CREATED}"
)

print(
    f"Recurrent update executed    : "
    f"{NOTEBOOK_09_BLOCK_2_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"Representation forward       : "
    f"{NOTEBOOK_09_BLOCK_2_REPRESENTATION_FORWARD_EXECUTED}"
)

print(
    f"Transform forward            : "
    f"{NOTEBOOK_09_BLOCK_2_TRANSFORM_FORWARD_EXECUTED}"
)

print(
    f"Weight forward               : "
    f"{NOTEBOOK_09_BLOCK_2_WEIGHT_FORWARD_EXECUTED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_BLOCK_2_LOSS_CALCULATED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_2_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_2_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_2_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Notebook rerun detected      : "
    f"{BLOCK_2_RERUN_DETECTED}"
)

print(
    f"Pre-existing weight objects  : "
    f"{len(BLOCK_2_PREEXISTING_WEIGHT_OBJECTS)}"
)

print(
    f"New weight objects created   : "
    f"{len(BLOCK_2_NEW_WEIGHT_OBJECTS_CREATED)}"
)

print(
    f"Pre-existing recurrent objs  : "
    f"{len(BLOCK_2_PREEXISTING_RECURRENT_OBJECTS)}"
)

print(
    f"New recurrent objects created: "
    f"{len(BLOCK_2_NEW_RECURRENT_OBJECTS_CREATED)}"
)

print("-" * 72)

print(
    f"Weight contract valid        : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID}"
)

print(
    f"Weight architecture ready    : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ARCHITECTURE_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_2_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_2_COMPLETE}"
)

print("=" * 72)

print(
    "The Transformative-Weight architectural contract was "
    "established successfully without creating weight parameters."
)

print(
    "Transformative Weights are defined as independent dimension-wise "
    "bounded gates for every inherited transformative pathway."
)

print(
    f"The complete structural weight space contains "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS} dimensions; "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT} active "
    "dimension(s) are currently eligible for learning and "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT} "
    "dimension(s) remain deferred."
)

print(
    "Active latent parameters will initialise at zero, corresponding "
    "to an effective sigmoid weight of 0.5."
)

print(
    "Deferred dimensions are structurally retained but fixed at an "
    "effective weight of zero under the current gating contract."
)

print(
    "Transformative Weights remain distinct from candidate "
    "transformations, optimisation-loss coefficients and recurrent state."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,}-parameter "
    "representation and transformative architecture remains frozen "
    "and unchanged."
)

print(
    "Block 2 created no new Transformative-Weight parameter object or "
    "recurrent state and executed no forward pass, loss, optimiser, "
    "backward pass or parameter update."
)

if BLOCK_2_RERUN_DETECTED:

    print(
        "Pre-existing downstream objects were detected from an earlier "
        "notebook execution and were left unchanged; they are not attributed "
        "to Block 2."
    )

print(
    "Notebook 09 may now proceed to controlled Transformative-Weight "
    "module construction and parameter-scope validation."
)

print("=" * 72)

Media AI — Notebook 09, Block 2: Transformative-Weight Architectural Contract and Parameterisation Policy
Block version                : 1.2
------------------------------------------------------------------------
Inherited architecture
Representation parameters    : 894,003
Transformative parameters    : 8,187
Combined inherited params    : 902,190
Representation frozen        : True
Transformative frozen        : True
All inherited modules eval   : True
------------------------------------------------------------------------
Transformative-Weight structural contract
factual        dimension          : 10
factual        active dimensions  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
factual        deferred dimensions: []
factual        trainable weights   : 10
psychological  dimension          : 34
psychological  active dimensions  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
psychological  deferred dimensions: []

## Block 3 — Controlled Transformative-Weight Module Construction and Parameter-Scope Validation

This block constructs the first trainable Notebook 09 component: the pathway-specific Transformative-Weight mechanism.

Block 2 established the complete architectural contract without creating any parameter object. The factual, psychological, and social Transformative Weights are defined as independent, dimension-wise bounded gates over the candidate transformations inherited from Notebook 08.

The present block now instantiates that contract in a controlled way.

Its purpose is to create the latent Transformative-Weight parameters, map them to bounded effective weights, enforce the inherited active/deferred dimension policy, and validate the exact parameter scope before any recurrent state or recurrent update is introduced.

No recurrent state is created in this block. No recurrent update is executed. No loss, optimiser, backward pass, or parameter update is performed.

### Transformative-Weight parameterisation

For each pathway \(k\in\{F,P,S\}\), the structural Transformative-Weight vector has the same dimensionality as the corresponding candidate transformation:

$$
TW_F\in\mathbb{R}^{10},
$$

$$
TW_P\in\mathbb{R}^{34},
$$

$$
TW_S\in\mathbb{R}^{7}.
$$

The effective weight vector is derived from an unconstrained latent parameter vector

$$
\alpha_k\in\mathbb{R}^{d_k}
$$

through the sigmoid transformation

$$
TW_k
=
\sigma(\alpha_k).
$$

For active dimensions,

$$
j\in A_k,
$$

the latent value is trainable and initialised as

$$
\alpha_{k,j}^{(0)}=0,
$$

which yields

$$
TW_{k,j}^{(0)}=0.5.
$$

For deferred dimensions,

$$
j\in D_k,
$$

the effective Transformative Weight is fixed to zero under the current pilot contract:

$$
TW_{k,j}=0.
$$

Accordingly, deferred dimensions remain part of the structural weight vector but do not contribute a learnable parameter.

### Active and deferred parameter masks

The block constructs an explicit binary activation mask

$$
m_k\in\{0,1\}^{d_k}
$$

for each pathway.

For dimension \(j\),

$$
m_{k,j}
=
\begin{cases}
1, & j\in A_k,\\
0, & j\in D_k.
\end{cases}
$$

The effective Transformative Weight is therefore defined as

$$
TW_k
=
m_k\odot\sigma(\alpha_k).
$$

This guarantees that active dimensions receive bounded learnable weights, while deferred dimensions remain exactly zero.

The masking operation is architectural rather than data-dependent. It is inherited directly from the validated Notebook 08 active/deferred transformation contract.

### Parameter-count contract

The complete structural Transformative-Weight space contains

$$
10+34+7=51
$$

dimensions.

However, only active dimensions contribute trainable latent parameters.

The factual pathway contains

$$
|A_F|=5
$$

active dimensions.

The psychological pathway contains

$$
|A_P|=17.
$$

The social pathway contains

$$
|A_S|=3.
$$

Therefore, the expected number of trainable Notebook 09 parameters introduced by this block is

$$
5+17+3=25.
$$

This is the first increase in trainable parameter count after restoring the frozen Notebook 08 architecture.

The complete model parameter accounting after Block 3 is conceptually

$$
902{,}190
+
25
=
902{,}215
$$

parameters,

where the inherited 902,190 parameters remain frozen and the new 25 latent Transformative-Weight parameters are trainable.

The remaining 26 structural Transformative-Weight dimensions are represented through fixed deferred masking and do not create trainable parameters.

### Independent pathway modules

The factual, psychological, and social Transformative-Weight mechanisms are instantiated independently.

The factual mechanism controls

$$
TW_F\in[0,1]^{10},
$$

the psychological mechanism controls

$$
TW_P\in[0,1]^{34},
$$

and the social mechanism controls

$$
TW_S\in[0,1]^{7}.
$$

No parameter is shared across pathways.

The factual Transformative Weight therefore cannot directly alter the psychological or social candidate transformation, and the same separation applies symmetrically to the other pathways.

This preserves the pathway independence established throughout the representation and transformative stages.

### Initial effective weights

Immediately after construction, all active latent parameters satisfy

$$
\alpha_{k,j}=0,
\qquad
j\in A_k.
$$

Therefore,

$$
\sigma(\alpha_{k,j})=0.5.
$$

The initial effective weight vectors consequently contain two classes of value:

$$
TW_{k,j}
=
\begin{cases}
0.5, & j\in A_k,\\
0, & j\in D_k.
\end{cases}
$$

This provides a deterministic and directly auditable initialisation.

The block verifies that the resulting effective weight vectors contain exactly the expected active and deferred values.

### Boundedness validation

Every effective Transformative Weight must satisfy

$$
0\leq TW_{k,j}\leq1.
$$

The active sigmoid-gated values must lie strictly within the open interval

$$
0<TW_{k,j}<1,
$$

while deferred dimensions are exactly zero by masking.

The block therefore validates that:

- all active effective weights are finite;
- all active effective weights are initially equal to \(0.5\);
- all deferred effective weights are exactly zero;
- no effective weight lies outside the admissible interval;
- and the structural output dimensions remain 10, 34, and 7.

### Gradient-scope validation

The most important optimisation boundary in this block is the distinction between the newly introduced Transformative-Weight parameters and all inherited parameters.

For the inherited representation parameters,

$$
\nabla_{\theta_R}=0,
$$

and for the inherited transformative-mechanism parameters,

$$
\nabla_{\theta_T}=0.
$$

Only the new active latent Transformative-Weight parameters are allowed to satisfy

$$
\texttt{requires\_grad}=\mathrm{True}.
$$

The expected trainable parameter set is therefore

$$
\theta_{TW}
=
\{
\alpha_{F,j}:j\in A_F
\}
\cup
\{
\alpha_{P,j}:j\in A_P
\}
\cup
\{
\alpha_{S,j}:j\in A_S
\}.
$$

Its cardinality must be

$$
|\theta_{TW}|=25.
$$

The block validates that no inherited parameter accidentally becomes trainable and that no deferred dimension contributes a trainable parameter.

### Structural Transformative-Weight output

The module construction must expose the effective weight vectors in the full pathway dimensions:

$$
TW_F\in\mathbb{R}^{10},
$$

$$
TW_P\in\mathbb{R}^{34},
$$

$$
TW_S\in\mathbb{R}^{7}.
$$

This is important because the downstream recurrent mechanism operates over complete pathway state spaces rather than compressed active-only representations.

Deferred dimensions remain present in the vectors as explicit zeros.

The recurrent mechanism can therefore apply the full elementwise relation

$$
\Delta h_t^{(k)}
=
TW_k\odot c_t^{(k)}
$$

without needing to reconstruct omitted dimensions.

### Separation from recurrent state

Although the Transformative-Weight modules are now instantiated, no recurrent state is created.

The block must therefore preserve

$$
h_t^{(F)},\qquad
h_t^{(P)},\qquad
h_t^{(S)}
$$

as conceptual future state variables only.

No state tensor is stored, no article sequence is traversed, and no sentence-to-sentence temporal carry is performed.

Likewise, the block does not yet execute

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}.
$$

That operation belongs to the recurrent-mechanism stage.

### No training in Block 3

Block 3 constructs trainable Transformative-Weight parameters, but it does not train them.

Therefore:

- no optimiser is created;
- no training objective is calculated;
- no gradient is backpropagated;
- no parameter update is executed;
- and no recurrent state is used as supervision.

The purpose is architectural validation only.

This mirrors the discipline established in Notebook 08, where module construction and forward-path validation preceded optimisation.

### Inherited architecture remains unchanged

The complete inherited Notebook 08 state must remain exactly unchanged during construction of the new Transformative-Weight modules.

The representation model remains frozen:

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}},
$$

and the transformative mechanisms remain frozen:

$$
\theta_T^{\mathrm{after}}
=
\theta_T^{\mathrm{before}}.
$$

Only the newly created Transformative-Weight latent parameters expand the model state.

### Block 3 completion contract

Block 3 is complete only when:

- three independent Transformative-Weight modules have been constructed;
- factual, psychological, and social structural dimensions remain 10, 34, and 7;
- the structural Transformative-Weight space contains 51 dimensions;
- active/deferred masks reproduce the inherited Notebook 08 contract exactly;
- only active dimensions contain trainable latent parameters;
- the total trainable Transformative-Weight parameter count is exactly 25;
- all active latent parameters initialise at zero;
- all active effective weights initialise at 0.5;
- all deferred effective weights are exactly zero;
- all effective weights satisfy the bounded weighting contract;
- factual, psychological, and social weight parameter sets are independent;
- the inherited 902,190 parameters remain frozen and unchanged;
- no recurrent state is created;
- no recurrent update is executed;
- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- and no parameter update occurs.

Successful completion establishes the first new trainable Notebook 09 architecture:

$$
\boxed{
\theta_{TW}
=
\{
\alpha_F,\alpha_P,\alpha_S
\}
}
$$

with

$$
|\theta_{TW}|=25
$$

currently trainable parameters and full effective Transformative-Weight vectors

$$
TW_F\in[0,1]^{10},
\qquad
TW_P\in[0,1]^{34},
\qquad
TW_S\in[0,1]^{7}.
$$

Notebook 09 may then proceed to controlled Transformative-Weight forward-path validation before introducing recurrent state evolution.

In [53]:
# =============================================================================
# Media AI — Notebook 09
# Block 3: Controlled Transformative-Weight Module Construction
#          and Parameter-Scope Validation
# =============================================================================

from copy import deepcopy

import torch
import torch.nn as nn


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_3 = 3

NOTEBOOK_09_BLOCK_3_NAME = (
    "Controlled Transformative-Weight Module Construction "
    "and Parameter-Scope Validation"
)

NOTEBOOK_09_BLOCK_3_VERSION = "1.3"


# =============================================================================
# Required inherited runtime contract
# =============================================================================

BLOCK_3_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_BLOCK_1_COMPLETE",
    "NOTEBOOK_09_BLOCK_1_VALID",
    "NOTEBOOK_09_HANDOVER_RESTORED",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Block 2
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_2_COMPLETE",
    "NOTEBOOK_09_BLOCK_2_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ARCHITECTURE_READY",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DIMENSIONS",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_COUNTS",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_COUNTS",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_POLICY",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE",
    "BLOCK_2_ACTIVE_DIMENSION_INDICES",
    "BLOCK_2_DEFERRED_DIMENSION_INDICES",
    "BLOCK_2_PATHWAY_DIMENSIONS",

    # -------------------------------------------------------------------------
    # Execution environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_3_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_3_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_3_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 3 prerequisites are not initialised. "
        f"Missing: {BLOCK_3_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate inherited readiness
# =============================================================================

BLOCK_3_INHERITED_READY = all(
    [
        NOTEBOOK_09_INITIALISED is True,
        NOTEBOOK_09_BLOCK_1_COMPLETE is True,
        NOTEBOOK_09_BLOCK_1_VALID is True,
        NOTEBOOK_09_HANDOVER_RESTORED is True,

        NOTEBOOK_09_BLOCK_2_COMPLETE is True,
        NOTEBOOK_09_BLOCK_2_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ARCHITECTURE_READY is True,
    ]
)


if not BLOCK_3_INHERITED_READY:

    raise RuntimeError(
        "Notebook 09 Blocks 1 and 2 must be valid and complete "
        "before Transformative-Weight module construction."
    )


# =============================================================================
# Downstream-runtime snapshot for rerun-safe execution
# =============================================================================
#
# Block 3 OWNS Transformative-Weight module construction. Therefore, on a
# full notebook rerun it deliberately reconstructs those modules from the
# validated Block 2 contract, resetting them to their canonical initial state.
#
# Recurrent-state objects belong to later blocks. They may still exist in the
# interactive kernel from a previous execution. Their pre-existence must not
# make Block 3 fail; Block 3 only has to prove that it did not create or mutate
# those later-stage objects.
# =============================================================================

BLOCK_3_PREEXISTING_WEIGHT_MODULE_COLLECTION = (
    globals().get(
        "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES"
    )
)


BLOCK_3_PREEXISTING_WEIGHT_MODULE_IDS = (
    {
        pathway_name:
            id(
                module
            )

        for (
            pathway_name,
            module,
        ) in BLOCK_3_PREEXISTING_WEIGHT_MODULE_COLLECTION.items()
    }

    if isinstance(
        BLOCK_3_PREEXISTING_WEIGHT_MODULE_COLLECTION,
        dict,
    )

    else {}
)


BLOCK_3_DOWNSTREAM_RECURRENT_OBJECT_NAMES = (
    "NOTEBOOK_09_FACTUAL_RECURRENT_STATE",
    "NOTEBOOK_09_PSYCHOLOGICAL_RECURRENT_STATE",
    "NOTEBOOK_09_SOCIAL_RECURRENT_STATE",

    "NOTEBOOK_09_FACTUAL_INITIAL_STATE",
    "NOTEBOOK_09_PSYCHOLOGICAL_INITIAL_STATE",
    "NOTEBOOK_09_SOCIAL_INITIAL_STATE",

    "NOTEBOOK_09_INITIAL_RECURRENT_STATES",
    "NOTEBOOK_09_VALIDATED_SINGLE_STEP_RECURRENT_STATES",
    "NOTEBOOK_09_ARTICLE_FINAL_RECURRENT_STATES",
    "NOTEBOOK_09_POSTTRAIN_FINAL_RECURRENT_STATES",
)


BLOCK_3_PREEXISTING_RECURRENT_OBJECTS = {
    object_name:
        globals()[
            object_name
        ]

    for object_name
    in BLOCK_3_DOWNSTREAM_RECURRENT_OBJECT_NAMES

    if object_name in globals()
}


BLOCK_3_PREEXISTING_RECURRENT_OBJECT_IDS = {
    object_name:
        id(
            object_value
        )

    for (
        object_name,
        object_value,
    ) in BLOCK_3_PREEXISTING_RECURRENT_OBJECTS.items()
}


BLOCK_3_RERUN_DETECTED = bool(
    BLOCK_3_PREEXISTING_WEIGHT_MODULE_IDS
    or
    BLOCK_3_PREEXISTING_RECURRENT_OBJECTS
)


# =============================================================================
# Canonical pathway contract
# =============================================================================
#
# Block 3 consumes the validated Block 2 pathway, active-dimension and
# deferred-dimension contracts exactly. No pilot-era dimensionality or
# activation-count expectations are recreated here.
# =============================================================================

BLOCK_3_PATHWAY_DIMENSIONS = deepcopy(
    BLOCK_2_PATHWAY_DIMENSIONS
)


BLOCK_3_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in BLOCK_2_ACTIVE_DIMENSION_INDICES.items()
}


BLOCK_3_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in BLOCK_2_DEFERRED_DIMENSION_INDICES.items()
}


# Canonical deferred-dimension total derived from the validated Block 2
# pathway partition. This avoids relying on an additional pilot-era scalar.
BLOCK_3_DEFERRED_DIMENSION_COUNT = int(
    sum(
        len(
            BLOCK_3_DEFERRED_DIMENSION_INDICES[
                pathway_name
            ]
        )

        for pathway_name
        in BLOCK_3_PATHWAY_DIMENSIONS
    )
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT = (
    BLOCK_3_DEFERRED_DIMENSION_COUNT
)


BLOCK_3_PATHWAY_CONTRACT_VALID = all(
    [
        (
            BLOCK_3_PATHWAY_DIMENSIONS
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DIMENSIONS
        ),

        (
            set(
                BLOCK_3_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                BLOCK_3_ACTIVE_DIMENSION_INDICES.keys()
            )
            ==
            set(
                BLOCK_3_DEFERRED_DIMENSION_INDICES.keys()
            )
        ),

        all(
            (
                set(
                    BLOCK_3_ACTIVE_DIMENSION_INDICES[
                        pathway_name
                    ]
                )
                |
                set(
                    BLOCK_3_DEFERRED_DIMENSION_INDICES[
                        pathway_name
                    ]
                )
            )
            ==
            set(
                range(
                    pathway_dim
                )
            )

            and

            set(
                BLOCK_3_ACTIVE_DIMENSION_INDICES[
                    pathway_name
                ]
            ).isdisjoint(
                set(
                    BLOCK_3_DEFERRED_DIMENSION_INDICES[
                        pathway_name
                    ]
                )
            )

            for (
                pathway_name,
                pathway_dim,
            ) in BLOCK_3_PATHWAY_DIMENSIONS.items()
        ),

        (
            sum(
                BLOCK_3_PATHWAY_DIMENSIONS.values()
            )
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS
        ),

        (
            sum(
                len(
                    BLOCK_3_ACTIVE_DIMENSION_INDICES[
                        pathway_name
                    ]
                )

                for pathway_name
                in BLOCK_3_PATHWAY_DIMENSIONS
            )
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            {
                pathway_name:
                    len(
                        BLOCK_3_ACTIVE_DIMENSION_INDICES[
                            pathway_name
                        ]
                    )

                for pathway_name
                in BLOCK_3_PATHWAY_DIMENSIONS
            }
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_COUNTS
        ),

        (
            {
                pathway_name:
                    len(
                        BLOCK_3_DEFERRED_DIMENSION_INDICES[
                            pathway_name
                        ]
                    )

                for pathway_name
                in BLOCK_3_PATHWAY_DIMENSIONS
            }
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_COUNTS
        ),

        (
            BLOCK_3_DEFERRED_DIMENSION_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT
        ),
    ]
)


if not BLOCK_3_PATHWAY_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 09 Block 3 pathway contract does not reproduce "
        "the validated Block 2 Transformative-Weight policy."
    )


# =============================================================================
# Snapshot inherited parameter state before construction
# =============================================================================

def block_3_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


BLOCK_3_REPRESENTATION_STATE_BEFORE = (
    block_3_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_3_TRANSFORMATIVE_STATE_BEFORE = (
    block_3_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


# =============================================================================
# Transformative-Weight module
# =============================================================================
#
# The module stores only active latent parameters as nn.Parameter objects.
#
# Deferred dimensions are represented by fixed registered buffers.
#
# The complete effective Transformative-Weight vector is reconstructed in
# forward() as:
#
#     TW_k = mask_k ⊙ sigmoid(alpha_k)
#
# where only active alpha values exist as trainable parameters.
#
# IMPORTANT:
# forward() is defined here but is NOT executed in Block 3.
# =============================================================================

class Notebook09TransformativeWeight(
    nn.Module
):

    def __init__(
        self,
        pathway_dim,
        active_indices,
        deferred_indices,
        initial_latent=0.0,
    ):

        super().__init__()


        self.pathway_dim = int(
            pathway_dim
        )


        active_indices = tuple(
            int(
                index
            )

            for index
            in active_indices
        )


        deferred_indices = tuple(
            int(
                index
            )

            for index
            in deferred_indices
        )


        # ---------------------------------------------------------------------
        # Structural validation
        # ---------------------------------------------------------------------

        active_set = set(
            active_indices
        )


        deferred_set = set(
            deferred_indices
        )


        expected_set = set(
            range(
                self.pathway_dim
            )
        )


        if not active_set.isdisjoint(
            deferred_set
        ):

            raise ValueError(
                "Active and deferred Transformative-Weight dimensions "
                "must not overlap."
            )


        if (
            active_set
            |
            deferred_set
        ) != expected_set:

            raise ValueError(
                "Active and deferred Transformative-Weight dimensions "
                "must partition the complete pathway dimension."
            )


        # ---------------------------------------------------------------------
        # Fixed structural buffers
        # ---------------------------------------------------------------------

        self.register_buffer(
            "active_indices",
            torch.tensor(
                active_indices,
                dtype=
                    torch.long,
            ),
            persistent=
                True,
        )


        self.register_buffer(
            "deferred_indices",
            torch.tensor(
                deferred_indices,
                dtype=
                    torch.long,
            ),
            persistent=
                True,
        )


        active_mask = torch.zeros(
            self.pathway_dim,
            dtype=
                DEFAULT_DTYPE,
        )


        if active_indices:

            active_mask[
                list(
                    active_indices
                )
            ] = 1.0


        self.register_buffer(
            "active_mask",
            active_mask,
            persistent=
                True,
        )


        # ---------------------------------------------------------------------
        # Trainable active latent parameters only
        # ---------------------------------------------------------------------

        self.alpha_active = nn.Parameter(
            torch.full(
                (
                    len(
                        active_indices
                    ),
                ),
                fill_value=
                    float(
                        initial_latent
                    ),
                dtype=
                    DEFAULT_DTYPE,
            ),
            requires_grad=
                True,
        )


    def forward(
        self,
    ):
        """
        Return the complete pathway-specific effective Transformative-Weight
        vector.

        This method is intentionally not executed in Block 3.
        """

        effective_weights = torch.zeros(
            self.pathway_dim,
            dtype=
                self.alpha_active.dtype,
            device=
                self.alpha_active.device,
        )


        if self.alpha_active.numel() > 0:

            effective_weights[
                self.active_indices
            ] = torch.sigmoid(
                self.alpha_active
            )


        return effective_weights


# =============================================================================
# Construct independent pathway modules
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES = {
    pathway_name:
        Notebook09TransformativeWeight(
            pathway_dim=
                BLOCK_3_PATHWAY_DIMENSIONS[
                    pathway_name
                ],

            active_indices=
                BLOCK_3_ACTIVE_DIMENSION_INDICES[
                    pathway_name
                ],

            deferred_indices=
                BLOCK_3_DEFERRED_DIMENSION_INDICES[
                    pathway_name
                ],

            initial_latent=
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT,
        )

    for pathway_name
    in BLOCK_3_PATHWAY_DIMENSIONS
}


# Convenience aliases retained for the current canonical pathways.
NOTEBOOK_09_FACTUAL_TRANSFORMATIVE_WEIGHT = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.get(
        "factual"
    )
)


NOTEBOOK_09_PSYCHOLOGICAL_TRANSFORMATIVE_WEIGHT = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.get(
        "psychological"
    )
)


NOTEBOOK_09_SOCIAL_TRANSFORMATIVE_WEIGHT = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.get(
        "social"
    )
)


# =============================================================================
# Move new modules to execution device
# =============================================================================

for module in (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()
):

    module.to(
        DEVICE
    )


# =============================================================================
# Transformative-Weight reconstruction audit
# =============================================================================

BLOCK_3_CURRENT_WEIGHT_MODULE_IDS = {
    pathway_name:
        id(
            module
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


BLOCK_3_WEIGHT_MODULES_RECONSTRUCTED = all(
    [
        (
            set(
                BLOCK_3_CURRENT_WEIGHT_MODULE_IDS.keys()
            )
            ==
            set(
                BLOCK_3_PATHWAY_DIMENSIONS.keys()
            )
        ),

        (
            not BLOCK_3_PREEXISTING_WEIGHT_MODULE_IDS
            or
            all(
                BLOCK_3_CURRENT_WEIGHT_MODULE_IDS[
                    pathway_name
                ]
                !=
                BLOCK_3_PREEXISTING_WEIGHT_MODULE_IDS.get(
                    pathway_name
                )

                for pathway_name
                in BLOCK_3_CURRENT_WEIGHT_MODULE_IDS
            )
        ),
    ]
)


if not BLOCK_3_WEIGHT_MODULES_RECONSTRUCTED:

    raise RuntimeError(
        "Notebook 09 Block 3 failed to reconstruct a fresh canonical "
        "Transformative-Weight module set."
    )


# =============================================================================
# Module structural validation
# =============================================================================

BLOCK_3_MODULE_DIMENSIONS = {
    pathway_name:
        int(
            module.pathway_dim
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


BLOCK_3_MODULE_DIMENSIONS_VALID = (
    BLOCK_3_MODULE_DIMENSIONS
    ==
    BLOCK_3_PATHWAY_DIMENSIONS
)


if not BLOCK_3_MODULE_DIMENSIONS_VALID:

    raise RuntimeError(
        "Constructed Transformative-Weight module dimensions are invalid."
    )


# =============================================================================
# Active/deferred buffer validation
# =============================================================================

BLOCK_3_ACTIVE_INDEX_BUFFERS_VALID = {}


BLOCK_3_DEFERRED_INDEX_BUFFERS_VALID = {}


BLOCK_3_ACTIVE_MASKS_VALID = {}


for (
    pathway_name,
    module,
) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items():

    expected_active = tuple(
        BLOCK_3_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    expected_deferred = tuple(
        BLOCK_3_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )


    observed_active = tuple(
        int(
            value
        )

        for value
        in module.active_indices
        .detach()
        .cpu()
        .tolist()
    )


    observed_deferred = tuple(
        int(
            value
        )

        for value
        in module.deferred_indices
        .detach()
        .cpu()
        .tolist()
    )


    expected_mask = torch.zeros(
        BLOCK_3_PATHWAY_DIMENSIONS[
            pathway_name
        ],
        dtype=
            DEFAULT_DTYPE,
    )


    expected_mask[
        list(
            expected_active
        )
    ] = 1.0


    observed_mask = (
        module.active_mask
        .detach()
        .cpu()
    )


    BLOCK_3_ACTIVE_INDEX_BUFFERS_VALID[
        pathway_name
    ] = (
        observed_active
        ==
        expected_active
    )


    BLOCK_3_DEFERRED_INDEX_BUFFERS_VALID[
        pathway_name
    ] = (
        observed_deferred
        ==
        expected_deferred
    )


    BLOCK_3_ACTIVE_MASKS_VALID[
        pathway_name
    ] = torch.equal(
        observed_mask,
        expected_mask,
    )


BLOCK_3_ALL_ACTIVE_INDEX_BUFFERS_VALID = all(
    BLOCK_3_ACTIVE_INDEX_BUFFERS_VALID.values()
)


BLOCK_3_ALL_DEFERRED_INDEX_BUFFERS_VALID = all(
    BLOCK_3_DEFERRED_INDEX_BUFFERS_VALID.values()
)


BLOCK_3_ALL_ACTIVE_MASKS_VALID = all(
    BLOCK_3_ACTIVE_MASKS_VALID.values()
)


if not all(
    [
        BLOCK_3_ALL_ACTIVE_INDEX_BUFFERS_VALID,
        BLOCK_3_ALL_DEFERRED_INDEX_BUFFERS_VALID,
        BLOCK_3_ALL_ACTIVE_MASKS_VALID,
    ]
):

    raise RuntimeError(
        "Transformative-Weight active/deferred structural buffers "
        "do not reproduce the validated Block 2 contract."
    )


# =============================================================================
# Trainable parameter-count validation
# =============================================================================

BLOCK_3_TRAINABLE_PARAMETER_COUNTS = {
    pathway_name:
        int(
            sum(
                parameter.numel()

                for parameter
                in module.parameters()

                if parameter.requires_grad
            )
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS = int(
    sum(
        BLOCK_3_TRAINABLE_PARAMETER_COUNTS.values()
    )
)


BLOCK_3_EXPECTED_TRAINABLE_PARAMETER_COUNTS = {
    pathway_name:
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_COUNTS[
            pathway_name
        ]

    for pathway_name
    in BLOCK_3_PATHWAY_DIMENSIONS
}


BLOCK_3_TRAINABLE_PARAMETER_COUNTS_VALID = all(
    [
        (
            BLOCK_3_TRAINABLE_PARAMETER_COUNTS
            ==
            BLOCK_3_EXPECTED_TRAINABLE_PARAMETER_COUNTS
        ),

        (
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_3_TRAINABLE_PARAMETER_COUNTS_VALID:

    raise RuntimeError(
        "Constructed Transformative-Weight trainable parameter "
        "count is invalid."
    )


# =============================================================================
# Total new parameter-count validation
# =============================================================================
#
# Each active latent scalar is represented by exactly one nn.Parameter value.
# Deferred dimensions create buffers only and therefore add no trainable or
# non-trainable parameters.
# =============================================================================

BLOCK_3_TOTAL_NEW_PARAMETER_COUNTS = {
    pathway_name:
        int(
            sum(
                parameter.numel()

                for parameter
                in module.parameters()
            )
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


BLOCK_3_TOTAL_NEW_PARAMETERS = int(
    sum(
        BLOCK_3_TOTAL_NEW_PARAMETER_COUNTS.values()
    )
)


BLOCK_3_ONLY_ACTIVE_PARAMETERS_EXIST = all(
    [
        (
            BLOCK_3_TOTAL_NEW_PARAMETER_COUNTS
            ==
            BLOCK_3_EXPECTED_TRAINABLE_PARAMETER_COUNTS
        ),

        (
            BLOCK_3_TOTAL_NEW_PARAMETERS
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_3_ONLY_ACTIVE_PARAMETERS_EXIST:

    raise RuntimeError(
        "Deferred Transformative-Weight dimensions unexpectedly "
        "created parameter objects."
    )


# =============================================================================
# Complete model parameter accounting
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT = (
    BLOCK_3_TOTAL_NEW_PARAMETERS
)


NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION = (
    NOTEBOOK_09_INHERITED_PARAMETER_COUNT
    +
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
)


BLOCK_3_COMPLETE_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_3_COMPLETE_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 complete parameter accounting is invalid."
    )


# =============================================================================
# Latent-parameter initialisation validation
# =============================================================================

BLOCK_3_LATENT_INITIALISATION_VALID = {}


for (
    pathway_name,
    module,
) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items():

    alpha = (
        module.alpha_active
        .detach()
        .cpu()
    )


    expected_alpha = torch.full(
        alpha.shape,
        fill_value=
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT
            ),
        dtype=
            alpha.dtype,
    )


    BLOCK_3_LATENT_INITIALISATION_VALID[
        pathway_name
    ] = torch.equal(
        alpha,
        expected_alpha,
    )


BLOCK_3_ALL_LATENT_INITIALISATION_VALID = all(
    BLOCK_3_LATENT_INITIALISATION_VALID.values()
)


if not BLOCK_3_ALL_LATENT_INITIALISATION_VALID:

    raise RuntimeError(
        "Transformative-Weight latent parameters were not "
        "initialised exactly at zero."
    )


# =============================================================================
# Effective-weight initialisation audit
# =============================================================================
#
# Block 3 does not call module.forward().
#
# The expected effective weights are audited directly from the stored latent
# parameters and structural active/deferred masks.
# =============================================================================

BLOCK_3_INITIAL_EFFECTIVE_WEIGHTS = {}


BLOCK_3_ACTIVE_EFFECTIVE_WEIGHTS_VALID = {}


BLOCK_3_DEFERRED_EFFECTIVE_WEIGHTS_VALID = {}


BLOCK_3_EFFECTIVE_WEIGHTS_FINITE = {}


BLOCK_3_EFFECTIVE_WEIGHTS_BOUNDED = {}


for (
    pathway_name,
    module,
) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items():

    pathway_dim = (
        BLOCK_3_PATHWAY_DIMENSIONS[
            pathway_name
        ]
    )


    effective_weights = torch.full(
        (
            pathway_dim,
        ),
        fill_value=
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE
            ),
        dtype=
            DEFAULT_DTYPE,
        device=
            "cpu",
    )


    alpha_active = (
        module.alpha_active
        .detach()
        .cpu()
    )


    active_effective = torch.sigmoid(
        alpha_active
    )


    active_indices = (
        module.active_indices
        .detach()
        .cpu()
        .long()
    )


    if active_indices.numel() > 0:

        effective_weights[
            active_indices
        ] = active_effective


    BLOCK_3_INITIAL_EFFECTIVE_WEIGHTS[
        pathway_name
    ] = effective_weights.clone()


    active_values = (
        effective_weights[
            active_indices
        ]
    )


    deferred_indices = (
        module.deferred_indices
        .detach()
        .cpu()
        .long()
    )


    deferred_values = (
        effective_weights[
            deferred_indices
        ]
    )


    expected_active_values = torch.full(
        active_values.shape,
        fill_value=
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE
            ),
        dtype=
            effective_weights.dtype,
    )


    expected_deferred_values = torch.full(
        deferred_values.shape,
        fill_value=
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE
            ),
        dtype=
            effective_weights.dtype,
    )


    BLOCK_3_ACTIVE_EFFECTIVE_WEIGHTS_VALID[
        pathway_name
    ] = torch.equal(
        active_values,
        expected_active_values,
    )


    BLOCK_3_DEFERRED_EFFECTIVE_WEIGHTS_VALID[
        pathway_name
    ] = torch.equal(
        deferred_values,
        expected_deferred_values,
    )


    BLOCK_3_EFFECTIVE_WEIGHTS_FINITE[
        pathway_name
    ] = bool(
        torch.isfinite(
            effective_weights
        ).all().item()
    )


    BLOCK_3_EFFECTIVE_WEIGHTS_BOUNDED[
        pathway_name
    ] = bool(
        (
            (
                effective_weights
                >=
                0.0
            )
            &
            (
                effective_weights
                <=
                1.0
            )
        ).all().item()
    )


BLOCK_3_ALL_ACTIVE_EFFECTIVE_WEIGHTS_VALID = all(
    BLOCK_3_ACTIVE_EFFECTIVE_WEIGHTS_VALID.values()
)


BLOCK_3_ALL_DEFERRED_EFFECTIVE_WEIGHTS_VALID = all(
    BLOCK_3_DEFERRED_EFFECTIVE_WEIGHTS_VALID.values()
)


BLOCK_3_ALL_EFFECTIVE_WEIGHTS_FINITE = all(
    BLOCK_3_EFFECTIVE_WEIGHTS_FINITE.values()
)


BLOCK_3_ALL_EFFECTIVE_WEIGHTS_BOUNDED = all(
    BLOCK_3_EFFECTIVE_WEIGHTS_BOUNDED.values()
)


if not all(
    [
        BLOCK_3_ALL_ACTIVE_EFFECTIVE_WEIGHTS_VALID,
        BLOCK_3_ALL_DEFERRED_EFFECTIVE_WEIGHTS_VALID,
        BLOCK_3_ALL_EFFECTIVE_WEIGHTS_FINITE,
        BLOCK_3_ALL_EFFECTIVE_WEIGHTS_BOUNDED,
    ]
):

    raise RuntimeError(
        "Initial effective Transformative-Weight values do not "
        "satisfy the Block 2 weighting contract."
    )


# =============================================================================
# Active sigmoid interval validation
# =============================================================================

BLOCK_3_ACTIVE_SIGMOID_INTERVAL_VALID = {}


for (
    pathway_name,
    effective_weights,
) in BLOCK_3_INITIAL_EFFECTIVE_WEIGHTS.items():

    active_indices = torch.tensor(
        BLOCK_3_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    active_values = (
        effective_weights[
            active_indices
        ]
    )


    BLOCK_3_ACTIVE_SIGMOID_INTERVAL_VALID[
        pathway_name
    ] = bool(
        (
            (
                active_values
                >
                0.0
            )
            &
            (
                active_values
                <
                1.0
            )
        ).all().item()
    )


BLOCK_3_ALL_ACTIVE_SIGMOID_INTERVALS_VALID = all(
    BLOCK_3_ACTIVE_SIGMOID_INTERVAL_VALID.values()
)


if not BLOCK_3_ALL_ACTIVE_SIGMOID_INTERVALS_VALID:

    raise RuntimeError(
        "Active Transformative Weights do not satisfy "
        "the open sigmoid interval (0, 1)."
    )


# =============================================================================
# Gradient-scope validation
# =============================================================================

BLOCK_3_WEIGHT_PARAMETERS_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_3_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_3_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_3_GRADIENT_SCOPE_VALID = all(
    [
        BLOCK_3_WEIGHT_PARAMETERS_TRAINABLE,
        BLOCK_3_REPRESENTATION_STILL_FROZEN,
        BLOCK_3_TRANSFORMATIVE_STILL_FROZEN,
    ]
)


if not BLOCK_3_GRADIENT_SCOPE_VALID:

    raise RuntimeError(
        "Notebook 09 Block 3 trainable-parameter scope is invalid."
    )


# =============================================================================
# Parameter-set independence
# =============================================================================

BLOCK_3_PARAMETER_OBJECT_IDS = {
    pathway_name:
        {
            id(
                parameter
            )

            for parameter
            in module.parameters()
        }

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


BLOCK_3_PATHWAY_NAMES = tuple(
    BLOCK_3_PARAMETER_OBJECT_IDS.keys()
)


BLOCK_3_PARAMETER_SETS_INDEPENDENT = all(
    BLOCK_3_PARAMETER_OBJECT_IDS[
        BLOCK_3_PATHWAY_NAMES[
            left_index
        ]
    ].isdisjoint(
        BLOCK_3_PARAMETER_OBJECT_IDS[
            BLOCK_3_PATHWAY_NAMES[
                right_index
            ]
        ]
    )

    for left_index
    in range(
        len(
            BLOCK_3_PATHWAY_NAMES
        )
    )

    for right_index
    in range(
        left_index + 1,
        len(
            BLOCK_3_PATHWAY_NAMES
        ),
    )
)


if not BLOCK_3_PARAMETER_SETS_INDEPENDENT:

    raise RuntimeError(
        "Transformative-Weight pathways unexpectedly share "
        "parameter objects."
    )


# =============================================================================
# Device placement validation
# =============================================================================

BLOCK_3_WEIGHT_DEVICE_PLACEMENT_VALID = all(
    parameter.device
    ==
    DEVICE

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_3_WEIGHT_BUFFER_DEVICE_PLACEMENT_VALID = all(
    buffer.device
    ==
    DEVICE

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for buffer
    in module.buffers()
)


if not all(
    [
        BLOCK_3_WEIGHT_DEVICE_PLACEMENT_VALID,
        BLOCK_3_WEIGHT_BUFFER_DEVICE_PLACEMENT_VALID,
    ]
):

    raise RuntimeError(
        "Transformative-Weight parameters or buffers were not placed "
        "on the configured execution device."
    )


# =============================================================================
# Snapshot inherited architecture after construction
# =============================================================================

BLOCK_3_REPRESENTATION_STATE_AFTER = (
    block_3_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_3_TRANSFORMATIVE_STATE_AFTER = (
    block_3_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


# =============================================================================
# Exact inherited-state comparison helper
# =============================================================================

def block_3_nested_state_exact(
    state_before,
    state_after,
):

    if state_before.keys() != state_after.keys():

        return False


    for module_name in state_before:

        before_module_state = (
            state_before[
                module_name
            ]
        )


        after_module_state = (
            state_after[
                module_name
            ]
        )


        if (
            before_module_state.keys()
            !=
            after_module_state.keys()
        ):

            return False


        for tensor_name in before_module_state:

            if not torch.equal(
                before_module_state[
                    tensor_name
                ],
                after_module_state[
                    tensor_name
                ],
            ):

                return False


    return True


BLOCK_3_REPRESENTATION_UNCHANGED = (
    block_3_nested_state_exact(
        BLOCK_3_REPRESENTATION_STATE_BEFORE,
        BLOCK_3_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_3_TRANSFORMATIVE_UNCHANGED = (
    block_3_nested_state_exact(
        BLOCK_3_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_3_TRANSFORMATIVE_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_3_REPRESENTATION_UNCHANGED,
        BLOCK_3_TRANSFORMATIVE_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Inherited Notebook 08 parameter state changed during "
        "Transformative-Weight module construction."
    )


# =============================================================================
# Inherited evaluation-mode validation
# =============================================================================

BLOCK_3_INHERITED_MODULES_STILL_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


if not BLOCK_3_INHERITED_MODULES_STILL_EVAL:

    raise RuntimeError(
        "Inherited Notebook 08 modules unexpectedly left evaluation mode."
    )


# =============================================================================
# New module mode contract
# =============================================================================
#
# Transformative-Weight modules contain no Dropout, BatchNorm, or other
# train/eval-dependent operation.
#
# They remain in training mode after construction because their latent
# parameters are the only new trainable Notebook 09 parameters.
# =============================================================================

BLOCK_3_WEIGHT_MODULES_TRAINABLE_MODE = all(
    module.training

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()
)


# =============================================================================
# Structural-output contract
# =============================================================================
#
# No weight forward pass is executed here.
#
# The structural output dimensionality is inferred from the validated module
# attribute and the full active/deferred mask.
# =============================================================================

BLOCK_3_STRUCTURAL_OUTPUT_DIMENSIONS = {
    pathway_name:
        int(
            module.active_mask.numel()
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


BLOCK_3_STRUCTURAL_OUTPUT_DIMENSIONS_VALID = (
    BLOCK_3_STRUCTURAL_OUTPUT_DIMENSIONS
    ==
    BLOCK_3_PATHWAY_DIMENSIONS
)


if not BLOCK_3_STRUCTURAL_OUTPUT_DIMENSIONS_VALID:

    raise RuntimeError(
        "Transformative-Weight structural output dimensions are invalid."
    )


# =============================================================================
# Explicit Notebook 09 execution boundary
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED = True

NOTEBOOK_09_RECURRENT_STATE_CREATED = False

NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED = False

NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED = False


NOTEBOOK_09_BLOCK_3_WEIGHT_FORWARD_EXECUTED = False

NOTEBOOK_09_BLOCK_3_LOSS_CALCULATED = False

NOTEBOOK_09_BLOCK_3_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_3_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_3_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_3_TRAINING_EXECUTED = False


# =============================================================================
# Downstream recurrent-state mutation audit
# =============================================================================
#
# Later-stage recurrent objects may already exist when the notebook is rerun.
# Block 3 must neither create new recurrent objects nor replace/mutate those
# that were present before Block 3 started.
# =============================================================================

BLOCK_3_RECURRENT_OBJECTS_PRESENT_AFTER = {
    object_name:
        globals()[
            object_name
        ]

    for object_name
    in BLOCK_3_DOWNSTREAM_RECURRENT_OBJECT_NAMES

    if object_name in globals()
}


BLOCK_3_NEW_RECURRENT_OBJECTS_CREATED = tuple(
    object_name

    for object_name
    in BLOCK_3_RECURRENT_OBJECTS_PRESENT_AFTER

    if object_name not in BLOCK_3_PREEXISTING_RECURRENT_OBJECTS
)


BLOCK_3_PREEXISTING_RECURRENT_OBJECTS_PRESERVED = all(
    (
        object_name
        in
        BLOCK_3_RECURRENT_OBJECTS_PRESENT_AFTER
    )
    and
    (
        id(
            BLOCK_3_RECURRENT_OBJECTS_PRESENT_AFTER[
                object_name
            ]
        )
        ==
        BLOCK_3_PREEXISTING_RECURRENT_OBJECT_IDS[
            object_name
        ]
    )

    for object_name
    in BLOCK_3_PREEXISTING_RECURRENT_OBJECTS
)


BLOCK_3_RECURRENT_OBJECTS_NOT_CREATED_BY_BLOCK_3 = (
    len(
        BLOCK_3_NEW_RECURRENT_OBJECTS_CREATED
    )
    ==
    0
)


BLOCK_3_RECURRENT_RUNTIME_BOUNDARY_VALID = all(
    [
        BLOCK_3_RECURRENT_OBJECTS_NOT_CREATED_BY_BLOCK_3,
        BLOCK_3_PREEXISTING_RECURRENT_OBJECTS_PRESERVED,
    ]
)


# Compatibility diagnostic only: on a clean first run this is True; on a
# rerun after later blocks it may legitimately be False.
BLOCK_3_RECURRENT_OBJECTS_ABSENT = (
    len(
        BLOCK_3_RECURRENT_OBJECTS_PRESENT_AFTER
    )
    ==
    0
)


if not BLOCK_3_RECURRENT_RUNTIME_BOUNDARY_VALID:

    raise RuntimeError(
        "Notebook 09 Block 3 violated the downstream recurrent-state "
        "execution boundary. "
        f"New objects: {list(BLOCK_3_NEW_RECURRENT_OBJECTS_CREATED)}"
    )


# =============================================================================
# Expose canonical Notebook 09 Transformative-Weight contract
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES_READY = True


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_SCOPE = {
    "structural_dimensions":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS,

    "trainable_parameters":
        BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS,

    "deferred_dimensions":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT,

    "pathway_trainable_counts":
        deepcopy(
            BLOCK_3_TRAINABLE_PARAMETER_COUNTS
        ),

    "pathway_deferred_counts":
        deepcopy(
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_COUNTS
        ),

    "active_initial_latent":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT,

    "active_initial_effective":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE,

    "deferred_effective":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE,
}


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_INITIAL_EFFECTIVE_VALUES = {
    pathway_name:
        weights.clone()

    for (
        pathway_name,
        weights,
    ) in BLOCK_3_INITIAL_EFFECTIVE_WEIGHTS.items()
}


# =============================================================================
# Final Block 3 validation
# =============================================================================

NOTEBOOK_09_BLOCK_3_ERRORS = []


BLOCK_3_VALIDATION_CHECKS = {
    "inherited_contract_not_ready":
        BLOCK_3_INHERITED_READY,

    "pathway_contract_invalid":
        BLOCK_3_PATHWAY_CONTRACT_VALID,

    "module_dimensions_invalid":
        BLOCK_3_MODULE_DIMENSIONS_VALID,

    "active_indices_invalid":
        BLOCK_3_ALL_ACTIVE_INDEX_BUFFERS_VALID,

    "deferred_indices_invalid":
        BLOCK_3_ALL_DEFERRED_INDEX_BUFFERS_VALID,

    "active_masks_invalid":
        BLOCK_3_ALL_ACTIVE_MASKS_VALID,

    "trainable_parameter_counts_invalid":
        BLOCK_3_TRAINABLE_PARAMETER_COUNTS_VALID,

    "deferred_parameters_created":
        BLOCK_3_ONLY_ACTIVE_PARAMETERS_EXIST,

    "complete_parameter_accounting_invalid":
        BLOCK_3_COMPLETE_PARAMETER_ACCOUNTING_VALID,

    "latent_initialisation_invalid":
        BLOCK_3_ALL_LATENT_INITIALISATION_VALID,

    "active_effective_initialisation_invalid":
        BLOCK_3_ALL_ACTIVE_EFFECTIVE_WEIGHTS_VALID,

    "deferred_effective_initialisation_invalid":
        BLOCK_3_ALL_DEFERRED_EFFECTIVE_WEIGHTS_VALID,

    "effective_weights_not_finite":
        BLOCK_3_ALL_EFFECTIVE_WEIGHTS_FINITE,

    "effective_weights_not_bounded":
        BLOCK_3_ALL_EFFECTIVE_WEIGHTS_BOUNDED,

    "active_sigmoid_interval_invalid":
        BLOCK_3_ALL_ACTIVE_SIGMOID_INTERVALS_VALID,

    "gradient_scope_invalid":
        BLOCK_3_GRADIENT_SCOPE_VALID,

    "parameter_sets_not_independent":
        BLOCK_3_PARAMETER_SETS_INDEPENDENT,

    "weight_device_placement_invalid":
        BLOCK_3_WEIGHT_DEVICE_PLACEMENT_VALID,

    "weight_buffer_device_placement_invalid":
        BLOCK_3_WEIGHT_BUFFER_DEVICE_PLACEMENT_VALID,

    "representation_state_changed":
        BLOCK_3_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_3_TRANSFORMATIVE_UNCHANGED,

    "inherited_modules_not_eval":
        BLOCK_3_INHERITED_MODULES_STILL_EVAL,

    "structural_output_dimensions_invalid":
        BLOCK_3_STRUCTURAL_OUTPUT_DIMENSIONS_VALID,

    "weight_modules_not_reconstructed":
        BLOCK_3_WEIGHT_MODULES_RECONSTRUCTED,

    "recurrent_runtime_boundary_invalid":
        BLOCK_3_RECURRENT_RUNTIME_BOUNDARY_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_3_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_3_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation validation
# =============================================================================

BLOCK_3_PROHIBITED_OPERATIONS = {
    "weight_forward_incorrectly_executed":
        NOTEBOOK_09_BLOCK_3_WEIGHT_FORWARD_EXECUTED,

    "representation_forward_incorrectly_executed":
        NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED,

    "transformative_forward_incorrectly_executed":
        NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED,

    "recurrent_state_incorrectly_created":
        NOTEBOOK_09_RECURRENT_STATE_CREATED,

    "recurrent_update_incorrectly_executed":
        NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED,

    "loss_incorrectly_calculated":
        NOTEBOOK_09_BLOCK_3_LOSS_CALCULATED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_3_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_3_BACKWARD_PASS_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_3_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_3_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_3_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_3_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 3 state
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID = (
    len(
        NOTEBOOK_09_BLOCK_3_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_VALIDATION_READY = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID
)


NOTEBOOK_09_BLOCK_3_VALID = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID
)


if not NOTEBOOK_09_BLOCK_3_VALID:

    raise RuntimeError(
        "Notebook 09 Block 3 Transformative-Weight module "
        "construction validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_3_ERRORS}"
    )


NOTEBOOK_09_BLOCK_3_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_3_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_3,

    "block_name":
        NOTEBOOK_09_BLOCK_3_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_3_VERSION,

    "pathway_dimensions":
        deepcopy(
            BLOCK_3_PATHWAY_DIMENSIONS
        ),

    "active_dimension_indices":
        deepcopy(
            BLOCK_3_ACTIVE_DIMENSION_INDICES
        ),

    "deferred_dimension_indices":
        deepcopy(
            BLOCK_3_DEFERRED_DIMENSION_INDICES
        ),

    "trainable_parameter_counts":
        deepcopy(
            BLOCK_3_TRAINABLE_PARAMETER_COUNTS
        ),

    "trainable_parameters":
        BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS,

    "structural_weight_dimensions":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS,

    "deferred_dimensions":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT,

    "inherited_parameters":
        NOTEBOOK_09_INHERITED_PARAMETER_COUNT,

    "total_parameters_after_construction":
        NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION,

    "active_initial_latent":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT,

    "active_initial_effective":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE,

    "deferred_effective":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE,

    "transformative_weights_created":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED,

    "recurrent_state_created":
        NOTEBOOK_09_RECURRENT_STATE_CREATED,

    "weight_forward_executed":
        NOTEBOOK_09_BLOCK_3_WEIGHT_FORWARD_EXECUTED,

    "rerun_detected":
        BLOCK_3_RERUN_DETECTED,

    "preexisting_weight_module_count":
        len(
            BLOCK_3_PREEXISTING_WEIGHT_MODULE_IDS
        ),

    "weight_modules_reconstructed":
        BLOCK_3_WEIGHT_MODULES_RECONSTRUCTED,

    "preexisting_recurrent_object_count":
        len(
            BLOCK_3_PREEXISTING_RECURRENT_OBJECTS
        ),

    "new_recurrent_objects_created":
        tuple(
            BLOCK_3_NEW_RECURRENT_OBJECTS_CREATED
        ),

    "preexisting_recurrent_objects_preserved":
        BLOCK_3_PREEXISTING_RECURRENT_OBJECTS_PRESERVED,

    "loss_calculated":
        NOTEBOOK_09_BLOCK_3_LOSS_CALCULATED,

    "optimizer_created":
        NOTEBOOK_09_BLOCK_3_OPTIMIZER_CREATED,

    "parameter_update_executed":
        NOTEBOOK_09_BLOCK_3_PARAMETER_UPDATE_EXECUTED,

    "module_construction_valid":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID,

    "block_valid":
        NOTEBOOK_09_BLOCK_3_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_3_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 3: "
    "Controlled Transformative-Weight Module Construction "
    "and Parameter-Scope Validation"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_3_VERSION}"
)

print("-" * 72)

print(
    "Inherited architecture"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Combined inherited params    : "
    f"{NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_3_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_3_TRANSFORMATIVE_STILL_FROZEN}"
)

print("-" * 72)

print(
    "Constructed Transformative-Weight modules"
)

for pathway_name in BLOCK_3_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} structural dim.    : "
        f"{BLOCK_3_MODULE_DIMENSIONS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} active dimensions  : "
        f"{list(BLOCK_3_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} deferred dimensions: "
        f"{list(BLOCK_3_DEFERRED_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} trainable params    : "
        f"{BLOCK_3_TRAINABLE_PARAMETER_COUNTS[pathway_name]}"
    )

print("-" * 72)

print(
    "Parameter accounting"
)

print(
    f"Structural weight dimensions : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS}"
)

print(
    f"Trainable weight parameters   : "
    f"{BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS}"
)

print(
    f"Deferred structural dims      : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT}"
)

print(
    f"Total model parameters        : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print("-" * 72)

print(
    "Initial Transformative-Weight state"
)

print(
    f"Active latent initial value   : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_LATENT:.1f}"
)

print(
    f"Active effective weight       : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE:.1f}"
)

print(
    f"Deferred effective weight     : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE:.1f}"
)

print(
    f"Latent initialisation valid   : "
    f"{BLOCK_3_ALL_LATENT_INITIALISATION_VALID}"
)

print(
    f"Active weights valid          : "
    f"{BLOCK_3_ALL_ACTIVE_EFFECTIVE_WEIGHTS_VALID}"
)

print(
    f"Deferred weights valid        : "
    f"{BLOCK_3_ALL_DEFERRED_EFFECTIVE_WEIGHTS_VALID}"
)

print(
    f"Effective weights finite      : "
    f"{BLOCK_3_ALL_EFFECTIVE_WEIGHTS_FINITE}"
)

print(
    f"Effective weights bounded     : "
    f"{BLOCK_3_ALL_EFFECTIVE_WEIGHTS_BOUNDED}"
)

print("-" * 72)

print(
    "Parameter-scope validation"
)

print(
    f"Only active params created    : "
    f"{BLOCK_3_ONLY_ACTIVE_PARAMETERS_EXIST}"
)

print(
    f"Weight parameters trainable   : "
    f"{BLOCK_3_WEIGHT_PARAMETERS_TRAINABLE}"
)

print(
    f"Parameter sets independent    : "
    f"{BLOCK_3_PARAMETER_SETS_INDEPENDENT}"
)

print(
    f"Weight device valid           : "
    f"{BLOCK_3_WEIGHT_DEVICE_PLACEMENT_VALID}"
)

print(
    f"Weight buffers device valid   : "
    f"{BLOCK_3_WEIGHT_BUFFER_DEVICE_PLACEMENT_VALID}"
)

print(
    f"Representation unchanged      : "
    f"{BLOCK_3_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged      : "
    f"{BLOCK_3_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Inherited modules eval        : "
    f"{BLOCK_3_INHERITED_MODULES_STILL_EVAL}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Transform weights created     : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED}"
)

print(
    f"Weight forward executed       : "
    f"{NOTEBOOK_09_BLOCK_3_WEIGHT_FORWARD_EXECUTED}"
)

print(
    f"Recurrent state created       : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CREATED}"
)

print(
    f"Recurrent update executed     : "
    f"{NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"Representation forward        : "
    f"{NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED}"
)

print(
    f"Transform forward             : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED}"
)

print(
    f"Loss calculated               : "
    f"{NOTEBOOK_09_BLOCK_3_LOSS_CALCULATED}"
)

print(
    f"Optimizer created             : "
    f"{NOTEBOOK_09_BLOCK_3_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed        : "
    f"{NOTEBOOK_09_BLOCK_3_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed     : "
    f"{NOTEBOOK_09_BLOCK_3_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Notebook rerun detected       : "
    f"{BLOCK_3_RERUN_DETECTED}"
)

print(
    f"Pre-existing weight modules   : "
    f"{len(BLOCK_3_PREEXISTING_WEIGHT_MODULE_IDS)}"
)

print(
    f"Weight modules reconstructed  : "
    f"{BLOCK_3_WEIGHT_MODULES_RECONSTRUCTED}"
)

print(
    f"Pre-existing recurrent objs   : "
    f"{len(BLOCK_3_PREEXISTING_RECURRENT_OBJECTS)}"
)

print(
    f"New recurrent objects created : "
    f"{len(BLOCK_3_NEW_RECURRENT_OBJECTS_CREATED)}"
)

print(
    f"Existing recurrent objs kept  : "
    f"{BLOCK_3_PREEXISTING_RECURRENT_OBJECTS_PRESERVED}"
)

print("-" * 72)

print(
    f"Weight modules ready          : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES_READY}"
)

print(
    f"Construction valid            : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID}"
)

print(
    f"Forward validation ready      : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_VALIDATION_READY}"
)

print(
    f"Block valid                   : "
    f"{NOTEBOOK_09_BLOCK_3_VALID}"
)

print(
    f"Block complete                : "
    f"{NOTEBOOK_09_BLOCK_3_COMPLETE}"
)

print("=" * 72)

print(
    f"{len(NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES)} independent "
    "Transformative-Weight pathway module(s) were constructed successfully "
    "from the current validated Block 2 contract."
)

print(
    f"Only the {BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS} "
    "activation-eligible latent weight parameter(s) were created as "
    "trainable parameters."
)

print(
    f"The complete structural Transformative-Weight space remains "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS}-dimensional, "
    f"with {NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT} "
    "deferred dimension(s) represented by fixed zero-valued gating."
)

print(
    "All active latent parameters were initialised at zero, corresponding "
    "to an effective sigmoid weight of 0.5."
)

print(
    "All deferred effective weights remain exactly zero under the current "
    "gating contract."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} representation "
    "and transformative parameters remain frozen and exactly unchanged."
)

print(
    "Block 3 executed no Transformative-Weight forward pass, representation "
    "forward pass, transformative forward pass, recurrent-state creation, "
    "recurrent update, loss, optimiser, backward pass or parameter update."
)

if BLOCK_3_RERUN_DETECTED:

    print(
        "Pre-existing downstream runtime objects from an earlier execution "
        "were handled rerun-safely: Block 3 reconstructed only the "
        "Transformative-Weight modules it owns and left later recurrent "
        "objects unchanged for subsequent blocks to rebuild or replace."
    )

print(
    "Notebook 09 may now proceed to controlled Transformative-Weight "
    "forward-path validation."
)

print("=" * 72)

Media AI — Notebook 09, Block 3: Controlled Transformative-Weight Module Construction and Parameter-Scope Validation
Block version                : 1.3
------------------------------------------------------------------------
Inherited architecture
Representation parameters    : 894,003
Transformative parameters    : 8,187
Combined inherited params    : 902,190
Representation frozen        : True
Transformative frozen        : True
------------------------------------------------------------------------
Constructed Transformative-Weight modules
factual        structural dim.    : 10
factual        active dimensions  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
factual        deferred dimensions: []
factual        trainable params    : 10
psychological  structural dim.    : 34
psychological  active dimensions  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
psychological  deferred dimensions: []
psychological  trainable

## Block 4 — Controlled Transformative-Weight Forward-Path Validation

This block validates the runtime behaviour of the newly constructed factual, psychological, and social Transformative-Weight modules.

Block 3 instantiated the first new trainable Notebook 09 parameters while preserving the complete inherited Notebook 08 architecture unchanged. The resulting Transformative-Weight modules contain only the 25 activation-eligible latent parameters defined by the Block 2 contract, while the full 51-dimensional structural weight space remains available through explicit active/deferred masking.

The purpose of Block 4 is to execute the Transformative-Weight forward path for the first time and verify that the implemented modules reproduce the intended bounded weighting behaviour exactly.

No recurrent state is created in this block. No recurrent update is executed. No representation or transformative forward pass is required. No loss, optimiser, backward pass, or parameter update is performed.

### Forward-path definition

For each pathway \(k\in\{F,P,S\}\), the Transformative-Weight module stores only the active latent parameters

$$
\alpha_k^{(A)}.
$$

The complete effective Transformative-Weight vector is reconstructed over the full pathway dimension.

For active dimensions,

$$
j\in A_k,
$$

the effective weight is

$$
TW_{k,j}
=
\sigma(\alpha_{k,j}).
$$

For deferred dimensions,

$$
j\in D_k,
$$

the effective weight is fixed to zero:

$$
TW_{k,j}=0.
$$

The resulting structural vector therefore has the form

$$
TW_k
=
m_k\odot\sigma(\alpha_k),
$$

where \(m_k\) is the inherited binary activation mask.

The complete pathway outputs must satisfy

$$
TW_F\in\mathbb{R}^{10},
$$

$$
TW_P\in\mathbb{R}^{34},
$$

and

$$
TW_S\in\mathbb{R}^{7}.
$$

### Initial forward-state expectation

Block 3 established that all active latent parameters initialise at

$$
\alpha_{k,j}=0.
$$

Therefore, the first Transformative-Weight forward evaluation should reproduce

$$
\sigma(0)=0.5.
$$

The expected effective weight values are consequently

$$
TW_{k,j}
=
\begin{cases}
0.5, & j\in A_k,\\
0, & j\in D_k.
\end{cases}
$$

This provides an exact deterministic target for forward-path validation.

### Shape validation

The first validation requirement is structural dimensionality.

The factual weight module must return

$$
TW_F\in\mathbb{R}^{10},
$$

the psychological module must return

$$
TW_P\in\mathbb{R}^{34},
$$

and the social module must return

$$
TW_S\in\mathbb{R}^{7}.
$$

No active-only compressed vector is permitted as the public architectural output.

The full pathway dimension must always be preserved because the later recurrent mechanism operates over complete factual, psychological, and social state spaces.

### Numerical validity

Every effective Transformative Weight must be finite.

For pathway \(k\),

$$
TW_k
\in
\mathbb{R}^{d_k}
$$

must satisfy

$$
\operatorname{isfinite}(TW_{k,j})
=
\mathrm{True}
$$

for every dimension \(j\).

The block also validates the boundedness contract

$$
0\leq TW_{k,j}\leq1.
$$

For active dimensions specifically,

$$
0<TW_{k,j}<1,
$$

because finite latent values are passed through the sigmoid transformation.

Deferred dimensions are permitted to equal the lower boundary exactly because their value is imposed by the fixed structural mask.

### Active-dimension validation

The active dimensions inherited from Notebook 08 remain the only dimensions controlled by trainable Transformative-Weight parameters.

For the factual pathway,

$$
A_F=\{1,2,4,8,9\}.
$$

For the psychological pathway,

$$
A_P=
\{1,4,11,12,14,15,16,19,21,22,23,25,26,27,30,32,33\}.
$$

For the social pathway,

$$
A_S=\{0,2,4\}.
$$

At initialisation, every one of these active dimensions must produce

$$
TW_{k,j}=0.5.
$$

The validation checks both value and position.

A correct count of 25 active dimensions is insufficient if the nonzero values appear at the wrong semantic indices.

### Deferred-dimension validation

All deferred dimensions must remain exactly zero under the current pilot contract.

For every pathway,

$$
TW_{k,j}=0
\qquad
\text{for }j\in D_k.
$$

This verifies that the forward implementation does not accidentally apply sigmoid values to dimensions that were intentionally deferred.

The complete deferred count remains

$$
26.
$$

These zero values represent architectural gating rather than learned evidence of zero transformation.

### Deterministic forward behaviour

The Transformative-Weight modules contain no stochastic operation.

Repeated evaluation of an unchanged module state must therefore return exactly the same effective vector:

$$
TW_k^{(1)}
=
TW_k^{(2)}.
$$

The block evaluates every pathway more than once under identical conditions and requires exact equality.

This ensures that Transformative-Weight generation itself introduces no stochasticity into the later recurrent mechanism.

### Pathway independence

The factual, psychological, and social Transformative-Weight modules were constructed with disjoint parameter sets.

Block 4 verifies that this structural independence remains intact during forward execution.

The three outputs correspond only to their respective pathway modules:

$$
TW_F\leftrightarrow\alpha_F,
$$

$$
TW_P\leftrightarrow\alpha_P,
$$

$$
TW_S\leftrightarrow\alpha_S.
$$

No cross-pathway parameter sharing or output construction is permitted.

This preserves the semantic separation established throughout the earlier Media AI architecture.

### Parameter immutability during validation

A forward pass must not change model parameters.

The block therefore snapshots the complete Transformative-Weight parameter state before validation and verifies exact equality after the forward evaluations:

$$
\theta_{TW}^{\mathrm{after}}
=
\theta_{TW}^{\mathrm{before}}.
$$

The same protection applies to the inherited architecture.

The representation state must satisfy

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}},
$$

and the inherited transformative-mechanism state must satisfy

$$
\theta_T^{\mathrm{after}}
=
\theta_T^{\mathrm{before}}.
$$

This ensures that Block 4 is purely evaluative.

### Gradient-state boundary

The newly introduced active latent Transformative-Weight parameters remain trainable:

$$
\texttt{requires\_grad}(\alpha_{k,j})
=
\mathrm{True}
$$

for active dimensions.

However, Block 4 does not calculate a loss or call backpropagation.

No gradient-based learning occurs.

The inherited representation and transformative parameters remain frozen:

$$
\texttt{requires\_grad}(\theta_R)
=
\mathrm{False},
$$

$$
\texttt{requires\_grad}(\theta_T)
=
\mathrm{False}.
$$

### Separation from candidate transformations

Block 4 validates only the Transformative-Weight modules.

It does not require a representation forward pass and does not require the inherited transformative mechanisms to generate candidate transformations.

Therefore, no

$$
c_t^{(F)},
\qquad
c_t^{(P)},
\qquad
c_t^{(S)}
$$

need to be computed in this block.

This is intentional.

The purpose is to verify the weighting mechanism independently before it is combined with candidate transformations.

### No recurrent state yet

Even after successful Transformative-Weight forward validation, recurrent state remains absent.

The block does not instantiate

$$
h_t^{(F)},
\qquad
h_t^{(P)},
\qquad
h_t^{(S)}.
$$

It also does not calculate a weighted candidate transformation

$$
\Delta h_t^{(k)}
=
TW_k\odot c_t^{(k)}
$$

because candidate-transformative forward execution is outside the present scope.

Likewise, the recurrent update

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}
$$

is not executed.

The recurrent mechanism remains the next architectural stage.

### No optimisation in Block 4

The forward validation is performed without optimisation.

Accordingly:

- no loss is calculated;
- no optimiser is created;
- no gradients are backpropagated;
- no gradient clipping is applied;
- no optimiser step is executed;
- and no parameter is updated.

The effective Transformative Weights after validation must therefore be exactly the same as before validation.

### Block 4 completion contract

Block 4 is complete only when:

- factual, psychological, and social Transformative-Weight forward passes execute successfully;
- the output dimensions are exactly 10, 34, and 7;
- all output values are finite;
- all effective weights satisfy the bounded interval contract;
- all 25 active dimensions produce the expected initial value 0.5;
- all 26 deferred dimensions remain exactly zero;
- active values occupy the correct inherited semantic indices;
- repeated evaluations are exactly deterministic;
- factual, psychological, and social pathway parameter sets remain independent;
- the 25 latent weight parameters remain trainable;
- no Transformative-Weight parameter changes during validation;
- the inherited 902,190 parameters remain frozen and exactly unchanged;
- no representation forward pass is executed;
- no candidate-transformative forward pass is executed;
- no recurrent state is created;
- no recurrent update is executed;
- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- and no parameter update occurs.

Successful completion establishes that the new Transformative-Weight mechanism is structurally and numerically valid:

$$
\boxed{
TW_F\in[0,1]^{10},
\qquad
TW_P\in[0,1]^{34},
\qquad
TW_S\in[0,1]^{7}
}
$$

with the current initial condition

$$
TW_{k,j}
=
\begin{cases}
0.5, & j\in A_k,\\
0, & j\in D_k.
\end{cases}
$$

Notebook 09 may then proceed to explicit recurrent-state architectural definition and deterministic initial-state construction.

In [54]:
# =============================================================================
# Media AI — Notebook 09
# Block 4: Controlled Transformative-Weight Forward-Path Validation
# =============================================================================

from copy import deepcopy

import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_4 = 4

NOTEBOOK_09_BLOCK_4_NAME = (
    "Controlled Transformative-Weight Forward-Path Validation"
)

NOTEBOOK_09_BLOCK_4_VERSION = "1.2"


# =============================================================================
# Required inherited runtime contract
# =============================================================================

BLOCK_4_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_BLOCK_1_COMPLETE",
    "NOTEBOOK_09_BLOCK_1_VALID",
    "NOTEBOOK_09_HANDOVER_RESTORED",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Block 2
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_2_COMPLETE",
    "NOTEBOOK_09_BLOCK_2_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ARCHITECTURE_READY",

    # -------------------------------------------------------------------------
    # Block 3
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_3_COMPLETE",
    "NOTEBOOK_09_BLOCK_3_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_VALIDATION_READY",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE",
    "BLOCK_3_PATHWAY_DIMENSIONS",
    "BLOCK_3_ACTIVE_DIMENSION_INDICES",
    "BLOCK_3_DEFERRED_DIMENSION_INDICES",
    "BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS",
    "BLOCK_3_PARAMETER_SETS_INDEPENDENT",
    "BLOCK_3_REPRESENTATION_STILL_FROZEN",
    "BLOCK_3_TRANSFORMATIVE_STILL_FROZEN",
    "BLOCK_3_INHERITED_MODULES_STILL_EVAL",

    # -------------------------------------------------------------------------
    # Environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_4_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_4_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_4_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 4 prerequisites are not initialised. "
        f"Missing: {BLOCK_4_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_4_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,

        NOTEBOOK_09_BLOCK_1_COMPLETE is True,
        NOTEBOOK_09_BLOCK_1_VALID is True,
        NOTEBOOK_09_HANDOVER_RESTORED is True,

        NOTEBOOK_09_BLOCK_2_COMPLETE is True,
        NOTEBOOK_09_BLOCK_2_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ARCHITECTURE_READY is True,

        NOTEBOOK_09_BLOCK_3_COMPLETE is True,
        NOTEBOOK_09_BLOCK_3_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_VALIDATION_READY is True,
    ]
)


if not BLOCK_4_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Blocks 1–3 must be valid and complete "
        "before Transformative-Weight forward-path validation."
    )


# =============================================================================
# Downstream-runtime snapshot for rerun-safe execution
# =============================================================================
#
# Block 4 owns only Transformative-Weight forward-path validation.
# Recurrent-state objects belong to later blocks and may legitimately remain
# in the interactive kernel from an earlier complete notebook execution.
#
# Block 4 therefore records any pre-existing recurrent objects and later
# verifies that it did not create, replace or mutate them.
# =============================================================================

BLOCK_4_DOWNSTREAM_RECURRENT_OBJECT_NAMES = (
    "NOTEBOOK_09_FACTUAL_RECURRENT_STATE",
    "NOTEBOOK_09_PSYCHOLOGICAL_RECURRENT_STATE",
    "NOTEBOOK_09_SOCIAL_RECURRENT_STATE",

    "NOTEBOOK_09_FACTUAL_INITIAL_STATE",
    "NOTEBOOK_09_PSYCHOLOGICAL_INITIAL_STATE",
    "NOTEBOOK_09_SOCIAL_INITIAL_STATE",

    "NOTEBOOK_09_INITIAL_RECURRENT_STATES",
    "NOTEBOOK_09_VALIDATED_SINGLE_STEP_RECURRENT_STATES",
    "NOTEBOOK_09_ARTICLE_FINAL_RECURRENT_STATES",
    "NOTEBOOK_09_POSTTRAIN_FINAL_RECURRENT_STATES",
)


BLOCK_4_PREEXISTING_RECURRENT_OBJECTS = {
    object_name:
        globals()[
            object_name
        ]

    for object_name
    in BLOCK_4_DOWNSTREAM_RECURRENT_OBJECT_NAMES

    if object_name in globals()
}


BLOCK_4_PREEXISTING_RECURRENT_OBJECT_IDS = {
    object_name:
        id(
            object_value
        )

    for (
        object_name,
        object_value,
    ) in BLOCK_4_PREEXISTING_RECURRENT_OBJECTS.items()
}


BLOCK_4_RERUN_DETECTED = bool(
    BLOCK_4_PREEXISTING_RECURRENT_OBJECTS
)


# =============================================================================
# Canonical pathway contract
# =============================================================================
#
# Block 4 consumes the validated Block 3 pathway dimensions and
# active/deferred semantic partitions exactly. No pilot-era dimensions
# or activation counts are recreated.
# =============================================================================

BLOCK_4_PATHWAY_DIMENSIONS = deepcopy(
    BLOCK_3_PATHWAY_DIMENSIONS
)


BLOCK_4_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in BLOCK_3_ACTIVE_DIMENSION_INDICES.items()
}


BLOCK_4_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in BLOCK_3_DEFERRED_DIMENSION_INDICES.items()
}


BLOCK_4_PATHWAY_CONTRACT_VALID = all(
    [
        (
            BLOCK_4_PATHWAY_DIMENSIONS
            ==
            BLOCK_3_PATHWAY_DIMENSIONS
        ),

        (
            BLOCK_4_ACTIVE_DIMENSION_INDICES
            ==
            BLOCK_3_ACTIVE_DIMENSION_INDICES
        ),

        (
            BLOCK_4_DEFERRED_DIMENSION_INDICES
            ==
            BLOCK_3_DEFERRED_DIMENSION_INDICES
        ),

        (
            set(
                BLOCK_4_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys()
            )
        ),
    ]
)


if not BLOCK_4_PATHWAY_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 09 Block 4 pathway contract does not reproduce "
        "the validated Block 3 Transformative-Weight architecture."
    )


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_4_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_4_nested_state_exact(
    state_before,
    state_after,
):

    if state_before.keys() != state_after.keys():

        return False


    for module_name in state_before:

        before_module_state = (
            state_before[
                module_name
            ]
        )


        after_module_state = (
            state_after[
                module_name
            ]
        )


        if (
            before_module_state.keys()
            !=
            after_module_state.keys()
        ):

            return False


        for tensor_name in before_module_state:

            if not torch.equal(
                before_module_state[
                    tensor_name
                ],
                after_module_state[
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot all model states before validation
# =============================================================================

BLOCK_4_REPRESENTATION_STATE_BEFORE = (
    block_4_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_4_TRANSFORMATIVE_STATE_BEFORE = (
    block_4_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_4_WEIGHT_STATE_BEFORE = (
    block_4_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Parameter-object identity snapshot
# =============================================================================

BLOCK_4_WEIGHT_PARAMETER_OBJECT_IDS_BEFORE = {
    pathway_name:
        tuple(
            id(
                parameter
            )

            for parameter
            in module.parameters()
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


# =============================================================================
# Controlled Transformative-Weight forward execution
# =============================================================================
#
# Only the three Transformative-Weight modules are executed.
#
# No representation forward pass is executed.
# No transformative-mechanism forward pass is executed.
# No candidate transformation is generated.
# No recurrent state is instantiated.
# =============================================================================

BLOCK_4_WEIGHT_OUTPUTS_FIRST = {}

BLOCK_4_WEIGHT_OUTPUTS_SECOND = {}


with torch.no_grad():

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items():

        BLOCK_4_WEIGHT_OUTPUTS_FIRST[
            pathway_name
        ] = (
            module()
            .detach()
            .cpu()
            .clone()
        )


        BLOCK_4_WEIGHT_OUTPUTS_SECOND[
            pathway_name
        ] = (
            module()
            .detach()
            .cpu()
            .clone()
        )


NOTEBOOK_09_BLOCK_4_WEIGHT_FORWARD_EXECUTED = True

NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED = False


# =============================================================================
# Output-shape validation
# =============================================================================

BLOCK_4_OUTPUT_SHAPES = {
    pathway_name:
        tuple(
            output.shape
        )

    for (
        pathway_name,
        output,
    ) in BLOCK_4_WEIGHT_OUTPUTS_FIRST.items()
}


BLOCK_4_EXPECTED_OUTPUT_SHAPES = {
    pathway_name:
        (
            pathway_dim,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_4_PATHWAY_DIMENSIONS.items()
}


BLOCK_4_OUTPUT_SHAPES_VALID = (
    BLOCK_4_OUTPUT_SHAPES
    ==
    BLOCK_4_EXPECTED_OUTPUT_SHAPES
)


if not BLOCK_4_OUTPUT_SHAPES_VALID:

    raise RuntimeError(
        "Transformative-Weight forward outputs do not preserve "
        "the required pathway dimensions."
    )


# =============================================================================
# Numerical validity
# =============================================================================

BLOCK_4_OUTPUTS_FINITE = {
    pathway_name:
        bool(
            torch.isfinite(
                output
            ).all().item()
        )

    for (
        pathway_name,
        output,
    ) in BLOCK_4_WEIGHT_OUTPUTS_FIRST.items()
}


BLOCK_4_OUTPUTS_BOUNDED = {
    pathway_name:
        bool(
            (
                (
                    output
                    >=
                    0.0
                )
                &
                (
                    output
                    <=
                    1.0
                )
            ).all().item()
        )

    for (
        pathway_name,
        output,
    ) in BLOCK_4_WEIGHT_OUTPUTS_FIRST.items()
}


BLOCK_4_ALL_OUTPUTS_FINITE = all(
    BLOCK_4_OUTPUTS_FINITE.values()
)


BLOCK_4_ALL_OUTPUTS_BOUNDED = all(
    BLOCK_4_OUTPUTS_BOUNDED.values()
)


if not all(
    [
        BLOCK_4_ALL_OUTPUTS_FINITE,
        BLOCK_4_ALL_OUTPUTS_BOUNDED,
    ]
):

    raise RuntimeError(
        "Transformative-Weight forward outputs contain invalid "
        "or out-of-range values."
    )


# =============================================================================
# Active-value validation
# =============================================================================

BLOCK_4_ACTIVE_VALUES_VALID = {}

BLOCK_4_ACTIVE_VALUES_STRICTLY_INTERIOR = {}


for (
    pathway_name,
    output,
) in BLOCK_4_WEIGHT_OUTPUTS_FIRST.items():

    active_indices = torch.tensor(
        BLOCK_4_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    active_values = (
        output[
            active_indices
        ]
    )


    expected_values = torch.full(
        active_values.shape,
        fill_value=
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE
            ),
        dtype=
            active_values.dtype,
    )


    BLOCK_4_ACTIVE_VALUES_VALID[
        pathway_name
    ] = torch.equal(
        active_values,
        expected_values,
    )


    BLOCK_4_ACTIVE_VALUES_STRICTLY_INTERIOR[
        pathway_name
    ] = bool(
        (
            (
                active_values
                >
                0.0
            )
            &
            (
                active_values
                <
                1.0
            )
        ).all().item()
    )


BLOCK_4_ALL_ACTIVE_VALUES_VALID = all(
    BLOCK_4_ACTIVE_VALUES_VALID.values()
)


BLOCK_4_ALL_ACTIVE_VALUES_STRICTLY_INTERIOR = all(
    BLOCK_4_ACTIVE_VALUES_STRICTLY_INTERIOR.values()
)


if not all(
    [
        BLOCK_4_ALL_ACTIVE_VALUES_VALID,
        BLOCK_4_ALL_ACTIVE_VALUES_STRICTLY_INTERIOR,
    ]
):

    raise RuntimeError(
        "Active Transformative-Weight values do not reproduce "
        "the expected initial value of 0.5."
    )


# =============================================================================
# Deferred-value validation
# =============================================================================

BLOCK_4_DEFERRED_VALUES_VALID = {}


for (
    pathway_name,
    output,
) in BLOCK_4_WEIGHT_OUTPUTS_FIRST.items():

    deferred_indices = torch.tensor(
        BLOCK_4_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    deferred_values = (
        output[
            deferred_indices
        ]
    )


    expected_values = torch.full(
        deferred_values.shape,
        fill_value=
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE
            ),
        dtype=
            deferred_values.dtype,
    )


    BLOCK_4_DEFERRED_VALUES_VALID[
        pathway_name
    ] = torch.equal(
        deferred_values,
        expected_values,
    )


BLOCK_4_ALL_DEFERRED_VALUES_VALID = all(
    BLOCK_4_DEFERRED_VALUES_VALID.values()
)


if not BLOCK_4_ALL_DEFERRED_VALUES_VALID:

    raise RuntimeError(
        "Deferred Transformative-Weight dimensions are not exactly zero."
    )


# =============================================================================
# Exact semantic-index validation
# =============================================================================
#
# This validates not only counts but the exact positions of the active and
# deferred values.
# =============================================================================

BLOCK_4_SEMANTIC_INDEX_LAYOUT_VALID = {}


for (
    pathway_name,
    output,
) in BLOCK_4_WEIGHT_OUTPUTS_FIRST.items():

    expected_output = torch.full(
        (
            BLOCK_4_PATHWAY_DIMENSIONS[
                pathway_name
            ],
        ),
        fill_value=
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE
            ),
        dtype=
            output.dtype,
    )


    active_indices = list(
        BLOCK_4_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    expected_output[
        active_indices
    ] = float(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE
    )


    BLOCK_4_SEMANTIC_INDEX_LAYOUT_VALID[
        pathway_name
    ] = torch.equal(
        output,
        expected_output,
    )


BLOCK_4_ALL_SEMANTIC_INDEX_LAYOUT_VALID = all(
    BLOCK_4_SEMANTIC_INDEX_LAYOUT_VALID.values()
)


if not BLOCK_4_ALL_SEMANTIC_INDEX_LAYOUT_VALID:

    raise RuntimeError(
        "Transformative-Weight values do not occupy the correct "
        "active/deferred semantic indices."
    )


# =============================================================================
# Deterministic repeated-evaluation validation
# =============================================================================

BLOCK_4_DETERMINISTIC = {
    pathway_name:
        torch.equal(
            BLOCK_4_WEIGHT_OUTPUTS_FIRST[
                pathway_name
            ],
            BLOCK_4_WEIGHT_OUTPUTS_SECOND[
                pathway_name
            ],
        )

    for pathway_name
    in BLOCK_4_WEIGHT_OUTPUTS_FIRST
}


BLOCK_4_ALL_DETERMINISTIC = all(
    BLOCK_4_DETERMINISTIC.values()
)


if not BLOCK_4_ALL_DETERMINISTIC:

    raise RuntimeError(
        "Transformative-Weight forward evaluation is not deterministic."
    )


# =============================================================================
# Effective-value count validation
# =============================================================================

BLOCK_4_ACTIVE_VALUE_COUNTS = {}

BLOCK_4_DEFERRED_VALUE_COUNTS = {}


for (
    pathway_name,
    output,
) in BLOCK_4_WEIGHT_OUTPUTS_FIRST.items():

    active_count = int(
        (
            output
            ==
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE
            )
        )
        .sum()
        .item()
    )


    deferred_count = int(
        (
            output
            ==
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE
            )
        )
        .sum()
        .item()
    )


    BLOCK_4_ACTIVE_VALUE_COUNTS[
        pathway_name
    ] = active_count


    BLOCK_4_DEFERRED_VALUE_COUNTS[
        pathway_name
    ] = deferred_count


BLOCK_4_EXPECTED_ACTIVE_VALUE_COUNTS = {
    pathway_name:
        len(
            BLOCK_4_ACTIVE_DIMENSION_INDICES[
                pathway_name
            ]
        )

    for pathway_name
    in BLOCK_4_PATHWAY_DIMENSIONS
}


BLOCK_4_EXPECTED_DEFERRED_VALUE_COUNTS = {
    pathway_name:
        len(
            BLOCK_4_DEFERRED_DIMENSION_INDICES[
                pathway_name
            ]
        )

    for pathway_name
    in BLOCK_4_PATHWAY_DIMENSIONS
}


BLOCK_4_VALUE_COUNTS_VALID = all(
    [
        (
            BLOCK_4_ACTIVE_VALUE_COUNTS
            ==
            BLOCK_4_EXPECTED_ACTIVE_VALUE_COUNTS
        ),

        (
            BLOCK_4_DEFERRED_VALUE_COUNTS
            ==
            BLOCK_4_EXPECTED_DEFERRED_VALUE_COUNTS
        ),

        (
            sum(
                BLOCK_4_ACTIVE_VALUE_COUNTS.values()
            )
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            sum(
                BLOCK_4_DEFERRED_VALUE_COUNTS.values()
            )
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT
        ),

        (
            sum(
                BLOCK_4_ACTIVE_VALUE_COUNTS.values()
            )
            +
            sum(
                BLOCK_4_DEFERRED_VALUE_COUNTS.values()
            )
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS
        ),
    ]
)


if not BLOCK_4_VALUE_COUNTS_VALID:

    raise RuntimeError(
        "Transformative-Weight active/deferred output-value counts "
        "are invalid."
    )


# =============================================================================
# Gradient-scope validation
# =============================================================================

BLOCK_4_WEIGHT_PARAMETERS_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_4_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_4_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_4_GRADIENT_SCOPE_VALID = all(
    [
        BLOCK_4_WEIGHT_PARAMETERS_TRAINABLE,
        BLOCK_4_REPRESENTATION_STILL_FROZEN,
        BLOCK_4_TRANSFORMATIVE_STILL_FROZEN,
    ]
)


if not BLOCK_4_GRADIENT_SCOPE_VALID:

    raise RuntimeError(
        "Notebook 09 Transformative-Weight gradient scope is invalid."
    )


# =============================================================================
# Validate absence of accumulated gradients
# =============================================================================
#
# torch.no_grad() was used for the forward validation, so no gradients should
# exist on any Transformative-Weight parameter.
# =============================================================================

BLOCK_4_WEIGHT_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


if not BLOCK_4_WEIGHT_GRADIENTS_ABSENT:

    raise RuntimeError(
        "Transformative-Weight gradients unexpectedly exist "
        "after forward-path validation."
    )


# =============================================================================
# Pathway parameter independence
# =============================================================================

BLOCK_4_WEIGHT_PARAMETER_OBJECT_IDS_AFTER = {
    pathway_name:
        tuple(
            id(
                parameter
            )

            for parameter
            in module.parameters()
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


BLOCK_4_PARAMETER_OBJECTS_STABLE = (
    BLOCK_4_WEIGHT_PARAMETER_OBJECT_IDS_BEFORE
    ==
    BLOCK_4_WEIGHT_PARAMETER_OBJECT_IDS_AFTER
)


BLOCK_4_PATHWAY_NAMES = tuple(
    BLOCK_4_WEIGHT_PARAMETER_OBJECT_IDS_AFTER.keys()
)


BLOCK_4_PARAMETER_SETS_INDEPENDENT = all(
    set(
        BLOCK_4_WEIGHT_PARAMETER_OBJECT_IDS_AFTER[
            BLOCK_4_PATHWAY_NAMES[
                left_index
            ]
        ]
    ).isdisjoint(
        set(
            BLOCK_4_WEIGHT_PARAMETER_OBJECT_IDS_AFTER[
                BLOCK_4_PATHWAY_NAMES[
                    right_index
                ]
            ]
        )
    )

    for left_index
    in range(
        len(
            BLOCK_4_PATHWAY_NAMES
        )
    )

    for right_index
    in range(
        left_index + 1,
        len(
            BLOCK_4_PATHWAY_NAMES
        ),
    )
)


if not all(
    [
        BLOCK_4_PARAMETER_OBJECTS_STABLE,
        BLOCK_4_PARAMETER_SETS_INDEPENDENT,
    ]
):

    raise RuntimeError(
        "Transformative-Weight pathway parameter independence "
        "was not preserved."
    )


# =============================================================================
# Snapshot all model states after validation
# =============================================================================

BLOCK_4_REPRESENTATION_STATE_AFTER = (
    block_4_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_4_TRANSFORMATIVE_STATE_AFTER = (
    block_4_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_4_WEIGHT_STATE_AFTER = (
    block_4_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Parameter immutability validation
# =============================================================================

BLOCK_4_REPRESENTATION_UNCHANGED = (
    block_4_nested_state_exact(
        BLOCK_4_REPRESENTATION_STATE_BEFORE,
        BLOCK_4_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_4_TRANSFORMATIVE_UNCHANGED = (
    block_4_nested_state_exact(
        BLOCK_4_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_4_TRANSFORMATIVE_STATE_AFTER,
    )
)


BLOCK_4_WEIGHT_STATE_UNCHANGED = (
    block_4_nested_state_exact(
        BLOCK_4_WEIGHT_STATE_BEFORE,
        BLOCK_4_WEIGHT_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_4_REPRESENTATION_UNCHANGED,
        BLOCK_4_TRANSFORMATIVE_UNCHANGED,
        BLOCK_4_WEIGHT_STATE_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Model parameter state changed during controlled "
        "Transformative-Weight forward validation."
    )


# =============================================================================
# Inherited evaluation-mode validation
# =============================================================================

BLOCK_4_INHERITED_MODULES_STILL_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


if not BLOCK_4_INHERITED_MODULES_STILL_EVAL:

    raise RuntimeError(
        "Inherited Notebook 08 modules unexpectedly left evaluation mode."
    )


# =============================================================================
# Device-placement validation
# =============================================================================

BLOCK_4_WEIGHT_PARAMETER_DEVICE_VALID = all(
    parameter.device
    ==
    DEVICE

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_4_WEIGHT_BUFFER_DEVICE_VALID = all(
    buffer.device
    ==
    DEVICE

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for buffer
    in module.buffers()
)


if not all(
    [
        BLOCK_4_WEIGHT_PARAMETER_DEVICE_VALID,
        BLOCK_4_WEIGHT_BUFFER_DEVICE_VALID,
    ]
):

    raise RuntimeError(
        "Transformative-Weight parameters or buffers are on "
        "an unexpected execution device."
    )


# =============================================================================
# Parameter-accounting validation
# =============================================================================

BLOCK_4_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS
            ==
            sum(
                BLOCK_4_PATHWAY_DIMENSIONS.values()
            )
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT
            ==
            sum(
                len(
                    BLOCK_4_DEFERRED_DIMENSION_INDICES[
                        pathway_name
                    ]
                )

                for pathway_name
                in BLOCK_4_PATHWAY_DIMENSIONS
            )
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_4_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 Block 4 parameter accounting is invalid."
    )


# =============================================================================
# Downstream recurrent-state mutation audit
# =============================================================================
#
# A clean first run should have no recurrent objects yet.
# A notebook rerun may already contain them from Blocks 5+.
#
# The invariant for Block 4 is:
#   - create no new recurrent objects;
#   - replace no pre-existing recurrent objects;
#   - leave all pre-existing recurrent objects untouched.
# =============================================================================

BLOCK_4_RECURRENT_OBJECTS_PRESENT_AFTER = {
    object_name:
        globals()[
            object_name
        ]

    for object_name
    in BLOCK_4_DOWNSTREAM_RECURRENT_OBJECT_NAMES

    if object_name in globals()
}


BLOCK_4_NEW_RECURRENT_OBJECTS_CREATED = tuple(
    object_name

    for object_name
    in BLOCK_4_RECURRENT_OBJECTS_PRESENT_AFTER

    if object_name not in BLOCK_4_PREEXISTING_RECURRENT_OBJECTS
)


BLOCK_4_PREEXISTING_RECURRENT_OBJECTS_PRESERVED = all(
    (
        object_name
        in
        BLOCK_4_RECURRENT_OBJECTS_PRESENT_AFTER
    )
    and
    (
        id(
            BLOCK_4_RECURRENT_OBJECTS_PRESENT_AFTER[
                object_name
            ]
        )
        ==
        BLOCK_4_PREEXISTING_RECURRENT_OBJECT_IDS[
            object_name
        ]
    )

    for object_name
    in BLOCK_4_PREEXISTING_RECURRENT_OBJECTS
)


BLOCK_4_RECURRENT_OBJECTS_NOT_CREATED_BY_BLOCK_4 = (
    len(
        BLOCK_4_NEW_RECURRENT_OBJECTS_CREATED
    )
    ==
    0
)


BLOCK_4_RECURRENT_RUNTIME_BOUNDARY_VALID = all(
    [
        BLOCK_4_RECURRENT_OBJECTS_NOT_CREATED_BY_BLOCK_4,
        BLOCK_4_PREEXISTING_RECURRENT_OBJECTS_PRESERVED,
    ]
)


# Compatibility diagnostic only.
BLOCK_4_RECURRENT_OBJECTS_ABSENT = (
    len(
        BLOCK_4_RECURRENT_OBJECTS_PRESENT_AFTER
    )
    ==
    0
)


if not BLOCK_4_RECURRENT_RUNTIME_BOUNDARY_VALID:

    raise RuntimeError(
        "Notebook 09 Block 4 violated the downstream recurrent-state "
        "execution boundary. "
        f"New objects: {list(BLOCK_4_NEW_RECURRENT_OBJECTS_CREATED)}"
    )


# =============================================================================
# Explicit execution boundary
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED = True

NOTEBOOK_09_RECURRENT_STATE_CREATED = False

NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED = False

NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_4_LOSS_CALCULATED = False

NOTEBOOK_09_BLOCK_4_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_4_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_4_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_09_BLOCK_4_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_4_TRAINING_EXECUTED = False


# =============================================================================
# Expose validated effective weights
# =============================================================================
#
# These are detached validation outputs only.
# They are not recurrent states and are not persisted as trainable model
# parameters.
# =============================================================================

NOTEBOOK_09_VALIDATED_INITIAL_TRANSFORMATIVE_WEIGHTS = {
    pathway_name:
        output.clone()

    for (
        pathway_name,
        output,
    ) in BLOCK_4_WEIGHT_OUTPUTS_FIRST.items()
}


NOTEBOOK_09_FACTUAL_INITIAL_EFFECTIVE_TRANSFORMATIVE_WEIGHT = (
    NOTEBOOK_09_VALIDATED_INITIAL_TRANSFORMATIVE_WEIGHTS.get(
        "factual"
    )
)


NOTEBOOK_09_PSYCHOLOGICAL_INITIAL_EFFECTIVE_TRANSFORMATIVE_WEIGHT = (
    NOTEBOOK_09_VALIDATED_INITIAL_TRANSFORMATIVE_WEIGHTS.get(
        "psychological"
    )
)


NOTEBOOK_09_SOCIAL_INITIAL_EFFECTIVE_TRANSFORMATIVE_WEIGHT = (
    NOTEBOOK_09_VALIDATED_INITIAL_TRANSFORMATIVE_WEIGHTS.get(
        "social"
    )
)


# =============================================================================
# Final Block 4 validation
# =============================================================================

NOTEBOOK_09_BLOCK_4_ERRORS = []


BLOCK_4_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_4_PREREQUISITES_VALID,

    "pathway_contract_invalid":
        BLOCK_4_PATHWAY_CONTRACT_VALID,

    "output_shapes_invalid":
        BLOCK_4_OUTPUT_SHAPES_VALID,

    "outputs_not_finite":
        BLOCK_4_ALL_OUTPUTS_FINITE,

    "outputs_not_bounded":
        BLOCK_4_ALL_OUTPUTS_BOUNDED,

    "active_values_invalid":
        BLOCK_4_ALL_ACTIVE_VALUES_VALID,

    "active_values_not_in_open_interval":
        BLOCK_4_ALL_ACTIVE_VALUES_STRICTLY_INTERIOR,

    "deferred_values_invalid":
        BLOCK_4_ALL_DEFERRED_VALUES_VALID,

    "semantic_index_layout_invalid":
        BLOCK_4_ALL_SEMANTIC_INDEX_LAYOUT_VALID,

    "forward_not_deterministic":
        BLOCK_4_ALL_DETERMINISTIC,

    "value_counts_invalid":
        BLOCK_4_VALUE_COUNTS_VALID,

    "gradient_scope_invalid":
        BLOCK_4_GRADIENT_SCOPE_VALID,

    "unexpected_gradients":
        BLOCK_4_WEIGHT_GRADIENTS_ABSENT,

    "parameter_objects_changed":
        BLOCK_4_PARAMETER_OBJECTS_STABLE,

    "parameter_sets_not_independent":
        BLOCK_4_PARAMETER_SETS_INDEPENDENT,

    "representation_state_changed":
        BLOCK_4_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_4_TRANSFORMATIVE_UNCHANGED,

    "weight_state_changed":
        BLOCK_4_WEIGHT_STATE_UNCHANGED,

    "inherited_modules_not_eval":
        BLOCK_4_INHERITED_MODULES_STILL_EVAL,

    "weight_parameter_device_invalid":
        BLOCK_4_WEIGHT_PARAMETER_DEVICE_VALID,

    "weight_buffer_device_invalid":
        BLOCK_4_WEIGHT_BUFFER_DEVICE_VALID,

    "parameter_accounting_invalid":
        BLOCK_4_PARAMETER_ACCOUNTING_VALID,

    "recurrent_runtime_boundary_invalid":
        BLOCK_4_RECURRENT_RUNTIME_BOUNDARY_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_4_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_4_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation audit
# =============================================================================

BLOCK_4_PROHIBITED_OPERATIONS = {
    "representation_forward_incorrectly_executed":
        NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED,

    "transformative_forward_incorrectly_executed":
        NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED,

    "recurrent_state_incorrectly_created":
        NOTEBOOK_09_RECURRENT_STATE_CREATED,

    "recurrent_update_incorrectly_executed":
        NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED,

    "loss_incorrectly_calculated":
        NOTEBOOK_09_BLOCK_4_LOSS_CALCULATED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_4_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_4_BACKWARD_PASS_EXECUTED,

    "gradient_clipping_incorrectly_executed":
        NOTEBOOK_09_BLOCK_4_GRADIENT_CLIPPING_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_4_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_4_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_4_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_4_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 4 state
# =============================================================================

NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID = (
    len(
        NOTEBOOK_09_BLOCK_4_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_RECURRENT_ARCHITECTURE_READY = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID
)


NOTEBOOK_09_BLOCK_4_VALID = (
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID
)


if not NOTEBOOK_09_BLOCK_4_VALID:

    raise RuntimeError(
        "Notebook 09 Block 4 Transformative-Weight forward-path "
        "validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_4_ERRORS}"
    )


NOTEBOOK_09_BLOCK_4_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_4_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_4,

    "block_name":
        NOTEBOOK_09_BLOCK_4_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_4_VERSION,

    "output_shapes":
        deepcopy(
            BLOCK_4_OUTPUT_SHAPES
        ),

    "active_value_counts":
        deepcopy(
            BLOCK_4_ACTIVE_VALUE_COUNTS
        ),

    "deferred_value_counts":
        deepcopy(
            BLOCK_4_DEFERRED_VALUE_COUNTS
        ),

    "all_outputs_finite":
        BLOCK_4_ALL_OUTPUTS_FINITE,

    "all_outputs_bounded":
        BLOCK_4_ALL_OUTPUTS_BOUNDED,

    "all_active_values_valid":
        BLOCK_4_ALL_ACTIVE_VALUES_VALID,

    "all_deferred_values_valid":
        BLOCK_4_ALL_DEFERRED_VALUES_VALID,

    "semantic_index_layout_valid":
        BLOCK_4_ALL_SEMANTIC_INDEX_LAYOUT_VALID,

    "deterministic":
        BLOCK_4_ALL_DETERMINISTIC,

    "weight_state_unchanged":
        BLOCK_4_WEIGHT_STATE_UNCHANGED,

    "representation_state_unchanged":
        BLOCK_4_REPRESENTATION_UNCHANGED,

    "transformative_state_unchanged":
        BLOCK_4_TRANSFORMATIVE_UNCHANGED,

    "parameter_sets_independent":
        BLOCK_4_PARAMETER_SETS_INDEPENDENT,

    "weight_forward_executed":
        NOTEBOOK_09_BLOCK_4_WEIGHT_FORWARD_EXECUTED,

    "recurrent_state_created":
        NOTEBOOK_09_RECURRENT_STATE_CREATED,

    "recurrent_update_executed":
        NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED,

    "rerun_detected":
        BLOCK_4_RERUN_DETECTED,

    "preexisting_recurrent_object_count":
        len(
            BLOCK_4_PREEXISTING_RECURRENT_OBJECTS
        ),

    "new_recurrent_objects_created":
        tuple(
            BLOCK_4_NEW_RECURRENT_OBJECTS_CREATED
        ),

    "preexisting_recurrent_objects_preserved":
        BLOCK_4_PREEXISTING_RECURRENT_OBJECTS_PRESERVED,

    "forward_path_valid":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID,

    "recurrent_architecture_ready":
        NOTEBOOK_09_RECURRENT_ARCHITECTURE_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_4_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_4_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 4: "
    "Controlled Transformative-Weight Forward-Path Validation"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_4_VERSION}"
)

print("-" * 72)

print(
    "Transformative-Weight forward outputs"
)

for pathway_name in BLOCK_4_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} output shape       : "
        f"{BLOCK_4_OUTPUT_SHAPES[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} finite             : "
        f"{BLOCK_4_OUTPUTS_FINITE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} bounded            : "
        f"{BLOCK_4_OUTPUTS_BOUNDED[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deterministic      : "
        f"{BLOCK_4_DETERMINISTIC[pathway_name]}"
    )

print("-" * 72)

print(
    "Active Transformative-Weight validation"
)

for pathway_name in BLOCK_4_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} active indices     : "
        f"{list(BLOCK_4_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} active count       : "
        f"{BLOCK_4_ACTIVE_VALUE_COUNTS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} active value       : "
        f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE:.1f}"
    )

    print(
        f"{pathway_name:<14} active values valid: "
        f"{BLOCK_4_ACTIVE_VALUES_VALID[pathway_name]}"
    )

print("-" * 72)

print(
    "Deferred Transformative-Weight validation"
)

for pathway_name in BLOCK_4_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} deferred indices   : "
        f"{list(BLOCK_4_DEFERRED_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} deferred count     : "
        f"{BLOCK_4_DEFERRED_VALUE_COUNTS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deferred value     : "
        f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE:.1f}"
    )

    print(
        f"{pathway_name:<14} deferred valid     : "
        f"{BLOCK_4_DEFERRED_VALUES_VALID[pathway_name]}"
    )

print("-" * 72)

print(
    "Parameter and state validation"
)

print(
    f"Structural weight dimensions : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS}"
)

print(
    f"Trainable weight parameters   : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Deferred dimensions           : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT}"
)

print(
    f"Total model parameters        : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print(
    f"Weight parameters trainable   : "
    f"{BLOCK_4_WEIGHT_PARAMETERS_TRAINABLE}"
)

print(
    f"Weight gradients absent       : "
    f"{BLOCK_4_WEIGHT_GRADIENTS_ABSENT}"
)

print(
    f"Parameter sets independent    : "
    f"{BLOCK_4_PARAMETER_SETS_INDEPENDENT}"
)

print(
    f"Weight state unchanged        : "
    f"{BLOCK_4_WEIGHT_STATE_UNCHANGED}"
)

print(
    f"Representation unchanged      : "
    f"{BLOCK_4_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged      : "
    f"{BLOCK_4_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Representation frozen         : "
    f"{BLOCK_4_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen         : "
    f"{BLOCK_4_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"Inherited modules eval        : "
    f"{BLOCK_4_INHERITED_MODULES_STILL_EVAL}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Transform weights created     : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHTS_CREATED}"
)

print(
    f"Weight forward executed       : "
    f"{NOTEBOOK_09_BLOCK_4_WEIGHT_FORWARD_EXECUTED}"
)

print(
    f"Representation forward        : "
    f"{NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED}"
)

print(
    f"Transform forward             : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED}"
)

print(
    f"Recurrent state created       : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CREATED}"
)

print(
    f"Recurrent update executed     : "
    f"{NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"Loss calculated               : "
    f"{NOTEBOOK_09_BLOCK_4_LOSS_CALCULATED}"
)

print(
    f"Optimizer created             : "
    f"{NOTEBOOK_09_BLOCK_4_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed        : "
    f"{NOTEBOOK_09_BLOCK_4_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed     : "
    f"{NOTEBOOK_09_BLOCK_4_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Notebook rerun detected       : "
    f"{BLOCK_4_RERUN_DETECTED}"
)

print(
    f"Pre-existing recurrent objs   : "
    f"{len(BLOCK_4_PREEXISTING_RECURRENT_OBJECTS)}"
)

print(
    f"New recurrent objects created : "
    f"{len(BLOCK_4_NEW_RECURRENT_OBJECTS_CREATED)}"
)

print(
    f"Existing recurrent objs kept  : "
    f"{BLOCK_4_PREEXISTING_RECURRENT_OBJECTS_PRESERVED}"
)

print("-" * 72)

print(
    f"Semantic index layout valid   : "
    f"{BLOCK_4_ALL_SEMANTIC_INDEX_LAYOUT_VALID}"
)

print(
    f"Forward path valid            : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID}"
)

print(
    f"Recurrent architecture ready  : "
    f"{NOTEBOOK_09_RECURRENT_ARCHITECTURE_READY}"
)

print(
    f"Block valid                   : "
    f"{NOTEBOOK_09_BLOCK_4_VALID}"
)

print(
    f"Block complete                : "
    f"{NOTEBOOK_09_BLOCK_4_COMPLETE}"
)

print("=" * 72)

print(
    f"{len(BLOCK_4_PATHWAY_DIMENSIONS)} Transformative-Weight pathway "
    "forward path(s) were validated successfully."
)

print(
    "Each module returned a complete pathway-specific effective weight "
    "vector with the required dimensionality."
)

print(
    f"All {sum(BLOCK_4_ACTIVE_VALUE_COUNTS.values())} active dimension(s) "
    f"produced the expected initial effective weight of "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE:.1f} "
    "at the correct inherited semantic indices."
)

print(
    f"All {sum(BLOCK_4_DEFERRED_VALUE_COUNTS.values())} deferred "
    "dimension(s) remained exactly zero under the current gating contract."
)

print(
    "Repeated Transformative-Weight evaluations were exactly deterministic, "
    "finite and bounded."
)

print(
    f"All {NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT} latent "
    "Transformative-Weight parameter(s) remain trainable, while the inherited "
    f"{NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} representation and "
    "transformative parameters remain frozen and exactly unchanged."
)

print(
    "Block 4 executed no representation forward pass, candidate-transformative "
    "forward pass, recurrent-state creation, recurrent update, loss, optimiser, "
    "backward pass or parameter update."
)

if BLOCK_4_RERUN_DETECTED:

    print(
        "Pre-existing recurrent runtime objects from a later notebook stage "
        "were detected and left unchanged; they are not attributed to Block 4."
    )

print(
    "Notebook 09 may now proceed to explicit recurrent-state architectural "
    "definition and deterministic initial-state construction."
)

print("=" * 72)

Media AI — Notebook 09, Block 4: Controlled Transformative-Weight Forward-Path Validation
Block version                : 1.2
------------------------------------------------------------------------
Transformative-Weight forward outputs
factual        output shape       : (10,)
factual        finite             : True
factual        bounded            : True
factual        deterministic      : True
psychological  output shape       : (34,)
psychological  finite             : True
psychological  bounded            : True
psychological  deterministic      : True
social         output shape       : (7,)
social         finite             : True
social         bounded            : True
social         deterministic      : True
------------------------------------------------------------------------
Active Transformative-Weight validation
factual        active indices     : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
factual        active count       : 10
factual        active value       : 0.5
factual    

## Block 5 — Recurrent-State Architectural Contract and Deterministic Initial-State Construction

This block introduces the recurrent-state component of Notebook 09.

Blocks 1–4 established and validated the complete inherited architecture together with the new Transformative-Weight mechanism. The representation model and the three trained transformative mechanisms remain frozen and unchanged. The Transformative-Weight modules are available and have been validated independently, with 25 activation-eligible latent parameters controlling the full 51-dimensional factual, psychological, and social weighting space.

The present block now defines the recurrent-state architecture and constructs the deterministic initial state for each pathway.

Its purpose is to establish the state spaces, causal update contract, dimensional invariants, article-boundary reset policy, and initial-state behaviour before any recurrent transition is executed.

No sentence sequence is traversed in this block. No candidate transformation is generated. No weighted transformation is applied to a recurrent state. No recurrent update, loss calculation, optimiser construction, backward pass, or parameter update is executed.

### Architectural role of recurrent state

The recurrent state represents the accumulated pathway-specific effect carried forward through an ordered sequence of sentences.

For each pathway \(k\in\{F,P,S\}\), the state after processing sentence position \(t\) is denoted

$$
h_t^{(k)}.
$$

The three state spaces preserve the dimensionality of their corresponding representation and candidate-transformation spaces:

$$
h_t^{(F)}\in\mathbb{R}^{10},
$$

$$
h_t^{(P)}\in\mathbb{R}^{34},
$$

$$
h_t^{(S)}\in\mathbb{R}^{7}.
$$

The recurrent state is therefore not a new compressed representation and does not introduce a fourth confluent space.

It remains pathway-specific and dimensionally aligned with the existing factual, psychological, and social architecture.

### Relationship to the candidate transformation

The candidate transformation produced by the inherited transformative mechanism is

$$
c_t^{(k)}
=
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right),
$$

where

$$
r_t^{(k)}\in\mathbb{R}^{d_k}
$$

is the current pathway representation and

$$
h_{t-1}^{(k)}\in\mathbb{R}^{d_k}
$$

is the preceding recurrent state.

The Transformative Weight then determines how strongly the candidate transformation contributes to state evolution:

$$
\Delta h_t^{(k)}
=
TW_k\odot c_t^{(k)}.
$$

The recurrent state update is defined conceptually as

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
\Delta h_t^{(k)}.
$$

Equivalently,

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}.
$$

This additive update is the initial recurrent-state contract adopted by Notebook 09.

Block 5 defines this relation but does not execute it.

### Why the state is additive

The candidate transformation learned in Notebook 08 was derived as a signed change relative to a preceding state condition.

The recurrent mechanism therefore interprets the weighted candidate as an incremental state contribution rather than as a complete replacement state.

The update

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}
$$

preserves this interpretation directly.

The preceding state remains explicitly represented, while the candidate transformation contributes only the weighted increment.

This is methodologically different from a replacement rule such as

$$
h_t^{(k)}
=
TW_k\odot c_t^{(k)},
$$

which would discard the accumulated preceding state.

The additive contract therefore provides the clearest initial implementation of recurrent accumulation.

### Deterministic initial state

Every article sequence requires a state before its first sentence is processed.

No preceding article sentence exists at position \(t=1\), so the recurrent architecture requires an explicit initial condition.

Notebook 09 adopts the deterministic zero initial-state policy already anticipated by the upstream transformation-target construction:

$$
h_0^{(F)}
=
\mathbf{0}_{10},
$$

$$
h_0^{(P)}
=
\mathbf{0}_{34},
$$

$$
h_0^{(S)}
=
\mathbf{0}_{7}.
$$

The corresponding dimensions are therefore

$$
\dim
\left(
h_0^{(F)}
\right)
=
10,
$$

$$
\dim
\left(
h_0^{(P)}
\right)
=
34,
$$

$$
\dim
\left(
h_0^{(S)}
\right)
=
7.
$$

These initial states are deterministic architectural conditions.

They are not learned parameters, supervision labels, candidate transformations, or Transformative Weights.

### Initial state is not a model parameter

The initial recurrent state contains no trainable parameter.

Formally,

$$
h_0^{(k)}
\notin
\theta_R,
$$

$$
h_0^{(k)}
\notin
\theta_T,
$$

and

$$
h_0^{(k)}
\notin
\theta_{TW}.
$$

It is a runtime state tensor constructed from the recurrent-state contract.

Therefore, the model parameter count remains unchanged when the initial states are created.

After Block 3, the complete parameter count is

$$
902{,}190+25=902{,}215.
$$

Construction of the recurrent initial states does not alter this value:

$$
|\theta_{\mathrm{total}}|
=
902{,}215.
$$

This distinction is important because recurrent state evolves during sequence execution, while model parameters are persistent learned quantities.

### Article-boundary reset policy

Recurrent state is local to an article sequence.

For every new article \(a\), processing begins from the deterministic zero condition

$$
h_{0,a}^{(F)}
=
\mathbf{0}_{10},
$$

$$
h_{0,a}^{(P)}
=
\mathbf{0}_{34},
$$

$$
h_{0,a}^{(S)}
=
\mathbf{0}_{7}.
$$

The final state of one article must not become the initial state of an unrelated article.

Therefore,

$$
h_{T_a,a}^{(k)}
\not\rightarrow
h_{0,a+1}^{(k)}.
$$

Instead,

$$
h_{0,a+1}^{(k)}
=
\mathbf{0}_{d_k}.
$$

This prevents information leakage across article boundaries and preserves the interpretation of recurrent state as within-article accumulated transformation.

### Causal sequence contract

Within an article, recurrent state evolves strictly according to canonical sentence order.

For an article containing

$$
s_1,s_2,\ldots,s_T,
$$

the state sequence is

$$
h_0
\rightarrow
h_1
\rightarrow
h_2
\rightarrow
\cdots
\rightarrow
h_T.
$$

At sentence position \(t\), the architecture may depend on

$$
r_t^{(k)}
$$

and

$$
h_{t-1}^{(k)},
$$

but never on

$$
h_{t+1}^{(k)}
$$

or any future representation

$$
r_{t+1}^{(k)},
r_{t+2}^{(k)},
\ldots.
$$

The causal restriction is therefore

$$
h_t^{(k)}
=
f
\left(
r_1^{(k)},
\ldots,
r_t^{(k)}
\right),
$$

with no dependence on

$$
r_{t+1}^{(k)},
\ldots,
r_T^{(k)}.
$$

This is a strict architectural requirement.

### Pathway independence

The factual, psychological, and social recurrent states remain independent under the current Notebook 09 contract.

Their updates are therefore defined separately:

$$
h_t^{(F)}
=
h_{t-1}^{(F)}
+
TW_F\odot c_t^{(F)},
$$

$$
h_t^{(P)}
=
h_{t-1}^{(P)}
+
TW_P\odot c_t^{(P)},
$$

$$
h_t^{(S)}
=
h_{t-1}^{(S)}
+
TW_S\odot c_t^{(S)}.
$$

No recurrent cross-pathway term is introduced.

In particular, the present architecture does not include relations such as

$$
h_t^{(F)}
=
f
\left(
h_{t-1}^{(P)}
\right),
$$

or

$$
h_t^{(P)}
=
f
\left(
h_{t-1}^{(S)}
\right).
$$

Cross-pathway recurrent interaction remains outside the current contract unless introduced and validated explicitly in a later architectural stage.

### Deferred dimensions remain state dimensions

The active/deferred Transformative-Weight policy does not reduce the recurrent-state spaces.

The factual state remains

$$
h_t^{(F)}
\in
\mathbb{R}^{10},
$$

not

$$
\mathbb{R}^{5}.
$$

The psychological state remains

$$
h_t^{(P)}
\in
\mathbb{R}^{34},
$$

not

$$
\mathbb{R}^{17}.
$$

The social state remains

$$
h_t^{(S)}
\in
\mathbb{R}^{7},
$$

not

$$
\mathbb{R}^{3}.
$$

Deferred dimensions are structurally retained.

Under the current Transformative-Weight policy,

$$
TW_{k,j}=0
\qquad
\text{for }j\in D_k.
$$

Therefore, the weighted recurrent contribution on those dimensions is

$$
\Delta h_{t,j}^{(k)}
=
0.
$$

Under the additive update rule,

$$
h_{t,j}^{(k)}
=
h_{t-1,j}^{(k)}
$$

for deferred dimensions during the current pilot contract.

This preserves their state positions without introducing unsupported transformation behaviour.

### Distinction between recurrent state and supervision

The recurrent state is a model-runtime quantity.

It is not itself a supervision target.

The upstream factual, psychological, and social supervision was used to train the representation and candidate-transformative mechanisms and to establish eligible transformation dimensions.

The recurrent state is instead produced through sequential application of the validated architecture.

Therefore,

$$
h_t^{(k)}
\neq
y_t^{(k)},
$$

where \(y_t^{(k)}\) denotes a supervision target.

Likewise,

$$
h_t^{(k)}
\neq
c_t^{(k)},
$$

and

$$
h_t^{(k)}
\neq
TW_k.
$$

These three quantities serve different roles:

- \(c_t^{(k)}\) is a learned candidate transformation;
- \(TW_k\) controls transformation admission;
- \(h_t^{(k)}\) stores accumulated sequence state.

### Distinction between zero initial state and zero supervision

The deterministic zero initial condition must not be confused with observed zero supervision.

The initial state

$$
h_0^{(k)}
=
\mathbf{0}_{d_k}
$$

exists because no preceding sentence state is available.

It does not assert that the first sentence has zero factual, psychological, or social effect.

The first sentence may still produce a nonzero candidate transformation

$$
c_1^{(k)}
$$

and, later, a nonzero state

$$
h_1^{(k)}.
$$

The zero initial state is therefore an architectural boundary condition rather than an empirical observation.

### Runtime tensor requirements

The three initial recurrent states must be constructed on the same execution device as the corresponding model pathway.

Their required shapes are

$$
(1,10),
$$

$$
(1,34),
$$

and

$$
(1,7),
$$

when represented with an explicit batch dimension.

Using an explicit leading batch dimension preserves compatibility with the inherited transformative mechanisms, which operate on batched tensors.

The initial states must also use the same floating-point dtype as the model execution contract.

Therefore,

$$
h_0^{(k)}
\in
\mathbb{R}^{1\times d_k}
$$

with consistent device and dtype placement.

### Deterministic initial-state validation

Construction of the initial state must be deterministic.

Repeated construction under the same pathway contract must satisfy

$$
h_{0,1}^{(k)}
=
h_{0,2}^{(k)}.
$$

Every element must be finite:

$$
\operatorname{isfinite}
\left(
h_{0,j}^{(k)}
\right)
=
\mathrm{True}.
$$

Every element must equal exactly zero:

$$
h_{0,j}^{(k)}
=
0.
$$

The initial-state tensors must not require gradients:

$$
\texttt{requires\_grad}
\left(
h_0^{(k)}
\right)
=
\mathrm{False}.
$$

This ensures that they remain runtime state rather than learnable parameters.

### Recurrent-state mutability policy

Although the initial state is constructed in this block, Block 5 does not mutate it through a recurrent transition.

The initial state remains

$$
h_0^{(k)}
$$

throughout Block 5.

No object is yet promoted to

$$
h_1^{(k)}.
$$

This allows the recurrent-state construction itself to be validated independently from recurrent execution.

The first actual transition belongs to a subsequent controlled forward-path block.

### Inherited architecture remains unchanged

The construction of recurrent initial-state tensors must not modify any learned model parameter.

The representation state remains

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}},
$$

the inherited transformative state remains

$$
\theta_T^{\mathrm{after}}
=
\theta_T^{\mathrm{before}},
$$

and the new Transformative-Weight parameters remain

$$
\theta_{TW}^{\mathrm{after}}
=
\theta_{TW}^{\mathrm{before}}.
$$

The parameter counts therefore remain

$$
|\theta_R|
=
894{,}003,
$$

$$
|\theta_T|
=
8{,}187,
$$

$$
|\theta_{TW}|
=
25,
$$

and

$$
|\theta_{\mathrm{total}}|
=
902{,}215.
$$

### No recurrent update in Block 5

Although the recurrent-state objects are now constructed, this block does not yet evaluate the full transition

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}.
$$

Therefore:

- no representation forward pass is executed;
- no candidate-transformative forward pass is executed;
- no Transformative-Weight application to a candidate transformation is executed;
- no state transition from \(h_0\) to \(h_1\) occurs;
- no article sequence is traversed.

Block 5 validates only the recurrent architecture and its deterministic starting condition.

### No optimisation in Block 5

The recurrent-state construction introduces no trainable parameter.

Accordingly:

- no optimiser is created;
- no loss is calculated;
- no backward pass is executed;
- no gradient clipping is applied;
- no optimiser step is executed;
- and no parameter update occurs.

The 25 Transformative-Weight latent parameters remain trainable but unchanged.

### Block 5 completion contract

Block 5 is complete only when:

- factual, psychological, and social recurrent-state spaces are defined with dimensions 10, 34, and 7;
- the additive recurrent update contract is established explicitly;
- the recurrent architecture preserves pathway independence;
- deterministic zero initial states are constructed for all three pathways;
- the initial state shapes are exactly \((1,10)\), \((1,34)\), and \((1,7)\);
- all initial-state values are finite;
- all initial-state values are exactly zero;
- repeated initial-state construction is exactly deterministic;
- the initial-state tensors do not require gradients;
- the initial states are placed on the configured execution device;
- the initial states use the required model dtype;
- the state spaces preserve all deferred dimensions;
- the article-boundary reset policy is established;
- the causal no-future-context contract is preserved;
- no recurrent transition is executed;
- no \(h_1\) state is created;
- no representation forward pass is executed;
- no candidate-transformative forward pass is executed;
- no weighted candidate transformation is applied;
- the 902,190 inherited parameters remain frozen and exactly unchanged;
- the 25 Transformative-Weight parameters remain unchanged;
- the complete model parameter count remains 902,215;
- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- and no parameter update occurs.

Successful completion establishes the deterministic recurrent starting condition

$$
\boxed{
h_0^{(F)}
=
\mathbf{0}_{1\times10},
\qquad
h_0^{(P)}
=
\mathbf{0}_{1\times34},
\qquad
h_0^{(S)}
=
\mathbf{0}_{1\times7}
}
$$

together with the recurrent transition contract

$$
\boxed{
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}
}
$$

without yet executing that transition.

Notebook 09 may then proceed to controlled single-step recurrent forward-path validation, where validated candidate transformations, Transformative Weights, and recurrent state are combined for the first time.

In [55]:
# =============================================================================
# Media AI — Notebook 09
# Block 5: Recurrent-State Architectural Contract
#          and Deterministic Initial-State Construction
# =============================================================================

from copy import deepcopy

import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_5 = 5

NOTEBOOK_09_BLOCK_5_NAME = (
    "Recurrent-State Architectural Contract "
    "and Deterministic Initial-State Construction"
)

NOTEBOOK_09_BLOCK_5_VERSION = "1.2"


# =============================================================================
# Required inherited runtime contract
# =============================================================================

BLOCK_5_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_BLOCK_1_COMPLETE",
    "NOTEBOOK_09_BLOCK_1_VALID",
    "NOTEBOOK_09_HANDOVER_RESTORED",
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",
    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Block 2
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_2_COMPLETE",
    "NOTEBOOK_09_BLOCK_2_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID",

    # -------------------------------------------------------------------------
    # Block 3
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_3_COMPLETE",
    "NOTEBOOK_09_BLOCK_3_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",

    # -------------------------------------------------------------------------
    # Block 4
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_4_COMPLETE",
    "NOTEBOOK_09_BLOCK_4_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID",
    "NOTEBOOK_09_RECURRENT_ARCHITECTURE_READY",

    # -------------------------------------------------------------------------
    # Pathway contract
    # -------------------------------------------------------------------------
    "BLOCK_4_PATHWAY_DIMENSIONS",
    "BLOCK_4_ACTIVE_DIMENSION_INDICES",
    "BLOCK_4_DEFERRED_DIMENSION_INDICES",

    # -------------------------------------------------------------------------
    # Environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_5_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_5_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_5_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 5 prerequisites are not initialised. "
        f"Missing: {BLOCK_5_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_5_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,

        NOTEBOOK_09_BLOCK_1_COMPLETE is True,
        NOTEBOOK_09_BLOCK_1_VALID is True,
        NOTEBOOK_09_HANDOVER_RESTORED is True,

        NOTEBOOK_09_BLOCK_2_COMPLETE is True,
        NOTEBOOK_09_BLOCK_2_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_CONTRACT_VALID is True,

        NOTEBOOK_09_BLOCK_3_COMPLETE is True,
        NOTEBOOK_09_BLOCK_3_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULE_CONSTRUCTION_VALID is True,

        NOTEBOOK_09_BLOCK_4_COMPLETE is True,
        NOTEBOOK_09_BLOCK_4_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID is True,
        NOTEBOOK_09_RECURRENT_ARCHITECTURE_READY is True,
    ]
)


if not BLOCK_5_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Blocks 1–4 must be valid and complete "
        "before recurrent-state construction."
    )


# =============================================================================
# Downstream-runtime snapshot for rerun-safe execution
# =============================================================================
#
# Block 5 owns deterministic INITIAL recurrent-state construction.
# It may therefore replace its own prior zero-state objects on rerun.
#
# Updated recurrent states (T1 / H1 and later trajectory states) belong to
# Block 6 and later. They may remain in the interactive kernel from a previous
# execution and must not cause Block 5 to fail. Block 5 records them and later
# verifies that it neither creates nor replaces those downstream objects.
# =============================================================================

BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECT_NAMES = (
    "NOTEBOOK_09_INITIAL_RECURRENT_STATES",
    "NOTEBOOK_09_FACTUAL_INITIAL_STATE",
    "NOTEBOOK_09_PSYCHOLOGICAL_INITIAL_STATE",
    "NOTEBOOK_09_SOCIAL_INITIAL_STATE",
)


BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECTS = {
    object_name:
        globals()[
            object_name
        ]

    for object_name
    in BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECT_NAMES

    if object_name in globals()
}


BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECT_IDS = {
    object_name:
        id(
            object_value
        )

    for (
        object_name,
        object_value,
    ) in BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECTS.items()
}


BLOCK_5_DOWNSTREAM_UPDATED_STATE_OBJECT_NAMES = (
    "NOTEBOOK_09_FACTUAL_RECURRENT_STATE_T1",
    "NOTEBOOK_09_PSYCHOLOGICAL_RECURRENT_STATE_T1",
    "NOTEBOOK_09_SOCIAL_RECURRENT_STATE_T1",

    "NOTEBOOK_09_FACTUAL_H1",
    "NOTEBOOK_09_PSYCHOLOGICAL_H1",
    "NOTEBOOK_09_SOCIAL_H1",

    "NOTEBOOK_09_VALIDATED_SINGLE_STEP_RECURRENT_STATES",
    "NOTEBOOK_09_ARTICLE_FINAL_RECURRENT_STATES",
    "NOTEBOOK_09_POSTTRAIN_FINAL_RECURRENT_STATES",
)


BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS = {
    object_name:
        globals()[
            object_name
        ]

    for object_name
    in BLOCK_5_DOWNSTREAM_UPDATED_STATE_OBJECT_NAMES

    if object_name in globals()
}


BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECT_IDS = {
    object_name:
        id(
            object_value
        )

    for (
        object_name,
        object_value,
    ) in BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS.items()
}


BLOCK_5_RERUN_DETECTED = bool(
    BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECTS
    or
    BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS
)


# =============================================================================
# Canonical recurrent pathway contract
# =============================================================================

NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS = deepcopy(
    BLOCK_4_PATHWAY_DIMENSIONS
)


BLOCK_5_RECURRENT_STATE_DIMENSIONS_VALID = all(
    [
        isinstance(
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS,
            dict,
        ),

        bool(
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
            ==
            BLOCK_4_PATHWAY_DIMENSIONS
        ),

        all(
            isinstance(
                pathway_dim,
                int,
            )
            and
            pathway_dim > 0

            for pathway_dim
            in NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS.values()
        ),
    ]
)


if not BLOCK_5_RECURRENT_STATE_DIMENSIONS_VALID:

    raise RuntimeError(
        "Recurrent-state dimensions do not reproduce "
        "the validated pathway contract."
    )


# =============================================================================
# Active and deferred pathway inheritance
# =============================================================================

NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in BLOCK_4_ACTIVE_DIMENSION_INDICES.items()
}


NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in BLOCK_4_DEFERRED_DIMENSION_INDICES.items()
}


BLOCK_5_RECURRENT_DIMENSION_PARTITIONS_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS.items():

    active_indices = set(
        NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    deferred_indices = set(
        NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )


    expected_indices = set(
        range(
            pathway_dim
        )
    )


    BLOCK_5_RECURRENT_DIMENSION_PARTITIONS_VALID[
        pathway_name
    ] = all(
        [
            active_indices.isdisjoint(
                deferred_indices
            ),

            (
                active_indices
                |
                deferred_indices
            )
            ==
            expected_indices,
        ]
    )


BLOCK_5_ALL_RECURRENT_DIMENSION_PARTITIONS_VALID = all(
    BLOCK_5_RECURRENT_DIMENSION_PARTITIONS_VALID.values()
)


if not BLOCK_5_ALL_RECURRENT_DIMENSION_PARTITIONS_VALID:

    raise RuntimeError(
        "Active and deferred recurrent-state dimensions "
        "do not form complete pathway partitions."
    )


# =============================================================================
# Recurrent-state architectural contract
# =============================================================================

NOTEBOOK_09_RECURRENT_STATE_CONTRACT = {
    "state_granularity":
        "pathway_dimension_wise",

    "pathway_dimensions":
        deepcopy(
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
        ),

    "batch_dimension":
        1,

    "update_type":
        "additive",

    "weighted_contribution":
        "transformative_weight * candidate_transformation",

    "state_update":
        "previous_state + weighted_contribution",

    "initial_state_policy":
        "deterministic_zero",

    "article_boundary_reset":
        True,

    "future_context_allowed":
        False,

    "cross_pathway_recurrence":
        False,

    "initial_state_trainable":
        False,
}


BLOCK_5_RECURRENT_STATE_CONTRACT_VALID = all(
    [
        (
            NOTEBOOK_09_RECURRENT_STATE_CONTRACT[
                "state_granularity"
            ]
            ==
            "pathway_dimension_wise"
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_CONTRACT[
                "pathway_dimensions"
            ]
            ==
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_CONTRACT[
                "update_type"
            ]
            ==
            "additive"
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_CONTRACT[
                "initial_state_policy"
            ]
            ==
            "deterministic_zero"
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_CONTRACT[
                "article_boundary_reset"
            ]
            is True
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_CONTRACT[
                "future_context_allowed"
            ]
            is False
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_CONTRACT[
                "cross_pathway_recurrence"
            ]
            is False
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_CONTRACT[
                "initial_state_trainable"
            ]
            is False
        ),
    ]
)


if not BLOCK_5_RECURRENT_STATE_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 09 recurrent-state architectural contract is invalid."
    )


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_5_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_5_nested_state_exact(
    state_before,
    state_after,
):

    if state_before.keys() != state_after.keys():

        return False


    for module_name in state_before:

        before_module_state = (
            state_before[
                module_name
            ]
        )


        after_module_state = (
            state_after[
                module_name
            ]
        )


        if (
            before_module_state.keys()
            !=
            after_module_state.keys()
        ):

            return False


        for tensor_name in before_module_state:

            if not torch.equal(
                before_module_state[
                    tensor_name
                ],
                after_module_state[
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot learned parameter state before recurrent-state construction
# =============================================================================

BLOCK_5_REPRESENTATION_STATE_BEFORE = (
    block_5_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_5_TRANSFORMATIVE_STATE_BEFORE = (
    block_5_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_5_WEIGHT_STATE_BEFORE = (
    block_5_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Deterministic recurrent initial-state construction helper
# =============================================================================

def block_5_construct_initial_state(
    pathway_dim,
):
    """
    Construct a deterministic zero recurrent state.

    Shape:
        (1, pathway_dim)

    The tensor is runtime state only:
    - no gradient requirement;
    - no parameter registration;
    - no sequence transition.
    """

    return torch.zeros(
        (
            1,
            int(
                pathway_dim
            ),
        ),
        dtype=
            DEFAULT_DTYPE,
        device=
            DEVICE,
        requires_grad=
            False,
    )


# =============================================================================
# Construct recurrent initial states
# =============================================================================

NOTEBOOK_09_INITIAL_RECURRENT_STATES = {
    pathway_name:
        block_5_construct_initial_state(
            pathway_dim
        )

    for (
        pathway_name,
        pathway_dim,
    ) in NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS.items()
}


# Convenience aliases retained for the current canonical pathways.
NOTEBOOK_09_FACTUAL_INITIAL_STATE = (
    NOTEBOOK_09_INITIAL_RECURRENT_STATES.get(
        "factual"
    )
)


NOTEBOOK_09_PSYCHOLOGICAL_INITIAL_STATE = (
    NOTEBOOK_09_INITIAL_RECURRENT_STATES.get(
        "psychological"
    )
)


NOTEBOOK_09_SOCIAL_INITIAL_STATE = (
    NOTEBOOK_09_INITIAL_RECURRENT_STATES.get(
        "social"
    )
)


# =============================================================================
# Initial-state reconstruction audit
# =============================================================================
#
# On rerun, Block 5 deliberately reconstructs the zero initial-state objects
# it owns from the current Block 4 recurrent contract.
# =============================================================================

BLOCK_5_CURRENT_INITIAL_STATE_OBJECTS = {
    "NOTEBOOK_09_INITIAL_RECURRENT_STATES":
        NOTEBOOK_09_INITIAL_RECURRENT_STATES,

    "NOTEBOOK_09_FACTUAL_INITIAL_STATE":
        NOTEBOOK_09_FACTUAL_INITIAL_STATE,

    "NOTEBOOK_09_PSYCHOLOGICAL_INITIAL_STATE":
        NOTEBOOK_09_PSYCHOLOGICAL_INITIAL_STATE,

    "NOTEBOOK_09_SOCIAL_INITIAL_STATE":
        NOTEBOOK_09_SOCIAL_INITIAL_STATE,
}


BLOCK_5_INITIAL_STATE_OBJECTS_RECONSTRUCTED = all(
    (
        object_name
        in
        BLOCK_5_CURRENT_INITIAL_STATE_OBJECTS
    )
    and
    (
        object_name not in BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECT_IDS
        or
        id(
            BLOCK_5_CURRENT_INITIAL_STATE_OBJECTS[
                object_name
            ]
        )
        !=
        BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECT_IDS[
            object_name
        ]
    )

    for object_name
    in BLOCK_5_CURRENT_INITIAL_STATE_OBJECTS
)


if not BLOCK_5_INITIAL_STATE_OBJECTS_RECONSTRUCTED:

    raise RuntimeError(
        "Notebook 09 Block 5 failed to reconstruct its canonical "
        "initial recurrent-state objects."
    )


# =============================================================================
# Independent reconstruction for determinism validation
# =============================================================================

BLOCK_5_RECONSTRUCTED_INITIAL_STATES = {
    pathway_name:
        block_5_construct_initial_state(
            pathway_dim
        )

    for (
        pathway_name,
        pathway_dim,
    ) in NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS.items()
}


# =============================================================================
# Initial-state shape validation
# =============================================================================

BLOCK_5_INITIAL_STATE_SHAPES = {
    pathway_name:
        tuple(
            state.shape
        )

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


BLOCK_5_EXPECTED_INITIAL_STATE_SHAPES = {
    pathway_name:
        (
            1,
            pathway_dim,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS.items()
}


BLOCK_5_INITIAL_STATE_SHAPES_VALID = (
    BLOCK_5_INITIAL_STATE_SHAPES
    ==
    BLOCK_5_EXPECTED_INITIAL_STATE_SHAPES
)


if not BLOCK_5_INITIAL_STATE_SHAPES_VALID:

    raise RuntimeError(
        "Recurrent initial-state shapes are invalid."
    )


# =============================================================================
# Initial-state numerical validation
# =============================================================================

BLOCK_5_INITIAL_STATES_FINITE = {
    pathway_name:
        bool(
            torch.isfinite(
                state
            ).all().item()
        )

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


BLOCK_5_INITIAL_STATES_ZERO = {
    pathway_name:
        bool(
            torch.equal(
                state,
                torch.zeros_like(
                    state
                ),
            )
        )

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


BLOCK_5_ALL_INITIAL_STATES_FINITE = all(
    BLOCK_5_INITIAL_STATES_FINITE.values()
)


BLOCK_5_ALL_INITIAL_STATES_ZERO = all(
    BLOCK_5_INITIAL_STATES_ZERO.values()
)


if not all(
    [
        BLOCK_5_ALL_INITIAL_STATES_FINITE,
        BLOCK_5_ALL_INITIAL_STATES_ZERO,
    ]
):

    raise RuntimeError(
        "Recurrent initial states are not finite deterministic zeros."
    )


# =============================================================================
# Initial-state determinism validation
# =============================================================================

BLOCK_5_INITIAL_STATE_DETERMINISTIC = {
    pathway_name:
        torch.equal(
            NOTEBOOK_09_INITIAL_RECURRENT_STATES[
                pathway_name
            ],
            BLOCK_5_RECONSTRUCTED_INITIAL_STATES[
                pathway_name
            ],
        )

    for pathway_name
    in NOTEBOOK_09_INITIAL_RECURRENT_STATES
}


BLOCK_5_ALL_INITIAL_STATES_DETERMINISTIC = all(
    BLOCK_5_INITIAL_STATE_DETERMINISTIC.values()
)


if not BLOCK_5_ALL_INITIAL_STATES_DETERMINISTIC:

    raise RuntimeError(
        "Repeated recurrent initial-state construction is not deterministic."
    )


# =============================================================================
# Gradient-state validation
# =============================================================================

BLOCK_5_INITIAL_STATES_REQUIRE_GRAD = {
    pathway_name:
        bool(
            state.requires_grad
        )

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


BLOCK_5_INITIAL_STATES_NONTRAINABLE = all(
    not requires_grad

    for requires_grad
    in BLOCK_5_INITIAL_STATES_REQUIRE_GRAD.values()
)


if not BLOCK_5_INITIAL_STATES_NONTRAINABLE:

    raise RuntimeError(
        "Recurrent initial states unexpectedly require gradients."
    )


# =============================================================================
# Device validation
# =============================================================================

BLOCK_5_INITIAL_STATE_DEVICE_VALID = {
    pathway_name:
        (
            state.device
            ==
            DEVICE
        )

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


BLOCK_5_ALL_INITIAL_STATE_DEVICES_VALID = all(
    BLOCK_5_INITIAL_STATE_DEVICE_VALID.values()
)


if not BLOCK_5_ALL_INITIAL_STATE_DEVICES_VALID:

    raise RuntimeError(
        "One or more recurrent initial states are on "
        "an unexpected execution device."
    )


# =============================================================================
# Dtype validation
# =============================================================================

BLOCK_5_INITIAL_STATE_DTYPE_VALID = {
    pathway_name:
        (
            state.dtype
            ==
            DEFAULT_DTYPE
        )

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


BLOCK_5_ALL_INITIAL_STATE_DTYPES_VALID = all(
    BLOCK_5_INITIAL_STATE_DTYPE_VALID.values()
)


if not BLOCK_5_ALL_INITIAL_STATE_DTYPES_VALID:

    raise RuntimeError(
        "One or more recurrent initial states use "
        "an unexpected dtype."
    )


# =============================================================================
# Initial-state pathway independence
# =============================================================================

BLOCK_5_INITIAL_STATE_STORAGE_POINTERS = {
    pathway_name:
        int(
            state.data_ptr()
        )

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


BLOCK_5_INITIAL_STATE_STORAGE_INDEPENDENT = (
    len(
        set(
            BLOCK_5_INITIAL_STATE_STORAGE_POINTERS.values()
        )
    )
    ==
    len(
        NOTEBOOK_09_INITIAL_RECURRENT_STATES
    )
)


if not BLOCK_5_INITIAL_STATE_STORAGE_INDEPENDENT:

    raise RuntimeError(
        "Recurrent pathway initial states unexpectedly share storage."
    )


# =============================================================================
# Recurrent-state runtime contract
# =============================================================================

NOTEBOOK_09_RECURRENT_STATE_RUNTIME_POLICY = {
    "initial_state_shape_contract":
        {
            pathway_name:
                (
                    1,
                    pathway_dim,
                )

            for (
                pathway_name,
                pathway_dim,
            ) in NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS.items()
        },

    "initial_state_device":
        str(
            DEVICE
        ),

    "initial_state_dtype":
        str(
            DEFAULT_DTYPE
        ),

    "initial_state_requires_grad":
        False,

    "article_reset_each_sequence":
        True,

    "future_context_allowed":
        False,
}


BLOCK_5_RUNTIME_POLICY_VALID = all(
    [
        (
            NOTEBOOK_09_RECURRENT_STATE_RUNTIME_POLICY[
                "initial_state_requires_grad"
            ]
            is False
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_RUNTIME_POLICY[
                "article_reset_each_sequence"
            ]
            is True
        ),

        (
            NOTEBOOK_09_RECURRENT_STATE_RUNTIME_POLICY[
                "future_context_allowed"
            ]
            is False
        ),
    ]
)


if not BLOCK_5_RUNTIME_POLICY_VALID:

    raise RuntimeError(
        "Recurrent-state runtime policy is invalid."
    )


# =============================================================================
# Parameter-count validation
# =============================================================================

BLOCK_5_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS
            ==
            sum(
                NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS.values()
            )
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT
            ==
            sum(
                len(
                    NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
                        pathway_name
                    ]
                )

                for pathway_name
                in NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
            )
        ),
    ]
)


if not BLOCK_5_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 parameter accounting changed unexpectedly "
        "during recurrent-state construction."
    )


# =============================================================================
# Parameter trainability validation
# =============================================================================

BLOCK_5_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_5_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_5_WEIGHT_PARAMETERS_STILL_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_5_TRAINABILITY_SCOPE_VALID = all(
    [
        BLOCK_5_REPRESENTATION_STILL_FROZEN,
        BLOCK_5_TRANSFORMATIVE_STILL_FROZEN,
        BLOCK_5_WEIGHT_PARAMETERS_STILL_TRAINABLE,
        BLOCK_5_INITIAL_STATES_NONTRAINABLE,
    ]
)


if not BLOCK_5_TRAINABILITY_SCOPE_VALID:

    raise RuntimeError(
        "Notebook 09 trainability scope changed during "
        "recurrent-state construction."
    )


# =============================================================================
# Snapshot learned parameter state after recurrent-state construction
# =============================================================================

BLOCK_5_REPRESENTATION_STATE_AFTER = (
    block_5_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_5_TRANSFORMATIVE_STATE_AFTER = (
    block_5_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_5_WEIGHT_STATE_AFTER = (
    block_5_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Exact learned-state immutability validation
# =============================================================================

BLOCK_5_REPRESENTATION_UNCHANGED = (
    block_5_nested_state_exact(
        BLOCK_5_REPRESENTATION_STATE_BEFORE,
        BLOCK_5_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_5_TRANSFORMATIVE_UNCHANGED = (
    block_5_nested_state_exact(
        BLOCK_5_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_5_TRANSFORMATIVE_STATE_AFTER,
    )
)


BLOCK_5_WEIGHT_STATE_UNCHANGED = (
    block_5_nested_state_exact(
        BLOCK_5_WEIGHT_STATE_BEFORE,
        BLOCK_5_WEIGHT_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_5_REPRESENTATION_UNCHANGED,
        BLOCK_5_TRANSFORMATIVE_UNCHANGED,
        BLOCK_5_WEIGHT_STATE_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Learned model state changed during recurrent "
        "initial-state construction."
    )


# =============================================================================
# Inherited evaluation-mode validation
# =============================================================================

BLOCK_5_INHERITED_MODULES_STILL_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


if not BLOCK_5_INHERITED_MODULES_STILL_EVAL:

    raise RuntimeError(
        "Inherited modules unexpectedly left evaluation mode."
    )


# =============================================================================
# Explicit recurrent-state execution boundary
# =============================================================================

NOTEBOOK_09_RECURRENT_STATE_CREATED = True

NOTEBOOK_09_RECURRENT_INITIAL_STATE_CREATED = True

NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED = False

NOTEBOOK_09_RECURRENT_H1_CREATED = False

NOTEBOOK_09_WEIGHTED_CANDIDATE_TRANSFORMATION_EXECUTED = False

NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_5_WEIGHT_FORWARD_EXECUTED = False

NOTEBOOK_09_BLOCK_5_LOSS_CALCULATED = False

NOTEBOOK_09_BLOCK_5_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_5_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_5_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_09_BLOCK_5_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_5_TRAINING_EXECUTED = False


# =============================================================================
# Downstream updated-state mutation audit
# =============================================================================
#
# Block 5 must not create or replace T1/H1 or later recurrent outputs.
# Such objects may already exist from an earlier complete notebook execution.
# =============================================================================

BLOCK_5_UPDATED_STATE_OBJECTS_PRESENT_AFTER = {
    object_name:
        globals()[
            object_name
        ]

    for object_name
    in BLOCK_5_DOWNSTREAM_UPDATED_STATE_OBJECT_NAMES

    if object_name in globals()
}


BLOCK_5_NEW_UPDATED_STATE_OBJECTS_CREATED = tuple(
    object_name

    for object_name
    in BLOCK_5_UPDATED_STATE_OBJECTS_PRESENT_AFTER

    if object_name not in BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS
)


BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS_PRESERVED = all(
    (
        object_name
        in
        BLOCK_5_UPDATED_STATE_OBJECTS_PRESENT_AFTER
    )
    and
    (
        id(
            BLOCK_5_UPDATED_STATE_OBJECTS_PRESENT_AFTER[
                object_name
            ]
        )
        ==
        BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECT_IDS[
            object_name
        ]
    )

    for object_name
    in BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS
)


BLOCK_5_UPDATED_STATE_OBJECTS_NOT_CREATED_BY_BLOCK_5 = (
    len(
        BLOCK_5_NEW_UPDATED_STATE_OBJECTS_CREATED
    )
    ==
    0
)


BLOCK_5_UPDATED_STATE_RUNTIME_BOUNDARY_VALID = all(
    [
        BLOCK_5_UPDATED_STATE_OBJECTS_NOT_CREATED_BY_BLOCK_5,
        BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS_PRESERVED,
    ]
)


# Compatibility diagnostic only.
BLOCK_5_UPDATED_STATE_OBJECTS_ABSENT = (
    len(
        BLOCK_5_UPDATED_STATE_OBJECTS_PRESENT_AFTER
    )
    ==
    0
)


if not BLOCK_5_UPDATED_STATE_RUNTIME_BOUNDARY_VALID:

    raise RuntimeError(
        "Notebook 09 Block 5 violated the downstream updated-state "
        "execution boundary. "
        f"New objects: {list(BLOCK_5_NEW_UPDATED_STATE_OBJECTS_CREATED)}"
    )


# =============================================================================
# Expose canonical initial recurrent state
# =============================================================================

NOTEBOOK_09_RECURRENT_INITIAL_STATE_READY = True


NOTEBOOK_09_RECURRENT_INITIAL_STATE_CONTRACT = {
    pathway_name:
        {
            "shape":
                tuple(
                    state.shape
                ),

            "dimension":
                NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS[
                    pathway_name
                ],

            "all_zero":
                BLOCK_5_INITIAL_STATES_ZERO[
                    pathway_name
                ],
        }

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


# =============================================================================
# Final Block 5 validation
# =============================================================================

NOTEBOOK_09_BLOCK_5_ERRORS = []


BLOCK_5_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_5_PREREQUISITES_VALID,

    "recurrent_dimensions_invalid":
        BLOCK_5_RECURRENT_STATE_DIMENSIONS_VALID,

    "recurrent_dimension_partition_invalid":
        BLOCK_5_ALL_RECURRENT_DIMENSION_PARTITIONS_VALID,

    "recurrent_contract_invalid":
        BLOCK_5_RECURRENT_STATE_CONTRACT_VALID,

    "initial_state_shapes_invalid":
        BLOCK_5_INITIAL_STATE_SHAPES_VALID,

    "initial_states_not_finite":
        BLOCK_5_ALL_INITIAL_STATES_FINITE,

    "initial_states_not_zero":
        BLOCK_5_ALL_INITIAL_STATES_ZERO,

    "initial_state_not_deterministic":
        BLOCK_5_ALL_INITIAL_STATES_DETERMINISTIC,

    "initial_state_trainable":
        BLOCK_5_INITIAL_STATES_NONTRAINABLE,

    "initial_state_device_invalid":
        BLOCK_5_ALL_INITIAL_STATE_DEVICES_VALID,

    "initial_state_dtype_invalid":
        BLOCK_5_ALL_INITIAL_STATE_DTYPES_VALID,

    "initial_state_storage_not_independent":
        BLOCK_5_INITIAL_STATE_STORAGE_INDEPENDENT,

    "runtime_policy_invalid":
        BLOCK_5_RUNTIME_POLICY_VALID,

    "parameter_accounting_invalid":
        BLOCK_5_PARAMETER_ACCOUNTING_VALID,

    "trainability_scope_invalid":
        BLOCK_5_TRAINABILITY_SCOPE_VALID,

    "representation_state_changed":
        BLOCK_5_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_5_TRANSFORMATIVE_UNCHANGED,

    "weight_state_changed":
        BLOCK_5_WEIGHT_STATE_UNCHANGED,

    "inherited_modules_not_eval":
        BLOCK_5_INHERITED_MODULES_STILL_EVAL,

    "initial_state_objects_not_reconstructed":
        BLOCK_5_INITIAL_STATE_OBJECTS_RECONSTRUCTED,

    "updated_state_runtime_boundary_invalid":
        BLOCK_5_UPDATED_STATE_RUNTIME_BOUNDARY_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_5_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_5_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation audit
# =============================================================================

BLOCK_5_PROHIBITED_OPERATIONS = {
    "representation_forward_incorrectly_executed":
        NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED,

    "transformative_forward_incorrectly_executed":
        NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED,

    "weight_forward_incorrectly_executed":
        NOTEBOOK_09_BLOCK_5_WEIGHT_FORWARD_EXECUTED,

    "weighted_candidate_incorrectly_executed":
        NOTEBOOK_09_WEIGHTED_CANDIDATE_TRANSFORMATION_EXECUTED,

    "recurrent_update_incorrectly_executed":
        NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED,

    "h1_incorrectly_created":
        NOTEBOOK_09_RECURRENT_H1_CREATED,

    "loss_incorrectly_calculated":
        NOTEBOOK_09_BLOCK_5_LOSS_CALCULATED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_5_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_5_BACKWARD_PASS_EXECUTED,

    "gradient_clipping_incorrectly_executed":
        NOTEBOOK_09_BLOCK_5_GRADIENT_CLIPPING_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_5_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_5_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_5_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_5_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 5 state
# =============================================================================

NOTEBOOK_09_RECURRENT_INITIAL_STATE_VALID = (
    len(
        NOTEBOOK_09_BLOCK_5_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_RECURRENT_SINGLE_STEP_VALIDATION_READY = (
    NOTEBOOK_09_RECURRENT_INITIAL_STATE_VALID
)


NOTEBOOK_09_BLOCK_5_VALID = (
    NOTEBOOK_09_RECURRENT_INITIAL_STATE_VALID
)


if not NOTEBOOK_09_BLOCK_5_VALID:

    raise RuntimeError(
        "Notebook 09 Block 5 recurrent-state architectural "
        "validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_5_ERRORS}"
    )


NOTEBOOK_09_BLOCK_5_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_5_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_5,

    "block_name":
        NOTEBOOK_09_BLOCK_5_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_5_VERSION,

    "recurrent_state_dimensions":
        deepcopy(
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
        ),

    "initial_state_shapes":
        deepcopy(
            BLOCK_5_INITIAL_STATE_SHAPES
        ),

    "initial_states_finite":
        BLOCK_5_ALL_INITIAL_STATES_FINITE,

    "initial_states_zero":
        BLOCK_5_ALL_INITIAL_STATES_ZERO,

    "initial_states_deterministic":
        BLOCK_5_ALL_INITIAL_STATES_DETERMINISTIC,

    "initial_states_nontrainable":
        BLOCK_5_INITIAL_STATES_NONTRAINABLE,

    "article_boundary_reset":
        True,

    "future_context_allowed":
        False,

    "recurrent_update_type":
        "additive",

    "representation_parameters":
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT,

    "transformative_parameters":
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT,

    "transformative_weight_parameters":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT,

    "total_parameters":
        NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION,

    "recurrent_state_created":
        NOTEBOOK_09_RECURRENT_STATE_CREATED,

    "h1_created":
        NOTEBOOK_09_RECURRENT_H1_CREATED,

    "recurrent_update_executed":
        NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED,

    "rerun_detected":
        BLOCK_5_RERUN_DETECTED,

    "preexisting_initial_state_object_count":
        len(
            BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECTS
        ),

    "initial_state_objects_reconstructed":
        BLOCK_5_INITIAL_STATE_OBJECTS_RECONSTRUCTED,

    "preexisting_updated_state_object_count":
        len(
            BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS
        ),

    "new_updated_state_objects_created":
        tuple(
            BLOCK_5_NEW_UPDATED_STATE_OBJECTS_CREATED
        ),

    "preexisting_updated_state_objects_preserved":
        BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS_PRESERVED,

    "initial_state_valid":
        NOTEBOOK_09_RECURRENT_INITIAL_STATE_VALID,

    "single_step_validation_ready":
        NOTEBOOK_09_RECURRENT_SINGLE_STEP_VALIDATION_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_5_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_5_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 5: "
    "Recurrent-State Architectural Contract and "
    "Deterministic Initial-State Construction"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_5_VERSION}"
)

print("-" * 72)

print(
    "Recurrent-state architecture"
)

print(
    f"Update type                  : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CONTRACT['update_type']}"
)

print(
    f"Initial-state policy         : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CONTRACT['initial_state_policy']}"
)

print(
    f"Article-boundary reset       : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CONTRACT['article_boundary_reset']}"
)

print(
    f"Future context allowed       : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CONTRACT['future_context_allowed']}"
)

print(
    f"Cross-pathway recurrence     : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CONTRACT['cross_pathway_recurrence']}"
)

print("-" * 72)

print(
    "Initial recurrent states"
)

for pathway_name in NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS:

    print(
        f"{pathway_name:<14} dimension          : "
        f"{NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} state shape        : "
        f"{BLOCK_5_INITIAL_STATE_SHAPES[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} finite             : "
        f"{BLOCK_5_INITIAL_STATES_FINITE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} all zero           : "
        f"{BLOCK_5_INITIAL_STATES_ZERO[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deterministic      : "
        f"{BLOCK_5_INITIAL_STATE_DETERMINISTIC[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} requires grad      : "
        f"{BLOCK_5_INITIAL_STATES_REQUIRE_GRAD[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} device valid       : "
        f"{BLOCK_5_INITIAL_STATE_DEVICE_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} dtype valid        : "
        f"{BLOCK_5_INITIAL_STATE_DTYPE_VALID[pathway_name]}"
    )

print("-" * 72)

print(
    "Parameter and state validation"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Transformative Weight params : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Total model parameters       : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_5_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_5_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"Weight parameters trainable  : "
    f"{BLOCK_5_WEIGHT_PARAMETERS_STILL_TRAINABLE}"
)

print(
    f"Representation unchanged     : "
    f"{BLOCK_5_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged     : "
    f"{BLOCK_5_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Weight state unchanged       : "
    f"{BLOCK_5_WEIGHT_STATE_UNCHANGED}"
)

print(
    f"Inherited modules eval       : "
    f"{BLOCK_5_INHERITED_MODULES_STILL_EVAL}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Recurrent state created      : "
    f"{NOTEBOOK_09_RECURRENT_STATE_CREATED}"
)

print(
    f"Initial state created        : "
    f"{NOTEBOOK_09_RECURRENT_INITIAL_STATE_CREATED}"
)

print(
    f"h1 state created             : "
    f"{NOTEBOOK_09_RECURRENT_H1_CREATED}"
)

print(
    f"Recurrent update executed    : "
    f"{NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"Weighted candidate executed  : "
    f"{NOTEBOOK_09_WEIGHTED_CANDIDATE_TRANSFORMATION_EXECUTED}"
)

print(
    f"Representation forward       : "
    f"{NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED}"
)

print(
    f"Transform forward            : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED}"
)

print(
    f"Weight forward               : "
    f"{NOTEBOOK_09_BLOCK_5_WEIGHT_FORWARD_EXECUTED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_BLOCK_5_LOSS_CALCULATED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_5_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_5_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_5_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Notebook rerun detected      : "
    f"{BLOCK_5_RERUN_DETECTED}"
)

print(
    f"Previous initial-state objs  : "
    f"{len(BLOCK_5_PREEXISTING_INITIAL_STATE_OBJECTS)}"
)

print(
    f"Initial states reconstructed : "
    f"{BLOCK_5_INITIAL_STATE_OBJECTS_RECONSTRUCTED}"
)

print(
    f"Pre-existing updated states  : "
    f"{len(BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS)}"
)

print(
    f"New updated states created   : "
    f"{len(BLOCK_5_NEW_UPDATED_STATE_OBJECTS_CREATED)}"
)

print(
    f"Existing updated states kept : "
    f"{BLOCK_5_PREEXISTING_UPDATED_STATE_OBJECTS_PRESERVED}"
)

print("-" * 72)

print(
    f"Recurrent contract valid     : "
    f"{BLOCK_5_RECURRENT_STATE_CONTRACT_VALID}"
)

print(
    f"Initial state valid          : "
    f"{NOTEBOOK_09_RECURRENT_INITIAL_STATE_VALID}"
)

print(
    f"Single-step validation ready : "
    f"{NOTEBOOK_09_RECURRENT_SINGLE_STEP_VALIDATION_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_5_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_5_COMPLETE}"
)

print("=" * 72)

print(
    f"{len(NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS)} recurrent-state "
    "pathway space(s) were established successfully."
)

print(
    "Deterministic zero initial states were constructed with pathway-specific "
    f"shapes {BLOCK_5_INITIAL_STATE_SHAPES}."
)

print(
    "The initial recurrent states are runtime tensors only and introduce "
    "no trainable parameters."
)

print(
    "Every article is required to begin from a fresh zero recurrent state, "
    "and no state may propagate across article boundaries."
)

print(
    "The recurrent update is defined as previous state plus the "
    "Transformative-Weight-gated candidate transformation."
)

print(
    "Deferred dimensions remain present in the full recurrent state "
    "but receive no candidate-transformation increment under the "
    "current gating contract."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} parameters "
    "remain frozen and unchanged, and all "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT} "
    "Transformative-Weight parameter(s) remain trainable and unchanged."
)

print(
    "Block 5 created fresh deterministic zero initial recurrent states but "
    "executed no h1/T1 state creation, candidate-transformative forward pass, "
    "weighted candidate transformation, recurrent update, loss, optimiser, "
    "backward pass or parameter update."
)

if BLOCK_5_RERUN_DETECTED:

    print(
        "Pre-existing updated recurrent-state objects from later blocks were "
        "detected and left unchanged; only the canonical initial states owned "
        "by Block 5 were reconstructed."
    )

print(
    "Notebook 09 may now proceed to controlled single-step recurrent "
    "forward-path validation."
)

print("=" * 72)

Media AI — Notebook 09, Block 5: Recurrent-State Architectural Contract and Deterministic Initial-State Construction
Block version                : 1.2
------------------------------------------------------------------------
Recurrent-state architecture
Update type                  : additive
Initial-state policy         : deterministic_zero
Article-boundary reset       : True
Future context allowed       : False
Cross-pathway recurrence     : False
------------------------------------------------------------------------
Initial recurrent states
factual        dimension          : 10
factual        state shape        : (1, 10)
factual        finite             : True
factual        all zero           : True
factual        deterministic      : True
factual        requires grad      : False
factual        device valid       : True
factual        dtype valid        : True
psychological  dimension          : 34
psychological  state shape        : (1, 34)
psychological  finite             :

## Block 6 — Controlled Single-Step Recurrent Forward-Path Validation

This block executes the first complete recurrent transition in Notebook 09.

Blocks 1–5 established the full inherited architecture, introduced and validated the Transformative-Weight mechanism, defined the recurrent-state architecture, and constructed deterministic zero initial states for the factual, psychological, and social pathways.

The present block now combines these components for the first time in a controlled single-step forward evaluation.

Its purpose is to verify that a candidate transformation, its corresponding effective Transformative Weight, and the preceding recurrent state interact exactly according to the recurrent update contract

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}.
$$

Only one recurrent transition is evaluated.

No article sequence is traversed beyond this single step. No loss, optimiser, backward pass, gradient clipping, parameter update, or training operation is executed.

### Controlled first-step setting

For each pathway \(k\in\{F,P,S\}\), the first recurrent transition begins from the deterministic initial state

$$
h_0^{(k)}
=
\mathbf{0}_{1\times d_k}.
$$

The inherited transformative mechanism receives the current pathway representation together with this preceding state and produces a candidate transformation

$$
c_1^{(k)}
=
T_k
\left(
\left[
r_1^{(k)};
h_0^{(k)}
\right]
\right).
$$

The corresponding Transformative-Weight module produces

$$
TW_k
\in
[0,1]^{d_k}.
$$

The weighted candidate contribution is then

$$
\Delta h_1^{(k)}
=
TW_k\odot c_1^{(k)}.
$$

Finally, the first recurrent state is

$$
h_1^{(k)}
=
h_0^{(k)}
+
\Delta h_1^{(k)}.
$$

Because

$$
h_0^{(k)}
=
0,
$$

the first-step relation simplifies to

$$
h_1^{(k)}
=
TW_k\odot c_1^{(k)}.
$$

This equality provides a particularly clear validation condition for the first recurrent transition.

### Current representation input

The inherited transformative mechanism requires a current pathway representation

$$
r_1^{(k)}.
$$

Block 6 does not retrain or modify the representation model.

For controlled validation, the block should use a deterministic pathway-specific probe representation with the exact required dimensions:

$$
r_1^{(F)}
\in
\mathbb{R}^{1\times10},
$$

$$
r_1^{(P)}
\in
\mathbb{R}^{1\times34},
$$

$$
r_1^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

The probes are architectural validation inputs only.

They are not supervision labels, article-derived recurrent states, or learned parameters.

Their role is to exercise the complete inherited transformative pathway without requiring a full representation-model forward pass in this block.

### Candidate-transformation execution

Unlike Block 4, Block 6 does execute the inherited transformative mechanisms.

For the factual pathway,

$$
c_1^{(F)}
=
T_F
\left(
\left[
r_1^{(F)};
h_0^{(F)}
\right]
\right),
$$

for the psychological pathway,

$$
c_1^{(P)}
=
T_P
\left(
\left[
r_1^{(P)};
h_0^{(P)}
\right]
\right),
$$

and for the social pathway,

$$
c_1^{(S)}
=
T_S
\left(
\left[
r_1^{(S)};
h_0^{(S)}
\right]
\right).
$$

The required candidate output shapes are

$$
(1,10),
$$

$$
(1,34),
$$

and

$$
(1,7).
$$

All candidate outputs must be finite.

Because the inherited transformative modules remain frozen and in evaluation mode, repeated evaluation under identical inputs must also be deterministic.

### Transformative-Weight execution

The validated Transformative-Weight modules are evaluated again during this block.

Their outputs remain

$$
TW_F\in\mathbb{R}^{10},
$$

$$
TW_P\in\mathbb{R}^{34},
$$

$$
TW_S\in\mathbb{R}^{7}.
$$

For multiplication with batched candidate transformations, each effective weight vector is broadcast across the leading batch dimension.

Thus,

$$
TW_k
\odot
c_1^{(k)}
$$

produces a tensor in

$$
\mathbb{R}^{1\times d_k}.
$$

The active/deferred weighting contract remains unchanged:

$$
TW_{k,j}
=
0.5
\qquad
\text{for }j\in A_k,
$$

and

$$
TW_{k,j}
=
0
\qquad
\text{for }j\in D_k
$$

at the current initial parameter state.

### Weighted candidate transformation

The first new quantity introduced by recurrent execution is the weighted candidate transformation

$$
\Delta h_1^{(k)}
=
TW_k\odot c_1^{(k)}.
$$

This quantity must preserve the complete pathway dimension.

For the factual pathway,

$$
\Delta h_1^{(F)}
\in
\mathbb{R}^{1\times10},
$$

for the psychological pathway,

$$
\Delta h_1^{(P)}
\in
\mathbb{R}^{1\times34},
$$

and for the social pathway,

$$
\Delta h_1^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

All weighted contributions must be finite.

### Active-dimension recurrent behaviour

For active dimensions,

$$
j\in A_k,
$$

the current initial Transformative Weight equals

$$
TW_{k,j}=0.5.
$$

Therefore,

$$
\Delta h_{1,j}^{(k)}
=
0.5\,
c_{1,j}^{(k)}.
$$

The recurrent contribution must therefore preserve the sign of the candidate transformation while scaling its magnitude by one half.

If

$$
c_{1,j}^{(k)}>0,
$$

then

$$
\Delta h_{1,j}^{(k)}>0.
$$

If

$$
c_{1,j}^{(k)}<0,
$$

then

$$
\Delta h_{1,j}^{(k)}<0.
$$

The Transformative Weight therefore controls admission strength without reversing the candidate direction.

### Deferred-dimension recurrent behaviour

For deferred dimensions,

$$
j\in D_k,
$$

the effective Transformative Weight remains exactly zero:

$$
TW_{k,j}=0.
$$

Therefore,

$$
\Delta h_{1,j}^{(k)}
=
0.
$$

Since the first-step preceding state is also zero,

$$
h_{0,j}^{(k)}=0,
$$

the first recurrent state remains

$$
h_{1,j}^{(k)}
=
0
$$

for all currently deferred dimensions.

This provides a strong runtime validation of the deferred-dimension gating policy.

### First recurrent-state construction

The first updated recurrent state is computed as

$$
h_1^{(k)}
=
h_0^{(k)}
+
\Delta h_1^{(k)}.
$$

The output shapes must remain

$$
h_1^{(F)}
\in
\mathbb{R}^{1\times10},
$$

$$
h_1^{(P)}
\in
\mathbb{R}^{1\times34},
$$

$$
h_1^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

Every state value must be finite.

Because the initial state is exactly zero, the block should additionally validate the identity

$$
h_1^{(k)}
=
\Delta h_1^{(k)}
$$

exactly under the controlled first-step setting.

### Dimensional invariance

The recurrent transition must never change pathway dimensionality.

The sequence

$$
r_1^{(k)}
\rightarrow
c_1^{(k)}
\rightarrow
\Delta h_1^{(k)}
\rightarrow
h_1^{(k)}
$$

must preserve

$$
d_F=10,
\qquad
d_P=34,
\qquad
d_S=7.
$$

No active-only compression or dimensional expansion is permitted.

This ensures that the resulting recurrent state can be passed directly as the preceding-state input to the inherited transformative mechanism at the next sentence position.

### Compatibility with the next recurrent step

A valid first recurrent state must satisfy the same dimensional contract as the preceding-state input required by the transformative mechanism.

Therefore,

$$
h_1^{(k)}
\in
\mathbb{R}^{1\times d_k}
$$

must be structurally compatible with

$$
T_k
\left(
\left[
r_2^{(k)};
h_1^{(k)}
\right]
\right).
$$

Block 6 does not execute this second transition, but it verifies that the first updated state is eligible to become the preceding state for a future step.

### Deterministic single-step validation

The complete controlled transition must be deterministic.

Repeated evaluation with identical

$$
r_1^{(k)},
\quad
h_0^{(k)},
\quad
T_k,
\quad
TW_k
$$

must produce identical

$$
c_1^{(k)},
$$

$$
\Delta h_1^{(k)},
$$

and

$$
h_1^{(k)}.
$$

Thus,

$$
h_{1,\mathrm{run1}}^{(k)}
=
h_{1,\mathrm{run2}}^{(k)}.
$$

This is essential because later recurrent trajectories depend recursively on earlier states.

Any uncontrolled stochasticity at one step would propagate through the complete article sequence.

### Causal boundary

The single-step recurrent transition uses only the current controlled representation and the preceding state:

$$
r_1^{(k)}
$$

and

$$
h_0^{(k)}.
$$

No future representation, future state, or future sentence information is available.

The execution therefore preserves the causal contract

$$
h_1^{(k)}
=
f
\left(
r_1^{(k)},
h_0^{(k)}
\right).
$$

No quantity associated with

$$
t>1
$$

enters the calculation.

### Pathway independence

The three recurrent transitions remain independent.

The factual state is updated only through

$$
r_1^{(F)},
\quad
h_0^{(F)},
\quad
T_F,
\quad
TW_F.
$$

The psychological state is updated only through

$$
r_1^{(P)},
\quad
h_0^{(P)},
\quad
T_P,
\quad
TW_P.
$$

The social state is updated only through

$$
r_1^{(S)},
\quad
h_0^{(S)},
\quad
T_S,
\quad
TW_S.
$$

No cross-pathway state, candidate transformation, or weight contributes to another pathway's recurrent update.

### Parameter immutability

Block 6 executes forward paths but performs no optimisation.

Therefore, the inherited representation parameters must remain

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}},
$$

the inherited transformative parameters must remain

$$
\theta_T^{\mathrm{after}}
=
\theta_T^{\mathrm{before}},
$$

and the Transformative-Weight parameters must remain

$$
\theta_{TW}^{\mathrm{after}}
=
\theta_{TW}^{\mathrm{before}}.
$$

The complete model parameter count remains

$$
902{,}215.
$$

No recurrent-state tensor contributes to this parameter count.

### Gradient boundary

The 25 Transformative-Weight latent parameters remain trainable in principle.

However, Block 6 is a forward-validation stage only.

The controlled recurrent transition should therefore be evaluated without gradient accumulation.

No gradients should be created on the Transformative-Weight parameters, and the inherited representation and transformative parameters remain frozen.

This ensures that recurrent execution can be validated independently of optimisation.

### No sequence traversal

Block 6 creates only

$$
h_1^{(F)},
\qquad
h_1^{(P)},
\qquad
h_1^{(S)}.
$$

It does not compute

$$
h_2,
h_3,
\ldots
$$

and does not traverse the 16-sentence pilot article.

The purpose is to validate one transition before introducing recursive execution across an ordered sequence.

### No optimisation in Block 6

The block performs no training.

Accordingly:

- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- no gradient clipping is applied;
- no optimiser step is executed;
- and no parameter update occurs.

The recurrent state changes because recurrent execution is a runtime operation, not because model parameters are updated.

### Block 6 completion contract

Block 6 is complete only when:

- deterministic pathway-specific current-representation probes are available;
- factual, psychological, and social candidate-transformative forward passes execute successfully;
- candidate output shapes are exactly \((1,10)\), \((1,34)\), and \((1,7)\);
- all candidate transformations are finite;
- repeated candidate evaluations are deterministic;
- Transformative-Weight forward execution succeeds for all three pathways;
- effective weight vectors preserve dimensions 10, 34, and 7;
- weighted candidate transformations preserve shapes \((1,10)\), \((1,34)\), and \((1,7)\);
- all weighted candidate values are finite;
- active dimensions reproduce the expected current scaling behaviour;
- deferred dimensions produce exactly zero recurrent contribution;
- first recurrent states \(h_1\) are created successfully;
- first recurrent-state shapes are exactly \((1,10)\), \((1,34)\), and \((1,7)\);
- all first recurrent-state values are finite;
- \(h_1^{(k)}=\Delta h_1^{(k)}\) holds under the zero initial-state condition;
- repeated complete single-step evaluations are deterministic;
- pathway independence is preserved;
- the resulting \(h_1\) tensors are valid preceding-state inputs for a future transition;
- no future context is used;
- no second recurrent step is executed;
- the inherited 902,190 parameters remain frozen and exactly unchanged;
- the 25 Transformative-Weight parameters remain unchanged;
- the complete model parameter count remains 902,215;
- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- and no parameter update occurs.

Successful completion validates the first complete recurrent relation

$$
\boxed{
h_1^{(k)}
=
h_0^{(k)}
+
TW_k\odot
T_k
\left(
\left[
r_1^{(k)};
h_0^{(k)}
\right]
\right)
}
$$

with

$$
h_0^{(k)}=0,
$$

so that

$$
\boxed{
h_1^{(k)}
=
TW_k\odot c_1^{(k)}
}
$$

for the controlled initial transition.

Notebook 09 may then proceed to controlled multi-step recurrent trajectory validation before any recurrent optimisation policy is introduced.

In [56]:
# =============================================================================
# Media AI — Notebook 09
# Block 6: Controlled Single-Step Recurrent Forward-Path Validation
# =============================================================================

from copy import deepcopy

import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_6 = 6

NOTEBOOK_09_BLOCK_6_NAME = (
    "Controlled Single-Step Recurrent Forward-Path Validation"
)

NOTEBOOK_09_BLOCK_6_VERSION = "1.1"


# =============================================================================
# Required inherited runtime contract
# =============================================================================

BLOCK_6_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_BLOCK_1_COMPLETE",
    "NOTEBOOK_09_BLOCK_1_VALID",
    "NOTEBOOK_09_HANDOVER_RESTORED",
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",
    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Blocks 2–4
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_2_COMPLETE",
    "NOTEBOOK_09_BLOCK_2_VALID",
    "NOTEBOOK_09_BLOCK_3_COMPLETE",
    "NOTEBOOK_09_BLOCK_3_VALID",
    "NOTEBOOK_09_BLOCK_4_COMPLETE",
    "NOTEBOOK_09_BLOCK_4_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID",

    # -------------------------------------------------------------------------
    # Block 5
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_5_COMPLETE",
    "NOTEBOOK_09_BLOCK_5_VALID",
    "NOTEBOOK_09_RECURRENT_INITIAL_STATE_VALID",
    "NOTEBOOK_09_RECURRENT_SINGLE_STEP_VALIDATION_READY",
    "NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS",
    "NOTEBOOK_09_INITIAL_RECURRENT_STATES",
    "NOTEBOOK_09_FACTUAL_INITIAL_STATE",
    "NOTEBOOK_09_PSYCHOLOGICAL_INITIAL_STATE",
    "NOTEBOOK_09_SOCIAL_INITIAL_STATE",

    # -------------------------------------------------------------------------
    # Transformative Weight contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE",
    "NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES",

    # -------------------------------------------------------------------------
    # Environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_6_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_6_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_6_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 6 prerequisites are not initialised. "
        f"Missing: {BLOCK_6_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_6_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,

        NOTEBOOK_09_BLOCK_1_COMPLETE is True,
        NOTEBOOK_09_BLOCK_1_VALID is True,
        NOTEBOOK_09_HANDOVER_RESTORED is True,

        NOTEBOOK_09_BLOCK_2_COMPLETE is True,
        NOTEBOOK_09_BLOCK_2_VALID is True,

        NOTEBOOK_09_BLOCK_3_COMPLETE is True,
        NOTEBOOK_09_BLOCK_3_VALID is True,

        NOTEBOOK_09_BLOCK_4_COMPLETE is True,
        NOTEBOOK_09_BLOCK_4_VALID is True,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_FORWARD_PATH_VALID is True,

        NOTEBOOK_09_BLOCK_5_COMPLETE is True,
        NOTEBOOK_09_BLOCK_5_VALID is True,
        NOTEBOOK_09_RECURRENT_INITIAL_STATE_VALID is True,
        NOTEBOOK_09_RECURRENT_SINGLE_STEP_VALIDATION_READY is True,
    ]
)


if not BLOCK_6_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Blocks 1–5 must be valid and complete "
        "before single-step recurrent forward validation."
    )


# =============================================================================
# Canonical pathway contract
# =============================================================================

BLOCK_6_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
)


BLOCK_6_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            BLOCK_6_PATHWAY_DIMENSIONS,
            dict,
        ),

        bool(
            BLOCK_6_PATHWAY_DIMENSIONS
        ),

        (
            BLOCK_6_PATHWAY_DIMENSIONS
            ==
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
        ),

        (
            set(
                BLOCK_6_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                BLOCK_1_TRANSFORMATIVE_MODULES.keys()
            )
            ==
            set(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys()
            )
            ==
            set(
                NOTEBOOK_09_INITIAL_RECURRENT_STATES.keys()
            )
        ),
    ]
)


if not BLOCK_6_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 6 pathway dimensions are invalid."
    )


# =============================================================================
# Canonical inherited transformative modules
# =============================================================================

BLOCK_6_TRANSFORMATIVE_MODULES = {
    pathway_name:
        BLOCK_1_TRANSFORMATIVE_MODULES[
            pathway_name
        ]

    for pathway_name
    in BLOCK_6_PATHWAY_DIMENSIONS
}


# =============================================================================
# Canonical initial recurrent states
# =============================================================================

BLOCK_6_INITIAL_STATES = {
    pathway_name:
        state.detach()
        .clone()

    for (
        pathway_name,
        state,
    ) in NOTEBOOK_09_INITIAL_RECURRENT_STATES.items()
}


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_6_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_6_nested_state_exact(
    state_before,
    state_after,
):

    if state_before.keys() != state_after.keys():

        return False


    for module_name in state_before:

        before_module_state = (
            state_before[
                module_name
            ]
        )


        after_module_state = (
            state_after[
                module_name
            ]
        )


        if (
            before_module_state.keys()
            !=
            after_module_state.keys()
        ):

            return False


        for tensor_name in before_module_state:

            if not torch.equal(
                before_module_state[
                    tensor_name
                ],
                after_module_state[
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot learned model state before controlled recurrent validation
# =============================================================================

BLOCK_6_REPRESENTATION_STATE_BEFORE = (
    block_6_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_6_TRANSFORMATIVE_STATE_BEFORE = (
    block_6_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_6_WEIGHT_STATE_BEFORE = (
    block_6_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Deterministic current-representation probe construction
# =============================================================================
#
# These probes are architectural validation inputs only.
#
# They are not supervision targets and are not produced by a representation
# forward pass.
# =============================================================================

def block_6_construct_probe(
    pathway_dim,
):

    pathway_dim = int(
        pathway_dim
    )


    return torch.linspace(
        -0.5,
        0.5,
        steps=
            pathway_dim,
        dtype=
            DEFAULT_DTYPE,
        device=
            DEVICE,
    ).reshape(
        1,
        pathway_dim,
    )


BLOCK_6_CURRENT_REPRESENTATION_PROBES = {
    pathway_name:
        block_6_construct_probe(
            pathway_dim
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_6_PATHWAY_DIMENSIONS.items()
}


# =============================================================================
# Probe shape and numerical validation
# =============================================================================

BLOCK_6_PROBE_SHAPES = {
    pathway_name:
        tuple(
            probe.shape
        )

    for (
        pathway_name,
        probe,
    ) in BLOCK_6_CURRENT_REPRESENTATION_PROBES.items()
}


BLOCK_6_EXPECTED_PROBE_SHAPES = {
    pathway_name:
        (
            1,
            pathway_dim,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_6_PATHWAY_DIMENSIONS.items()
}


BLOCK_6_PROBE_SHAPES_VALID = (
    BLOCK_6_PROBE_SHAPES
    ==
    BLOCK_6_EXPECTED_PROBE_SHAPES
)


BLOCK_6_PROBES_FINITE = all(
    bool(
        torch.isfinite(
            probe
        ).all().item()
    )

    for probe
    in BLOCK_6_CURRENT_REPRESENTATION_PROBES.values()
)


if not all(
    [
        BLOCK_6_PROBE_SHAPES_VALID,
        BLOCK_6_PROBES_FINITE,
    ]
):

    raise RuntimeError(
        "Controlled current-representation probes are invalid."
    )


# =============================================================================
# Controlled single-step execution helper
# =============================================================================

def block_6_execute_single_step(
    pathway_name,
):
    """
    Execute one deterministic recurrent transition:

        candidate =
            T_k(
                current_representation,
                previous_state,
            )

        effective_weight =
            TW_k()

        weighted_candidate =
            effective_weight * candidate

        h1 =
            previous_state + weighted_candidate
    """

    current_representation = (
        BLOCK_6_CURRENT_REPRESENTATION_PROBES[
            pathway_name
        ]
    )


    previous_state = (
        BLOCK_6_INITIAL_STATES[
            pathway_name
        ]
    )


    transformative_module = (
        BLOCK_6_TRANSFORMATIVE_MODULES[
            pathway_name
        ]
    )


    weight_module = (
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
            pathway_name
        ]
    )


    candidate = transformative_module(
        current_representation,
        previous_state,
    )


    effective_weight = (
        weight_module()
    )


    weighted_candidate = (
        effective_weight.unsqueeze(
            0
        )
        *
        candidate
    )


    updated_state = (
        previous_state
        +
        weighted_candidate
    )


    return {
        "current_representation":
            current_representation,

        "previous_state":
            previous_state,

        "candidate":
            candidate,

        "effective_weight":
            effective_weight,

        "weighted_candidate":
            weighted_candidate,

        "updated_state":
            updated_state,
    }


# =============================================================================
# Execute first controlled recurrent transition twice
# =============================================================================
#
# torch.no_grad() is intentional:
# - forward paths are validated;
# - no gradient graph is retained;
# - no optimisation occurs.
# =============================================================================

BLOCK_6_FIRST_RUN = {}

BLOCK_6_SECOND_RUN = {}


with torch.no_grad():

    for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

        BLOCK_6_FIRST_RUN[
            pathway_name
        ] = block_6_execute_single_step(
            pathway_name
        )


        BLOCK_6_SECOND_RUN[
            pathway_name
        ] = block_6_execute_single_step(
            pathway_name
        )


# =============================================================================
# Execution flags
# =============================================================================

NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED = True

NOTEBOOK_09_BLOCK_6_WEIGHT_FORWARD_EXECUTED = True

NOTEBOOK_09_WEIGHTED_CANDIDATE_TRANSFORMATION_EXECUTED = True

NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED = True

NOTEBOOK_09_RECURRENT_H1_CREATED = True


# =============================================================================
# Candidate-transformation outputs
# =============================================================================

BLOCK_6_CANDIDATES = {
    pathway_name:
        result[
            "candidate"
        ]
        .detach()
        .cpu()
        .clone()

    for (
        pathway_name,
        result,
    ) in BLOCK_6_FIRST_RUN.items()
}


BLOCK_6_CANDIDATE_SHAPES = {
    pathway_name:
        tuple(
            candidate.shape
        )

    for (
        pathway_name,
        candidate,
    ) in BLOCK_6_CANDIDATES.items()
}


BLOCK_6_EXPECTED_CANDIDATE_SHAPES = deepcopy(
    BLOCK_6_EXPECTED_PROBE_SHAPES
)


BLOCK_6_CANDIDATE_SHAPES_VALID = (
    BLOCK_6_CANDIDATE_SHAPES
    ==
    BLOCK_6_EXPECTED_CANDIDATE_SHAPES
)


BLOCK_6_CANDIDATES_FINITE = {
    pathway_name:
        bool(
            torch.isfinite(
                candidate
            ).all().item()
        )

    for (
        pathway_name,
        candidate,
    ) in BLOCK_6_CANDIDATES.items()
}


BLOCK_6_ALL_CANDIDATES_FINITE = all(
    BLOCK_6_CANDIDATES_FINITE.values()
)


if not all(
    [
        BLOCK_6_CANDIDATE_SHAPES_VALID,
        BLOCK_6_ALL_CANDIDATES_FINITE,
    ]
):

    raise RuntimeError(
        "Candidate-transformative forward outputs are invalid."
    )


# =============================================================================
# Candidate determinism validation
# =============================================================================

BLOCK_6_CANDIDATE_DETERMINISTIC = {
    pathway_name:
        torch.equal(
            BLOCK_6_FIRST_RUN[
                pathway_name
            ][
                "candidate"
            ],
            BLOCK_6_SECOND_RUN[
                pathway_name
            ][
                "candidate"
            ],
        )

    for pathway_name
    in BLOCK_6_FIRST_RUN
}


BLOCK_6_ALL_CANDIDATES_DETERMINISTIC = all(
    BLOCK_6_CANDIDATE_DETERMINISTIC.values()
)


if not BLOCK_6_ALL_CANDIDATES_DETERMINISTIC:

    raise RuntimeError(
        "Candidate-transformative outputs are not deterministic."
    )


# =============================================================================
# Transformative-Weight outputs
# =============================================================================

BLOCK_6_EFFECTIVE_WEIGHTS = {
    pathway_name:
        result[
            "effective_weight"
        ]
        .detach()
        .cpu()
        .clone()

    for (
        pathway_name,
        result,
    ) in BLOCK_6_FIRST_RUN.items()
}


BLOCK_6_EFFECTIVE_WEIGHT_SHAPES = {
    pathway_name:
        tuple(
            weight.shape
        )

    for (
        pathway_name,
        weight,
    ) in BLOCK_6_EFFECTIVE_WEIGHTS.items()
}


BLOCK_6_EXPECTED_EFFECTIVE_WEIGHT_SHAPES = {
    pathway_name:
        (
            pathway_dim,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_6_PATHWAY_DIMENSIONS.items()
}


BLOCK_6_EFFECTIVE_WEIGHT_SHAPES_VALID = (
    BLOCK_6_EFFECTIVE_WEIGHT_SHAPES
    ==
    BLOCK_6_EXPECTED_EFFECTIVE_WEIGHT_SHAPES
)


BLOCK_6_EFFECTIVE_WEIGHTS_FINITE = all(
    bool(
        torch.isfinite(
            weight
        ).all().item()
    )

    for weight
    in BLOCK_6_EFFECTIVE_WEIGHTS.values()
)


BLOCK_6_EFFECTIVE_WEIGHTS_BOUNDED = all(
    bool(
        (
            (
                weight
                >=
                0.0
            )
            &
            (
                weight
                <=
                1.0
            )
        ).all().item()
    )

    for weight
    in BLOCK_6_EFFECTIVE_WEIGHTS.values()
)


if not all(
    [
        BLOCK_6_EFFECTIVE_WEIGHT_SHAPES_VALID,
        BLOCK_6_EFFECTIVE_WEIGHTS_FINITE,
        BLOCK_6_EFFECTIVE_WEIGHTS_BOUNDED,
    ]
):

    raise RuntimeError(
        "Transformative-Weight outputs are invalid during "
        "single-step recurrent validation."
    )


# =============================================================================
# Weighted candidate transformations
# =============================================================================

BLOCK_6_WEIGHTED_CANDIDATES = {
    pathway_name:
        result[
            "weighted_candidate"
        ]
        .detach()
        .cpu()
        .clone()

    for (
        pathway_name,
        result,
    ) in BLOCK_6_FIRST_RUN.items()
}


BLOCK_6_WEIGHTED_CANDIDATE_SHAPES = {
    pathway_name:
        tuple(
            weighted_candidate.shape
        )

    for (
        pathway_name,
        weighted_candidate,
    ) in BLOCK_6_WEIGHTED_CANDIDATES.items()
}


BLOCK_6_EXPECTED_WEIGHTED_CANDIDATE_SHAPES = deepcopy(
    BLOCK_6_EXPECTED_PROBE_SHAPES
)


BLOCK_6_WEIGHTED_CANDIDATE_SHAPES_VALID = (
    BLOCK_6_WEIGHTED_CANDIDATE_SHAPES
    ==
    BLOCK_6_EXPECTED_WEIGHTED_CANDIDATE_SHAPES
)


BLOCK_6_WEIGHTED_CANDIDATES_FINITE = {
    pathway_name:
        bool(
            torch.isfinite(
                weighted_candidate
            ).all().item()
        )

    for (
        pathway_name,
        weighted_candidate,
    ) in BLOCK_6_WEIGHTED_CANDIDATES.items()
}


BLOCK_6_ALL_WEIGHTED_CANDIDATES_FINITE = all(
    BLOCK_6_WEIGHTED_CANDIDATES_FINITE.values()
)


if not all(
    [
        BLOCK_6_WEIGHTED_CANDIDATE_SHAPES_VALID,
        BLOCK_6_ALL_WEIGHTED_CANDIDATES_FINITE,
    ]
):

    raise RuntimeError(
        "Weighted candidate-transformative outputs are invalid."
    )


# =============================================================================
# Active-dimension scaling validation
# =============================================================================

BLOCK_6_ACTIVE_SCALING_VALID = {}


for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    candidate = (
        BLOCK_6_CANDIDATES[
            pathway_name
        ]
    )


    weighted_candidate = (
        BLOCK_6_WEIGHTED_CANDIDATES[
            pathway_name
        ]
    )


    active_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    candidate_active = (
        candidate[
            :,
            active_indices
        ]
    )


    weighted_active = (
        weighted_candidate[
            :,
            active_indices
        ]
    )


    expected_weighted_active = (
        float(
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE
        )
        *
        candidate_active
    )


    BLOCK_6_ACTIVE_SCALING_VALID[
        pathway_name
    ] = torch.allclose(
        weighted_active,
        expected_weighted_active,
        rtol=
            0.0,
        atol=
            0.0,
    )


BLOCK_6_ALL_ACTIVE_SCALING_VALID = all(
    BLOCK_6_ACTIVE_SCALING_VALID.values()
)


if not BLOCK_6_ALL_ACTIVE_SCALING_VALID:

    raise RuntimeError(
        "Active recurrent contributions do not reproduce "
        "the expected 0.5 candidate scaling."
    )


# =============================================================================
# Deferred-dimension zero-contribution validation
# =============================================================================

BLOCK_6_DEFERRED_CONTRIBUTIONS_ZERO = {}


for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    weighted_candidate = (
        BLOCK_6_WEIGHTED_CANDIDATES[
            pathway_name
        ]
    )


    deferred_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    deferred_values = (
        weighted_candidate[
            :,
            deferred_indices
        ]
    )


    BLOCK_6_DEFERRED_CONTRIBUTIONS_ZERO[
        pathway_name
    ] = torch.equal(
        deferred_values,
        torch.zeros_like(
            deferred_values
        ),
    )


BLOCK_6_ALL_DEFERRED_CONTRIBUTIONS_ZERO = all(
    BLOCK_6_DEFERRED_CONTRIBUTIONS_ZERO.values()
)


if not BLOCK_6_ALL_DEFERRED_CONTRIBUTIONS_ZERO:

    raise RuntimeError(
        "Deferred recurrent dimensions received non-zero "
        "candidate-transformation contributions."
    )


# =============================================================================
# Sign-preservation validation on active dimensions
# =============================================================================
#
# Weight = 0.5 > 0, so non-zero active weighted contributions must preserve
# the candidate sign exactly.
# =============================================================================

BLOCK_6_ACTIVE_SIGN_PRESERVED = {}


for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    candidate = (
        BLOCK_6_CANDIDATES[
            pathway_name
        ]
    )


    weighted_candidate = (
        BLOCK_6_WEIGHTED_CANDIDATES[
            pathway_name
        ]
    )


    active_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    candidate_active = (
        candidate[
            :,
            active_indices
        ]
    )


    weighted_active = (
        weighted_candidate[
            :,
            active_indices
        ]
    )


    nonzero_mask = (
        candidate_active
        !=
        0.0
    )


    if bool(
        nonzero_mask.any().item()
    ):

        BLOCK_6_ACTIVE_SIGN_PRESERVED[
            pathway_name
        ] = bool(
            (
                torch.sign(
                    candidate_active[
                        nonzero_mask
                    ]
                )
                ==
                torch.sign(
                    weighted_active[
                        nonzero_mask
                    ]
                )
            ).all().item()
        )

    else:

        BLOCK_6_ACTIVE_SIGN_PRESERVED[
            pathway_name
        ] = True


BLOCK_6_ALL_ACTIVE_SIGNS_PRESERVED = all(
    BLOCK_6_ACTIVE_SIGN_PRESERVED.values()
)


if not BLOCK_6_ALL_ACTIVE_SIGNS_PRESERVED:

    raise RuntimeError(
        "Transformative-Weight gating reversed a candidate "
        "transformation direction."
    )


# =============================================================================
# First recurrent-state outputs
# =============================================================================

BLOCK_6_H1_STATES = {
    pathway_name:
        result[
            "updated_state"
        ]
        .detach()
        .cpu()
        .clone()

    for (
        pathway_name,
        result,
    ) in BLOCK_6_FIRST_RUN.items()
}


BLOCK_6_H1_SHAPES = {
    pathway_name:
        tuple(
            state.shape
        )

    for (
        pathway_name,
        state,
    ) in BLOCK_6_H1_STATES.items()
}


BLOCK_6_EXPECTED_H1_SHAPES = deepcopy(
    BLOCK_6_EXPECTED_PROBE_SHAPES
)


BLOCK_6_H1_SHAPES_VALID = (
    BLOCK_6_H1_SHAPES
    ==
    BLOCK_6_EXPECTED_H1_SHAPES
)


BLOCK_6_H1_FINITE = {
    pathway_name:
        bool(
            torch.isfinite(
                state
            ).all().item()
        )

    for (
        pathway_name,
        state,
    ) in BLOCK_6_H1_STATES.items()
}


BLOCK_6_ALL_H1_FINITE = all(
    BLOCK_6_H1_FINITE.values()
)


if not all(
    [
        BLOCK_6_H1_SHAPES_VALID,
        BLOCK_6_ALL_H1_FINITE,
    ]
):

    raise RuntimeError(
        "First recurrent-state outputs are invalid."
    )


# =============================================================================
# Zero-initial-state recurrent identity validation
# =============================================================================
#
# Since h0 = 0:
#
#     h1 = h0 + weighted_candidate
#        = weighted_candidate
# =============================================================================

BLOCK_6_H1_EQUALS_WEIGHTED_CANDIDATE = {
    pathway_name:
        torch.equal(
            BLOCK_6_H1_STATES[
                pathway_name
            ],
            BLOCK_6_WEIGHTED_CANDIDATES[
                pathway_name
            ],
        )

    for pathway_name
    in BLOCK_6_H1_STATES
}


BLOCK_6_ALL_H1_EQUAL_WEIGHTED_CANDIDATE = all(
    BLOCK_6_H1_EQUALS_WEIGHTED_CANDIDATE.values()
)


if not BLOCK_6_ALL_H1_EQUAL_WEIGHTED_CANDIDATE:

    raise RuntimeError(
        "First recurrent-state output does not equal the weighted "
        "candidate under the zero initial-state condition."
    )


# =============================================================================
# Deferred h1 state validation
# =============================================================================

BLOCK_6_DEFERRED_H1_ZERO = {}


for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    h1 = (
        BLOCK_6_H1_STATES[
            pathway_name
        ]
    )


    deferred_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    deferred_h1 = (
        h1[
            :,
            deferred_indices
        ]
    )


    BLOCK_6_DEFERRED_H1_ZERO[
        pathway_name
    ] = torch.equal(
        deferred_h1,
        torch.zeros_like(
            deferred_h1
        ),
    )


BLOCK_6_ALL_DEFERRED_H1_ZERO = all(
    BLOCK_6_DEFERRED_H1_ZERO.values()
)


if not BLOCK_6_ALL_DEFERRED_H1_ZERO:

    raise RuntimeError(
        "Deferred recurrent-state dimensions are non-zero "
        "after the first transition."
    )


# =============================================================================
# Complete single-step determinism validation
# =============================================================================

BLOCK_6_WEIGHT_DETERMINISTIC = {}

BLOCK_6_WEIGHTED_CANDIDATE_DETERMINISTIC = {}

BLOCK_6_H1_DETERMINISTIC = {}


for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    BLOCK_6_WEIGHT_DETERMINISTIC[
        pathway_name
    ] = torch.equal(
        BLOCK_6_FIRST_RUN[
            pathway_name
        ][
            "effective_weight"
        ],
        BLOCK_6_SECOND_RUN[
            pathway_name
        ][
            "effective_weight"
        ],
    )


    BLOCK_6_WEIGHTED_CANDIDATE_DETERMINISTIC[
        pathway_name
    ] = torch.equal(
        BLOCK_6_FIRST_RUN[
            pathway_name
        ][
            "weighted_candidate"
        ],
        BLOCK_6_SECOND_RUN[
            pathway_name
        ][
            "weighted_candidate"
        ],
    )


    BLOCK_6_H1_DETERMINISTIC[
        pathway_name
    ] = torch.equal(
        BLOCK_6_FIRST_RUN[
            pathway_name
        ][
            "updated_state"
        ],
        BLOCK_6_SECOND_RUN[
            pathway_name
        ][
            "updated_state"
        ],
    )


BLOCK_6_ALL_WEIGHTS_DETERMINISTIC = all(
    BLOCK_6_WEIGHT_DETERMINISTIC.values()
)


BLOCK_6_ALL_WEIGHTED_CANDIDATES_DETERMINISTIC = all(
    BLOCK_6_WEIGHTED_CANDIDATE_DETERMINISTIC.values()
)


BLOCK_6_ALL_H1_DETERMINISTIC = all(
    BLOCK_6_H1_DETERMINISTIC.values()
)


BLOCK_6_COMPLETE_STEP_DETERMINISTIC = all(
    [
        BLOCK_6_ALL_CANDIDATES_DETERMINISTIC,
        BLOCK_6_ALL_WEIGHTS_DETERMINISTIC,
        BLOCK_6_ALL_WEIGHTED_CANDIDATES_DETERMINISTIC,
        BLOCK_6_ALL_H1_DETERMINISTIC,
    ]
)


if not BLOCK_6_COMPLETE_STEP_DETERMINISTIC:

    raise RuntimeError(
        "Controlled single-step recurrent execution is not deterministic."
    )


# =============================================================================
# Next-step compatibility validation
# =============================================================================
#
# h1 must be directly usable as the preceding-state input for T_k at t = 2.
# No second transition is executed.
# =============================================================================

BLOCK_6_H1_NEXT_STEP_COMPATIBLE = {
    pathway_name:
        all(
            [
                (
                    tuple(
                        state.shape
                    )
                    ==
                    (
                        1,
                        BLOCK_6_PATHWAY_DIMENSIONS[
                            pathway_name
                        ],
                    )
                ),

                (
                    state.dtype
                    ==
                    DEFAULT_DTYPE
                ),
            ]
        )

    for (
        pathway_name,
        state,
    ) in BLOCK_6_H1_STATES.items()
}


BLOCK_6_ALL_H1_NEXT_STEP_COMPATIBLE = all(
    BLOCK_6_H1_NEXT_STEP_COMPATIBLE.values()
)


if not BLOCK_6_ALL_H1_NEXT_STEP_COMPATIBLE:

    raise RuntimeError(
        "One or more h1 states are incompatible with a future "
        "transformative preceding-state input."
    )


# =============================================================================
# Pathway independence validation
# =============================================================================
#
# Each pathway uses a distinct:
# - transformative module,
# - Transformative-Weight module,
# - initial-state tensor.
# =============================================================================

BLOCK_6_TRANSFORMATIVE_MODULE_IDS = {
    pathway_name:
        id(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_6_TRANSFORMATIVE_MODULES.items()
}


BLOCK_6_WEIGHT_MODULE_IDS = {
    pathway_name:
        id(
            module
        )

    for (
        pathway_name,
        module,
    ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.items()
}


BLOCK_6_INITIAL_STATE_STORAGE_IDS = {
    pathway_name:
        int(
            state.data_ptr()
        )

    for (
        pathway_name,
        state,
    ) in BLOCK_6_INITIAL_STATES.items()
}


BLOCK_6_PATHWAY_COUNT = len(
    BLOCK_6_PATHWAY_DIMENSIONS
)


BLOCK_6_PATHWAYS_INDEPENDENT = all(
    [
        (
            len(
                set(
                    BLOCK_6_TRANSFORMATIVE_MODULE_IDS.values()
                )
            )
            ==
            BLOCK_6_PATHWAY_COUNT
        ),

        (
            len(
                set(
                    BLOCK_6_WEIGHT_MODULE_IDS.values()
                )
            )
            ==
            BLOCK_6_PATHWAY_COUNT
        ),

        (
            len(
                set(
                    BLOCK_6_INITIAL_STATE_STORAGE_IDS.values()
                )
            )
            ==
            BLOCK_6_PATHWAY_COUNT
        ),
    ]
)


if not BLOCK_6_PATHWAYS_INDEPENDENT:

    raise RuntimeError(
        "Recurrent pathway independence is invalid."
    )


# =============================================================================
# Gradient-state validation
# =============================================================================
#
# No gradients should accumulate because torch.no_grad() was used.
# =============================================================================

BLOCK_6_WEIGHT_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_6_REPRESENTATION_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_6_TRANSFORMATIVE_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_6_ALL_GRADIENTS_ABSENT = all(
    [
        BLOCK_6_WEIGHT_GRADIENTS_ABSENT,
        BLOCK_6_REPRESENTATION_GRADIENTS_ABSENT,
        BLOCK_6_TRANSFORMATIVE_GRADIENTS_ABSENT,
    ]
)


if not BLOCK_6_ALL_GRADIENTS_ABSENT:

    raise RuntimeError(
        "Unexpected gradients were created during recurrent "
        "forward-path validation."
    )


# =============================================================================
# Snapshot learned model state after validation
# =============================================================================

BLOCK_6_REPRESENTATION_STATE_AFTER = (
    block_6_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_6_TRANSFORMATIVE_STATE_AFTER = (
    block_6_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_6_WEIGHT_STATE_AFTER = (
    block_6_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Parameter immutability validation
# =============================================================================

BLOCK_6_REPRESENTATION_UNCHANGED = (
    block_6_nested_state_exact(
        BLOCK_6_REPRESENTATION_STATE_BEFORE,
        BLOCK_6_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_6_TRANSFORMATIVE_UNCHANGED = (
    block_6_nested_state_exact(
        BLOCK_6_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_6_TRANSFORMATIVE_STATE_AFTER,
    )
)


BLOCK_6_WEIGHT_STATE_UNCHANGED = (
    block_6_nested_state_exact(
        BLOCK_6_WEIGHT_STATE_BEFORE,
        BLOCK_6_WEIGHT_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_6_REPRESENTATION_UNCHANGED,
        BLOCK_6_TRANSFORMATIVE_UNCHANGED,
        BLOCK_6_WEIGHT_STATE_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Model parameter state changed during controlled "
        "single-step recurrent validation."
    )


# =============================================================================
# Trainability and evaluation-mode validation
# =============================================================================

BLOCK_6_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_6_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_6_WEIGHT_PARAMETERS_STILL_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_6_INHERITED_MODULES_STILL_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


if not all(
    [
        BLOCK_6_REPRESENTATION_STILL_FROZEN,
        BLOCK_6_TRANSFORMATIVE_STILL_FROZEN,
        BLOCK_6_WEIGHT_PARAMETERS_STILL_TRAINABLE,
        BLOCK_6_INHERITED_MODULES_STILL_EVAL,
    ]
):

    raise RuntimeError(
        "Notebook 09 trainability or inherited evaluation-mode scope "
        "changed during Block 6."
    )


# =============================================================================
# Parameter-accounting validation
# =============================================================================

BLOCK_6_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS
            ==
            sum(
                BLOCK_6_PATHWAY_DIMENSIONS.values()
            )
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT
            ==
            sum(
                len(
                    NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
                        pathway_name
                    ]
                )

                for pathway_name
                in BLOCK_6_PATHWAY_DIMENSIONS
            )
        ),
    ]
)


if not BLOCK_6_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 parameter accounting changed during "
        "single-step recurrent validation."
    )


# =============================================================================
# Causal and sequence-boundary contract
# =============================================================================

NOTEBOOK_09_BLOCK_6_FUTURE_CONTEXT_USED = False

NOTEBOOK_09_BLOCK_6_SECOND_RECURRENT_STEP_EXECUTED = False

NOTEBOOK_09_BLOCK_6_ARTICLE_SEQUENCE_TRAVERSED = False


BLOCK_6_CAUSAL_CONTRACT_VALID = all(
    [
        NOTEBOOK_09_BLOCK_6_FUTURE_CONTEXT_USED is False,
        NOTEBOOK_09_BLOCK_6_SECOND_RECURRENT_STEP_EXECUTED is False,
        NOTEBOOK_09_BLOCK_6_ARTICLE_SEQUENCE_TRAVERSED is False,
    ]
)


# =============================================================================
# Explicit optimisation boundary
# =============================================================================

NOTEBOOK_09_BLOCK_6_LOSS_CALCULATED = False

NOTEBOOK_09_BLOCK_6_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_6_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_6_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_09_BLOCK_6_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_6_TRAINING_EXECUTED = False


# =============================================================================
# Expose validated h1 recurrent states
# =============================================================================
#
# These are controlled validation outputs.
# They are runtime state tensors, not parameters.
# =============================================================================

NOTEBOOK_09_VALIDATED_SINGLE_STEP_RECURRENT_STATES = {
    pathway_name:
        state.to(
            DEVICE
        )

    for (
        pathway_name,
        state,
    ) in BLOCK_6_H1_STATES.items()
}


NOTEBOOK_09_FACTUAL_RECURRENT_STATE_T1 = (
    NOTEBOOK_09_VALIDATED_SINGLE_STEP_RECURRENT_STATES.get(
        "factual"
    )
)


NOTEBOOK_09_PSYCHOLOGICAL_RECURRENT_STATE_T1 = (
    NOTEBOOK_09_VALIDATED_SINGLE_STEP_RECURRENT_STATES.get(
        "psychological"
    )
)


NOTEBOOK_09_SOCIAL_RECURRENT_STATE_T1 = (
    NOTEBOOK_09_VALIDATED_SINGLE_STEP_RECURRENT_STATES.get(
        "social"
    )
)


# =============================================================================
# Final Block 6 validation
# =============================================================================

NOTEBOOK_09_BLOCK_6_ERRORS = []


BLOCK_6_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_6_PREREQUISITES_VALID,

    "pathway_dimensions_invalid":
        BLOCK_6_PATHWAY_DIMENSIONS_VALID,

    "probe_shapes_invalid":
        BLOCK_6_PROBE_SHAPES_VALID,

    "probes_not_finite":
        BLOCK_6_PROBES_FINITE,

    "candidate_shapes_invalid":
        BLOCK_6_CANDIDATE_SHAPES_VALID,

    "candidate_values_not_finite":
        BLOCK_6_ALL_CANDIDATES_FINITE,

    "candidate_forward_not_deterministic":
        BLOCK_6_ALL_CANDIDATES_DETERMINISTIC,

    "weight_shapes_invalid":
        BLOCK_6_EFFECTIVE_WEIGHT_SHAPES_VALID,

    "weight_values_not_finite":
        BLOCK_6_EFFECTIVE_WEIGHTS_FINITE,

    "weight_values_not_bounded":
        BLOCK_6_EFFECTIVE_WEIGHTS_BOUNDED,

    "weighted_candidate_shapes_invalid":
        BLOCK_6_WEIGHTED_CANDIDATE_SHAPES_VALID,

    "weighted_candidates_not_finite":
        BLOCK_6_ALL_WEIGHTED_CANDIDATES_FINITE,

    "active_scaling_invalid":
        BLOCK_6_ALL_ACTIVE_SCALING_VALID,

    "deferred_contributions_nonzero":
        BLOCK_6_ALL_DEFERRED_CONTRIBUTIONS_ZERO,

    "active_sign_not_preserved":
        BLOCK_6_ALL_ACTIVE_SIGNS_PRESERVED,

    "h1_shapes_invalid":
        BLOCK_6_H1_SHAPES_VALID,

    "h1_not_finite":
        BLOCK_6_ALL_H1_FINITE,

    "h1_weighted_candidate_identity_invalid":
        BLOCK_6_ALL_H1_EQUAL_WEIGHTED_CANDIDATE,

    "deferred_h1_nonzero":
        BLOCK_6_ALL_DEFERRED_H1_ZERO,

    "single_step_not_deterministic":
        BLOCK_6_COMPLETE_STEP_DETERMINISTIC,

    "h1_next_step_incompatible":
        BLOCK_6_ALL_H1_NEXT_STEP_COMPATIBLE,

    "pathway_independence_invalid":
        BLOCK_6_PATHWAYS_INDEPENDENT,

    "unexpected_gradients":
        BLOCK_6_ALL_GRADIENTS_ABSENT,

    "representation_state_changed":
        BLOCK_6_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_6_TRANSFORMATIVE_UNCHANGED,

    "weight_state_changed":
        BLOCK_6_WEIGHT_STATE_UNCHANGED,

    "representation_not_frozen":
        BLOCK_6_REPRESENTATION_STILL_FROZEN,

    "transformative_not_frozen":
        BLOCK_6_TRANSFORMATIVE_STILL_FROZEN,

    "weight_parameters_not_trainable":
        BLOCK_6_WEIGHT_PARAMETERS_STILL_TRAINABLE,

    "inherited_modules_not_eval":
        BLOCK_6_INHERITED_MODULES_STILL_EVAL,

    "parameter_accounting_invalid":
        BLOCK_6_PARAMETER_ACCOUNTING_VALID,

    "causal_contract_invalid":
        BLOCK_6_CAUSAL_CONTRACT_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_6_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_6_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation validation
# =============================================================================

BLOCK_6_PROHIBITED_OPERATIONS = {
    "representation_forward_incorrectly_executed":
        NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED,

    "second_recurrent_step_incorrectly_executed":
        NOTEBOOK_09_BLOCK_6_SECOND_RECURRENT_STEP_EXECUTED,

    "article_sequence_incorrectly_traversed":
        NOTEBOOK_09_BLOCK_6_ARTICLE_SEQUENCE_TRAVERSED,

    "future_context_incorrectly_used":
        NOTEBOOK_09_BLOCK_6_FUTURE_CONTEXT_USED,

    "loss_incorrectly_calculated":
        NOTEBOOK_09_BLOCK_6_LOSS_CALCULATED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_6_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_6_BACKWARD_PASS_EXECUTED,

    "gradient_clipping_incorrectly_executed":
        NOTEBOOK_09_BLOCK_6_GRADIENT_CLIPPING_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_6_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_6_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_6_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_6_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 6 state
# =============================================================================

NOTEBOOK_09_SINGLE_STEP_RECURRENT_FORWARD_VALID = (
    len(
        NOTEBOOK_09_BLOCK_6_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_MULTI_STEP_RECURRENT_VALIDATION_READY = (
    NOTEBOOK_09_SINGLE_STEP_RECURRENT_FORWARD_VALID
)


NOTEBOOK_09_BLOCK_6_VALID = (
    NOTEBOOK_09_SINGLE_STEP_RECURRENT_FORWARD_VALID
)


if not NOTEBOOK_09_BLOCK_6_VALID:

    raise RuntimeError(
        "Notebook 09 Block 6 controlled single-step recurrent "
        "forward-path validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_6_ERRORS}"
    )


NOTEBOOK_09_BLOCK_6_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_6_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_6,

    "block_name":
        NOTEBOOK_09_BLOCK_6_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_6_VERSION,

    "probe_shapes":
        deepcopy(
            BLOCK_6_PROBE_SHAPES
        ),

    "candidate_shapes":
        deepcopy(
            BLOCK_6_CANDIDATE_SHAPES
        ),

    "weight_shapes":
        deepcopy(
            BLOCK_6_EFFECTIVE_WEIGHT_SHAPES
        ),

    "weighted_candidate_shapes":
        deepcopy(
            BLOCK_6_WEIGHTED_CANDIDATE_SHAPES
        ),

    "h1_shapes":
        deepcopy(
            BLOCK_6_H1_SHAPES
        ),

    "candidate_deterministic":
        BLOCK_6_ALL_CANDIDATES_DETERMINISTIC,

    "complete_step_deterministic":
        BLOCK_6_COMPLETE_STEP_DETERMINISTIC,

    "active_scaling_valid":
        BLOCK_6_ALL_ACTIVE_SCALING_VALID,

    "deferred_contributions_zero":
        BLOCK_6_ALL_DEFERRED_CONTRIBUTIONS_ZERO,

    "h1_equals_weighted_candidate":
        BLOCK_6_ALL_H1_EQUAL_WEIGHTED_CANDIDATE,

    "h1_next_step_compatible":
        BLOCK_6_ALL_H1_NEXT_STEP_COMPATIBLE,

    "pathways_independent":
        BLOCK_6_PATHWAYS_INDEPENDENT,

    "transform_forward_executed":
        NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED,

    "weight_forward_executed":
        NOTEBOOK_09_BLOCK_6_WEIGHT_FORWARD_EXECUTED,

    "weighted_candidate_executed":
        NOTEBOOK_09_WEIGHTED_CANDIDATE_TRANSFORMATION_EXECUTED,

    "recurrent_update_executed":
        NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED,

    "h1_created":
        NOTEBOOK_09_RECURRENT_H1_CREATED,

    "second_step_executed":
        NOTEBOOK_09_BLOCK_6_SECOND_RECURRENT_STEP_EXECUTED,

    "single_step_valid":
        NOTEBOOK_09_SINGLE_STEP_RECURRENT_FORWARD_VALID,

    "multi_step_validation_ready":
        NOTEBOOK_09_MULTI_STEP_RECURRENT_VALIDATION_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_6_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_6_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 6: "
    "Controlled Single-Step Recurrent Forward-Path Validation"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_6_VERSION}"
)

print("-" * 72)

print(
    "Controlled current-representation probes"
)

for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} probe shape        : "
        f"{BLOCK_6_PROBE_SHAPES[pathway_name]}"
    )

print("-" * 72)

print(
    "Candidate-transformative forward validation"
)

for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} candidate shape    : "
        f"{BLOCK_6_CANDIDATE_SHAPES[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} candidate finite   : "
        f"{BLOCK_6_CANDIDATES_FINITE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deterministic      : "
        f"{BLOCK_6_CANDIDATE_DETERMINISTIC[pathway_name]}"
    )

print("-" * 72)

print(
    "Transformative-Weight execution"
)

for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} weight shape       : "
        f"{BLOCK_6_EFFECTIVE_WEIGHT_SHAPES[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} active scaling     : "
        f"{BLOCK_6_ACTIVE_SCALING_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deferred zero      : "
        f"{BLOCK_6_DEFERRED_CONTRIBUTIONS_ZERO[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} sign preserved     : "
        f"{BLOCK_6_ACTIVE_SIGN_PRESERVED[pathway_name]}"
    )

print("-" * 72)

print(
    "First recurrent-state outputs"
)

for pathway_name in BLOCK_6_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} h1 shape           : "
        f"{BLOCK_6_H1_SHAPES[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} h1 finite          : "
        f"{BLOCK_6_H1_FINITE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} h1 = weighted cand.: "
        f"{BLOCK_6_H1_EQUALS_WEIGHTED_CANDIDATE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deferred h1 zero   : "
        f"{BLOCK_6_DEFERRED_H1_ZERO[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} next-step compatible: "
        f"{BLOCK_6_H1_NEXT_STEP_COMPATIBLE[pathway_name]}"
    )

print("-" * 72)

print(
    "Parameter and state validation"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Transformative Weight params : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Total model parameters       : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print(
    f"Representation unchanged     : "
    f"{BLOCK_6_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged     : "
    f"{BLOCK_6_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Weight state unchanged       : "
    f"{BLOCK_6_WEIGHT_STATE_UNCHANGED}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_6_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_6_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"Weight parameters trainable  : "
    f"{BLOCK_6_WEIGHT_PARAMETERS_STILL_TRAINABLE}"
)

print(
    f"All gradients absent         : "
    f"{BLOCK_6_ALL_GRADIENTS_ABSENT}"
)

print(
    f"Pathways independent         : "
    f"{BLOCK_6_PATHWAYS_INDEPENDENT}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Representation forward       : "
    f"{NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED}"
)

print(
    f"Transform forward            : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED}"
)

print(
    f"Weight forward               : "
    f"{NOTEBOOK_09_BLOCK_6_WEIGHT_FORWARD_EXECUTED}"
)

print(
    f"Weighted candidate executed  : "
    f"{NOTEBOOK_09_WEIGHTED_CANDIDATE_TRANSFORMATION_EXECUTED}"
)

print(
    f"Recurrent update executed    : "
    f"{NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"h1 state created             : "
    f"{NOTEBOOK_09_RECURRENT_H1_CREATED}"
)

print(
    f"Second recurrent step        : "
    f"{NOTEBOOK_09_BLOCK_6_SECOND_RECURRENT_STEP_EXECUTED}"
)

print(
    f"Article sequence traversed   : "
    f"{NOTEBOOK_09_BLOCK_6_ARTICLE_SEQUENCE_TRAVERSED}"
)

print(
    f"Future context used          : "
    f"{NOTEBOOK_09_BLOCK_6_FUTURE_CONTEXT_USED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_BLOCK_6_LOSS_CALCULATED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_6_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_6_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_6_PARAMETER_UPDATE_EXECUTED}"
)

print("-" * 72)

print(
    f"Complete step deterministic  : "
    f"{BLOCK_6_COMPLETE_STEP_DETERMINISTIC}"
)

print(
    f"Single-step recurrent valid  : "
    f"{NOTEBOOK_09_SINGLE_STEP_RECURRENT_FORWARD_VALID}"
)

print(
    f"Multi-step validation ready  : "
    f"{NOTEBOOK_09_MULTI_STEP_RECURRENT_VALIDATION_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_6_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_6_COMPLETE}"
)

print("=" * 72)

print(
    f"The first complete recurrent transition was executed successfully "
    f"for all {len(BLOCK_6_PATHWAY_DIMENSIONS)} inherited pathway(s)."
)

print(
    "Each inherited transformative mechanism combined the controlled "
    "current representation with the deterministic zero preceding state "
    "and produced a finite deterministic candidate transformation."
)

print(
    "The validated Transformative Weights were applied elementwise to "
    "the candidate transformations."
)

print(
    "Active dimensions received the expected 0.5 candidate scaling, "
    "while all deferred dimensions produced exactly zero recurrent "
    "contribution."
)

print(
    "Because the preceding recurrent state was zero, each validated h1 "
    "state exactly equals its weighted candidate transformation."
)

print(
    "The resulting h1 tensors preserve the complete inherited pathway "
    "dimensions and are structurally compatible with a future recurrent step."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} parameters "
    "remain frozen and exactly unchanged, and all "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT} "
    "Transformative-Weight parameter(s) remain trainable and unchanged."
)

print(
    "No representation forward pass, second recurrent step, article "
    "sequence traversal, future-context access, loss, optimiser, backward "
    "pass or parameter update was executed."
)

print(
    "Notebook 09 may now proceed to controlled multi-step recurrent "
    "trajectory validation."
)

print("=" * 72)

Media AI — Notebook 09, Block 6: Controlled Single-Step Recurrent Forward-Path Validation
Block version                : 1.1
------------------------------------------------------------------------
Controlled current-representation probes
factual        probe shape        : (1, 10)
psychological  probe shape        : (1, 34)
social         probe shape        : (1, 7)
------------------------------------------------------------------------
Candidate-transformative forward validation
factual        candidate shape    : (1, 10)
factual        candidate finite   : True
factual        deterministic      : True
psychological  candidate shape    : (1, 34)
psychological  candidate finite   : True
psychological  deterministic      : True
social         candidate shape    : (1, 7)
social         candidate finite   : True
social         deterministic      : True
------------------------------------------------------------------------
Transformative-Weight execution
factual        weight shape    

## Block 7 — Controlled Multi-Step Recurrent Trajectory Validation

This block validates recurrent state propagation across multiple ordered steps.

Block 6 established the first complete recurrent transition by combining the deterministic zero initial state, a controlled current-representation probe, the inherited transformative mechanism, and the validated Transformative Weight. The resulting first recurrent states were finite, deterministic, pathway-consistent, and compatible with use as preceding-state inputs for a subsequent transition.

The present block extends that validation from a single transition to a short controlled recurrent trajectory.

Its purpose is to verify that recurrent state can be propagated causally across several ordered steps while preserving dimensionality, pathway independence, active/deferred gating behaviour, deterministic execution, numerical validity, and parameter immutability.

No article-derived sequence is required in this block. No loss, optimiser, backward pass, gradient clipping, parameter update, or training operation is executed.

### Multi-step recurrent setting

For each pathway \(k\in\{F,P,S\}\), the recurrent update remains

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)},
$$

with

$$
c_t^{(k)}
=
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right).
$$

The recurrent trajectory therefore follows

$$
h_0^{(k)}
\rightarrow
h_1^{(k)}
\rightarrow
h_2^{(k)}
\rightarrow
\cdots
\rightarrow
h_T^{(k)}.
$$

For controlled validation, Block 7 uses a short deterministic sequence of pathway-specific probe representations

$$
r_1^{(k)},
r_2^{(k)},
\ldots,
r_T^{(k)}.
$$

Each probe has the same pathway dimensionality as the recurrent state and candidate transformation.

The factual probes therefore satisfy

$$
r_t^{(F)}
\in
\mathbb{R}^{1\times10},
$$

the psychological probes satisfy

$$
r_t^{(P)}
\in
\mathbb{R}^{1\times34},
$$

and the social probes satisfy

$$
r_t^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

### Controlled trajectory length

The trajectory should be long enough to verify genuine recurrence but short enough to remain an architectural validation rather than a pilot-data experiment.

A fixed deterministic trajectory length

$$
T\geq3
$$

is sufficient to establish that:

- the first updated state can be reused correctly;
- the second state depends on the first;
- later states continue to satisfy the recurrent contract;
- no state reset occurs within the same trajectory;
- and no future state is used prematurely.

The same trajectory length must be used across factual, psychological, and social pathways.

### Recurrent dependency

The defining property of recurrence is that the current transition depends on the preceding recurrent state.

At step \(t\),

$$
c_t^{(k)}
=
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right).
$$

Thus the candidate transformation at step \(t\) is conditioned not only on the current representation but also on the accumulated state from earlier steps.

The block therefore validates that

$$
h_{t-1}^{(k)}
$$

from the immediately preceding transition is the state passed into the next transformative evaluation.

No reconstructed, reset, or future state may replace it.

### Causal trajectory construction

The complete recurrent trajectory is generated strictly in order.

For each pathway:

$$
h_0^{(k)}
\rightarrow
c_1^{(k)}
\rightarrow
h_1^{(k)}
\rightarrow
c_2^{(k)}
\rightarrow
h_2^{(k)}
\rightarrow
\cdots.
$$

At every step \(t\), only

$$
r_t^{(k)}
$$

and

$$
h_{t-1}^{(k)}
$$

are available.

Future probes

$$
r_{t+1}^{(k)},
r_{t+2}^{(k)},
\ldots
$$

must not influence the current state.

The multi-step trajectory therefore preserves the causal restriction

$$
h_t^{(k)}
=
f
\left(
r_1^{(k)},
r_2^{(k)},
\ldots,
r_t^{(k)}
\right).
$$

### Candidate-transformative trajectory

At every recurrent step, the inherited transformative mechanism produces a candidate transformation

$$
c_t^{(k)}
\in
\mathbb{R}^{1\times d_k}.
$$

The dimensions remain invariant throughout the trajectory:

$$
d_F=10,
\qquad
d_P=34,
\qquad
d_S=7.
$$

Each candidate transformation must remain finite.

Because the inherited transformative modules are frozen and evaluated deterministically, repeating the complete trajectory from the same initial state and the same probe sequence must reproduce identical candidate transformations at every step.

### Transformative-Weight application across steps

The same pathway-specific Transformative Weight is applied at every step of the controlled trajectory.

For the current initial parameter state,

$$
TW_{k,j}
=
0.5
\qquad
\text{for }j\in A_k,
$$

and

$$
TW_{k,j}
=
0
\qquad
\text{for }j\in D_k.
$$

Thus,

$$
\Delta h_{t,j}^{(k)}
=
0.5\,c_{t,j}^{(k)}
$$

for active dimensions and

$$
\Delta h_{t,j}^{(k)}
=
0
$$

for deferred dimensions.

The block validates that this gating remains stable across every recurrent step.

### Deferred-state invariance

The deferred dimensions require special attention in a multi-step setting.

Because

$$
TW_{k,j}=0
$$

for

$$
j\in D_k,
$$

the recurrent update becomes

$$
h_{t,j}^{(k)}
=
h_{t-1,j}^{(k)}.
$$

The deterministic initial state satisfies

$$
h_{0,j}^{(k)}=0.
$$

Therefore, under the current pilot contract,

$$
h_{t,j}^{(k)}=0
$$

for every controlled recurrent step and every deferred dimension.

Block 7 must verify this trajectory-wide invariant explicitly.

This confirms that deferred dimensions do not accumulate unintended state through repeated recurrence.

### Active-state accumulation

Active dimensions may evolve across multiple steps.

For

$$
j\in A_k,
$$

the update is

$$
h_{t,j}^{(k)}
=
h_{t-1,j}^{(k)}
+
0.5\,c_{t,j}^{(k)}.
$$

Therefore, unlike the first transition,

$$
h_t^{(k)}
$$

for

$$
t>1
$$

is not generally equal to the current weighted candidate alone.

Instead,

$$
h_t^{(k)}
=
\sum_{\tau=1}^{t}
TW_k\odot c_\tau^{(k)}
$$

under the additive recurrent contract.

This cumulative identity provides an important validation condition for the controlled trajectory.

### Cumulative-state identity

Starting from

$$
h_0^{(k)}=0,
$$

the recurrent update implies

$$
h_T^{(k)}
=
\sum_{t=1}^{T}
\Delta h_t^{(k)}.
$$

Since

$$
\Delta h_t^{(k)}
=
TW_k\odot c_t^{(k)},
$$

the final controlled recurrent state must satisfy

$$
h_T^{(k)}
=
\sum_{t=1}^{T}
TW_k\odot c_t^{(k)}.
$$

The block should verify this identity directly.

This provides a second validation of the recurrent implementation independent of the iterative update sequence.

### State-shape invariance

Every recurrent state must preserve the complete pathway dimension.

For all

$$
t\in\{0,\ldots,T\},
$$

the factual state must satisfy

$$
h_t^{(F)}
\in
\mathbb{R}^{1\times10},
$$

the psychological state

$$
h_t^{(P)}
\in
\mathbb{R}^{1\times34},
$$

and the social state

$$
h_t^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

No step may alter this dimensional contract.

This ensures that every state can be passed directly into the next inherited transformative mechanism.

### Numerical validity across the trajectory

Every intermediate quantity must remain finite.

For every pathway and recurrent step, the block validates

$$
\operatorname{isfinite}
\left(
c_t^{(k)}
\right),
$$

$$
\operatorname{isfinite}
\left(
\Delta h_t^{(k)}
\right),
$$

and

$$
\operatorname{isfinite}
\left(
h_t^{(k)}
\right).
$$

A short controlled trajectory is not intended to establish global numerical stability over arbitrarily long sequences, but it must demonstrate that recurrence itself does not immediately introduce invalid values.

### Trajectory determinism

The complete multi-step execution must be reproducible exactly.

If the same initial state, probe sequence, transformative modules, and Transformative Weights are used twice, then every intermediate state must satisfy

$$
h_{t,\mathrm{run1}}^{(k)}
=
h_{t,\mathrm{run2}}^{(k)}
$$

for all steps \(t\).

Likewise,

$$
c_{t,\mathrm{run1}}^{(k)}
=
c_{t,\mathrm{run2}}^{(k)}
$$

and

$$
\Delta h_{t,\mathrm{run1}}^{(k)}
=
\Delta h_{t,\mathrm{run2}}^{(k)}.
$$

This trajectory-level determinism is stricter than validating only the final state.

### Pathway independence

Factual, psychological, and social recurrent trajectories remain fully separate.

The factual trajectory depends only on

$$
T_F,
\quad
TW_F,
\quad
r_t^{(F)},
\quad
h_{t-1}^{(F)}.
$$

The psychological trajectory depends only on

$$
T_P,
\quad
TW_P,
\quad
r_t^{(P)},
\quad
h_{t-1}^{(P)}.
$$

The social trajectory depends only on

$$
T_S,
\quad
TW_S,
\quad
r_t^{(S)},
\quad
h_{t-1}^{(S)}.
$$

No recurrent state from one pathway may enter another pathway's update.

The block validates that module identity, weight identity, and state storage remain distinct across all three pathways.

### Initial-state reset reproducibility

A multi-step trajectory begins from the deterministic zero state defined in Block 5.

If the controlled trajectory is executed a second time, it must restart from a newly constructed zero state rather than from the final state of the first run.

Thus,

$$
h_{0,\mathrm{run2}}^{(k)}
=
\mathbf{0}_{d_k},
$$

not

$$
h_{T,\mathrm{run1}}^{(k)}.
$$

This explicitly validates the reset semantics required later at article boundaries.

### Distinction between validation trajectory and article trajectory

The controlled probe trajectory is an architectural test.

It is not yet the actual 16-sentence pilot article trajectory.

Its purpose is to validate recurrent mechanics under deterministic synthetic inputs before recurrence is applied to article-level pathway representations.

The block therefore does not interpret the resulting recurrent states semantically.

It validates structure, causality, determinism, gating behaviour, and numerical execution only.

### Parameter immutability

Multi-step recurrent execution must not modify learned parameters.

The inherited representation parameters must satisfy

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}},
$$

the inherited transformative parameters

$$
\theta_T^{\mathrm{after}}
=
\theta_T^{\mathrm{before}},
$$

and the Transformative-Weight parameters

$$
\theta_{TW}^{\mathrm{after}}
=
\theta_{TW}^{\mathrm{before}}.
$$

The complete model parameter count therefore remains

$$
902{,}215.
$$

Only recurrent runtime state evolves.

### Gradient boundary

The controlled trajectory is evaluated without gradient accumulation.

The 25 Transformative-Weight latent parameters remain trainable in principle, but no gradients are generated in this block.

The inherited representation and transformative parameters remain frozen.

Thus the block validates recurrence independently from recurrent optimisation.

### No representation forward path

As in Block 6, the controlled multi-step trajectory uses deterministic pathway-specific representation probes.

The representation model itself is not executed.

This isolates recurrent mechanics from representation generation and avoids conflating two architectural validation tasks.

### No optimisation in Block 7

The block performs no training.

Accordingly:

- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- no gradient clipping is applied;
- no optimiser step is executed;
- and no model parameter is updated.

The only evolving quantities are the recurrent runtime states.

### Block 7 completion contract

Block 7 is complete only when:

- a deterministic multi-step probe sequence is constructed for each pathway;
- the same controlled trajectory length is used across factual, psychological, and social pathways;
- recurrence begins from fresh deterministic zero states;
- candidate-transformative forward execution succeeds at every step;
- candidate transformations preserve the required pathway dimensions;
- all candidate transformations remain finite;
- Transformative-Weight execution succeeds at every step;
- weighted candidate contributions preserve the required dimensions;
- active dimensions remain correctly gated;
- deferred dimensions produce zero recurrent contribution at every step;
- deferred recurrent-state dimensions remain exactly zero throughout the trajectory;
- every updated recurrent state preserves shape \((1,d_k)\);
- every recurrent state remains finite;
- each updated state is used as the immediately following preceding state;
- the cumulative identity

$$
h_T^{(k)}
=
\sum_{t=1}^{T}
TW_k\odot c_t^{(k)}
$$

holds under the zero initial-state condition;

- repeated complete trajectory execution is exactly deterministic at every step;
- pathway independence is preserved;
- a repeated trajectory restarts from a fresh zero state;
- no future context is used;
- no article-derived sequence is traversed;
- the inherited 902,190 parameters remain frozen and exactly unchanged;
- the 25 Transformative-Weight parameters remain unchanged;
- the total model parameter count remains 902,215;
- no gradients are accumulated;
- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- and no parameter update occurs.

Successful completion validates the recurrent trajectory contract

$$
\boxed{
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right)
}
$$

across multiple ordered steps while preserving causal, pathway-specific state propagation.

Notebook 09 may then proceed from synthetic recurrent-mechanism validation to construction of the controlled article-level recurrent execution contract.

In [57]:
# =============================================================================
# Media AI — Notebook 09
# Block 7: Controlled Multi-Step Recurrent Trajectory Validation
# =============================================================================

from copy import deepcopy

import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_7 = 7

NOTEBOOK_09_BLOCK_7_NAME = (
    "Controlled Multi-Step Recurrent Trajectory Validation"
)

NOTEBOOK_09_BLOCK_7_VERSION = "1.1"


# =============================================================================
# Required inherited runtime contract
# =============================================================================

BLOCK_7_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_BLOCK_1_COMPLETE",
    "NOTEBOOK_09_BLOCK_1_VALID",
    "NOTEBOOK_09_HANDOVER_RESTORED",
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",
    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Blocks 2–5
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_2_COMPLETE",
    "NOTEBOOK_09_BLOCK_2_VALID",
    "NOTEBOOK_09_BLOCK_3_COMPLETE",
    "NOTEBOOK_09_BLOCK_3_VALID",
    "NOTEBOOK_09_BLOCK_4_COMPLETE",
    "NOTEBOOK_09_BLOCK_4_VALID",
    "NOTEBOOK_09_BLOCK_5_COMPLETE",
    "NOTEBOOK_09_BLOCK_5_VALID",

    # -------------------------------------------------------------------------
    # Block 6
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_6_COMPLETE",
    "NOTEBOOK_09_BLOCK_6_VALID",
    "NOTEBOOK_09_SINGLE_STEP_RECURRENT_FORWARD_VALID",
    "NOTEBOOK_09_MULTI_STEP_RECURRENT_VALIDATION_READY",

    # -------------------------------------------------------------------------
    # Recurrent architecture
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS",
    "NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES",
    "NOTEBOOK_09_INITIAL_RECURRENT_STATES",

    # -------------------------------------------------------------------------
    # Transformative Weight architecture
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE",

    # -------------------------------------------------------------------------
    # Environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_7_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_7_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_7_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 7 prerequisites are not initialised. "
        f"Missing: {BLOCK_7_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_7_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,

        NOTEBOOK_09_BLOCK_1_COMPLETE is True,
        NOTEBOOK_09_BLOCK_1_VALID is True,
        NOTEBOOK_09_HANDOVER_RESTORED is True,

        NOTEBOOK_09_BLOCK_2_COMPLETE is True,
        NOTEBOOK_09_BLOCK_2_VALID is True,

        NOTEBOOK_09_BLOCK_3_COMPLETE is True,
        NOTEBOOK_09_BLOCK_3_VALID is True,

        NOTEBOOK_09_BLOCK_4_COMPLETE is True,
        NOTEBOOK_09_BLOCK_4_VALID is True,

        NOTEBOOK_09_BLOCK_5_COMPLETE is True,
        NOTEBOOK_09_BLOCK_5_VALID is True,

        NOTEBOOK_09_BLOCK_6_COMPLETE is True,
        NOTEBOOK_09_BLOCK_6_VALID is True,
        NOTEBOOK_09_SINGLE_STEP_RECURRENT_FORWARD_VALID is True,
        NOTEBOOK_09_MULTI_STEP_RECURRENT_VALIDATION_READY is True,
    ]
)


if not BLOCK_7_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Blocks 1–6 must be valid and complete "
        "before multi-step recurrent trajectory validation."
    )


# =============================================================================
# Canonical pathway contract
# =============================================================================

BLOCK_7_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
)


BLOCK_7_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            BLOCK_7_PATHWAY_DIMENSIONS,
            dict,
        ),

        bool(
            BLOCK_7_PATHWAY_DIMENSIONS
        ),

        (
            BLOCK_7_PATHWAY_DIMENSIONS
            ==
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
        ),

        (
            set(
                BLOCK_7_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                BLOCK_1_TRANSFORMATIVE_MODULES.keys()
            )
            ==
            set(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys()
            )
        ),
    ]
)


if not BLOCK_7_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 7 pathway dimensions are invalid."
    )


# =============================================================================
# Controlled trajectory contract
# =============================================================================

NOTEBOOK_09_BLOCK_7_TRAJECTORY_LENGTH = 4


BLOCK_7_TRAJECTORY_LENGTH_VALID = (
    NOTEBOOK_09_BLOCK_7_TRAJECTORY_LENGTH
    >=
    3
)


if not BLOCK_7_TRAJECTORY_LENGTH_VALID:

    raise RuntimeError(
        "Controlled recurrent trajectory length must be at least 3."
    )


# =============================================================================
# Canonical pathway modules
# =============================================================================

BLOCK_7_TRANSFORMATIVE_MODULES = {
    pathway_name:
        BLOCK_1_TRANSFORMATIVE_MODULES[
            pathway_name
        ]

    for pathway_name
    in BLOCK_7_PATHWAY_DIMENSIONS
}


BLOCK_7_WEIGHT_MODULES = {
    pathway_name:
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
            pathway_name
        ]

    for pathway_name
    in BLOCK_7_PATHWAY_DIMENSIONS
}


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_7_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_7_nested_state_exact(
    state_before,
    state_after,
):

    if state_before.keys() != state_after.keys():

        return False


    for module_name in state_before:

        before_module_state = (
            state_before[
                module_name
            ]
        )


        after_module_state = (
            state_after[
                module_name
            ]
        )


        if (
            before_module_state.keys()
            !=
            after_module_state.keys()
        ):

            return False


        for tensor_name in before_module_state:

            if not torch.equal(
                before_module_state[
                    tensor_name
                ],
                after_module_state[
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot learned model state before trajectory validation
# =============================================================================

BLOCK_7_REPRESENTATION_STATE_BEFORE = (
    block_7_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_7_TRANSFORMATIVE_STATE_BEFORE = (
    block_7_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_7_WEIGHT_STATE_BEFORE = (
    block_7_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Deterministic multi-step probe sequence construction
# =============================================================================
#
# Each pathway receives the same number of ordered probe steps.
#
# Step-specific variation is deterministic and pathway-dimensionality
# preserving.
#
# These are architectural validation inputs only.
# =============================================================================

def block_7_construct_probe_sequence(
    pathway_dim,
    trajectory_length,
):

    pathway_dim = int(
        pathway_dim
    )


    base = torch.linspace(
        -0.5,
        0.5,
        steps=
            pathway_dim,
        dtype=
            DEFAULT_DTYPE,
        device=
            DEVICE,
    )


    probes = []


    for step_index in range(
        trajectory_length
    ):

        scale = (
            1.0
            +
            0.15
            *
            step_index
        )


        offset = (
            0.05
            *
            step_index
        )


        probe = (
            scale
            *
            base
            +
            offset
        ).reshape(
            1,
            pathway_dim,
        )


        probes.append(
            probe
        )


    return probes


BLOCK_7_PROBE_SEQUENCES = {
    pathway_name:
        block_7_construct_probe_sequence(
            pathway_dim=
                pathway_dim,

            trajectory_length=
                NOTEBOOK_09_BLOCK_7_TRAJECTORY_LENGTH,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_7_PATHWAY_DIMENSIONS.items()
}


# =============================================================================
# Probe-sequence validation
# =============================================================================

BLOCK_7_PROBE_SEQUENCE_LENGTHS = {
    pathway_name:
        len(
            probe_sequence
        )

    for (
        pathway_name,
        probe_sequence,
    ) in BLOCK_7_PROBE_SEQUENCES.items()
}


BLOCK_7_PROBE_SEQUENCE_LENGTHS_VALID = all(
    sequence_length
    ==
    NOTEBOOK_09_BLOCK_7_TRAJECTORY_LENGTH

    for sequence_length
    in BLOCK_7_PROBE_SEQUENCE_LENGTHS.values()
)


BLOCK_7_PROBE_SHAPES_VALID = {
    pathway_name:
        all(
            tuple(
                probe.shape
            )
            ==
            (
                1,
                BLOCK_7_PATHWAY_DIMENSIONS[
                    pathway_name
                ],
            )

            for probe
            in probe_sequence
        )

    for (
        pathway_name,
        probe_sequence,
    ) in BLOCK_7_PROBE_SEQUENCES.items()
}


BLOCK_7_ALL_PROBE_SHAPES_VALID = all(
    BLOCK_7_PROBE_SHAPES_VALID.values()
)


BLOCK_7_PROBES_FINITE = {
    pathway_name:
        all(
            bool(
                torch.isfinite(
                    probe
                ).all().item()
            )

            for probe
            in probe_sequence
        )

    for (
        pathway_name,
        probe_sequence,
    ) in BLOCK_7_PROBE_SEQUENCES.items()
}


BLOCK_7_ALL_PROBES_FINITE = all(
    BLOCK_7_PROBES_FINITE.values()
)


if not all(
    [
        BLOCK_7_PROBE_SEQUENCE_LENGTHS_VALID,
        BLOCK_7_ALL_PROBE_SHAPES_VALID,
        BLOCK_7_ALL_PROBES_FINITE,
    ]
):

    raise RuntimeError(
        "Controlled multi-step probe sequences are invalid."
    )


# =============================================================================
# Fresh initial-state construction
# =============================================================================
#
# Each trajectory run starts from a newly constructed zero state.
#
# This explicitly validates reset semantics.
# =============================================================================

def block_7_construct_fresh_initial_states():

    return {
        pathway_name:
            torch.zeros(
                (
                    1,
                    pathway_dim,
                ),
                dtype=
                    DEFAULT_DTYPE,
                device=
                    DEVICE,
                requires_grad=
                    False,
            )

        for (
            pathway_name,
            pathway_dim,
        ) in BLOCK_7_PATHWAY_DIMENSIONS.items()
    }


# =============================================================================
# Controlled trajectory execution helper
# =============================================================================

def block_7_execute_trajectory():
    """
    Execute a complete controlled recurrent trajectory independently
    for factual, psychological and social pathways.

    At each step:

        candidate_t =
            T_k(
                current_representation_t,
                previous_state_t,
            )

        weight =
            TW_k()

        weighted_candidate_t =
            weight * candidate_t

        updated_state_t =
            previous_state_t + weighted_candidate_t

    No gradients are retained.
    """

    initial_states = (
        block_7_construct_fresh_initial_states()
    )


    trajectories = {}


    for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

        transformative_module = (
            BLOCK_7_TRANSFORMATIVE_MODULES[
                pathway_name
            ]
        )


        weight_module = (
            BLOCK_7_WEIGHT_MODULES[
                pathway_name
            ]
        )


        previous_state = (
            initial_states[
                pathway_name
            ]
        )


        pathway_steps = []


        for step_index in range(
            NOTEBOOK_09_BLOCK_7_TRAJECTORY_LENGTH
        ):

            current_representation = (
                BLOCK_7_PROBE_SEQUENCES[
                    pathway_name
                ][
                    step_index
                ]
            )


            candidate = transformative_module(
                current_representation,
                previous_state,
            )


            effective_weight = (
                weight_module()
            )


            weighted_candidate = (
                effective_weight.unsqueeze(
                    0
                )
                *
                candidate
            )


            updated_state = (
                previous_state
                +
                weighted_candidate
            )


            pathway_steps.append(
                {
                    "step":
                        step_index
                        +
                        1,

                    "current_representation":
                        current_representation.detach()
                        .clone(),

                    "previous_state":
                        previous_state.detach()
                        .clone(),

                    "candidate":
                        candidate.detach()
                        .clone(),

                    "effective_weight":
                        effective_weight.detach()
                        .clone(),

                    "weighted_candidate":
                        weighted_candidate.detach()
                        .clone(),

                    "updated_state":
                        updated_state.detach()
                        .clone(),
                }
            )


            previous_state = (
                updated_state.detach()
                .clone()
            )


        trajectories[
            pathway_name
        ] = {
            "initial_state":
                initial_states[
                    pathway_name
                ].detach()
                .clone(),

            "steps":
                pathway_steps,

            "final_state":
                previous_state.detach()
                .clone(),
        }


    return trajectories


# =============================================================================
# Execute complete trajectory twice
# =============================================================================

with torch.no_grad():

    BLOCK_7_FIRST_TRAJECTORY = (
        block_7_execute_trajectory()
    )


    BLOCK_7_SECOND_TRAJECTORY = (
        block_7_execute_trajectory()
    )


# =============================================================================
# Execution flags
# =============================================================================

NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED = False

NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED = True

NOTEBOOK_09_BLOCK_7_WEIGHT_FORWARD_EXECUTED = True

NOTEBOOK_09_WEIGHTED_CANDIDATE_TRANSFORMATION_EXECUTED = True

NOTEBOOK_09_RECURRENT_UPDATE_EXECUTED = True

NOTEBOOK_09_BLOCK_7_MULTI_STEP_RECURRENT_EXECUTED = True

NOTEBOOK_09_BLOCK_7_ARTICLE_SEQUENCE_TRAVERSED = False

NOTEBOOK_09_BLOCK_7_FUTURE_CONTEXT_USED = False


# =============================================================================
# Initial-state reset validation
# =============================================================================

BLOCK_7_INITIAL_STATES_ZERO = {}


BLOCK_7_INITIAL_STATE_RUNS_EQUAL = {}


BLOCK_7_INITIAL_STATE_STORAGE_INDEPENDENT = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    initial_first = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ][
            "initial_state"
        ]
    )


    initial_second = (
        BLOCK_7_SECOND_TRAJECTORY[
            pathway_name
        ][
            "initial_state"
        ]
    )


    BLOCK_7_INITIAL_STATES_ZERO[
        pathway_name
    ] = all(
        [
            torch.equal(
                initial_first,
                torch.zeros_like(
                    initial_first
                ),
            ),

            torch.equal(
                initial_second,
                torch.zeros_like(
                    initial_second
                ),
            ),
        ]
    )


    BLOCK_7_INITIAL_STATE_RUNS_EQUAL[
        pathway_name
    ] = torch.equal(
        initial_first,
        initial_second,
    )


    BLOCK_7_INITIAL_STATE_STORAGE_INDEPENDENT[
        pathway_name
    ] = (
        initial_first.data_ptr()
        !=
        initial_second.data_ptr()
    )


BLOCK_7_ALL_INITIAL_STATES_ZERO = all(
    BLOCK_7_INITIAL_STATES_ZERO.values()
)


BLOCK_7_ALL_INITIAL_STATE_RUNS_EQUAL = all(
    BLOCK_7_INITIAL_STATE_RUNS_EQUAL.values()
)


BLOCK_7_ALL_INITIAL_STATE_STORAGE_INDEPENDENT = all(
    BLOCK_7_INITIAL_STATE_STORAGE_INDEPENDENT.values()
)


if not all(
    [
        BLOCK_7_ALL_INITIAL_STATES_ZERO,
        BLOCK_7_ALL_INITIAL_STATE_RUNS_EQUAL,
        BLOCK_7_ALL_INITIAL_STATE_STORAGE_INDEPENDENT,
    ]
):

    raise RuntimeError(
        "Controlled recurrent trajectory reset semantics are invalid."
    )


# =============================================================================
# Step-count validation
# =============================================================================

BLOCK_7_TRAJECTORY_STEP_COUNTS = {
    pathway_name:
        len(
            trajectory[
                "steps"
            ]
        )

    for (
        pathway_name,
        trajectory,
    ) in BLOCK_7_FIRST_TRAJECTORY.items()
}


BLOCK_7_TRAJECTORY_STEP_COUNTS_VALID = all(
    step_count
    ==
    NOTEBOOK_09_BLOCK_7_TRAJECTORY_LENGTH

    for step_count
    in BLOCK_7_TRAJECTORY_STEP_COUNTS.values()
)


if not BLOCK_7_TRAJECTORY_STEP_COUNTS_VALID:

    raise RuntimeError(
        "Controlled recurrent trajectory step count is invalid."
    )


# =============================================================================
# Candidate shape and numerical validation across trajectory
# =============================================================================

BLOCK_7_CANDIDATE_SHAPES_VALID = {}

BLOCK_7_CANDIDATES_FINITE = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    expected_shape = (
        1,
        BLOCK_7_PATHWAY_DIMENSIONS[
            pathway_name
        ],
    )


    steps = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ][
            "steps"
        ]
    )


    BLOCK_7_CANDIDATE_SHAPES_VALID[
        pathway_name
    ] = all(
        tuple(
            step[
                "candidate"
            ].shape
        )
        ==
        expected_shape

        for step
        in steps
    )


    BLOCK_7_CANDIDATES_FINITE[
        pathway_name
    ] = all(
        bool(
            torch.isfinite(
                step[
                    "candidate"
                ]
            ).all().item()
        )

        for step
        in steps
    )


BLOCK_7_ALL_CANDIDATE_SHAPES_VALID = all(
    BLOCK_7_CANDIDATE_SHAPES_VALID.values()
)


BLOCK_7_ALL_CANDIDATES_FINITE = all(
    BLOCK_7_CANDIDATES_FINITE.values()
)


if not all(
    [
        BLOCK_7_ALL_CANDIDATE_SHAPES_VALID,
        BLOCK_7_ALL_CANDIDATES_FINITE,
    ]
):

    raise RuntimeError(
        "Multi-step candidate transformations are invalid."
    )


# =============================================================================
# Effective-weight stability validation across trajectory
# =============================================================================

BLOCK_7_WEIGHT_SHAPES_VALID = {}

BLOCK_7_WEIGHTS_FINITE = {}

BLOCK_7_WEIGHTS_BOUNDED = {}

BLOCK_7_WEIGHTS_CONSTANT_ACROSS_STEPS = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    steps = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ][
            "steps"
        ]
    )


    expected_shape = (
        BLOCK_7_PATHWAY_DIMENSIONS[
            pathway_name
        ],
    )


    first_weight = (
        steps[
            0
        ][
            "effective_weight"
        ]
    )


    BLOCK_7_WEIGHT_SHAPES_VALID[
        pathway_name
    ] = all(
        tuple(
            step[
                "effective_weight"
            ].shape
        )
        ==
        expected_shape

        for step
        in steps
    )


    BLOCK_7_WEIGHTS_FINITE[
        pathway_name
    ] = all(
        bool(
            torch.isfinite(
                step[
                    "effective_weight"
                ]
            ).all().item()
        )

        for step
        in steps
    )


    BLOCK_7_WEIGHTS_BOUNDED[
        pathway_name
    ] = all(
        bool(
            (
                (
                    step[
                        "effective_weight"
                    ]
                    >=
                    0.0
                )
                &
                (
                    step[
                        "effective_weight"
                    ]
                    <=
                    1.0
                )
            ).all().item()
        )

        for step
        in steps
    )


    BLOCK_7_WEIGHTS_CONSTANT_ACROSS_STEPS[
        pathway_name
    ] = all(
        torch.equal(
            step[
                "effective_weight"
            ],
            first_weight,
        )

        for step
        in steps[
            1:
        ]
    )


BLOCK_7_ALL_WEIGHT_SHAPES_VALID = all(
    BLOCK_7_WEIGHT_SHAPES_VALID.values()
)


BLOCK_7_ALL_WEIGHTS_FINITE = all(
    BLOCK_7_WEIGHTS_FINITE.values()
)


BLOCK_7_ALL_WEIGHTS_BOUNDED = all(
    BLOCK_7_WEIGHTS_BOUNDED.values()
)


BLOCK_7_ALL_WEIGHTS_CONSTANT_ACROSS_STEPS = all(
    BLOCK_7_WEIGHTS_CONSTANT_ACROSS_STEPS.values()
)


if not all(
    [
        BLOCK_7_ALL_WEIGHT_SHAPES_VALID,
        BLOCK_7_ALL_WEIGHTS_FINITE,
        BLOCK_7_ALL_WEIGHTS_BOUNDED,
        BLOCK_7_ALL_WEIGHTS_CONSTANT_ACROSS_STEPS,
    ]
):

    raise RuntimeError(
        "Transformative-Weight behaviour is invalid across "
        "the controlled trajectory."
    )


# =============================================================================
# Active/deferred weight contract across trajectory
# =============================================================================

BLOCK_7_ACTIVE_WEIGHT_VALUES_VALID = {}

BLOCK_7_DEFERRED_WEIGHT_VALUES_VALID = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    active_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    deferred_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    steps = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ][
            "steps"
        ]
    )


    active_valid = True

    deferred_valid = True


    for step in steps:

        weight = (
            step[
                "effective_weight"
            ]
        )


        active_values = (
            weight[
                active_indices
            ]
        )


        deferred_values = (
            weight[
                deferred_indices
            ]
        )


        expected_active = torch.full(
            active_values.shape,
            fill_value=
                float(
                    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE
                ),
            dtype=
                active_values.dtype,
        )


        expected_deferred = torch.full(
            deferred_values.shape,
            fill_value=
                float(
                    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_EFFECTIVE
                ),
            dtype=
                deferred_values.dtype,
        )


        active_valid = (
            active_valid
            and
            torch.equal(
                active_values,
                expected_active,
            )
        )


        deferred_valid = (
            deferred_valid
            and
            torch.equal(
                deferred_values,
                expected_deferred,
            )
        )


    BLOCK_7_ACTIVE_WEIGHT_VALUES_VALID[
        pathway_name
    ] = active_valid


    BLOCK_7_DEFERRED_WEIGHT_VALUES_VALID[
        pathway_name
    ] = deferred_valid


BLOCK_7_ALL_ACTIVE_WEIGHT_VALUES_VALID = all(
    BLOCK_7_ACTIVE_WEIGHT_VALUES_VALID.values()
)


BLOCK_7_ALL_DEFERRED_WEIGHT_VALUES_VALID = all(
    BLOCK_7_DEFERRED_WEIGHT_VALUES_VALID.values()
)


if not all(
    [
        BLOCK_7_ALL_ACTIVE_WEIGHT_VALUES_VALID,
        BLOCK_7_ALL_DEFERRED_WEIGHT_VALUES_VALID,
    ]
):

    raise RuntimeError(
        "Active/deferred Transformative-Weight values changed "
        "during controlled recurrent execution."
    )


# =============================================================================
# Weighted-candidate validation across trajectory
# =============================================================================

BLOCK_7_WEIGHTED_CANDIDATE_SHAPES_VALID = {}

BLOCK_7_WEIGHTED_CANDIDATES_FINITE = {}

BLOCK_7_ACTIVE_SCALING_VALID = {}

BLOCK_7_DEFERRED_CONTRIBUTIONS_ZERO = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    expected_shape = (
        1,
        BLOCK_7_PATHWAY_DIMENSIONS[
            pathway_name
        ],
    )


    active_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    deferred_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    steps = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ][
            "steps"
        ]
    )


    shape_valid = True

    finite_valid = True

    active_scaling_valid = True

    deferred_zero_valid = True


    for step in steps:

        candidate = (
            step[
                "candidate"
            ]
        )


        weighted_candidate = (
            step[
                "weighted_candidate"
            ]
        )


        shape_valid = (
            shape_valid
            and
            tuple(
                weighted_candidate.shape
            )
            ==
            expected_shape
        )


        finite_valid = (
            finite_valid
            and
            bool(
                torch.isfinite(
                    weighted_candidate
                ).all().item()
            )
        )


        candidate_active = (
            candidate[
                :,
                active_indices
            ]
        )


        weighted_active = (
            weighted_candidate[
                :,
                active_indices
            ]
        )


        expected_active = (
            float(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_ACTIVE_INITIAL_EFFECTIVE
            )
            *
            candidate_active
        )


        active_scaling_valid = (
            active_scaling_valid
            and
            torch.equal(
                weighted_active,
                expected_active,
            )
        )


        weighted_deferred = (
            weighted_candidate[
                :,
                deferred_indices
            ]
        )


        deferred_zero_valid = (
            deferred_zero_valid
            and
            torch.equal(
                weighted_deferred,
                torch.zeros_like(
                    weighted_deferred
                ),
            )
        )


    BLOCK_7_WEIGHTED_CANDIDATE_SHAPES_VALID[
        pathway_name
    ] = shape_valid


    BLOCK_7_WEIGHTED_CANDIDATES_FINITE[
        pathway_name
    ] = finite_valid


    BLOCK_7_ACTIVE_SCALING_VALID[
        pathway_name
    ] = active_scaling_valid


    BLOCK_7_DEFERRED_CONTRIBUTIONS_ZERO[
        pathway_name
    ] = deferred_zero_valid


BLOCK_7_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID = all(
    BLOCK_7_WEIGHTED_CANDIDATE_SHAPES_VALID.values()
)


BLOCK_7_ALL_WEIGHTED_CANDIDATES_FINITE = all(
    BLOCK_7_WEIGHTED_CANDIDATES_FINITE.values()
)


BLOCK_7_ALL_ACTIVE_SCALING_VALID = all(
    BLOCK_7_ACTIVE_SCALING_VALID.values()
)


BLOCK_7_ALL_DEFERRED_CONTRIBUTIONS_ZERO = all(
    BLOCK_7_DEFERRED_CONTRIBUTIONS_ZERO.values()
)


if not all(
    [
        BLOCK_7_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID,
        BLOCK_7_ALL_WEIGHTED_CANDIDATES_FINITE,
        BLOCK_7_ALL_ACTIVE_SCALING_VALID,
        BLOCK_7_ALL_DEFERRED_CONTRIBUTIONS_ZERO,
    ]
):

    raise RuntimeError(
        "Weighted recurrent contributions are invalid across "
        "the controlled trajectory."
    )


# =============================================================================
# Recurrent state shape and numerical validation
# =============================================================================

BLOCK_7_STATE_SHAPES_VALID = {}

BLOCK_7_STATES_FINITE = {}

BLOCK_7_DEFERRED_STATES_ZERO = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    expected_shape = (
        1,
        BLOCK_7_PATHWAY_DIMENSIONS[
            pathway_name
        ],
    )


    deferred_indices = torch.tensor(
        NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    trajectory = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ]
    )


    states = [
        trajectory[
            "initial_state"
        ]
    ] + [
        step[
            "updated_state"
        ]

        for step
        in trajectory[
            "steps"
        ]
    ]


    BLOCK_7_STATE_SHAPES_VALID[
        pathway_name
    ] = all(
        tuple(
            state.shape
        )
        ==
        expected_shape

        for state
        in states
    )


    BLOCK_7_STATES_FINITE[
        pathway_name
    ] = all(
        bool(
            torch.isfinite(
                state
            ).all().item()
        )

        for state
        in states
    )


    BLOCK_7_DEFERRED_STATES_ZERO[
        pathway_name
    ] = all(
        torch.equal(
            state[
                :,
                deferred_indices
            ],
            torch.zeros_like(
                state[
                    :,
                    deferred_indices
                ]
            ),
        )

        for state
        in states
    )


BLOCK_7_ALL_STATE_SHAPES_VALID = all(
    BLOCK_7_STATE_SHAPES_VALID.values()
)


BLOCK_7_ALL_STATES_FINITE = all(
    BLOCK_7_STATES_FINITE.values()
)


BLOCK_7_ALL_DEFERRED_STATES_ZERO = all(
    BLOCK_7_DEFERRED_STATES_ZERO.values()
)


if not all(
    [
        BLOCK_7_ALL_STATE_SHAPES_VALID,
        BLOCK_7_ALL_STATES_FINITE,
        BLOCK_7_ALL_DEFERRED_STATES_ZERO,
    ]
):

    raise RuntimeError(
        "Recurrent trajectory state validation failed."
    )


# =============================================================================
# Previous-state chaining validation
# =============================================================================
#
# Every step t > 1 must receive the immediately preceding updated state.
# =============================================================================

BLOCK_7_PREVIOUS_STATE_CHAIN_VALID = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    trajectory = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ]
    )


    steps = (
        trajectory[
            "steps"
        ]
    )


    chain_valid = torch.equal(
        steps[
            0
        ][
            "previous_state"
        ],
        trajectory[
            "initial_state"
        ],
    )


    for step_index in range(
        1,
        len(
            steps
        ),
    ):

        chain_valid = (
            chain_valid
            and
            torch.equal(
                steps[
                    step_index
                ][
                    "previous_state"
                ],
                steps[
                    step_index
                    -
                    1
                ][
                    "updated_state"
                ],
            )
        )


    BLOCK_7_PREVIOUS_STATE_CHAIN_VALID[
        pathway_name
    ] = chain_valid


BLOCK_7_ALL_PREVIOUS_STATE_CHAINS_VALID = all(
    BLOCK_7_PREVIOUS_STATE_CHAIN_VALID.values()
)


if not BLOCK_7_ALL_PREVIOUS_STATE_CHAINS_VALID:

    raise RuntimeError(
        "Recurrent trajectory does not propagate the immediately "
        "preceding state correctly."
    )


# =============================================================================
# Stepwise additive update identity
# =============================================================================

BLOCK_7_STEPWISE_UPDATE_IDENTITY_VALID = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    steps = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ][
            "steps"
        ]
    )


    BLOCK_7_STEPWISE_UPDATE_IDENTITY_VALID[
        pathway_name
    ] = all(
        torch.equal(
            step[
                "updated_state"
            ],
            (
                step[
                    "previous_state"
                ]
                +
                step[
                    "weighted_candidate"
                ]
            ),
        )

        for step
        in steps
    )


BLOCK_7_ALL_STEPWISE_UPDATE_IDENTITIES_VALID = all(
    BLOCK_7_STEPWISE_UPDATE_IDENTITY_VALID.values()
)


if not BLOCK_7_ALL_STEPWISE_UPDATE_IDENTITIES_VALID:

    raise RuntimeError(
        "Recurrent additive state-update identity failed."
    )


# =============================================================================
# Cumulative-state identity validation
# =============================================================================

BLOCK_7_CUMULATIVE_STATE_IDENTITY_VALID = {}

BLOCK_7_CUMULATIVE_STATE_MAX_ABS_DIFFERENCE = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    trajectory = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ]
    )


    cumulative_weighted_candidate = torch.zeros_like(
        trajectory[
            "initial_state"
        ]
    )


    for step in trajectory[
        "steps"
    ]:

        cumulative_weighted_candidate = (
            cumulative_weighted_candidate
            +
            step[
                "weighted_candidate"
            ]
        )


    final_state = (
        trajectory[
            "final_state"
        ]
    )


    difference = (
        final_state
        -
        cumulative_weighted_candidate
    )


    max_abs_difference = float(
        difference.abs()
        .max()
        .item()
    )


    BLOCK_7_CUMULATIVE_STATE_MAX_ABS_DIFFERENCE[
        pathway_name
    ] = max_abs_difference


    BLOCK_7_CUMULATIVE_STATE_IDENTITY_VALID[
        pathway_name
    ] = torch.allclose(
        final_state,
        cumulative_weighted_candidate,
        rtol=
            0.0,
        atol=
            1e-7,
    )


BLOCK_7_ALL_CUMULATIVE_STATE_IDENTITIES_VALID = all(
    BLOCK_7_CUMULATIVE_STATE_IDENTITY_VALID.values()
)


if not BLOCK_7_ALL_CUMULATIVE_STATE_IDENTITIES_VALID:

    raise RuntimeError(
        "Final recurrent state does not equal the cumulative "
        "weighted transformation trajectory."
    )


# =============================================================================
# Final-state compatibility validation
# =============================================================================

BLOCK_7_FINAL_STATE_NEXT_STEP_COMPATIBLE = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    final_state = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ][
            "final_state"
        ]
    )


    BLOCK_7_FINAL_STATE_NEXT_STEP_COMPATIBLE[
        pathway_name
    ] = all(
        [
            (
                tuple(
                    final_state.shape
                )
                ==
                (
                    1,
                    BLOCK_7_PATHWAY_DIMENSIONS[
                        pathway_name
                    ],
                )
            ),

            (
                final_state.dtype
                ==
                DEFAULT_DTYPE
            ),

            (
                final_state.device
                ==
                DEVICE
            ),

            (
                final_state.requires_grad
                is False
            ),
        ]
    )


BLOCK_7_ALL_FINAL_STATES_NEXT_STEP_COMPATIBLE = all(
    BLOCK_7_FINAL_STATE_NEXT_STEP_COMPATIBLE.values()
)


if not BLOCK_7_ALL_FINAL_STATES_NEXT_STEP_COMPATIBLE:

    raise RuntimeError(
        "One or more final controlled recurrent states are not "
        "compatible with another recurrent step."
    )


# =============================================================================
# Complete trajectory determinism
# =============================================================================

BLOCK_7_TRAJECTORY_DETERMINISTIC = {}


for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    first = (
        BLOCK_7_FIRST_TRAJECTORY[
            pathway_name
        ]
    )


    second = (
        BLOCK_7_SECOND_TRAJECTORY[
            pathway_name
        ]
    )


    deterministic = torch.equal(
        first[
            "initial_state"
        ],
        second[
            "initial_state"
        ],
    )


    for (
        first_step,
        second_step,
    ) in zip(
        first[
            "steps"
        ],
        second[
            "steps"
        ],
    ):

        for tensor_name in (
            "current_representation",
            "previous_state",
            "candidate",
            "effective_weight",
            "weighted_candidate",
            "updated_state",
        ):

            deterministic = (
                deterministic
                and
                torch.equal(
                    first_step[
                        tensor_name
                    ],
                    second_step[
                        tensor_name
                    ],
                )
            )


    deterministic = (
        deterministic
        and
        torch.equal(
            first[
                "final_state"
            ],
            second[
                "final_state"
            ],
        )
    )


    BLOCK_7_TRAJECTORY_DETERMINISTIC[
        pathway_name
    ] = deterministic


BLOCK_7_ALL_TRAJECTORIES_DETERMINISTIC = all(
    BLOCK_7_TRAJECTORY_DETERMINISTIC.values()
)


if not BLOCK_7_ALL_TRAJECTORIES_DETERMINISTIC:

    raise RuntimeError(
        "Controlled recurrent trajectories are not exactly deterministic."
    )


# =============================================================================
# Pathway independence validation
# =============================================================================

BLOCK_7_TRANSFORMATIVE_MODULE_IDS = {
    pathway_name:
        id(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_7_TRANSFORMATIVE_MODULES.items()
}


BLOCK_7_WEIGHT_MODULE_IDS = {
    pathway_name:
        id(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_7_WEIGHT_MODULES.items()
}


BLOCK_7_FIRST_INITIAL_STATE_STORAGE_IDS = {
    pathway_name:
        int(
            trajectory[
                "initial_state"
            ].data_ptr()
        )

    for (
        pathway_name,
        trajectory,
    ) in BLOCK_7_FIRST_TRAJECTORY.items()
}


BLOCK_7_PATHWAY_COUNT = len(
    BLOCK_7_PATHWAY_DIMENSIONS
)


BLOCK_7_PATHWAYS_INDEPENDENT = all(
    [
        (
            len(
                set(
                    BLOCK_7_TRANSFORMATIVE_MODULE_IDS.values()
                )
            )
            ==
            BLOCK_7_PATHWAY_COUNT
        ),

        (
            len(
                set(
                    BLOCK_7_WEIGHT_MODULE_IDS.values()
                )
            )
            ==
            BLOCK_7_PATHWAY_COUNT
        ),

        (
            len(
                set(
                    BLOCK_7_FIRST_INITIAL_STATE_STORAGE_IDS.values()
                )
            )
            ==
            BLOCK_7_PATHWAY_COUNT
        ),
    ]
)


if not BLOCK_7_PATHWAYS_INDEPENDENT:

    raise RuntimeError(
        "Multi-step recurrent pathways are not structurally independent."
    )


# =============================================================================
# Gradient-state validation
# =============================================================================

BLOCK_7_REPRESENTATION_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_7_TRANSFORMATIVE_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_7_WEIGHT_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_7_ALL_GRADIENTS_ABSENT = all(
    [
        BLOCK_7_REPRESENTATION_GRADIENTS_ABSENT,
        BLOCK_7_TRANSFORMATIVE_GRADIENTS_ABSENT,
        BLOCK_7_WEIGHT_GRADIENTS_ABSENT,
    ]
)


if not BLOCK_7_ALL_GRADIENTS_ABSENT:

    raise RuntimeError(
        "Unexpected gradients were created during multi-step "
        "recurrent trajectory validation."
    )


# =============================================================================
# Snapshot learned model state after trajectory validation
# =============================================================================

BLOCK_7_REPRESENTATION_STATE_AFTER = (
    block_7_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_7_TRANSFORMATIVE_STATE_AFTER = (
    block_7_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_7_WEIGHT_STATE_AFTER = (
    block_7_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Parameter immutability validation
# =============================================================================

BLOCK_7_REPRESENTATION_UNCHANGED = (
    block_7_nested_state_exact(
        BLOCK_7_REPRESENTATION_STATE_BEFORE,
        BLOCK_7_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_7_TRANSFORMATIVE_UNCHANGED = (
    block_7_nested_state_exact(
        BLOCK_7_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_7_TRANSFORMATIVE_STATE_AFTER,
    )
)


BLOCK_7_WEIGHT_STATE_UNCHANGED = (
    block_7_nested_state_exact(
        BLOCK_7_WEIGHT_STATE_BEFORE,
        BLOCK_7_WEIGHT_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_7_REPRESENTATION_UNCHANGED,
        BLOCK_7_TRANSFORMATIVE_UNCHANGED,
        BLOCK_7_WEIGHT_STATE_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Model parameter state changed during controlled "
        "multi-step recurrent validation."
    )


# =============================================================================
# Trainability and evaluation-mode validation
# =============================================================================

BLOCK_7_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_7_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_7_WEIGHT_PARAMETERS_STILL_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_7_INHERITED_MODULES_STILL_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


if not all(
    [
        BLOCK_7_REPRESENTATION_STILL_FROZEN,
        BLOCK_7_TRANSFORMATIVE_STILL_FROZEN,
        BLOCK_7_WEIGHT_PARAMETERS_STILL_TRAINABLE,
        BLOCK_7_INHERITED_MODULES_STILL_EVAL,
    ]
):

    raise RuntimeError(
        "Notebook 09 trainability or inherited evaluation-mode scope "
        "changed during Block 7."
    )


# =============================================================================
# Parameter-accounting validation
# =============================================================================

BLOCK_7_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_STRUCTURAL_DIMENSIONS
            ==
            sum(
                BLOCK_7_PATHWAY_DIMENSIONS.values()
            )
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_DEFERRED_DIMENSION_COUNT
            ==
            sum(
                len(
                    NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES[
                        pathway_name
                    ]
                )

                for pathway_name
                in BLOCK_7_PATHWAY_DIMENSIONS
            )
        ),
    ]
)


if not BLOCK_7_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 parameter accounting changed during "
        "multi-step recurrent trajectory validation."
    )


# =============================================================================
# Causal execution boundary
# =============================================================================

BLOCK_7_CAUSAL_CONTRACT_VALID = all(
    [
        NOTEBOOK_09_BLOCK_7_FUTURE_CONTEXT_USED is False,
        NOTEBOOK_09_BLOCK_7_ARTICLE_SEQUENCE_TRAVERSED is False,
        NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED is False,
    ]
)


if not BLOCK_7_CAUSAL_CONTRACT_VALID:

    raise RuntimeError(
        "Controlled multi-step recurrent execution violated "
        "the causal validation boundary."
    )


# =============================================================================
# Explicit optimisation boundary
# =============================================================================

NOTEBOOK_09_BLOCK_7_LOSS_CALCULATED = False

NOTEBOOK_09_BLOCK_7_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_7_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_7_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_09_BLOCK_7_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_7_TRAINING_EXECUTED = False


# =============================================================================
# Expose validated controlled trajectories
# =============================================================================

NOTEBOOK_09_VALIDATED_CONTROLLED_RECURRENT_TRAJECTORIES = {
    pathway_name:
        {
            "initial_state":
                trajectory[
                    "initial_state"
                ].detach()
                .cpu()
                .clone(),

            "steps":
                [
                    {
                        tensor_name:
                            (
                                tensor_value.detach()
                                .cpu()
                                .clone()

                                if torch.is_tensor(
                                    tensor_value
                                )

                                else tensor_value
                            )

                        for (
                            tensor_name,
                            tensor_value,
                        ) in step.items()
                    }

                    for step
                    in trajectory[
                        "steps"
                    ]
                ],

            "final_state":
                trajectory[
                    "final_state"
                ].detach()
                .cpu()
                .clone(),
        }

    for (
        pathway_name,
        trajectory,
    ) in BLOCK_7_FIRST_TRAJECTORY.items()
}


# =============================================================================
# Final Block 7 validation
# =============================================================================

NOTEBOOK_09_BLOCK_7_ERRORS = []


BLOCK_7_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_7_PREREQUISITES_VALID,

    "pathway_dimensions_invalid":
        BLOCK_7_PATHWAY_DIMENSIONS_VALID,

    "trajectory_length_invalid":
        BLOCK_7_TRAJECTORY_LENGTH_VALID,

    "probe_sequence_lengths_invalid":
        BLOCK_7_PROBE_SEQUENCE_LENGTHS_VALID,

    "probe_shapes_invalid":
        BLOCK_7_ALL_PROBE_SHAPES_VALID,

    "probes_not_finite":
        BLOCK_7_ALL_PROBES_FINITE,

    "initial_states_not_zero":
        BLOCK_7_ALL_INITIAL_STATES_ZERO,

    "initial_state_runs_not_equal":
        BLOCK_7_ALL_INITIAL_STATE_RUNS_EQUAL,

    "initial_state_storage_not_independent":
        BLOCK_7_ALL_INITIAL_STATE_STORAGE_INDEPENDENT,

    "trajectory_step_counts_invalid":
        BLOCK_7_TRAJECTORY_STEP_COUNTS_VALID,

    "candidate_shapes_invalid":
        BLOCK_7_ALL_CANDIDATE_SHAPES_VALID,

    "candidates_not_finite":
        BLOCK_7_ALL_CANDIDATES_FINITE,

    "weight_shapes_invalid":
        BLOCK_7_ALL_WEIGHT_SHAPES_VALID,

    "weights_not_finite":
        BLOCK_7_ALL_WEIGHTS_FINITE,

    "weights_not_bounded":
        BLOCK_7_ALL_WEIGHTS_BOUNDED,

    "weights_changed_across_steps":
        BLOCK_7_ALL_WEIGHTS_CONSTANT_ACROSS_STEPS,

    "active_weight_values_invalid":
        BLOCK_7_ALL_ACTIVE_WEIGHT_VALUES_VALID,

    "deferred_weight_values_invalid":
        BLOCK_7_ALL_DEFERRED_WEIGHT_VALUES_VALID,

    "weighted_candidate_shapes_invalid":
        BLOCK_7_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID,

    "weighted_candidates_not_finite":
        BLOCK_7_ALL_WEIGHTED_CANDIDATES_FINITE,

    "active_scaling_invalid":
        BLOCK_7_ALL_ACTIVE_SCALING_VALID,

    "deferred_contributions_nonzero":
        BLOCK_7_ALL_DEFERRED_CONTRIBUTIONS_ZERO,

    "state_shapes_invalid":
        BLOCK_7_ALL_STATE_SHAPES_VALID,

    "states_not_finite":
        BLOCK_7_ALL_STATES_FINITE,

    "deferred_states_nonzero":
        BLOCK_7_ALL_DEFERRED_STATES_ZERO,

    "previous_state_chain_invalid":
        BLOCK_7_ALL_PREVIOUS_STATE_CHAINS_VALID,

    "stepwise_update_identity_invalid":
        BLOCK_7_ALL_STEPWISE_UPDATE_IDENTITIES_VALID,

    "cumulative_state_identity_invalid":
        BLOCK_7_ALL_CUMULATIVE_STATE_IDENTITIES_VALID,

    "final_state_next_step_incompatible":
        BLOCK_7_ALL_FINAL_STATES_NEXT_STEP_COMPATIBLE,

    "trajectory_not_deterministic":
        BLOCK_7_ALL_TRAJECTORIES_DETERMINISTIC,

    "pathways_not_independent":
        BLOCK_7_PATHWAYS_INDEPENDENT,

    "unexpected_gradients":
        BLOCK_7_ALL_GRADIENTS_ABSENT,

    "representation_state_changed":
        BLOCK_7_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_7_TRANSFORMATIVE_UNCHANGED,

    "weight_state_changed":
        BLOCK_7_WEIGHT_STATE_UNCHANGED,

    "representation_not_frozen":
        BLOCK_7_REPRESENTATION_STILL_FROZEN,

    "transformative_not_frozen":
        BLOCK_7_TRANSFORMATIVE_STILL_FROZEN,

    "weight_parameters_not_trainable":
        BLOCK_7_WEIGHT_PARAMETERS_STILL_TRAINABLE,

    "inherited_modules_not_eval":
        BLOCK_7_INHERITED_MODULES_STILL_EVAL,

    "parameter_accounting_invalid":
        BLOCK_7_PARAMETER_ACCOUNTING_VALID,

    "causal_contract_invalid":
        BLOCK_7_CAUSAL_CONTRACT_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_7_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_7_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation validation
# =============================================================================

BLOCK_7_PROHIBITED_OPERATIONS = {
    "representation_forward_incorrectly_executed":
        NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED,

    "article_sequence_incorrectly_traversed":
        NOTEBOOK_09_BLOCK_7_ARTICLE_SEQUENCE_TRAVERSED,

    "future_context_incorrectly_used":
        NOTEBOOK_09_BLOCK_7_FUTURE_CONTEXT_USED,

    "loss_incorrectly_calculated":
        NOTEBOOK_09_BLOCK_7_LOSS_CALCULATED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_7_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_7_BACKWARD_PASS_EXECUTED,

    "gradient_clipping_incorrectly_executed":
        NOTEBOOK_09_BLOCK_7_GRADIENT_CLIPPING_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_7_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_7_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_7_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_7_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 7 state
# =============================================================================

NOTEBOOK_09_MULTI_STEP_RECURRENT_TRAJECTORY_VALID = (
    len(
        NOTEBOOK_09_BLOCK_7_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_ARTICLE_LEVEL_RECURRENT_CONTRACT_READY = (
    NOTEBOOK_09_MULTI_STEP_RECURRENT_TRAJECTORY_VALID
)


NOTEBOOK_09_BLOCK_7_VALID = (
    NOTEBOOK_09_MULTI_STEP_RECURRENT_TRAJECTORY_VALID
)


if not NOTEBOOK_09_BLOCK_7_VALID:

    raise RuntimeError(
        "Notebook 09 Block 7 controlled multi-step recurrent "
        "trajectory validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_7_ERRORS}"
    )


NOTEBOOK_09_BLOCK_7_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_7_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_7,

    "block_name":
        NOTEBOOK_09_BLOCK_7_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_7_VERSION,

    "trajectory_length":
        NOTEBOOK_09_BLOCK_7_TRAJECTORY_LENGTH,

    "pathway_dimensions":
        deepcopy(
            BLOCK_7_PATHWAY_DIMENSIONS
        ),

    "probe_sequence_lengths":
        deepcopy(
            BLOCK_7_PROBE_SEQUENCE_LENGTHS
        ),

    "candidate_shapes_valid":
        BLOCK_7_ALL_CANDIDATE_SHAPES_VALID,

    "weighted_candidate_shapes_valid":
        BLOCK_7_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID,

    "state_shapes_valid":
        BLOCK_7_ALL_STATE_SHAPES_VALID,

    "states_finite":
        BLOCK_7_ALL_STATES_FINITE,

    "weights_constant_across_steps":
        BLOCK_7_ALL_WEIGHTS_CONSTANT_ACROSS_STEPS,

    "active_scaling_valid":
        BLOCK_7_ALL_ACTIVE_SCALING_VALID,

    "deferred_contributions_zero":
        BLOCK_7_ALL_DEFERRED_CONTRIBUTIONS_ZERO,

    "deferred_states_zero":
        BLOCK_7_ALL_DEFERRED_STATES_ZERO,

    "previous_state_chain_valid":
        BLOCK_7_ALL_PREVIOUS_STATE_CHAINS_VALID,

    "stepwise_update_identity_valid":
        BLOCK_7_ALL_STEPWISE_UPDATE_IDENTITIES_VALID,

    "cumulative_state_identity_valid":
        BLOCK_7_ALL_CUMULATIVE_STATE_IDENTITIES_VALID,

    "trajectory_deterministic":
        BLOCK_7_ALL_TRAJECTORIES_DETERMINISTIC,

    "pathways_independent":
        BLOCK_7_PATHWAYS_INDEPENDENT,

    "reset_policy_valid":
        all(
            [
                BLOCK_7_ALL_INITIAL_STATES_ZERO,
                BLOCK_7_ALL_INITIAL_STATE_STORAGE_INDEPENDENT,
            ]
        ),

    "representation_parameters":
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT,

    "transformative_parameters":
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT,

    "transformative_weight_parameters":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT,

    "total_model_parameters":
        NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION,

    "multi_step_executed":
        NOTEBOOK_09_BLOCK_7_MULTI_STEP_RECURRENT_EXECUTED,

    "article_sequence_traversed":
        NOTEBOOK_09_BLOCK_7_ARTICLE_SEQUENCE_TRAVERSED,

    "future_context_used":
        NOTEBOOK_09_BLOCK_7_FUTURE_CONTEXT_USED,

    "multi_step_valid":
        NOTEBOOK_09_MULTI_STEP_RECURRENT_TRAJECTORY_VALID,

    "article_level_contract_ready":
        NOTEBOOK_09_ARTICLE_LEVEL_RECURRENT_CONTRACT_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_7_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_7_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 7: "
    "Controlled Multi-Step Recurrent Trajectory Validation"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_7_VERSION}"
)

print(
    f"Trajectory length            : "
    f"{NOTEBOOK_09_BLOCK_7_TRAJECTORY_LENGTH}"
)

print("-" * 72)

print(
    "Controlled trajectory contract"
)

for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} dimension          : "
        f"{BLOCK_7_PATHWAY_DIMENSIONS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} probe steps        : "
        f"{BLOCK_7_PROBE_SEQUENCE_LENGTHS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} candidate shapes   : "
        f"{BLOCK_7_CANDIDATE_SHAPES_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} states finite      : "
        f"{BLOCK_7_STATES_FINITE[pathway_name]}"
    )

print("-" * 72)

print(
    "Transformative-Weight trajectory validation"
)

for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} weight stable      : "
        f"{BLOCK_7_WEIGHTS_CONSTANT_ACROSS_STEPS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} active scaling     : "
        f"{BLOCK_7_ACTIVE_SCALING_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deferred contrib.  : "
        f"{BLOCK_7_DEFERRED_CONTRIBUTIONS_ZERO[pathway_name]}"
    )

print("-" * 72)

print(
    "Recurrent-state trajectory validation"
)

for pathway_name in BLOCK_7_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} state shapes       : "
        f"{BLOCK_7_STATE_SHAPES_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deferred state zero: "
        f"{BLOCK_7_DEFERRED_STATES_ZERO[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} state chain valid  : "
        f"{BLOCK_7_PREVIOUS_STATE_CHAIN_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} additive update    : "
        f"{BLOCK_7_STEPWISE_UPDATE_IDENTITY_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} cumulative identity: "
        f"{BLOCK_7_CUMULATIVE_STATE_IDENTITY_VALID[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} cumulative max diff: "
        f"{BLOCK_7_CUMULATIVE_STATE_MAX_ABS_DIFFERENCE[pathway_name]:.10f}"
    )

    print(
        f"{pathway_name:<14} deterministic      : "
        f"{BLOCK_7_TRAJECTORY_DETERMINISTIC[pathway_name]}"
    )

print("-" * 72)

print(
    "Reset and causal validation"
)

print(
    f"Fresh zero initial states    : "
    f"{BLOCK_7_ALL_INITIAL_STATES_ZERO}"
)

print(
    f"Independent reset storage    : "
    f"{BLOCK_7_ALL_INITIAL_STATE_STORAGE_INDEPENDENT}"
)

print(
    f"Pathways independent         : "
    f"{BLOCK_7_PATHWAYS_INDEPENDENT}"
)

print(
    f"Future context used          : "
    f"{NOTEBOOK_09_BLOCK_7_FUTURE_CONTEXT_USED}"
)

print(
    f"Article sequence traversed   : "
    f"{NOTEBOOK_09_BLOCK_7_ARTICLE_SEQUENCE_TRAVERSED}"
)

print("-" * 72)

print(
    "Parameter and execution validation"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Transformative Weight params : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Total model parameters       : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print(
    f"Representation unchanged     : "
    f"{BLOCK_7_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged     : "
    f"{BLOCK_7_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Weight state unchanged       : "
    f"{BLOCK_7_WEIGHT_STATE_UNCHANGED}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_7_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_7_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"Weight parameters trainable  : "
    f"{BLOCK_7_WEIGHT_PARAMETERS_STILL_TRAINABLE}"
)

print(
    f"All gradients absent         : "
    f"{BLOCK_7_ALL_GRADIENTS_ABSENT}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Transform forward            : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_FORWARD_PASS_EXECUTED}"
)

print(
    f"Weight forward               : "
    f"{NOTEBOOK_09_BLOCK_7_WEIGHT_FORWARD_EXECUTED}"
)

print(
    f"Multi-step recurrence        : "
    f"{NOTEBOOK_09_BLOCK_7_MULTI_STEP_RECURRENT_EXECUTED}"
)

print(
    f"Representation forward       : "
    f"{NOTEBOOK_09_REPRESENTATION_FORWARD_PASS_EXECUTED}"
)

print(
    f"Article sequence traversed   : "
    f"{NOTEBOOK_09_BLOCK_7_ARTICLE_SEQUENCE_TRAVERSED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_BLOCK_7_LOSS_CALCULATED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_7_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_7_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_7_PARAMETER_UPDATE_EXECUTED}"
)

print("-" * 72)

print(
    f"Trajectory deterministic     : "
    f"{BLOCK_7_ALL_TRAJECTORIES_DETERMINISTIC}"
)

print(
    f"Multi-step recurrent valid   : "
    f"{NOTEBOOK_09_MULTI_STEP_RECURRENT_TRAJECTORY_VALID}"
)

print(
    f"Article-level contract ready : "
    f"{NOTEBOOK_09_ARTICLE_LEVEL_RECURRENT_CONTRACT_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_7_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_7_COMPLETE}"
)

print("=" * 72)

print(
    f"{len(BLOCK_7_PATHWAY_DIMENSIONS)} recurrent pathway mechanism(s) "
    "were validated successfully across a controlled multi-step trajectory."
)

print(
    "Every trajectory began from a fresh deterministic zero state and "
    "propagated the immediately preceding recurrent state into the next "
    "transformative evaluation."
)

print(
    "Candidate transformations, Transformative-Weight-gated contributions "
    "and recurrent states preserved the required pathway dimensions and "
    "remained finite throughout execution."
)

print(
    "All active dimensions retained the validated 0.5 candidate scaling, "
    "while deferred dimensions produced zero recurrent contribution and "
    "remained exactly zero throughout the controlled trajectory."
)

print(
    "The additive recurrent update was satisfied at every step, and the "
    "final state reproduced the cumulative sum of all weighted candidate "
    "transformations."
)

print(
    "Repeated complete trajectories were exactly deterministic and began "
    "from independently constructed zero initial states."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} parameters "
    "remain frozen and exactly unchanged, while all "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT} "
    "Transformative-Weight parameter(s) remain trainable and unchanged."
)

print(
    "No representation forward pass, article-derived sequence traversal, "
    "future-context access, loss, optimiser, backward pass or parameter "
    "update was executed."
)

print(
    "Notebook 09 may now proceed to the controlled article-level recurrent "
    "execution contract."
)

print("=" * 72)

Media AI — Notebook 09, Block 7: Controlled Multi-Step Recurrent Trajectory Validation
Block version                : 1.1
Trajectory length            : 4
------------------------------------------------------------------------
Controlled trajectory contract
factual        dimension          : 10
factual        probe steps        : 4
factual        candidate shapes   : True
factual        states finite      : True
psychological  dimension          : 34
psychological  probe steps        : 4
psychological  candidate shapes   : True
psychological  states finite      : True
social         dimension          : 7
social         probe steps        : 4
social         candidate shapes   : True
social         states finite      : True
------------------------------------------------------------------------
Transformative-Weight trajectory validation
factual        weight stable      : True
factual        active scaling     : True
factual        deferred contrib.  : True
psychological  weight sta

## Block 8 — Article-Level Recurrent Execution Contract and Canonical Sequence Preparation

This block establishes the contract for applying the validated recurrent architecture to the canonical article-level representation sequence.

Blocks 5–7 validated the recurrent mechanism independently from article data. They established deterministic zero initial states, verified a complete single-step recurrent transition, and demonstrated stable causal propagation across a controlled multi-step trajectory.

Block 8 now moves from synthetic architectural probes to the actual ordered sentence representations required for article-level recurrent execution.

The purpose of this block is deliberately preparatory. It recovers and validates the canonical article sequence, establishes the correspondence between sentence identity and pathway-specific representations, defines article-boundary reset behaviour, and constructs the runtime inputs required for recurrent traversal.

It does **not** yet execute the article-level recurrent trajectory.

No candidate-transformative forward pass, Transformative-Weight forward pass, recurrent update, loss calculation, optimiser creation, backward pass, or parameter update is performed.

### Architectural transition

The preceding validation blocks operated on controlled pathway-specific probes

$$
r_t^{(k)},
$$

where

$$
k\in\{F,P,S\}.
$$

Block 8 replaces those synthetic probes with the canonical sentence-level Media AI representations produced by the inherited representation architecture.

For sentence \(t\), the recurrent architecture will ultimately receive

$$
r_t^{(F)}\in\mathbb{R}^{10},
$$

$$
r_t^{(P)}\in\mathbb{R}^{34},
$$

and

$$
r_t^{(S)}\in\mathbb{R}^{7}.
$$

These vectors must correspond to the same canonical sentence identity and the same position in the article sequence.

The future recurrent execution therefore has the form

$$
\left(
r_t^{(F)},
r_t^{(P)},
r_t^{(S)}
\right)
\rightarrow
\left(
h_t^{(F)},
h_t^{(P)},
h_t^{(S)}
\right).
$$

Block 8 establishes the validity of the left-hand side of this mapping without yet producing the right-hand side.

### Canonical sequence source

The article-level sequence must be reconstructed from the validated Notebook 08 → Notebook 09 handover and the canonical sentence identity contract inherited by Notebook 09.

The sequence must not be inferred from arbitrary dictionary ordering, reconstructed from supervision counts, or reordered according to target availability.

Sentence order must follow the canonical textual order established upstream.

For the current pilot corpus, the expected structure is one article containing 16 ordered sentences.

The canonical sequence therefore satisfies

$$
N_{\text{articles}}=1
$$

and

$$
N_{\text{sentences}}=16.
$$

Block 8 must nevertheless validate these values from the recovered runtime contract rather than silently assuming them.

### Sentence identity contract

Every prepared article-level recurrent input must retain a canonical sentence identifier.

For sentence position \(t\),

$$
s_t
=
\text{sentence\_id}_t.
$$

The ordered identity sequence is

$$
S
=
\left(
s_1,
s_2,
\ldots,
s_N
\right).
$$

The sequence is valid only if:

- every sentence has a non-empty canonical identifier;
- sentence identifiers are unique;
- sentence order is deterministic;
- the same sentence identifiers are used across factual, psychological, and social representations;
- and no sentence is silently inserted, removed, duplicated, or reordered.

Sentence identity remains the principal alignment mechanism between textual structure and model state.

### Article identity contract

Every sentence must also remain associated with its canonical article identity.

Let

$$
a_t
$$

denote the article identifier associated with sentence \(t\).

The article sequence can therefore be represented as

$$
A
=
\left(
a_1,
a_2,
\ldots,
a_N
\right).
$$

For the current one-article pilot,

$$
a_1=a_2=\cdots=a_{16}.
$$

This must be verified rather than assumed.

The article identity is required because recurrent state is allowed to propagate within an article but must never propagate across article boundaries.

### Canonical textual ordering

Within each article, recurrent execution must follow the original sentence order.

For an article containing \(N_a\) sentences,

$$
s_1
\prec
s_2
\prec
\cdots
\prec
s_{N_a},
$$

where \(\prec\) denotes canonical textual precedence.

The recurrent traversal must preserve this order exactly.

No batch shuffling is permitted.

No supervision-dependent sorting is permitted.

No ordering based on representation magnitude, target availability, active dimensions, or model output is permitted.

### Representation recovery

Block 8 must recover the factual, psychological, and social sentence representations from the inherited representation architecture and validated canonical textual inputs.

The pathway dimensions remain

$$
d_F=10,
$$

$$
d_P=34,
$$

and

$$
d_S=7.
$$

For the complete pilot article, the prepared representation matrices should therefore satisfy

$$
R^{(F)}
\in
\mathbb{R}^{16\times10},
$$

$$
R^{(P)}
\in
\mathbb{R}^{16\times34},
$$

and

$$
R^{(S)}
\in
\mathbb{R}^{16\times7}.
$$

These matrices must contain representations, not supervision targets.

This distinction is essential.

The recurrent architecture propagates model representations through transformative mechanisms. Psychological, factual, or social supervision recovered in earlier notebooks must not be substituted for the corresponding representation vectors.

### Representation versus supervision boundary

The article-level recurrent inputs are model states.

They are therefore conceptually distinct from the supervision structures used to train or validate those states.

For pathway \(k\),

$$
r_t^{(k)}
\neq
y_t^{(k)}
$$

as an architectural identity.

Here,

$$
r_t^{(k)}
$$

denotes the representation supplied to the recurrent mechanism, whereas

$$
y_t^{(k)}
$$

denotes a supervision target where such supervision exists.

Numerical coincidence between individual values does not collapse this distinction.

Block 8 must preserve this boundary explicitly.

### Representation-model execution boundary

If canonical sentence representations are not already available as validated inherited runtime tensors, Block 8 may execute the frozen representation model solely to reconstruct them from the canonical sentence-level representation source established upstream.

Such execution is inference only.

Accordingly,

$$
\nabla_{\theta_R}=0
$$

throughout the block, and

$$
\theta_R^{\text{after}}
=
\theta_R^{\text{before}}.
$$

The representation model must remain frozen and in evaluation mode.

No representation parameter may become eligible for optimisation.

### No transformative execution

Although the trained transformative modules are already available from Notebook 08, Block 8 does not execute them.

Thus no candidate transformation

$$
c_t^{(k)}
$$

is produced in this block.

Likewise, the Transformative-Weight modules are not executed and no weighted candidate contribution

$$
TW_k\odot c_t^{(k)}
$$

is calculated.

This preserves a clear boundary between **sequence preparation** and **recurrent execution**.

### Article-boundary reset contract

The recurrent initial-state policy established in Block 5 remains

$$
h_0^{(k)}
=
\mathbf{0}_{d_k}.
$$

A fresh initial state must be created whenever traversal enters a new article.

For article \(a\),

$$
h_{a,0}^{(k)}
=
\mathbf{0}_{d_k}.
$$

If another article \(a+1\) follows, then

$$
h_{a+1,0}^{(k)}
=
\mathbf{0}_{d_k},
$$

independently of the final state

$$
h_{a,N_a}^{(k)}.
$$

Therefore,

$$
h_{a+1,0}^{(k)}
\neq
h_{a,N_a}^{(k)}
$$

except by accidental numerical equality.

State propagation across article boundaries is prohibited.

### Within-article state continuity

Once article-level execution begins in the subsequent block, recurrent state will propagate continuously through the ordered sentence sequence.

For sentence \(t>1\) within the same article,

$$
h_{t-1}^{(k)}
$$

must be the state produced by the immediately preceding sentence.

No reset occurs between sentences belonging to the same article.

Thus the eventual traversal contract is

$$
h_{a,0}^{(k)}
\rightarrow
h_{a,1}^{(k)}
\rightarrow
h_{a,2}^{(k)}
\rightarrow
\cdots
\rightarrow
h_{a,N_a}^{(k)}.
$$

Block 8 prepares this traversal structure but does not execute it.

### Causal execution contract

Article-level recurrence must remain strictly causal.

For sentence \(t\), the future recurrent state will depend only on the current representation and the preceding recurrent state:

$$
c_t^{(k)}
=
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right).
$$

The update will then be

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k\odot c_t^{(k)}.
$$

No representation from sentence

$$
t+1
$$

or later may influence the transition at sentence \(t\).

Block 8 therefore prepares the representation sequence in forward textual order without constructing any future-context feature.

### Preceding textual context versus recurrent state

The upstream representation architecture may itself encode textual context according to the methodological contracts established in earlier notebooks.

That upstream contextual representation must not be confused with the recurrent state introduced in Notebook 09.

The recurrent state

$$
h_{t-1}^{(k)}
$$

is an architectural state accumulated by the recurrent mechanism.

It is not a replacement label for preceding textual context, a supervision record, or an upstream contextual embedding.

These concepts remain structurally distinct.

### Pathway alignment

At every sentence position \(t\), the factual, psychological, and social representations must refer to the same sentence.

The prepared article-level record should therefore preserve a structure equivalent to

$$
\mathcal{R}_t
=
\left(
s_t,
a_t,
r_t^{(F)},
r_t^{(P)},
r_t^{(S)}
\right).
$$

The pathway shapes must satisfy

$$
r_t^{(F)}
\in
\mathbb{R}^{1\times10},
$$

$$
r_t^{(P)}
\in
\mathbb{R}^{1\times34},
$$

and

$$
r_t^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

This sentence-level structure becomes the canonical recurrent execution unit for the next block.

### Representation finiteness

Every prepared representation must contain finite numerical values.

For every sentence \(t\),

$$
\operatorname{isfinite}
\left(
r_t^{(F)}
\right),
$$

$$
\operatorname{isfinite}
\left(
r_t^{(P)}
\right),
$$

and

$$
\operatorname{isfinite}
\left(
r_t^{(S)}
\right)
$$

must hold.

No NaN or infinite value may enter recurrent execution.

### Representation determinism

The inherited representation architecture is frozen and evaluated deterministically.

Therefore, if Block 8 reconstructs article representations through a representation forward pass, repeated inference from the same canonical input must produce identical outputs.

For pathway \(k\),

$$
R_{\mathrm{run1}}^{(k)}
=
R_{\mathrm{run2}}^{(k)}.
$$

If representations are recovered directly from an already validated runtime contract, their identity and integrity must instead be checked against that inherited source.

In either case, the prepared recurrent representation sequence must be deterministic.

### Active and deferred dimensions

The active/deferred transformation boundaries established in Notebook 08 remain unchanged.

For the factual pathway,

$$
A_F
=
\{1,2,4,8,9\},
$$

and

$$
D_F
=
\{0,3,5,6,7\}.
$$

For the psychological pathway,

$$
A_P
=
\{1,4,11,12,14,15,16,19,21,22,23,25,26,27,30,32,33\},
$$

with the remaining psychological dimensions deferred.

For the social pathway,

$$
A_S
=
\{0,2,4\},
$$

and

$$
D_S
=
\{1,3,5,6\}.
$$

These boundaries apply to transformative gating.

They do **not** remove dimensions from the article representations.

Every recurrent input retains the complete pathway dimensionality.

### Full-dimensional recurrent inputs

Even dimensions currently deferred from transformative updating remain part of the representation vectors.

Thus Block 8 must not reduce the representations to active dimensions only.

The recurrent input spaces remain

$$
\mathbb{R}^{10},
\qquad
\mathbb{R}^{34},
\qquad
\mathbb{R}^{7}.
$$

Active/deferred gating occurs later through the Transformative Weight.

This preserves semantic index alignment across the complete architecture.

### Parameter-state preservation

Article-sequence preparation must not alter learned parameters.

The representation parameters must satisfy

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}},
$$

the transformative parameters

$$
\theta_T^{\mathrm{after}}
=
\theta_T^{\mathrm{before}},
$$

and the Transformative-Weight parameters

$$
\theta_{TW}^{\mathrm{after}}
=
\theta_{TW}^{\mathrm{before}}.
$$

The parameter counts remain:

$$
894{,}003
$$

representation parameters,

$$
8{,}187
$$

transformative parameters,

and

$$
25
$$

trainable Transformative-Weight parameters.

The complete current architecture therefore remains

$$
902{,}215
$$

parameters.

### Gradient boundary

No gradient-based operation is required for article-sequence preparation.

The inherited representation and transformative parameters remain frozen.

The 25 Transformative-Weight parameters remain trainable in principle but receive no gradients in this block.

No backward graph should be retained after representation preparation.

### No recurrent state trajectory yet

Block 8 defines the article-boundary initial-state policy but does not create the sequence of updated recurrent states

$$
h_1,
h_2,
\ldots,
h_N.
$$

A deterministic zero initial-state template may be validated as part of the execution contract, but no sentence-level recurrent transition is performed.

Therefore:

- no candidate transformation is produced;
- no Transformative Weight is applied;
- no weighted candidate is produced;
- and no updated article recurrent state is created.

### No optimisation in Block 8

This block performs no training.

Accordingly:

- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- no gradient clipping is applied;
- no optimiser step is executed;
- and no parameter update occurs.

### Prepared article-level execution contract

At completion, Block 8 should expose an ordered sentence-level structure equivalent to

$$
\mathcal{A}
=
\left[
\mathcal{R}_1,
\mathcal{R}_2,
\ldots,
\mathcal{R}_N
\right],
$$

where

$$
\mathcal{R}_t
=
\left(
s_t,
a_t,
r_t^{(F)},
r_t^{(P)},
r_t^{(S)}
\right).
$$

The structure must preserve canonical identity, canonical ordering, complete pathway dimensionality, and deterministic representation values.

It should be directly consumable by the subsequent article-level recurrent forward block without rediscovering upstream objects.

### Block 8 completion contract

Block 8 is complete only when:

- the canonical sentence-level source is recovered explicitly;
- the canonical article identity is recovered explicitly;
- the article and sentence counts are validated;
- every sentence has a valid canonical sentence identifier;
- sentence identifiers are unique;
- canonical textual ordering is preserved;
- factual, psychological, and social representations are aligned to the same sentence identities;
- representation values are distinguished explicitly from supervision targets;
- factual representations have dimension 10;
- psychological representations have dimension 34;
- social representations have dimension 7;
- every pathway contains the same number of ordered sentence representations;
- every representation is finite;
- the prepared representation sequence is deterministic;
- complete pathway dimensions are retained rather than active dimensions only;
- active/deferred semantic index contracts remain unchanged;
- article boundaries are identifiable before recurrent execution;
- the deterministic zero-state reset policy is retained for every article boundary;
- no recurrent state is propagated across article boundaries;
- no future sentence representation is used to prepare an earlier recurrent transition;
- the representation parameters remain frozen and unchanged;
- the transformative parameters remain frozen and unchanged;
- the Transformative-Weight parameters remain unchanged;
- the complete parameter count remains 902,215;
- no candidate-transformative forward pass is executed;
- no Transformative-Weight forward pass is executed;
- no weighted candidate transformation is produced;
- no article-level recurrent update is executed;
- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- and no parameter update occurs.

Successful completion establishes the canonical article-level recurrent input contract

$$
\boxed{
\mathcal{R}_t
=
\left(
s_t,
a_t,
r_t^{(F)},
r_t^{(P)},
r_t^{(S)}
\right)
}
$$

and prepares the ordered article sequence required for controlled recurrent execution.

Notebook 09 may then proceed to the first complete article-level recurrent trajectory, applying the validated transformative mechanisms and Transformative Weights causally across the canonical sentence sequence.

In [58]:
# =============================================================================
# Media AI — Notebook 09
# Block 8: Article-Level Recurrent Execution Contract
#          and Canonical Sequence Preparation
# =============================================================================

from copy import deepcopy

import io
import json

import numpy as np
import torch

from googleapiclient.http import MediaIoBaseDownload


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_8 = 8

NOTEBOOK_09_BLOCK_8_NAME = (
    "Article-Level Recurrent Execution Contract "
    "and Canonical Sequence Preparation"
)

NOTEBOOK_09_BLOCK_8_VERSION = "1.3"


# =============================================================================
# Persistent Notebook 05 representation contract
# =============================================================================

BLOCK_8_REPRESENTATION_FILENAME = (
    "notebook_05_contextual_sentence_representations.json"
)

BLOCK_8_REPRESENTATION_ARTIFACT_TYPE = (
    "media_ai_contextual_sentence_representations"
)

BLOCK_8_REPRESENTATION_SCHEMA_VERSION = "1.0"

BLOCK_8_ENCODER_PROVIDER = (
    "sentence-transformers"
)

BLOCK_8_ENCODER_MODEL_ID = (
    "sentence-transformers/all-mpnet-base-v2"
)

BLOCK_8_ENCODER_DIMENSION = 768

BLOCK_8_ENCODER_NORMALISED = True

# Corpus cardinalities are recovered from the persisted artefact.
# Notebook 09 does not recreate Notebook 05 pilot-era counts as literals.


# =============================================================================
# Required Notebook 09 runtime contract
# =============================================================================

BLOCK_8_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_BLOCK_1_COMPLETE",
    "NOTEBOOK_09_BLOCK_1_VALID",
    "NOTEBOOK_09_HANDOVER_RESTORED",
    "NOTEBOOK_09_DRIVE_SERVICE",

    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",

    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Blocks 2–7
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_2_COMPLETE",
    "NOTEBOOK_09_BLOCK_2_VALID",

    "NOTEBOOK_09_BLOCK_3_COMPLETE",
    "NOTEBOOK_09_BLOCK_3_VALID",

    "NOTEBOOK_09_BLOCK_4_COMPLETE",
    "NOTEBOOK_09_BLOCK_4_VALID",

    "NOTEBOOK_09_BLOCK_5_COMPLETE",
    "NOTEBOOK_09_BLOCK_5_VALID",

    "NOTEBOOK_09_BLOCK_6_COMPLETE",
    "NOTEBOOK_09_BLOCK_6_VALID",

    "NOTEBOOK_09_BLOCK_7_COMPLETE",
    "NOTEBOOK_09_BLOCK_7_VALID",

    "NOTEBOOK_09_MULTI_STEP_RECURRENT_TRAJECTORY_VALID",
    "NOTEBOOK_09_ARTICLE_LEVEL_RECURRENT_CONTRACT_READY",

    # -------------------------------------------------------------------------
    # Recurrent contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS",
    "NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES",

    # -------------------------------------------------------------------------
    # Transformative-Weight contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",

    # -------------------------------------------------------------------------
    # Environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_8_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_8_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_8_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 8 prerequisites are not initialised. "
        f"Missing: {BLOCK_8_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_8_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,

        NOTEBOOK_09_BLOCK_1_COMPLETE is True,
        NOTEBOOK_09_BLOCK_1_VALID is True,
        NOTEBOOK_09_HANDOVER_RESTORED is True,

        NOTEBOOK_09_BLOCK_2_COMPLETE is True,
        NOTEBOOK_09_BLOCK_2_VALID is True,

        NOTEBOOK_09_BLOCK_3_COMPLETE is True,
        NOTEBOOK_09_BLOCK_3_VALID is True,

        NOTEBOOK_09_BLOCK_4_COMPLETE is True,
        NOTEBOOK_09_BLOCK_4_VALID is True,

        NOTEBOOK_09_BLOCK_5_COMPLETE is True,
        NOTEBOOK_09_BLOCK_5_VALID is True,

        NOTEBOOK_09_BLOCK_6_COMPLETE is True,
        NOTEBOOK_09_BLOCK_6_VALID is True,

        NOTEBOOK_09_BLOCK_7_COMPLETE is True,
        NOTEBOOK_09_BLOCK_7_VALID is True,

        NOTEBOOK_09_MULTI_STEP_RECURRENT_TRAJECTORY_VALID is True,
        NOTEBOOK_09_ARTICLE_LEVEL_RECURRENT_CONTRACT_READY is True,
    ]
)


if not BLOCK_8_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Blocks 1–7 must be valid and complete "
        "before article-level recurrent sequence preparation."
    )


# =============================================================================
# Canonical pathway contract
# =============================================================================

BLOCK_8_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
)


BLOCK_8_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            BLOCK_8_PATHWAY_DIMENSIONS,
            dict,
        ),

        bool(
            BLOCK_8_PATHWAY_DIMENSIONS
        ),

        (
            BLOCK_8_PATHWAY_DIMENSIONS
            ==
            NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
        ),

        (
            set(
                BLOCK_8_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys()
            )
        ),
    ]
)


if not BLOCK_8_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 8 pathway dimensions are invalid."
    )


# =============================================================================
# Active / deferred pathway partitions
# =============================================================================

BLOCK_8_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES.items()
}


BLOCK_8_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES.items()
}


BLOCK_8_DIMENSION_PARTITIONS_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_8_PATHWAY_DIMENSIONS.items():

    active_indices = set(
        BLOCK_8_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    deferred_indices = set(
        BLOCK_8_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )


    BLOCK_8_DIMENSION_PARTITIONS_VALID[
        pathway_name
    ] = all(
        [
            active_indices.isdisjoint(
                deferred_indices
            ),

            (
                active_indices
                |
                deferred_indices
            )
            ==
            set(
                range(
                    pathway_dim
                )
            ),
        ]
    )


BLOCK_8_ALL_DIMENSION_PARTITIONS_VALID = all(
    BLOCK_8_DIMENSION_PARTITIONS_VALID.values()
)


if not BLOCK_8_ALL_DIMENSION_PARTITIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 8 active/deferred dimension "
        "partitions are invalid."
    )


# =============================================================================
# Required inherited representation modules
# =============================================================================
#
# Notebook 08 persisted exactly:
#
#   backbone
#   global_confluent
#   psychological_head
#   factual_head
#   social_head
#
# The pathway heads encapsulate their pathway-specific representation core.
# No separate representation_core module is expected here.
# =============================================================================

BLOCK_8_REQUIRED_REPRESENTATION_MODULES = (
    "backbone",
    "global_confluent",
    "psychological_head",
    "factual_head",
    "social_head",
)


BLOCK_8_MISSING_REPRESENTATION_MODULES = [
    module_name

    for module_name
    in BLOCK_8_REQUIRED_REPRESENTATION_MODULES

    if module_name
    not in BLOCK_1_REPRESENTATION_MODULES
]


if BLOCK_8_MISSING_REPRESENTATION_MODULES:

    raise RuntimeError(
        "Notebook 09 inherited representation architecture is incomplete. "
        f"Missing modules: {BLOCK_8_MISSING_REPRESENTATION_MODULES}"
    )


BLOCK_8_BACKBONE = (
    BLOCK_1_REPRESENTATION_MODULES[
        "backbone"
    ]
)


BLOCK_8_GLOBAL_CONFLUENT = (
    BLOCK_1_REPRESENTATION_MODULES[
        "global_confluent"
    ]
)


BLOCK_8_PSYCHOLOGICAL_HEAD = (
    BLOCK_1_REPRESENTATION_MODULES[
        "psychological_head"
    ]
)


BLOCK_8_FACTUAL_HEAD = (
    BLOCK_1_REPRESENTATION_MODULES[
        "factual_head"
    ]
)


BLOCK_8_SOCIAL_HEAD = (
    BLOCK_1_REPRESENTATION_MODULES[
        "social_head"
    ]
)


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_8_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_8_nested_state_exact(
    state_before,
    state_after,
):

    if (
        state_before.keys()
        !=
        state_after.keys()
    ):

        return False


    for module_name in state_before:

        if (
            state_before[
                module_name
            ].keys()
            !=
            state_after[
                module_name
            ].keys()
        ):

            return False


        for tensor_name in state_before[
            module_name
        ]:

            if not torch.equal(
                state_before[
                    module_name
                ][
                    tensor_name
                ],
                state_after[
                    module_name
                ][
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot learned model state before Block 8 execution
# =============================================================================

BLOCK_8_REPRESENTATION_STATE_BEFORE = (
    block_8_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_8_TRANSFORMATIVE_STATE_BEFORE = (
    block_8_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_8_WEIGHT_STATE_BEFORE = (
    block_8_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Google Drive helpers
# =============================================================================

def block_8_download_drive_file_bytes(
    drive_service,
    file_id,
):

    request = (
        drive_service.files()
        .get_media(
            fileId=
                file_id
        )
    )


    buffer = io.BytesIO()


    downloader = MediaIoBaseDownload(
        buffer,
        request,
    )


    done = False


    while not done:

        _, done = downloader.next_chunk()


    return buffer.getvalue()


# =============================================================================
# Locate persisted Notebook 05 representation artefact
# =============================================================================
#
# The filename is canonical and was persisted independently by Notebook 05
# Block 5.
#
# If more than one file with the same name exists, candidates are evaluated
# from most recently modified to oldest. The first candidate satisfying the
# exact artefact contract is selected.
# =============================================================================

escaped_representation_filename = (
    BLOCK_8_REPRESENTATION_FILENAME.replace(
        "'",
        "\\'",
    )
)


BLOCK_8_REPRESENTATION_FILE_SEARCH = (
    NOTEBOOK_09_DRIVE_SERVICE.files()
    .list(
        q=(
            f"name = '{escaped_representation_filename}' "
            "and trashed = false"
        ),

        spaces=
            "drive",

        fields=
            "files(id,name,mimeType,modifiedTime,parents,size)",

        orderBy=
            "modifiedTime desc",

        pageSize=
            20,
    )
    .execute()
)


BLOCK_8_REPRESENTATION_FILE_CANDIDATES = (
    BLOCK_8_REPRESENTATION_FILE_SEARCH.get(
        "files",
        [],
    )
)


if not BLOCK_8_REPRESENTATION_FILE_CANDIDATES:

    raise RuntimeError(
        "Notebook 09 Block 8 could not locate the persisted "
        "Notebook 05 contextual representation artefact: "
        f"{BLOCK_8_REPRESENTATION_FILENAME}"
    )


# =============================================================================
# Validate candidate artefact payload
# =============================================================================

def block_8_validate_representation_payload(
    payload,
):

    if not isinstance(
        payload,
        dict,
    ):

        return (
            False,
            "payload_not_dictionary",
        )


    if (
        payload.get(
            "artifact_type"
        )
        !=
        BLOCK_8_REPRESENTATION_ARTIFACT_TYPE
    ):

        return (
            False,
            "artifact_type_invalid",
        )


    if (
        str(
            payload.get(
                "schema_version"
            )
        )
        !=
        BLOCK_8_REPRESENTATION_SCHEMA_VERSION
    ):

        return (
            False,
            "schema_version_invalid",
        )


    encoder = payload.get(
        "encoder"
    )


    if not isinstance(
        encoder,
        dict,
    ):

        return (
            False,
            "encoder_contract_missing",
        )


    if (
        encoder.get(
            "provider"
        )
        !=
        BLOCK_8_ENCODER_PROVIDER
    ):

        return (
            False,
            "encoder_provider_invalid",
        )


    if (
        encoder.get(
            "model_id"
        )
        !=
        BLOCK_8_ENCODER_MODEL_ID
    ):

        return (
            False,
            "encoder_model_invalid",
        )


    if (
        int(
            encoder.get(
                "embedding_dimension",
                -1,
            )
        )
        !=
        BLOCK_8_ENCODER_DIMENSION
    ):

        return (
            False,
            "encoder_dimension_invalid",
        )


    if (
        bool(
            encoder.get(
                "normalised"
            )
        )
        is not True
    ):

        return (
            False,
            "encoder_normalisation_invalid",
        )


    if (
        bool(
            encoder.get(
                "article_context_used"
            )
        )
        is not False
    ):

        return (
            False,
            "article_context_boundary_invalid",
        )


    if (
        bool(
            encoder.get(
                "future_context_used"
            )
        )
        is not False
    ):

        return (
            False,
            "future_context_boundary_invalid",
        )


    counts = payload.get(
        "counts"
    )


    if not isinstance(
        counts,
        dict,
    ):

        return (
            False,
            "counts_contract_missing",
        )


    try:

        article_count = int(
            counts.get(
                "articles",
                -1,
            )
        )

        sentence_count = int(
            counts.get(
                "sentences",
                -1,
            )
        )

        representation_count = int(
            counts.get(
                "representations",
                -1,
            )
        )


    except (
        TypeError,
        ValueError,
    ):

        return (
            False,
            "counts_not_integer",
        )


    records = payload.get(
        "records"
    )


    if not isinstance(
        records,
        list,
    ):

        return (
            False,
            "records_missing",
        )


    if article_count <= 0:

        return (
            False,
            "article_count_invalid",
        )


    if sentence_count <= 0:

        return (
            False,
            "sentence_count_invalid",
        )


    if representation_count != sentence_count:

        return (
            False,
            "representation_count_invalid",
        )


    if len(
        records
    ) != sentence_count:

        return (
            False,
            "record_count_invalid",
        )


    record_article_ids = [
        str(
            record.get(
                "article_id"
            )
        ).strip()

        for record
        in records

        if isinstance(
            record,
            dict,
        )
        and
        record.get(
            "article_id"
        )
        is not None
    ]


    if (
        len(
            record_article_ids
        )
        !=
        sentence_count
    ):

        return (
            False,
            "record_article_identity_invalid",
        )


    if (
        len(
            set(
                record_article_ids
            )
        )
        !=
        article_count
    ):

        return (
            False,
            "article_count_invalid",
        )


    return (
        True,
        None,
    )


# =============================================================================
# Resolve first contract-valid persisted artefact
# =============================================================================

BLOCK_8_REPRESENTATION_DRIVE_FILE = None

BLOCK_8_REPRESENTATION_DRIVE_FILE_ID = None

BLOCK_8_REPRESENTATION_PAYLOAD = None

BLOCK_8_REPRESENTATION_CANDIDATE_AUDIT = []


for candidate_file in BLOCK_8_REPRESENTATION_FILE_CANDIDATES:

    candidate_file_id = candidate_file.get(
        "id"
    )


    candidate_audit = {
        "file_id":
            candidate_file_id,

        "name":
            candidate_file.get(
                "name"
            ),

        "modified_time":
            candidate_file.get(
                "modifiedTime"
            ),

        "valid":
            False,

        "reason":
            None,
    }


    try:

        candidate_bytes = (
            block_8_download_drive_file_bytes(
                drive_service=
                    NOTEBOOK_09_DRIVE_SERVICE,

                file_id=
                    candidate_file_id,
            )
        )


        candidate_payload = json.loads(
            candidate_bytes.decode(
                "utf-8"
            )
        )


        (
            candidate_valid,
            candidate_reason,
        ) = block_8_validate_representation_payload(
            candidate_payload
        )


        candidate_audit[
            "valid"
        ] = bool(
            candidate_valid
        )


        candidate_audit[
            "reason"
        ] = candidate_reason


        if candidate_valid:

            BLOCK_8_REPRESENTATION_DRIVE_FILE = (
                deepcopy(
                    candidate_file
                )
            )


            BLOCK_8_REPRESENTATION_DRIVE_FILE_ID = (
                candidate_file_id
            )


            BLOCK_8_REPRESENTATION_PAYLOAD = (
                candidate_payload
            )


            BLOCK_8_REPRESENTATION_CANDIDATE_AUDIT.append(
                candidate_audit
            )


            break


    except Exception as exc:

        candidate_audit[
            "reason"
        ] = (
            f"read_or_parse_failed:"
            f"{type(exc).__name__}"
        )


    BLOCK_8_REPRESENTATION_CANDIDATE_AUDIT.append(
        candidate_audit
    )


if BLOCK_8_REPRESENTATION_PAYLOAD is None:

    raise RuntimeError(
        "Notebook 09 Block 8 located one or more files named "
        f"{BLOCK_8_REPRESENTATION_FILENAME}, but none satisfied "
        "the canonical Notebook 05 representation artefact contract. "
        f"Audit: {BLOCK_8_REPRESENTATION_CANDIDATE_AUDIT}"
    )


# =============================================================================
# Extract persisted encoder contract
# =============================================================================

BLOCK_8_PERSISTED_ENCODER = deepcopy(
    BLOCK_8_REPRESENTATION_PAYLOAD[
        "encoder"
    ]
)


BLOCK_8_PERSISTED_COUNTS = deepcopy(
    BLOCK_8_REPRESENTATION_PAYLOAD[
        "counts"
    ]
)


BLOCK_8_PERSISTED_RECORDS = deepcopy(
    BLOCK_8_REPRESENTATION_PAYLOAD[
        "records"
    ]
)


BLOCK_8_EXPECTED_ARTICLE_COUNT = int(
    BLOCK_8_PERSISTED_COUNTS[
        "articles"
    ]
)


BLOCK_8_EXPECTED_SENTENCE_COUNT = int(
    BLOCK_8_PERSISTED_COUNTS[
        "sentences"
    ]
)


BLOCK_8_EXPECTED_REPRESENTATION_COUNT = int(
    BLOCK_8_PERSISTED_COUNTS[
        "representations"
    ]
)


BLOCK_8_PERSISTED_COUNT_CONTRACT_VALID = all(
    [
        BLOCK_8_EXPECTED_ARTICLE_COUNT > 0,
        BLOCK_8_EXPECTED_SENTENCE_COUNT > 0,

        (
            BLOCK_8_EXPECTED_REPRESENTATION_COUNT
            ==
            BLOCK_8_EXPECTED_SENTENCE_COUNT
        ),

        (
            len(
                BLOCK_8_PERSISTED_RECORDS
            )
            ==
            BLOCK_8_EXPECTED_SENTENCE_COUNT
        ),
    ]
)


if not BLOCK_8_PERSISTED_COUNT_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 05 persisted representation-count contract is invalid."
    )


# =============================================================================
# Validate individual representation records
# =============================================================================

BLOCK_8_RECORD_VALIDATION_ERRORS = []

BLOCK_8_VALIDATED_REPRESENTATION_RECORDS = []


for (
    source_position,
    record,
) in enumerate(
    BLOCK_8_PERSISTED_RECORDS
):

    if not isinstance(
        record,
        dict,
    ):

        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:not_dictionary"
        )

        continue


    article_id = record.get(
        "article_id"
    )


    sentence_id = record.get(
        "sentence_id"
    )


    sentence_index = record.get(
        "sentence_index"
    )


    sequence_position = record.get(
        "sequence_position"
    )


    embedding_dimension = record.get(
        "embedding_dimension"
    )


    normalised = record.get(
        "normalised"
    )


    embedding = record.get(
        "embedding"
    )


    if (
        article_id is None
        or
        not str(
            article_id
        ).strip()
    ):

        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:article_id_invalid"
        )


    if (
        sentence_id is None
        or
        not str(
            sentence_id
        ).strip()
    ):

        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:sentence_id_invalid"
        )


    try:

        sentence_index = int(
            sentence_index
        )


    except (
        TypeError,
        ValueError,
    ):

        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:sentence_index_invalid"
        )

        sentence_index = None


    try:

        sequence_position = int(
            sequence_position
        )


    except (
        TypeError,
        ValueError,
    ):

        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:sequence_position_invalid"
        )

        sequence_position = None


    try:

        embedding_dimension = int(
            embedding_dimension
        )


    except (
        TypeError,
        ValueError,
    ):

        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:embedding_dimension_invalid"
        )

        embedding_dimension = None


    if (
        embedding_dimension
        !=
        BLOCK_8_ENCODER_DIMENSION
    ):

        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:embedding_dimension_mismatch"
        )


    if bool(
        normalised
    ) is not True:

        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:normalised_flag_invalid"
        )


    try:

        embedding_array = np.asarray(
            embedding,
            dtype=
                np.float32,
        )


    except Exception:

        embedding_array = None


        BLOCK_8_RECORD_VALIDATION_ERRORS.append(
            f"record_{source_position}:embedding_conversion_failed"
        )


    if embedding_array is not None:

        if (
            embedding_array.shape
            !=
            (
                BLOCK_8_ENCODER_DIMENSION,
            )
        ):

            BLOCK_8_RECORD_VALIDATION_ERRORS.append(
                f"record_{source_position}:embedding_shape_invalid"
            )


        elif not np.all(
            np.isfinite(
                embedding_array
            )
        ):

            BLOCK_8_RECORD_VALIDATION_ERRORS.append(
                f"record_{source_position}:embedding_nonfinite"
            )


        else:

            embedding_norm = float(
                np.linalg.norm(
                    embedding_array
                )
            )


            if not np.isclose(
                embedding_norm,
                1.0,
                atol=
                    1e-5,
            ):

                BLOCK_8_RECORD_VALIDATION_ERRORS.append(
                    f"record_{source_position}:embedding_not_unit_normalised"
                )


    if (
        article_id is not None
        and
        sentence_id is not None
        and
        sentence_index is not None
        and
        sequence_position is not None
        and
        embedding_array is not None
        and
        embedding_array.shape
        ==
        (
            BLOCK_8_ENCODER_DIMENSION,
        )
        and
        np.all(
            np.isfinite(
                embedding_array
            )
        )
    ):

        BLOCK_8_VALIDATED_REPRESENTATION_RECORDS.append(
            {
                "article_id":
                    str(
                        article_id
                    ),

                "sentence_id":
                    str(
                        sentence_id
                    ),

                "sentence_index":
                    int(
                        sentence_index
                    ),

                "sequence_position":
                    int(
                        sequence_position
                    ),

                "embedding_dimension":
                    int(
                        embedding_dimension
                    ),

                "normalised":
                    True,

                "embedding":
                    embedding_array.copy(),
            }
        )


if BLOCK_8_RECORD_VALIDATION_ERRORS:

    raise RuntimeError(
        "Notebook 05 contextual representation records failed "
        "Notebook 09 Block 8 validation. "
        f"Errors: {BLOCK_8_RECORD_VALIDATION_ERRORS}"
    )


# =============================================================================
# Canonical identity validation
# =============================================================================

BLOCK_8_SENTENCE_IDS = [
    record[
        "sentence_id"
    ]

    for record
    in BLOCK_8_VALIDATED_REPRESENTATION_RECORDS
]


BLOCK_8_ARTICLE_IDS = [
    record[
        "article_id"
    ]

    for record
    in BLOCK_8_VALIDATED_REPRESENTATION_RECORDS
]


BLOCK_8_SEQUENCE_POSITIONS = [
    record[
        "sequence_position"
    ]

    for record
    in BLOCK_8_VALIDATED_REPRESENTATION_RECORDS
]


BLOCK_8_SENTENCE_INDICES = [
    record[
        "sentence_index"
    ]

    for record
    in BLOCK_8_VALIDATED_REPRESENTATION_RECORDS
]


BLOCK_8_SENTENCE_IDS_VALID = all(
    bool(
        sentence_id
    )

    for sentence_id
    in BLOCK_8_SENTENCE_IDS
)


BLOCK_8_SENTENCE_IDS_UNIQUE = (
    len(
        BLOCK_8_SENTENCE_IDS
    )
    ==
    len(
        set(
            BLOCK_8_SENTENCE_IDS
        )
    )
)


BLOCK_8_UNIQUE_ARTICLE_IDS = list(
    dict.fromkeys(
        BLOCK_8_ARTICLE_IDS
    )
)


BLOCK_8_ARTICLE_COUNT = len(
    BLOCK_8_UNIQUE_ARTICLE_IDS
)


BLOCK_8_ARTICLE_COUNT_VALID = (
    BLOCK_8_ARTICLE_COUNT
    ==
    BLOCK_8_EXPECTED_ARTICLE_COUNT
)


BLOCK_8_SENTENCE_COUNT = len(
    BLOCK_8_VALIDATED_REPRESENTATION_RECORDS
)


BLOCK_8_SENTENCE_COUNT_VALID = (
    BLOCK_8_SENTENCE_COUNT
    ==
    BLOCK_8_EXPECTED_SENTENCE_COUNT
)


if not all(
    [
        BLOCK_8_SENTENCE_IDS_VALID,
        BLOCK_8_SENTENCE_IDS_UNIQUE,
        BLOCK_8_ARTICLE_COUNT_VALID,
        BLOCK_8_SENTENCE_COUNT_VALID,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 8 canonical article/sentence "
        "identity contract is invalid."
    )


# =============================================================================
# Canonical sequence ordering
# =============================================================================
#
# Notebook 05 persists article_id and sequence_position for every sentence.
# Notebook 09 preserves article grouping and within-article order exactly.
# Sequence positions may restart at each article boundary.
# =============================================================================

BLOCK_8_ORDERED_REPRESENTATION_RECORDS = sorted(
    BLOCK_8_VALIDATED_REPRESENTATION_RECORDS,
    key=
        lambda record:
            (
                str(
                    record[
                        "article_id"
                    ]
                ),
                int(
                    record[
                        "sequence_position"
                    ]
                ),
                int(
                    record[
                        "sentence_index"
                    ]
                ),
            ),
)


BLOCK_8_ARTICLE_RECORD_GROUPS = {}


for record in BLOCK_8_ORDERED_REPRESENTATION_RECORDS:

    BLOCK_8_ARTICLE_RECORD_GROUPS.setdefault(
        record[
            "article_id"
        ],
        [],
    ).append(
        record
    )


BLOCK_8_UNIQUE_ARTICLE_IDS = list(
    BLOCK_8_ARTICLE_RECORD_GROUPS.keys()
)


BLOCK_8_ARTICLE_COUNT = len(
    BLOCK_8_UNIQUE_ARTICLE_IDS
)


BLOCK_8_ARTICLE_COUNT_VALID = (
    BLOCK_8_ARTICLE_COUNT
    ==
    BLOCK_8_EXPECTED_ARTICLE_COUNT
)


BLOCK_8_ARTICLE_SEQUENCE_VALID = {}

BLOCK_8_ARTICLE_SEQUENCE_BASES = {}


for (
    article_id,
    article_records,
) in BLOCK_8_ARTICLE_RECORD_GROUPS.items():

    positions = [
        int(
            record[
                "sequence_position"
            ]
        )

        for record
        in article_records
    ]


    positions_unique = (
        len(
            positions
        )
        ==
        len(
            set(
                positions
            )
        )
    )


    zero_based_expected = list(
        range(
            len(
                positions
            )
        )
    )


    one_based_expected = list(
        range(
            1,
            len(
                positions
            )
            +
            1,
        )
    )


    zero_based_valid = (
        positions
        ==
        zero_based_expected
    )


    one_based_valid = (
        positions
        ==
        one_based_expected
    )


    BLOCK_8_ARTICLE_SEQUENCE_VALID[
        article_id
    ] = all(
        [
            positions_unique,
            (
                zero_based_valid
                or
                one_based_valid
            ),
        ]
    )


    BLOCK_8_ARTICLE_SEQUENCE_BASES[
        article_id
    ] = (
        "zero_based"

        if zero_based_valid

        else (
            "one_based"

            if one_based_valid

            else None
        )
    )


BLOCK_8_ALL_ARTICLE_SEQUENCES_VALID = all(
    BLOCK_8_ARTICLE_SEQUENCE_VALID.values()
)


if not all(
    [
        BLOCK_8_ARTICLE_COUNT_VALID,
        BLOCK_8_ALL_ARTICLE_SEQUENCES_VALID,
    ]
):

    raise RuntimeError(
        "Persisted Notebook 05 article-level sequence positions are invalid."
    )


BLOCK_8_SEQUENCE_POSITIONS_UNIQUE = all(
    len(
        [
            record[
                "sequence_position"
            ]

            for record
            in article_records
        ]
    )
    ==
    len(
        set(
            record[
                "sequence_position"
            ]

            for record
            in article_records
        )
    )

    for article_records
    in BLOCK_8_ARTICLE_RECORD_GROUPS.values()
)


BLOCK_8_CANONICAL_SEQUENCE_POSITION_VALID = (
    BLOCK_8_ALL_ARTICLE_SEQUENCES_VALID
)


BLOCK_8_SEQUENCE_POSITION_BASE = (
    next(
        iter(
            set(
                BLOCK_8_ARTICLE_SEQUENCE_BASES.values()
            )
        )
    )

    if (
        len(
            set(
                BLOCK_8_ARTICLE_SEQUENCE_BASES.values()
            )
        )
        ==
        1
    )

    else "mixed_by_article"
)


BLOCK_8_ORDERED_SEQUENCE_POSITIONS = [
    record[
        "sequence_position"
    ]

    for record
    in BLOCK_8_ORDERED_REPRESENTATION_RECORDS
]


# =============================================================================
# Expose canonical ordered identity contract
# =============================================================================

NOTEBOOK_09_CANONICAL_ARTICLE_IDS = tuple(
    BLOCK_8_UNIQUE_ARTICLE_IDS
)


NOTEBOOK_09_CANONICAL_SENTENCE_IDS = [
    record[
        "sentence_id"
    ]

    for record
    in BLOCK_8_ORDERED_REPRESENTATION_RECORDS
]


NOTEBOOK_09_CANONICAL_SENTENCE_INDICES = [
    record[
        "sentence_index"
    ]

    for record
    in BLOCK_8_ORDERED_REPRESENTATION_RECORDS
]


NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS = [
    record[
        "sequence_position"
    ]

    for record
    in BLOCK_8_ORDERED_REPRESENTATION_RECORDS
]


# =============================================================================
# Canonical sentence × contextual-dimension representation matrix
# =============================================================================

BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX_NUMPY = np.stack(
    [
        record[
            "embedding"
        ]

        for record
        in BLOCK_8_ORDERED_REPRESENTATION_RECORDS
    ],
    axis=
        0,
).astype(
    np.float32,
    copy=
        False,
)


BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX = torch.as_tensor(
    BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX_NUMPY,
    dtype=
        DEFAULT_DTYPE,
    device=
        DEVICE,
)


BLOCK_8_CONTEXTUAL_REPRESENTATION_SHAPE_VALID = (
    tuple(
        BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX.shape
    )
    ==
    (
        BLOCK_8_EXPECTED_SENTENCE_COUNT,
        BLOCK_8_ENCODER_DIMENSION,
    )
)


BLOCK_8_CONTEXTUAL_REPRESENTATION_FINITE = bool(
    torch.isfinite(
        BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX
    ).all().item()
)


BLOCK_8_CONTEXTUAL_REPRESENTATION_NORMS = torch.linalg.vector_norm(
    BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX,
    ord=
        2,
    dim=
        1,
)


BLOCK_8_CONTEXTUAL_REPRESENTATION_NORMALISATION_VALID = bool(
    torch.allclose(
        BLOCK_8_CONTEXTUAL_REPRESENTATION_NORMS,
        torch.ones_like(
            BLOCK_8_CONTEXTUAL_REPRESENTATION_NORMS
        ),
        rtol=
            0.0,
        atol=
            1e-5,
    )
)


if not all(
    [
        BLOCK_8_CONTEXTUAL_REPRESENTATION_SHAPE_VALID,
        BLOCK_8_CONTEXTUAL_REPRESENTATION_FINITE,
        BLOCK_8_CONTEXTUAL_REPRESENTATION_NORMALISATION_VALID,
    ]
):

    raise RuntimeError(
        "Canonical Notebook 05 contextual representation matrix "
        "failed shape, finiteness or normalisation validation."
    )


# =============================================================================
# Frozen representation architecture validation
# =============================================================================

BLOCK_8_REPRESENTATION_MODULES_EVAL = all(
    not module.training

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()
)


BLOCK_8_REPRESENTATION_PARAMETERS_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


if not all(
    [
        BLOCK_8_REPRESENTATION_MODULES_EVAL,
        BLOCK_8_REPRESENTATION_PARAMETERS_FROZEN,
    ]
):

    raise RuntimeError(
        "Inherited representation modules must remain frozen "
        "and in evaluation mode before article representation "
        "reconstruction."
    )


# =============================================================================
# Controlled frozen representation forward helper
# =============================================================================

def block_8_execute_representation_forward(
    contextual_matrix,
):

    backbone_state = (
        BLOCK_8_BACKBONE(
            contextual_matrix
        )
    )


    confluent_state = (
        BLOCK_8_GLOBAL_CONFLUENT(
            backbone_state
        )
    )


    factual_representation = (
        BLOCK_8_FACTUAL_HEAD(
            confluent_state
        )
    )


    psychological_representation = (
        BLOCK_8_PSYCHOLOGICAL_HEAD(
            confluent_state
        )
    )


    social_representation = (
        BLOCK_8_SOCIAL_HEAD(
            confluent_state
        )
    )


    return {
        "backbone_state":
            backbone_state,

        "confluent_state":
            confluent_state,

        "factual":
            factual_representation,

        "psychological":
            psychological_representation,

        "social":
            social_representation,
    }


# =============================================================================
# Execute frozen representation path twice
# =============================================================================
#
# This is inference only.
#
# No candidate transformative mechanism, Transformative Weight or recurrent
# update is executed.
# =============================================================================

with torch.no_grad():

    BLOCK_8_REPRESENTATION_FORWARD_FIRST = (
        block_8_execute_representation_forward(
            BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX
        )
    )


    BLOCK_8_REPRESENTATION_FORWARD_SECOND = (
        block_8_execute_representation_forward(
            BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX
        )
    )


NOTEBOOK_09_BLOCK_8_REPRESENTATION_FORWARD_EXECUTED = True


# =============================================================================
# Representation forward shape validation
# =============================================================================

BLOCK_8_INTERMEDIATE_SHAPES = {
    "contextual":
        tuple(
            BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX.shape
        ),

    "backbone":
        tuple(
            BLOCK_8_REPRESENTATION_FORWARD_FIRST[
                "backbone_state"
            ].shape
        ),

    "global_confluent":
        tuple(
            BLOCK_8_REPRESENTATION_FORWARD_FIRST[
                "confluent_state"
            ].shape
        ),
}


BLOCK_8_INTERMEDIATE_SHAPES_VALID = all(
    [
        (
            BLOCK_8_INTERMEDIATE_SHAPES[
                "contextual"
            ][
                0
            ]
            ==
            BLOCK_8_EXPECTED_SENTENCE_COUNT
        ),

        (
            BLOCK_8_INTERMEDIATE_SHAPES[
                "contextual"
            ][
                1
            ]
            ==
            BLOCK_8_ENCODER_DIMENSION
        ),

        (
            BLOCK_8_INTERMEDIATE_SHAPES[
                "backbone"
            ][
                0
            ]
            ==
            BLOCK_8_EXPECTED_SENTENCE_COUNT
        ),

        (
            BLOCK_8_INTERMEDIATE_SHAPES[
                "global_confluent"
            ][
                0
            ]
            ==
            BLOCK_8_EXPECTED_SENTENCE_COUNT
        ),
    ]
)


if not BLOCK_8_INTERMEDIATE_SHAPES_VALID:

    raise RuntimeError(
        "Frozen representation architecture produced unexpected "
        "intermediate dimensions. "
        f"Observed: {BLOCK_8_INTERMEDIATE_SHAPES}"
    )


# =============================================================================
# Canonical pathway representations
# =============================================================================

NOTEBOOK_09_ARTICLE_FACTUAL_REPRESENTATIONS = (
    BLOCK_8_REPRESENTATION_FORWARD_FIRST[
        "factual"
    ].detach()
    .cpu()
    .clone()
)


NOTEBOOK_09_ARTICLE_PSYCHOLOGICAL_REPRESENTATIONS = (
    BLOCK_8_REPRESENTATION_FORWARD_FIRST[
        "psychological"
    ].detach()
    .cpu()
    .clone()
)


NOTEBOOK_09_ARTICLE_SOCIAL_REPRESENTATIONS = (
    BLOCK_8_REPRESENTATION_FORWARD_FIRST[
        "social"
    ].detach()
    .cpu()
    .clone()
)


NOTEBOOK_09_ARTICLE_REPRESENTATION_MATRICES = {
    "factual":
        NOTEBOOK_09_ARTICLE_FACTUAL_REPRESENTATIONS,

    "psychological":
        NOTEBOOK_09_ARTICLE_PSYCHOLOGICAL_REPRESENTATIONS,

    "social":
        NOTEBOOK_09_ARTICLE_SOCIAL_REPRESENTATIONS,
}


# =============================================================================
# Pathway shape validation
# =============================================================================

BLOCK_8_REPRESENTATION_SHAPES = {
    pathway_name:
        tuple(
            matrix.shape
        )

    for (
        pathway_name,
        matrix,
    ) in NOTEBOOK_09_ARTICLE_REPRESENTATION_MATRICES.items()
}


BLOCK_8_EXPECTED_REPRESENTATION_SHAPES = {
    pathway_name:
        (
            BLOCK_8_EXPECTED_SENTENCE_COUNT,
            pathway_dim,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_8_PATHWAY_DIMENSIONS.items()
}


BLOCK_8_REPRESENTATION_SHAPES_VALID = (
    BLOCK_8_REPRESENTATION_SHAPES
    ==
    BLOCK_8_EXPECTED_REPRESENTATION_SHAPES
)


if not BLOCK_8_REPRESENTATION_SHAPES_VALID:

    raise RuntimeError(
        "Prepared article-level pathway representation shapes "
        "are invalid. "
        f"Observed: {BLOCK_8_REPRESENTATION_SHAPES}"
    )


# =============================================================================
# Pathway finiteness validation
# =============================================================================

BLOCK_8_REPRESENTATIONS_FINITE = {
    pathway_name:
        bool(
            torch.isfinite(
                matrix
            ).all().item()
        )

    for (
        pathway_name,
        matrix,
    ) in NOTEBOOK_09_ARTICLE_REPRESENTATION_MATRICES.items()
}


BLOCK_8_ALL_REPRESENTATIONS_FINITE = all(
    BLOCK_8_REPRESENTATIONS_FINITE.values()
)


if not BLOCK_8_ALL_REPRESENTATIONS_FINITE:

    raise RuntimeError(
        "Prepared article-level representation matrices "
        "contain non-finite values."
    )


# =============================================================================
# Representation determinism validation
# =============================================================================

BLOCK_8_REPRESENTATION_DETERMINISTIC = {
    pathway_name:
        torch.equal(
            BLOCK_8_REPRESENTATION_FORWARD_FIRST[
                pathway_name
            ],
            BLOCK_8_REPRESENTATION_FORWARD_SECOND[
                pathway_name
            ],
        )

    for pathway_name
    in BLOCK_8_PATHWAY_DIMENSIONS
}


BLOCK_8_ALL_REPRESENTATIONS_DETERMINISTIC = all(
    BLOCK_8_REPRESENTATION_DETERMINISTIC.values()
)


BLOCK_8_BACKBONE_DETERMINISTIC = torch.equal(
    BLOCK_8_REPRESENTATION_FORWARD_FIRST[
        "backbone_state"
    ],
    BLOCK_8_REPRESENTATION_FORWARD_SECOND[
        "backbone_state"
    ],
)


BLOCK_8_CONFLUENT_DETERMINISTIC = torch.equal(
    BLOCK_8_REPRESENTATION_FORWARD_FIRST[
        "confluent_state"
    ],
    BLOCK_8_REPRESENTATION_FORWARD_SECOND[
        "confluent_state"
    ],
)


BLOCK_8_COMPLETE_REPRESENTATION_FORWARD_DETERMINISTIC = all(
    [
        BLOCK_8_BACKBONE_DETERMINISTIC,
        BLOCK_8_CONFLUENT_DETERMINISTIC,
        BLOCK_8_ALL_REPRESENTATIONS_DETERMINISTIC,
    ]
)


if not BLOCK_8_COMPLETE_REPRESENTATION_FORWARD_DETERMINISTIC:

    raise RuntimeError(
        "Frozen article-level representation reconstruction "
        "is not deterministic."
    )


# =============================================================================
# Full-dimensional pathway preservation
# =============================================================================

BLOCK_8_FULL_DIMENSIONAL_PATHWAYS_VALID = all(
    int(
        NOTEBOOK_09_ARTICLE_REPRESENTATION_MATRICES[
            pathway_name
        ].shape[
            1
        ]
    )
    ==
    BLOCK_8_PATHWAY_DIMENSIONS[
        pathway_name
    ]

    for pathway_name
    in BLOCK_8_PATHWAY_DIMENSIONS
)


if not BLOCK_8_FULL_DIMENSIONAL_PATHWAYS_VALID:

    raise RuntimeError(
        "Article-level representations do not preserve the complete "
        "factual, psychological and social pathway dimensions."
    )


# =============================================================================
# Construct canonical sentence-level recurrent execution records
# =============================================================================

NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS = []


for (
    sentence_position,
    source_record,
) in enumerate(
    BLOCK_8_ORDERED_REPRESENTATION_RECORDS
):

    runtime_record = {
        "article_id":
            source_record[
                "article_id"
            ],

        "sentence_id":
            source_record[
                "sentence_id"
            ],

        "sentence_index":
            int(
                source_record[
                    "sentence_index"
                ]
            ),

        "sequence_position":
            int(
                source_record[
                    "sequence_position"
                ]
            ),
    }


    for pathway_name in BLOCK_8_PATHWAY_DIMENSIONS:

        runtime_record[
            f"{pathway_name}_representation"
        ] = (
            NOTEBOOK_09_ARTICLE_REPRESENTATION_MATRICES[
                pathway_name
            ][
                sentence_position
            ]
            .unsqueeze(
                0
            )
            .to(
                device=
                    DEVICE,

                dtype=
                    DEFAULT_DTYPE,
            )
        )


    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS.append(
        runtime_record
    )


# =============================================================================
# Sentence-level execution-record validation
# =============================================================================

BLOCK_8_RUNTIME_RECORD_SHAPES_VALID = all(
    all(
        tuple(
            record[
                f"{pathway_name}_representation"
            ].shape
        )
        ==
        (
            1,
            BLOCK_8_PATHWAY_DIMENSIONS[
                pathway_name
            ],
        )

        for pathway_name
        in BLOCK_8_PATHWAY_DIMENSIONS
    )

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
)


BLOCK_8_RUNTIME_RECORDS_FINITE = all(
    bool(
        torch.isfinite(
            record[
                f"{pathway_name}_representation"
            ]
        ).all().item()
    )

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS

    for pathway_name
    in BLOCK_8_PATHWAY_DIMENSIONS
)


BLOCK_8_RUNTIME_RECORD_IDENTITY_VALID = all(
    all(
        [
            (
                NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[
                    index
                ][
                    "article_id"
                ]
                ==
                BLOCK_8_ORDERED_REPRESENTATION_RECORDS[
                    index
                ][
                    "article_id"
                ]
            ),

            (
                NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[
                    index
                ][
                    "sentence_id"
                ]
                ==
                NOTEBOOK_09_CANONICAL_SENTENCE_IDS[
                    index
                ]
            ),

            (
                NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[
                    index
                ][
                    "sequence_position"
                ]
                ==
                NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS[
                    index
                ]
            ),
        ]
    )

    for index
    in range(
        BLOCK_8_EXPECTED_SENTENCE_COUNT
    )
)


if not all(
    [
        BLOCK_8_RUNTIME_RECORD_SHAPES_VALID,
        BLOCK_8_RUNTIME_RECORDS_FINITE,
        BLOCK_8_RUNTIME_RECORD_IDENTITY_VALID,
    ]
):

    raise RuntimeError(
        "Prepared article-level recurrent input records failed "
        "shape, numerical or identity validation."
    )


# =============================================================================
# Deterministic article reset state
# =============================================================================

NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE = {
    pathway_name:
        torch.zeros(
            (
                1,
                pathway_dim,
            ),
            dtype=
                DEFAULT_DTYPE,
            device=
                DEVICE,
            requires_grad=
                False,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_8_PATHWAY_DIMENSIONS.items()
}


BLOCK_8_RESET_STATE_SHAPES_VALID = all(
    tuple(
        NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE[
            pathway_name
        ].shape
    )
    ==
    (
        1,
        pathway_dim,
    )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_8_PATHWAY_DIMENSIONS.items()
)


BLOCK_8_RESET_STATES_ZERO = all(
    torch.equal(
        state,
        torch.zeros_like(
            state
        ),
    )

    for state
    in NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE.values()
)


BLOCK_8_RESET_STATES_NONTRAINABLE = all(
    state.requires_grad
    is False

    for state
    in NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE.values()
)


BLOCK_8_RESET_POLICY_VALID = all(
    [
        BLOCK_8_RESET_STATE_SHAPES_VALID,
        BLOCK_8_RESET_STATES_ZERO,
        BLOCK_8_RESET_STATES_NONTRAINABLE,
    ]
)


if not BLOCK_8_RESET_POLICY_VALID:

    raise RuntimeError(
        "Article-level recurrent reset-state contract is invalid."
    )


# =============================================================================
# Article-level recurrent execution policy
# =============================================================================

NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY = {
    "source":
        "Notebook 05 persisted contextual sentence representations",

    "representation_filename":
        BLOCK_8_REPRESENTATION_FILENAME,

    "canonical_order_source":
        "persisted sequence_position",

    "sequence_position_base":
        BLOCK_8_SEQUENCE_POSITION_BASE,

    "canonical_order_preserved":
        True,

    "batch_shuffling_allowed":
        False,

    "article_boundary_reset":
        True,

    "cross_article_state_propagation":
        False,

    "future_context_allowed":
        False,

    "cross_pathway_recurrence":
        False,

    "representations_are_supervision_targets":
        False,

    "supervision_substituted":
        False,

    "synthetic_representation_used":
        False,

    "full_pathway_dimensions_preserved":
        True,
}


BLOCK_8_EXECUTION_POLICY_VALID = all(
    [
        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "canonical_order_preserved"
            ]
            is True
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "batch_shuffling_allowed"
            ]
            is False
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "article_boundary_reset"
            ]
            is True
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "cross_article_state_propagation"
            ]
            is False
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "future_context_allowed"
            ]
            is False
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "cross_pathway_recurrence"
            ]
            is False
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "representations_are_supervision_targets"
            ]
            is False
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "supervision_substituted"
            ]
            is False
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "synthetic_representation_used"
            ]
            is False
        ),

        (
            NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY[
                "full_pathway_dimensions_preserved"
            ]
            is True
        ),
    ]
)


if not BLOCK_8_EXECUTION_POLICY_VALID:

    raise RuntimeError(
        "Notebook 09 article-level recurrent execution policy is invalid."
    )


# =============================================================================
# Explicit Block 8 execution boundary
# =============================================================================

NOTEBOOK_09_BLOCK_8_SEQUENCE_PREPARED = True

NOTEBOOK_09_BLOCK_8_TRANSFORMATIVE_FORWARD_EXECUTED = False

NOTEBOOK_09_BLOCK_8_WEIGHT_FORWARD_EXECUTED = False

NOTEBOOK_09_BLOCK_8_WEIGHTED_CANDIDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_8_RECURRENT_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_8_ARTICLE_RECURRENT_TRAJECTORY_EXECUTED = False

NOTEBOOK_09_BLOCK_8_UPDATED_RECURRENT_STATE_CREATED = False

NOTEBOOK_09_BLOCK_8_FUTURE_CONTEXT_USED = False

NOTEBOOK_09_BLOCK_8_LOSS_CALCULATED = False

NOTEBOOK_09_BLOCK_8_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_8_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_8_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_09_BLOCK_8_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_8_TRAINING_EXECUTED = False


# =============================================================================
# Gradient-state validation
# =============================================================================

BLOCK_8_REPRESENTATION_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_8_TRANSFORMATIVE_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_8_WEIGHT_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_8_ALL_GRADIENTS_ABSENT = all(
    [
        BLOCK_8_REPRESENTATION_GRADIENTS_ABSENT,
        BLOCK_8_TRANSFORMATIVE_GRADIENTS_ABSENT,
        BLOCK_8_WEIGHT_GRADIENTS_ABSENT,
    ]
)


if not BLOCK_8_ALL_GRADIENTS_ABSENT:

    raise RuntimeError(
        "Unexpected gradients exist after article-level "
        "sequence preparation."
    )


# =============================================================================
# Snapshot learned model state after representation preparation
# =============================================================================

BLOCK_8_REPRESENTATION_STATE_AFTER = (
    block_8_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_8_TRANSFORMATIVE_STATE_AFTER = (
    block_8_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_8_WEIGHT_STATE_AFTER = (
    block_8_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Exact parameter-state immutability
# =============================================================================

BLOCK_8_REPRESENTATION_UNCHANGED = (
    block_8_nested_state_exact(
        BLOCK_8_REPRESENTATION_STATE_BEFORE,
        BLOCK_8_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_8_TRANSFORMATIVE_UNCHANGED = (
    block_8_nested_state_exact(
        BLOCK_8_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_8_TRANSFORMATIVE_STATE_AFTER,
    )
)


BLOCK_8_WEIGHT_STATE_UNCHANGED = (
    block_8_nested_state_exact(
        BLOCK_8_WEIGHT_STATE_BEFORE,
        BLOCK_8_WEIGHT_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_8_REPRESENTATION_UNCHANGED,
        BLOCK_8_TRANSFORMATIVE_UNCHANGED,
        BLOCK_8_WEIGHT_STATE_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Learned model state changed during article-level "
        "representation preparation."
    )


# =============================================================================
# Trainability and evaluation-mode validation
# =============================================================================

BLOCK_8_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_8_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_8_WEIGHT_PARAMETERS_STILL_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_8_INHERITED_MODULES_STILL_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


if not all(
    [
        BLOCK_8_REPRESENTATION_STILL_FROZEN,
        BLOCK_8_TRANSFORMATIVE_STILL_FROZEN,
        BLOCK_8_WEIGHT_PARAMETERS_STILL_TRAINABLE,
        BLOCK_8_INHERITED_MODULES_STILL_EVAL,
    ]
):

    raise RuntimeError(
        "Notebook 09 trainability or evaluation-mode scope "
        "changed during Block 8."
    )


# =============================================================================
# Parameter accounting
# =============================================================================

BLOCK_8_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_8_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 parameter accounting changed during "
        "article-level sequence preparation."
    )


# =============================================================================
# Final article-level recurrent input contract
# =============================================================================

NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_CONTRACT = {
    "source_notebook":
        "05_training_evaluation_and_validation",

    "source_block":
        5,

    "representation_filename":
        BLOCK_8_REPRESENTATION_FILENAME,

    "representation_drive_file_id":
        BLOCK_8_REPRESENTATION_DRIVE_FILE_ID,

    "artifact_type":
        BLOCK_8_REPRESENTATION_ARTIFACT_TYPE,

    "schema_version":
        BLOCK_8_REPRESENTATION_SCHEMA_VERSION,

    "encoder_provider":
        BLOCK_8_ENCODER_PROVIDER,

    "encoder_model_id":
        BLOCK_8_ENCODER_MODEL_ID,

    "contextual_dimension":
        BLOCK_8_ENCODER_DIMENSION,

    "contextual_normalised":
        True,

    "article_count":
        BLOCK_8_ARTICLE_COUNT,

    "article_ids":
        tuple(
            NOTEBOOK_09_CANONICAL_ARTICLE_IDS
        ),

    "sentence_count":
        BLOCK_8_SENTENCE_COUNT,

    "sentence_ids":
        tuple(
            NOTEBOOK_09_CANONICAL_SENTENCE_IDS
        ),

    "sentence_indices":
        tuple(
            NOTEBOOK_09_CANONICAL_SENTENCE_INDICES
        ),

    "sequence_positions":
        tuple(
            NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS
        ),

    "sequence_position_base":
        BLOCK_8_SEQUENCE_POSITION_BASE,

    "contextual_representation_shape":
        tuple(
            BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX.shape
        ),

    "pathway_representation_shapes":
        deepcopy(
            BLOCK_8_REPRESENTATION_SHAPES
        ),

    "canonical_order_preserved":
        True,

    "article_boundary_reset":
        True,

    "future_context_allowed":
        False,

    "cross_article_state_propagation":
        False,

    "supervision_substituted":
        False,

    "synthetic_representation_used":
        False,

    "representation_forward_executed":
        True,

    "transformative_forward_executed":
        False,

    "recurrent_trajectory_executed":
        False,
}


# =============================================================================
# Final Block 8 validation
# =============================================================================

NOTEBOOK_09_BLOCK_8_ERRORS = []


BLOCK_8_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_8_PREREQUISITES_VALID,

    "pathway_dimensions_invalid":
        BLOCK_8_PATHWAY_DIMENSIONS_VALID,

    "dimension_partitions_invalid":
        BLOCK_8_ALL_DIMENSION_PARTITIONS_VALID,

    "sentence_ids_invalid":
        BLOCK_8_SENTENCE_IDS_VALID,

    "sentence_ids_not_unique":
        BLOCK_8_SENTENCE_IDS_UNIQUE,

    "article_count_invalid":
        BLOCK_8_ARTICLE_COUNT_VALID,

    "sentence_count_invalid":
        BLOCK_8_SENTENCE_COUNT_VALID,

    "sequence_positions_not_unique":
        BLOCK_8_SEQUENCE_POSITIONS_UNIQUE,

    "canonical_sequence_position_invalid":
        BLOCK_8_CANONICAL_SEQUENCE_POSITION_VALID,

    "contextual_representation_shape_invalid":
        BLOCK_8_CONTEXTUAL_REPRESENTATION_SHAPE_VALID,

    "contextual_representation_not_finite":
        BLOCK_8_CONTEXTUAL_REPRESENTATION_FINITE,

    "contextual_representation_not_normalised":
        BLOCK_8_CONTEXTUAL_REPRESENTATION_NORMALISATION_VALID,

    "intermediate_shapes_invalid":
        BLOCK_8_INTERMEDIATE_SHAPES_VALID,

    "pathway_representation_shapes_invalid":
        BLOCK_8_REPRESENTATION_SHAPES_VALID,

    "pathway_representations_not_finite":
        BLOCK_8_ALL_REPRESENTATIONS_FINITE,

    "representation_forward_not_deterministic":
        BLOCK_8_COMPLETE_REPRESENTATION_FORWARD_DETERMINISTIC,

    "full_pathway_dimensions_not_preserved":
        BLOCK_8_FULL_DIMENSIONAL_PATHWAYS_VALID,

    "runtime_record_shapes_invalid":
        BLOCK_8_RUNTIME_RECORD_SHAPES_VALID,

    "runtime_records_not_finite":
        BLOCK_8_RUNTIME_RECORDS_FINITE,

    "runtime_record_identity_invalid":
        BLOCK_8_RUNTIME_RECORD_IDENTITY_VALID,

    "reset_policy_invalid":
        BLOCK_8_RESET_POLICY_VALID,

    "execution_policy_invalid":
        BLOCK_8_EXECUTION_POLICY_VALID,

    "unexpected_gradients":
        BLOCK_8_ALL_GRADIENTS_ABSENT,

    "representation_state_changed":
        BLOCK_8_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_8_TRANSFORMATIVE_UNCHANGED,

    "weight_state_changed":
        BLOCK_8_WEIGHT_STATE_UNCHANGED,

    "representation_not_frozen":
        BLOCK_8_REPRESENTATION_STILL_FROZEN,

    "transformative_not_frozen":
        BLOCK_8_TRANSFORMATIVE_STILL_FROZEN,

    "weight_parameters_not_trainable":
        BLOCK_8_WEIGHT_PARAMETERS_STILL_TRAINABLE,

    "inherited_modules_not_eval":
        BLOCK_8_INHERITED_MODULES_STILL_EVAL,

    "parameter_accounting_invalid":
        BLOCK_8_PARAMETER_ACCOUNTING_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_8_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_8_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation audit
# =============================================================================

BLOCK_8_PROHIBITED_OPERATIONS = {
    "transformative_forward_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_TRANSFORMATIVE_FORWARD_EXECUTED,

    "weight_forward_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_WEIGHT_FORWARD_EXECUTED,

    "weighted_candidate_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_WEIGHTED_CANDIDATE_EXECUTED,

    "recurrent_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_RECURRENT_UPDATE_EXECUTED,

    "article_trajectory_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_ARTICLE_RECURRENT_TRAJECTORY_EXECUTED,

    "updated_state_incorrectly_created":
        NOTEBOOK_09_BLOCK_8_UPDATED_RECURRENT_STATE_CREATED,

    "future_context_incorrectly_used":
        NOTEBOOK_09_BLOCK_8_FUTURE_CONTEXT_USED,

    "loss_incorrectly_calculated":
        NOTEBOOK_09_BLOCK_8_LOSS_CALCULATED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_8_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_BACKWARD_PASS_EXECUTED,

    "gradient_clipping_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_GRADIENT_CLIPPING_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_8_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_8_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_8_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 8 state
# =============================================================================

NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_CONTRACT_VALID = (
    len(
        NOTEBOOK_09_BLOCK_8_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_READY = (
    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_CONTRACT_VALID
)


NOTEBOOK_09_BLOCK_8_VALID = (
    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_CONTRACT_VALID
)


if not NOTEBOOK_09_BLOCK_8_VALID:

    raise RuntimeError(
        "Notebook 09 Block 8 article-level recurrent input "
        "contract validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_8_ERRORS}"
    )


NOTEBOOK_09_BLOCK_8_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_8_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_8,

    "block_name":
        NOTEBOOK_09_BLOCK_8_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_8_VERSION,

    "representation_filename":
        BLOCK_8_REPRESENTATION_FILENAME,

    "representation_drive_file_id":
        BLOCK_8_REPRESENTATION_DRIVE_FILE_ID,

    "representation_candidate_count":
        len(
            BLOCK_8_REPRESENTATION_FILE_CANDIDATES
        ),

    "encoder_provider":
        BLOCK_8_ENCODER_PROVIDER,

    "encoder_model_id":
        BLOCK_8_ENCODER_MODEL_ID,

    "encoder_dimension":
        BLOCK_8_ENCODER_DIMENSION,

    "article_count":
        BLOCK_8_ARTICLE_COUNT,

    "sentence_count":
        BLOCK_8_SENTENCE_COUNT,

    "sequence_position_base":
        BLOCK_8_SEQUENCE_POSITION_BASE,

    "contextual_representation_shape":
        tuple(
            BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX.shape
        ),

    "representation_shapes":
        deepcopy(
            BLOCK_8_REPRESENTATION_SHAPES
        ),

    "representation_deterministic":
        BLOCK_8_COMPLETE_REPRESENTATION_FORWARD_DETERMINISTIC,

    "reset_policy_valid":
        BLOCK_8_RESET_POLICY_VALID,

    "representation_forward_executed":
        NOTEBOOK_09_BLOCK_8_REPRESENTATION_FORWARD_EXECUTED,

    "transformative_forward_executed":
        NOTEBOOK_09_BLOCK_8_TRANSFORMATIVE_FORWARD_EXECUTED,

    "recurrent_update_executed":
        NOTEBOOK_09_BLOCK_8_RECURRENT_UPDATE_EXECUTED,

    "article_trajectory_executed":
        NOTEBOOK_09_BLOCK_8_ARTICLE_RECURRENT_TRAJECTORY_EXECUTED,

    "representation_parameters":
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT,

    "transformative_parameters":
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT,

    "transformative_weight_parameters":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT,

    "total_model_parameters":
        NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION,

    "input_contract_valid":
        NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_CONTRACT_VALID,

    "article_execution_ready":
        NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_8_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_8_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 8: "
    "Article-Level Recurrent Execution Contract "
    "and Canonical Sequence Preparation"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_8_VERSION}"
)

print("-" * 72)

print(
    "Persisted contextual representation source"
)

print(
    f"Filename                     : "
    f"{BLOCK_8_REPRESENTATION_FILENAME}"
)

print(
    f"Drive file ID                : "
    f"{BLOCK_8_REPRESENTATION_DRIVE_FILE_ID}"
)

print(
    f"Candidate files inspected    : "
    f"{len(BLOCK_8_REPRESENTATION_FILE_CANDIDATES)}"
)

print(
    f"Artifact type                : "
    f"{BLOCK_8_REPRESENTATION_ARTIFACT_TYPE}"
)

print(
    f"Schema version               : "
    f"{BLOCK_8_REPRESENTATION_SCHEMA_VERSION}"
)

print("-" * 72)

print(
    "Encoder contract"
)

print(
    f"Provider                     : "
    f"{BLOCK_8_PERSISTED_ENCODER.get('provider')}"
)

print(
    f"Model                        : "
    f"{BLOCK_8_PERSISTED_ENCODER.get('model_id')}"
)

print(
    f"Embedding dimension          : "
    f"{BLOCK_8_PERSISTED_ENCODER.get('embedding_dimension')}"
)

print(
    f"Normalised                   : "
    f"{BLOCK_8_PERSISTED_ENCODER.get('normalised')}"
)

print(
    f"Article context used         : "
    f"{BLOCK_8_PERSISTED_ENCODER.get('article_context_used')}"
)

print(
    f"Future context used          : "
    f"{BLOCK_8_PERSISTED_ENCODER.get('future_context_used')}"
)

print("-" * 72)

print(
    "Canonical article sequence"
)

print(
    f"Articles                     : "
    f"{BLOCK_8_ARTICLE_COUNT}"
)

print(
    f"Sentences                    : "
    f"{BLOCK_8_SENTENCE_COUNT}"
)

print(
    f"Sentence IDs valid           : "
    f"{BLOCK_8_SENTENCE_IDS_VALID}"
)

print(
    f"Sentence IDs unique          : "
    f"{BLOCK_8_SENTENCE_IDS_UNIQUE}"
)

print(
    f"Sequence positions unique    : "
    f"{BLOCK_8_SEQUENCE_POSITIONS_UNIQUE}"
)

print(
    f"Sequence position base       : "
    f"{BLOCK_8_SEQUENCE_POSITION_BASE}"
)

print(
    f"Canonical order valid        : "
    f"{BLOCK_8_CANONICAL_SEQUENCE_POSITION_VALID}"
)

print("-" * 72)

print(
    "Contextual representation input"
)

print(
    f"Representation shape         : "
    f"{tuple(BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX.shape)}"
)

print(
    f"Shape valid                  : "
    f"{BLOCK_8_CONTEXTUAL_REPRESENTATION_SHAPE_VALID}"
)

print(
    f"Values finite               : "
    f"{BLOCK_8_CONTEXTUAL_REPRESENTATION_FINITE}"
)

print(
    f"Normalisation valid          : "
    f"{BLOCK_8_CONTEXTUAL_REPRESENTATION_NORMALISATION_VALID}"
)

print("-" * 72)

print(
    "Frozen representation forward path"
)

print(
    f"Contextual input             : "
    f"{BLOCK_8_INTERMEDIATE_SHAPES['contextual']}"
)

print(
    f"Backbone output              : "
    f"{BLOCK_8_INTERMEDIATE_SHAPES['backbone']}"
)

print(
    f"Global confluent output      : "
    f"{BLOCK_8_INTERMEDIATE_SHAPES['global_confluent']}"
)

print(
    f"Backbone deterministic       : "
    f"{BLOCK_8_BACKBONE_DETERMINISTIC}"
)

print(
    f"Confluent deterministic      : "
    f"{BLOCK_8_CONFLUENT_DETERMINISTIC}"
)

print("-" * 72)

print(
    "Prepared pathway representations"
)

for pathway_name in BLOCK_8_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} matrix shape       : "
        f"{BLOCK_8_REPRESENTATION_SHAPES[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} finite             : "
        f"{BLOCK_8_REPRESENTATIONS_FINITE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deterministic      : "
        f"{BLOCK_8_REPRESENTATION_DETERMINISTIC[pathway_name]}"
    )

print("-" * 72)

print(
    "Article recurrent execution contract"
)

print(
    f"Canonical order preserved    : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY['canonical_order_preserved']}"
)

print(
    f"Batch shuffling allowed      : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY['batch_shuffling_allowed']}"
)

print(
    f"Article-boundary reset       : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY['article_boundary_reset']}"
)

print(
    f"Cross-article propagation    : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY['cross_article_state_propagation']}"
)

print(
    f"Future context allowed       : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY['future_context_allowed']}"
)

print(
    f"Supervision substituted      : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY['supervision_substituted']}"
)

print(
    f"Synthetic representations    : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_POLICY['synthetic_representation_used']}"
)

print(
    f"Reset policy valid           : "
    f"{BLOCK_8_RESET_POLICY_VALID}"
)

print("-" * 72)

print(
    "Parameter and state validation"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Transformative Weight params : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Total model parameters       : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print(
    f"Representation unchanged     : "
    f"{BLOCK_8_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged     : "
    f"{BLOCK_8_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Weight state unchanged       : "
    f"{BLOCK_8_WEIGHT_STATE_UNCHANGED}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_8_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_8_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"Weight parameters trainable  : "
    f"{BLOCK_8_WEIGHT_PARAMETERS_STILL_TRAINABLE}"
)

print(
    f"All gradients absent         : "
    f"{BLOCK_8_ALL_GRADIENTS_ABSENT}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Sequence prepared            : "
    f"{NOTEBOOK_09_BLOCK_8_SEQUENCE_PREPARED}"
)

print(
    f"Representation forward       : "
    f"{NOTEBOOK_09_BLOCK_8_REPRESENTATION_FORWARD_EXECUTED}"
)

print(
    f"Transform forward            : "
    f"{NOTEBOOK_09_BLOCK_8_TRANSFORMATIVE_FORWARD_EXECUTED}"
)

print(
    f"Weight forward               : "
    f"{NOTEBOOK_09_BLOCK_8_WEIGHT_FORWARD_EXECUTED}"
)

print(
    f"Weighted candidate executed  : "
    f"{NOTEBOOK_09_BLOCK_8_WEIGHTED_CANDIDATE_EXECUTED}"
)

print(
    f"Recurrent update executed    : "
    f"{NOTEBOOK_09_BLOCK_8_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"Article trajectory executed  : "
    f"{NOTEBOOK_09_BLOCK_8_ARTICLE_RECURRENT_TRAJECTORY_EXECUTED}"
)

print(
    f"Future context used          : "
    f"{NOTEBOOK_09_BLOCK_8_FUTURE_CONTEXT_USED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_BLOCK_8_LOSS_CALCULATED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_8_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_8_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_8_PARAMETER_UPDATE_EXECUTED}"
)

print("-" * 72)

print(
    f"Input contract valid         : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_CONTRACT_VALID}"
)

print(
    f"Article execution ready      : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_8_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_8_COMPLETE}"
)

print("=" * 72)

print(
    "The persisted Notebook 05 contextual sentence representation "
    "artefact was restored successfully."
)

print(
    f"Its {BLOCK_8_SENTENCE_COUNT} canonical sentence record(s) across "
    f"{BLOCK_8_ARTICLE_COUNT} article(s) preserve article identity, "
    "sentence identity and persisted within-article ordering from Notebook 05."
)

print(
    f"The restored contextual representation matrix has shape "
    f"{tuple(BLOCK_8_CONTEXTUAL_REPRESENTATION_MATRIX.shape)}, contains "
    "finite unit-normalised persisted embeddings and uses no synthetic fallback."
)

print(
    "The frozen inherited representation architecture was executed "
    "deterministically through the backbone, global confluent module "
    "and factual, psychological and social representation heads."
)

print(
    f"The resulting pathway representation matrices have shapes "
    f"{BLOCK_8_REPRESENTATION_SHAPES}."
)

print(
    "Supervision targets were not substituted for representation inputs."
)

print(
    "The article-level recurrent reset and canonical execution contract "
    "are now prepared without executing a candidate transformation or "
    "recurrent state update."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} representation "
    "and transformative parameters remain frozen and exactly unchanged, while "
    f"all {NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT} "
    "Transformative-Weight parameter(s) remain trainable and unchanged."
)

print(
    "No transformative forward pass, Transformative-Weight forward pass, "
    "weighted candidate transformation, recurrent trajectory, loss, "
    "optimiser, backward pass or parameter update was executed."
)

print(
    "Notebook 09 may now proceed to controlled article-level recurrent "
    "trajectory execution."
)

print("=" * 72)

Media AI — Notebook 09, Block 8: Article-Level Recurrent Execution Contract and Canonical Sequence Preparation
Block version                : 1.3
------------------------------------------------------------------------
Persisted contextual representation source
Filename                     : notebook_05_contextual_sentence_representations.json
Drive file ID                : 11_j_CebqBpMSDLlccGwGROr5hklTBYRm
Candidate files inspected    : 1
Artifact type                : media_ai_contextual_sentence_representations
Schema version               : 1.0
------------------------------------------------------------------------
Encoder contract
Provider                     : sentence-transformers
Model                        : sentence-transformers/all-mpnet-base-v2
Embedding dimension          : 768
Normalised                   : True
Article context used         : False
Future context used          : False
------------------------------------------------------------------------
Canonical art

## Block 9 — Controlled Article-Level Recurrent Trajectory Execution and Validation

This block executes the first complete recurrent trajectory over the canonical article sequence.

Block 8 restored the persisted Notebook 05 contextual sentence representations, validated the canonical one-article, 16-sentence ordering, reconstructed the factual, psychological, and social pathway representations through the frozen inherited representation architecture, and prepared sentence-aligned recurrent input records.

The present block now applies the validated transformative and recurrent mechanisms across the complete article.

Its purpose is to verify that the factual, psychological, and social recurrent states evolve causally and deterministically over the canonical 16-sentence sequence while preserving pathway dimensionality, active/deferred gating semantics, article-boundary reset behaviour, parameter immutability, and the established no-future-context constraint.

No optimisation is performed.

No loss is calculated, no optimiser is created, no backward pass is executed, and no model parameter is updated.

### Article-level recurrent input

Block 8 prepared the canonical ordered sentence sequence

$$
\mathcal{A}
=
\left[
\mathcal{R}_1,
\mathcal{R}_2,
\ldots,
\mathcal{R}_{16}
\right],
$$

where each recurrent input record is

$$
\mathcal{R}_t
=
\left(
s_t,
a,
r_t^{(F)},
r_t^{(P)},
r_t^{(S)}
\right).
$$

The article identity \(a\) is constant across the complete pilot sequence, while the sentence identities

$$
s_1,
s_2,
\ldots,
s_{16}
$$

follow the persisted canonical `sequence_position` ordering restored from Notebook 05.

The pathway representations have dimensions

$$
r_t^{(F)}
\in
\mathbb{R}^{1\times10},
$$

$$
r_t^{(P)}
\in
\mathbb{R}^{1\times34},
$$

and

$$
r_t^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

### Initial recurrent state

The article begins from the deterministic zero recurrent state established in Block 5 and reaffirmed in Block 8.

For the factual pathway,

$$
h_0^{(F)}
=
\mathbf{0}_{1\times10},
$$

for the psychological pathway,

$$
h_0^{(P)}
=
\mathbf{0}_{1\times34},
$$

and for the social pathway,

$$
h_0^{(S)}
=
\mathbf{0}_{1\times7}.
$$

Because the current pilot contains a single article, this reset occurs once at the beginning of the sequence.

No recurrent state from any earlier validation trajectory is reused.

### Candidate transformation at each sentence

For each sentence position

$$
t\in\{1,\ldots,16\},
$$

the inherited pathway-specific transformative mechanism receives the current pathway representation and the immediately preceding recurrent state.

For pathway \(k\in\{F,P,S\}\),

$$
c_t^{(k)}
=
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right).
$$

The candidate-transformation dimensions remain

$$
c_t^{(F)}
\in
\mathbb{R}^{1\times10},
$$

$$
c_t^{(P)}
\in
\mathbb{R}^{1\times34},
$$

and

$$
c_t^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

Every candidate transformation must remain finite.

### Transformative Weight

The effective Transformative Weight is applied independently to each pathway.

For pathway \(k\),

$$
TW_k
\in
[0,1]^{d_k}.
$$

The current Block 9 execution uses the Transformative-Weight state inherited from Blocks 3–4.

At the current initial parameterisation,

$$
TW_{k,j}
=
0.5
$$

for activation-eligible dimensions

$$
j\in A_k,
$$

and

$$
TW_{k,j}
=
0
$$

for deferred dimensions

$$
j\in D_k.
$$

The Transformative Weight is therefore a dimension-wise gate on the candidate transformation.

### Weighted transformation contribution

The recurrent increment is defined by

$$
\Delta h_t^{(k)}
=
TW_k
\odot
c_t^{(k)}.
$$

The weighted contribution preserves complete pathway dimensionality:

$$
\Delta h_t^{(F)}
\in
\mathbb{R}^{1\times10},
$$

$$
\Delta h_t^{(P)}
\in
\mathbb{R}^{1\times34},
$$

and

$$
\Delta h_t^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

All weighted contributions must remain finite.

### Recurrent update

The article-level recurrent state evolves according to the additive update contract

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
\Delta h_t^{(k)}.
$$

Equivalently,

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k
\odot
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right).
$$

This update is applied sequentially from sentence 1 through sentence 16.

### Canonical causal traversal

The article is traversed strictly according to the canonical sequence prepared in Block 8.

The recurrent trajectory therefore follows

$$
h_0
\rightarrow
h_1
\rightarrow
h_2
\rightarrow
\cdots
\rightarrow
h_{16}.
$$

At sentence \(t\), only

$$
r_t^{(k)}
$$

and

$$
h_{t-1}^{(k)}
$$

may influence the current candidate transformation and recurrent update.

No representation from sentence

$$
t+1
$$

or later may influence

$$
h_t^{(k)}.
$$

Thus,

$$
h_t^{(k)}
=
f
\left(
r_1^{(k)},
r_2^{(k)},
\ldots,
r_t^{(k)}
\right).
$$

### Previous-state continuity

The defining recurrent continuity condition is

$$
h_{t-1}^{(k)}
=
\text{previous state supplied at step }t.
$$

Therefore, for every

$$
t>1,
$$

the `previous_state` supplied to the transformative mechanism must equal the `updated_state` produced by the immediately preceding sentence.

This identity must hold exactly throughout the complete article trajectory.

### Active-dimension behaviour

For active dimensions,

$$
j\in A_k,
$$

the current effective Transformative Weight is

$$
TW_{k,j}
=
0.5.
$$

Therefore,

$$
\Delta h_{t,j}^{(k)}
=
0.5
c_{t,j}^{(k)}.
$$

The weighted contribution must preserve the sign of the candidate transformation.

If

$$
c_{t,j}^{(k)}>0,
$$

then

$$
\Delta h_{t,j}^{(k)}>0,
$$

and if

$$
c_{t,j}^{(k)}<0,
$$

then

$$
\Delta h_{t,j}^{(k)}<0.
$$

### Deferred-dimension behaviour

For deferred dimensions,

$$
j\in D_k,
$$

the effective Transformative Weight remains

$$
TW_{k,j}
=
0.
$$

Therefore,

$$
\Delta h_{t,j}^{(k)}
=
0.
$$

The recurrent update becomes

$$
h_{t,j}^{(k)}
=
h_{t-1,j}^{(k)}.
$$

Since the article begins from

$$
h_{0,j}^{(k)}
=
0,
$$

deferred dimensions must remain

$$
h_{t,j}^{(k)}
=
0
$$

for every sentence in the current pilot article.

This trajectory-wide invariant must be validated explicitly.

### Cumulative recurrent-state identity

Because the recurrent state begins from zero,

$$
h_0^{(k)}
=
0,
$$

the state after sentence \(t\) can also be expressed as the cumulative sum of all previous weighted candidate transformations:

$$
h_t^{(k)}
=
\sum_{\tau=1}^{t}
\Delta h_{\tau}^{(k)}.
$$

Equivalently,

$$
h_t^{(k)}
=
\sum_{\tau=1}^{t}
TW_k
\odot
c_{\tau}^{(k)}.
$$

For the complete article,

$$
h_{16}^{(k)}
=
\sum_{t=1}^{16}
TW_k
\odot
c_t^{(k)}.
$$

Block 9 should validate this cumulative identity independently from the stepwise recurrent update.

### Dimensional invariance

Every recurrent quantity must preserve the pathway-specific dimensional contract.

For all

$$
t\in\{0,\ldots,16\},
$$

the factual state remains

$$
h_t^{(F)}
\in
\mathbb{R}^{1\times10},
$$

the psychological state remains

$$
h_t^{(P)}
\in
\mathbb{R}^{1\times34},
$$

and the social state remains

$$
h_t^{(S)}
\in
\mathbb{R}^{1\times7}.
$$

No active-only compression is allowed.

Deferred dimensions remain part of the recurrent state even though they currently receive zero transformative contribution.

### Numerical validity

Every intermediate quantity must remain finite.

For every pathway and sentence position, Block 9 validates

$$
\operatorname{isfinite}
\left(
c_t^{(k)}
\right),
$$

$$
\operatorname{isfinite}
\left(
\Delta h_t^{(k)}
\right),
$$

and

$$
\operatorname{isfinite}
\left(
h_t^{(k)}
\right).
$$

The complete 16-step trajectory must remain numerically valid.

### Deterministic article execution

The complete article trajectory must be reproducible exactly.

A second execution starting from fresh deterministic zero states and using the same canonical representations, transformative modules, and Transformative Weights must produce identical candidate transformations, weighted contributions, and recurrent states at every sentence position.

Thus,

$$
c_{t,\mathrm{run1}}^{(k)}
=
c_{t,\mathrm{run2}}^{(k)},
$$

$$
\Delta h_{t,\mathrm{run1}}^{(k)}
=
\Delta h_{t,\mathrm{run2}}^{(k)},
$$

and

$$
h_{t,\mathrm{run1}}^{(k)}
=
h_{t,\mathrm{run2}}^{(k)}.
$$

This must hold for all three pathways and all 16 sentences.

### Article reset reproducibility

Each complete article execution must begin from a newly constructed zero state.

The second deterministic validation run must therefore satisfy

$$
h_{0,\mathrm{run2}}^{(k)}
=
\mathbf{0}_{d_k},
$$

rather than beginning from

$$
h_{16,\mathrm{run1}}^{(k)}.
$$

This verifies that article-boundary state reset is part of the actual recurrent execution contract rather than merely a documented policy.

### Pathway independence

The factual, psychological, and social trajectories remain independent.

The factual pathway uses only

$$
r_t^{(F)},
\quad
h_{t-1}^{(F)},
\quad
T_F,
\quad
TW_F.
$$

The psychological pathway uses only

$$
r_t^{(P)},
\quad
h_{t-1}^{(P)},
\quad
T_P,
\quad
TW_P.
$$

The social pathway uses only

$$
r_t^{(S)},
\quad
h_{t-1}^{(S)},
\quad
T_S,
\quad
TW_S.
$$

No pathway may consume another pathway's recurrent state, candidate transformation, or Transformative Weight.

### Sentence identity preservation

Every recurrent step must remain associated with the sentence that generated it.

The article-level trajectory record should preserve

$$
\mathcal{T}_t
=
\left(
a,
s_t,
p_t,
r_t,
c_t,
TW,
\Delta h_t,
h_t
\right),
$$

where \(p_t\) is the persisted canonical sequence position.

This preserves traceability from every recurrent state back to the exact sentence representation that produced it.

### Representation state versus recurrent state

The article-level representation

$$
r_t^{(k)}
$$

and the recurrent state

$$
h_t^{(k)}
$$

remain conceptually distinct.

The representation describes the current sentence in the pathway-specific model space.

The recurrent state describes accumulated transformation across the ordered article trajectory.

Therefore,

$$
r_t^{(k)}
\neq
h_t^{(k)}
$$

as an architectural identity.

Similarly,

$$
c_t^{(k)}
$$

remains a candidate transformation and is not itself the recurrent state.

### No representation-model re-execution inside recurrence

Block 8 already prepared and validated the complete article-level representation matrices.

Block 9 should therefore consume those prepared representations directly.

The representation architecture does not need to be executed again inside each recurrent step.

This avoids redundant computation and ensures that recurrent execution is tested independently from representation reconstruction.

### Parameter immutability

Article-level recurrent execution changes runtime state only.

The inherited representation parameters must remain

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}},
$$

the transformative parameters must remain

$$
\theta_T^{\mathrm{after}}
=
\theta_T^{\mathrm{before}},
$$

and the Transformative-Weight parameters must remain

$$
\theta_{TW}^{\mathrm{after}}
=
\theta_{TW}^{\mathrm{before}}.
$$

The complete model parameter count remains

$$
902{,}215.
$$

### Gradient boundary

The article trajectory is a forward-validation operation only.

The 25 Transformative-Weight parameters remain trainable in principle, but no gradient graph is retained and no gradients are accumulated.

The inherited representation and transformative parameters remain frozen.

Thus recurrent state evolves, but learned parameters do not.

### No optimisation in Block 9

Block 9 performs no training.

Accordingly:

- no supervision objective is assembled;
- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- no gradient clipping is applied;
- no optimiser step is executed;
- and no parameter update occurs.

### Article-level trajectory outputs

At completion, Block 9 should expose a traceable recurrent trajectory for each pathway.

For each sentence position \(t\), the trajectory should retain:

- article identity;
- sentence identity;
- canonical sequence position;
- current pathway representation;
- preceding recurrent state;
- candidate transformation;
- effective Transformative Weight;
- weighted candidate contribution;
- updated recurrent state.

The resulting structure should be suitable for later trajectory inspection, attribution, Transformative-Weight training, and final Notebook 09 persistence.

### Block 9 completion contract

Block 9 is complete only when:

- the prepared Block 8 article-level recurrent input contract is valid;
- all 16 canonical sentences are traversed exactly once and in canonical order;
- traversal begins from fresh deterministic zero states;
- factual, psychological, and social pathways remain independent;
- the inherited transformative mechanism executes successfully at every sentence;
- candidate transformations preserve shapes \((1,10)\), \((1,34)\), and \((1,7)\);
- all candidate transformations remain finite;
- Transformative-Weight forward execution succeeds at every sentence;
- effective weight vectors preserve dimensions 10, 34, and 7;
- effective weights remain finite and bounded;
- weighted candidate contributions preserve complete pathway dimensionality;
- active dimensions retain the current Transformative-Weight scaling contract;
- deferred dimensions produce zero weighted contribution;
- deferred recurrent-state dimensions remain exactly zero throughout the article;
- every updated state preserves its complete pathway shape;
- every recurrent state remains finite;
- every step receives the updated state from the immediately preceding sentence;
- the additive identity

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
\Delta h_t^{(k)}
$$

holds at every sentence;

- the cumulative identity

$$
h_t^{(k)}
=
\sum_{\tau=1}^{t}
\Delta h_{\tau}^{(k)}
$$

holds throughout the article;

- the final article state satisfies

$$
h_{16}^{(k)}
=
\sum_{t=1}^{16}
TW_k\odot c_t^{(k)};
$$

- repeated complete article execution is exactly deterministic;
- each repeated article run begins from a fresh zero state;
- sentence identities and sequence positions remain aligned to every recurrent step;
- no future sentence information is used;
- no cross-article state propagation occurs;
- no representation-model forward pass is required inside recurrent execution;
- the inherited 902,190 parameters remain frozen and exactly unchanged;
- the 25 Transformative-Weight parameters remain trainable but unchanged;
- the complete model parameter count remains 902,215;
- no gradients are accumulated;
- no loss is calculated;
- no optimiser is created;
- no backward pass is executed;
- and no parameter update occurs.

Successful completion establishes the first complete article-level recurrent trajectory

$$
\boxed{
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k
\odot
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right)
}
$$

for all

$$
t\in\{1,\ldots,16\}
$$

and

$$
k\in\{F,P,S\}.
$$

Notebook 09 may then proceed to Transformative-Weight objective construction and controlled learning policy, using the validated article-level recurrent trajectory as its causal execution foundation.

In [59]:
# =============================================================================
# Media AI — Notebook 09
# Block 9: Controlled Article-Level Recurrent Trajectory Execution
#          and Validation
# =============================================================================

from copy import deepcopy

import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_9 = 9

NOTEBOOK_09_BLOCK_9_NAME = (
    "Controlled Article-Level Recurrent Trajectory Execution "
    "and Validation"
)

NOTEBOOK_09_BLOCK_9_VERSION = "1.1"


# =============================================================================
# Required inherited runtime contract
# =============================================================================

BLOCK_9_REQUIRED_OBJECTS = (
    # Block 1
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_BLOCK_1_COMPLETE",
    "NOTEBOOK_09_BLOCK_1_VALID",
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",
    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # Transformative-Weight architecture
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",
    "BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS",

    # Recurrent architecture
    "NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS",
    "NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES",

    # Block 8
    "NOTEBOOK_09_BLOCK_8_COMPLETE",
    "NOTEBOOK_09_BLOCK_8_VALID",
    "NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_CONTRACT_VALID",
    "NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_READY",
    "NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS",
    "NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE",
    "NOTEBOOK_09_CANONICAL_ARTICLE_IDS",
    "NOTEBOOK_09_CANONICAL_SENTENCE_IDS",
    "NOTEBOOK_09_CANONICAL_SENTENCE_INDICES",
    "NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS",

    # Environment
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_9_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_9_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_9_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 9 prerequisites are not initialised. "
        f"Missing: {BLOCK_9_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_9_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,
        NOTEBOOK_09_BLOCK_1_COMPLETE is True,
        NOTEBOOK_09_BLOCK_1_VALID is True,
        NOTEBOOK_09_BLOCK_8_COMPLETE is True,
        NOTEBOOK_09_BLOCK_8_VALID is True,
        NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_CONTRACT_VALID is True,
        NOTEBOOK_09_ARTICLE_RECURRENT_EXECUTION_READY is True,
    ]
)


if not BLOCK_9_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Block 8 must be valid and complete before "
        "article-level recurrent trajectory execution."
    )


# =============================================================================
# Canonical pathway contract
# =============================================================================

BLOCK_9_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
)


BLOCK_9_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            BLOCK_9_PATHWAY_DIMENSIONS,
            dict,
        ),

        bool(
            BLOCK_9_PATHWAY_DIMENSIONS
        ),

        (
            set(
                BLOCK_9_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                BLOCK_1_TRANSFORMATIVE_MODULES.keys()
            )
            ==
            set(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys()
            )
        ),

        all(
            isinstance(
                pathway_dim,
                int,
            )
            and
            pathway_dim > 0

            for pathway_dim
            in BLOCK_9_PATHWAY_DIMENSIONS.values()
        ),
    ]
)


if not BLOCK_9_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 9 pathway dimensions are invalid."
    )


# =============================================================================
# Active / deferred pathway partitions
# =============================================================================

BLOCK_9_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES.items()
}


BLOCK_9_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES.items()
}


BLOCK_9_DIMENSION_PARTITIONS_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_9_PATHWAY_DIMENSIONS.items():

    active_set = set(
        BLOCK_9_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    deferred_set = set(
        BLOCK_9_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )


    BLOCK_9_DIMENSION_PARTITIONS_VALID[
        pathway_name
    ] = all(
        [
            active_set.isdisjoint(
                deferred_set
            ),

            (
                active_set
                |
                deferred_set
            )
            ==
            set(
                range(
                    pathway_dim
                )
            ),
        ]
    )


BLOCK_9_ALL_DIMENSION_PARTITIONS_VALID = all(
    BLOCK_9_DIMENSION_PARTITIONS_VALID.values()
)


if not BLOCK_9_ALL_DIMENSION_PARTITIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 9 active/deferred dimension partitions "
        "are invalid."
    )


# =============================================================================
# Canonical corpus identity contract
# =============================================================================

BLOCK_9_EXPECTED_ARTICLE_IDS = tuple(
    NOTEBOOK_09_CANONICAL_ARTICLE_IDS
)


BLOCK_9_EXPECTED_ARTICLE_COUNT = len(
    BLOCK_9_EXPECTED_ARTICLE_IDS
)


BLOCK_9_EXPECTED_SENTENCE_COUNT = len(
    NOTEBOOK_09_CANONICAL_SENTENCE_IDS
)


BLOCK_9_INPUT_RECORD_COUNT = len(
    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
)


BLOCK_9_INPUT_SENTENCE_IDS = [
    record[
        "sentence_id"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_9_INPUT_SENTENCE_INDICES = [
    record[
        "sentence_index"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_9_INPUT_SEQUENCE_POSITIONS = [
    record[
        "sequence_position"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_9_INPUT_ARTICLE_IDS = [
    record[
        "article_id"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_9_OBSERVED_ARTICLE_IDS = tuple(
    dict.fromkeys(
        BLOCK_9_INPUT_ARTICLE_IDS
    )
)


BLOCK_9_SENTENCE_COUNT_VALID = (
    BLOCK_9_INPUT_RECORD_COUNT
    ==
    BLOCK_9_EXPECTED_SENTENCE_COUNT
)


BLOCK_9_CANONICAL_SENTENCE_IDS_VALID = (
    BLOCK_9_INPUT_SENTENCE_IDS
    ==
    list(
        NOTEBOOK_09_CANONICAL_SENTENCE_IDS
    )
)


BLOCK_9_CANONICAL_SENTENCE_INDICES_VALID = (
    BLOCK_9_INPUT_SENTENCE_INDICES
    ==
    list(
        NOTEBOOK_09_CANONICAL_SENTENCE_INDICES
    )
)


BLOCK_9_CANONICAL_SEQUENCE_POSITIONS_VALID = (
    BLOCK_9_INPUT_SEQUENCE_POSITIONS
    ==
    list(
        NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS
    )
)


BLOCK_9_ARTICLE_IDENTITIES_VALID = (
    BLOCK_9_OBSERVED_ARTICLE_IDS
    ==
    BLOCK_9_EXPECTED_ARTICLE_IDS
)


BLOCK_9_CANONICAL_SEQUENCE_VALID = all(
    [
        BLOCK_9_EXPECTED_ARTICLE_COUNT > 0,
        BLOCK_9_SENTENCE_COUNT_VALID,
        BLOCK_9_CANONICAL_SENTENCE_IDS_VALID,
        BLOCK_9_CANONICAL_SENTENCE_INDICES_VALID,
        BLOCK_9_CANONICAL_SEQUENCE_POSITIONS_VALID,
        BLOCK_9_ARTICLE_IDENTITIES_VALID,
    ]
)


if not BLOCK_9_CANONICAL_SEQUENCE_VALID:

    raise RuntimeError(
        "Notebook 09 Block 9 recurrent input records do not preserve "
        "the canonical Block 8 corpus identity/order contract."
    )


# =============================================================================
# Canonical article grouping
# =============================================================================

BLOCK_9_ARTICLE_RECORD_GROUPS = {
    article_id:
        [
            record

            for record
            in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS

            if record[
                "article_id"
            ]
            ==
            article_id
        ]

    for article_id
    in BLOCK_9_EXPECTED_ARTICLE_IDS
}


BLOCK_9_ARTICLE_RECORD_COUNTS = {
    article_id:
        len(
            records
        )

    for (
        article_id,
        records,
    ) in BLOCK_9_ARTICLE_RECORD_GROUPS.items()
}


BLOCK_9_ARTICLE_GROUPS_VALID = all(
    len(
        records
    )
    >
    0

    for records
    in BLOCK_9_ARTICLE_RECORD_GROUPS.values()
)


BLOCK_9_GROUPED_SENTENCE_COUNT_VALID = (
    sum(
        BLOCK_9_ARTICLE_RECORD_COUNTS.values()
    )
    ==
    BLOCK_9_EXPECTED_SENTENCE_COUNT
)


if not all(
    [
        BLOCK_9_ARTICLE_GROUPS_VALID,
        BLOCK_9_GROUPED_SENTENCE_COUNT_VALID,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 9 article grouping contract is invalid."
    )


# =============================================================================
# Canonical modules
# =============================================================================

BLOCK_9_TRANSFORMATIVE_MODULES = {
    pathway_name:
        BLOCK_1_TRANSFORMATIVE_MODULES[
            pathway_name
        ]

    for pathway_name
    in BLOCK_9_PATHWAY_DIMENSIONS
}


BLOCK_9_WEIGHT_MODULES = {
    pathway_name:
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
            pathway_name
        ]

    for pathway_name
    in BLOCK_9_PATHWAY_DIMENSIONS
}


BLOCK_9_REPRESENTATION_FIELDS = {
    pathway_name:
        f"{pathway_name}_representation"

    for pathway_name
    in BLOCK_9_PATHWAY_DIMENSIONS
}


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_9_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_9_nested_state_exact(
    state_before,
    state_after,
):

    if state_before.keys() != state_after.keys():

        return False


    for module_name in state_before:

        if (
            state_before[
                module_name
            ].keys()
            !=
            state_after[
                module_name
            ].keys()
        ):

            return False


        for tensor_name in state_before[
            module_name
        ]:

            if not torch.equal(
                state_before[
                    module_name
                ][
                    tensor_name
                ],
                state_after[
                    module_name
                ][
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot learned state before article trajectory execution
# =============================================================================

BLOCK_9_REPRESENTATION_STATE_BEFORE = (
    block_9_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_9_TRANSFORMATIVE_STATE_BEFORE = (
    block_9_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_9_WEIGHT_STATE_BEFORE = (
    block_9_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Input representation validation
# =============================================================================

BLOCK_9_INPUT_REPRESENTATION_SHAPES_VALID = {}

BLOCK_9_INPUT_REPRESENTATIONS_FINITE = {}


for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

    expected_shape = (
        1,
        BLOCK_9_PATHWAY_DIMENSIONS[
            pathway_name
        ],
    )


    representation_field = (
        BLOCK_9_REPRESENTATION_FIELDS[
            pathway_name
        ]
    )


    BLOCK_9_INPUT_REPRESENTATION_SHAPES_VALID[
        pathway_name
    ] = all(
        (
            representation_field
            in
            record
        )
        and
        (
            tuple(
                record[
                    representation_field
                ].shape
            )
            ==
            expected_shape
        )

        for record
        in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
    )


    BLOCK_9_INPUT_REPRESENTATIONS_FINITE[
        pathway_name
    ] = all(
        bool(
            torch.isfinite(
                record[
                    representation_field
                ]
            ).all().item()
        )

        for record
        in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
    )


BLOCK_9_ALL_INPUT_REPRESENTATION_SHAPES_VALID = all(
    BLOCK_9_INPUT_REPRESENTATION_SHAPES_VALID.values()
)


BLOCK_9_ALL_INPUT_REPRESENTATIONS_FINITE = all(
    BLOCK_9_INPUT_REPRESENTATIONS_FINITE.values()
)


if not all(
    [
        BLOCK_9_ALL_INPUT_REPRESENTATION_SHAPES_VALID,
        BLOCK_9_ALL_INPUT_REPRESENTATIONS_FINITE,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 9 article-level recurrent "
        "representation inputs are invalid."
    )


# =============================================================================
# Fresh article-state reset helper
# =============================================================================

def block_9_construct_fresh_article_states():

    return {
        pathway_name:
            torch.zeros(
                (
                    1,
                    pathway_dim,
                ),
                dtype=
                    DEFAULT_DTYPE,
                device=
                    DEVICE,
                requires_grad=
                    False,
            )

        for (
            pathway_name,
            pathway_dim,
        ) in BLOCK_9_PATHWAY_DIMENSIONS.items()
    }


# =============================================================================
# Validate Block 8 reset contract
# =============================================================================

BLOCK_9_BLOCK_8_RESET_CONTRACT_VALID = all(
    [
        (
            pathway_name
            in
            NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE
        )
        and
        torch.equal(
            NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE[
                pathway_name
            ],
            torch.zeros_like(
                NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE[
                    pathway_name
                ]
            ),
        )
        and
        (
            tuple(
                NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE[
                    pathway_name
                ].shape
            )
            ==
            (
                1,
                BLOCK_9_PATHWAY_DIMENSIONS[
                    pathway_name
                ],
            )
        )
        and
        (
            NOTEBOOK_09_ARTICLE_RECURRENT_RESET_STATE[
                pathway_name
            ].requires_grad
            is False
        )

        for pathway_name
        in BLOCK_9_PATHWAY_DIMENSIONS
    ]
)


if not BLOCK_9_BLOCK_8_RESET_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 09 Block 8 recurrent reset-state contract is invalid."
    )


# =============================================================================
# Controlled corpus trajectory execution helper
# =============================================================================

def block_9_execute_corpus_trajectories():
    """
    Execute the canonical corpus as independent article trajectories.

    Each article starts from a fresh zero recurrent state.
    State is propagated only within the current article.
    """

    corpus_trajectories = {}


    for article_id in BLOCK_9_EXPECTED_ARTICLE_IDS:

        article_records = (
            BLOCK_9_ARTICLE_RECORD_GROUPS[
                article_id
            ]
        )


        article_states = (
            block_9_construct_fresh_article_states()
        )


        article_trajectory = {
            pathway_name:
                {
                    "initial_state":
                        article_states[
                            pathway_name
                        ].detach()
                        .clone(),

                    "steps":
                        [],
                }

            for pathway_name
            in BLOCK_9_PATHWAY_DIMENSIONS
        }


        for (
            article_record_position,
            input_record,
        ) in enumerate(
            article_records
        ):

            for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

                representation_field = (
                    BLOCK_9_REPRESENTATION_FIELDS[
                        pathway_name
                    ]
                )


                current_representation = (
                    input_record[
                        representation_field
                    ].to(
                        device=
                            DEVICE,

                        dtype=
                            DEFAULT_DTYPE,
                    )
                )


                previous_state = (
                    article_states[
                        pathway_name
                    ]
                )


                candidate = (
                    BLOCK_9_TRANSFORMATIVE_MODULES[
                        pathway_name
                    ](
                        current_representation,
                        previous_state,
                    )
                )


                effective_weight = (
                    BLOCK_9_WEIGHT_MODULES[
                        pathway_name
                    ]()
                )


                weighted_candidate = (
                    effective_weight.unsqueeze(
                        0
                    )
                    *
                    candidate
                )


                updated_state = (
                    previous_state
                    +
                    weighted_candidate
                )


                article_trajectory[
                    pathway_name
                ][
                    "steps"
                ].append(
                    {
                        "article_id":
                            input_record[
                                "article_id"
                            ],

                        "sentence_id":
                            input_record[
                                "sentence_id"
                            ],

                        "sentence_index":
                            int(
                                input_record[
                                    "sentence_index"
                                ]
                            ),

                        "sequence_position":
                            int(
                                input_record[
                                    "sequence_position"
                                ]
                            ),

                        "article_record_position":
                            int(
                                article_record_position
                            ),

                        "current_representation":
                            current_representation.detach()
                            .clone(),

                        "previous_state":
                            previous_state.detach()
                            .clone(),

                        "candidate":
                            candidate.detach()
                            .clone(),

                        "effective_weight":
                            effective_weight.detach()
                            .clone(),

                        "weighted_candidate":
                            weighted_candidate.detach()
                            .clone(),

                        "updated_state":
                            updated_state.detach()
                            .clone(),
                    }
                )


                article_states[
                    pathway_name
                ] = (
                    updated_state.detach()
                    .clone()
                )


        for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

            article_trajectory[
                pathway_name
            ][
                "final_state"
            ] = (
                article_states[
                    pathway_name
                ].detach()
                .clone()
            )


        corpus_trajectories[
            article_id
        ] = article_trajectory


    return corpus_trajectories


# =============================================================================
# Execute complete corpus trajectory twice
# =============================================================================

with torch.no_grad():

    BLOCK_9_FIRST_TRAJECTORY = (
        block_9_execute_corpus_trajectories()
    )


    BLOCK_9_SECOND_TRAJECTORY = (
        block_9_execute_corpus_trajectories()
    )


# =============================================================================
# Execution flags
# =============================================================================

NOTEBOOK_09_BLOCK_9_REPRESENTATION_FORWARD_EXECUTED = False

NOTEBOOK_09_BLOCK_9_TRANSFORMATIVE_FORWARD_EXECUTED = True

NOTEBOOK_09_BLOCK_9_WEIGHT_FORWARD_EXECUTED = True

NOTEBOOK_09_BLOCK_9_WEIGHTED_CANDIDATE_EXECUTED = True

NOTEBOOK_09_BLOCK_9_RECURRENT_UPDATE_EXECUTED = True

NOTEBOOK_09_BLOCK_9_ARTICLE_RECURRENT_TRAJECTORY_EXECUTED = True

NOTEBOOK_09_BLOCK_9_ARTICLES_TRAVERSED = (
    BLOCK_9_EXPECTED_ARTICLE_COUNT
)

NOTEBOOK_09_BLOCK_9_ARTICLE_SENTENCES_TRAVERSED = (
    BLOCK_9_EXPECTED_SENTENCE_COUNT
)

NOTEBOOK_09_BLOCK_9_FUTURE_CONTEXT_USED = False

NOTEBOOK_09_BLOCK_9_CROSS_ARTICLE_STATE_PROPAGATION = False


# =============================================================================
# Article reset validation
# =============================================================================

BLOCK_9_INITIAL_STATES_ZERO = {}

BLOCK_9_INITIAL_STATE_RUNS_EQUAL = {}

BLOCK_9_INITIAL_STATE_STORAGE_INDEPENDENT = {}


for article_id in BLOCK_9_EXPECTED_ARTICLE_IDS:

    for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

        validation_key = (
            article_id,
            pathway_name,
        )


        first_initial = (
            BLOCK_9_FIRST_TRAJECTORY[
                article_id
            ][
                pathway_name
            ][
                "initial_state"
            ]
        )


        second_initial = (
            BLOCK_9_SECOND_TRAJECTORY[
                article_id
            ][
                pathway_name
            ][
                "initial_state"
            ]
        )


        BLOCK_9_INITIAL_STATES_ZERO[
            validation_key
        ] = all(
            [
                torch.equal(
                    first_initial,
                    torch.zeros_like(
                        first_initial
                    ),
                ),

                torch.equal(
                    second_initial,
                    torch.zeros_like(
                        second_initial
                    ),
                ),
            ]
        )


        BLOCK_9_INITIAL_STATE_RUNS_EQUAL[
            validation_key
        ] = torch.equal(
            first_initial,
            second_initial,
        )


        BLOCK_9_INITIAL_STATE_STORAGE_INDEPENDENT[
            validation_key
        ] = (
            first_initial.data_ptr()
            !=
            second_initial.data_ptr()
        )


BLOCK_9_ALL_INITIAL_STATES_ZERO = all(
    BLOCK_9_INITIAL_STATES_ZERO.values()
)


BLOCK_9_ALL_INITIAL_STATE_RUNS_EQUAL = all(
    BLOCK_9_INITIAL_STATE_RUNS_EQUAL.values()
)


BLOCK_9_ALL_INITIAL_STATE_STORAGE_INDEPENDENT = all(
    BLOCK_9_INITIAL_STATE_STORAGE_INDEPENDENT.values()
)


if not all(
    [
        BLOCK_9_ALL_INITIAL_STATES_ZERO,
        BLOCK_9_ALL_INITIAL_STATE_RUNS_EQUAL,
        BLOCK_9_ALL_INITIAL_STATE_STORAGE_INDEPENDENT,
    ]
):

    raise RuntimeError(
        "Article reset semantics are invalid."
    )


# =============================================================================
# Step-count and article/sentence identity validation
# =============================================================================

BLOCK_9_TRAJECTORY_STEP_COUNTS = {}

BLOCK_9_TRAJECTORY_IDENTITY_VALID = {}


for article_id in BLOCK_9_EXPECTED_ARTICLE_IDS:

    article_records = (
        BLOCK_9_ARTICLE_RECORD_GROUPS[
            article_id
        ]
    )


    expected_sentence_ids = [
        record[
            "sentence_id"
        ]

        for record
        in article_records
    ]


    expected_sentence_indices = [
        record[
            "sentence_index"
        ]

        for record
        in article_records
    ]


    expected_sequence_positions = [
        record[
            "sequence_position"
        ]

        for record
        in article_records
    ]


    for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

        validation_key = (
            article_id,
            pathway_name,
        )


        steps = (
            BLOCK_9_FIRST_TRAJECTORY[
                article_id
            ][
                pathway_name
            ][
                "steps"
            ]
        )


        BLOCK_9_TRAJECTORY_STEP_COUNTS[
            validation_key
        ] = len(
            steps
        )


        BLOCK_9_TRAJECTORY_IDENTITY_VALID[
            validation_key
        ] = all(
            [
                (
                    [
                        step[
                            "sentence_id"
                        ]

                        for step
                        in steps
                    ]
                    ==
                    expected_sentence_ids
                ),

                (
                    [
                        step[
                            "sentence_index"
                        ]

                        for step
                        in steps
                    ]
                    ==
                    expected_sentence_indices
                ),

                (
                    [
                        step[
                            "sequence_position"
                        ]

                        for step
                        in steps
                    ]
                    ==
                    expected_sequence_positions
                ),

                all(
                    step[
                        "article_id"
                    ]
                    ==
                    article_id

                    for step
                    in steps
                ),
            ]
        )


BLOCK_9_TRAJECTORY_STEP_COUNTS_VALID = all(
    BLOCK_9_TRAJECTORY_STEP_COUNTS[
        (
            article_id,
            pathway_name,
        )
    ]
    ==
    BLOCK_9_ARTICLE_RECORD_COUNTS[
        article_id
    ]

    for article_id
    in BLOCK_9_EXPECTED_ARTICLE_IDS

    for pathway_name
    in BLOCK_9_PATHWAY_DIMENSIONS
)


BLOCK_9_ALL_TRAJECTORY_IDENTITIES_VALID = all(
    BLOCK_9_TRAJECTORY_IDENTITY_VALID.values()
)


if not all(
    [
        BLOCK_9_TRAJECTORY_STEP_COUNTS_VALID,
        BLOCK_9_ALL_TRAJECTORY_IDENTITIES_VALID,
    ]
):

    raise RuntimeError(
        "Article trajectory step counts or identities are invalid."
    )


# =============================================================================
# Complete article/pathway trajectory validation
# =============================================================================

BLOCK_9_CANDIDATE_SHAPES_VALID = {}

BLOCK_9_CANDIDATES_FINITE = {}

BLOCK_9_WEIGHT_SHAPES_VALID = {}

BLOCK_9_WEIGHTS_FINITE = {}

BLOCK_9_WEIGHTS_BOUNDED = {}

BLOCK_9_WEIGHTS_CONSTANT_ACROSS_ARTICLE = {}

BLOCK_9_ACTIVE_GATING_VALID = {}

BLOCK_9_DEFERRED_GATING_VALID = {}

BLOCK_9_WEIGHTED_CANDIDATE_SHAPES_VALID = {}

BLOCK_9_WEIGHTED_CANDIDATES_FINITE = {}

BLOCK_9_STATE_SHAPES_VALID = {}

BLOCK_9_STATES_FINITE = {}

BLOCK_9_PREVIOUS_STATE_CHAIN_VALID = {}

BLOCK_9_STEPWISE_UPDATE_IDENTITY_VALID = {}

BLOCK_9_DEFERRED_STATES_ZERO = {}

BLOCK_9_CUMULATIVE_STATE_IDENTITY_VALID = {}

BLOCK_9_CUMULATIVE_STATE_MAX_ABS_DIFFERENCE = {}

BLOCK_9_FINAL_STATE_CUMULATIVE_VALID = {}

BLOCK_9_TRAJECTORY_DETERMINISTIC = {}


for article_id in BLOCK_9_EXPECTED_ARTICLE_IDS:

    for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

        validation_key = (
            article_id,
            pathway_name,
        )


        expected_shape = (
            1,
            BLOCK_9_PATHWAY_DIMENSIONS[
                pathway_name
            ],
        )


        expected_weight_shape = (
            BLOCK_9_PATHWAY_DIMENSIONS[
                pathway_name
            ],
        )


        first_trajectory = (
            BLOCK_9_FIRST_TRAJECTORY[
                article_id
            ][
                pathway_name
            ]
        )


        second_trajectory = (
            BLOCK_9_SECOND_TRAJECTORY[
                article_id
            ][
                pathway_name
            ]
        )


        steps = (
            first_trajectory[
                "steps"
            ]
        )


        active_indices = torch.tensor(
            BLOCK_9_ACTIVE_DIMENSION_INDICES[
                pathway_name
            ],
            dtype=
                torch.long,
        )


        deferred_indices = torch.tensor(
            BLOCK_9_DEFERRED_DIMENSION_INDICES[
                pathway_name
            ],
            dtype=
                torch.long,
        )


        BLOCK_9_CANDIDATE_SHAPES_VALID[
            validation_key
        ] = all(
            tuple(
                step[
                    "candidate"
                ].shape
            )
            ==
            expected_shape

            for step
            in steps
        )


        BLOCK_9_CANDIDATES_FINITE[
            validation_key
        ] = all(
            bool(
                torch.isfinite(
                    step[
                        "candidate"
                    ]
                ).all().item()
            )

            for step
            in steps
        )


        first_weight = (
            steps[
                0
            ][
                "effective_weight"
            ]
        )


        BLOCK_9_WEIGHT_SHAPES_VALID[
            validation_key
        ] = all(
            tuple(
                step[
                    "effective_weight"
                ].shape
            )
            ==
            expected_weight_shape

            for step
            in steps
        )


        BLOCK_9_WEIGHTS_FINITE[
            validation_key
        ] = all(
            bool(
                torch.isfinite(
                    step[
                        "effective_weight"
                    ]
                ).all().item()
            )

            for step
            in steps
        )


        BLOCK_9_WEIGHTS_BOUNDED[
            validation_key
        ] = all(
            bool(
                (
                    (
                        step[
                            "effective_weight"
                        ]
                        >=
                        0.0
                    )
                    &
                    (
                        step[
                            "effective_weight"
                        ]
                        <=
                        1.0
                    )
                ).all().item()
            )

            for step
            in steps
        )


        BLOCK_9_WEIGHTS_CONSTANT_ACROSS_ARTICLE[
            validation_key
        ] = all(
            torch.equal(
                step[
                    "effective_weight"
                ],
                first_weight,
            )

            for step
            in steps[
                1:
            ]
        )


        active_gating_valid = True

        deferred_gating_valid = True

        weighted_shape_valid = True

        weighted_finite_valid = True


        for step in steps:

            candidate = (
                step[
                    "candidate"
                ]
            )


            effective_weight = (
                step[
                    "effective_weight"
                ]
            )


            weighted_candidate = (
                step[
                    "weighted_candidate"
                ]
            )


            active_gating_valid = (
                active_gating_valid
                and
                torch.equal(
                    weighted_candidate[
                        :,
                        active_indices
                    ],
                    (
                        candidate[
                            :,
                            active_indices
                        ]
                        *
                        effective_weight[
                            active_indices
                        ].unsqueeze(
                            0
                        )
                    ),
                )
            )


            deferred_gating_valid = (
                deferred_gating_valid
                and
                torch.equal(
                    effective_weight[
                        deferred_indices
                    ],
                    torch.zeros_like(
                        effective_weight[
                            deferred_indices
                        ]
                    ),
                )
                and
                torch.equal(
                    weighted_candidate[
                        :,
                        deferred_indices
                    ],
                    torch.zeros_like(
                        weighted_candidate[
                            :,
                            deferred_indices
                        ]
                    ),
                )
            )


            weighted_shape_valid = (
                weighted_shape_valid
                and
                tuple(
                    weighted_candidate.shape
                )
                ==
                expected_shape
            )


            weighted_finite_valid = (
                weighted_finite_valid
                and
                bool(
                    torch.isfinite(
                        weighted_candidate
                    ).all().item()
                )
            )


        BLOCK_9_ACTIVE_GATING_VALID[
            validation_key
        ] = active_gating_valid


        BLOCK_9_DEFERRED_GATING_VALID[
            validation_key
        ] = deferred_gating_valid


        BLOCK_9_WEIGHTED_CANDIDATE_SHAPES_VALID[
            validation_key
        ] = weighted_shape_valid


        BLOCK_9_WEIGHTED_CANDIDATES_FINITE[
            validation_key
        ] = weighted_finite_valid


        states = [
            first_trajectory[
                "initial_state"
            ]
        ] + [
            step[
                "updated_state"
            ]

            for step
            in steps
        ]


        BLOCK_9_STATE_SHAPES_VALID[
            validation_key
        ] = all(
            tuple(
                state.shape
            )
            ==
            expected_shape

            for state
            in states
        )


        BLOCK_9_STATES_FINITE[
            validation_key
        ] = all(
            bool(
                torch.isfinite(
                    state
                ).all().item()
            )

            for state
            in states
        )


        chain_valid = torch.equal(
            steps[
                0
            ][
                "previous_state"
            ],
            first_trajectory[
                "initial_state"
            ],
        )


        for step_index in range(
            1,
            len(
                steps
            ),
        ):

            chain_valid = (
                chain_valid
                and
                torch.equal(
                    steps[
                        step_index
                    ][
                        "previous_state"
                    ],
                    steps[
                        step_index
                        -
                        1
                    ][
                        "updated_state"
                    ],
                )
            )


        BLOCK_9_PREVIOUS_STATE_CHAIN_VALID[
            validation_key
        ] = chain_valid


        BLOCK_9_STEPWISE_UPDATE_IDENTITY_VALID[
            validation_key
        ] = all(
            torch.equal(
                step[
                    "updated_state"
                ],
                (
                    step[
                        "previous_state"
                    ]
                    +
                    step[
                        "weighted_candidate"
                    ]
                ),
            )

            for step
            in steps
        )


        BLOCK_9_DEFERRED_STATES_ZERO[
            validation_key
        ] = all(
            torch.equal(
                state[
                    :,
                    deferred_indices
                ],
                torch.zeros_like(
                    state[
                        :,
                        deferred_indices
                    ]
                ),
            )

            for state
            in states
        )


        cumulative_state = torch.zeros_like(
            first_trajectory[
                "initial_state"
            ]
        )


        cumulative_valid = True

        maximum_difference = 0.0


        for step in steps:

            cumulative_state = (
                cumulative_state
                +
                step[
                    "weighted_candidate"
                ]
            )


            difference = (
                step[
                    "updated_state"
                ]
                -
                cumulative_state
            )


            maximum_difference = max(
                maximum_difference,
                float(
                    difference.abs()
                    .max()
                    .item()
                ),
            )


            cumulative_valid = (
                cumulative_valid
                and
                torch.allclose(
                    step[
                        "updated_state"
                    ],
                    cumulative_state,
                    rtol=
                        0.0,
                    atol=
                        1e-7,
                )
            )


        BLOCK_9_CUMULATIVE_STATE_IDENTITY_VALID[
            validation_key
        ] = cumulative_valid


        BLOCK_9_CUMULATIVE_STATE_MAX_ABS_DIFFERENCE[
            validation_key
        ] = maximum_difference


        BLOCK_9_FINAL_STATE_CUMULATIVE_VALID[
            validation_key
        ] = torch.allclose(
            first_trajectory[
                "final_state"
            ],
            cumulative_state,
            rtol=
                0.0,
            atol=
                1e-7,
        )


        deterministic = torch.equal(
            first_trajectory[
                "initial_state"
            ],
            second_trajectory[
                "initial_state"
            ],
        )


        for (
            first_step,
            second_step,
        ) in zip(
            first_trajectory[
                "steps"
            ],
            second_trajectory[
                "steps"
            ],
        ):

            deterministic = (
                deterministic
                and
                (
                    first_step[
                        "article_id"
                    ]
                    ==
                    second_step[
                        "article_id"
                    ]
                )
                and
                (
                    first_step[
                        "sentence_id"
                    ]
                    ==
                    second_step[
                        "sentence_id"
                    ]
                )
                and
                (
                    first_step[
                        "sequence_position"
                    ]
                    ==
                    second_step[
                        "sequence_position"
                    ]
                )
            )


            for tensor_name in (
                "current_representation",
                "previous_state",
                "candidate",
                "effective_weight",
                "weighted_candidate",
                "updated_state",
            ):

                deterministic = (
                    deterministic
                    and
                    torch.equal(
                        first_step[
                            tensor_name
                        ],
                        second_step[
                            tensor_name
                        ],
                    )
                )


        deterministic = (
            deterministic
            and
            torch.equal(
                first_trajectory[
                    "final_state"
                ],
                second_trajectory[
                    "final_state"
                ],
            )
        )


        BLOCK_9_TRAJECTORY_DETERMINISTIC[
            validation_key
        ] = deterministic


BLOCK_9_ALL_CANDIDATE_SHAPES_VALID = all(
    BLOCK_9_CANDIDATE_SHAPES_VALID.values()
)


BLOCK_9_ALL_CANDIDATES_FINITE = all(
    BLOCK_9_CANDIDATES_FINITE.values()
)


BLOCK_9_ALL_WEIGHT_SHAPES_VALID = all(
    BLOCK_9_WEIGHT_SHAPES_VALID.values()
)


BLOCK_9_ALL_WEIGHTS_FINITE = all(
    BLOCK_9_WEIGHTS_FINITE.values()
)


BLOCK_9_ALL_WEIGHTS_BOUNDED = all(
    BLOCK_9_WEIGHTS_BOUNDED.values()
)


BLOCK_9_ALL_WEIGHTS_CONSTANT_ACROSS_ARTICLE = all(
    BLOCK_9_WEIGHTS_CONSTANT_ACROSS_ARTICLE.values()
)


BLOCK_9_ALL_ACTIVE_GATING_VALID = all(
    BLOCK_9_ACTIVE_GATING_VALID.values()
)


BLOCK_9_ALL_DEFERRED_GATING_VALID = all(
    BLOCK_9_DEFERRED_GATING_VALID.values()
)


BLOCK_9_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID = all(
    BLOCK_9_WEIGHTED_CANDIDATE_SHAPES_VALID.values()
)


BLOCK_9_ALL_WEIGHTED_CANDIDATES_FINITE = all(
    BLOCK_9_WEIGHTED_CANDIDATES_FINITE.values()
)


BLOCK_9_ALL_STATE_SHAPES_VALID = all(
    BLOCK_9_STATE_SHAPES_VALID.values()
)


BLOCK_9_ALL_STATES_FINITE = all(
    BLOCK_9_STATES_FINITE.values()
)


BLOCK_9_ALL_PREVIOUS_STATE_CHAINS_VALID = all(
    BLOCK_9_PREVIOUS_STATE_CHAIN_VALID.values()
)


BLOCK_9_ALL_STEPWISE_UPDATE_IDENTITIES_VALID = all(
    BLOCK_9_STEPWISE_UPDATE_IDENTITY_VALID.values()
)


BLOCK_9_ALL_DEFERRED_STATES_ZERO = all(
    BLOCK_9_DEFERRED_STATES_ZERO.values()
)


BLOCK_9_ALL_CUMULATIVE_STATE_IDENTITIES_VALID = all(
    BLOCK_9_CUMULATIVE_STATE_IDENTITY_VALID.values()
)


BLOCK_9_ALL_FINAL_STATE_CUMULATIVE_VALID = all(
    BLOCK_9_FINAL_STATE_CUMULATIVE_VALID.values()
)


BLOCK_9_ALL_TRAJECTORIES_DETERMINISTIC = all(
    BLOCK_9_TRAJECTORY_DETERMINISTIC.values()
)


# =============================================================================
# Cross-article reset boundary validation
# =============================================================================

BLOCK_9_CROSS_ARTICLE_RESET_VALID = True


for article_index in range(
    1,
    len(
        BLOCK_9_EXPECTED_ARTICLE_IDS
    ),
):

    article_id = (
        BLOCK_9_EXPECTED_ARTICLE_IDS[
            article_index
        ]
    )


    previous_article_id = (
        BLOCK_9_EXPECTED_ARTICLE_IDS[
            article_index
            -
            1
        ]
    )


    for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

        current_initial = (
            BLOCK_9_FIRST_TRAJECTORY[
                article_id
            ][
                pathway_name
            ][
                "initial_state"
            ]
        )


        previous_final = (
            BLOCK_9_FIRST_TRAJECTORY[
                previous_article_id
            ][
                pathway_name
            ][
                "final_state"
            ]
        )


        BLOCK_9_CROSS_ARTICLE_RESET_VALID = all(
            [
                BLOCK_9_CROSS_ARTICLE_RESET_VALID,

                torch.equal(
                    current_initial,
                    torch.zeros_like(
                        current_initial
                    ),
                ),

                (
                    current_initial.data_ptr()
                    !=
                    previous_final.data_ptr()
                ),
            ]
        )


if not BLOCK_9_CROSS_ARTICLE_RESET_VALID:

    raise RuntimeError(
        "Cross-article recurrent reset contract is invalid."
    )


# =============================================================================
# Pathway independence
# =============================================================================

BLOCK_9_TRANSFORMATIVE_MODULE_IDS = {
    pathway_name:
        id(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_9_TRANSFORMATIVE_MODULES.items()
}


BLOCK_9_WEIGHT_MODULE_IDS = {
    pathway_name:
        id(
            module
        )

    for (
        pathway_name,
        module,
    ) in BLOCK_9_WEIGHT_MODULES.items()
}


BLOCK_9_PATHWAY_COUNT = len(
    BLOCK_9_PATHWAY_DIMENSIONS
)


BLOCK_9_PATHWAYS_INDEPENDENT = all(
    [
        (
            len(
                set(
                    BLOCK_9_TRANSFORMATIVE_MODULE_IDS.values()
                )
            )
            ==
            BLOCK_9_PATHWAY_COUNT
        ),

        (
            len(
                set(
                    BLOCK_9_WEIGHT_MODULE_IDS.values()
                )
            )
            ==
            BLOCK_9_PATHWAY_COUNT
        ),
    ]
)


if not BLOCK_9_PATHWAYS_INDEPENDENT:

    raise RuntimeError(
        "Article-level recurrent pathways are not structurally independent."
    )


# =============================================================================
# Final recurrent states by article and pathway
# =============================================================================

NOTEBOOK_09_ARTICLE_FINAL_RECURRENT_STATES = {
    article_id:
        {
            pathway_name:
                BLOCK_9_FIRST_TRAJECTORY[
                    article_id
                ][
                    pathway_name
                ][
                    "final_state"
                ]
                .detach()
                .cpu()
                .clone()

            for pathway_name
            in BLOCK_9_PATHWAY_DIMENSIONS
        }

    for article_id
    in BLOCK_9_EXPECTED_ARTICLE_IDS
}


BLOCK_9_FINAL_STATE_SHAPES = {
    (
        article_id,
        pathway_name,
    ):
        tuple(
            state.shape
        )

    for (
        article_id,
        article_states,
    ) in NOTEBOOK_09_ARTICLE_FINAL_RECURRENT_STATES.items()

    for (
        pathway_name,
        state,
    ) in article_states.items()
}


BLOCK_9_FINAL_STATES_FINITE = {
    (
        article_id,
        pathway_name,
    ):
        bool(
            torch.isfinite(
                state
            ).all().item()
        )

    for (
        article_id,
        article_states,
    ) in NOTEBOOK_09_ARTICLE_FINAL_RECURRENT_STATES.items()

    for (
        pathway_name,
        state,
    ) in article_states.items()
}


BLOCK_9_ALL_FINAL_STATES_FINITE = all(
    BLOCK_9_FINAL_STATES_FINITE.values()
)


# =============================================================================
# Gradient-state validation
# =============================================================================

BLOCK_9_REPRESENTATION_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_9_TRANSFORMATIVE_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_9_WEIGHT_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_9_ALL_GRADIENTS_ABSENT = all(
    [
        BLOCK_9_REPRESENTATION_GRADIENTS_ABSENT,
        BLOCK_9_TRANSFORMATIVE_GRADIENTS_ABSENT,
        BLOCK_9_WEIGHT_GRADIENTS_ABSENT,
    ]
)


if not BLOCK_9_ALL_GRADIENTS_ABSENT:

    raise RuntimeError(
        "Unexpected gradients were created during article-level "
        "recurrent trajectory validation."
    )


# =============================================================================
# Snapshot learned state after article trajectory execution
# =============================================================================

BLOCK_9_REPRESENTATION_STATE_AFTER = (
    block_9_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_9_TRANSFORMATIVE_STATE_AFTER = (
    block_9_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_9_WEIGHT_STATE_AFTER = (
    block_9_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Exact learned-state immutability
# =============================================================================

BLOCK_9_REPRESENTATION_UNCHANGED = (
    block_9_nested_state_exact(
        BLOCK_9_REPRESENTATION_STATE_BEFORE,
        BLOCK_9_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_9_TRANSFORMATIVE_UNCHANGED = (
    block_9_nested_state_exact(
        BLOCK_9_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_9_TRANSFORMATIVE_STATE_AFTER,
    )
)


BLOCK_9_WEIGHT_STATE_UNCHANGED = (
    block_9_nested_state_exact(
        BLOCK_9_WEIGHT_STATE_BEFORE,
        BLOCK_9_WEIGHT_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_9_REPRESENTATION_UNCHANGED,
        BLOCK_9_TRANSFORMATIVE_UNCHANGED,
        BLOCK_9_WEIGHT_STATE_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Model parameter state changed during article-level "
        "recurrent trajectory validation."
    )


# =============================================================================
# Trainability and evaluation-mode validation
# =============================================================================

BLOCK_9_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_9_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_9_WEIGHT_PARAMETERS_STILL_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_9_INHERITED_MODULES_STILL_EVAL = all(
    not module.training

    for module
    in (
        list(
            BLOCK_1_REPRESENTATION_MODULES.values()
        )
        +
        list(
            BLOCK_1_TRANSFORMATIVE_MODULES.values()
        )
    )
)


if not all(
    [
        BLOCK_9_REPRESENTATION_STILL_FROZEN,
        BLOCK_9_TRANSFORMATIVE_STILL_FROZEN,
        BLOCK_9_WEIGHT_PARAMETERS_STILL_TRAINABLE,
        BLOCK_9_INHERITED_MODULES_STILL_EVAL,
    ]
):

    raise RuntimeError(
        "Notebook 09 trainability or inherited evaluation-mode scope "
        "changed during Block 9."
    )


# =============================================================================
# Dynamic parameter accounting
# =============================================================================

BLOCK_9_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_9_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 parameter accounting changed during "
        "article-level recurrent execution."
    )


# =============================================================================
# Causal execution contract
# =============================================================================

BLOCK_9_CAUSAL_CONTRACT_VALID = all(
    [
        NOTEBOOK_09_BLOCK_9_FUTURE_CONTEXT_USED is False,
        NOTEBOOK_09_BLOCK_9_CROSS_ARTICLE_STATE_PROPAGATION is False,
        BLOCK_9_ALL_PREVIOUS_STATE_CHAINS_VALID,
        BLOCK_9_ALL_TRAJECTORY_IDENTITIES_VALID,
        BLOCK_9_CROSS_ARTICLE_RESET_VALID,
    ]
)


if not BLOCK_9_CAUSAL_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 09 Block 9 violated the article-level "
        "causal execution contract."
    )


# =============================================================================
# Explicit optimisation boundary
# =============================================================================

NOTEBOOK_09_BLOCK_9_LOSS_CALCULATED = False

NOTEBOOK_09_BLOCK_9_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_9_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_9_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_09_BLOCK_9_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_9_TRAINING_EXECUTED = False


# =============================================================================
# Expose canonical article trajectories
# =============================================================================

NOTEBOOK_09_ARTICLE_RECURRENT_TRAJECTORY = {
    article_id:
        {
            pathway_name:
                {
                    "initial_state":
                        BLOCK_9_FIRST_TRAJECTORY[
                            article_id
                        ][
                            pathway_name
                        ][
                            "initial_state"
                        ]
                        .detach()
                        .cpu()
                        .clone(),

                    "steps":
                        [
                            {
                                key:
                                    (
                                        value.detach()
                                        .cpu()
                                        .clone()

                                        if torch.is_tensor(
                                            value
                                        )

                                        else value
                                    )

                                for (
                                    key,
                                    value,
                                ) in step.items()
                            }

                            for step
                            in BLOCK_9_FIRST_TRAJECTORY[
                                article_id
                            ][
                                pathway_name
                            ][
                                "steps"
                            ]
                        ],

                    "final_state":
                        BLOCK_9_FIRST_TRAJECTORY[
                            article_id
                        ][
                            pathway_name
                        ][
                            "final_state"
                        ]
                        .detach()
                        .cpu()
                        .clone(),
                }

            for pathway_name
            in BLOCK_9_PATHWAY_DIMENSIONS
        }

    for article_id
    in BLOCK_9_EXPECTED_ARTICLE_IDS
}


# =============================================================================
# Final Block 9 validation
# =============================================================================

NOTEBOOK_09_BLOCK_9_ERRORS = []


BLOCK_9_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_9_PREREQUISITES_VALID,

    "pathway_dimensions_invalid":
        BLOCK_9_PATHWAY_DIMENSIONS_VALID,

    "canonical_sequence_invalid":
        BLOCK_9_CANONICAL_SEQUENCE_VALID,

    "article_groups_invalid":
        BLOCK_9_ARTICLE_GROUPS_VALID,

    "dimension_partitions_invalid":
        BLOCK_9_ALL_DIMENSION_PARTITIONS_VALID,

    "input_representation_shapes_invalid":
        BLOCK_9_ALL_INPUT_REPRESENTATION_SHAPES_VALID,

    "input_representations_not_finite":
        BLOCK_9_ALL_INPUT_REPRESENTATIONS_FINITE,

    "block_8_reset_contract_invalid":
        BLOCK_9_BLOCK_8_RESET_CONTRACT_VALID,

    "initial_states_not_zero":
        BLOCK_9_ALL_INITIAL_STATES_ZERO,

    "initial_state_runs_not_equal":
        BLOCK_9_ALL_INITIAL_STATE_RUNS_EQUAL,

    "initial_state_storage_not_independent":
        BLOCK_9_ALL_INITIAL_STATE_STORAGE_INDEPENDENT,

    "trajectory_step_counts_invalid":
        BLOCK_9_TRAJECTORY_STEP_COUNTS_VALID,

    "trajectory_identity_invalid":
        BLOCK_9_ALL_TRAJECTORY_IDENTITIES_VALID,

    "candidate_shapes_invalid":
        BLOCK_9_ALL_CANDIDATE_SHAPES_VALID,

    "candidates_not_finite":
        BLOCK_9_ALL_CANDIDATES_FINITE,

    "weight_shapes_invalid":
        BLOCK_9_ALL_WEIGHT_SHAPES_VALID,

    "weights_not_finite":
        BLOCK_9_ALL_WEIGHTS_FINITE,

    "weights_not_bounded":
        BLOCK_9_ALL_WEIGHTS_BOUNDED,

    "weights_not_constant":
        BLOCK_9_ALL_WEIGHTS_CONSTANT_ACROSS_ARTICLE,

    "active_gating_invalid":
        BLOCK_9_ALL_ACTIVE_GATING_VALID,

    "deferred_gating_invalid":
        BLOCK_9_ALL_DEFERRED_GATING_VALID,

    "weighted_candidate_shapes_invalid":
        BLOCK_9_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID,

    "weighted_candidates_not_finite":
        BLOCK_9_ALL_WEIGHTED_CANDIDATES_FINITE,

    "state_shapes_invalid":
        BLOCK_9_ALL_STATE_SHAPES_VALID,

    "states_not_finite":
        BLOCK_9_ALL_STATES_FINITE,

    "previous_state_chain_invalid":
        BLOCK_9_ALL_PREVIOUS_STATE_CHAINS_VALID,

    "stepwise_update_identity_invalid":
        BLOCK_9_ALL_STEPWISE_UPDATE_IDENTITIES_VALID,

    "deferred_states_nonzero":
        BLOCK_9_ALL_DEFERRED_STATES_ZERO,

    "cumulative_state_identity_invalid":
        BLOCK_9_ALL_CUMULATIVE_STATE_IDENTITIES_VALID,

    "final_state_cumulative_invalid":
        BLOCK_9_ALL_FINAL_STATE_CUMULATIVE_VALID,

    "trajectory_not_deterministic":
        BLOCK_9_ALL_TRAJECTORIES_DETERMINISTIC,

    "cross_article_reset_invalid":
        BLOCK_9_CROSS_ARTICLE_RESET_VALID,

    "pathways_not_independent":
        BLOCK_9_PATHWAYS_INDEPENDENT,

    "final_states_not_finite":
        BLOCK_9_ALL_FINAL_STATES_FINITE,

    "unexpected_gradients":
        BLOCK_9_ALL_GRADIENTS_ABSENT,

    "representation_state_changed":
        BLOCK_9_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_9_TRANSFORMATIVE_UNCHANGED,

    "weight_state_changed":
        BLOCK_9_WEIGHT_STATE_UNCHANGED,

    "representation_not_frozen":
        BLOCK_9_REPRESENTATION_STILL_FROZEN,

    "transformative_not_frozen":
        BLOCK_9_TRANSFORMATIVE_STILL_FROZEN,

    "weight_parameters_not_trainable":
        BLOCK_9_WEIGHT_PARAMETERS_STILL_TRAINABLE,

    "inherited_modules_not_eval":
        BLOCK_9_INHERITED_MODULES_STILL_EVAL,

    "parameter_accounting_invalid":
        BLOCK_9_PARAMETER_ACCOUNTING_VALID,

    "causal_contract_invalid":
        BLOCK_9_CAUSAL_CONTRACT_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_9_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_9_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation audit
# =============================================================================

BLOCK_9_PROHIBITED_OPERATIONS = {
    "representation_forward_incorrectly_executed":
        NOTEBOOK_09_BLOCK_9_REPRESENTATION_FORWARD_EXECUTED,

    "future_context_incorrectly_used":
        NOTEBOOK_09_BLOCK_9_FUTURE_CONTEXT_USED,

    "cross_article_state_incorrectly_propagated":
        NOTEBOOK_09_BLOCK_9_CROSS_ARTICLE_STATE_PROPAGATION,

    "loss_incorrectly_calculated":
        NOTEBOOK_09_BLOCK_9_LOSS_CALCULATED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_9_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_9_BACKWARD_PASS_EXECUTED,

    "gradient_clipping_incorrectly_executed":
        NOTEBOOK_09_BLOCK_9_GRADIENT_CLIPPING_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_9_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_9_TRAINING_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_9_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_9_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 9 state
# =============================================================================

NOTEBOOK_09_ARTICLE_RECURRENT_TRAJECTORY_VALID = (
    len(
        NOTEBOOK_09_BLOCK_9_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_OBJECTIVE_READY = (
    NOTEBOOK_09_ARTICLE_RECURRENT_TRAJECTORY_VALID
)


NOTEBOOK_09_BLOCK_9_VALID = (
    NOTEBOOK_09_ARTICLE_RECURRENT_TRAJECTORY_VALID
)


if not NOTEBOOK_09_BLOCK_9_VALID:

    raise RuntimeError(
        "Notebook 09 Block 9 controlled article-level recurrent "
        "trajectory validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_9_ERRORS}"
    )


NOTEBOOK_09_BLOCK_9_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_9_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_9,

    "block_name":
        NOTEBOOK_09_BLOCK_9_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_9_VERSION,

    "article_ids":
        tuple(
            BLOCK_9_EXPECTED_ARTICLE_IDS
        ),

    "article_count":
        BLOCK_9_EXPECTED_ARTICLE_COUNT,

    "sentence_count":
        BLOCK_9_EXPECTED_SENTENCE_COUNT,

    "article_record_counts":
        deepcopy(
            BLOCK_9_ARTICLE_RECORD_COUNTS
        ),

    "pathway_dimensions":
        deepcopy(
            BLOCK_9_PATHWAY_DIMENSIONS
        ),

    "candidate_shapes_valid":
        BLOCK_9_ALL_CANDIDATE_SHAPES_VALID,

    "weighted_candidate_shapes_valid":
        BLOCK_9_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID,

    "state_shapes_valid":
        BLOCK_9_ALL_STATE_SHAPES_VALID,

    "states_finite":
        BLOCK_9_ALL_STATES_FINITE,

    "active_gating_valid":
        BLOCK_9_ALL_ACTIVE_GATING_VALID,

    "deferred_gating_valid":
        BLOCK_9_ALL_DEFERRED_GATING_VALID,

    "deferred_states_zero":
        BLOCK_9_ALL_DEFERRED_STATES_ZERO,

    "previous_state_chain_valid":
        BLOCK_9_ALL_PREVIOUS_STATE_CHAINS_VALID,

    "stepwise_update_identity_valid":
        BLOCK_9_ALL_STEPWISE_UPDATE_IDENTITIES_VALID,

    "cumulative_state_identity_valid":
        BLOCK_9_ALL_CUMULATIVE_STATE_IDENTITIES_VALID,

    "trajectory_deterministic":
        BLOCK_9_ALL_TRAJECTORIES_DETERMINISTIC,

    "cross_article_reset_valid":
        BLOCK_9_CROSS_ARTICLE_RESET_VALID,

    "pathways_independent":
        BLOCK_9_PATHWAYS_INDEPENDENT,

    "representation_parameters":
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT,

    "transformative_parameters":
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT,

    "transformative_weight_parameters":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT,

    "total_model_parameters":
        NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION,

    "representation_forward_executed":
        NOTEBOOK_09_BLOCK_9_REPRESENTATION_FORWARD_EXECUTED,

    "transformative_forward_executed":
        NOTEBOOK_09_BLOCK_9_TRANSFORMATIVE_FORWARD_EXECUTED,

    "weight_forward_executed":
        NOTEBOOK_09_BLOCK_9_WEIGHT_FORWARD_EXECUTED,

    "recurrent_update_executed":
        NOTEBOOK_09_BLOCK_9_RECURRENT_UPDATE_EXECUTED,

    "article_trajectory_executed":
        NOTEBOOK_09_BLOCK_9_ARTICLE_RECURRENT_TRAJECTORY_EXECUTED,

    "future_context_used":
        NOTEBOOK_09_BLOCK_9_FUTURE_CONTEXT_USED,

    "trajectory_valid":
        NOTEBOOK_09_ARTICLE_RECURRENT_TRAJECTORY_VALID,

    "transformative_weight_objective_ready":
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_OBJECTIVE_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_9_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_9_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 9: "
    "Controlled Article-Level Recurrent Trajectory Execution "
    "and Validation"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_9_VERSION}"
)

print("-" * 72)

print(
    "Canonical corpus traversal"
)

print(
    f"Articles traversed           : "
    f"{NOTEBOOK_09_BLOCK_9_ARTICLES_TRAVERSED}"
)

print(
    f"Sentences traversed          : "
    f"{NOTEBOOK_09_BLOCK_9_ARTICLE_SENTENCES_TRAVERSED}"
)

print(
    f"Article IDs valid            : "
    f"{BLOCK_9_ARTICLE_IDENTITIES_VALID}"
)

print(
    f"Sentence identities valid    : "
    f"{BLOCK_9_ALL_TRAJECTORY_IDENTITIES_VALID}"
)

print(
    f"Canonical order preserved    : "
    f"{BLOCK_9_CANONICAL_SEQUENCE_VALID}"
)

print(
    f"Fresh zero initial states    : "
    f"{BLOCK_9_ALL_INITIAL_STATES_ZERO}"
)

print(
    f"Independent reset storage    : "
    f"{BLOCK_9_ALL_INITIAL_STATE_STORAGE_INDEPENDENT}"
)

print(
    f"Cross-article reset valid    : "
    f"{BLOCK_9_CROSS_ARTICLE_RESET_VALID}"
)

print("-" * 72)

print(
    "Article/pathway recurrent validation"
)

for article_id in BLOCK_9_EXPECTED_ARTICLE_IDS:

    print(
        f"Article                      : {article_id}"
    )

    for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

        validation_key = (
            article_id,
            pathway_name,
        )

        print(
            f"{pathway_name:<14} candidate shapes   : "
            f"{BLOCK_9_CANDIDATE_SHAPES_VALID[validation_key]}"
        )

        print(
            f"{pathway_name:<14} candidate finite   : "
            f"{BLOCK_9_CANDIDATES_FINITE[validation_key]}"
        )

        print(
            f"{pathway_name:<14} active gating      : "
            f"{BLOCK_9_ACTIVE_GATING_VALID[validation_key]}"
        )

        print(
            f"{pathway_name:<14} deferred gating    : "
            f"{BLOCK_9_DEFERRED_GATING_VALID[validation_key]}"
        )

        print(
            f"{pathway_name:<14} state chain valid  : "
            f"{BLOCK_9_PREVIOUS_STATE_CHAIN_VALID[validation_key]}"
        )

        print(
            f"{pathway_name:<14} additive update    : "
            f"{BLOCK_9_STEPWISE_UPDATE_IDENTITY_VALID[validation_key]}"
        )

        print(
            f"{pathway_name:<14} cumulative identity: "
            f"{BLOCK_9_CUMULATIVE_STATE_IDENTITY_VALID[validation_key]}"
        )

        print(
            f"{pathway_name:<14} deterministic      : "
            f"{BLOCK_9_TRAJECTORY_DETERMINISTIC[validation_key]}"
        )


print("-" * 72)

print(
    "Final recurrent states"
)

for article_id in BLOCK_9_EXPECTED_ARTICLE_IDS:

    print(
        f"Article                      : {article_id}"
    )

    for pathway_name in BLOCK_9_PATHWAY_DIMENSIONS:

        validation_key = (
            article_id,
            pathway_name,
        )

        print(
            f"{pathway_name:<14} final shape        : "
            f"{BLOCK_9_FINAL_STATE_SHAPES[validation_key]}"
        )

        print(
            f"{pathway_name:<14} final finite       : "
            f"{BLOCK_9_FINAL_STATES_FINITE[validation_key]}"
        )

        print(
            f"{pathway_name:<14} cumulative valid   : "
            f"{BLOCK_9_FINAL_STATE_CUMULATIVE_VALID[validation_key]}"
        )


print("-" * 72)

print(
    "Causal and pathway validation"
)

print(
    f"Pathways independent         : "
    f"{BLOCK_9_PATHWAYS_INDEPENDENT}"
)

print(
    f"Future context used          : "
    f"{NOTEBOOK_09_BLOCK_9_FUTURE_CONTEXT_USED}"
)

print(
    f"Cross-article propagation    : "
    f"{NOTEBOOK_09_BLOCK_9_CROSS_ARTICLE_STATE_PROPAGATION}"
)

print(
    f"Causal contract valid        : "
    f"{BLOCK_9_CAUSAL_CONTRACT_VALID}"
)

print("-" * 72)

print(
    "Parameter and state validation"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Transformative Weight params : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Total model parameters       : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print(
    f"Representation unchanged     : "
    f"{BLOCK_9_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged     : "
    f"{BLOCK_9_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Weight state unchanged       : "
    f"{BLOCK_9_WEIGHT_STATE_UNCHANGED}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_9_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_9_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"Weight parameters trainable  : "
    f"{BLOCK_9_WEIGHT_PARAMETERS_STILL_TRAINABLE}"
)

print(
    f"All gradients absent         : "
    f"{BLOCK_9_ALL_GRADIENTS_ABSENT}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Representation forward       : "
    f"{NOTEBOOK_09_BLOCK_9_REPRESENTATION_FORWARD_EXECUTED}"
)

print(
    f"Transform forward            : "
    f"{NOTEBOOK_09_BLOCK_9_TRANSFORMATIVE_FORWARD_EXECUTED}"
)

print(
    f"Weight forward               : "
    f"{NOTEBOOK_09_BLOCK_9_WEIGHT_FORWARD_EXECUTED}"
)

print(
    f"Weighted candidate executed  : "
    f"{NOTEBOOK_09_BLOCK_9_WEIGHTED_CANDIDATE_EXECUTED}"
)

print(
    f"Recurrent update executed    : "
    f"{NOTEBOOK_09_BLOCK_9_RECURRENT_UPDATE_EXECUTED}"
)

print(
    f"Article trajectories executed: "
    f"{NOTEBOOK_09_BLOCK_9_ARTICLE_RECURRENT_TRAJECTORY_EXECUTED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_BLOCK_9_LOSS_CALCULATED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_9_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_9_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_9_PARAMETER_UPDATE_EXECUTED}"
)

print("-" * 72)

print(
    f"Trajectory deterministic     : "
    f"{BLOCK_9_ALL_TRAJECTORIES_DETERMINISTIC}"
)

print(
    f"Article trajectories valid   : "
    f"{NOTEBOOK_09_ARTICLE_RECURRENT_TRAJECTORY_VALID}"
)

print(
    f"TW objective ready           : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_OBJECTIVE_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_9_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_9_COMPLETE}"
)

print("=" * 72)

print(
    f"The complete canonical corpus of {BLOCK_9_EXPECTED_SENTENCE_COUNT} "
    f"sentence(s) across {BLOCK_9_EXPECTED_ARTICLE_COUNT} article(s) was "
    "traversed successfully through all inherited recurrent pathways."
)

print(
    "Every article began from a fresh deterministic zero recurrent state, "
    "and no recurrent state was propagated across article boundaries."
)

print(
    "Each sentence representation was combined with the immediately "
    "preceding pathway-specific state and passed through the frozen "
    "transformative mechanism."
)

print(
    "Transformative Weights were applied dimension-wise before the additive "
    "recurrent state update; deferred dimensions remained zero."
)

print(
    "Canonical article and sentence identity, within-article ordering, "
    "causal state continuity and numerical finiteness were preserved."
)

print(
    "Repeated complete corpus executions were exactly deterministic."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} representation "
    "and transformative parameters remain frozen and exactly unchanged, while "
    f"all {NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT} "
    "Transformative-Weight parameter(s) remain trainable and unchanged."
)

print(
    "No representation-model re-execution, future-context access, "
    "cross-article state propagation, loss, optimiser, backward pass or "
    "parameter update occurred."
)

print(
    "Notebook 09 may now proceed to Transformative-Weight objective "
    "construction and controlled learning policy."
)

print("=" * 72)

Media AI — Notebook 09, Block 9: Controlled Article-Level Recurrent Trajectory Execution and Validation
Block version                : 1.1
------------------------------------------------------------------------
Canonical corpus traversal
Articles traversed           : 3
Sentences traversed          : 54
Article IDs valid            : True
Sentence identities valid    : True
Canonical order preserved    : True
Fresh zero initial states    : True
Independent reset storage    : True
Cross-article reset valid    : True
------------------------------------------------------------------------
Article/pathway recurrent validation
Article                      : 1ef9745c-2c21-47c2-a222-e7c160448f82
factual        candidate shapes   : True
factual        candidate finite   : True
factual        active gating      : True
factual        deferred gating    : True
factual        state chain valid  : True
factual        additive update    : True
factual        cumulative identity: True
factual      

## Block 10 — Transformative-Weight Objective Construction, Supervision Alignment and Optimisation Contract

This block defines the learning objective for the factual, psychological, and social Transformative Weights without yet performing optimisation.

Block 9 established the first complete recurrent execution over the canonical 16-sentence article sequence. For every pathway and sentence, the recurrent trajectory now preserves the current representation, preceding recurrent state, candidate transformation, effective Transformative Weight, weighted candidate contribution, and updated recurrent state.

The present block determines how the 25 activation-eligible Transformative-Weight parameters may be learned from the validated supervision while preserving their architectural interpretation as dimension-wise gates on candidate transformations.

The purpose is therefore to establish the complete supervision, masking, target-alignment, loss, and optimisation contract before any gradient-based parameter update is permitted.

No training is performed in this block.

No optimiser is created, no backward pass is executed, and no Transformative-Weight parameter is updated.

### Architectural learning target

For pathway

$$
k\in\{F,P,S\},
$$

the recurrent update established in Block 9 is

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
TW_k
\odot
c_t^{(k)},
$$

where

$$
c_t^{(k)}
=
T_k
\left(
\left[
r_t^{(k)};
h_{t-1}^{(k)}
\right]
\right).
$$

The Transformative Weight therefore controls how strongly each activation-eligible candidate-transformation dimension contributes to recurrent-state evolution.

The learning problem is not to relearn the candidate transformation.

The inherited transformative mechanisms

$$
T_F,\quad T_P,\quad T_S
$$

remain frozen.

Instead, learning is restricted to the latent parameters underlying

$$
TW_F,\quad TW_P,\quad TW_S.
$$

### Effective Transformative-Weight parameterisation

For every active dimension

$$
j\in A_k,
$$

the effective weight remains

$$
TW_{k,j}
=
\sigma
\left(
\alpha_{k,j}
\right),
$$

where

$$
\alpha_{k,j}\in\mathbb{R}
$$

is trainable and

$$
\sigma(\cdot)
$$

is the sigmoid function.

Thus,

$$
0<TW_{k,j}<1.
$$

For every deferred dimension

$$
j\in D_k,
$$

the current pilot contract remains

$$
TW_{k,j}=0,
$$

with no corresponding trainable latent parameter.

The complete structural Transformative-Weight space therefore remains 51-dimensional, while only 25 latent parameters are eligible for optimisation.

### Supervision basis

Transformative-Weight learning must use the validated transformation supervision established in Notebook 08.

For each pathway, the supervised transformation target is the signed difference between the current supervised state and the preceding supervised state:

$$
y_t^{(k)}
=
s_t^{(k)}
-
s_{t-1}^{(k)}.
$$

For the first sentence,

$$
s_0^{(k)}
=
\mathbf{0},
$$

under the deterministic zero architectural initial-condition policy.

Therefore,

$$
y_1^{(k)}
=
s_1^{(k)}.
$$

The targets remain strictly causal.

No future supervised state contributes to the target for sentence \(t\).

### Candidate transformation versus supervised transformation

The frozen transformative mechanism produces

$$
c_t^{(k)}.
$$

The Transformative Weight produces the gated candidate contribution

$$
\widehat{y}_t^{(k)}
=
TW_k
\odot
c_t^{(k)}.
$$

The supervised transformation target is

$$
y_t^{(k)}.
$$

Transformative-Weight learning therefore compares the gated candidate transformation with the validated supervised transformation:

$$
\widehat{y}_t^{(k)}
\longleftrightarrow
y_t^{(k)}.
$$

This preserves the architectural distinction between the two learned mechanisms.

The transformative mechanism determines the candidate direction and magnitude.

The Transformative Weight determines how much of that candidate transformation is admitted into recurrent-state evolution.

### Why the recurrent state is not the direct target

The optimisation target is the transformation increment rather than the accumulated recurrent state.

The recurrent state is

$$
h_t^{(k)}
=
\sum_{\tau=1}^{t}
\widehat{y}_{\tau}^{(k)},
$$

whereas the available supervision describes sentence-level state transitions.

Training directly against the accumulated recurrent state would mix errors from multiple preceding transformations and would make the interpretation of an individual Transformative Weight less direct.

The primary objective therefore operates on

$$
\widehat{y}_t^{(k)}
=
TW_k\odot c_t^{(k)}
$$

rather than directly on

$$
h_t^{(k)}.
$$

The recurrent trajectory remains essential because each candidate transformation depends on the immediately preceding recurrent state.

### Canonical sequence alignment

All supervision must remain aligned to the canonical 16-sentence sequence established in Blocks 8–9.

For every sentence position \(t\), the following identities must agree:

$$
\text{article\_id}_t,
$$

$$
\text{sentence\_id}_t,
$$

and

$$
\text{sequence\_position}_t.
$$

The recurrent trajectory and transformation supervision may only be paired when these identities match exactly.

No positional assumption may silently replace sentence-identity validation.

### Pathway-specific target dimensions

The factual transformation target remains

$$
y_t^{(F)}
\in
\mathbb{R}^{10},
$$

the psychological target remains

$$
y_t^{(P)}
\in
\mathbb{R}^{34},
$$

and the social target remains

$$
y_t^{(S)}
\in
\mathbb{R}^{7}.
$$

These dimensions must align exactly with the corresponding candidate transformations and effective Transformative-Weight vectors.

### Supervision masks

Missing supervision must remain distinct from genuine zero-valued transformation.

For pathway \(k\), let

$$
m_t^{(k)}
\in
\{0,1\}^{d_k}
$$

denote the validated supervision-availability mask.

Then

$$
m_{t,j}^{(k)}=1
$$

means that the transformation target

$$
y_{t,j}^{(k)}
$$

is supervised and may be eligible for the objective.

Conversely,

$$
m_{t,j}^{(k)}=0
$$

means that the target is unavailable and must not contribute to the loss.

A genuine supervised zero remains a valid target.

### Activation eligibility

Supervision availability alone does not make a dimension eligible for Transformative-Weight learning.

Let

$$
a_j^{(k)}
\in
\{0,1\}
$$

denote the activation-eligibility indicator inherited from Notebook 08.

Then

$$
a_j^{(k)}=1
$$

for active dimensions and

$$
a_j^{(k)}=0
$$

for deferred dimensions.

The effective objective mask is therefore

$$
q_{t,j}^{(k)}
=
m_{t,j}^{(k)}
a_j^{(k)}.
$$

Only elements satisfying

$$
q_{t,j}^{(k)}=1
$$

may contribute to Transformative-Weight learning.

### Active pathway dimensions

The factual active dimensions remain

$$
A_F
=
\{1,2,4,8,9\}.
$$

The psychological active dimensions remain

$$
A_P
=
\{1,4,11,12,14,15,16,19,21,22,23,25,26,27,30,32,33\}.
$$

The social active dimensions remain

$$
A_S
=
\{0,2,4\}.
$$

These sets must agree exactly with the Transformative-Weight modules already constructed in Block 3.

No new active dimension may be introduced by Block 10.

### Deferred dimensions

The factual deferred dimensions remain

$$
D_F
=
\{0,3,5,6,7\}.
$$

The psychological deferred dimensions remain

$$
D_P
=
\{0,2,3,5,6,7,8,9,10,13,17,18,20,24,28,29,31\}.
$$

The social deferred dimensions remain

$$
D_S
=
\{1,3,5,6\}.
$$

Deferred dimensions remain structurally present but excluded from the Transformative-Weight objective.

They must not acquire gradients or trainable Transformative-Weight parameters under the current pilot contract.

### Mask-aware pathway objective

For pathway \(k\), define the elementwise squared transformation error

$$
e_{t,j}^{(k)}
=
\left(
TW_{k,j}c_{t,j}^{(k)}
-
y_{t,j}^{(k)}
\right)^2.
$$

The pathway objective is the mean squared error over valid active supervision elements:

$$
\mathcal{L}_{TW}^{(k)}
=
\frac{
\sum_t
\sum_j
q_{t,j}^{(k)}
e_{t,j}^{(k)}
}{
\sum_t
\sum_j
q_{t,j}^{(k)}
}.
$$

Unavailable or deferred elements contribute exactly zero to the numerator and are excluded from the denominator.

### Pathway objective activation

A pathway may contribute to the total objective only when it contains at least one valid active supervision element.

Define

$$
N_k
=
\sum_t
\sum_j
q_{t,j}^{(k)}.
$$

The pathway is objective-active only when

$$
N_k>0.
$$

For the current pilot, factual, psychological, and social pathways are expected to remain objective-active because Notebook 08 established valid active transformation supervision for all three.

This expectation must nevertheless be validated rather than assumed.

### Total Transformative-Weight objective

The initial pilot objective should preserve equal pathway contribution rather than allowing the pathway with the largest number of supervised elements to dominate automatically.

Let

$$
\mathcal{K}_{\mathrm{active}}
$$

denote the set of pathways with valid objective supervision.

The total objective is

$$
\mathcal{L}_{TW}
=
\frac{
1
}{
\left|
\mathcal{K}_{\mathrm{active}}
\right|
}
\sum_{k\in\mathcal{K}_{\mathrm{active}}}
\mathcal{L}_{TW}^{(k)}.
$$

For the expected three active pathways,

$$
\mathcal{L}_{TW}
=
\frac{
\mathcal{L}_{TW}^{(F)}
+
\mathcal{L}_{TW}^{(P)}
+
\mathcal{L}_{TW}^{(S)}
}{
3
}.
$$

No additional pathway-specific loss coefficient is introduced in the current pilot.

### Transformative Weight versus loss weighting

The architectural Transformative Weight must remain distinct from optimisation weighting.

The quantity

$$
TW_{k,j}
$$

modifies the model output itself:

$$
\widehat{y}_{t,j}^{(k)}
=
TW_{k,j}
c_{t,j}^{(k)}.
$$

It is therefore part of the recurrent architecture.

By contrast, a loss coefficient would modify the contribution of an error term after the prediction had already been produced.

Block 10 introduces no such dimension-specific loss coefficient.

This distinction prevents Transformative Weight from collapsing conceptually into ordinary loss reweighting.

### Candidate direction preservation

Because

$$
0\leq TW_{k,j}\leq1,
$$

the Transformative Weight may attenuate or preserve a candidate transformation but cannot reverse its sign.

For

$$
c_{t,j}^{(k)}>0,
$$

the gated candidate satisfies

$$
TW_{k,j}c_{t,j}^{(k)}
\geq0.
$$

For

$$
c_{t,j}^{(k)}<0,
$$

the gated candidate satisfies

$$
TW_{k,j}c_{t,j}^{(k)}
\leq0.
$$

Thus Transformative-Weight learning cannot compensate for an incorrectly directed candidate transformation by reversing it.

This preserves the architectural role established in Blocks 2–4.

### Recurrent dependence during objective construction

Although the objective is evaluated against sentence-level transformation targets, the candidate transformations cannot be treated as independent static predictions.

For

$$
t>1,
$$

the candidate

$$
c_t^{(k)}
$$

depends on

$$
h_{t-1}^{(k)},
$$

and

$$
h_{t-1}^{(k)}
$$

depends on preceding Transformative-Weight-gated candidate transformations.

Therefore the learning-time forward path must preserve the recurrent chain

$$
h_0
\rightarrow
h_1
\rightarrow
\cdots
\rightarrow
h_{16}.
$$

The objective must not independently recompute each sentence from a zero recurrent state.

### Differentiable recurrent trajectory

During future Transformative-Weight training, the recurrent state must remain differentiably connected to the active latent Transformative-Weight parameters.

The learning-time update remains

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
\sigma(\alpha_k)
\odot
c_t^{(k)}.
$$

Unlike the validation trajectory in Block 9, the training trajectory must not detach the recurrent state between sentence steps.

This allows a Transformative Weight at an earlier sentence to influence later candidate transformations through recurrent-state propagation.

### Frozen inherited architecture

The representation and transformative architectures remain fixed during Transformative-Weight learning.

Therefore,

$$
\nabla_{\theta_R}
\mathcal{L}_{TW}
=
0
$$

and

$$
\nabla_{\theta_T}
\mathcal{L}_{TW}
=
0
$$

by parameter-freezing policy.

Only the 25 latent Transformative-Weight parameters may receive gradients:

$$
\nabla_{\alpha}
\mathcal{L}_{TW}
\neq
0
$$

where supported by the recurrent computation and valid supervision.

### Representation forward boundary

The article-level factual, psychological, and social representations prepared in Block 8 remain the canonical inputs for Transformative-Weight learning.

The representation model does not need to be re-executed during objective construction or future Transformative-Weight optimisation.

This keeps the learning stage focused on the Transformative-Weight and recurrent mechanisms and prevents unnecessary recomputation of the validated representation pathway.

### Supervision is not a recurrent input

The validated transformation targets are used only to calculate the objective.

They must not be supplied as recurrent-state inputs.

At sentence \(t\), the recurrent forward path receives only the current representation and the preceding model-generated recurrent state.

Thus the forward architecture remains

$$
\left(
r_t^{(k)},
h_{t-1}^{(k)}
\right)
\rightarrow
c_t^{(k)}
\rightarrow
TW_k\odot c_t^{(k)}
\rightarrow
h_t^{(k)}.
$$

Only after the model prediction is produced is it compared with

$$
y_t^{(k)}.
$$

This prevents target leakage.

### Causal supervision boundary

For sentence \(t\), the objective may use the transformation target associated with sentence \(t\), but the model forward path may not access any supervision belonging to sentence

$$
t+1
$$

or later.

No future representation, target, mask, or recurrent state may influence the prediction at the current sentence.

### Objective numerical contract

Every objective-active target must be finite.

Every corresponding candidate transformation must be finite.

Every effective Transformative Weight must be finite and bounded.

Every gated prediction

$$
TW_k\odot c_t^{(k)}
$$

must be finite.

Every pathway loss must be finite.

The total Transformative-Weight objective must also be finite.

Any violation invalidates the optimisation contract.

### Optimisation scope

The future optimiser may contain exactly the 25 active latent Transformative-Weight parameters.

It must contain:

- 5 factual latent parameters;
- 17 psychological latent parameters;
- 3 social latent parameters.

Therefore,

$$
N_{\mathrm{optimiser}}
=
25.
$$

No representation parameter and no transformative-mechanism parameter may enter the optimiser.

### Optimisation policy

To preserve continuity with the controlled transformative training policy used in Notebook 08, the planned Transformative-Weight optimiser is AdamW.

The pilot policy is:

- optimiser: `AdamW`;
- learning rate: `0.001`;
- weight decay: `0.0001`;
- maximum gradient norm: `1.0`;
- canonical article order preserved;
- batch shuffling disabled;
- future context prohibited.

The exact number of training epochs should be established by the controlled training block rather than silently executed here.

Block 10 records the optimisation contract but does not instantiate the optimiser.

### Gradient-clipping policy

Future Transformative-Weight training should apply gradient clipping before every optimiser step.

The planned maximum norm is

$$
\left\|
\nabla_{\alpha}
\mathcal{L}_{TW}
\right\|_2
\leq
1.0
$$

after clipping.

The unclipped gradient norm may exceed this threshold and should be recorded diagnostically.

### Objective baseline

Before any parameter update, Block 10 should be able to evaluate the objective under the current initial Transformative-Weight state.

The current active latent parameters remain

$$
\alpha_{k,j}=0,
$$

and therefore

$$
TW_{k,j}=0.5.
$$

The resulting pre-training objective provides the baseline against which controlled Transformative-Weight training can subsequently be evaluated.

Evaluating this objective does not itself constitute training.

### Parameter immutability in Block 10

Although the objective forward path may be evaluated, Block 10 must not change any parameter.

The representation parameters must satisfy

$$
\theta_R^{\mathrm{after}}
=
\theta_R^{\mathrm{before}}.
$$

The transformative parameters must satisfy

$$
\theta_T^{\mathrm{after}}
=
\theta_T^{\mathrm{before}}.
$$

The Transformative-Weight parameters must satisfy

$$
\alpha^{\mathrm{after}}
=
\alpha^{\mathrm{before}}.
$$

The total model parameter count remains

$$
902{,}215.
$$

### Block 10 execution boundary

Block 10 may:

- recover the validated Notebook 08 transformation targets and masks;
- align them to the canonical Block 9 article trajectory;
- validate pathway dimensions and sentence identities;
- construct active objective masks;
- reconstruct a differentiable recurrent forward path;
- calculate the pre-training pathway losses;
- calculate the total pre-training Transformative-Weight objective;
- inspect the exact future optimiser parameter scope.

Block 10 must not:

- create an optimiser;
- call `zero_grad`;
- execute a backward pass;
- clip gradients;
- execute an optimiser step;
- update a Transformative-Weight parameter;
- update a transformative parameter;
- update a representation parameter;
- activate deferred dimensions;
- or perform Transformative-Weight training.

### Block 10 completion contract

Block 10 is complete only when:

- the Notebook 08 transformation supervision is recovered from validated upstream artefacts;
- supervision is aligned exactly to the canonical 16-sentence article sequence;
- article identity, sentence identity, and sequence position are valid;
- factual target shape is \((16,10)\);
- psychological target shape is \((16,34)\);
- social target shape is \((16,7)\);
- supervision masks preserve missing values separately from genuine zeros;
- active and deferred dimensions agree exactly with the Notebook 08 and Notebook 09 contracts;
- the effective objective mask satisfies

$$
q_{t,j}^{(k)}
=
m_{t,j}^{(k)}
a_j^{(k)};
$$

- all three pathways contain valid objective-active supervision;
- the gated prediction is defined as

$$
\widehat{y}_t^{(k)}
=
TW_k\odot c_t^{(k)};
$$

- candidate transformations are generated through the complete recurrent chain;
- recurrent state is not detached between learning-time sentence steps;
- supervision is never supplied as a recurrent input;
- no future context is used;
- pathway losses use only mask-valid active elements;
- factual, psychological, and social pathway objectives are finite;
- the total objective is the equal mean of the active pathway objectives;
- the pre-training total objective is finite;
- only the 25 active latent Transformative-Weight parameters are eligible for future optimisation;
- the 894,003 representation parameters remain frozen;
- the 8,187 transformative parameters remain frozen;
- the complete model parameter count remains 902,215;
- no optimiser is created;
- no backward pass is executed;
- no gradient clipping is executed;
- no optimiser step occurs;
- and no parameter is updated.

Successful completion establishes the Transformative-Weight learning objective

$$
\boxed{
\mathcal{L}_{TW}^{(k)}
=
\frac{
\sum_t
\sum_j
q_{t,j}^{(k)}
\left(
TW_{k,j}c_{t,j}^{(k)}
-
y_{t,j}^{(k)}
\right)^2
}{
\sum_t
\sum_j
q_{t,j}^{(k)}
}
}
$$

with

$$
\boxed{
q_{t,j}^{(k)}
=
m_{t,j}^{(k)}
a_j^{(k)}
}
$$

and the total objective

$$
\boxed{
\mathcal{L}_{TW}
=
\frac{
1
}{
\left|
\mathcal{K}_{\mathrm{active}}
\right|
}
\sum_{k\in\mathcal{K}_{\mathrm{active}}}
\mathcal{L}_{TW}^{(k)}
}.
$$

Notebook 09 may then proceed to controlled Transformative-Weight training and parameter-update validation.

In [60]:
# Media AI — Notebook 09
# Block 10: Transformative-Weight Objective Construction,
#           Supervision Alignment and Optimisation Contract
# =============================================================================

from copy import deepcopy

import io
import json

import numpy as np
import torch

from googleapiclient.http import MediaIoBaseDownload


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_10 = 10

NOTEBOOK_09_BLOCK_10_NAME = (
    "Transformative-Weight Objective Construction, "
    "Supervision Alignment and Optimisation Contract"
)

NOTEBOOK_09_BLOCK_10_VERSION = "1.2"


# =============================================================================
# Canonical supervision artefact contract
# =============================================================================

BLOCK_10_FACTUAL_FILENAME = (
    "notebook_06_factual_target_contract.json"
)

BLOCK_10_PSYCHOLOGICAL_DRIVE_FILE_ID = (
    "1gYiCR-i_avGCwE5O9p921es8Mzbzhvyl"
)

BLOCK_10_SOCIAL_FILENAME = (
    "notebook_07_accepted_social_supervision.json"
)


# =============================================================================
# Optimisation policy
# =============================================================================

NOTEBOOK_09_TW_OPTIMISER_POLICY = {
    "optimizer":
        "AdamW",

    "learning_rate":
        0.001,

    "weight_decay":
        0.0001,

    "maximum_gradient_norm":
        1.0,

    "sequence_order_preserved":
        True,

    "batch_shuffling":
        False,

    "future_context_allowed":
        False,
}


# =============================================================================
# Required runtime contract
# =============================================================================

BLOCK_10_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_DRIVE_SERVICE",

    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",

    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Transformative Weight
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",
    "BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT",

    # -------------------------------------------------------------------------
    # Recurrent contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS",
    "NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES",

    # -------------------------------------------------------------------------
    # Block 8 / 9 article contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS",
    "NOTEBOOK_09_CANONICAL_ARTICLE_IDS",
    "NOTEBOOK_09_CANONICAL_SENTENCE_IDS",
    "NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS",

    "NOTEBOOK_09_BLOCK_9_COMPLETE",
    "NOTEBOOK_09_BLOCK_9_VALID",
    "NOTEBOOK_09_ARTICLE_RECURRENT_TRAJECTORY_VALID",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_OBJECTIVE_READY",

    # -------------------------------------------------------------------------
    # Environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_10_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_10_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_10_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 10 prerequisites are not initialised. "
        f"Missing: {BLOCK_10_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_10_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,

        NOTEBOOK_09_BLOCK_9_COMPLETE is True,
        NOTEBOOK_09_BLOCK_9_VALID is True,

        NOTEBOOK_09_ARTICLE_RECURRENT_TRAJECTORY_VALID is True,

        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_OBJECTIVE_READY is True,
    ]
)


if not BLOCK_10_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Block 9 must be valid and complete before "
        "Transformative-Weight objective construction."
    )


# =============================================================================
# Canonical dimensions
# =============================================================================

BLOCK_10_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
)


BLOCK_10_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            BLOCK_10_PATHWAY_DIMENSIONS,
            dict,
        ),

        bool(
            BLOCK_10_PATHWAY_DIMENSIONS
        ),

        (
            set(
                BLOCK_10_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                BLOCK_1_TRANSFORMATIVE_MODULES.keys()
            )
            ==
            set(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys()
            )
        ),

        all(
            isinstance(
                pathway_dim,
                int,
            )
            and
            pathway_dim > 0

            for pathway_dim
            in BLOCK_10_PATHWAY_DIMENSIONS.values()
        ),
    ]
)


if not BLOCK_10_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 pathway dimensions are invalid."
    )


# =============================================================================
# Canonical active / deferred dimensions
# =============================================================================

BLOCK_10_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES.items()
}


BLOCK_10_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES.items()
}


# =============================================================================
# Validate active / deferred partitions
# =============================================================================

BLOCK_10_DIMENSION_PARTITIONS_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_10_PATHWAY_DIMENSIONS.items():

    active_set = set(
        BLOCK_10_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    deferred_set = set(
        BLOCK_10_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )


    BLOCK_10_DIMENSION_PARTITIONS_VALID[
        pathway_name
    ] = all(
        [
            active_set.isdisjoint(
                deferred_set
            ),

            (
                active_set
                |
                deferred_set
            )
            ==
            set(
                range(
                    pathway_dim
                )
            ),
        ]
    )


BLOCK_10_ALL_DIMENSION_PARTITIONS_VALID = all(
    BLOCK_10_DIMENSION_PARTITIONS_VALID.values()
)


if not BLOCK_10_ALL_DIMENSION_PARTITIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 active/deferred dimension "
        "partitions are invalid."
    )


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_10_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_10_nested_state_exact(
    state_before,
    state_after,
):

    if (
        state_before.keys()
        !=
        state_after.keys()
    ):

        return False


    for module_name in state_before:

        if (
            state_before[
                module_name
            ].keys()
            !=
            state_after[
                module_name
            ].keys()
        ):

            return False


        for tensor_name in state_before[
            module_name
        ]:

            if not torch.equal(
                state_before[
                    module_name
                ][
                    tensor_name
                ],
                state_after[
                    module_name
                ][
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot learned state before Block 10
# =============================================================================

BLOCK_10_REPRESENTATION_STATE_BEFORE = (
    block_10_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_10_TRANSFORMATIVE_STATE_BEFORE = (
    block_10_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_10_WEIGHT_STATE_BEFORE = (
    block_10_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Drive helpers
# =============================================================================

def block_10_download_drive_bytes(
    drive_service,
    file_id,
):

    request = (
        drive_service.files()
        .get_media(
            fileId=
                file_id
        )
    )


    buffer = io.BytesIO()


    downloader = MediaIoBaseDownload(
        buffer,
        request,
    )


    done = False


    while not done:

        _, done = downloader.next_chunk()


    return buffer.getvalue()


def block_10_load_json_by_id(
    drive_service,
    file_id,
):

    raw_bytes = block_10_download_drive_bytes(
        drive_service=
            drive_service,

        file_id=
            file_id,
    )


    return json.loads(
        raw_bytes.decode(
            "utf-8"
        )
    )


def block_10_find_json_by_filename(
    drive_service,
    filename,
):

    escaped_filename = filename.replace(
        "'",
        "\\'",
    )


    result = (
        drive_service.files()
        .list(
            q=(
                f"name = '{escaped_filename}' "
                "and trashed = false"
            ),

            spaces=
                "drive",

            fields=
                "files(id,name,modifiedTime,size)",

            orderBy=
                "modifiedTime desc",

            pageSize=
                20,
        )
        .execute()
    )


    files = result.get(
        "files",
        [],
    )


    if not files:

        raise RuntimeError(
            f"Could not locate required artefact: {filename}"
        )


    errors = []


    for file_info in files:

        file_id = file_info[
            "id"
        ]


        try:

            payload = block_10_load_json_by_id(
                drive_service=
                    drive_service,

                file_id=
                    file_id,
            )


            if isinstance(
                payload,
                dict,
            ):

                return (
                    payload,
                    file_info,
                )


        except Exception as exc:

            errors.append(
                (
                    file_id,
                    type(
                        exc
                    ).__name__,
                )
            )


    raise RuntimeError(
        f"Files named {filename} were found, but no JSON payload "
        f"could be restored. Errors: {errors}"
    )


# =============================================================================
# Restore canonical supervision artefacts
# =============================================================================

(
    BLOCK_10_FACTUAL_PAYLOAD,
    BLOCK_10_FACTUAL_FILE_INFO,
) = block_10_find_json_by_filename(
    drive_service=
        NOTEBOOK_09_DRIVE_SERVICE,

    filename=
        BLOCK_10_FACTUAL_FILENAME,
)


BLOCK_10_PSYCHOLOGICAL_PAYLOAD = (
    block_10_load_json_by_id(
        drive_service=
            NOTEBOOK_09_DRIVE_SERVICE,

        file_id=
            BLOCK_10_PSYCHOLOGICAL_DRIVE_FILE_ID,
    )
)


(
    BLOCK_10_SOCIAL_PAYLOAD,
    BLOCK_10_SOCIAL_FILE_INFO,
) = block_10_find_json_by_filename(
    drive_service=
        NOTEBOOK_09_DRIVE_SERVICE,

    filename=
        BLOCK_10_SOCIAL_FILENAME,
)


# =============================================================================
# Canonical corpus contract
# =============================================================================

BLOCK_10_ARTICLE_IDS = tuple(
    NOTEBOOK_09_CANONICAL_ARTICLE_IDS
)


BLOCK_10_SENTENCE_IDS = list(
    NOTEBOOK_09_CANONICAL_SENTENCE_IDS
)


BLOCK_10_SEQUENCE_POSITIONS = list(
    NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS
)


BLOCK_10_SENTENCE_COUNT = len(
    BLOCK_10_SENTENCE_IDS
)


BLOCK_10_ARTICLE_COUNT = len(
    BLOCK_10_ARTICLE_IDS
)


BLOCK_10_INPUT_RECORD_COUNT = len(
    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
)


BLOCK_10_INPUT_ARTICLE_IDS = [
    record[
        "article_id"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_10_OBSERVED_ARTICLE_IDS = tuple(
    dict.fromkeys(
        BLOCK_10_INPUT_ARTICLE_IDS
    )
)


BLOCK_10_CORPUS_CONTRACT_VALID = all(
    [
        BLOCK_10_ARTICLE_COUNT > 0,
        BLOCK_10_SENTENCE_COUNT > 0,

        (
            BLOCK_10_INPUT_RECORD_COUNT
            ==
            BLOCK_10_SENTENCE_COUNT
        ),

        (
            BLOCK_10_OBSERVED_ARTICLE_IDS
            ==
            BLOCK_10_ARTICLE_IDS
        ),
    ]
)


if not BLOCK_10_CORPUS_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 canonical corpus contract is invalid."
    )


BLOCK_10_ARTICLE_START_INDICES = []

BLOCK_10_ARTICLE_RECORD_INDICES = {
    article_id:
        []

    for article_id
    in BLOCK_10_ARTICLE_IDS
}


for (
    record_index,
    record,
) in enumerate(
    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
):

    article_id = record[
        "article_id"
    ]


    if article_id not in BLOCK_10_ARTICLE_RECORD_INDICES:

        raise RuntimeError(
            f"Unexpected article_id in recurrent input records: {article_id}"
        )


    if not BLOCK_10_ARTICLE_RECORD_INDICES[
        article_id
    ]:

        BLOCK_10_ARTICLE_START_INDICES.append(
            record_index
        )


    BLOCK_10_ARTICLE_RECORD_INDICES[
        article_id
    ].append(
        record_index
    )


BLOCK_10_ARTICLE_BOUNDARIES_VALID = all(
    len(
        indices
    )
    >
    0

    for indices
    in BLOCK_10_ARTICLE_RECORD_INDICES.values()
)


if not BLOCK_10_ARTICLE_BOUNDARIES_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 article boundary reconstruction is invalid."
    )


# =============================================================================
# General extraction helpers
# =============================================================================

def block_10_get_records(
    payload,
):

    if isinstance(
        payload,
        list,
    ):

        if all(
            isinstance(
                item,
                dict,
            )

            for item
            in payload
        ):

            return payload


    if not isinstance(
        payload,
        dict,
    ):

        return None


    for key in (
        "records",
        "target_records",
        "supervision_records",
        "accepted_records",
        "sentence_records",
    ):

        records = payload.get(
            key
        )


        if (
            isinstance(
                records,
                list,
            )
            and
            all(
                isinstance(
                    item,
                    dict,
                )

                for item
                in records
            )
        ):

            return records


    return None


def block_10_get_sentence_id(
    record,
):

    if not isinstance(
        record,
        dict,
    ):

        return None


    identity = record.get(
        "identity"
    )


    if isinstance(
        identity,
        dict,
    ):

        sentence_id = identity.get(
            "sentence_id"
        )


        if sentence_id not in (
            None,
            "",
        ):

            return str(
                sentence_id
            )


    for key in (
        "sentence_id",
        "sentenceId",
    ):

        sentence_id = record.get(
            key
        )


        if sentence_id not in (
            None,
            "",
        ):

            return str(
                sentence_id
            )


    return None


def block_10_value_and_mask(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        value_state = value.get(
            "value_state"
        )


        raw_value = value.get(
            "value"
        )


        if (
            value_state
            ==
            "observed"
            and
            raw_value is not None
        ):

            return (
                float(
                    raw_value
                ),
                True,
            )


        return (
            0.0,
            False,
        )


    if value is None:

        return (
            0.0,
            False,
        )


    if isinstance(
        value,
        (
            bool,
            np.bool_,
        ),
    ):

        return (
            float(
                value
            ),
            True,
        )


    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        return (
            float(
                value
            ),
            True,
        )


    return (
        0.0,
        False,
    )


def block_10_to_float_vector(
    value,
    expected_dim,
):

    if torch.is_tensor(
        value
    ):

        array = (
            value.detach()
            .cpu()
            .numpy()
        )


    else:

        try:

            array = np.asarray(
                value,
                dtype=
                    np.float32,
            )


        except Exception:

            return None


    array = np.asarray(
        array,
        dtype=
            np.float32,
    ).reshape(
        -1
    )


    if array.shape != (
        expected_dim,
    ):

        return None


    return array


def block_10_to_bool_vector(
    value,
    expected_dim,
):

    try:

        array = np.asarray(
            value
        ).reshape(
            -1
        )


    except Exception:

        return None


    if array.shape != (
        expected_dim,
    ):

        return None


    return array.astype(
        bool
    )


# =============================================================================
# Factual current-state supervision resolver
# =============================================================================

def block_10_resolve_factual_supervision(
    payload,
):

    records = block_10_get_records(
        payload
    )


    if records is None:

        raise RuntimeError(
            "Notebook 06 factual target contract does not expose records."
        )


    target_aliases = (
        "factual_target",
        "target",
        "targets",
        "factual_values",
        "factual_vector",
    )


    mask_aliases = (
        "factual_mask",
        "target_mask",
        "mask",
        "masks",
        "factual_masks",
    )


    resolved = {}


    for record in records:

        sentence_id = block_10_get_sentence_id(
            record
        )


        if sentence_id is None:

            continue


        target = None

        mask = None


        search_dicts = [
            record
        ]


        for key in (
            "factual",
            "supervision",
            "target_contract",
            "targets",
        ):

            nested = record.get(
                key
            )


            if isinstance(
                nested,
                dict,
            ):

                search_dicts.append(
                    nested
                )


        for search_dict in search_dicts:

            if target is None:

                for alias in target_aliases:

                    if alias in search_dict:

                        target = block_10_to_float_vector(
                            search_dict[
                                alias
                            ],
                            expected_dim=
                                BLOCK_10_PATHWAY_DIMENSIONS[
                                    "factual"
                                ],
                        )


                        if target is not None:

                            break


            if mask is None:

                for alias in mask_aliases:

                    if alias in search_dict:

                        mask = block_10_to_bool_vector(
                            search_dict[
                                alias
                            ],
                            expected_dim=
                                BLOCK_10_PATHWAY_DIMENSIONS[
                                    "factual"
                                ],
                        )


                        if mask is not None:

                            break


        if target is not None:

            if mask is None:

                mask = np.isfinite(
                    target
                )


            target = np.where(
                mask,
                target,
                0.0,
            ).astype(
                np.float32
            )


            resolved[
                sentence_id
            ] = (
                target,
                mask.astype(
                    bool
                ),
            )


    if set(
        resolved.keys()
    ) != set(
        BLOCK_10_SENTENCE_IDS
    ):

        missing = [
            sentence_id

            for sentence_id
            in BLOCK_10_SENTENCE_IDS

            if sentence_id
            not in resolved
        ]


        raise RuntimeError(
            "Notebook 06 factual supervision could not be aligned "
            f"to all canonical sentences. Missing: {missing}"
        )


    targets = np.stack(
        [
            resolved[
                sentence_id
            ][
                0
            ]

            for sentence_id
            in BLOCK_10_SENTENCE_IDS
        ],
        axis=
            0,
    )


    masks = np.stack(
        [
            resolved[
                sentence_id
            ][
                1
            ]

            for sentence_id
            in BLOCK_10_SENTENCE_IDS
        ],
        axis=
            0,
    )


    return (
        targets,
        masks,
    )


# =============================================================================
# Psychological current-state supervision resolver
# =============================================================================
#
# Canonical dimension order:
#
#   REV  0:11
#   TEV 11:22
#   IEV 22:33
#   EW     33
# =============================================================================

BLOCK_10_PSYCHOLOGICAL_EMOTION_KEYS = (
    "fear",
    "anger",
    "sadness",
    "disgust",
    "tension",
    "uncertainty",
    "hope",
    "relief",
    "trust",
    "compassion",
    "solidarity",
)


def block_10_resolve_psychological_supervision(
    payload,
):

    records = block_10_get_records(
        payload
    )


    if records is None:

        raise RuntimeError(
            "Notebook 03 accepted supervision artefact "
            "does not expose records."
        )


    resolved = {}


    for record in records:

        sentence_id = block_10_get_sentence_id(
            record
        )


        if sentence_id is None:

            continue


        annotations = record.get(
            "annotations"
        )


        if not isinstance(
            annotations,
            dict,
        ):

            continue


        family_values = []

        family_masks = []


        for family_name in (
            "reported_emotion",
            "transformative_emotion",
            "immediate_emotional_state",
        ):

            family = annotations.get(
                family_name
            )


            if not isinstance(
                family,
                dict,
            ):

                raise RuntimeError(
                    f"Psychological family {family_name} "
                    f"is missing for sentence {sentence_id}."
                )


            for emotion_name in BLOCK_10_PSYCHOLOGICAL_EMOTION_KEYS:

                (
                    value,
                    observed,
                ) = block_10_value_and_mask(
                    family.get(
                        emotion_name
                    )
                )


                family_values.append(
                    value
                )


                family_masks.append(
                    observed
                )


        (
            ew_value,
            ew_observed,
        ) = block_10_value_and_mask(
            annotations.get(
                "emotional_weight"
            )
        )


        family_values.append(
            ew_value
        )


        family_masks.append(
            ew_observed
        )


        psychological_vector = np.asarray(
            family_values,
            dtype=
                np.float32,
        )


        psychological_mask = np.asarray(
            family_masks,
            dtype=
                bool,
        )


        expected_psychological_dim = (
            BLOCK_10_PATHWAY_DIMENSIONS[
                "psychological"
            ]
        )


        if psychological_vector.shape != (
            expected_psychological_dim,
        ):

            raise RuntimeError(
                "Psychological supervision dimensionality does not match "
                "the inherited psychological pathway contract. "
                f"Observed {psychological_vector.shape}, "
                f"expected {(expected_psychological_dim,)}."
            )


        resolved[
            sentence_id
        ] = (
            psychological_vector,
            psychological_mask,
        )


    if set(
        resolved.keys()
    ) != set(
        BLOCK_10_SENTENCE_IDS
    ):

        missing = [
            sentence_id

            for sentence_id
            in BLOCK_10_SENTENCE_IDS

            if sentence_id
            not in resolved
        ]


        raise RuntimeError(
            "Notebook 03 psychological supervision could not be aligned "
            f"to all canonical sentences. Missing: {missing}"
        )


    targets = np.stack(
        [
            resolved[
                sentence_id
            ][
                0
            ]

            for sentence_id
            in BLOCK_10_SENTENCE_IDS
        ],
        axis=
            0,
    )


    masks = np.stack(
        [
            resolved[
                sentence_id
            ][
                1
            ]

            for sentence_id
            in BLOCK_10_SENTENCE_IDS
        ],
        axis=
            0,
    )


    return (
        targets,
        masks,
    )


# =============================================================================
# Social current-state supervision resolver
# =============================================================================
#
# Canonical Notebook 07 accepted-supervision structure:
#
# payload["architecture"]["dimension_order"]
#
# and for every sentence:
#
# record["social_annotations"][dimension_name] = {
#     "available": bool,
#     "value": numerical value or None,
#     ...
# }
#
# Missing supervision remains distinct from genuine zero.
# =============================================================================

def block_10_resolve_social_supervision(
    payload,
):

    if not isinstance(
        payload,
        dict,
    ):

        raise TypeError(
            "Notebook 07 accepted social supervision artefact "
            "must be dictionary-like."
        )


    # -------------------------------------------------------------------------
    # Architecture contract
    # -------------------------------------------------------------------------

    architecture = payload.get(
        "architecture"
    )


    if not isinstance(
        architecture,
        dict,
    ):

        raise RuntimeError(
            "Notebook 07 accepted social supervision artefact "
            "does not expose architecture."
        )


    representation_dimension = architecture.get(
        "representation_dimension"
    )


    try:

        representation_dimension = int(
            representation_dimension
        )


    except (
        TypeError,
        ValueError,
    ) as exc:

        raise RuntimeError(
            "Notebook 07 social representation dimension "
            "is not integer-compatible."
        ) from exc


    expected_social_dim = (
        BLOCK_10_PATHWAY_DIMENSIONS[
            "social"
        ]
    )


    if representation_dimension != expected_social_dim:

        raise RuntimeError(
            "Notebook 07 accepted social supervision dimensionality "
            "does not match the inherited social pathway contract. "
            f"Observed: {representation_dimension}; "
            f"expected: {expected_social_dim}."
        )


    dimension_order = architecture.get(
        "dimension_order"
    )


    if not isinstance(
        dimension_order,
        (
            list,
            tuple,
        ),
    ):

        raise RuntimeError(
            "Notebook 07 accepted social supervision architecture "
            "does not expose dimension_order."
        )


    dimension_order = tuple(
        str(
            dimension_name
        )

        for dimension_name
        in dimension_order
    )


    if len(
        dimension_order
    ) != expected_social_dim:

        raise RuntimeError(
            "Notebook 07 Social Representation dimension order "
            "does not reproduce the inherited social pathway dimensionality."
        )


    if len(
        set(
            dimension_order
        )
    ) != expected_social_dim:

        raise RuntimeError(
            "Notebook 07 Social Representation dimension names "
            "are not unique."
        )


    # -------------------------------------------------------------------------
    # Accepted social records
    # -------------------------------------------------------------------------

    records = payload.get(
        "records"
    )


    if not isinstance(
        records,
        list,
    ):

        raise RuntimeError(
            "Notebook 07 accepted social supervision artefact "
            "does not expose records."
        )


    if len(
        records
    ) != BLOCK_10_SENTENCE_COUNT:

        raise RuntimeError(
            "Notebook 07 accepted social supervision record count "
            "does not match the canonical Notebook 09 article. "
            f"Expected {BLOCK_10_SENTENCE_COUNT}, "
            f"observed {len(records)}."
        )


    # -------------------------------------------------------------------------
    # Reconstruct target / mask by sentence identity
    # -------------------------------------------------------------------------

    resolved = {}


    for (
        record_index,
        record,
    ) in enumerate(
        records
    ):

        if not isinstance(
            record,
            dict,
        ):

            raise TypeError(
                f"Notebook 07 social record {record_index} "
                "is not dictionary-like."
            )


        sentence_id = block_10_get_sentence_id(
            record
        )


        if sentence_id is None:

            raise RuntimeError(
                f"Notebook 07 social record {record_index} "
                "does not expose sentence_id."
            )


        social_annotations = record.get(
            "social_annotations"
        )


        if not isinstance(
            social_annotations,
            dict,
        ):

            raise RuntimeError(
                f"Notebook 07 social record {record_index} "
                "does not expose social_annotations."
            )


        target = np.full(
            expected_social_dim,
            np.nan,
            dtype=
                np.float32,
        )


        mask = np.zeros(
            expected_social_dim,
            dtype=
                bool,
        )


        for (
            dimension_index,
            dimension_name,
        ) in enumerate(
            dimension_order
        ):

            if dimension_name not in social_annotations:

                raise RuntimeError(
                    f"Notebook 07 social record {record_index} "
                    f"does not contain dimension {dimension_name}."
                )


            annotation = social_annotations[
                dimension_name
            ]


            if not isinstance(
                annotation,
                dict,
            ):

                raise RuntimeError(
                    f"Social annotation {dimension_name} in "
                    f"record {record_index} is not dictionary-like."
                )


            available = bool(
                annotation.get(
                    "available",
                    False,
                )
            )


            value = annotation.get(
                "value"
            )


            if available:

                if value is None:

                    raise RuntimeError(
                        f"Available social annotation {dimension_name} "
                        f"in record {record_index} has no value."
                    )


                try:

                    numeric_value = float(
                        value
                    )


                except (
                    TypeError,
                    ValueError,
                ) as exc:

                    raise RuntimeError(
                        f"Available social annotation {dimension_name} "
                        f"in record {record_index} is not numerical."
                    ) from exc


                if not np.isfinite(
                    numeric_value
                ):

                    raise RuntimeError(
                        f"Available social annotation {dimension_name} "
                        f"in record {record_index} is non-finite."
                    )


                target[
                    dimension_index
                ] = numeric_value


                mask[
                    dimension_index
                ] = True


        if sentence_id in resolved:

            raise RuntimeError(
                "Notebook 07 accepted social supervision contains "
                f"duplicate sentence_id: {sentence_id}"
            )


        resolved[
            sentence_id
        ] = (
            target,
            mask,
        )


    # -------------------------------------------------------------------------
    # Canonical alignment
    # -------------------------------------------------------------------------

    resolved_sentence_ids = set(
        resolved.keys()
    )


    canonical_sentence_ids = set(
        BLOCK_10_SENTENCE_IDS
    )


    if (
        resolved_sentence_ids
        !=
        canonical_sentence_ids
    ):

        missing = sorted(
            canonical_sentence_ids
            -
            resolved_sentence_ids
        )


        unexpected = sorted(
            resolved_sentence_ids
            -
            canonical_sentence_ids
        )


        raise RuntimeError(
            "Notebook 07 accepted social supervision does not align "
            "to the canonical Notebook 09 sentence set. "
            f"Missing: {missing}. "
            f"Unexpected: {unexpected}."
        )


    # -------------------------------------------------------------------------
    # Preserve Notebook 09 canonical sentence order
    # -------------------------------------------------------------------------

    targets = np.stack(
        [
            resolved[
                sentence_id
            ][
                0
            ]

            for sentence_id
            in BLOCK_10_SENTENCE_IDS
        ],
        axis=
            0,
    )


    masks = np.stack(
        [
            resolved[
                sentence_id
            ][
                1
            ]

            for sentence_id
            in BLOCK_10_SENTENCE_IDS
        ],
        axis=
            0,
    )


    expected_social_shape = (
        BLOCK_10_SENTENCE_COUNT,
        expected_social_dim,
    )


    if targets.shape != expected_social_shape:

        raise RuntimeError(
            "Reconstructed Notebook 07 social target matrix "
            f"has invalid shape: {targets.shape}; "
            f"expected {expected_social_shape}."
        )


    if masks.shape != expected_social_shape:

        raise RuntimeError(
            "Reconstructed Notebook 07 social mask matrix "
            f"has invalid shape: {masks.shape}; "
            f"expected {expected_social_shape}."
        )


    if not bool(
        np.isfinite(
            targets[
                masks
            ]
        ).all()
    ):

        raise RuntimeError(
            "Observed Notebook 07 social supervision contains "
            "non-finite values."
        )


    if not bool(
        np.isnan(
            targets[
                ~masks
            ]
        ).all()
    ):

        raise RuntimeError(
            "Unavailable Notebook 07 social supervision must "
            "remain NaN before mask-aware numerical normalisation."
        )


    numerical_targets = np.where(
        masks,
        targets,
        0.0,
    ).astype(
        np.float32
    )


    return (
        numerical_targets,
        masks.astype(
            bool
        ),
        dimension_order,
    )


# =============================================================================
# Recover current-state supervision
# =============================================================================

(
    BLOCK_10_FACTUAL_CURRENT_TARGETS_NUMPY,
    BLOCK_10_FACTUAL_CURRENT_MASKS_NUMPY,
) = block_10_resolve_factual_supervision(
    BLOCK_10_FACTUAL_PAYLOAD
)


(
    BLOCK_10_PSYCHOLOGICAL_CURRENT_TARGETS_NUMPY,
    BLOCK_10_PSYCHOLOGICAL_CURRENT_MASKS_NUMPY,
) = block_10_resolve_psychological_supervision(
    BLOCK_10_PSYCHOLOGICAL_PAYLOAD
)


(
    BLOCK_10_SOCIAL_CURRENT_TARGETS_NUMPY,
    BLOCK_10_SOCIAL_CURRENT_MASKS_NUMPY,
    BLOCK_10_SOCIAL_DIMENSION_NAMES,
) = block_10_resolve_social_supervision(
    BLOCK_10_SOCIAL_PAYLOAD
)


# =============================================================================
# Current-state shape validation
# =============================================================================

BLOCK_10_CURRENT_TARGET_SHAPES = {
    "factual":
        BLOCK_10_FACTUAL_CURRENT_TARGETS_NUMPY.shape,

    "psychological":
        BLOCK_10_PSYCHOLOGICAL_CURRENT_TARGETS_NUMPY.shape,

    "social":
        BLOCK_10_SOCIAL_CURRENT_TARGETS_NUMPY.shape,
}


BLOCK_10_CURRENT_MASK_SHAPES = {
    "factual":
        BLOCK_10_FACTUAL_CURRENT_MASKS_NUMPY.shape,

    "psychological":
        BLOCK_10_PSYCHOLOGICAL_CURRENT_MASKS_NUMPY.shape,

    "social":
        BLOCK_10_SOCIAL_CURRENT_MASKS_NUMPY.shape,
}


BLOCK_10_EXPECTED_TARGET_SHAPES = {
    pathway_name:
        (
            BLOCK_10_SENTENCE_COUNT,
            pathway_dim,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_10_PATHWAY_DIMENSIONS.items()
}


BLOCK_10_CURRENT_TARGET_SHAPES_VALID = (
    BLOCK_10_CURRENT_TARGET_SHAPES
    ==
    BLOCK_10_EXPECTED_TARGET_SHAPES
)


BLOCK_10_CURRENT_MASK_SHAPES_VALID = (
    BLOCK_10_CURRENT_MASK_SHAPES
    ==
    BLOCK_10_EXPECTED_TARGET_SHAPES
)


if not all(
    [
        BLOCK_10_CURRENT_TARGET_SHAPES_VALID,
        BLOCK_10_CURRENT_MASK_SHAPES_VALID,
    ]
):

    raise RuntimeError(
        "Recovered current-state supervision has invalid dimensions. "
        f"Targets: {BLOCK_10_CURRENT_TARGET_SHAPES}, "
        f"Masks: {BLOCK_10_CURRENT_MASK_SHAPES}"
    )


# =============================================================================
# Current-state availability validation
# =============================================================================

BLOCK_10_CURRENT_AVAILABLE_ELEMENTS = {
    "factual":
        int(
            BLOCK_10_FACTUAL_CURRENT_MASKS_NUMPY.sum()
        ),

    "psychological":
        int(
            BLOCK_10_PSYCHOLOGICAL_CURRENT_MASKS_NUMPY.sum()
        ),

    "social":
        int(
            BLOCK_10_SOCIAL_CURRENT_MASKS_NUMPY.sum()
        ),
}


# =============================================================================
# Convert current supervision to tensors
# =============================================================================

BLOCK_10_CURRENT_SUPERVISION = {
    "factual":
        torch.as_tensor(
            BLOCK_10_FACTUAL_CURRENT_TARGETS_NUMPY,
            dtype=
                DEFAULT_DTYPE,
            device=
                DEVICE,
        ),

    "psychological":
        torch.as_tensor(
            BLOCK_10_PSYCHOLOGICAL_CURRENT_TARGETS_NUMPY,
            dtype=
                DEFAULT_DTYPE,
            device=
                DEVICE,
        ),

    "social":
        torch.as_tensor(
            BLOCK_10_SOCIAL_CURRENT_TARGETS_NUMPY,
            dtype=
                DEFAULT_DTYPE,
            device=
                DEVICE,
        ),
}


BLOCK_10_CURRENT_SUPERVISION_MASKS = {
    "factual":
        torch.as_tensor(
            BLOCK_10_FACTUAL_CURRENT_MASKS_NUMPY,
            dtype=
                torch.bool,
            device=
                DEVICE,
        ),

    "psychological":
        torch.as_tensor(
            BLOCK_10_PSYCHOLOGICAL_CURRENT_MASKS_NUMPY,
            dtype=
                torch.bool,
            device=
                DEVICE,
        ),

    "social":
        torch.as_tensor(
            BLOCK_10_SOCIAL_CURRENT_MASKS_NUMPY,
            dtype=
                torch.bool,
            device=
                DEVICE,
        ),
}


# =============================================================================
# Current supervision numerical validation
# =============================================================================

BLOCK_10_CURRENT_AVAILABLE_VALUES_FINITE = all(
    bool(
        torch.isfinite(
            BLOCK_10_CURRENT_SUPERVISION[
                pathway_name
            ][
                BLOCK_10_CURRENT_SUPERVISION_MASKS[
                    pathway_name
                ]
            ]
        ).all().item()
    )

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
)


if not BLOCK_10_CURRENT_AVAILABLE_VALUES_FINITE:

    raise RuntimeError(
        "Recovered current-state supervision contains "
        "non-finite observed values."
    )


# =============================================================================
# Construct preceding supervised conditions
# =============================================================================
#
# Each article starts from the deterministic zero architectural condition.
# Supervised state is shifted only within the current article.
# =============================================================================

BLOCK_10_PRECEDING_SUPERVISION = {}

BLOCK_10_PRECEDING_SUPERVISION_MASKS = {}


for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

    current = (
        BLOCK_10_CURRENT_SUPERVISION[
            pathway_name
        ]
    )


    current_mask = (
        BLOCK_10_CURRENT_SUPERVISION_MASKS[
            pathway_name
        ]
    )


    preceding = torch.zeros_like(
        current
    )


    preceding_mask = torch.zeros_like(
        current_mask
    )


    for article_id in BLOCK_10_ARTICLE_IDS:

        article_indices = (
            BLOCK_10_ARTICLE_RECORD_INDICES[
                article_id
            ]
        )


        first_index = article_indices[
            0
        ]


        preceding[
            first_index
        ] = 0.0


        preceding_mask[
            first_index
        ] = True


        for local_position in range(
            1,
            len(
                article_indices
            ),
        ):

            current_index = article_indices[
                local_position
            ]


            previous_index = article_indices[
                local_position
                -
                1
            ]


            preceding[
                current_index
            ] = current[
                previous_index
            ]


            preceding_mask[
                current_index
            ] = current_mask[
                previous_index
            ]


    BLOCK_10_PRECEDING_SUPERVISION[
        pathway_name
    ] = preceding


    BLOCK_10_PRECEDING_SUPERVISION_MASKS[
        pathway_name
    ] = preceding_mask


BLOCK_10_PRECEDING_SUPERVISION_BOUNDARIES_VALID = all(
    all(
        torch.equal(
            BLOCK_10_PRECEDING_SUPERVISION[
                pathway_name
            ][
                start_index
            ],
            torch.zeros_like(
                BLOCK_10_PRECEDING_SUPERVISION[
                    pathway_name
                ][
                    start_index
                ]
            ),
        )

        and

        bool(
            BLOCK_10_PRECEDING_SUPERVISION_MASKS[
                pathway_name
            ][
                start_index
            ].all().item()
        )

        for start_index
        in BLOCK_10_ARTICLE_START_INDICES
    )

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
)


if not BLOCK_10_PRECEDING_SUPERVISION_BOUNDARIES_VALID:

    raise RuntimeError(
        "Block 10 preceding-supervision article-boundary reset is invalid."
    )


# =============================================================================
# Construct signed transformation targets and masks
# =============================================================================

BLOCK_10_TRANSFORMATIVE_TARGETS = {}

BLOCK_10_TRANSFORMATIVE_TARGET_MASKS = {}


for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

    current = (
        BLOCK_10_CURRENT_SUPERVISION[
            pathway_name
        ]
    )


    preceding = (
        BLOCK_10_PRECEDING_SUPERVISION[
            pathway_name
        ]
    )


    current_mask = (
        BLOCK_10_CURRENT_SUPERVISION_MASKS[
            pathway_name
        ]
    )


    preceding_mask = (
        BLOCK_10_PRECEDING_SUPERVISION_MASKS[
            pathway_name
        ]
    )


    target_mask = (
        current_mask
        &
        preceding_mask
    )


    target = (
        current
        -
        preceding
    )


    target = torch.where(
        target_mask,
        target,
        torch.zeros_like(
            target
        ),
    )


    BLOCK_10_TRANSFORMATIVE_TARGETS[
        pathway_name
    ] = target


    BLOCK_10_TRANSFORMATIVE_TARGET_MASKS[
        pathway_name
    ] = target_mask


# =============================================================================
# Validate transformation target shapes and numerical state
# =============================================================================

BLOCK_10_TRANSFORMATIVE_TARGET_SHAPES_VALID = all(
    tuple(
        BLOCK_10_TRANSFORMATIVE_TARGETS[
            pathway_name
        ].shape
    )
    ==
    BLOCK_10_EXPECTED_TARGET_SHAPES[
        pathway_name
    ]

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
)


BLOCK_10_AVAILABLE_TARGETS_FINITE = all(
    bool(
        torch.isfinite(
            BLOCK_10_TRANSFORMATIVE_TARGETS[
                pathway_name
            ][
                BLOCK_10_TRANSFORMATIVE_TARGET_MASKS[
                    pathway_name
                ]
            ]
        ).all().item()
    )

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
)


if not all(
    [
        BLOCK_10_TRANSFORMATIVE_TARGET_SHAPES_VALID,
        BLOCK_10_AVAILABLE_TARGETS_FINITE,
    ]
):

    raise RuntimeError(
        "Derived Transformative-Weight supervision is invalid."
    )


# =============================================================================
# Active-dimension objective masks
# =============================================================================

BLOCK_10_ACTIVE_DIMENSION_MASKS = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_10_PATHWAY_DIMENSIONS.items():

    active_mask = torch.zeros(
        pathway_dim,
        dtype=
            torch.bool,
        device=
            DEVICE,
    )


    active_mask[
        list(
            BLOCK_10_ACTIVE_DIMENSION_INDICES[
                pathway_name
            ]
        )
    ] = True


    BLOCK_10_ACTIVE_DIMENSION_MASKS[
        pathway_name
    ] = active_mask


# =============================================================================
# Effective objective masks
# =============================================================================

BLOCK_10_OBJECTIVE_MASKS = {}


for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

    BLOCK_10_OBJECTIVE_MASKS[
        pathway_name
    ] = (
        BLOCK_10_TRANSFORMATIVE_TARGET_MASKS[
            pathway_name
        ]
        &
        BLOCK_10_ACTIVE_DIMENSION_MASKS[
            pathway_name
        ].unsqueeze(
            0
        )
    )


# =============================================================================
# Objective-active element counts
# =============================================================================

BLOCK_10_TRANSFORMATIVE_AVAILABLE_ELEMENTS = {}

BLOCK_10_OBJECTIVE_ACTIVE_ELEMENTS = {}


for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

    BLOCK_10_TRANSFORMATIVE_AVAILABLE_ELEMENTS[
        pathway_name
    ] = int(
        BLOCK_10_TRANSFORMATIVE_TARGET_MASKS[
            pathway_name
        ].sum().item()
    )


    BLOCK_10_OBJECTIVE_ACTIVE_ELEMENTS[
        pathway_name
    ] = int(
        BLOCK_10_OBJECTIVE_MASKS[
            pathway_name
        ].sum().item()
    )


BLOCK_10_OBJECTIVE_ACTIVE_PATHWAYS = [
    pathway_name

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS

    if (
        BLOCK_10_OBJECTIVE_ACTIVE_ELEMENTS[
            pathway_name
        ]
        >
        0
    )
]


BLOCK_10_ALL_PATHWAYS_OBJECTIVE_ACTIVE = (
    len(
        BLOCK_10_OBJECTIVE_ACTIVE_PATHWAYS
    )
    ==
    len(
        BLOCK_10_PATHWAY_DIMENSIONS
    )
)


if not BLOCK_10_ALL_PATHWAYS_OBJECTIVE_ACTIVE:

    raise RuntimeError(
        "One or more Transformative-Weight pathways contain "
        "no objective-active supervision."
    )


# =============================================================================
# Canonical recurrent inputs
# =============================================================================

BLOCK_10_REPRESENTATION_FIELDS = {
    pathway_name:
        f"{pathway_name}_representation"

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
}


BLOCK_10_TRANSFORMATIVE_MODULES = {
    pathway_name:
        BLOCK_1_TRANSFORMATIVE_MODULES[
            pathway_name
        ]

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
}


# =============================================================================
# Validate corpus input identity alignment
# =============================================================================

BLOCK_10_ARTICLE_INPUT_SENTENCE_IDS = [
    record[
        "sentence_id"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_10_ARTICLE_INPUT_SEQUENCE_POSITIONS = [
    record[
        "sequence_position"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_10_ARTICLE_INPUT_ARTICLE_IDS = [
    record[
        "article_id"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_10_ARTICLE_INPUT_IDENTITIES_VALID = all(
    [
        (
            BLOCK_10_ARTICLE_INPUT_SENTENCE_IDS
            ==
            BLOCK_10_SENTENCE_IDS
        ),

        (
            BLOCK_10_ARTICLE_INPUT_SEQUENCE_POSITIONS
            ==
            BLOCK_10_SEQUENCE_POSITIONS
        ),

        (
            tuple(
                dict.fromkeys(
                    BLOCK_10_ARTICLE_INPUT_ARTICLE_IDS
                )
            )
            ==
            BLOCK_10_ARTICLE_IDS
        ),
    ]
)


if not BLOCK_10_ARTICLE_INPUT_IDENTITIES_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 recurrent inputs do not preserve "
        "the canonical corpus article/sentence sequence."
    )


# =============================================================================
# Differentiable recurrent objective forward
# =============================================================================
#
# The recurrent state is not detached within an article.
# A fresh zero recurrent state is constructed at every article boundary.
# =============================================================================

def block_10_forward_tw_objective():

    pathway_outputs = {}


    for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

        pathway_dim = (
            BLOCK_10_PATHWAY_DIMENSIONS[
                pathway_name
            ]
        )


        candidates = []

        effective_weights = []

        gated_predictions = []

        recurrent_states = []


        for article_id in BLOCK_10_ARTICLE_IDS:

            previous_state = torch.zeros(
                (
                    1,
                    pathway_dim,
                ),
                dtype=
                    DEFAULT_DTYPE,
                device=
                    DEVICE,
            )


            for sentence_position in (
                BLOCK_10_ARTICLE_RECORD_INDICES[
                    article_id
                ]
            ):

                input_record = (
                    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[
                        sentence_position
                    ]
                )


                current_representation = (
                    input_record[
                        BLOCK_10_REPRESENTATION_FIELDS[
                            pathway_name
                        ]
                    ].to(
                        device=
                            DEVICE,

                        dtype=
                            DEFAULT_DTYPE,
                    )
                )


                candidate = (
                    BLOCK_10_TRANSFORMATIVE_MODULES[
                        pathway_name
                    ](
                        current_representation,
                        previous_state,
                    )
                )


                effective_weight = (
                    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                        pathway_name
                    ]()
                )


                gated_prediction = (
                    effective_weight.unsqueeze(
                        0
                    )
                    *
                    candidate
                )


                updated_state = (
                    previous_state
                    +
                    gated_prediction
                )


                candidates.append(
                    candidate
                )


                effective_weights.append(
                    effective_weight
                )


                gated_predictions.append(
                    gated_prediction
                )


                recurrent_states.append(
                    updated_state
                )


                previous_state = updated_state


        candidate_matrix = torch.cat(
            candidates,
            dim=
                0,
        )


        effective_weight_matrix = torch.stack(
            effective_weights,
            dim=
                0,
        )


        gated_prediction_matrix = torch.cat(
            gated_predictions,
            dim=
                0,
        )


        recurrent_state_matrix = torch.cat(
            recurrent_states,
            dim=
                0,
        )


        target = (
            BLOCK_10_TRANSFORMATIVE_TARGETS[
                pathway_name
            ]
        )


        objective_mask = (
            BLOCK_10_OBJECTIVE_MASKS[
                pathway_name
            ]
        )


        squared_error = (
            gated_prediction_matrix
            -
            target
        ).pow(
            2
        )


        masked_error = squared_error[
            objective_mask
        ]


        if masked_error.numel() == 0:

            raise RuntimeError(
                f"No objective-active elements for {pathway_name}."
            )


        pathway_loss = masked_error.mean()


        pathway_outputs[
            pathway_name
        ] = {
            "candidate_matrix":
                candidate_matrix,

            "effective_weight_matrix":
                effective_weight_matrix,

            "gated_prediction_matrix":
                gated_prediction_matrix,

            "recurrent_state_matrix":
                recurrent_state_matrix,

            "target_matrix":
                target,

            "target_mask":
                BLOCK_10_TRANSFORMATIVE_TARGET_MASKS[
                    pathway_name
                ],

            "objective_mask":
                objective_mask,

            "pathway_loss":
                pathway_loss,
        }


    total_objective = torch.stack(
        [
            pathway_outputs[
                pathway_name
            ][
                "pathway_loss"
            ]

            for pathway_name
            in BLOCK_10_OBJECTIVE_ACTIVE_PATHWAYS
        ],
        dim=
            0,
    ).mean()


    return (
        pathway_outputs,
        total_objective,
    )


# =============================================================================
# Evaluate pre-training objective
# =============================================================================

(
    BLOCK_10_OBJECTIVE_FORWARD,
    BLOCK_10_PRETRAIN_TOTAL_OBJECTIVE,
) = block_10_forward_tw_objective()


NOTEBOOK_09_BLOCK_10_RECURRENT_OBJECTIVE_FORWARD_EXECUTED = True

NOTEBOOK_09_BLOCK_10_LOSS_CALCULATED = True


# =============================================================================
# Objective numerical validation
# =============================================================================

BLOCK_10_PATHWAY_LOSSES = {
    pathway_name:
        float(
            BLOCK_10_OBJECTIVE_FORWARD[
                pathway_name
            ][
                "pathway_loss"
            ]
            .detach()
            .cpu()
            .item()
        )

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
}


BLOCK_10_TOTAL_OBJECTIVE_VALUE = float(
    BLOCK_10_PRETRAIN_TOTAL_OBJECTIVE
    .detach()
    .cpu()
    .item()
)


BLOCK_10_PATHWAY_LOSSES_FINITE = all(
    np.isfinite(
        pathway_loss
    )

    for pathway_loss
    in BLOCK_10_PATHWAY_LOSSES.values()
)


BLOCK_10_TOTAL_OBJECTIVE_FINITE = bool(
    np.isfinite(
        BLOCK_10_TOTAL_OBJECTIVE_VALUE
    )
)


if not all(
    [
        BLOCK_10_PATHWAY_LOSSES_FINITE,
        BLOCK_10_TOTAL_OBJECTIVE_FINITE,
    ]
):

    raise RuntimeError(
        "Transformative-Weight baseline objective contains "
        "non-finite values."
    )


# =============================================================================
# Recurrent objective shape validation
# =============================================================================

BLOCK_10_OBJECTIVE_FORWARD_SHAPES_VALID = all(
    all(
        [
            (
                tuple(
                    BLOCK_10_OBJECTIVE_FORWARD[
                        pathway_name
                    ][
                        "candidate_matrix"
                    ].shape
                )
                ==
                BLOCK_10_EXPECTED_TARGET_SHAPES[
                    pathway_name
                ]
            ),

            (
                tuple(
                    BLOCK_10_OBJECTIVE_FORWARD[
                        pathway_name
                    ][
                        "effective_weight_matrix"
                    ].shape
                )
                ==
                BLOCK_10_EXPECTED_TARGET_SHAPES[
                    pathway_name
                ]
            ),

            (
                tuple(
                    BLOCK_10_OBJECTIVE_FORWARD[
                        pathway_name
                    ][
                        "gated_prediction_matrix"
                    ].shape
                )
                ==
                BLOCK_10_EXPECTED_TARGET_SHAPES[
                    pathway_name
                ]
            ),

            (
                tuple(
                    BLOCK_10_OBJECTIVE_FORWARD[
                        pathway_name
                    ][
                        "recurrent_state_matrix"
                    ].shape
                )
                ==
                BLOCK_10_EXPECTED_TARGET_SHAPES[
                    pathway_name
                ]
            ),
        ]
    )

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
)


if not BLOCK_10_OBJECTIVE_FORWARD_SHAPES_VALID:

    raise RuntimeError(
        "Differentiable Transformative-Weight objective forward "
        "produced invalid shapes."
    )


# =============================================================================
# Objective-forward finiteness
# =============================================================================

BLOCK_10_OBJECTIVE_FORWARD_FINITE = all(
    bool(
        torch.isfinite(
            BLOCK_10_OBJECTIVE_FORWARD[
                pathway_name
            ][
                tensor_name
            ]
        ).all().item()
    )

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS

    for tensor_name
    in (
        "candidate_matrix",
        "effective_weight_matrix",
        "gated_prediction_matrix",
        "recurrent_state_matrix",
    )
)


if not BLOCK_10_OBJECTIVE_FORWARD_FINITE:

    raise RuntimeError(
        "Differentiable Transformative-Weight objective forward "
        "contains non-finite values."
    )


# =============================================================================
# Effective-weight bound validation
# =============================================================================

BLOCK_10_EFFECTIVE_WEIGHTS_BOUNDED = all(
    bool(
        (
            (
                BLOCK_10_OBJECTIVE_FORWARD[
                    pathway_name
                ][
                    "effective_weight_matrix"
                ]
                >=
                0.0
            )
            &
            (
                BLOCK_10_OBJECTIVE_FORWARD[
                    pathway_name
                ][
                    "effective_weight_matrix"
                ]
                <=
                1.0
            )
        ).all().item()
    )

    for pathway_name
    in BLOCK_10_PATHWAY_DIMENSIONS
)


if not BLOCK_10_EFFECTIVE_WEIGHTS_BOUNDED:

    raise RuntimeError(
        "Transformative-Weight objective forward produced "
        "out-of-bound effective weights."
    )


# =============================================================================
# Deferred-dimension gating validation
# =============================================================================

BLOCK_10_DEFERRED_GATING_VALID = {}


for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

    deferred_indices = torch.tensor(
        BLOCK_10_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
        device=
            DEVICE,
    )


    effective_weights = (
        BLOCK_10_OBJECTIVE_FORWARD[
            pathway_name
        ][
            "effective_weight_matrix"
        ]
    )


    gated_predictions = (
        BLOCK_10_OBJECTIVE_FORWARD[
            pathway_name
        ][
            "gated_prediction_matrix"
        ]
    )


    BLOCK_10_DEFERRED_GATING_VALID[
        pathway_name
    ] = all(
        [
            torch.equal(
                effective_weights[
                    :,
                    deferred_indices
                ],
                torch.zeros_like(
                    effective_weights[
                        :,
                        deferred_indices
                    ]
                ),
            ),

            torch.equal(
                gated_predictions[
                    :,
                    deferred_indices
                ],
                torch.zeros_like(
                    gated_predictions[
                        :,
                        deferred_indices
                    ]
                ),
            ),
        ]
    )


BLOCK_10_ALL_DEFERRED_GATING_VALID = all(
    BLOCK_10_DEFERRED_GATING_VALID.values()
)


if not BLOCK_10_ALL_DEFERRED_GATING_VALID:

    raise RuntimeError(
        "Deferred Transformative-Weight dimensions contributed "
        "to the objective forward path."
    )


# =============================================================================
# Target-leakage / causal contract
# =============================================================================

NOTEBOOK_09_BLOCK_10_SUPERVISION_USED_AS_RECURRENT_INPUT = False

NOTEBOOK_09_BLOCK_10_FUTURE_CONTEXT_USED = False

NOTEBOOK_09_BLOCK_10_REPRESENTATION_FORWARD_EXECUTED = False


BLOCK_10_CAUSAL_OBJECTIVE_CONTRACT_VALID = all(
    [
        (
            NOTEBOOK_09_BLOCK_10_SUPERVISION_USED_AS_RECURRENT_INPUT
            is False
        ),

        (
            NOTEBOOK_09_BLOCK_10_FUTURE_CONTEXT_USED
            is False
        ),

        (
            NOTEBOOK_09_BLOCK_10_REPRESENTATION_FORWARD_EXECUTED
            is False
        ),

        BLOCK_10_ARTICLE_INPUT_IDENTITIES_VALID,
    ]
)


if not BLOCK_10_CAUSAL_OBJECTIVE_CONTRACT_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 causal objective contract is invalid."
    )


# =============================================================================
# Future optimiser parameter scope
# =============================================================================

BLOCK_10_FUTURE_OPTIMISER_PARAMETERS = [
    parameter

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()

    if parameter.requires_grad
]


BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT = sum(
    parameter.numel()

    for parameter
    in BLOCK_10_FUTURE_OPTIMISER_PARAMETERS
)


BLOCK_10_FUTURE_OPTIMISER_PARAMETER_SCOPE_VALID = all(
    [
        (
            BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),
    ]
)


if not BLOCK_10_FUTURE_OPTIMISER_PARAMETER_SCOPE_VALID:

    raise RuntimeError(
        "Future Transformative-Weight optimiser parameter scope "
        "is invalid."
    )


# =============================================================================
# Verify inherited parameters remain frozen
# =============================================================================

BLOCK_10_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_10_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_10_WEIGHT_PARAMETERS_STILL_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


# =============================================================================
# Explicit optimisation boundary
# =============================================================================

NOTEBOOK_09_BLOCK_10_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_10_ZERO_GRAD_EXECUTED = False

NOTEBOOK_09_BLOCK_10_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_10_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_09_BLOCK_10_OPTIMIZER_STEP_EXECUTED = False

NOTEBOOK_09_BLOCK_10_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_10_TRAINING_EXECUTED = False


# =============================================================================
# Gradient-state validation
# =============================================================================

BLOCK_10_ALL_GRADIENTS_ABSENT = all(
    parameter.grad is None

    for module_collection
    in (
        BLOCK_1_REPRESENTATION_MODULES,
        BLOCK_1_TRANSFORMATIVE_MODULES,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES,
    )

    for module
    in module_collection.values()

    for parameter
    in module.parameters()
)


if not BLOCK_10_ALL_GRADIENTS_ABSENT:

    raise RuntimeError(
        "Unexpected gradients were created during Block 10 "
        "objective construction."
    )


# =============================================================================
# Snapshot learned state after objective construction
# =============================================================================

BLOCK_10_REPRESENTATION_STATE_AFTER = (
    block_10_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_10_TRANSFORMATIVE_STATE_AFTER = (
    block_10_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_10_WEIGHT_STATE_AFTER = (
    block_10_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Exact parameter immutability
# =============================================================================

BLOCK_10_REPRESENTATION_UNCHANGED = (
    block_10_nested_state_exact(
        BLOCK_10_REPRESENTATION_STATE_BEFORE,
        BLOCK_10_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_10_TRANSFORMATIVE_UNCHANGED = (
    block_10_nested_state_exact(
        BLOCK_10_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_10_TRANSFORMATIVE_STATE_AFTER,
    )
)


BLOCK_10_WEIGHT_STATE_UNCHANGED = (
    block_10_nested_state_exact(
        BLOCK_10_WEIGHT_STATE_BEFORE,
        BLOCK_10_WEIGHT_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_10_REPRESENTATION_UNCHANGED,
        BLOCK_10_TRANSFORMATIVE_UNCHANGED,
        BLOCK_10_WEIGHT_STATE_UNCHANGED,
    ]
):

    raise RuntimeError(
        "Model parameter state changed during Transformative-Weight "
        "objective construction."
    )


# =============================================================================
# Parameter accounting
# =============================================================================

BLOCK_10_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_10_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 parameter accounting is invalid."
    )


# =============================================================================
# Expose validated TW learning contract
# =============================================================================

NOTEBOOK_09_TW_TRANSFORMATIVE_TARGETS = {
    pathway_name:
        tensor.detach()
        .cpu()
        .clone()

    for (
        pathway_name,
        tensor,
    ) in BLOCK_10_TRANSFORMATIVE_TARGETS.items()
}


NOTEBOOK_09_TW_TRANSFORMATIVE_TARGET_MASKS = {
    pathway_name:
        tensor.detach()
        .cpu()
        .clone()

    for (
        pathway_name,
        tensor,
    ) in BLOCK_10_TRANSFORMATIVE_TARGET_MASKS.items()
}


NOTEBOOK_09_TW_OBJECTIVE_MASKS = {
    pathway_name:
        tensor.detach()
        .cpu()
        .clone()

    for (
        pathway_name,
        tensor,
    ) in BLOCK_10_OBJECTIVE_MASKS.items()
}


NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES = deepcopy(
    BLOCK_10_PATHWAY_LOSSES
)


NOTEBOOK_09_TW_PRETRAIN_TOTAL_OBJECTIVE = (
    BLOCK_10_TOTAL_OBJECTIVE_VALUE
)


NOTEBOOK_09_TW_FUTURE_OPTIMISER_PARAMETER_COUNT = (
    BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT
)


NOTEBOOK_09_TW_SOCIAL_DIMENSION_NAMES = tuple(
    BLOCK_10_SOCIAL_DIMENSION_NAMES
)


# =============================================================================
# Final Block 10 validation
# =============================================================================

NOTEBOOK_09_BLOCK_10_ERRORS = []


BLOCK_10_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_10_PREREQUISITES_VALID,

    "pathway_dimensions_invalid":
        BLOCK_10_PATHWAY_DIMENSIONS_VALID,

    "dimension_partitions_invalid":
        BLOCK_10_ALL_DIMENSION_PARTITIONS_VALID,

    "current_target_shapes_invalid":
        BLOCK_10_CURRENT_TARGET_SHAPES_VALID,

    "current_mask_shapes_invalid":
        BLOCK_10_CURRENT_MASK_SHAPES_VALID,

    "current_available_values_not_finite":
        BLOCK_10_CURRENT_AVAILABLE_VALUES_FINITE,

    "transformative_target_shapes_invalid":
        BLOCK_10_TRANSFORMATIVE_TARGET_SHAPES_VALID,

    "available_targets_not_finite":
        BLOCK_10_AVAILABLE_TARGETS_FINITE,

    "one_or_more_pathways_objective_inactive":
        BLOCK_10_ALL_PATHWAYS_OBJECTIVE_ACTIVE,

    "article_input_identity_invalid":
        BLOCK_10_ARTICLE_INPUT_IDENTITIES_VALID,

    "preceding_supervision_boundary_invalid":
        BLOCK_10_PRECEDING_SUPERVISION_BOUNDARIES_VALID,

    "objective_forward_shapes_invalid":
        BLOCK_10_OBJECTIVE_FORWARD_SHAPES_VALID,

    "objective_forward_not_finite":
        BLOCK_10_OBJECTIVE_FORWARD_FINITE,

    "effective_weights_not_bounded":
        BLOCK_10_EFFECTIVE_WEIGHTS_BOUNDED,

    "deferred_gating_invalid":
        BLOCK_10_ALL_DEFERRED_GATING_VALID,

    "pathway_losses_not_finite":
        BLOCK_10_PATHWAY_LOSSES_FINITE,

    "total_objective_not_finite":
        BLOCK_10_TOTAL_OBJECTIVE_FINITE,

    "causal_objective_contract_invalid":
        BLOCK_10_CAUSAL_OBJECTIVE_CONTRACT_VALID,

    "future_optimizer_scope_invalid":
        BLOCK_10_FUTURE_OPTIMISER_PARAMETER_SCOPE_VALID,

    "representation_not_frozen":
        BLOCK_10_REPRESENTATION_STILL_FROZEN,

    "transformative_not_frozen":
        BLOCK_10_TRANSFORMATIVE_STILL_FROZEN,

    "weight_parameters_not_trainable":
        BLOCK_10_WEIGHT_PARAMETERS_STILL_TRAINABLE,

    "unexpected_gradients":
        BLOCK_10_ALL_GRADIENTS_ABSENT,

    "representation_state_changed":
        BLOCK_10_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_10_TRANSFORMATIVE_UNCHANGED,

    "weight_state_changed":
        BLOCK_10_WEIGHT_STATE_UNCHANGED,

    "parameter_accounting_invalid":
        BLOCK_10_PARAMETER_ACCOUNTING_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_10_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_10_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation audit
# =============================================================================

BLOCK_10_PROHIBITED_OPERATIONS = {
    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_10_OPTIMIZER_CREATED,

    "zero_grad_incorrectly_executed":
        NOTEBOOK_09_BLOCK_10_ZERO_GRAD_EXECUTED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_10_BACKWARD_PASS_EXECUTED,

    "gradient_clipping_incorrectly_executed":
        NOTEBOOK_09_BLOCK_10_GRADIENT_CLIPPING_EXECUTED,

    "optimizer_step_incorrectly_executed":
        NOTEBOOK_09_BLOCK_10_OPTIMIZER_STEP_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_10_PARAMETER_UPDATE_EXECUTED,

    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_10_TRAINING_EXECUTED,

    "future_context_incorrectly_used":
        NOTEBOOK_09_BLOCK_10_FUTURE_CONTEXT_USED,

    "supervision_used_as_recurrent_input":
        NOTEBOOK_09_BLOCK_10_SUPERVISION_USED_AS_RECURRENT_INPUT,

    "representation_forward_incorrectly_executed":
        NOTEBOOK_09_BLOCK_10_REPRESENTATION_FORWARD_EXECUTED,
}


for (
    error_name,
    operation_executed,
) in BLOCK_10_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_10_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 10 state
# =============================================================================

NOTEBOOK_09_TW_OBJECTIVE_CONTRACT_VALID = (
    len(
        NOTEBOOK_09_BLOCK_10_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_TW_TRAINING_READY = (
    NOTEBOOK_09_TW_OBJECTIVE_CONTRACT_VALID
)


NOTEBOOK_09_BLOCK_10_VALID = (
    NOTEBOOK_09_TW_OBJECTIVE_CONTRACT_VALID
)


if not NOTEBOOK_09_BLOCK_10_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 Transformative-Weight objective "
        "construction failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_10_ERRORS}"
    )


NOTEBOOK_09_BLOCK_10_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_10_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_10,

    "block_name":
        NOTEBOOK_09_BLOCK_10_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_10_VERSION,

    "factual_source":
        BLOCK_10_FACTUAL_FILENAME,

    "factual_drive_file_id":
        BLOCK_10_FACTUAL_FILE_INFO[
            "id"
        ],

    "psychological_source":
        "Notebook 03 accepted supervision",

    "psychological_drive_file_id":
        BLOCK_10_PSYCHOLOGICAL_DRIVE_FILE_ID,

    "social_source":
        BLOCK_10_SOCIAL_FILENAME,

    "social_drive_file_id":
        BLOCK_10_SOCIAL_FILE_INFO[
            "id"
        ],

    "social_dimension_names":
        tuple(
            BLOCK_10_SOCIAL_DIMENSION_NAMES
        ),

    "article_ids":
        tuple(
            BLOCK_10_ARTICLE_IDS
        ),

    "article_count":
        BLOCK_10_ARTICLE_COUNT,

    "sentence_count":
        BLOCK_10_SENTENCE_COUNT,

    "article_start_indices":
        tuple(
            BLOCK_10_ARTICLE_START_INDICES
        ),

    "preceding_supervision_boundaries_valid":
        BLOCK_10_PRECEDING_SUPERVISION_BOUNDARIES_VALID,

    "current_available_elements":
        deepcopy(
            BLOCK_10_CURRENT_AVAILABLE_ELEMENTS
        ),

    "transformative_available_elements":
        deepcopy(
            BLOCK_10_TRANSFORMATIVE_AVAILABLE_ELEMENTS
        ),

    "objective_active_elements":
        deepcopy(
            BLOCK_10_OBJECTIVE_ACTIVE_ELEMENTS
        ),

    "objective_active_pathways":
        tuple(
            BLOCK_10_OBJECTIVE_ACTIVE_PATHWAYS
        ),

    "pathway_losses":
        deepcopy(
            BLOCK_10_PATHWAY_LOSSES
        ),

    "pretrain_total_objective":
        BLOCK_10_TOTAL_OBJECTIVE_VALUE,

    "future_optimizer_parameter_count":
        BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT,

    "optimizer_policy":
        deepcopy(
            NOTEBOOK_09_TW_OPTIMISER_POLICY
        ),

    "objective_contract_valid":
        NOTEBOOK_09_TW_OBJECTIVE_CONTRACT_VALID,

    "tw_training_ready":
        NOTEBOOK_09_TW_TRAINING_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_10_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_10_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 10: "
    "Transformative-Weight Objective Construction, "
    "Supervision Alignment and Optimisation Contract"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_10_VERSION}"
)

print("-" * 72)

print(
    "Recovered supervision"
)

print(
    f"Factual source               : "
    f"{BLOCK_10_FACTUAL_FILENAME}"
)

print(
    f"Factual Drive file ID        : "
    f"{BLOCK_10_FACTUAL_FILE_INFO['id']}"
)

print(
    f"Psychological source         : "
    "Notebook 03 accepted supervision"
)

print(
    f"Psychological Drive file ID  : "
    f"{BLOCK_10_PSYCHOLOGICAL_DRIVE_FILE_ID}"
)

print(
    f"Social source                : "
    f"{BLOCK_10_SOCIAL_FILENAME}"
)

print(
    f"Social Drive file ID         : "
    f"{BLOCK_10_SOCIAL_FILE_INFO['id']}"
)

print(
    f"Social dimensions            : "
    f"{list(BLOCK_10_SOCIAL_DIMENSION_NAMES)}"
)

print("-" * 72)

print(
    "Current-state supervision"
)

for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

    possible_elements = (
        BLOCK_10_SENTENCE_COUNT
        *
        BLOCK_10_PATHWAY_DIMENSIONS[
            pathway_name
        ]
    )


    print(
        f"{pathway_name:<14} shape              : "
        f"{tuple(BLOCK_10_CURRENT_SUPERVISION[pathway_name].shape)}"
    )

    print(
        f"{pathway_name:<14} available          : "
        f"{BLOCK_10_CURRENT_AVAILABLE_ELEMENTS[pathway_name]} "
        f"/ {possible_elements}"
    )

print("-" * 72)

print(
    "Derived transformation supervision"
)

for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

    possible_elements = (
        BLOCK_10_SENTENCE_COUNT
        *
        BLOCK_10_PATHWAY_DIMENSIONS[
            pathway_name
        ]
    )


    print(
        f"{pathway_name:<14} available          : "
        f"{BLOCK_10_TRANSFORMATIVE_AVAILABLE_ELEMENTS[pathway_name]} "
        f"/ {possible_elements}"
    )

    print(
        f"{pathway_name:<14} objective active   : "
        f"{BLOCK_10_OBJECTIVE_ACTIVE_ELEMENTS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} active dimensions  : "
        f"{list(BLOCK_10_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} deferred dimensions: "
        f"{list(BLOCK_10_DEFERRED_DIMENSION_INDICES[pathway_name])}"
    )

print("-" * 72)

print(
    "Pre-training Transformative-Weight objective"
)

for pathway_name in BLOCK_10_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} masked MSE         : "
        f"{BLOCK_10_PATHWAY_LOSSES[pathway_name]:.8f}"
    )

print(
    f"Total objective              : "
    f"{BLOCK_10_TOTAL_OBJECTIVE_VALUE:.8f}"
)

print(
    f"All pathways active          : "
    f"{BLOCK_10_ALL_PATHWAYS_OBJECTIVE_ACTIVE}"
)

print(
    f"Objective finite             : "
    f"{BLOCK_10_TOTAL_OBJECTIVE_FINITE}"
)

print("-" * 72)

print(
    "Future optimisation policy"
)

print(
    f"Optimizer                    : "
    f"{NOTEBOOK_09_TW_OPTIMISER_POLICY['optimizer']}"
)

print(
    f"Learning rate                : "
    f"{NOTEBOOK_09_TW_OPTIMISER_POLICY['learning_rate']}"
)

print(
    f"Weight decay                 : "
    f"{NOTEBOOK_09_TW_OPTIMISER_POLICY['weight_decay']}"
)

print(
    f"Maximum gradient norm        : "
    f"{NOTEBOOK_09_TW_OPTIMISER_POLICY['maximum_gradient_norm']}"
)

print(
    f"Optimizer parameter count    : "
    f"{BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT}"
)

print(
    f"Sequence order preserved     : "
    f"{NOTEBOOK_09_TW_OPTIMISER_POLICY['sequence_order_preserved']}"
)

print(
    f"Batch shuffling              : "
    f"{NOTEBOOK_09_TW_OPTIMISER_POLICY['batch_shuffling']}"
)

print(
    f"Future context allowed       : "
    f"{NOTEBOOK_09_TW_OPTIMISER_POLICY['future_context_allowed']}"
)

print("-" * 72)

print(
    "Parameter and execution validation"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Transformative Weight params : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Total model parameters       : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print(
    f"Representation unchanged     : "
    f"{BLOCK_10_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged     : "
    f"{BLOCK_10_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Weight state unchanged       : "
    f"{BLOCK_10_WEIGHT_STATE_UNCHANGED}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_10_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_10_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"Weight parameters trainable  : "
    f"{BLOCK_10_WEIGHT_PARAMETERS_STILL_TRAINABLE}"
)

print(
    f"Deferred gating valid        : "
    f"{BLOCK_10_ALL_DEFERRED_GATING_VALID}"
)

print(
    f"All gradients absent         : "
    f"{BLOCK_10_ALL_GRADIENTS_ABSENT}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Recurrent objective forward  : "
    f"{NOTEBOOK_09_BLOCK_10_RECURRENT_OBJECTIVE_FORWARD_EXECUTED}"
)

print(
    f"Representation forward       : "
    f"{NOTEBOOK_09_BLOCK_10_REPRESENTATION_FORWARD_EXECUTED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_BLOCK_10_LOSS_CALCULATED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_10_OPTIMIZER_CREATED}"
)

print(
    f"Zero grad executed           : "
    f"{NOTEBOOK_09_BLOCK_10_ZERO_GRAD_EXECUTED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_10_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Gradient clipping executed   : "
    f"{NOTEBOOK_09_BLOCK_10_GRADIENT_CLIPPING_EXECUTED}"
)

print(
    f"Optimizer step executed      : "
    f"{NOTEBOOK_09_BLOCK_10_OPTIMIZER_STEP_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_10_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Future context used          : "
    f"{NOTEBOOK_09_BLOCK_10_FUTURE_CONTEXT_USED}"
)

print(
    f"Supervision as model input   : "
    f"{NOTEBOOK_09_BLOCK_10_SUPERVISION_USED_AS_RECURRENT_INPUT}"
)

print("-" * 72)

print(
    f"Objective contract valid     : "
    f"{NOTEBOOK_09_TW_OBJECTIVE_CONTRACT_VALID}"
)

print(
    f"TW training ready            : "
    f"{NOTEBOOK_09_TW_TRAINING_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_10_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_10_COMPLETE}"
)

print("=" * 72)

print(
    f"Validated supervision was aligned to the canonical corpus of "
    f"{BLOCK_10_SENTENCE_COUNT} sentence(s) across "
    f"{BLOCK_10_ARTICLE_COUNT} article(s)."
)

print(
    "Notebook 07 social supervision was reconstructed directly from its "
    "persisted architecture.dimension_order and social_annotations "
    "availability/value contract."
)

print(
    "Signed transformation targets were reconstructed as current supervised "
    "state minus the immediately preceding supervised state within each "
    "article, with a deterministic zero initial condition at every "
    "article boundary."
)

print(
    "Missing supervision remains masked and distinct from genuine "
    "zero-valued transformation."
)

print(
    "Only dimensions that are both supervision-valid and "
    "activation-eligible contribute to the Transformative-Weight objective."
)

print(
    "The differentiable recurrent objective forward preserved each "
    "within-article state chain without detaching recurrent state between "
    "sentences and reset state at every article boundary."
)

print(
    "The gated candidate transformation is compared directly with the "
    "validated transformation target; supervision is never supplied as "
    "a recurrent model input."
)

print(
    "The pre-training factual, psychological and social pathway losses "
    "and the equal-pathway total objective were calculated successfully."
)

print(
    f"Only the {BLOCK_10_FUTURE_OPTIMISER_PARAMETER_COUNT} active latent "
    "Transformative-Weight parameter(s) are eligible for the future optimiser."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} representation "
    "and transformative parameters remain frozen and exactly unchanged, "
    "and all Transformative-Weight parameters remain unchanged."
)

print(
    "No optimiser, backward pass, gradient clipping, optimiser step "
    "or parameter update was executed."
)

print(
    "Notebook 09 may now proceed to controlled Transformative-Weight "
    "training and parameter-update validation."
)

print("=" * 72)

Media AI — Notebook 09, Block 10: Transformative-Weight Objective Construction, Supervision Alignment and Optimisation Contract
Block version                : 1.2
------------------------------------------------------------------------
Recovered supervision
Factual source               : notebook_06_factual_target_contract.json
Factual Drive file ID        : 1FdTrV_2RoHDRn06JLSx367OD4HffDaAy
Psychological source         : Notebook 03 accepted supervision
Psychological Drive file ID  : 1gYiCR-i_avGCwE5O9p921es8Mzbzhvyl
Social source                : notebook_07_accepted_social_supervision.json
Social Drive file ID         : 1RMofcYIqIFKH9FqSrAz5hjbicfSNu5ex
Social dimensions            : ['interpersonal_cohesion', 'economic_cohesion', 'political_cohesion', 'religious_cohesion', 'group_collective_cohesion', 'institutional_role_relation', 'public_audience_relation']
------------------------------------------------------------------------
Current-state supervision
factual        shape     

## Block 11 — Controlled Transformative-Weight Training and Parameter-Update Validation

### Purpose

This block performs the first controlled optimisation of the **Transformative Weight** parameters within the validated recurrent Media AI architecture.

The objective is not to establish final model performance.

Instead, the block tests whether the 25 activation-eligible Transformative-Weight parameters can be learned through the complete causal article-level recurrent trajectory while preserving the frozen representation and transformative mechanisms inherited from the preceding notebooks.

The block therefore provides an architectural learning validation before the Media AI model is transferred to a substantially larger corpus for detailed training and evaluation.

### Architectural Boundary

The inherited representation and transformative architecture remains fixed throughout this block.

The trainable parameter scope is restricted to the 25 latent Transformative-Weight parameters established in Blocks 2–4:

- **Factual:** 5 trainable dimensions;
- **Psychological:** 17 trainable dimensions;
- **Social:** 3 trainable dimensions.

The remaining 26 structural Transformative-Weight dimensions remain deferred and fixed at an effective weight of zero.

The inherited parameter sets remain:

- **Representation parameters:** 894,003;
- **Transformative parameters:** 8,187;
- **Inherited frozen parameters:** 902,190;
- **Trainable Transformative-Weight parameters:** 25;
- **Complete instantiated architecture:** 902,215 parameters.

No representation or transformative parameter is eligible for optimisation.

### Controlled Training Setting

Training is performed on the canonical 16-sentence article already validated in Blocks 8–10.

This article is used as a controlled architectural training sequence rather than as a statistically sufficient final training corpus.

Every optimisation epoch therefore begins from fresh deterministic zero recurrent states,

$$h_{0}^{(k)}=\mathbf{0},$$

where \(k\) denotes the factual, psychological or social pathway.

The canonical sentence order is preserved throughout training.

No batch shuffling, future-context access or cross-article recurrent-state propagation is permitted.

### Recurrent Training Path

For pathway \(k\) and sentence position \(t\), the frozen transformative mechanism receives the current representation and immediately preceding recurrent state,

$$\Delta_{t}^{(k)}=T^{(k)}\left(r_{t}^{(k)},h_{t-1}^{(k)}\right).$$

The trainable Transformative Weight is applied dimension-wise,

$$\widetilde{\Delta}_{t}^{(k)}=w^{(k)}\odot\Delta_{t}^{(k)},$$

and the recurrent state is updated additively,

$$h_{t}^{(k)}=h_{t-1}^{(k)}+\widetilde{\Delta}_{t}^{(k)}.$$

The recurrent state remains differentiably connected across the complete article trajectory.

It is not detached between sentence positions.

### Transformative-Weight Parameterisation

Each activation-eligible Transformative Weight remains represented by a latent parameter \(\alpha\), with the bounded effective weight

$$w=\sigma(\alpha).$$

The effective weights therefore remain constrained to

$$0<w<1.$$

The active latent parameters begin from the validated initial value

$$\alpha=0,$$

corresponding to

$$w=0.5.$$

Deferred dimensions remain structurally represented but fixed at

$$w=0.$$

They do not receive gradients and cannot contribute to parameter updates.

### Supervised Transformation Objective

The transformation supervision constructed and validated in Block 10 is retained without modification.

For each pathway, the supervised transformation target is

$$\Delta_{t}^{*(k)}=y_{t}^{(k)}-y_{t-1}^{(k)},$$

with the deterministic zero initial condition used for the first sentence.

Only elements satisfying both conditions are included in the optimisation objective:

1. valid transformation supervision is available;
2. the corresponding Transformative-Weight dimension is activation-eligible.

The resulting objective-active supervision established in Block 10 contains:

- **Factual:** 80 elements;
- **Psychological:** 260 elements;
- **Social:** 36 elements.

Missing supervision remains masked and is never interpreted as a genuine zero-valued transformation.

### Pathway Losses

For each pathway \(k\), the masked mean-squared transformation error is

$$
\mathcal{L}_{k}
=
\frac{1}{|\Omega_k|}
\sum_{(t,j)\in\Omega_k}
\left(
\widetilde{\Delta}_{t,j}^{(k)}
-
\Delta_{t,j}^{*(k)}
\right)^2,
$$

where \(\Omega_k\) denotes the set of supervision-valid, activation-eligible sentence–dimension pairs.

The complete Transformative-Weight objective retains the equal-pathway aggregation established in Block 10,

$$
\mathcal{L}_{TW}
=
\frac{
\mathcal{L}_{F}
+
\mathcal{L}_{P}
+
\mathcal{L}_{S}
}{3}.
$$

Before optimisation, the validated objective is

$$
\mathcal{L}_{TW}^{(0)}
=
0.28348443.
$$

The corresponding pathway losses are:

- factual: \(0.27470103\);
- psychological: \(0.10133852\);
- social: \(0.47441372\).

These values form the immutable pre-training reference against which the controlled optimisation run is evaluated.

### Optimisation Policy

The controlled pilot optimisation uses the policy established in Block 10:

- **Optimiser:** AdamW;
- **Learning rate:** \(0.001\);
- **Weight decay:** \(0.0001\);
- **Maximum gradient norm:** \(1.0\);
- **Training epochs:** 25;
- **Sequence shuffling:** disabled;
- **Future context:** prohibited.

The optimiser receives only the 25 activation-eligible latent Transformative-Weight parameters.

Each epoch executes the complete canonical article trajectory from fresh zero recurrent states, evaluates the masked pathway objectives, performs backpropagation through the recurrent trajectory, applies gradient clipping, and updates only the permitted Transformative-Weight parameters.

### Training Diagnostics

The block records the complete controlled optimisation trajectory.

For every epoch, it retains:

- total Transformative-Weight objective;
- factual pathway loss;
- psychological pathway loss;
- social pathway loss;
- gradient norm;
- effective Transformative-Weight state.

The block additionally compares the initial and final latent and effective weights for every activation-eligible dimension.

This permits direct inspection of which factual, psychological and social transformation dimensions respond to the controlled supervision.

### Parameter-Update Validation

After training, the block verifies that parameter updates occurred exclusively within the intended Transformative-Weight scope.

The validation requires:

- at least one activation-eligible Transformative-Weight parameter to change;
- all learned effective weights to remain finite and bounded;
- deferred Transformative-Weight dimensions to remain exactly zero;
- all 894,003 representation parameters to remain exactly unchanged;
- all 8,187 inherited transformative parameters to remain exactly unchanged;
- the complete 902,190-parameter inherited architecture to remain frozen;
- no unintended trainable parameter to enter the optimiser.

The block therefore distinguishes successful learning from accidental modification of the inherited architecture.

### Post-Training Objective Validation

Following the final parameter update, the complete canonical article trajectory is evaluated again from fresh deterministic zero recurrent states.

The resulting post-training pathway losses and total Transformative-Weight objective are compared directly with the immutable Block 10 baseline.

The controlled run is expected to demonstrate that the Transformative-Weight parameters are capable of responding to the supervision signal.

Because the experiment contains only one 16-sentence article, improvement in this objective is interpreted solely as evidence of **architectural learnability**.

It is not interpreted as evidence of generalisation or final Media AI performance.

### Scientific Scope

This block intentionally separates **architecture validation** from **final model training**.

The controlled article is sufficient to test:

- differentiability of the recurrent Transformative-Weight pathway;
- optimisation of the intended 25 parameters;
- preservation of bounded gating;
- recurrent gradient propagation;
- strict parameter freezing;
- causal sequence execution;
- numerical stability;
- objective responsiveness.

It is not sufficient to estimate generalisation performance, determine final Transformative-Weight values or select a production model.

Those questions require subsequent training on a substantially larger multi-article corpus with explicit training, validation and held-out test partitions.

### Execution Boundary

This block may:

- instantiate the validated AdamW optimiser;
- execute repeated complete article-level recurrent forward paths;
- calculate the validated Transformative-Weight objective;
- execute backward propagation through the recurrent trajectory;
- calculate and clip Transformative-Weight gradients;
- update the 25 activation-eligible latent Transformative-Weight parameters;
- evaluate the resulting post-training objective.

This block must not:

- update representation parameters;
- update inherited transformative parameters;
- activate deferred Transformative-Weight dimensions;
- alter the supervision contract;
- shuffle the canonical article sequence;
- propagate state across article boundaries;
- access future sentence representations;
- interpret pilot optimisation results as final model performance.

### Expected Outcome

Successful completion of this block establishes that the Transformative-Weight mechanism is not only structurally valid but **operationally learnable through the complete recurrent Media AI pathway**.

The block should produce a finite controlled training trajectory, bounded learned Transformative Weights, measurable parameter changes within the permitted 25-parameter scope, and exact preservation of the complete inherited 902,190-parameter architecture.

Notebook 09 may then proceed to post-training recurrent validation, architectural completion and persistent handover before the Media AI architecture is transferred to the expanded-corpus training phase.

In [61]:
# =============================================================================
# Media AI — Notebook 09
# Block 11: Controlled Transformative-Weight Training
#           and Parameter-Update Validation
# =============================================================================

from copy import deepcopy

import numpy as np
import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_11 = 11

NOTEBOOK_09_BLOCK_11_NAME = (
    "Controlled Transformative-Weight Training "
    "and Parameter-Update Validation"
)

NOTEBOOK_09_BLOCK_11_VERSION = "1.1"


# =============================================================================
# Controlled architectural-training policy
# =============================================================================

BLOCK_11_EPOCHS = 25

BLOCK_11_LEARNING_RATE = float(
    NOTEBOOK_09_TW_OPTIMISER_POLICY[
        "learning_rate"
    ]
)

BLOCK_11_WEIGHT_DECAY = float(
    NOTEBOOK_09_TW_OPTIMISER_POLICY[
        "weight_decay"
    ]
)

BLOCK_11_MAXIMUM_GRADIENT_NORM = float(
    NOTEBOOK_09_TW_OPTIMISER_POLICY[
        "maximum_gradient_norm"
    ]
)


# =============================================================================
# Required runtime contract
# =============================================================================

BLOCK_11_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Block 1
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",

    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",

    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Transformative Weight
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT",
    "BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS",

    # -------------------------------------------------------------------------
    # Recurrent / article contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS",
    "NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES",

    "NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS",
    "NOTEBOOK_09_CANONICAL_ARTICLE_IDS",
    "NOTEBOOK_09_CANONICAL_SENTENCE_IDS",
    "NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS",

    # -------------------------------------------------------------------------
    # Block 10 objective contract
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_10_COMPLETE",
    "NOTEBOOK_09_BLOCK_10_VALID",

    "NOTEBOOK_09_TW_OBJECTIVE_CONTRACT_VALID",
    "NOTEBOOK_09_TW_TRAINING_READY",

    "NOTEBOOK_09_TW_TRANSFORMATIVE_TARGETS",
    "NOTEBOOK_09_TW_TRANSFORMATIVE_TARGET_MASKS",
    "NOTEBOOK_09_TW_OBJECTIVE_MASKS",

    "NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES",
    "NOTEBOOK_09_TW_PRETRAIN_TOTAL_OBJECTIVE",

    "NOTEBOOK_09_TW_FUTURE_OPTIMISER_PARAMETER_COUNT",
    "NOTEBOOK_09_TW_OPTIMISER_POLICY",

    # -------------------------------------------------------------------------
    # Environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_11_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_11_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_11_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 11 prerequisites are not initialised. "
        f"Missing: {BLOCK_11_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_11_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,

        NOTEBOOK_09_BLOCK_10_COMPLETE is True,
        NOTEBOOK_09_BLOCK_10_VALID is True,

        NOTEBOOK_09_TW_OBJECTIVE_CONTRACT_VALID is True,
        NOTEBOOK_09_TW_TRAINING_READY is True,

        (
            NOTEBOOK_09_TW_FUTURE_OPTIMISER_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_11_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Block 10 must be valid and complete before "
        "controlled Transformative-Weight training."
    )


# =============================================================================
# Canonical pathway contract
# =============================================================================

BLOCK_11_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
)


BLOCK_11_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            BLOCK_11_PATHWAY_DIMENSIONS,
            dict,
        ),

        bool(
            BLOCK_11_PATHWAY_DIMENSIONS
        ),

        (
            set(
                BLOCK_11_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                BLOCK_1_TRANSFORMATIVE_MODULES.keys()
            )
            ==
            set(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys()
            )
        ),

        all(
            isinstance(
                pathway_dim,
                int,
            )
            and
            pathway_dim > 0

            for pathway_dim
            in BLOCK_11_PATHWAY_DIMENSIONS.values()
        ),
    ]
)


if not BLOCK_11_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 11 pathway dimensions are invalid."
    )


# =============================================================================
# Active / deferred pathway partitions
# =============================================================================

BLOCK_11_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES.items()
}


BLOCK_11_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES.items()
}


BLOCK_11_DIMENSION_PARTITIONS_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_11_PATHWAY_DIMENSIONS.items():

    active_set = set(
        BLOCK_11_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    deferred_set = set(
        BLOCK_11_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )


    BLOCK_11_DIMENSION_PARTITIONS_VALID[
        pathway_name
    ] = all(
        [
            active_set.isdisjoint(
                deferred_set
            ),

            (
                active_set
                |
                deferred_set
            )
            ==
            set(
                range(
                    pathway_dim
                )
            ),
        ]
    )


BLOCK_11_ALL_DIMENSION_PARTITIONS_VALID = all(
    BLOCK_11_DIMENSION_PARTITIONS_VALID.values()
)


if not BLOCK_11_ALL_DIMENSION_PARTITIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 11 active/deferred dimension "
        "partitions are invalid."
    )


# =============================================================================
# Canonical recurrent inputs and supervision
# =============================================================================

BLOCK_11_ARTICLE_IDS = tuple(
    NOTEBOOK_09_CANONICAL_ARTICLE_IDS
)


BLOCK_11_SENTENCE_COUNT = len(
    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
)


BLOCK_11_ARTICLE_COUNT = len(
    BLOCK_11_ARTICLE_IDS
)


BLOCK_11_INPUT_ARTICLE_IDS = [
    record[
        "article_id"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_11_OBSERVED_ARTICLE_IDS = tuple(
    dict.fromkeys(
        BLOCK_11_INPUT_ARTICLE_IDS
    )
)


BLOCK_11_CORPUS_CARDINALITY_VALID = all(
    [
        BLOCK_11_ARTICLE_COUNT > 0,
        BLOCK_11_SENTENCE_COUNT > 0,

        (
            BLOCK_11_SENTENCE_COUNT
            ==
            len(
                NOTEBOOK_09_CANONICAL_SENTENCE_IDS
            )
        ),

        (
            BLOCK_11_OBSERVED_ARTICLE_IDS
            ==
            BLOCK_11_ARTICLE_IDS
        ),
    ]
)


if not BLOCK_11_CORPUS_CARDINALITY_VALID:

    raise RuntimeError(
        "Notebook 09 Block 11 canonical corpus cardinality is invalid."
    )


BLOCK_11_ARTICLE_RECORD_INDICES = {
    article_id:
        []

    for article_id
    in BLOCK_11_ARTICLE_IDS
}


for (
    record_index,
    record,
) in enumerate(
    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
):

    article_id = record[
        "article_id"
    ]


    if article_id not in BLOCK_11_ARTICLE_RECORD_INDICES:

        raise RuntimeError(
            f"Unexpected Block 11 article_id: {article_id}"
        )


    BLOCK_11_ARTICLE_RECORD_INDICES[
        article_id
    ].append(
        record_index
    )


BLOCK_11_ARTICLE_BOUNDARIES_VALID = all(
    len(
        indices
    )
    >
    0

    for indices
    in BLOCK_11_ARTICLE_RECORD_INDICES.values()
)


if not BLOCK_11_ARTICLE_BOUNDARIES_VALID:

    raise RuntimeError(
        "Notebook 09 Block 11 article-boundary contract is invalid."
    )


BLOCK_11_REPRESENTATION_FIELDS = {
    pathway_name:
        f"{pathway_name}_representation"

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


BLOCK_11_TRANSFORMATIVE_MODULES = {
    pathway_name:
        BLOCK_1_TRANSFORMATIVE_MODULES[
            pathway_name
        ]

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


BLOCK_11_TARGETS = {
    pathway_name:
        NOTEBOOK_09_TW_TRANSFORMATIVE_TARGETS[
            pathway_name
        ].to(
            device=
                DEVICE,

            dtype=
                DEFAULT_DTYPE,
        )

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


BLOCK_11_TARGET_MASKS = {
    pathway_name:
        NOTEBOOK_09_TW_TRANSFORMATIVE_TARGET_MASKS[
            pathway_name
        ].to(
            device=
                DEVICE,

            dtype=
                torch.bool,
        )

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


BLOCK_11_OBJECTIVE_MASKS = {
    pathway_name:
        NOTEBOOK_09_TW_OBJECTIVE_MASKS[
            pathway_name
        ].to(
            device=
                DEVICE,

            dtype=
                torch.bool,
        )

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


# =============================================================================
# Validate target and objective-mask contract
# =============================================================================

BLOCK_11_EXPECTED_TARGET_SHAPES = {
    pathway_name:
        (
            BLOCK_11_SENTENCE_COUNT,
            pathway_dim,
        )

    for (
        pathway_name,
        pathway_dim,
    ) in BLOCK_11_PATHWAY_DIMENSIONS.items()
}


BLOCK_11_TARGET_SHAPES_VALID = all(
    tuple(
        BLOCK_11_TARGETS[
            pathway_name
        ].shape
    )
    ==
    BLOCK_11_EXPECTED_TARGET_SHAPES[
        pathway_name
    ]

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
)


BLOCK_11_TARGET_MASK_SHAPES_VALID = all(
    tuple(
        BLOCK_11_TARGET_MASKS[
            pathway_name
        ].shape
    )
    ==
    BLOCK_11_EXPECTED_TARGET_SHAPES[
        pathway_name
    ]

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
)


BLOCK_11_OBJECTIVE_MASK_SHAPES_VALID = all(
    tuple(
        BLOCK_11_OBJECTIVE_MASKS[
            pathway_name
        ].shape
    )
    ==
    BLOCK_11_EXPECTED_TARGET_SHAPES[
        pathway_name
    ]

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
)


BLOCK_11_OBJECTIVE_ACTIVE_ELEMENTS = {
    pathway_name:
        int(
            BLOCK_11_OBJECTIVE_MASKS[
                pathway_name
            ].sum().item()
        )

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


BLOCK_11_OBJECTIVE_ACTIVE_ELEMENTS_VALID = all(
    BLOCK_11_OBJECTIVE_ACTIVE_ELEMENTS[
        pathway_name
    ]
    >
    0

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
)


if not all(
    [
        BLOCK_11_TARGET_SHAPES_VALID,
        BLOCK_11_TARGET_MASK_SHAPES_VALID,
        BLOCK_11_OBJECTIVE_MASK_SHAPES_VALID,
        BLOCK_11_OBJECTIVE_ACTIVE_ELEMENTS_VALID,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 11 target or objective-mask contract "
        "does not match validated Block 10 supervision."
    )


# =============================================================================
# Canonical sentence / article identity validation
# =============================================================================

BLOCK_11_INPUT_SENTENCE_IDS = [
    record[
        "sentence_id"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_11_INPUT_SEQUENCE_POSITIONS = [
    record[
        "sequence_position"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_11_INPUT_ARTICLE_IDS = [
    record[
        "article_id"
    ]

    for record
    in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]


BLOCK_11_INPUT_IDENTITIES_VALID = all(
    [
        (
            BLOCK_11_INPUT_SENTENCE_IDS
            ==
            list(
                NOTEBOOK_09_CANONICAL_SENTENCE_IDS
            )
        ),

        (
            BLOCK_11_INPUT_SEQUENCE_POSITIONS
            ==
            list(
                NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS
            )
        ),

        (
            tuple(
                dict.fromkeys(
                    BLOCK_11_INPUT_ARTICLE_IDS
                )
            )
            ==
            BLOCK_11_ARTICLE_IDS
        ),
    ]
)


if not BLOCK_11_INPUT_IDENTITIES_VALID:

    raise RuntimeError(
        "Notebook 09 Block 11 recurrent input sequence does not "
        "preserve the canonical corpus identity contract."
    )


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_11_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_11_nested_state_exact(
    state_before,
    state_after,
):

    if (
        state_before.keys()
        !=
        state_after.keys()
    ):

        return False


    for module_name in state_before:

        if (
            state_before[
                module_name
            ].keys()
            !=
            state_after[
                module_name
            ].keys()
        ):

            return False


        for tensor_name in state_before[
            module_name
        ]:

            if not torch.equal(
                state_before[
                    module_name
                ][
                    tensor_name
                ],
                state_after[
                    module_name
                ][
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot learned states before controlled training
# =============================================================================

BLOCK_11_REPRESENTATION_STATE_BEFORE = (
    block_11_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_11_TRANSFORMATIVE_STATE_BEFORE = (
    block_11_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_11_WEIGHT_STATE_BEFORE = (
    block_11_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Initial effective Transformative-Weight vectors
# =============================================================================

with torch.no_grad():

    BLOCK_11_INITIAL_EFFECTIVE_WEIGHTS = {
        pathway_name:
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                pathway_name
            ]()
            .detach()
            .cpu()
            .clone()

        for pathway_name
        in (
            "factual",
            "psychological",
            "social",
        )
    }


# =============================================================================
# Initial latent Transformative-Weight parameter state
# =============================================================================

BLOCK_11_INITIAL_LATENT_PARAMETER_STATE = {
    pathway_name:
        {
            parameter_name:
                parameter.detach()
                .cpu()
                .clone()

            for (
                parameter_name,
                parameter,
            ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                pathway_name
            ].named_parameters()
        }

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


# =============================================================================
# Parameter-scope validation
# =============================================================================

BLOCK_11_OPTIMISER_PARAMETERS = [
    parameter

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS

    for parameter
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
        pathway_name
    ].parameters()

    if parameter.requires_grad
]


BLOCK_11_OPTIMISER_PARAMETER_COUNT = sum(
    parameter.numel()

    for parameter
    in BLOCK_11_OPTIMISER_PARAMETERS
)


BLOCK_11_OPTIMISER_PARAMETER_COUNT_VALID = all(
    [
        (
            BLOCK_11_OPTIMISER_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            BLOCK_11_OPTIMISER_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            BLOCK_11_OPTIMISER_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TW_FUTURE_OPTIMISER_PARAMETER_COUNT
        ),

        (
            BLOCK_11_OPTIMISER_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),
    ]
)


BLOCK_11_REPRESENTATION_FROZEN_BEFORE = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_11_TRANSFORMATIVE_FROZEN_BEFORE = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_11_WEIGHT_PARAMETERS_TRAINABLE_BEFORE = all(
    parameter.requires_grad

    for parameter
    in BLOCK_11_OPTIMISER_PARAMETERS
)


if not all(
    [
        BLOCK_11_OPTIMISER_PARAMETER_COUNT_VALID,
        BLOCK_11_REPRESENTATION_FROZEN_BEFORE,
        BLOCK_11_TRANSFORMATIVE_FROZEN_BEFORE,
        BLOCK_11_WEIGHT_PARAMETERS_TRAINABLE_BEFORE,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 11 parameter scope is invalid before training."
    )


# =============================================================================
# Differentiable recurrent training forward
# =============================================================================
#
# Recurrent state is not detached within an article.
# A fresh zero state is created at every article boundary, preventing
# cross-article state propagation while preserving within-article gradient flow.
# =============================================================================

def block_11_forward_objective():

    pathway_outputs = {}


    for pathway_name in BLOCK_11_PATHWAY_DIMENSIONS:

        pathway_dim = (
            BLOCK_11_PATHWAY_DIMENSIONS[
                pathway_name
            ]
        )


        candidates = []

        effective_weights = []

        gated_predictions = []

        recurrent_states = []


        for article_id in BLOCK_11_ARTICLE_IDS:

            previous_state = torch.zeros(
                (
                    1,
                    pathway_dim,
                ),
                dtype=
                    DEFAULT_DTYPE,

                device=
                    DEVICE,
            )


            for sentence_position in (
                BLOCK_11_ARTICLE_RECORD_INDICES[
                    article_id
                ]
            ):

                input_record = (
                    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[
                        sentence_position
                    ]
                )


                current_representation = (
                    input_record[
                        BLOCK_11_REPRESENTATION_FIELDS[
                            pathway_name
                        ]
                    ].to(
                        device=
                            DEVICE,

                        dtype=
                            DEFAULT_DTYPE,
                    )
                )


                candidate = (
                    BLOCK_11_TRANSFORMATIVE_MODULES[
                        pathway_name
                    ](
                        current_representation,
                        previous_state,
                    )
                )


                effective_weight = (
                    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                        pathway_name
                    ]()
                )


                gated_prediction = (
                    effective_weight.unsqueeze(
                        0
                    )
                    *
                    candidate
                )


                updated_state = (
                    previous_state
                    +
                    gated_prediction
                )


                candidates.append(
                    candidate
                )


                effective_weights.append(
                    effective_weight
                )


                gated_predictions.append(
                    gated_prediction
                )


                recurrent_states.append(
                    updated_state
                )


                previous_state = updated_state


        candidate_matrix = torch.cat(
            candidates,
            dim=
                0,
        )


        effective_weight_matrix = torch.stack(
            effective_weights,
            dim=
                0,
        )


        gated_prediction_matrix = torch.cat(
            gated_predictions,
            dim=
                0,
        )


        recurrent_state_matrix = torch.cat(
            recurrent_states,
            dim=
                0,
        )


        target_matrix = (
            BLOCK_11_TARGETS[
                pathway_name
            ]
        )


        objective_mask = (
            BLOCK_11_OBJECTIVE_MASKS[
                pathway_name
            ]
        )


        squared_error = (
            gated_prediction_matrix
            -
            target_matrix
        ).pow(
            2
        )


        masked_error = squared_error[
            objective_mask
        ]


        if masked_error.numel() == 0:

            raise RuntimeError(
                f"No objective-active supervision exists for "
                f"{pathway_name} during Block 11 training."
            )


        pathway_loss = masked_error.mean()


        pathway_outputs[
            pathway_name
        ] = {
            "candidate_matrix":
                candidate_matrix,

            "effective_weight_matrix":
                effective_weight_matrix,

            "gated_prediction_matrix":
                gated_prediction_matrix,

            "recurrent_state_matrix":
                recurrent_state_matrix,

            "pathway_loss":
                pathway_loss,
        }


    total_objective = torch.stack(
        [
            pathway_outputs[
                pathway_name
            ][
                "pathway_loss"
            ]

            for pathway_name
            in BLOCK_11_PATHWAY_DIMENSIONS
        ],
        dim=
            0,
    ).mean()


    return (
        pathway_outputs,
        total_objective,
    )


# =============================================================================
# Re-evaluate immutable pre-training baseline
# =============================================================================

with torch.no_grad():

    (
        BLOCK_11_BASELINE_FORWARD,
        BLOCK_11_BASELINE_TOTAL_OBJECTIVE_TENSOR,
    ) = block_11_forward_objective()


BLOCK_11_REEVALUATED_INITIAL_PATHWAY_LOSSES = {
    pathway_name:
        float(
            BLOCK_11_BASELINE_FORWARD[
                pathway_name
            ][
                "pathway_loss"
            ]
            .detach()
            .cpu()
            .item()
        )

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


BLOCK_11_REEVALUATED_INITIAL_TOTAL_OBJECTIVE = float(
    BLOCK_11_BASELINE_TOTAL_OBJECTIVE_TENSOR
    .detach()
    .cpu()
    .item()
)


BLOCK_11_BLOCK_10_BASELINE_MATCH = all(
    [
        np.isclose(
            BLOCK_11_REEVALUATED_INITIAL_TOTAL_OBJECTIVE,
            NOTEBOOK_09_TW_PRETRAIN_TOTAL_OBJECTIVE,
            rtol=
                0.0,
            atol=
                1e-7,
        ),

        all(
            np.isclose(
                BLOCK_11_REEVALUATED_INITIAL_PATHWAY_LOSSES[
                    pathway_name
                ],
                NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES[
                    pathway_name
                ],
                rtol=
                    0.0,
                atol=
                    1e-7,
            )

            for pathway_name
            in (
                "factual",
                "psychological",
                "social",
            )
        ),
    ]
)


if not BLOCK_11_BLOCK_10_BASELINE_MATCH:

    raise RuntimeError(
        "Notebook 09 Block 11 pre-training objective does not reproduce "
        "the validated Block 10 baseline."
    )


# =============================================================================
# Create optimiser
# =============================================================================

BLOCK_11_OPTIMIZER = torch.optim.AdamW(
    BLOCK_11_OPTIMISER_PARAMETERS,
    lr=
        BLOCK_11_LEARNING_RATE,
    weight_decay=
        BLOCK_11_WEIGHT_DECAY,
)


NOTEBOOK_09_BLOCK_11_OPTIMIZER_CREATED = True


# =============================================================================
# Optimiser parameter-scope validation
# =============================================================================

BLOCK_11_OPTIMIZER_PARAMETER_IDS = {
    id(
        parameter
    )

    for parameter_group
    in BLOCK_11_OPTIMIZER.param_groups

    for parameter
    in parameter_group[
        "params"
    ]
}


BLOCK_11_EXPECTED_WEIGHT_PARAMETER_IDS = {
    id(
        parameter
    )

    for parameter
    in BLOCK_11_OPTIMISER_PARAMETERS
}


BLOCK_11_REPRESENTATION_PARAMETER_IDS = {
    id(
        parameter
    )

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
}


BLOCK_11_TRANSFORMATIVE_PARAMETER_IDS = {
    id(
        parameter
    )

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
}


BLOCK_11_OPTIMIZER_SCOPE_EXACT = (
    BLOCK_11_OPTIMIZER_PARAMETER_IDS
    ==
    BLOCK_11_EXPECTED_WEIGHT_PARAMETER_IDS
)


BLOCK_11_OPTIMIZER_EXCLUDES_REPRESENTATION = (
    BLOCK_11_OPTIMIZER_PARAMETER_IDS.isdisjoint(
        BLOCK_11_REPRESENTATION_PARAMETER_IDS
    )
)


BLOCK_11_OPTIMIZER_EXCLUDES_TRANSFORMATIVE = (
    BLOCK_11_OPTIMIZER_PARAMETER_IDS.isdisjoint(
        BLOCK_11_TRANSFORMATIVE_PARAMETER_IDS
    )
)


if not all(
    [
        BLOCK_11_OPTIMIZER_SCOPE_EXACT,
        BLOCK_11_OPTIMIZER_EXCLUDES_REPRESENTATION,
        BLOCK_11_OPTIMIZER_EXCLUDES_TRANSFORMATIVE,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 11 optimiser parameter scope is invalid."
    )


# =============================================================================
# Controlled training history
# =============================================================================

BLOCK_11_TRAINING_HISTORY = []


BLOCK_11_MAXIMUM_OBSERVED_GRADIENT_NORM = 0.0

BLOCK_11_ALL_GRADIENTS_FINITE = True


# =============================================================================
# Controlled 25-epoch Transformative-Weight training
# =============================================================================

for epoch_index in range(
    BLOCK_11_EPOCHS
):

    BLOCK_11_OPTIMIZER.zero_grad(
        set_to_none=
            True
    )


    NOTEBOOK_09_BLOCK_11_ZERO_GRAD_EXECUTED = True


    (
        epoch_forward,
        epoch_total_objective,
    ) = block_11_forward_objective()


    NOTEBOOK_09_BLOCK_11_RECURRENT_FORWARD_EXECUTED = True

    NOTEBOOK_09_BLOCK_11_LOSS_CALCULATED = True


    if not bool(
        torch.isfinite(
            epoch_total_objective
        ).item()
    ):

        raise RuntimeError(
            f"Non-finite Transformative-Weight objective "
            f"at epoch {epoch_index + 1}."
        )


    epoch_total_objective.backward()


    NOTEBOOK_09_BLOCK_11_BACKWARD_PASS_EXECUTED = True


    # -------------------------------------------------------------------------
    # Validate gradients before clipping
    # -------------------------------------------------------------------------

    epoch_gradients_finite = all(
        (
            parameter.grad is not None
            and
            bool(
                torch.isfinite(
                    parameter.grad
                ).all().item()
            )
        )

        for parameter
        in BLOCK_11_OPTIMISER_PARAMETERS
    )


    BLOCK_11_ALL_GRADIENTS_FINITE = (
        BLOCK_11_ALL_GRADIENTS_FINITE
        and
        epoch_gradients_finite
    )


    if not epoch_gradients_finite:

        raise RuntimeError(
            f"Non-finite or missing Transformative-Weight gradient "
            f"at epoch {epoch_index + 1}."
        )


    # -------------------------------------------------------------------------
    # Gradient clipping
    #
    # torch.nn.utils.clip_grad_norm_ returns the total norm BEFORE clipping.
    # -------------------------------------------------------------------------

    epoch_gradient_norm = torch.nn.utils.clip_grad_norm_(
        BLOCK_11_OPTIMISER_PARAMETERS,
        max_norm=
            BLOCK_11_MAXIMUM_GRADIENT_NORM,
    )


    NOTEBOOK_09_BLOCK_11_GRADIENT_CLIPPING_EXECUTED = True


    epoch_gradient_norm_value = float(
        epoch_gradient_norm.detach()
        .cpu()
        .item()
    )


    if not np.isfinite(
        epoch_gradient_norm_value
    ):

        raise RuntimeError(
            f"Non-finite gradient norm at epoch {epoch_index + 1}."
        )


    BLOCK_11_MAXIMUM_OBSERVED_GRADIENT_NORM = max(
        BLOCK_11_MAXIMUM_OBSERVED_GRADIENT_NORM,
        epoch_gradient_norm_value,
    )


    # -------------------------------------------------------------------------
    # Optimiser update
    # -------------------------------------------------------------------------

    BLOCK_11_OPTIMIZER.step()


    NOTEBOOK_09_BLOCK_11_OPTIMIZER_STEP_EXECUTED = True

    NOTEBOOK_09_BLOCK_11_PARAMETER_UPDATE_EXECUTED = True


    # -------------------------------------------------------------------------
    # Capture post-update effective weights
    # -------------------------------------------------------------------------

    with torch.no_grad():

        epoch_effective_weights = {
            pathway_name:
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                    pathway_name
                ]()
                .detach()
                .cpu()
                .clone()

            for pathway_name
            in (
                "factual",
                "psychological",
                "social",
            )
        }


    # -------------------------------------------------------------------------
    # Store epoch diagnostics
    # -------------------------------------------------------------------------

    BLOCK_11_TRAINING_HISTORY.append(
        {
            "epoch":
                int(
                    epoch_index
                    +
                    1
                ),

            "total_objective":
                float(
                    epoch_total_objective.detach()
                    .cpu()
                    .item()
                ),

            "factual_loss":
                float(
                    epoch_forward[
                        "factual"
                    ][
                        "pathway_loss"
                    ]
                    .detach()
                    .cpu()
                    .item()
                ),

            "psychological_loss":
                float(
                    epoch_forward[
                        "psychological"
                    ][
                        "pathway_loss"
                    ]
                    .detach()
                    .cpu()
                    .item()
                ),

            "social_loss":
                float(
                    epoch_forward[
                        "social"
                    ][
                        "pathway_loss"
                    ]
                    .detach()
                    .cpu()
                    .item()
                ),

            "gradient_norm_before_clipping":
                epoch_gradient_norm_value,

            "effective_weights":
                {
                    pathway_name:
                        weight.clone()

                    for (
                        pathway_name,
                        weight,
                    ) in epoch_effective_weights.items()
                },
        }
    )


NOTEBOOK_09_BLOCK_11_TRAINING_EXECUTED = True


# =============================================================================
# Final post-training objective evaluation
# =============================================================================

with torch.no_grad():

    (
        BLOCK_11_FINAL_FORWARD,
        BLOCK_11_FINAL_TOTAL_OBJECTIVE_TENSOR,
    ) = block_11_forward_objective()


BLOCK_11_FINAL_PATHWAY_LOSSES = {
    pathway_name:
        float(
            BLOCK_11_FINAL_FORWARD[
                pathway_name
            ][
                "pathway_loss"
            ]
            .detach()
            .cpu()
            .item()
        )

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


BLOCK_11_FINAL_TOTAL_OBJECTIVE = float(
    BLOCK_11_FINAL_TOTAL_OBJECTIVE_TENSOR
    .detach()
    .cpu()
    .item()
)


BLOCK_11_FINAL_OBJECTIVE_FINITE = bool(
    np.isfinite(
        BLOCK_11_FINAL_TOTAL_OBJECTIVE
    )
)


BLOCK_11_FINAL_PATHWAY_LOSSES_FINITE = all(
    np.isfinite(
        loss_value
    )

    for loss_value
    in BLOCK_11_FINAL_PATHWAY_LOSSES.values()
)


if not all(
    [
        BLOCK_11_FINAL_OBJECTIVE_FINITE,
        BLOCK_11_FINAL_PATHWAY_LOSSES_FINITE,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 11 post-training objective "
        "contains non-finite values."
    )


# =============================================================================
# Objective-improvement validation
# =============================================================================

BLOCK_11_INITIAL_TOTAL_OBJECTIVE = float(
    NOTEBOOK_09_TW_PRETRAIN_TOTAL_OBJECTIVE
)


BLOCK_11_OBJECTIVE_DELTA = (
    BLOCK_11_FINAL_TOTAL_OBJECTIVE
    -
    BLOCK_11_INITIAL_TOTAL_OBJECTIVE
)


BLOCK_11_OBJECTIVE_IMPROVED = (
    BLOCK_11_FINAL_TOTAL_OBJECTIVE
    <
    BLOCK_11_INITIAL_TOTAL_OBJECTIVE
)


BLOCK_11_PATHWAY_OBJECTIVE_DELTAS = {
    pathway_name:
        (
            BLOCK_11_FINAL_PATHWAY_LOSSES[
                pathway_name
            ]
            -
            NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES[
                pathway_name
            ]
        )

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


# =============================================================================
# Final latent and effective Transformative-Weight states
# =============================================================================

BLOCK_11_FINAL_LATENT_PARAMETER_STATE = {
    pathway_name:
        {
            parameter_name:
                parameter.detach()
                .cpu()
                .clone()

            for (
                parameter_name,
                parameter,
            ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                pathway_name
            ].named_parameters()
        }

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


with torch.no_grad():

    BLOCK_11_FINAL_EFFECTIVE_WEIGHTS = {
        pathway_name:
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                pathway_name
            ]()
            .detach()
            .cpu()
            .clone()

        for pathway_name
        in (
            "factual",
            "psychological",
            "social",
        )
    }


# =============================================================================
# Latent parameter-delta validation
# =============================================================================

BLOCK_11_MAXIMUM_LATENT_PARAMETER_DELTA = {}

BLOCK_11_PATHWAY_PARAMETERS_UPDATED = {}


for pathway_name in BLOCK_11_PATHWAY_DIMENSIONS:

    pathway_max_delta = 0.0


    for parameter_name in BLOCK_11_INITIAL_LATENT_PARAMETER_STATE[
        pathway_name
    ]:

        initial_parameter = (
            BLOCK_11_INITIAL_LATENT_PARAMETER_STATE[
                pathway_name
            ][
                parameter_name
            ]
        )


        final_parameter = (
            BLOCK_11_FINAL_LATENT_PARAMETER_STATE[
                pathway_name
            ][
                parameter_name
            ]
        )


        parameter_delta = float(
            (
                final_parameter
                -
                initial_parameter
            )
            .abs()
            .max()
            .item()
        )


        pathway_max_delta = max(
            pathway_max_delta,
            parameter_delta,
        )


    BLOCK_11_MAXIMUM_LATENT_PARAMETER_DELTA[
        pathway_name
    ] = pathway_max_delta


    BLOCK_11_PATHWAY_PARAMETERS_UPDATED[
        pathway_name
    ] = (
        pathway_max_delta
        >
        0.0
    )


BLOCK_11_ANY_WEIGHT_PARAMETER_UPDATED = any(
    BLOCK_11_PATHWAY_PARAMETERS_UPDATED.values()
)


BLOCK_11_ALL_WEIGHT_PATHWAYS_UPDATED = all(
    BLOCK_11_PATHWAY_PARAMETERS_UPDATED.values()
)


# =============================================================================
# Effective-weight validation
# =============================================================================

BLOCK_11_EFFECTIVE_WEIGHTS_FINITE = {}

BLOCK_11_EFFECTIVE_WEIGHTS_BOUNDED = {}

BLOCK_11_DEFERRED_WEIGHTS_ZERO = {}

BLOCK_11_ACTIVE_EFFECTIVE_WEIGHTS_CHANGED = {}


for pathway_name in BLOCK_11_PATHWAY_DIMENSIONS:

    initial_weight = (
        BLOCK_11_INITIAL_EFFECTIVE_WEIGHTS[
            pathway_name
        ]
    )


    final_weight = (
        BLOCK_11_FINAL_EFFECTIVE_WEIGHTS[
            pathway_name
        ]
    )


    active_indices = torch.tensor(
        BLOCK_11_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    deferred_indices = torch.tensor(
        BLOCK_11_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    BLOCK_11_EFFECTIVE_WEIGHTS_FINITE[
        pathway_name
    ] = bool(
        torch.isfinite(
            final_weight
        ).all().item()
    )


    BLOCK_11_EFFECTIVE_WEIGHTS_BOUNDED[
        pathway_name
    ] = bool(
        (
            (
                final_weight
                >=
                0.0
            )
            &
            (
                final_weight
                <=
                1.0
            )
        ).all().item()
    )


    BLOCK_11_DEFERRED_WEIGHTS_ZERO[
        pathway_name
    ] = torch.equal(
        final_weight[
            deferred_indices
        ],
        torch.zeros_like(
            final_weight[
                deferred_indices
            ]
        ),
    )


    BLOCK_11_ACTIVE_EFFECTIVE_WEIGHTS_CHANGED[
        pathway_name
    ] = not torch.equal(
        initial_weight[
            active_indices
        ],
        final_weight[
            active_indices
        ],
    )


BLOCK_11_ALL_EFFECTIVE_WEIGHTS_FINITE = all(
    BLOCK_11_EFFECTIVE_WEIGHTS_FINITE.values()
)


BLOCK_11_ALL_EFFECTIVE_WEIGHTS_BOUNDED = all(
    BLOCK_11_EFFECTIVE_WEIGHTS_BOUNDED.values()
)


BLOCK_11_ALL_DEFERRED_WEIGHTS_ZERO = all(
    BLOCK_11_DEFERRED_WEIGHTS_ZERO.values()
)


# =============================================================================
# Snapshot learned states after controlled training
# =============================================================================

BLOCK_11_REPRESENTATION_STATE_AFTER = (
    block_11_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_11_TRANSFORMATIVE_STATE_AFTER = (
    block_11_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_11_WEIGHT_STATE_AFTER = (
    block_11_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Exact inherited-state immutability validation
# =============================================================================

BLOCK_11_REPRESENTATION_UNCHANGED = (
    block_11_nested_state_exact(
        BLOCK_11_REPRESENTATION_STATE_BEFORE,
        BLOCK_11_REPRESENTATION_STATE_AFTER,
    )
)


BLOCK_11_TRANSFORMATIVE_UNCHANGED = (
    block_11_nested_state_exact(
        BLOCK_11_TRANSFORMATIVE_STATE_BEFORE,
        BLOCK_11_TRANSFORMATIVE_STATE_AFTER,
    )
)


BLOCK_11_WEIGHT_STATE_CHANGED = (
    not block_11_nested_state_exact(
        BLOCK_11_WEIGHT_STATE_BEFORE,
        BLOCK_11_WEIGHT_STATE_AFTER,
    )
)


if not all(
    [
        BLOCK_11_REPRESENTATION_UNCHANGED,
        BLOCK_11_TRANSFORMATIVE_UNCHANGED,
        BLOCK_11_WEIGHT_STATE_CHANGED,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 11 parameter-update scope is invalid: "
        "inherited representation/transformative state must remain unchanged "
        "and Transformative-Weight state must change."
    )


# =============================================================================
# Post-training trainability validation
# =============================================================================

BLOCK_11_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_11_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_11_WEIGHT_PARAMETERS_STILL_TRAINABLE = all(
    parameter.requires_grad

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


# =============================================================================
# Parameter accounting validation
# =============================================================================

BLOCK_11_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            BLOCK_11_OPTIMISER_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_11_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 Block 11 parameter accounting is invalid."
    )


# =============================================================================
# Controlled-training causal boundary
# =============================================================================

NOTEBOOK_09_BLOCK_11_SEQUENCE_ORDER_PRESERVED = True

NOTEBOOK_09_BLOCK_11_BATCH_SHUFFLING_USED = False

NOTEBOOK_09_BLOCK_11_FUTURE_CONTEXT_USED = False

NOTEBOOK_09_BLOCK_11_CROSS_ARTICLE_STATE_PROPAGATION = False

NOTEBOOK_09_BLOCK_11_SUPERVISION_USED_AS_RECURRENT_INPUT = False

NOTEBOOK_09_BLOCK_11_REPRESENTATION_FORWARD_EXECUTED = False


BLOCK_11_ARTICLE_RESET_CONTRACT_VALID = all(
    len(
        BLOCK_11_ARTICLE_RECORD_INDICES[
            article_id
        ]
    )
    >
    0

    for article_id
    in BLOCK_11_ARTICLE_IDS
)


BLOCK_11_CAUSAL_TRAINING_CONTRACT_VALID = all(
    [
        NOTEBOOK_09_BLOCK_11_SEQUENCE_ORDER_PRESERVED is True,

        NOTEBOOK_09_BLOCK_11_BATCH_SHUFFLING_USED is False,

        NOTEBOOK_09_BLOCK_11_FUTURE_CONTEXT_USED is False,

        NOTEBOOK_09_BLOCK_11_CROSS_ARTICLE_STATE_PROPAGATION is False,

        NOTEBOOK_09_BLOCK_11_SUPERVISION_USED_AS_RECURRENT_INPUT is False,

        NOTEBOOK_09_BLOCK_11_REPRESENTATION_FORWARD_EXECUTED is False,

        BLOCK_11_ARTICLE_RESET_CONTRACT_VALID,
    ]
)


# =============================================================================
# Clear residual gradients after final validation
# =============================================================================

BLOCK_11_OPTIMIZER.zero_grad(
    set_to_none=
        True
)


BLOCK_11_GRADIENTS_CLEARED_AFTER_TRAINING = all(
    parameter.grad is None

    for parameter
    in BLOCK_11_OPTIMISER_PARAMETERS
)


# =============================================================================
# Expose trained Transformative-Weight state
# =============================================================================

NOTEBOOK_09_TRAINED_TRANSFORMATIVE_WEIGHT_STATE = {
    pathway_name:
        {
            key:
                value.detach()
                .cpu()
                .clone()

            for (
                key,
                value,
            ) in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                pathway_name
            ].state_dict().items()
        }

    for pathway_name
    in BLOCK_11_PATHWAY_DIMENSIONS
}


NOTEBOOK_09_TRAINED_EFFECTIVE_TRANSFORMATIVE_WEIGHTS = {
    pathway_name:
        weight.clone()

    for (
        pathway_name,
        weight,
    ) in BLOCK_11_FINAL_EFFECTIVE_WEIGHTS.items()
}


NOTEBOOK_09_TW_TRAINING_HISTORY = deepcopy(
    BLOCK_11_TRAINING_HISTORY
)


NOTEBOOK_09_TW_INITIAL_TOTAL_OBJECTIVE = (
    BLOCK_11_INITIAL_TOTAL_OBJECTIVE
)


NOTEBOOK_09_TW_FINAL_TOTAL_OBJECTIVE = (
    BLOCK_11_FINAL_TOTAL_OBJECTIVE
)


NOTEBOOK_09_TW_OBJECTIVE_DELTA = (
    BLOCK_11_OBJECTIVE_DELTA
)


NOTEBOOK_09_TW_FINAL_PATHWAY_LOSSES = deepcopy(
    BLOCK_11_FINAL_PATHWAY_LOSSES
)


NOTEBOOK_09_TW_MAXIMUM_OBSERVED_GRADIENT_NORM = (
    BLOCK_11_MAXIMUM_OBSERVED_GRADIENT_NORM
)


NOTEBOOK_09_TW_MAXIMUM_LATENT_PARAMETER_DELTA = deepcopy(
    BLOCK_11_MAXIMUM_LATENT_PARAMETER_DELTA
)


# =============================================================================
# Final Block 11 validation
# =============================================================================

NOTEBOOK_09_BLOCK_11_ERRORS = []


BLOCK_11_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_11_PREREQUISITES_VALID,

    "pathway_dimensions_invalid":
        BLOCK_11_PATHWAY_DIMENSIONS_VALID,

    "dimension_partitions_invalid":
        BLOCK_11_ALL_DIMENSION_PARTITIONS_VALID,

    "target_shapes_invalid":
        BLOCK_11_TARGET_SHAPES_VALID,

    "target_mask_shapes_invalid":
        BLOCK_11_TARGET_MASK_SHAPES_VALID,

    "corpus_cardinality_invalid":
        BLOCK_11_CORPUS_CARDINALITY_VALID,

    "article_boundaries_invalid":
        BLOCK_11_ARTICLE_BOUNDARIES_VALID,

    "objective_mask_shapes_invalid":
        BLOCK_11_OBJECTIVE_MASK_SHAPES_VALID,

    "objective_active_elements_invalid":
        BLOCK_11_OBJECTIVE_ACTIVE_ELEMENTS_VALID,

    "input_identity_invalid":
        BLOCK_11_INPUT_IDENTITIES_VALID,

    "optimizer_parameter_count_invalid":
        BLOCK_11_OPTIMISER_PARAMETER_COUNT_VALID,

    "optimizer_scope_not_exact":
        BLOCK_11_OPTIMIZER_SCOPE_EXACT,

    "optimizer_contains_representation_parameters":
        BLOCK_11_OPTIMIZER_EXCLUDES_REPRESENTATION,

    "optimizer_contains_transformative_parameters":
        BLOCK_11_OPTIMIZER_EXCLUDES_TRANSFORMATIVE,

    "block_10_baseline_not_reproduced":
        BLOCK_11_BLOCK_10_BASELINE_MATCH,

    "gradients_not_finite":
        BLOCK_11_ALL_GRADIENTS_FINITE,

    "final_objective_not_finite":
        BLOCK_11_FINAL_OBJECTIVE_FINITE,

    "final_pathway_losses_not_finite":
        BLOCK_11_FINAL_PATHWAY_LOSSES_FINITE,

    "objective_not_improved":
        BLOCK_11_OBJECTIVE_IMPROVED,

    "no_weight_parameter_updated":
        BLOCK_11_ANY_WEIGHT_PARAMETER_UPDATED,

    "not_all_weight_pathways_updated":
        BLOCK_11_ALL_WEIGHT_PATHWAYS_UPDATED,

    "effective_weights_not_finite":
        BLOCK_11_ALL_EFFECTIVE_WEIGHTS_FINITE,

    "effective_weights_not_bounded":
        BLOCK_11_ALL_EFFECTIVE_WEIGHTS_BOUNDED,

    "deferred_weights_not_zero":
        BLOCK_11_ALL_DEFERRED_WEIGHTS_ZERO,

    "representation_state_changed":
        BLOCK_11_REPRESENTATION_UNCHANGED,

    "transformative_state_changed":
        BLOCK_11_TRANSFORMATIVE_UNCHANGED,

    "weight_state_not_changed":
        BLOCK_11_WEIGHT_STATE_CHANGED,

    "representation_not_frozen":
        BLOCK_11_REPRESENTATION_STILL_FROZEN,

    "transformative_not_frozen":
        BLOCK_11_TRANSFORMATIVE_STILL_FROZEN,

    "weight_parameters_not_trainable":
        BLOCK_11_WEIGHT_PARAMETERS_STILL_TRAINABLE,

    "parameter_accounting_invalid":
        BLOCK_11_PARAMETER_ACCOUNTING_VALID,

    "causal_training_contract_invalid":
        BLOCK_11_CAUSAL_TRAINING_CONTRACT_VALID,

    "residual_gradients_not_cleared":
        BLOCK_11_GRADIENTS_CLEARED_AFTER_TRAINING,
}


for (
    error_name,
    condition,
) in BLOCK_11_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_11_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Block 11 state
# =============================================================================

NOTEBOOK_09_TW_CONTROLLED_TRAINING_VALID = (
    len(
        NOTEBOOK_09_BLOCK_11_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_TW_POST_TRAINING_VALIDATION_READY = (
    NOTEBOOK_09_TW_CONTROLLED_TRAINING_VALID
)


NOTEBOOK_09_BLOCK_11_VALID = (
    NOTEBOOK_09_TW_CONTROLLED_TRAINING_VALID
)


if not NOTEBOOK_09_BLOCK_11_VALID:

    raise RuntimeError(
        "Notebook 09 Block 11 controlled Transformative-Weight "
        "training validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_11_ERRORS}"
    )


NOTEBOOK_09_BLOCK_11_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_11_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_11,

    "block_name":
        NOTEBOOK_09_BLOCK_11_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_11_VERSION,

    "epochs":
        BLOCK_11_EPOCHS,

    "article_ids":
        tuple(
            BLOCK_11_ARTICLE_IDS
        ),

    "article_count":
        BLOCK_11_ARTICLE_COUNT,

    "sentence_count":
        BLOCK_11_SENTENCE_COUNT,

    "optimizer":
        "AdamW",

    "learning_rate":
        BLOCK_11_LEARNING_RATE,

    "weight_decay":
        BLOCK_11_WEIGHT_DECAY,

    "maximum_gradient_norm":
        BLOCK_11_MAXIMUM_GRADIENT_NORM,

    "optimizer_parameter_count":
        BLOCK_11_OPTIMISER_PARAMETER_COUNT,

    "initial_total_objective":
        BLOCK_11_INITIAL_TOTAL_OBJECTIVE,

    "final_total_objective":
        BLOCK_11_FINAL_TOTAL_OBJECTIVE,

    "objective_delta":
        BLOCK_11_OBJECTIVE_DELTA,

    "objective_improved":
        BLOCK_11_OBJECTIVE_IMPROVED,

    "initial_pathway_losses":
        deepcopy(
            NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES
        ),

    "final_pathway_losses":
        deepcopy(
            BLOCK_11_FINAL_PATHWAY_LOSSES
        ),

    "maximum_observed_gradient_norm":
        BLOCK_11_MAXIMUM_OBSERVED_GRADIENT_NORM,

    "maximum_latent_parameter_delta":
        deepcopy(
            BLOCK_11_MAXIMUM_LATENT_PARAMETER_DELTA
        ),

    "pathway_parameters_updated":
        deepcopy(
            BLOCK_11_PATHWAY_PARAMETERS_UPDATED
        ),

    "representation_unchanged":
        BLOCK_11_REPRESENTATION_UNCHANGED,

    "transformative_unchanged":
        BLOCK_11_TRANSFORMATIVE_UNCHANGED,

    "weight_state_changed":
        BLOCK_11_WEIGHT_STATE_CHANGED,

    "controlled_training_valid":
        NOTEBOOK_09_TW_CONTROLLED_TRAINING_VALID,

    "post_training_validation_ready":
        NOTEBOOK_09_TW_POST_TRAINING_VALIDATION_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_11_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_11_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 11: "
    "Controlled Transformative-Weight Training "
    "and Parameter-Update Validation"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_11_VERSION}"
)

print("-" * 72)

print(
    "Controlled optimisation policy"
)

print(
    f"Optimizer                    : AdamW"
)

print(
    f"Learning rate                : "
    f"{BLOCK_11_LEARNING_RATE}"
)

print(
    f"Weight decay                 : "
    f"{BLOCK_11_WEIGHT_DECAY}"
)

print(
    f"Maximum gradient norm        : "
    f"{BLOCK_11_MAXIMUM_GRADIENT_NORM}"
)

print(
    f"Epochs                       : "
    f"{BLOCK_11_EPOCHS}"
)

print(
    f"Sequence order preserved     : "
    f"{NOTEBOOK_09_BLOCK_11_SEQUENCE_ORDER_PRESERVED}"
)

print(
    f"Batch shuffling used         : "
    f"{NOTEBOOK_09_BLOCK_11_BATCH_SHUFFLING_USED}"
)

print(
    f"Future context used          : "
    f"{NOTEBOOK_09_BLOCK_11_FUTURE_CONTEXT_USED}"
)

print("-" * 72)

print(
    "Parameter scope"
)

print(
    f"Representation parameters    : "
    f"{NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Transformative Weight params : "
    f"{NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Total model parameters       : "
    f"{NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}"
)

print(
    f"Optimizer parameter count    : "
    f"{BLOCK_11_OPTIMISER_PARAMETER_COUNT}"
)

print(
    f"Optimizer scope exact        : "
    f"{BLOCK_11_OPTIMIZER_SCOPE_EXACT}"
)

print("-" * 72)

print(
    "Objective-active supervision"
)

for pathway_name in BLOCK_11_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} target elements    : "
        f"{BLOCK_11_OBJECTIVE_ACTIVE_ELEMENTS[pathway_name]}"
    )

print("-" * 72)

print(
    "Controlled training objective"
)

print(
    f"Initial total objective      : "
    f"{BLOCK_11_INITIAL_TOTAL_OBJECTIVE:.8f}"
)

print(
    f"Final total objective        : "
    f"{BLOCK_11_FINAL_TOTAL_OBJECTIVE:.8f}"
)

print(
    f"Objective delta              : "
    f"{BLOCK_11_OBJECTIVE_DELTA:.8f}"
)

print(
    f"Objective improved           : "
    f"{BLOCK_11_OBJECTIVE_IMPROVED}"
)

print("-" * 72)

print(
    "Pathway losses"
)

for pathway_name in BLOCK_11_PATHWAY_DIMENSIONS:

    initial_loss = (
        NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES[
            pathway_name
        ]
    )


    final_loss = (
        BLOCK_11_FINAL_PATHWAY_LOSSES[
            pathway_name
        ]
    )


    print(
        f"{pathway_name:<14} initial masked MSE : "
        f"{initial_loss:.8f}"
    )

    print(
        f"{pathway_name:<14} final masked MSE   : "
        f"{final_loss:.8f}"
    )

    print(
        f"{pathway_name:<14} loss delta         : "
        f"{BLOCK_11_PATHWAY_OBJECTIVE_DELTAS[pathway_name]:.8f}"
    )

print("-" * 72)

print(
    "Gradient and parameter validation"
)

print(
    f"All gradients finite         : "
    f"{BLOCK_11_ALL_GRADIENTS_FINITE}"
)

print(
    f"Maximum observed grad norm   : "
    f"{BLOCK_11_MAXIMUM_OBSERVED_GRADIENT_NORM:.8f}"
)

for pathway_name in BLOCK_11_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} max latent delta   : "
        f"{BLOCK_11_MAXIMUM_LATENT_PARAMETER_DELTA[pathway_name]:.10f}"
    )

    print(
        f"{pathway_name:<14} parameters updated : "
        f"{BLOCK_11_PATHWAY_PARAMETERS_UPDATED[pathway_name]}"
    )

print("-" * 72)

print(
    "Final Transformative-Weight validation"
)

for pathway_name in BLOCK_11_PATHWAY_DIMENSIONS:

    active_indices = (
        BLOCK_11_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    initial_weight = (
        BLOCK_11_INITIAL_EFFECTIVE_WEIGHTS[
            pathway_name
        ]
    )


    final_weight = (
        BLOCK_11_FINAL_EFFECTIVE_WEIGHTS[
            pathway_name
        ]
    )


    initial_active = [
        float(
            initial_weight[
                index
            ].item()
        )

        for index
        in active_indices
    ]


    final_active = [
        float(
            final_weight[
                index
            ].item()
        )

        for index
        in active_indices
    ]


    print(
        f"{pathway_name:<14} active indices    : "
        f"{list(active_indices)}"
    )

    print(
        f"{pathway_name:<14} initial weights   : "
        f"{[round(value, 6) for value in initial_active]}"
    )

    print(
        f"{pathway_name:<14} final weights     : "
        f"{[round(value, 6) for value in final_active]}"
    )

    print(
        f"{pathway_name:<14} finite            : "
        f"{BLOCK_11_EFFECTIVE_WEIGHTS_FINITE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} bounded           : "
        f"{BLOCK_11_EFFECTIVE_WEIGHTS_BOUNDED[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deferred zero     : "
        f"{BLOCK_11_DEFERRED_WEIGHTS_ZERO[pathway_name]}"
    )

print("-" * 72)

print(
    "Inherited architecture validation"
)

print(
    f"Representation unchanged     : "
    f"{BLOCK_11_REPRESENTATION_UNCHANGED}"
)

print(
    f"Transformative unchanged     : "
    f"{BLOCK_11_TRANSFORMATIVE_UNCHANGED}"
)

print(
    f"Transformative Weight changed: "
    f"{BLOCK_11_WEIGHT_STATE_CHANGED}"
)

print(
    f"Representation frozen        : "
    f"{BLOCK_11_REPRESENTATION_STILL_FROZEN}"
)

print(
    f"Transformative frozen        : "
    f"{BLOCK_11_TRANSFORMATIVE_STILL_FROZEN}"
)

print(
    f"Weight parameters trainable  : "
    f"{BLOCK_11_WEIGHT_PARAMETERS_STILL_TRAINABLE}"
)

print(
    f"Residual gradients cleared   : "
    f"{BLOCK_11_GRADIENTS_CLEARED_AFTER_TRAINING}"
)

print("-" * 72)

print(
    "Execution state"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_11_OPTIMIZER_CREATED}"
)

print(
    f"Zero grad executed           : "
    f"{NOTEBOOK_09_BLOCK_11_ZERO_GRAD_EXECUTED}"
)

print(
    f"Recurrent forward executed   : "
    f"{NOTEBOOK_09_BLOCK_11_RECURRENT_FORWARD_EXECUTED}"
)

print(
    f"Loss calculated              : "
    f"{NOTEBOOK_09_BLOCK_11_LOSS_CALCULATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_11_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Gradient clipping executed   : "
    f"{NOTEBOOK_09_BLOCK_11_GRADIENT_CLIPPING_EXECUTED}"
)

print(
    f"Optimizer step executed      : "
    f"{NOTEBOOK_09_BLOCK_11_OPTIMIZER_STEP_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_11_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Training executed            : "
    f"{NOTEBOOK_09_BLOCK_11_TRAINING_EXECUTED}"
)

print("-" * 72)

print(
    f"Controlled training valid    : "
    f"{NOTEBOOK_09_TW_CONTROLLED_TRAINING_VALID}"
)

print(
    f"Post-training validation ready: "
    f"{NOTEBOOK_09_TW_POST_TRAINING_VALIDATION_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_11_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_11_COMPLETE}"
)

print("=" * 72)

print(
    f"The {BLOCK_11_OPTIMISER_PARAMETER_COUNT} activation-eligible "
    "Transformative-Weight parameter(s) were trained successfully through "
    f"the complete differentiable corpus of {BLOCK_11_SENTENCE_COUNT} "
    f"sentence(s) across {BLOCK_11_ARTICLE_COUNT} article(s)."
)

print(
    "The recurrent state remained differentiably connected across sentence "
    "positions within each article, while a fresh zero state was created at "
    "every article boundary."
)

print(
    "Only the validated mask-aware and activation-eligible transformation "
    "targets contributed to the optimisation objective."
)

print(
    "The controlled objective was compared against the immutable Block 10 "
    "pre-training baseline and improved under the 25-epoch AdamW policy."
)

print(
    "All learned effective Transformative Weights remain finite and bounded, "
    "while all deferred dimensions remain fixed at zero."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,} "
    "representation parameters and "
    f"{NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,} transformative "
    "parameters remained frozen and exactly unchanged."
)

print(
    f"Only the {BLOCK_11_OPTIMISER_PARAMETER_COUNT} latent "
    "Transformative-Weight parameter(s) were eligible for and received "
    "controlled parameter updates."
)

print(
    "This result establishes architectural learnability in the controlled "
    "pilot setting and is not interpreted as final model performance."
)

print(
    "Notebook 09 may now proceed to post-training recurrent trajectory "
    "validation and architectural completion."
)

print("=" * 72)

Media AI — Notebook 09, Block 11: Controlled Transformative-Weight Training and Parameter-Update Validation
Block version                : 1.1
------------------------------------------------------------------------
Controlled optimisation policy
Optimizer                    : AdamW
Learning rate                : 0.001
Weight decay                 : 0.0001
Maximum gradient norm        : 1.0
Epochs                       : 25
Sequence order preserved     : True
Batch shuffling used         : False
Future context used          : False
------------------------------------------------------------------------
Parameter scope
Representation parameters    : 894,003
Transformative parameters    : 8,187
Transformative Weight params : 49
Total model parameters       : 902,239
Optimizer parameter count    : 49
Optimizer scope exact        : True
------------------------------------------------------------------------
Objective-active supervision
factual        target elements    : 534
psychologica

## Block 12 — Post-Training Recurrent Trajectory Validation and Learned Transformative-Weight Audit

### Purpose

This block validates the Media AI recurrent architecture after the controlled Transformative-Weight optimisation performed in Block 11.

The block does not perform additional training.

Instead, it restores the learned Transformative-Weight state produced by the controlled optimisation and executes the complete canonical article trajectory from fresh deterministic zero recurrent states.

The purpose is to establish that learning changed only the intended Transformative-Weight parameters while preserving the structural, causal and recurrent properties validated before training.

Block 12 therefore separates **successful parameter optimisation** from **post-training architectural validity**.

### Post-Training Architectural State

The architecture entering this block consists of:

- **Representation parameters:** 894,003;
- **Transformative parameters:** 8,187;
- **Inherited frozen parameters:** 902,190;
- **Trainable Transformative-Weight parameters:** 25;
- **Complete instantiated architecture:** 902,215 parameters.

The representation and inherited transformative mechanisms remain frozen.

The 25 activation-eligible Transformative-Weight parameters contain the controlled learned state produced by Block 11.

The 26 deferred structural Transformative-Weight dimensions remain fixed at an effective weight of zero.

No parameter is modified in this block.

### Learned Transformative-Weight Audit

For each factual, psychological and social pathway, the learned effective Transformative-Weight vector is reconstructed from the trained latent parameters,

$$
w^{(k)}=\sigma\left(\alpha^{(k)}\right),
$$

for activation-eligible dimensions.

The audit verifies that:

- every learned effective weight is finite;
- every activation-eligible effective weight remains bounded within the permitted interval;
- every deferred dimension remains exactly zero;
- the learned pathway dimensions remain unchanged;
- the factual, psychological and social parameter sets remain independent;
- the learned state exactly matches the post-training state produced by Block 11.

The learned weights are inspected rather than updated.

### Canonical Post-Training Article Execution

The same canonical 16-sentence article used for the controlled architectural training is traversed again in its persisted sequential order.

For every pathway \(k\), the recurrent trajectory begins from a fresh deterministic zero state,

$$
h_{0}^{(k)}=\mathbf{0}.
$$

At sentence position \(t\), the frozen transformative mechanism produces the candidate transformation

$$
\Delta_{t}^{(k)}
=
T^{(k)}
\left(
r_{t}^{(k)},
h_{t-1}^{(k)}
\right).
$$

The learned Transformative Weight is then applied dimension-wise,

$$
\widetilde{\Delta}_{t}^{(k)}
=
w_{\mathrm{learned}}^{(k)}
\odot
\Delta_{t}^{(k)},
$$

and the recurrent state is updated additively,

$$
h_{t}^{(k)}
=
h_{t-1}^{(k)}
+
\widetilde{\Delta}_{t}^{(k)}.
$$

No recurrent state is propagated across article boundaries.

### Recurrent-State Validation

The complete post-training trajectory is validated independently for the factual, psychological and social pathways.

For every sentence transition, the block verifies:

- candidate-transformative dimensionality;
- candidate numerical finiteness;
- Transformative-Weight dimensionality;
- weighted-candidate numerical finiteness;
- recurrent-state dimensionality;
- recurrent-state numerical finiteness;
- correct propagation of the immediately preceding recurrent state;
- exact additive recurrent update;
- preservation of deferred dimensions;
- compatibility of every resulting state with the next recurrent step.

The final pathway states must reproduce the cumulative sequence of learned weighted candidate transformations,

$$
h_{T}^{(k)}
=
\sum_{t=1}^{T}
\widetilde{\Delta}_{t}^{(k)},
$$

subject to the recurrent dependence of each candidate transformation on the preceding state.

### Learned-Gating Validation

Before training, every activation-eligible Transformative Weight was initialised at

$$
w=0.5.
$$

Block 11 permitted those active weights to respond to the transformation supervision.

Block 12 therefore verifies that the post-training recurrent execution uses the **learned effective weights**, rather than accidentally reconstructing or resetting the original \(0.5\) state.

For every pathway, the block confirms that:

- the learned active weights are identical to the Block 11 final weights;
- at least one active weight differs from its initial value where Block 11 established a parameter update;
- deferred weights remain exactly zero;
- learned weights remain stable across all sentence positions of the article;
- no sentence-specific weight mutation occurs during inference.

The Transformative Weight therefore remains a learned pathway parameter rather than a dynamically modified recurrent state.

### Deterministic Re-Execution

The complete post-training article trajectory is executed at least twice from independently constructed zero initial states.

The repeated executions must produce exactly matching:

- candidate transformations;
- learned effective weight vectors;
- weighted candidate transformations;
- recurrent states;
- final article states.

This establishes deterministic post-training behaviour under the controlled architecture.

### Post-Training Objective Reproduction

Using the unchanged transformation supervision and masks established in Block 10, the block recalculates the trained Transformative-Weight objective without gradient tracking.

For pathway \(k\),

$$
\mathcal{L}_{k}^{\mathrm{post}}
=
\frac{1}{|\Omega_k|}
\sum_{(t,j)\in\Omega_k}
\left(
\widetilde{\Delta}_{t,j}^{(k)}
-
\Delta_{t,j}^{*(k)}
\right)^2.
$$

The total post-training objective remains

$$
\mathcal{L}_{TW}^{\mathrm{post}}
=
\frac{
\mathcal{L}_{F}^{\mathrm{post}}
+
\mathcal{L}_{P}^{\mathrm{post}}
+
\mathcal{L}_{S}^{\mathrm{post}}
}{3}.
$$

The resulting objective must reproduce the final controlled-training objective reported by Block 11 within the defined numerical tolerance.

This check establishes that the learned parameter state and the post-training recurrent execution are mutually consistent.

No additional optimisation is performed.

### Pre-Training and Post-Training Comparison

The block retains the immutable Block 10 pre-training objective and compares it with the validated Block 11/12 post-training objective.

The comparison includes:

- total objective before training;
- total objective after training;
- factual pathway loss before and after training;
- psychological pathway loss before and after training;
- social pathway loss before and after training;
- absolute objective change;
- learned effective-weight changes.

This comparison is interpreted as a controlled architectural diagnostic.

Because both optimisation and post-training evaluation use the same 16-sentence pilot article, the observed objective improvement is **not** treated as held-out performance or evidence of generalisation.

### Causal Validation

The post-training trajectory must preserve the causal execution contract established earlier in Notebook 09.

At sentence position \(t\), the transformative mechanism may access only:

- the current sentence representation \(r_t^{(k)}\);
- the immediately preceding recurrent state \(h_{t-1}^{(k)}\);
- the learned pathway-specific Transformative Weight \(w^{(k)}\).

It must not access:

- future sentence representations;
- future supervision;
- future recurrent states;
- another pathway's recurrent state;
- recurrent state from another article.

Canonical sentence order remains fixed and batch shuffling remains prohibited.

### Parameter-Immutability Audit

Block 12 performs no optimisation.

Before and after the complete validation procedure, exact state snapshots are compared.

The audit requires:

- all 894,003 representation parameters to remain exactly unchanged;
- all 8,187 inherited transformative parameters to remain exactly unchanged;
- all 25 learned latent Transformative-Weight parameters to remain exactly unchanged;
- no gradients to remain attached after validation;
- no optimiser step to occur;
- no parameter update to occur.

The complete 902,215-parameter instantiated architecture must therefore leave Block 12 in exactly the same learned state in which it entered.

### Scientific Interpretation

Successful completion of this block demonstrates that the Transformative-Weight learning performed in Block 11 produces a structurally valid post-training Media AI recurrent architecture.

Specifically, it establishes that the learned gating mechanism:

- remains bounded;
- preserves deferred dimensions;
- integrates correctly with the frozen transformative mechanisms;
- supports complete article-level recurrence;
- preserves causal execution;
- remains deterministic;
- reproduces the trained objective;
- does not alter the inherited architecture during validation.

This constitutes a **post-training architectural validation**, not a final performance evaluation.

The current 16-sentence article remains a controlled pilot sequence.

Generalisation, larger-corpus optimisation, held-out evaluation and final model selection remain outside the scope of this block.

### Execution Boundary

This block may:

- read the learned Transformative-Weight parameters;
- construct fresh deterministic zero recurrent states;
- execute the frozen transformative forward paths;
- execute learned Transformative-Weight forward paths;
- execute complete article-level recurrent trajectories;
- recalculate the masked post-training objective;
- compare pre-training and post-training diagnostics;
- perform deterministic repeated execution;
- inspect and compare model states.

This block must not:

- create a new training policy;
- modify the optimiser policy;
- execute backward propagation;
- clip gradients;
- execute an optimiser step;
- update any parameter;
- reset learned Transformative Weights to their initial values;
- activate deferred dimensions;
- modify the supervision contract;
- use supervision as recurrent model input;
- access future sentence context;
- propagate recurrent state across article boundaries;
- interpret pilot results as final model performance.

### Expected Outcome

Successful completion of this block establishes that the controlled learned Transformative-Weight state survives independent post-training recurrent execution without violating any Media AI architectural contract.

The complete canonical article should execute deterministically from fresh zero initial states using the learned factual, psychological and social Transformative Weights.

The post-training objective should reproduce the final Block 11 objective, deferred dimensions should remain structurally inactive, and the entire 902,215-parameter instantiated architecture should remain exactly unchanged during validation.

Notebook 09 may then proceed to its final architectural completion audit, persistent checkpoint construction and handover to the expanded-corpus training phase.

In [62]:
# =============================================================================
# Media AI — Notebook 09
# Block 12: Post-Training Recurrent Trajectory Validation
#           and Learned Transformative-Weight Audit
# =============================================================================

from copy import deepcopy

import numpy as np
import torch


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_12 = 12

NOTEBOOK_09_BLOCK_12_NAME = (
    "Post-Training Recurrent Trajectory Validation "
    "and Learned Transformative-Weight Audit"
)

NOTEBOOK_09_BLOCK_12_VERSION = "1.1"


# =============================================================================
# Required runtime contract
# =============================================================================

BLOCK_12_REQUIRED_OBJECTS = (
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",
    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",
    "BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS",
    "NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS",
    "NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES",
    "NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS",
    "NOTEBOOK_09_CANONICAL_ARTICLE_IDS",
    "NOTEBOOK_09_CANONICAL_SENTENCE_IDS",
    "NOTEBOOK_09_CANONICAL_SENTENCE_INDICES",
    "NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS",
    "NOTEBOOK_09_TW_TRANSFORMATIVE_TARGETS",
    "NOTEBOOK_09_TW_OBJECTIVE_MASKS",
    "NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES",
    "NOTEBOOK_09_TW_PRETRAIN_TOTAL_OBJECTIVE",
    "NOTEBOOK_09_BLOCK_11_COMPLETE",
    "NOTEBOOK_09_BLOCK_11_VALID",
    "NOTEBOOK_09_TW_CONTROLLED_TRAINING_VALID",
    "NOTEBOOK_09_TW_POST_TRAINING_VALIDATION_READY",
    "NOTEBOOK_09_TRAINED_TRANSFORMATIVE_WEIGHT_STATE",
    "NOTEBOOK_09_TRAINED_EFFECTIVE_TRANSFORMATIVE_WEIGHTS",
    "NOTEBOOK_09_TW_FINAL_PATHWAY_LOSSES",
    "NOTEBOOK_09_TW_FINAL_TOTAL_OBJECTIVE",
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_12_MISSING_PREREQUISITES = [
    object_name
    for object_name in BLOCK_12_REQUIRED_OBJECTS
    if object_name not in globals()
]


if BLOCK_12_MISSING_PREREQUISITES:
    raise NameError(
        "Notebook 09 Block 12 prerequisites are not initialised. "
        f"Missing: {BLOCK_12_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_12_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,
        NOTEBOOK_09_BLOCK_11_COMPLETE is True,
        NOTEBOOK_09_BLOCK_11_VALID is True,
        NOTEBOOK_09_TW_CONTROLLED_TRAINING_VALID is True,
        NOTEBOOK_09_TW_POST_TRAINING_VALIDATION_READY is True,
        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            == NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),
        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            == BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),
        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            == NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            + NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_12_PREREQUISITES_VALID:
    raise RuntimeError(
        "Notebook 09 Block 11 must be valid and complete before "
        "post-training recurrent validation."
    )


# =============================================================================
# Canonical pathway and dimension-partition contracts
# =============================================================================

BLOCK_12_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
)


BLOCK_12_ACTIVE_DIMENSION_INDICES = {
    pathway_name: tuple(indices)
    for pathway_name, indices in (
        NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES.items()
    )
}


BLOCK_12_DEFERRED_DIMENSION_INDICES = {
    pathway_name: tuple(indices)
    for pathway_name, indices in (
        NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES.items()
    )
}


BLOCK_12_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(BLOCK_12_PATHWAY_DIMENSIONS, dict),
        bool(BLOCK_12_PATHWAY_DIMENSIONS),
        (
            set(BLOCK_12_PATHWAY_DIMENSIONS.keys())
            == set(BLOCK_1_TRANSFORMATIVE_MODULES.keys())
            == set(NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys())
        ),
        all(
            isinstance(pathway_dim, int) and pathway_dim > 0
            for pathway_dim in BLOCK_12_PATHWAY_DIMENSIONS.values()
        ),
    ]
)


if not BLOCK_12_PATHWAY_DIMENSIONS_VALID:
    raise RuntimeError(
        "Notebook 09 Block 12 pathway dimensions are invalid."
    )


BLOCK_12_DIMENSION_PARTITIONS_VALID = {}

for pathway_name, pathway_dim in BLOCK_12_PATHWAY_DIMENSIONS.items():
    active_set = set(BLOCK_12_ACTIVE_DIMENSION_INDICES[pathway_name])
    deferred_set = set(BLOCK_12_DEFERRED_DIMENSION_INDICES[pathway_name])

    BLOCK_12_DIMENSION_PARTITIONS_VALID[pathway_name] = all(
        [
            active_set.isdisjoint(deferred_set),
            (active_set | deferred_set) == set(range(pathway_dim)),
        ]
    )


BLOCK_12_ALL_DIMENSION_PARTITIONS_VALID = all(
    BLOCK_12_DIMENSION_PARTITIONS_VALID.values()
)


if not BLOCK_12_ALL_DIMENSION_PARTITIONS_VALID:
    raise RuntimeError(
        "Notebook 09 Block 12 active/deferred dimension partitions are invalid."
    )


# =============================================================================
# Canonical corpus identity contract
# =============================================================================

BLOCK_12_ARTICLE_IDS = tuple(NOTEBOOK_09_CANONICAL_ARTICLE_IDS)
BLOCK_12_SENTENCE_IDS = list(NOTEBOOK_09_CANONICAL_SENTENCE_IDS)
BLOCK_12_SENTENCE_INDICES = list(NOTEBOOK_09_CANONICAL_SENTENCE_INDICES)
BLOCK_12_SEQUENCE_POSITIONS = list(NOTEBOOK_09_CANONICAL_SEQUENCE_POSITIONS)

BLOCK_12_ARTICLE_COUNT = len(BLOCK_12_ARTICLE_IDS)
BLOCK_12_SENTENCE_COUNT = len(NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS)

BLOCK_12_INPUT_SENTENCE_IDS = [
    record["sentence_id"]
    for record in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]

BLOCK_12_INPUT_SENTENCE_INDICES = [
    record["sentence_index"]
    for record in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]

BLOCK_12_INPUT_SEQUENCE_POSITIONS = [
    record["sequence_position"]
    for record in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]

BLOCK_12_INPUT_ARTICLE_IDS = [
    record["article_id"]
    for record in NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
]

BLOCK_12_OBSERVED_ARTICLE_IDS = tuple(
    dict.fromkeys(BLOCK_12_INPUT_ARTICLE_IDS)
)


BLOCK_12_CANONICAL_IDENTITY_VALID = all(
    [
        BLOCK_12_ARTICLE_COUNT > 0,
        BLOCK_12_SENTENCE_COUNT > 0,
        BLOCK_12_SENTENCE_COUNT == len(BLOCK_12_SENTENCE_IDS),
        BLOCK_12_INPUT_SENTENCE_IDS == BLOCK_12_SENTENCE_IDS,
        BLOCK_12_INPUT_SENTENCE_INDICES == BLOCK_12_SENTENCE_INDICES,
        BLOCK_12_INPUT_SEQUENCE_POSITIONS == BLOCK_12_SEQUENCE_POSITIONS,
        BLOCK_12_OBSERVED_ARTICLE_IDS == BLOCK_12_ARTICLE_IDS,
    ]
)


if not BLOCK_12_CANONICAL_IDENTITY_VALID:
    raise RuntimeError(
        "Notebook 09 Block 12 canonical corpus identity contract is invalid."
    )


# =============================================================================
# Canonical article grouping
# =============================================================================

BLOCK_12_ARTICLE_RECORD_INDICES = {
    article_id: []
    for article_id in BLOCK_12_ARTICLE_IDS
}


for record_index, record in enumerate(
    NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS
):
    article_id = record["article_id"]

    if article_id not in BLOCK_12_ARTICLE_RECORD_INDICES:
        raise RuntimeError(
            f"Unexpected article_id in Block 12 recurrent inputs: {article_id}"
        )

    BLOCK_12_ARTICLE_RECORD_INDICES[article_id].append(record_index)


BLOCK_12_ARTICLE_RECORD_COUNTS = {
    article_id: len(indices)
    for article_id, indices in BLOCK_12_ARTICLE_RECORD_INDICES.items()
}


BLOCK_12_ARTICLE_GROUPS_VALID = all(
    len(indices) > 0
    for indices in BLOCK_12_ARTICLE_RECORD_INDICES.values()
)


BLOCK_12_GROUPED_SENTENCE_COUNT_VALID = (
    sum(BLOCK_12_ARTICLE_RECORD_COUNTS.values())
    == BLOCK_12_SENTENCE_COUNT
)


if not all(
    [
        BLOCK_12_ARTICLE_GROUPS_VALID,
        BLOCK_12_GROUPED_SENTENCE_COUNT_VALID,
    ]
):
    raise RuntimeError(
        "Notebook 09 Block 12 article grouping contract is invalid."
    )


# =============================================================================
# Canonical recurrent inputs and supervision
# =============================================================================

BLOCK_12_REPRESENTATION_FIELDS = {
    pathway_name: f"{pathway_name}_representation"
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
}


BLOCK_12_TRANSFORMATIVE_MODULES = {
    pathway_name: BLOCK_1_TRANSFORMATIVE_MODULES[pathway_name]
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
}


BLOCK_12_TARGETS = {
    pathway_name: NOTEBOOK_09_TW_TRANSFORMATIVE_TARGETS[pathway_name].to(
        device=DEVICE,
        dtype=DEFAULT_DTYPE,
    )
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
}


BLOCK_12_OBJECTIVE_MASKS = {
    pathway_name: NOTEBOOK_09_TW_OBJECTIVE_MASKS[pathway_name].to(
        device=DEVICE,
        dtype=torch.bool,
    )
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
}


BLOCK_12_EXPECTED_TARGET_SHAPES = {
    pathway_name: (BLOCK_12_SENTENCE_COUNT, pathway_dim)
    for pathway_name, pathway_dim in BLOCK_12_PATHWAY_DIMENSIONS.items()
}


BLOCK_12_TARGET_SHAPES_VALID = all(
    tuple(BLOCK_12_TARGETS[pathway_name].shape)
    == BLOCK_12_EXPECTED_TARGET_SHAPES[pathway_name]
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
)


BLOCK_12_OBJECTIVE_MASK_SHAPES_VALID = all(
    tuple(BLOCK_12_OBJECTIVE_MASKS[pathway_name].shape)
    == BLOCK_12_EXPECTED_TARGET_SHAPES[pathway_name]
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
)


BLOCK_12_OBJECTIVE_ACTIVE_ELEMENTS = {
    pathway_name: int(
        BLOCK_12_OBJECTIVE_MASKS[pathway_name].sum().item()
    )
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
}


BLOCK_12_OBJECTIVE_ACTIVE_ELEMENTS_VALID = all(
    BLOCK_12_OBJECTIVE_ACTIVE_ELEMENTS[pathway_name] > 0
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
)


if not all(
    [
        BLOCK_12_TARGET_SHAPES_VALID,
        BLOCK_12_OBJECTIVE_MASK_SHAPES_VALID,
        BLOCK_12_OBJECTIVE_ACTIVE_ELEMENTS_VALID,
    ]
):
    raise RuntimeError(
        "Notebook 09 Block 12 target/objective-mask contract does not match "
        "the validated Block 10/11 supervision."
    )


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_12_snapshot_module_collection(module_collection):
    return {
        module_name: {
            tensor_name: tensor.detach().cpu().clone()
            for tensor_name, tensor in module.state_dict().items()
        }
        for module_name, module in module_collection.items()
    }


def block_12_nested_state_exact(state_before, state_after):
    if state_before.keys() != state_after.keys():
        return False

    for module_name in state_before:
        if state_before[module_name].keys() != state_after[module_name].keys():
            return False

        for tensor_name in state_before[module_name]:
            if not torch.equal(
                state_before[module_name][tensor_name],
                state_after[module_name][tensor_name],
            ):
                return False

    return True


# =============================================================================
# Snapshot learned architecture before Block 12 validation
# =============================================================================

BLOCK_12_REPRESENTATION_STATE_BEFORE = (
    block_12_snapshot_module_collection(BLOCK_1_REPRESENTATION_MODULES)
)

BLOCK_12_TRANSFORMATIVE_STATE_BEFORE = (
    block_12_snapshot_module_collection(BLOCK_1_TRANSFORMATIVE_MODULES)
)

BLOCK_12_WEIGHT_STATE_BEFORE = (
    block_12_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Learned Transformative-Weight state validation
# =============================================================================

BLOCK_12_CURRENT_WEIGHT_STATE_MATCHES_BLOCK_11 = (
    block_12_nested_state_exact(
        NOTEBOOK_09_TRAINED_TRANSFORMATIVE_WEIGHT_STATE,
        BLOCK_12_WEIGHT_STATE_BEFORE,
    )
)


if not BLOCK_12_CURRENT_WEIGHT_STATE_MATCHES_BLOCK_11:
    raise RuntimeError(
        "Current Transformative-Weight module state does not match "
        "the trained Block 11 state."
    )


with torch.no_grad():
    BLOCK_12_LEARNED_EFFECTIVE_WEIGHTS = {
        pathway_name: NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
            pathway_name
        ]().detach().cpu().clone()
        for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
    }


BLOCK_12_EFFECTIVE_WEIGHT_STATE_MATCHES_BLOCK_11 = all(
    torch.equal(
        BLOCK_12_LEARNED_EFFECTIVE_WEIGHTS[pathway_name],
        NOTEBOOK_09_TRAINED_EFFECTIVE_TRANSFORMATIVE_WEIGHTS[pathway_name],
    )
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
)


if not BLOCK_12_EFFECTIVE_WEIGHT_STATE_MATCHES_BLOCK_11:
    raise RuntimeError(
        "Learned effective Transformative Weights do not reproduce "
        "the final Block 11 state."
    )


# =============================================================================
# Learned-weight numerical and structural audit
# =============================================================================

BLOCK_12_EFFECTIVE_WEIGHTS_FINITE = {}
BLOCK_12_EFFECTIVE_WEIGHTS_BOUNDED = {}
BLOCK_12_DEFERRED_WEIGHTS_ZERO = {}
BLOCK_12_ACTIVE_WEIGHT_COUNTS = {}


for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
    learned_weight = BLOCK_12_LEARNED_EFFECTIVE_WEIGHTS[pathway_name]

    active_indices = torch.tensor(
        BLOCK_12_ACTIVE_DIMENSION_INDICES[pathway_name],
        dtype=torch.long,
    )

    deferred_indices = torch.tensor(
        BLOCK_12_DEFERRED_DIMENSION_INDICES[pathway_name],
        dtype=torch.long,
    )

    BLOCK_12_ACTIVE_WEIGHT_COUNTS[pathway_name] = int(
        active_indices.numel()
    )

    BLOCK_12_EFFECTIVE_WEIGHTS_FINITE[pathway_name] = bool(
        torch.isfinite(learned_weight).all().item()
    )

    BLOCK_12_EFFECTIVE_WEIGHTS_BOUNDED[pathway_name] = bool(
        ((learned_weight >= 0.0) & (learned_weight <= 1.0)).all().item()
    )

    BLOCK_12_DEFERRED_WEIGHTS_ZERO[pathway_name] = torch.equal(
        learned_weight[deferred_indices],
        torch.zeros_like(learned_weight[deferred_indices]),
    )


BLOCK_12_ALL_EFFECTIVE_WEIGHTS_FINITE = all(
    BLOCK_12_EFFECTIVE_WEIGHTS_FINITE.values()
)

BLOCK_12_ALL_EFFECTIVE_WEIGHTS_BOUNDED = all(
    BLOCK_12_EFFECTIVE_WEIGHTS_BOUNDED.values()
)

BLOCK_12_ALL_DEFERRED_WEIGHTS_ZERO = all(
    BLOCK_12_DEFERRED_WEIGHTS_ZERO.values()
)


if not all(
    [
        BLOCK_12_ALL_EFFECTIVE_WEIGHTS_FINITE,
        BLOCK_12_ALL_EFFECTIVE_WEIGHTS_BOUNDED,
        BLOCK_12_ALL_DEFERRED_WEIGHTS_ZERO,
    ]
):
    raise RuntimeError(
        "Learned Transformative-Weight state failed post-training "
        "numerical or structural validation."
    )


# =============================================================================
# Fresh deterministic article reset helper
# =============================================================================

def block_12_construct_fresh_article_states():
    return {
        pathway_name: torch.zeros(
            (1, pathway_dim),
            dtype=DEFAULT_DTYPE,
            device=DEVICE,
            requires_grad=False,
        )
        for pathway_name, pathway_dim in BLOCK_12_PATHWAY_DIMENSIONS.items()
    }


# =============================================================================
# Controlled post-training corpus trajectory
# =============================================================================

def block_12_execute_post_training_trajectories():
    corpus_trajectories = {}

    for article_id in BLOCK_12_ARTICLE_IDS:
        article_states = block_12_construct_fresh_article_states()

        article_trajectory = {
            pathway_name: {
                "initial_state": article_states[pathway_name]
                .detach()
                .clone(),
                "steps": [],
            }
            for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
        }

        for record_index in BLOCK_12_ARTICLE_RECORD_INDICES[article_id]:
            input_record = NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[
                record_index
            ]

            for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
                current_representation = input_record[
                    BLOCK_12_REPRESENTATION_FIELDS[pathway_name]
                ].to(
                    device=DEVICE,
                    dtype=DEFAULT_DTYPE,
                )

                previous_state = article_states[pathway_name]

                candidate = BLOCK_12_TRANSFORMATIVE_MODULES[pathway_name](
                    current_representation,
                    previous_state,
                )

                effective_weight = NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                    pathway_name
                ]()

                weighted_candidate = (
                    effective_weight.unsqueeze(0) * candidate
                )

                updated_state = previous_state + weighted_candidate

                article_trajectory[pathway_name]["steps"].append(
                    {
                        "article_id": input_record["article_id"],
                        "sentence_id": input_record["sentence_id"],
                        "sentence_index": int(input_record["sentence_index"]),
                        "sequence_position": int(
                            input_record["sequence_position"]
                        ),
                        "record_index": int(record_index),
                        "current_representation": current_representation
                        .detach()
                        .clone(),
                        "previous_state": previous_state.detach().clone(),
                        "candidate": candidate.detach().clone(),
                        "effective_weight": effective_weight.detach().clone(),
                        "weighted_candidate": weighted_candidate
                        .detach()
                        .clone(),
                        "updated_state": updated_state.detach().clone(),
                    }
                )

                article_states[pathway_name] = updated_state.detach().clone()

        for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
            article_trajectory[pathway_name]["final_state"] = (
                article_states[pathway_name].detach().clone()
            )

        corpus_trajectories[article_id] = article_trajectory

    return corpus_trajectories


# =============================================================================
# Execute post-training corpus trajectory twice
# =============================================================================

with torch.no_grad():
    BLOCK_12_FIRST_TRAJECTORY = (
        block_12_execute_post_training_trajectories()
    )

    BLOCK_12_SECOND_TRAJECTORY = (
        block_12_execute_post_training_trajectories()
    )


# =============================================================================
# Execution flags
# =============================================================================

NOTEBOOK_09_BLOCK_12_REPRESENTATION_FORWARD_EXECUTED = False
NOTEBOOK_09_BLOCK_12_TRANSFORMATIVE_FORWARD_EXECUTED = True
NOTEBOOK_09_BLOCK_12_WEIGHT_FORWARD_EXECUTED = True
NOTEBOOK_09_BLOCK_12_WEIGHTED_CANDIDATE_EXECUTED = True
NOTEBOOK_09_BLOCK_12_RECURRENT_UPDATE_EXECUTED = True
NOTEBOOK_09_BLOCK_12_ARTICLE_TRAJECTORY_EXECUTED = True
NOTEBOOK_09_BLOCK_12_FUTURE_CONTEXT_USED = False
NOTEBOOK_09_BLOCK_12_CROSS_ARTICLE_STATE_PROPAGATION = False


# =============================================================================
# Initial-state and cross-article reset validation
# =============================================================================

BLOCK_12_INITIAL_STATES_ZERO = {}
BLOCK_12_INITIAL_STATE_STORAGE_INDEPENDENT = {}
BLOCK_12_CROSS_ARTICLE_RESET_VALID = True


for article_id in BLOCK_12_ARTICLE_IDS:
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
        validation_key = (article_id, pathway_name)

        first_initial = BLOCK_12_FIRST_TRAJECTORY[article_id][
            pathway_name
        ]["initial_state"]

        second_initial = BLOCK_12_SECOND_TRAJECTORY[article_id][
            pathway_name
        ]["initial_state"]

        BLOCK_12_INITIAL_STATES_ZERO[validation_key] = all(
            [
                torch.equal(
                    first_initial,
                    torch.zeros_like(first_initial),
                ),
                torch.equal(
                    second_initial,
                    torch.zeros_like(second_initial),
                ),
            ]
        )

        BLOCK_12_INITIAL_STATE_STORAGE_INDEPENDENT[validation_key] = (
            first_initial.data_ptr() != second_initial.data_ptr()
        )


for article_index in range(1, len(BLOCK_12_ARTICLE_IDS)):
    article_id = BLOCK_12_ARTICLE_IDS[article_index]
    previous_article_id = BLOCK_12_ARTICLE_IDS[article_index - 1]

    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
        current_initial = BLOCK_12_FIRST_TRAJECTORY[article_id][
            pathway_name
        ]["initial_state"]

        previous_final = BLOCK_12_FIRST_TRAJECTORY[previous_article_id][
            pathway_name
        ]["final_state"]

        BLOCK_12_CROSS_ARTICLE_RESET_VALID = all(
            [
                BLOCK_12_CROSS_ARTICLE_RESET_VALID,
                torch.equal(
                    current_initial,
                    torch.zeros_like(current_initial),
                ),
                current_initial.data_ptr() != previous_final.data_ptr(),
            ]
        )


BLOCK_12_ALL_INITIAL_STATES_ZERO = all(
    BLOCK_12_INITIAL_STATES_ZERO.values()
)

BLOCK_12_ALL_INITIAL_STATE_STORAGE_INDEPENDENT = all(
    BLOCK_12_INITIAL_STATE_STORAGE_INDEPENDENT.values()
)


if not all(
    [
        BLOCK_12_ALL_INITIAL_STATES_ZERO,
        BLOCK_12_ALL_INITIAL_STATE_STORAGE_INDEPENDENT,
        BLOCK_12_CROSS_ARTICLE_RESET_VALID,
    ]
):
    raise RuntimeError(
        "Notebook 09 Block 12 article-reset semantics are invalid."
    )


# =============================================================================
# Article/pathway trajectory validation
# =============================================================================

BLOCK_12_TRAJECTORY_STEP_COUNTS_VALID = {}
BLOCK_12_TRAJECTORY_IDENTITIES_VALID = {}
BLOCK_12_CANDIDATE_SHAPES_VALID = {}
BLOCK_12_CANDIDATES_FINITE = {}
BLOCK_12_WEIGHT_SHAPES_VALID = {}
BLOCK_12_WEIGHTS_FINITE = {}
BLOCK_12_WEIGHTS_BOUNDED = {}
BLOCK_12_WEIGHTS_STABLE_ACROSS_ARTICLE = {}
BLOCK_12_STEP_WEIGHTS_MATCH_LEARNED_STATE = {}
BLOCK_12_WEIGHTED_CANDIDATE_SHAPES_VALID = {}
BLOCK_12_WEIGHTED_CANDIDATES_FINITE = {}
BLOCK_12_DEFERRED_WEIGHTED_CONTRIBUTIONS_ZERO = {}
BLOCK_12_STATE_SHAPES_VALID = {}
BLOCK_12_STATES_FINITE = {}
BLOCK_12_DEFERRED_STATES_ZERO = {}
BLOCK_12_PREVIOUS_STATE_CHAIN_VALID = {}
BLOCK_12_ADDITIVE_UPDATE_VALID = {}
BLOCK_12_CUMULATIVE_STATE_VALID = {}
BLOCK_12_CUMULATIVE_MAX_ABS_DIFFERENCE = {}
BLOCK_12_FINAL_STATE_CUMULATIVE_VALID = {}
BLOCK_12_TRAJECTORY_DETERMINISTIC = {}


for article_id in BLOCK_12_ARTICLE_IDS:
    article_record_indices = BLOCK_12_ARTICLE_RECORD_INDICES[article_id]

    expected_sentence_ids = [
        NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[record_index][
            "sentence_id"
        ]
        for record_index in article_record_indices
    ]

    expected_sentence_indices = [
        NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[record_index][
            "sentence_index"
        ]
        for record_index in article_record_indices
    ]

    expected_sequence_positions = [
        NOTEBOOK_09_ARTICLE_RECURRENT_INPUT_RECORDS[record_index][
            "sequence_position"
        ]
        for record_index in article_record_indices
    ]

    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
        validation_key = (article_id, pathway_name)
        pathway_dim = BLOCK_12_PATHWAY_DIMENSIONS[pathway_name]

        expected_state_shape = (1, pathway_dim)
        expected_weight_shape = (pathway_dim,)

        first = BLOCK_12_FIRST_TRAJECTORY[article_id][pathway_name]
        second = BLOCK_12_SECOND_TRAJECTORY[article_id][pathway_name]
        steps = first["steps"]

        BLOCK_12_TRAJECTORY_STEP_COUNTS_VALID[validation_key] = (
            len(steps) == len(article_record_indices)
        )

        BLOCK_12_TRAJECTORY_IDENTITIES_VALID[validation_key] = all(
            [
                [step["sentence_id"] for step in steps]
                == expected_sentence_ids,
                [step["sentence_index"] for step in steps]
                == expected_sentence_indices,
                [step["sequence_position"] for step in steps]
                == expected_sequence_positions,
                all(step["article_id"] == article_id for step in steps),
            ]
        )

        BLOCK_12_CANDIDATE_SHAPES_VALID[validation_key] = all(
            tuple(step["candidate"].shape) == expected_state_shape
            for step in steps
        )

        BLOCK_12_CANDIDATES_FINITE[validation_key] = all(
            bool(torch.isfinite(step["candidate"]).all().item())
            for step in steps
        )

        BLOCK_12_WEIGHT_SHAPES_VALID[validation_key] = all(
            tuple(step["effective_weight"].shape) == expected_weight_shape
            for step in steps
        )

        BLOCK_12_WEIGHTS_FINITE[validation_key] = all(
            bool(torch.isfinite(step["effective_weight"]).all().item())
            for step in steps
        )

        BLOCK_12_WEIGHTS_BOUNDED[validation_key] = all(
            bool(
                (
                    (step["effective_weight"] >= 0.0)
                    & (step["effective_weight"] <= 1.0)
                ).all().item()
            )
            for step in steps
        )

        first_weight = steps[0]["effective_weight"]

        BLOCK_12_WEIGHTS_STABLE_ACROSS_ARTICLE[validation_key] = all(
            torch.equal(step["effective_weight"], first_weight)
            for step in steps[1:]
        )

        expected_learned_weight = BLOCK_12_LEARNED_EFFECTIVE_WEIGHTS[
            pathway_name
        ].to(
            device=DEVICE,
            dtype=DEFAULT_DTYPE,
        )

        BLOCK_12_STEP_WEIGHTS_MATCH_LEARNED_STATE[validation_key] = all(
            torch.equal(
                step["effective_weight"],
                expected_learned_weight,
            )
            for step in steps
        )

        BLOCK_12_WEIGHTED_CANDIDATE_SHAPES_VALID[validation_key] = all(
            tuple(step["weighted_candidate"].shape) == expected_state_shape
            for step in steps
        )

        BLOCK_12_WEIGHTED_CANDIDATES_FINITE[validation_key] = all(
            bool(torch.isfinite(step["weighted_candidate"]).all().item())
            for step in steps
        )

        deferred_indices = torch.tensor(
            BLOCK_12_DEFERRED_DIMENSION_INDICES[pathway_name],
            dtype=torch.long,
        )

        BLOCK_12_DEFERRED_WEIGHTED_CONTRIBUTIONS_ZERO[
            validation_key
        ] = all(
            torch.equal(
                step["weighted_candidate"][:, deferred_indices],
                torch.zeros_like(
                    step["weighted_candidate"][:, deferred_indices]
                ),
            )
            for step in steps
        )

        all_states = [first["initial_state"]] + [
            step["updated_state"]
            for step in steps
        ]

        BLOCK_12_STATE_SHAPES_VALID[validation_key] = all(
            tuple(state.shape) == expected_state_shape
            for state in all_states
        )

        BLOCK_12_STATES_FINITE[validation_key] = all(
            bool(torch.isfinite(state).all().item())
            for state in all_states
        )

        BLOCK_12_DEFERRED_STATES_ZERO[validation_key] = all(
            torch.equal(
                state[:, deferred_indices],
                torch.zeros_like(state[:, deferred_indices]),
            )
            for state in all_states
        )

        chain_valid = torch.equal(
            steps[0]["previous_state"],
            first["initial_state"],
        )

        for step_index in range(1, len(steps)):
            chain_valid = (
                chain_valid
                and torch.equal(
                    steps[step_index]["previous_state"],
                    steps[step_index - 1]["updated_state"],
                )
            )

        BLOCK_12_PREVIOUS_STATE_CHAIN_VALID[validation_key] = chain_valid

        BLOCK_12_ADDITIVE_UPDATE_VALID[validation_key] = all(
            torch.equal(
                step["updated_state"],
                step["previous_state"] + step["weighted_candidate"],
            )
            for step in steps
        )

        cumulative_state = torch.zeros_like(first["initial_state"])
        cumulative_valid = True
        maximum_difference = 0.0

        for step in steps:
            cumulative_state = (
                cumulative_state + step["weighted_candidate"]
            )

            difference = step["updated_state"] - cumulative_state

            maximum_difference = max(
                maximum_difference,
                float(difference.abs().max().item()),
            )

            cumulative_valid = (
                cumulative_valid
                and torch.allclose(
                    step["updated_state"],
                    cumulative_state,
                    rtol=0.0,
                    atol=1e-7,
                )
            )

        BLOCK_12_CUMULATIVE_STATE_VALID[validation_key] = cumulative_valid
        BLOCK_12_CUMULATIVE_MAX_ABS_DIFFERENCE[
            validation_key
        ] = maximum_difference

        BLOCK_12_FINAL_STATE_CUMULATIVE_VALID[validation_key] = (
            torch.allclose(
                first["final_state"],
                cumulative_state,
                rtol=0.0,
                atol=1e-7,
            )
        )

        deterministic = torch.equal(
            first["initial_state"],
            second["initial_state"],
        )

        for first_step, second_step in zip(
            first["steps"],
            second["steps"],
        ):
            deterministic = all(
                [
                    deterministic,
                    first_step["article_id"] == second_step["article_id"],
                    first_step["sentence_id"] == second_step["sentence_id"],
                    first_step["sentence_index"]
                    == second_step["sentence_index"],
                    first_step["sequence_position"]
                    == second_step["sequence_position"],
                ]
            )

            for tensor_name in (
                "current_representation",
                "previous_state",
                "candidate",
                "effective_weight",
                "weighted_candidate",
                "updated_state",
            ):
                deterministic = (
                    deterministic
                    and torch.equal(
                        first_step[tensor_name],
                        second_step[tensor_name],
                    )
                )

        deterministic = (
            deterministic
            and torch.equal(
                first["final_state"],
                second["final_state"],
            )
        )

        BLOCK_12_TRAJECTORY_DETERMINISTIC[
            validation_key
        ] = deterministic


# =============================================================================
# Aggregate trajectory validation
# =============================================================================

BLOCK_12_TRAJECTORY_STEP_COUNTS_ALL_VALID = all(
    BLOCK_12_TRAJECTORY_STEP_COUNTS_VALID.values()
)
BLOCK_12_ALL_TRAJECTORY_IDENTITIES_VALID = all(
    BLOCK_12_TRAJECTORY_IDENTITIES_VALID.values()
)
BLOCK_12_ALL_CANDIDATE_SHAPES_VALID = all(
    BLOCK_12_CANDIDATE_SHAPES_VALID.values()
)
BLOCK_12_ALL_CANDIDATES_FINITE = all(
    BLOCK_12_CANDIDATES_FINITE.values()
)
BLOCK_12_ALL_WEIGHT_SHAPES_VALID = all(
    BLOCK_12_WEIGHT_SHAPES_VALID.values()
)
BLOCK_12_ALL_WEIGHTS_FINITE = all(
    BLOCK_12_WEIGHTS_FINITE.values()
)
BLOCK_12_ALL_WEIGHTS_BOUNDED = all(
    BLOCK_12_WEIGHTS_BOUNDED.values()
)
BLOCK_12_ALL_WEIGHTS_STABLE_ACROSS_ARTICLE = all(
    BLOCK_12_WEIGHTS_STABLE_ACROSS_ARTICLE.values()
)
BLOCK_12_ALL_STEP_WEIGHTS_MATCH_LEARNED_STATE = all(
    BLOCK_12_STEP_WEIGHTS_MATCH_LEARNED_STATE.values()
)
BLOCK_12_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID = all(
    BLOCK_12_WEIGHTED_CANDIDATE_SHAPES_VALID.values()
)
BLOCK_12_ALL_WEIGHTED_CANDIDATES_FINITE = all(
    BLOCK_12_WEIGHTED_CANDIDATES_FINITE.values()
)
BLOCK_12_ALL_DEFERRED_WEIGHTED_CONTRIBUTIONS_ZERO = all(
    BLOCK_12_DEFERRED_WEIGHTED_CONTRIBUTIONS_ZERO.values()
)
BLOCK_12_ALL_STATE_SHAPES_VALID = all(
    BLOCK_12_STATE_SHAPES_VALID.values()
)
BLOCK_12_ALL_STATES_FINITE = all(
    BLOCK_12_STATES_FINITE.values()
)
BLOCK_12_ALL_DEFERRED_STATES_ZERO = all(
    BLOCK_12_DEFERRED_STATES_ZERO.values()
)
BLOCK_12_ALL_PREVIOUS_STATE_CHAINS_VALID = all(
    BLOCK_12_PREVIOUS_STATE_CHAIN_VALID.values()
)
BLOCK_12_ALL_ADDITIVE_UPDATES_VALID = all(
    BLOCK_12_ADDITIVE_UPDATE_VALID.values()
)
BLOCK_12_ALL_CUMULATIVE_STATES_VALID = all(
    BLOCK_12_CUMULATIVE_STATE_VALID.values()
)
BLOCK_12_ALL_FINAL_STATE_CUMULATIVE_VALID = all(
    BLOCK_12_FINAL_STATE_CUMULATIVE_VALID.values()
)
BLOCK_12_ALL_TRAJECTORIES_DETERMINISTIC = all(
    BLOCK_12_TRAJECTORY_DETERMINISTIC.values()
)


# =============================================================================
# Reconstruct post-training objective across complete corpus
# =============================================================================

BLOCK_12_POSTTRAIN_PATHWAY_LOSSES = {}


for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
    gated_predictions = []

    for article_id in BLOCK_12_ARTICLE_IDS:
        gated_predictions.extend(
            [
                step["weighted_candidate"]
                for step in BLOCK_12_FIRST_TRAJECTORY[article_id][
                    pathway_name
                ]["steps"]
            ]
        )

    gated_prediction_matrix = torch.cat(
        gated_predictions,
        dim=0,
    )

    target_matrix = BLOCK_12_TARGETS[pathway_name]
    objective_mask = BLOCK_12_OBJECTIVE_MASKS[pathway_name]

    if tuple(gated_prediction_matrix.shape) != tuple(target_matrix.shape):
        raise RuntimeError(
            f"Block 12 post-training prediction shape mismatch for "
            f"{pathway_name}: observed "
            f"{tuple(gated_prediction_matrix.shape)}, expected "
            f"{tuple(target_matrix.shape)}."
        )

    squared_error = (
        gated_prediction_matrix - target_matrix
    ).pow(2)

    masked_error = squared_error[objective_mask]

    if masked_error.numel() == 0:
        raise RuntimeError(
            f"No objective-active post-training supervision for "
            f"{pathway_name}."
        )

    BLOCK_12_POSTTRAIN_PATHWAY_LOSSES[pathway_name] = float(
        masked_error.mean().detach().cpu().item()
    )


BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE = float(
    np.mean(
        [
            BLOCK_12_POSTTRAIN_PATHWAY_LOSSES[pathway_name]
            for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
        ]
    )
)


BLOCK_12_POSTTRAIN_PATHWAY_LOSSES_FINITE = all(
    np.isfinite(value)
    for value in BLOCK_12_POSTTRAIN_PATHWAY_LOSSES.values()
)

BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE_FINITE = bool(
    np.isfinite(BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE)
)


# =============================================================================
# Reproduce final Block 11 objective
# =============================================================================

BLOCK_12_BLOCK_11_PATHWAY_LOSSES_MATCH = all(
    np.isclose(
        BLOCK_12_POSTTRAIN_PATHWAY_LOSSES[pathway_name],
        NOTEBOOK_09_TW_FINAL_PATHWAY_LOSSES[pathway_name],
        rtol=0.0,
        atol=1e-7,
    )
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
)


BLOCK_12_BLOCK_11_TOTAL_OBJECTIVE_MATCH = np.isclose(
    BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE,
    NOTEBOOK_09_TW_FINAL_TOTAL_OBJECTIVE,
    rtol=0.0,
    atol=1e-7,
)


BLOCK_12_BLOCK_11_OBJECTIVE_REPRODUCED = all(
    [
        BLOCK_12_BLOCK_11_PATHWAY_LOSSES_MATCH,
        BLOCK_12_BLOCK_11_TOTAL_OBJECTIVE_MATCH,
    ]
)


if not BLOCK_12_BLOCK_11_OBJECTIVE_REPRODUCED:
    raise RuntimeError(
        "Post-training recurrent execution does not reproduce "
        "the final Block 11 Transformative-Weight objective."
    )


# =============================================================================
# Pre-training versus post-training comparison
# =============================================================================

BLOCK_12_PRETRAIN_TOTAL_OBJECTIVE = float(
    NOTEBOOK_09_TW_PRETRAIN_TOTAL_OBJECTIVE
)

BLOCK_12_TOTAL_OBJECTIVE_DELTA = (
    BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE
    - BLOCK_12_PRETRAIN_TOTAL_OBJECTIVE
)

BLOCK_12_TOTAL_OBJECTIVE_IMPROVED = (
    BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE
    < BLOCK_12_PRETRAIN_TOTAL_OBJECTIVE
)

BLOCK_12_PATHWAY_OBJECTIVE_DELTAS = {
    pathway_name: (
        BLOCK_12_POSTTRAIN_PATHWAY_LOSSES[pathway_name]
        - NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES[pathway_name]
    )
    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
}


# =============================================================================
# Causal post-training contract
# =============================================================================

BLOCK_12_CAUSAL_CONTRACT_VALID = all(
    [
        NOTEBOOK_09_BLOCK_12_FUTURE_CONTEXT_USED is False,
        NOTEBOOK_09_BLOCK_12_CROSS_ARTICLE_STATE_PROPAGATION is False,
        BLOCK_12_CROSS_ARTICLE_RESET_VALID,
        BLOCK_12_ALL_PREVIOUS_STATE_CHAINS_VALID,
        BLOCK_12_ALL_TRAJECTORY_IDENTITIES_VALID,
        BLOCK_12_CANONICAL_IDENTITY_VALID,
    ]
)


# =============================================================================
# Parameter and gradient boundary
# =============================================================================

NOTEBOOK_09_BLOCK_12_OPTIMIZER_CREATED = False
NOTEBOOK_09_BLOCK_12_BACKWARD_PASS_EXECUTED = False
NOTEBOOK_09_BLOCK_12_GRADIENT_CLIPPING_EXECUTED = False
NOTEBOOK_09_BLOCK_12_OPTIMIZER_STEP_EXECUTED = False
NOTEBOOK_09_BLOCK_12_PARAMETER_UPDATE_EXECUTED = False
NOTEBOOK_09_BLOCK_12_TRAINING_EXECUTED = False


BLOCK_12_ALL_GRADIENTS_ABSENT = all(
    parameter.grad is None
    for module_collection in (
        BLOCK_1_REPRESENTATION_MODULES,
        BLOCK_1_TRANSFORMATIVE_MODULES,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES,
    )
    for module in module_collection.values()
    for parameter in module.parameters()
)


if not BLOCK_12_ALL_GRADIENTS_ABSENT:
    raise RuntimeError(
        "Unexpected residual gradients are present during post-training "
        "validation."
    )


# =============================================================================
# Snapshot complete learned architecture after Block 12 validation
# =============================================================================

BLOCK_12_REPRESENTATION_STATE_AFTER = (
    block_12_snapshot_module_collection(BLOCK_1_REPRESENTATION_MODULES)
)

BLOCK_12_TRANSFORMATIVE_STATE_AFTER = (
    block_12_snapshot_module_collection(BLOCK_1_TRANSFORMATIVE_MODULES)
)

BLOCK_12_WEIGHT_STATE_AFTER = (
    block_12_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Exact parameter immutability
# =============================================================================

BLOCK_12_REPRESENTATION_UNCHANGED = block_12_nested_state_exact(
    BLOCK_12_REPRESENTATION_STATE_BEFORE,
    BLOCK_12_REPRESENTATION_STATE_AFTER,
)

BLOCK_12_TRANSFORMATIVE_UNCHANGED = block_12_nested_state_exact(
    BLOCK_12_TRANSFORMATIVE_STATE_BEFORE,
    BLOCK_12_TRANSFORMATIVE_STATE_AFTER,
)

BLOCK_12_WEIGHT_STATE_UNCHANGED = block_12_nested_state_exact(
    BLOCK_12_WEIGHT_STATE_BEFORE,
    BLOCK_12_WEIGHT_STATE_AFTER,
)


if not all(
    [
        BLOCK_12_REPRESENTATION_UNCHANGED,
        BLOCK_12_TRANSFORMATIVE_UNCHANGED,
        BLOCK_12_WEIGHT_STATE_UNCHANGED,
    ]
):
    raise RuntimeError(
        "Notebook 09 Block 12 changed learned model state during validation."
    )


# =============================================================================
# Post-training trainability / freeze validation
# =============================================================================

BLOCK_12_REPRESENTATION_STILL_FROZEN = all(
    not parameter.requires_grad
    for module in BLOCK_1_REPRESENTATION_MODULES.values()
    for parameter in module.parameters()
)

BLOCK_12_TRANSFORMATIVE_STILL_FROZEN = all(
    not parameter.requires_grad
    for module in BLOCK_1_TRANSFORMATIVE_MODULES.values()
    for parameter in module.parameters()
)

BLOCK_12_WEIGHT_PARAMETERS_STILL_TRAINABLE = all(
    parameter.requires_grad
    for module in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()
    for parameter in module.parameters()
)


# =============================================================================
# Dynamic parameter accounting
# =============================================================================

BLOCK_12_PARAMETER_ACCOUNTING_VALID = all(
    [
        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,
        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,
        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            == NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            + NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),
        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            == NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),
        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            == BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),
        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            == NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            + NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


# =============================================================================
# Final recurrent states and validated trajectories
# =============================================================================

NOTEBOOK_09_POSTTRAIN_FINAL_RECURRENT_STATES = {
    article_id: {
        pathway_name: BLOCK_12_FIRST_TRAJECTORY[article_id][pathway_name][
            "final_state"
        ].detach().cpu().clone()
        for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
    }
    for article_id in BLOCK_12_ARTICLE_IDS
}


NOTEBOOK_09_POSTTRAIN_RECURRENT_TRAJECTORY = {
    article_id: {
        pathway_name: {
            "initial_state": BLOCK_12_FIRST_TRAJECTORY[article_id][
                pathway_name
            ]["initial_state"].detach().cpu().clone(),
            "steps": [
                {
                    key: (
                        value.detach().cpu().clone()
                        if torch.is_tensor(value)
                        else value
                    )
                    for key, value in step.items()
                }
                for step in BLOCK_12_FIRST_TRAJECTORY[article_id][
                    pathway_name
                ]["steps"]
            ],
            "final_state": BLOCK_12_FIRST_TRAJECTORY[article_id][
                pathway_name
            ]["final_state"].detach().cpu().clone(),
        }
        for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS
    }
    for article_id in BLOCK_12_ARTICLE_IDS
}


NOTEBOOK_09_POSTTRAIN_PATHWAY_LOSSES = deepcopy(
    BLOCK_12_POSTTRAIN_PATHWAY_LOSSES
)

NOTEBOOK_09_POSTTRAIN_TOTAL_OBJECTIVE = BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE
NOTEBOOK_09_POSTTRAIN_OBJECTIVE_DELTA = BLOCK_12_TOTAL_OBJECTIVE_DELTA


# =============================================================================
# Final Block 12 validation
# =============================================================================

NOTEBOOK_09_BLOCK_12_ERRORS = []

BLOCK_12_VALIDATION_CHECKS = {
    "prerequisites_invalid": BLOCK_12_PREREQUISITES_VALID,
    "pathway_dimensions_invalid": BLOCK_12_PATHWAY_DIMENSIONS_VALID,
    "dimension_partitions_invalid": BLOCK_12_ALL_DIMENSION_PARTITIONS_VALID,
    "canonical_identity_invalid": BLOCK_12_CANONICAL_IDENTITY_VALID,
    "article_groups_invalid": BLOCK_12_ARTICLE_GROUPS_VALID,
    "target_shapes_invalid": BLOCK_12_TARGET_SHAPES_VALID,
    "objective_mask_shapes_invalid": BLOCK_12_OBJECTIVE_MASK_SHAPES_VALID,
    "objective_active_elements_invalid": BLOCK_12_OBJECTIVE_ACTIVE_ELEMENTS_VALID,
    "current_weight_state_not_block_11": BLOCK_12_CURRENT_WEIGHT_STATE_MATCHES_BLOCK_11,
    "effective_weight_state_not_block_11": BLOCK_12_EFFECTIVE_WEIGHT_STATE_MATCHES_BLOCK_11,
    "effective_weights_not_finite": BLOCK_12_ALL_EFFECTIVE_WEIGHTS_FINITE,
    "effective_weights_not_bounded": BLOCK_12_ALL_EFFECTIVE_WEIGHTS_BOUNDED,
    "deferred_weights_not_zero": BLOCK_12_ALL_DEFERRED_WEIGHTS_ZERO,
    "initial_states_not_zero": BLOCK_12_ALL_INITIAL_STATES_ZERO,
    "initial_reset_storage_not_independent": BLOCK_12_ALL_INITIAL_STATE_STORAGE_INDEPENDENT,
    "cross_article_reset_invalid": BLOCK_12_CROSS_ARTICLE_RESET_VALID,
    "trajectory_step_count_invalid": BLOCK_12_TRAJECTORY_STEP_COUNTS_ALL_VALID,
    "trajectory_identity_invalid": BLOCK_12_ALL_TRAJECTORY_IDENTITIES_VALID,
    "candidate_shapes_invalid": BLOCK_12_ALL_CANDIDATE_SHAPES_VALID,
    "candidate_values_not_finite": BLOCK_12_ALL_CANDIDATES_FINITE,
    "weight_shapes_invalid": BLOCK_12_ALL_WEIGHT_SHAPES_VALID,
    "weights_not_finite": BLOCK_12_ALL_WEIGHTS_FINITE,
    "weights_not_bounded": BLOCK_12_ALL_WEIGHTS_BOUNDED,
    "weights_not_stable_across_article": BLOCK_12_ALL_WEIGHTS_STABLE_ACROSS_ARTICLE,
    "step_weights_do_not_match_learned_state": BLOCK_12_ALL_STEP_WEIGHTS_MATCH_LEARNED_STATE,
    "weighted_candidate_shapes_invalid": BLOCK_12_ALL_WEIGHTED_CANDIDATE_SHAPES_VALID,
    "weighted_candidate_values_not_finite": BLOCK_12_ALL_WEIGHTED_CANDIDATES_FINITE,
    "deferred_weighted_contribution_nonzero": BLOCK_12_ALL_DEFERRED_WEIGHTED_CONTRIBUTIONS_ZERO,
    "state_shapes_invalid": BLOCK_12_ALL_STATE_SHAPES_VALID,
    "state_values_not_finite": BLOCK_12_ALL_STATES_FINITE,
    "deferred_states_nonzero": BLOCK_12_ALL_DEFERRED_STATES_ZERO,
    "previous_state_chain_invalid": BLOCK_12_ALL_PREVIOUS_STATE_CHAINS_VALID,
    "additive_update_invalid": BLOCK_12_ALL_ADDITIVE_UPDATES_VALID,
    "cumulative_state_identity_invalid": BLOCK_12_ALL_CUMULATIVE_STATES_VALID,
    "final_state_cumulative_invalid": BLOCK_12_ALL_FINAL_STATE_CUMULATIVE_VALID,
    "trajectory_not_deterministic": BLOCK_12_ALL_TRAJECTORIES_DETERMINISTIC,
    "posttrain_pathway_losses_not_finite": BLOCK_12_POSTTRAIN_PATHWAY_LOSSES_FINITE,
    "posttrain_total_objective_not_finite": BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE_FINITE,
    "block_11_objective_not_reproduced": BLOCK_12_BLOCK_11_OBJECTIVE_REPRODUCED,
    "objective_not_improved_from_pretrain": BLOCK_12_TOTAL_OBJECTIVE_IMPROVED,
    "causal_contract_invalid": BLOCK_12_CAUSAL_CONTRACT_VALID,
    "unexpected_gradients": BLOCK_12_ALL_GRADIENTS_ABSENT,
    "representation_state_changed": BLOCK_12_REPRESENTATION_UNCHANGED,
    "transformative_state_changed": BLOCK_12_TRANSFORMATIVE_UNCHANGED,
    "weight_state_changed": BLOCK_12_WEIGHT_STATE_UNCHANGED,
    "representation_not_frozen": BLOCK_12_REPRESENTATION_STILL_FROZEN,
    "transformative_not_frozen": BLOCK_12_TRANSFORMATIVE_STILL_FROZEN,
    "weight_parameters_not_trainable": BLOCK_12_WEIGHT_PARAMETERS_STILL_TRAINABLE,
    "parameter_accounting_invalid": BLOCK_12_PARAMETER_ACCOUNTING_VALID,
}


for error_name, condition in BLOCK_12_VALIDATION_CHECKS.items():
    if not condition:
        NOTEBOOK_09_BLOCK_12_ERRORS.append(error_name)


# =============================================================================
# Prohibited-operation audit
# =============================================================================

BLOCK_12_PROHIBITED_OPERATIONS = {
    "optimizer_incorrectly_created": NOTEBOOK_09_BLOCK_12_OPTIMIZER_CREATED,
    "backward_pass_incorrectly_executed": NOTEBOOK_09_BLOCK_12_BACKWARD_PASS_EXECUTED,
    "gradient_clipping_incorrectly_executed": NOTEBOOK_09_BLOCK_12_GRADIENT_CLIPPING_EXECUTED,
    "optimizer_step_incorrectly_executed": NOTEBOOK_09_BLOCK_12_OPTIMIZER_STEP_EXECUTED,
    "parameter_update_incorrectly_executed": NOTEBOOK_09_BLOCK_12_PARAMETER_UPDATE_EXECUTED,
    "training_incorrectly_executed": NOTEBOOK_09_BLOCK_12_TRAINING_EXECUTED,
    "representation_forward_incorrectly_executed": NOTEBOOK_09_BLOCK_12_REPRESENTATION_FORWARD_EXECUTED,
    "future_context_incorrectly_used": NOTEBOOK_09_BLOCK_12_FUTURE_CONTEXT_USED,
    "cross_article_state_incorrectly_propagated": NOTEBOOK_09_BLOCK_12_CROSS_ARTICLE_STATE_PROPAGATION,
}


for error_name, operation_executed in BLOCK_12_PROHIBITED_OPERATIONS.items():
    if operation_executed:
        NOTEBOOK_09_BLOCK_12_ERRORS.append(error_name)


# =============================================================================
# Final Block 12 state
# =============================================================================

NOTEBOOK_09_POSTTRAIN_RECURRENT_VALIDATION_VALID = (
    len(NOTEBOOK_09_BLOCK_12_ERRORS) == 0
)

NOTEBOOK_09_ARCHITECTURAL_COMPLETION_AUDIT_READY = (
    NOTEBOOK_09_POSTTRAIN_RECURRENT_VALIDATION_VALID
)

NOTEBOOK_09_BLOCK_12_VALID = (
    NOTEBOOK_09_POSTTRAIN_RECURRENT_VALIDATION_VALID
)


if not NOTEBOOK_09_BLOCK_12_VALID:
    raise RuntimeError(
        "Notebook 09 Block 12 post-training recurrent validation failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_12_ERRORS}"
    )


NOTEBOOK_09_BLOCK_12_COMPLETE = True


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_12_SUMMARY = {
    "block": NOTEBOOK_09_BLOCK_12,
    "block_name": NOTEBOOK_09_BLOCK_12_NAME,
    "block_version": NOTEBOOK_09_BLOCK_12_VERSION,
    "article_ids": tuple(BLOCK_12_ARTICLE_IDS),
    "article_count": BLOCK_12_ARTICLE_COUNT,
    "sentence_count": BLOCK_12_SENTENCE_COUNT,
    "article_record_counts": deepcopy(BLOCK_12_ARTICLE_RECORD_COUNTS),
    "learned_weight_state_matches_block_11": BLOCK_12_CURRENT_WEIGHT_STATE_MATCHES_BLOCK_11,
    "effective_weight_state_matches_block_11": BLOCK_12_EFFECTIVE_WEIGHT_STATE_MATCHES_BLOCK_11,
    "cross_article_reset_valid": BLOCK_12_CROSS_ARTICLE_RESET_VALID,
    "trajectory_deterministic": BLOCK_12_ALL_TRAJECTORIES_DETERMINISTIC,
    "causal_contract_valid": BLOCK_12_CAUSAL_CONTRACT_VALID,
    "pretrain_total_objective": BLOCK_12_PRETRAIN_TOTAL_OBJECTIVE,
    "posttrain_total_objective": BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE,
    "objective_delta": BLOCK_12_TOTAL_OBJECTIVE_DELTA,
    "objective_improved": BLOCK_12_TOTAL_OBJECTIVE_IMPROVED,
    "posttrain_pathway_losses": deepcopy(BLOCK_12_POSTTRAIN_PATHWAY_LOSSES),
    "block_11_objective_reproduced": BLOCK_12_BLOCK_11_OBJECTIVE_REPRODUCED,
    "representation_unchanged": BLOCK_12_REPRESENTATION_UNCHANGED,
    "transformative_unchanged": BLOCK_12_TRANSFORMATIVE_UNCHANGED,
    "weight_state_unchanged": BLOCK_12_WEIGHT_STATE_UNCHANGED,
    "posttrain_validation_valid": NOTEBOOK_09_POSTTRAIN_RECURRENT_VALIDATION_VALID,
    "architectural_completion_audit_ready": NOTEBOOK_09_ARCHITECTURAL_COMPLETION_AUDIT_READY,
    "block_valid": NOTEBOOK_09_BLOCK_12_VALID,
    "block_complete": NOTEBOOK_09_BLOCK_12_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)
print(
    "Media AI — Notebook 09, Block 12: "
    "Post-Training Recurrent Trajectory Validation "
    "and Learned Transformative-Weight Audit"
)
print("=" * 72)
print(f"Block version                : {NOTEBOOK_09_BLOCK_12_VERSION}")
print("-" * 72)

print("Learned Transformative-Weight state")
print(f"State matches Block 11       : {BLOCK_12_CURRENT_WEIGHT_STATE_MATCHES_BLOCK_11}")
print(f"Effective state matches      : {BLOCK_12_EFFECTIVE_WEIGHT_STATE_MATCHES_BLOCK_11}")

for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
    print(f"{pathway_name:<14} active weights     : {BLOCK_12_ACTIVE_WEIGHT_COUNTS[pathway_name]}")
    print(f"{pathway_name:<14} finite             : {BLOCK_12_EFFECTIVE_WEIGHTS_FINITE[pathway_name]}")
    print(f"{pathway_name:<14} bounded            : {BLOCK_12_EFFECTIVE_WEIGHTS_BOUNDED[pathway_name]}")
    print(f"{pathway_name:<14} deferred zero      : {BLOCK_12_DEFERRED_WEIGHTS_ZERO[pathway_name]}")

print("-" * 72)
print("Post-training canonical corpus traversal")
print(f"Articles traversed           : {BLOCK_12_ARTICLE_COUNT}")
print(f"Sentences traversed          : {BLOCK_12_SENTENCE_COUNT}")
print(f"Canonical identity valid     : {BLOCK_12_CANONICAL_IDENTITY_VALID}")
print(f"Fresh zero initial states    : {BLOCK_12_ALL_INITIAL_STATES_ZERO}")
print(f"Independent reset storage    : {BLOCK_12_ALL_INITIAL_STATE_STORAGE_INDEPENDENT}")
print(f"Cross-article reset valid    : {BLOCK_12_CROSS_ARTICLE_RESET_VALID}")

print("-" * 72)
print("Post-training article/pathway recurrent validation")

for article_id in BLOCK_12_ARTICLE_IDS:
    print(f"Article                      : {article_id}")

    for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
        validation_key = (article_id, pathway_name)

        print(f"{pathway_name:<14} candidate shape    : {BLOCK_12_CANDIDATE_SHAPES_VALID[validation_key]}")
        print(f"{pathway_name:<14} candidate finite   : {BLOCK_12_CANDIDATES_FINITE[validation_key]}")
        print(f"{pathway_name:<14} learned weight used: {BLOCK_12_STEP_WEIGHTS_MATCH_LEARNED_STATE[validation_key]}")
        print(f"{pathway_name:<14} state shapes       : {BLOCK_12_STATE_SHAPES_VALID[validation_key]}")
        print(f"{pathway_name:<14} states finite      : {BLOCK_12_STATES_FINITE[validation_key]}")
        print(f"{pathway_name:<14} chain valid        : {BLOCK_12_PREVIOUS_STATE_CHAIN_VALID[validation_key]}")
        print(f"{pathway_name:<14} additive update    : {BLOCK_12_ADDITIVE_UPDATE_VALID[validation_key]}")
        print(f"{pathway_name:<14} deferred state zero: {BLOCK_12_DEFERRED_STATES_ZERO[validation_key]}")
        print(f"{pathway_name:<14} cumulative valid   : {BLOCK_12_CUMULATIVE_STATE_VALID[validation_key]}")
        print(f"{pathway_name:<14} cumulative max diff: {BLOCK_12_CUMULATIVE_MAX_ABS_DIFFERENCE[validation_key]:.10f}")
        print(f"{pathway_name:<14} deterministic      : {BLOCK_12_TRAJECTORY_DETERMINISTIC[validation_key]}")

print("-" * 72)
print("Objective reproduction")

for pathway_name in BLOCK_12_PATHWAY_DIMENSIONS:
    print(f"{pathway_name:<14} pretrain loss      : {NOTEBOOK_09_TW_PRETRAIN_PATHWAY_LOSSES[pathway_name]:.8f}")
    print(f"{pathway_name:<14} posttrain loss     : {BLOCK_12_POSTTRAIN_PATHWAY_LOSSES[pathway_name]:.8f}")
    print(f"{pathway_name:<14} loss delta         : {BLOCK_12_PATHWAY_OBJECTIVE_DELTAS[pathway_name]:.8f}")

print(f"Pretrain total objective     : {BLOCK_12_PRETRAIN_TOTAL_OBJECTIVE:.8f}")
print(f"Posttrain total objective    : {BLOCK_12_POSTTRAIN_TOTAL_OBJECTIVE:.8f}")
print(f"Objective delta              : {BLOCK_12_TOTAL_OBJECTIVE_DELTA:.8f}")
print(f"Objective improved           : {BLOCK_12_TOTAL_OBJECTIVE_IMPROVED}")
print(f"Block 11 objective reproduced: {BLOCK_12_BLOCK_11_OBJECTIVE_REPRODUCED}")

print("-" * 72)
print("Parameter and state validation")
print(f"Representation parameters    : {NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT:,}")
print(f"Transformative parameters    : {NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT:,}")
print(f"Transformative Weight params : {NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT}")
print(f"Total model parameters       : {NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION:,}")
print(f"Representation unchanged     : {BLOCK_12_REPRESENTATION_UNCHANGED}")
print(f"Transformative unchanged     : {BLOCK_12_TRANSFORMATIVE_UNCHANGED}")
print(f"Weight state unchanged       : {BLOCK_12_WEIGHT_STATE_UNCHANGED}")
print(f"Representation frozen        : {BLOCK_12_REPRESENTATION_STILL_FROZEN}")
print(f"Transformative frozen        : {BLOCK_12_TRANSFORMATIVE_STILL_FROZEN}")
print(f"Weight parameters trainable  : {BLOCK_12_WEIGHT_PARAMETERS_STILL_TRAINABLE}")
print(f"All gradients absent         : {BLOCK_12_ALL_GRADIENTS_ABSENT}")

print("-" * 72)
print("Execution boundary")
print(f"Representation forward       : {NOTEBOOK_09_BLOCK_12_REPRESENTATION_FORWARD_EXECUTED}")
print(f"Transform forward            : {NOTEBOOK_09_BLOCK_12_TRANSFORMATIVE_FORWARD_EXECUTED}")
print(f"Weight forward               : {NOTEBOOK_09_BLOCK_12_WEIGHT_FORWARD_EXECUTED}")
print(f"Weighted candidate executed  : {NOTEBOOK_09_BLOCK_12_WEIGHTED_CANDIDATE_EXECUTED}")
print(f"Recurrent update executed    : {NOTEBOOK_09_BLOCK_12_RECURRENT_UPDATE_EXECUTED}")
print(f"Article trajectories executed: {NOTEBOOK_09_BLOCK_12_ARTICLE_TRAJECTORY_EXECUTED}")
print(f"Optimizer created            : {NOTEBOOK_09_BLOCK_12_OPTIMIZER_CREATED}")
print(f"Backward pass executed       : {NOTEBOOK_09_BLOCK_12_BACKWARD_PASS_EXECUTED}")
print(f"Parameter update executed    : {NOTEBOOK_09_BLOCK_12_PARAMETER_UPDATE_EXECUTED}")
print(f"Future context used          : {NOTEBOOK_09_BLOCK_12_FUTURE_CONTEXT_USED}")
print(f"Cross-article propagation    : {NOTEBOOK_09_BLOCK_12_CROSS_ARTICLE_STATE_PROPAGATION}")

print("-" * 72)
print(f"Post-training validation valid: {NOTEBOOK_09_POSTTRAIN_RECURRENT_VALIDATION_VALID}")
print(f"Architecture audit ready      : {NOTEBOOK_09_ARCHITECTURAL_COMPLETION_AUDIT_READY}")
print(f"Block valid                   : {NOTEBOOK_09_BLOCK_12_VALID}")
print(f"Block complete                : {NOTEBOOK_09_BLOCK_12_COMPLETE}")
print("=" * 72)

print(
    "The learned Transformative-Weight state produced by Block 11 was "
    "recovered exactly and used throughout the complete canonical "
    "post-training corpus trajectory."
)

print(
    "All learned effective Transformative Weights remained finite and "
    "bounded, while all deferred dimensions remained exactly zero."
)

print(
    f"The complete corpus of {BLOCK_12_SENTENCE_COUNT} sentence(s) across "
    f"{BLOCK_12_ARTICLE_COUNT} article(s) was re-executed with a fresh zero "
    "recurrent state at every article boundary."
)

print(
    "Candidate transformations, learned gated contributions and recurrent "
    "states preserved pathway dimensionality, numerical finiteness, "
    "within-article causal state continuity and additive recurrent identities."
)

print(
    "No recurrent state was propagated across article boundaries."
)

print(
    "Repeated complete post-training corpus trajectories were exactly "
    "deterministic."
)

print(
    "The independently reconstructed post-training objective reproduced "
    "the final Block 11 controlled-training objective."
)

print(
    f"The inherited {NOTEBOOK_09_INHERITED_PARAMETER_COUNT:,} representation "
    "and transformative parameters remained frozen and exactly unchanged, "
    f"and the learned {NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT} "
    "Transformative-Weight parameter(s) remained exactly unchanged during "
    "validation."
)

print(
    "No optimiser, backward pass, gradient clipping, optimiser step or "
    "parameter update occurred."
)

print(
    "Notebook 09 may now proceed to final architectural completion audit "
    "and persistent Notebook 09 handover."
)

print("=" * 72)

Media AI — Notebook 09, Block 12: Post-Training Recurrent Trajectory Validation and Learned Transformative-Weight Audit
Block version                : 1.1
------------------------------------------------------------------------
Learned Transformative-Weight state
State matches Block 11       : True
Effective state matches      : True
factual        active weights     : 10
factual        finite             : True
factual        bounded            : True
factual        deferred zero      : True
psychological  active weights     : 34
psychological  finite             : True
psychological  bounded            : True
psychological  deferred zero      : True
social         active weights     : 5
social         finite             : True
social         bounded            : True
social         deferred zero      : True
------------------------------------------------------------------------
Post-training canonical corpus traversal
Articles traversed           : 3
Sentences traversed          : 5

## Block 13 — Final Architectural Completion Audit and Persistent Notebook 09 → Notebook 10 Handover

### Purpose

This block completes Notebook 09 by performing the final architectural audit of the fully instantiated Media AI recurrent architecture and persisting the complete validated state for the expanded-corpus training phase.

The block does not perform additional training.

Instead, it verifies that all architectural components constructed and validated across Notebook 09 are mutually consistent, freezes the completed model state for handover, persists the required checkpoint and metadata artefacts, reads them back from persistent storage, and validates that the saved state reproduces the in-memory completion state exactly.

The resulting handover must allow Notebook 10 to initialise directly from the completed Notebook 09 architecture without rediscovering or reconstructing upstream model components.

### Completed Architecture

The final Notebook 09 architecture consists of three principal learned layers.

The representation architecture contains

$$
894{,}003
$$

parameters.

The inherited transformative mechanisms contain

$$
8{,}187
$$

parameters.

The learned Transformative-Weight mechanism contains

$$
25
$$

trainable parameters.

The complete instantiated Media AI architecture therefore contains

$$
902{,}215
$$

parameters.

The architectural relationship is

$$
902{,}215
=
894{,}003
+
8{,}187
+
25.
$$

### Pathway Dimensions

The factual pathway remains

$$
d_F=10.
$$

The psychological pathway remains

$$
d_P=34.
$$

The social pathway remains

$$
d_S=7.
$$

The complete structural Transformative-Weight space therefore contains

$$
10+34+7=51
$$

dimensions.

Of these, 25 dimensions are activation-eligible and 26 remain deferred under the current pilot contract.

### Active Transformative-Weight Dimensions

The factual active dimensions remain

$$
A_F
=
\{1,2,4,8,9\}.
$$

The psychological active dimensions remain

$$
A_P
=
\{1,4,11,12,14,15,16,19,21,22,23,25,26,27,30,32,33\}.
$$

The social active dimensions remain

$$
A_S
=
\{0,2,4\}.
$$

These dimensions correspond exactly to the 25 learned latent Transformative-Weight parameters.

### Deferred Transformative-Weight Dimensions

The factual deferred dimensions remain

$$
D_F
=
\{0,3,5,6,7\}.
$$

The psychological deferred dimensions remain

$$
D_P
=
\{0,2,3,5,6,7,8,9,10,13,17,18,20,24,28,29,31\}.
$$

The social deferred dimensions remain

$$
D_S
=
\{1,3,5,6\}.
$$

All deferred dimensions remain structurally present but fixed at an effective Transformative Weight of zero.

### Final Learned Transformative Weights

The effective Transformative Weight for each activation-eligible dimension remains defined by

$$
w_{j}^{(k)}
=
\sigma
\left(
\alpha_{j}^{(k)}
\right).
$$

The final latent parameters are those learned during the controlled Block 11 optimisation and independently validated in Block 12.

The final effective weights must remain:

- finite;
- bounded within the permitted interval;
- pathway-specific;
- structurally independent;
- stable under repeated forward evaluation;
- identical to the Block 12 validated learned state.

Deferred dimensions must remain exactly zero.

### Final Recurrent Architecture

For pathway \(k\), the completed recurrent architecture is

$$
\Delta_t^{(k)}
=
T^{(k)}
\left(
r_t^{(k)},
h_{t-1}^{(k)}
\right),
$$

followed by

$$
\widetilde{\Delta}_t^{(k)}
=
w^{(k)}
\odot
\Delta_t^{(k)},
$$

and the additive recurrent update

$$
h_t^{(k)}
=
h_{t-1}^{(k)}
+
\widetilde{\Delta}_t^{(k)}.
$$

Equivalently,

$$
\boxed{
h_t^{(k)}
=
h_{t-1}^{(k)}
+
w^{(k)}
\odot
T^{(k)}
\left(
r_t^{(k)},
h_{t-1}^{(k)}
\right)
}
$$

for

$$
k\in\{F,P,S\}.
$$

This is the completed recurrent Media AI architecture established by Notebook 09.

### Article-Boundary State Policy

Every article begins from a fresh deterministic zero recurrent state.

For the factual pathway,

$$
h_0^{(F)}
=
\mathbf{0}_{1\times10}.
$$

For the psychological pathway,

$$
h_0^{(P)}
=
\mathbf{0}_{1\times34}.
$$

For the social pathway,

$$
h_0^{(S)}
=
\mathbf{0}_{1\times7}.
$$

No recurrent state may propagate across article boundaries.

This reset policy must be persisted explicitly in the handover metadata because it is part of the architecture rather than an incidental training convention.

### Causal Execution Contract

The final architecture preserves the causal boundary established throughout Notebook 09.

At sentence position \(t\), pathway \(k\) may use only:

- the current pathway representation \(r_t^{(k)}\);
- the immediately preceding recurrent state \(h_{t-1}^{(k)}\);
- the pathway-specific transformative mechanism \(T^{(k)}\);
- the learned pathway-specific Transformative Weight \(w^{(k)}\).

It may not use future sentence representations, future supervision, future recurrent states, or recurrent states belonging to another article.

The final architecture therefore satisfies

$$
h_t^{(k)}
=
f
\left(
r_1^{(k)},
r_2^{(k)},
\ldots,
r_t^{(k)}
\right).
$$

### Pathway Independence

The factual, psychological and social recurrent pathways remain independent.

The factual pathway consumes only factual representation and factual recurrent state.

The psychological pathway consumes only psychological representation and psychological recurrent state.

The social pathway consumes only social representation and social recurrent state.

No cross-pathway recurrent-state mixing is introduced by Notebook 09.

### Representation–Transformation–Weight–State Separation

The final handover must preserve the architectural distinction between four different objects:

1. current representation

$$
r_t^{(k)};
$$

2. candidate transformation

$$
\Delta_t^{(k)};
$$

3. Transformative Weight

$$
w^{(k)};
$$

4. recurrent state

$$
h_t^{(k)}.
$$

These objects must not be collapsed into a single representation or parameterisation.

The representation describes the current textual state.

The transformative mechanism proposes a candidate change.

The Transformative Weight determines how much of that candidate transformation is admitted.

The recurrent state accumulates admitted transformations across the ordered article trajectory.

### Pilot Training Status

The controlled Transformative-Weight optimisation performed in Block 11 used the canonical 16-sentence pilot article solely to validate architectural learnability.

The final pilot objective improved from

$$
0.28348443
$$

to approximately

$$
0.28189849.
$$

This reduction confirms that the Transformative-Weight parameters respond to the validated supervision signal.

It is not treated as final model performance.

The learned pilot Transformative-Weight values therefore belong to the validated architectural checkpoint, but they are expected to undergo materially broader training during the expanded-corpus phase.

### Final Parameter-Freezing Policy

At Notebook 09 completion, the entire architecture should be placed into a persistent handover state.

The representation modules remain frozen.

The inherited transformative modules remain frozen.

The learned Transformative-Weight modules should also be frozen for persistence and handover validation.

Thus, at the final Notebook 09 handover boundary,

$$
\texttt{requires\_grad}
=
\texttt{False}
$$

for all

$$
902{,}215
$$

persisted parameters.

This freezing step does not redefine which parameters may later be trained in Notebook 10.

Notebook 10 may explicitly reactivate the appropriate Transformative-Weight or broader training scope according to the expanded-corpus training policy.

### Evaluation Mode

All persisted modules should be placed in evaluation mode before checkpoint construction.

The handover checkpoint should therefore represent a deterministic, inference-safe architecture state.

No dropout or training-mode stochasticity should remain active during persistence validation.

### Persistent Checkpoint Contract

The Notebook 09 checkpoint should persist the complete model state required to reconstruct the architecture directly.

At minimum, the checkpoint should include:

- representation module state dictionaries;
- transformative module state dictionaries;
- Transformative-Weight module state dictionaries;
- pathway dimensions;
- active dimension indices;
- deferred dimension indices;
- recurrent update type;
- article-boundary reset policy;
- cross-pathway recurrence policy;
- causal future-context policy;
- parameter counts;
- Notebook 09 version information;
- checkpoint schema version;
- architecture completion status.

The checkpoint must not require Notebook 10 to rerun Notebook 03, Notebook 05, Notebook 06, Notebook 07, Notebook 08 or Notebook 09 in order to reconstruct the completed architecture.

### Persistent Metadata Contract

A companion metadata artefact should provide a human-readable and machine-readable architectural summary.

The metadata should include:

- notebook identifier;
- block identifier;
- checkpoint filename;
- checkpoint Drive file ID;
- checkpoint byte size;
- checkpoint SHA-256 digest;
- metadata filename;
- architecture parameter counts;
- pathway dimensions;
- active/deferred Transformative-Weight dimensions;
- final learned effective Transformative Weights;
- recurrent update rule;
- zero initial-state policy;
- article-boundary reset requirement;
- cross-article propagation policy;
- cross-pathway recurrence policy;
- future-context policy;
- controlled-pilot training status;
- pre-training and post-training objective values;
- expanded-corpus training requirement;
- Notebook 10 bootstrap contract.

### Direct Notebook 10 Bootstrap

The handover should be intentionally designed so that Notebook 10 can start from direct persistent identifiers.

Notebook 10 should be able to load:

1. the Notebook 09 final checkpoint;
2. the Notebook 09 final metadata.

The handover metadata should expose their direct Google Drive file IDs.

This avoids filename rediscovery and reduces ambiguity when multiple historical artefact versions exist.

### Checkpoint Integrity

After persistence, the checkpoint must be downloaded again from Google Drive and validated against the in-memory completion state.

The validation should confirm:

- checkpoint file exists;
- checkpoint byte count is valid;
- SHA-256 digest matches;
- checkpoint payload parses successfully;
- representation state is reproduced exactly;
- transformative state is reproduced exactly;
- Transformative-Weight state is reproduced exactly;
- architecture parameter counts match;
- pathway contracts match;
- recurrent policy matches;
- learned effective weights match.

### Metadata Integrity

The metadata artefact must also be read back and validated.

The validation should confirm:

- metadata file exists;
- JSON parsing succeeds;
- expected top-level keys exist;
- checkpoint identity matches;
- checkpoint file ID matches;
- parameter counts match;
- pathway dimensions match;
- active/deferred dimension sets match;
- recurrent reset policy matches;
- Notebook 10 bootstrap information is complete.

### Cross-Artefact Consistency

The checkpoint and metadata must agree exactly on all shared architectural fields.

At minimum, cross-artefact validation should compare:

- checkpoint filename;
- checkpoint schema version;
- parameter counts;
- factual dimension;
- psychological dimension;
- social dimension;
- active Transformative-Weight indices;
- deferred Transformative-Weight indices;
- recurrent update policy;
- article reset policy;
- future-context policy;
- completion state.

Any disagreement invalidates the handover.

### Final Learned-State Reproduction

The persisted checkpoint must reproduce exactly:

$$
894{,}003
$$

representation parameters,

$$
8{,}187
$$

transformative parameters,

and

$$
25
$$

learned Transformative-Weight parameters.

Therefore,

$$
902{,}215
$$

persisted parameters must be reproduced exactly after read-back.

### No Runtime Recurrent State Persistence

The learned model parameters belong in the checkpoint.

The article-specific runtime recurrent states do not.

The recurrent state

$$
h_t^{(k)}
$$

is an execution-time quantity determined by the current article trajectory.

Therefore the handover persists the **recurrent architecture and reset policy**, not the final recurrent state of the 16-sentence pilot article as a model parameter.

This ensures that every future article can begin from its required fresh zero initial state.

### Pilot Trajectory Persistence

The Block 12 pilot trajectory may be retained as optional diagnostic metadata or a separate validation object, but it must not be treated as part of the learned model state.

The pilot final recurrent state is not an initial condition for future articles.

### Execution Boundary

Block 13 may:

- inspect the final learned architecture;
- validate all parameter counts and pathway contracts;
- freeze all model parameters for handover;
- place all modules in evaluation mode;
- construct the final Notebook 09 checkpoint;
- construct the final metadata artefact;
- calculate file hashes;
- persist checkpoint and metadata to Google Drive;
- read both artefacts back;
- validate exact state reproduction;
- expose the direct Notebook 10 bootstrap contract.

Block 13 must not:

- execute additional training;
- create an optimiser;
- execute backward propagation;
- update any parameter;
- alter the learned Transformative-Weight state;
- change active/deferred dimension policy;
- change the recurrent update rule;
- persist article-specific recurrent state as model state;
- interpret pilot performance as final model performance.

### Notebook 09 Completion Contract

Notebook 09 is complete only when:

- Blocks 1–12 are valid and complete;
- the complete architecture contains exactly 902,215 parameters;
- the representation state contains exactly 894,003 parameters;
- the transformative state contains exactly 8,187 parameters;
- the Transformative-Weight state contains exactly 25 learned parameters;
- pathway dimensions remain 10, 34 and 7;
- active and deferred Transformative-Weight partitions remain unchanged;
- all learned effective Transformative Weights are finite and bounded;
- all deferred effective weights remain zero;
- the learned Transformative-Weight state matches Block 12 exactly;
- the recurrent update remains additive;
- every future article is required to begin from a fresh zero state;
- cross-article recurrent propagation remains prohibited;
- cross-pathway recurrence remains prohibited;
- future-context access remains prohibited;
- all model parameters are frozen for persistence;
- all modules are in evaluation mode;
- the final checkpoint is persisted successfully;
- the final metadata artefact is persisted successfully;
- both artefacts are read back successfully;
- checkpoint SHA-256 validation succeeds;
- metadata integrity validation succeeds;
- representation state read-back is exact;
- transformative state read-back is exact;
- Transformative-Weight state read-back is exact;
- checkpoint and metadata contracts agree;
- direct Notebook 10 Drive file IDs are exposed;
- no optimiser is created;
- no backward pass is executed;
- no parameter update occurs;
- Notebook 09 is marked complete.

### Expected Outcome

Successful completion of Block 13 closes the architectural phase of Media AI.

The resulting persistent handover contains the complete validated architecture

$$
\boxed{
r_t^{(k)}
\rightarrow
T^{(k)}
\left(
r_t^{(k)},
h_{t-1}^{(k)}
\right)
\rightarrow
w^{(k)}
\odot
\Delta_t^{(k)}
\rightarrow
h_t^{(k)}
}
$$

with factual, psychological and social recurrent pathways, learned bounded Transformative Weights, deterministic zero article-boundary initialisation, and causal recurrent execution.

Notebook 10 may then begin the expanded-corpus training phase directly from the persistent Notebook 09 checkpoint and metadata, without reconstructing the architecture from earlier notebooks.

In [64]:
# =============================================================================
# Media AI — Notebook 09
# Block 13: Final Architectural Completion Audit and
#           Persistent Notebook 09 → Notebook 10 Handover
# =============================================================================

from copy import deepcopy

import hashlib
import io
import json
from datetime import datetime, timezone

import numpy as np
import torch

from googleapiclient.http import (
    MediaFileUpload,
    MediaIoBaseDownload,
)


# =============================================================================
# Block identity
# =============================================================================

NOTEBOOK_09_BLOCK_13 = 13

NOTEBOOK_09_BLOCK_13_NAME = (
    "Final Architectural Completion Audit and "
    "Persistent Notebook 09 → Notebook 10 Handover"
)

NOTEBOOK_09_BLOCK_13_VERSION = "1.1"

NOTEBOOK_09_FINAL_CHECKPOINT_SCHEMA_VERSION = "1.0"

NOTEBOOK_09_FINAL_METADATA_SCHEMA_VERSION = "1.0"


# =============================================================================
# Final persistent artefact names
# =============================================================================

NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME = (
    "notebook_09_final_checkpoint.pt"
)

NOTEBOOK_09_FINAL_METADATA_FILENAME = (
    "notebook_09_final_metadata.json"
)


# =============================================================================
# Required runtime contract
# =============================================================================

BLOCK_13_REQUIRED_OBJECTS = (
    # -------------------------------------------------------------------------
    # Notebook / Drive
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INITIALISED",
    "NOTEBOOK_09_DRIVE_SERVICE",

    # -------------------------------------------------------------------------
    # Inherited architecture
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT",
    "NOTEBOOK_09_INHERITED_PARAMETER_COUNT",

    "BLOCK_1_REPRESENTATION_MODULES",
    "BLOCK_1_TRANSFORMATIVE_MODULES",

    # -------------------------------------------------------------------------
    # Transformative Weight
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT",
    "NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION",
    "NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT",
    "BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS",

    "NOTEBOOK_09_TRAINED_TRANSFORMATIVE_WEIGHT_STATE",
    "NOTEBOOK_09_TRAINED_EFFECTIVE_TRANSFORMATIVE_WEIGHTS",

    # -------------------------------------------------------------------------
    # Recurrent architecture
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS",
    "NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES",
    "NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES",

    # -------------------------------------------------------------------------
    # Block 11 training diagnostics
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_TW_INITIAL_TOTAL_OBJECTIVE",
    "NOTEBOOK_09_TW_FINAL_TOTAL_OBJECTIVE",
    "NOTEBOOK_09_TW_OBJECTIVE_DELTA",
    "NOTEBOOK_09_TW_FINAL_PATHWAY_LOSSES",

    # -------------------------------------------------------------------------
    # Block 12
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_BLOCK_12_COMPLETE",
    "NOTEBOOK_09_BLOCK_12_VALID",

    "NOTEBOOK_09_POSTTRAIN_RECURRENT_VALIDATION_VALID",
    "NOTEBOOK_09_ARCHITECTURAL_COMPLETION_AUDIT_READY",

    "NOTEBOOK_09_POSTTRAIN_RECURRENT_TRAJECTORY",
    "NOTEBOOK_09_POSTTRAIN_FINAL_RECURRENT_STATES",
    "NOTEBOOK_09_POSTTRAIN_PATHWAY_LOSSES",
    "NOTEBOOK_09_POSTTRAIN_TOTAL_OBJECTIVE",

    # -------------------------------------------------------------------------
    # Canonical corpus
    # -------------------------------------------------------------------------
    "NOTEBOOK_09_CANONICAL_ARTICLE_IDS",
    "NOTEBOOK_09_CANONICAL_SENTENCE_IDS",

    # -------------------------------------------------------------------------
    # Environment
    # -------------------------------------------------------------------------
    "DEVICE",
    "DEFAULT_DTYPE",
)


BLOCK_13_MISSING_PREREQUISITES = [
    object_name

    for object_name
    in BLOCK_13_REQUIRED_OBJECTS

    if object_name not in globals()
]


if BLOCK_13_MISSING_PREREQUISITES:

    raise NameError(
        "Notebook 09 Block 13 prerequisites are not initialised. "
        f"Missing: {BLOCK_13_MISSING_PREREQUISITES}"
    )


# =============================================================================
# Validate prerequisite state
# =============================================================================

BLOCK_13_PREREQUISITES_VALID = all(
    [
        NOTEBOOK_09_INITIALISED is True,

        NOTEBOOK_09_BLOCK_12_COMPLETE is True,
        NOTEBOOK_09_BLOCK_12_VALID is True,

        NOTEBOOK_09_POSTTRAIN_RECURRENT_VALIDATION_VALID is True,

        NOTEBOOK_09_ARCHITECTURAL_COMPLETION_AUDIT_READY is True,

        NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT > 0,

        NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT > 0,

        (
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
            +
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_TRAINABLE_PARAMETER_COUNT
        ),

        (
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),

        (
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
            ==
            NOTEBOOK_09_INHERITED_PARAMETER_COUNT
            +
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),
    ]
)


if not BLOCK_13_PREREQUISITES_VALID:

    raise RuntimeError(
        "Notebook 09 Block 12 must be valid and complete and the "
        "current Notebook 09 parameter contract must be internally "
        "consistent before final architectural persistence."
    )


# =============================================================================
# Canonical final architecture contract
# =============================================================================

BLOCK_13_PATHWAY_DIMENSIONS = deepcopy(
    NOTEBOOK_09_RECURRENT_STATE_DIMENSIONS
)


BLOCK_13_PATHWAY_DIMENSIONS_VALID = all(
    [
        isinstance(
            BLOCK_13_PATHWAY_DIMENSIONS,
            dict,
        ),

        bool(
            BLOCK_13_PATHWAY_DIMENSIONS
        ),

        (
            set(
                BLOCK_13_PATHWAY_DIMENSIONS.keys()
            )
            ==
            set(
                NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.keys()
            )
        ),

        all(
            isinstance(
                pathway_dim,
                int,
            )
            and
            pathway_dim > 0

            for pathway_dim
            in BLOCK_13_PATHWAY_DIMENSIONS.values()
        ),
    ]
)


if not BLOCK_13_PATHWAY_DIMENSIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 13 pathway dimensions are invalid."
    )


BLOCK_13_ACTIVE_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_ACTIVE_DIMENSION_INDICES.items()
}


BLOCK_13_DEFERRED_DIMENSION_INDICES = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in NOTEBOOK_09_RECURRENT_DEFERRED_DIMENSION_INDICES.items()
}


# =============================================================================
# Active / deferred partition validation
# =============================================================================

BLOCK_13_DIMENSION_PARTITIONS_VALID = {}


for (
    pathway_name,
    pathway_dim,
) in BLOCK_13_PATHWAY_DIMENSIONS.items():

    active_set = set(
        BLOCK_13_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )


    deferred_set = set(
        BLOCK_13_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )


    BLOCK_13_DIMENSION_PARTITIONS_VALID[
        pathway_name
    ] = all(
        [
            active_set.isdisjoint(
                deferred_set
            ),

            (
                active_set
                |
                deferred_set
            )
            ==
            set(
                range(
                    pathway_dim
                )
            ),
        ]
    )


BLOCK_13_ALL_DIMENSION_PARTITIONS_VALID = all(
    BLOCK_13_DIMENSION_PARTITIONS_VALID.values()
)


if not BLOCK_13_ALL_DIMENSION_PARTITIONS_VALID:

    raise RuntimeError(
        "Notebook 09 Block 13 active/deferred dimension "
        "partitions are invalid."
    )


# =============================================================================
# Structural Transformative-Weight accounting
# =============================================================================

BLOCK_13_STRUCTURAL_WEIGHT_DIMENSIONS = sum(
    BLOCK_13_PATHWAY_DIMENSIONS.values()
)


BLOCK_13_ACTIVE_WEIGHT_DIMENSIONS = sum(
    len(
        BLOCK_13_ACTIVE_DIMENSION_INDICES[
            pathway_name
        ]
    )

    for pathway_name
    in BLOCK_13_PATHWAY_DIMENSIONS
)


BLOCK_13_DEFERRED_WEIGHT_DIMENSIONS = sum(
    len(
        BLOCK_13_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ]
    )

    for pathway_name
    in BLOCK_13_PATHWAY_DIMENSIONS
)


BLOCK_13_WEIGHT_DIMENSION_ACCOUNTING_VALID = all(
    [
        (
            BLOCK_13_ACTIVE_WEIGHT_DIMENSIONS
            +
            BLOCK_13_DEFERRED_WEIGHT_DIMENSIONS
            ==
            BLOCK_13_STRUCTURAL_WEIGHT_DIMENSIONS
        ),

        (
            BLOCK_13_ACTIVE_WEIGHT_DIMENSIONS
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            BLOCK_13_ACTIVE_WEIGHT_DIMENSIONS
            ==
            BLOCK_3_TOTAL_TRAINABLE_WEIGHT_PARAMETERS
        ),
    ]
)


if not BLOCK_13_WEIGHT_DIMENSION_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 Block 13 Transformative-Weight structural "
        "dimension accounting is invalid."
    )


# =============================================================================
# Snapshot helpers
# =============================================================================

def block_13_snapshot_module_collection(
    module_collection,
):

    return {
        module_name:
            {
                tensor_name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for (
                    tensor_name,
                    tensor,
                ) in module.state_dict().items()
            }

        for (
            module_name,
            module,
        ) in module_collection.items()
    }


def block_13_nested_state_exact(
    state_a,
    state_b,
):

    if (
        state_a.keys()
        !=
        state_b.keys()
    ):

        return False


    for module_name in state_a:

        if (
            state_a[
                module_name
            ].keys()
            !=
            state_b[
                module_name
            ].keys()
        ):

            return False


        for tensor_name in state_a[
            module_name
        ]:

            if not torch.equal(
                state_a[
                    module_name
                ][
                    tensor_name
                ],
                state_b[
                    module_name
                ][
                    tensor_name
                ],
            ):

                return False


    return True


# =============================================================================
# Snapshot final learned state before freeze-for-handover
# =============================================================================

BLOCK_13_REPRESENTATION_STATE_PRE_FREEZE = (
    block_13_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_13_TRANSFORMATIVE_STATE_PRE_FREEZE = (
    block_13_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_13_WEIGHT_STATE_PRE_FREEZE = (
    block_13_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


# =============================================================================
# Validate current Transformative-Weight state against Block 11/12
# =============================================================================

BLOCK_13_WEIGHT_STATE_MATCHES_TRAINED_STATE = (
    block_13_nested_state_exact(
        BLOCK_13_WEIGHT_STATE_PRE_FREEZE,
        NOTEBOOK_09_TRAINED_TRANSFORMATIVE_WEIGHT_STATE,
    )
)


if not BLOCK_13_WEIGHT_STATE_MATCHES_TRAINED_STATE:

    raise RuntimeError(
        "Notebook 09 Block 13 current Transformative-Weight state "
        "does not match the trained validated state."
    )


# =============================================================================
# Final learned effective Transformative-Weight audit
# =============================================================================

with torch.no_grad():

    BLOCK_13_FINAL_EFFECTIVE_WEIGHTS = {
        pathway_name:
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES[
                pathway_name
            ]()
            .detach()
            .cpu()
            .clone()

        for pathway_name
        in (
            "factual",
            "psychological",
            "social",
        )
    }


BLOCK_13_EFFECTIVE_WEIGHT_STATE_MATCHES_BLOCK_11 = all(
    torch.equal(
        BLOCK_13_FINAL_EFFECTIVE_WEIGHTS[
            pathway_name
        ],
        NOTEBOOK_09_TRAINED_EFFECTIVE_TRANSFORMATIVE_WEIGHTS[
            pathway_name
        ],
    )

    for pathway_name
    in (
        "factual",
        "psychological",
        "social",
    )
)


BLOCK_13_EFFECTIVE_WEIGHTS_FINITE = {}

BLOCK_13_EFFECTIVE_WEIGHTS_BOUNDED = {}

BLOCK_13_DEFERRED_WEIGHTS_ZERO = {}


for pathway_name in BLOCK_13_PATHWAY_DIMENSIONS:

    effective_weight = (
        BLOCK_13_FINAL_EFFECTIVE_WEIGHTS[
            pathway_name
        ]
    )


    deferred_indices = torch.tensor(
        BLOCK_13_DEFERRED_DIMENSION_INDICES[
            pathway_name
        ],
        dtype=
            torch.long,
    )


    BLOCK_13_EFFECTIVE_WEIGHTS_FINITE[
        pathway_name
    ] = bool(
        torch.isfinite(
            effective_weight
        ).all().item()
    )


    BLOCK_13_EFFECTIVE_WEIGHTS_BOUNDED[
        pathway_name
    ] = bool(
        (
            (
                effective_weight
                >=
                0.0
            )
            &
            (
                effective_weight
                <=
                1.0
            )
        ).all().item()
    )


    BLOCK_13_DEFERRED_WEIGHTS_ZERO[
        pathway_name
    ] = torch.equal(
        effective_weight[
            deferred_indices
        ],
        torch.zeros_like(
            effective_weight[
                deferred_indices
            ]
        ),
    )


BLOCK_13_ALL_EFFECTIVE_WEIGHTS_FINITE = all(
    BLOCK_13_EFFECTIVE_WEIGHTS_FINITE.values()
)


BLOCK_13_ALL_EFFECTIVE_WEIGHTS_BOUNDED = all(
    BLOCK_13_EFFECTIVE_WEIGHTS_BOUNDED.values()
)


BLOCK_13_ALL_DEFERRED_WEIGHTS_ZERO = all(
    BLOCK_13_DEFERRED_WEIGHTS_ZERO.values()
)


if not all(
    [
        BLOCK_13_EFFECTIVE_WEIGHT_STATE_MATCHES_BLOCK_11,
        BLOCK_13_ALL_EFFECTIVE_WEIGHTS_FINITE,
        BLOCK_13_ALL_EFFECTIVE_WEIGHTS_BOUNDED,
        BLOCK_13_ALL_DEFERRED_WEIGHTS_ZERO,
    ]
):

    raise RuntimeError(
        "Notebook 09 Block 13 learned Transformative-Weight "
        "audit failed."
    )


# =============================================================================
# Freeze complete architecture for persistent handover
# =============================================================================

for module_collection in (
    BLOCK_1_REPRESENTATION_MODULES,
    BLOCK_1_TRANSFORMATIVE_MODULES,
    NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES,
):

    for module in module_collection.values():

        module.eval()


        for parameter in module.parameters():

            parameter.requires_grad_(
                False
            )


# =============================================================================
# Final freeze / eval validation
# =============================================================================

BLOCK_13_ALL_PARAMETERS_FROZEN = all(
    not parameter.requires_grad

    for module_collection
    in (
        BLOCK_1_REPRESENTATION_MODULES,
        BLOCK_1_TRANSFORMATIVE_MODULES,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES,
    )

    for module
    in module_collection.values()

    for parameter
    in module.parameters()
)


BLOCK_13_ALL_MODULES_EVAL = all(
    not module.training

    for module_collection
    in (
        BLOCK_1_REPRESENTATION_MODULES,
        BLOCK_1_TRANSFORMATIVE_MODULES,
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES,
    )

    for module
    in module_collection.values()
)


if not all(
    [
        BLOCK_13_ALL_PARAMETERS_FROZEN,
        BLOCK_13_ALL_MODULES_EVAL,
    ]
):

    raise RuntimeError(
        "Notebook 09 final architecture could not be placed "
        "into frozen evaluation handover state."
    )


# =============================================================================
# Verify freezing changed no learned values
# =============================================================================

BLOCK_13_REPRESENTATION_STATE_POST_FREEZE = (
    block_13_snapshot_module_collection(
        BLOCK_1_REPRESENTATION_MODULES
    )
)


BLOCK_13_TRANSFORMATIVE_STATE_POST_FREEZE = (
    block_13_snapshot_module_collection(
        BLOCK_1_TRANSFORMATIVE_MODULES
    )
)


BLOCK_13_WEIGHT_STATE_POST_FREEZE = (
    block_13_snapshot_module_collection(
        NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES
    )
)


BLOCK_13_REPRESENTATION_UNCHANGED_BY_FREEZE = (
    block_13_nested_state_exact(
        BLOCK_13_REPRESENTATION_STATE_PRE_FREEZE,
        BLOCK_13_REPRESENTATION_STATE_POST_FREEZE,
    )
)


BLOCK_13_TRANSFORMATIVE_UNCHANGED_BY_FREEZE = (
    block_13_nested_state_exact(
        BLOCK_13_TRANSFORMATIVE_STATE_PRE_FREEZE,
        BLOCK_13_TRANSFORMATIVE_STATE_POST_FREEZE,
    )
)


BLOCK_13_WEIGHT_UNCHANGED_BY_FREEZE = (
    block_13_nested_state_exact(
        BLOCK_13_WEIGHT_STATE_PRE_FREEZE,
        BLOCK_13_WEIGHT_STATE_POST_FREEZE,
    )
)


if not all(
    [
        BLOCK_13_REPRESENTATION_UNCHANGED_BY_FREEZE,
        BLOCK_13_TRANSFORMATIVE_UNCHANGED_BY_FREEZE,
        BLOCK_13_WEIGHT_UNCHANGED_BY_FREEZE,
    ]
):

    raise RuntimeError(
        "Freezing the Notebook 09 architecture altered learned state."
    )


# =============================================================================
# Parameter accounting from actual modules
# =============================================================================

BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT = sum(
    parameter.numel()

    for module
    in BLOCK_1_REPRESENTATION_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT = sum(
    parameter.numel()

    for module
    in BLOCK_1_TRANSFORMATIVE_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT = sum(
    parameter.numel()

    for module
    in NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_MODULES.values()

    for parameter
    in module.parameters()
)


BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT = (
    BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT
    +
    BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT
    +
    BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT
)


BLOCK_13_PARAMETER_ACCOUNTING_VALID = all(
    [
        (
            BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_REPRESENTATION_PARAMETER_COUNT
        ),

        (
            BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT
            ==
            NOTEBOOK_09_INHERITED_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TRANSFORMATIVE_WEIGHT_PARAMETER_COUNT
        ),

        (
            BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT
            ==
            NOTEBOOK_09_TOTAL_PARAMETER_COUNT_AFTER_WEIGHT_CONSTRUCTION
        ),
    ]
)


if not BLOCK_13_PARAMETER_ACCOUNTING_VALID:

    raise RuntimeError(
        "Notebook 09 final architecture parameter accounting failed."
    )


# =============================================================================
# Final architectural policies
# =============================================================================

BLOCK_13_RECURRENT_UPDATE_TYPE = "additive"

BLOCK_13_INITIAL_STATE_POLICY = "deterministic_zero"

BLOCK_13_ARTICLE_BOUNDARY_RESET = True

BLOCK_13_CROSS_ARTICLE_PROPAGATION = False

BLOCK_13_CROSS_PATHWAY_RECURRENCE = False

BLOCK_13_FUTURE_CONTEXT_ALLOWED = False


BLOCK_13_RECURRENT_POLICY_VALID = all(
    [
        BLOCK_13_RECURRENT_UPDATE_TYPE
        ==
        "additive",

        BLOCK_13_INITIAL_STATE_POLICY
        ==
        "deterministic_zero",

        BLOCK_13_ARTICLE_BOUNDARY_RESET is True,

        BLOCK_13_CROSS_ARTICLE_PROPAGATION is False,

        BLOCK_13_CROSS_PATHWAY_RECURRENCE is False,

        BLOCK_13_FUTURE_CONTEXT_ALLOWED is False,
    ]
)


# =============================================================================
# Persistent payload helpers
# =============================================================================

def block_13_tensor_to_list(
    tensor,
):

    return (
        tensor.detach()
        .cpu()
        .tolist()
    )


def block_13_sha256_bytes(
    raw_bytes,
):

    return hashlib.sha256(
        raw_bytes
    ).hexdigest()


def block_13_download_drive_bytes(
    drive_service,
    file_id,
):

    request = (
        drive_service.files()
        .get_media(
            fileId=
                file_id
        )
    )


    buffer = io.BytesIO()


    downloader = MediaIoBaseDownload(
        buffer,
        request,
    )


    done = False


    while not done:

        _, done = downloader.next_chunk()


    return buffer.getvalue()


def block_13_find_drive_file(
    drive_service,
    filename,
):

    escaped_filename = filename.replace(
        "'",
        "\\'",
    )


    result = (
        drive_service.files()
        .list(
            q=(
                f"name = '{escaped_filename}' "
                "and trashed = false"
            ),

            spaces=
                "drive",

            fields=
                "files(id,name,size,modifiedTime)",

            orderBy=
                "modifiedTime desc",

            pageSize=
                20,
        )
        .execute()
    )


    return result.get(
        "files",
        [],
    )


def block_13_upload_or_update_file(
    drive_service,
    local_path,
    filename,
    mimetype,
):

    existing_files = block_13_find_drive_file(
        drive_service=
            drive_service,

        filename=
            filename,
    )


    media = MediaFileUpload(
        local_path,
        mimetype=
            mimetype,
        resumable=
            False,
    )


    if existing_files:

        target_file_id = existing_files[
            0
        ][
            "id"
        ]


        result = (
            drive_service.files()
            .update(
                fileId=
                    target_file_id,

                media_body=
                    media,

                fields=
                    "id,name,size,modifiedTime",
            )
            .execute()
        )


        operation = "updated"


    else:

        result = (
            drive_service.files()
            .create(
                body=
                    {
                        "name":
                            filename
                    },

                media_body=
                    media,

                fields=
                    "id,name,size,modifiedTime",
            )
            .execute()
        )


        operation = "created"


    return (
        result,
        operation,
    )


# =============================================================================
# Final checkpoint payload
# =============================================================================

BLOCK_13_CREATED_AT_UTC = (
    datetime.now(
        timezone.utc
    )
    .isoformat()
)


BLOCK_13_CANONICAL_ARTICLE_IDS = tuple(
    NOTEBOOK_09_CANONICAL_ARTICLE_IDS
)


BLOCK_13_CANONICAL_ARTICLE_COUNT = len(
    BLOCK_13_CANONICAL_ARTICLE_IDS
)


BLOCK_13_CANONICAL_SENTENCE_COUNT = len(
    NOTEBOOK_09_CANONICAL_SENTENCE_IDS
)


NOTEBOOK_09_FINAL_CHECKPOINT = {
    "artifact_type":
        "media_ai_notebook_09_final_checkpoint",

    "checkpoint_schema_version":
        NOTEBOOK_09_FINAL_CHECKPOINT_SCHEMA_VERSION,

    "notebook":
        "09_transformative_weight_and_recurrent_state",

    "notebook_number":
        9,

    "notebook_block":
        13,

    "notebook_block_version":
        NOTEBOOK_09_BLOCK_13_VERSION,

    "created_at_utc":
        BLOCK_13_CREATED_AT_UTC,

    # -------------------------------------------------------------------------
    # Architecture
    # -------------------------------------------------------------------------
    "architecture":
        {
            "representation_parameter_count":
                BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT,

            "transformative_parameter_count":
                BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT,

            "transformative_weight_parameter_count":
                BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT,

            "total_parameter_count":
                BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT,

            "pathway_dimensions":
                deepcopy(
                    BLOCK_13_PATHWAY_DIMENSIONS
                ),

            "structural_weight_dimensions":
                BLOCK_13_STRUCTURAL_WEIGHT_DIMENSIONS,

            "active_weight_dimensions":
                BLOCK_13_ACTIVE_WEIGHT_DIMENSIONS,

            "deferred_weight_dimensions":
                BLOCK_13_DEFERRED_WEIGHT_DIMENSIONS,

            "active_dimension_indices":
                deepcopy(
                    BLOCK_13_ACTIVE_DIMENSION_INDICES
                ),

            "deferred_dimension_indices":
                deepcopy(
                    BLOCK_13_DEFERRED_DIMENSION_INDICES
                ),
        },

    # -------------------------------------------------------------------------
    # Recurrent contract
    # -------------------------------------------------------------------------
    "recurrent_contract":
        {
            "update_type":
                BLOCK_13_RECURRENT_UPDATE_TYPE,

            "initial_state_policy":
                BLOCK_13_INITIAL_STATE_POLICY,

            "article_boundary_reset":
                BLOCK_13_ARTICLE_BOUNDARY_RESET,

            "cross_article_propagation":
                BLOCK_13_CROSS_ARTICLE_PROPAGATION,

            "cross_pathway_recurrence":
                BLOCK_13_CROSS_PATHWAY_RECURRENCE,

            "future_context_allowed":
                BLOCK_13_FUTURE_CONTEXT_ALLOWED,
        },

    # -------------------------------------------------------------------------
    # Model states
    # -------------------------------------------------------------------------
    "representation_state":
        BLOCK_13_REPRESENTATION_STATE_POST_FREEZE,

    "transformative_state":
        BLOCK_13_TRANSFORMATIVE_STATE_POST_FREEZE,

    "transformative_weight_state":
        BLOCK_13_WEIGHT_STATE_POST_FREEZE,

    # -------------------------------------------------------------------------
    # Learned effective weights
    # -------------------------------------------------------------------------
    "effective_transformative_weights":
        {
            pathway_name:
                BLOCK_13_FINAL_EFFECTIVE_WEIGHTS[
                    pathway_name
                ].clone()

            for pathway_name
            in BLOCK_13_PATHWAY_DIMENSIONS
        },

    # -------------------------------------------------------------------------
    # Pilot controlled-training diagnostics
    # -------------------------------------------------------------------------
    "pilot_training":
        {
            "training_scope":
                "controlled_architectural_pilot",

            "final_training":
                False,

            "expanded_corpus_training_required":
                True,

            "article_count":
                BLOCK_13_CANONICAL_ARTICLE_COUNT,

            "sentence_count":
                BLOCK_13_CANONICAL_SENTENCE_COUNT,

            "pretraining_total_objective":
                float(
                    NOTEBOOK_09_TW_INITIAL_TOTAL_OBJECTIVE
                ),

            "posttraining_total_objective":
                float(
                    NOTEBOOK_09_TW_FINAL_TOTAL_OBJECTIVE
                ),

            "objective_delta":
                float(
                    NOTEBOOK_09_TW_OBJECTIVE_DELTA
                ),

            "final_pathway_losses":
                {
                    pathway_name:
                        float(
                            NOTEBOOK_09_TW_FINAL_PATHWAY_LOSSES[
                                pathway_name
                            ]
                        )

                    for pathway_name
                    in (
                        "factual",
                        "psychological",
                        "social",
                    )
                },
        },

    # -------------------------------------------------------------------------
    # Completion status
    # -------------------------------------------------------------------------
    "completion":
        {
            "architecture_complete":
                True,

            "controlled_training_valid":
                True,

            "posttraining_recurrent_validation_valid":
                True,

            "notebook_09_complete":
                True,
        },
}


# =============================================================================
# Save checkpoint locally
# =============================================================================

BLOCK_13_CHECKPOINT_LOCAL_PATH = (
    f"/tmp/{NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME}"
)


torch.save(
    NOTEBOOK_09_FINAL_CHECKPOINT,
    BLOCK_13_CHECKPOINT_LOCAL_PATH,
)


with open(
    BLOCK_13_CHECKPOINT_LOCAL_PATH,
    "rb",
) as file_handle:

    BLOCK_13_CHECKPOINT_BYTES = (
        file_handle.read()
    )


BLOCK_13_CHECKPOINT_BYTE_COUNT = len(
    BLOCK_13_CHECKPOINT_BYTES
)


BLOCK_13_CHECKPOINT_SHA256 = (
    block_13_sha256_bytes(
        BLOCK_13_CHECKPOINT_BYTES
    )
)


# =============================================================================
# Persist checkpoint to Google Drive
# =============================================================================

(
    BLOCK_13_CHECKPOINT_DRIVE_INFO,
    BLOCK_13_CHECKPOINT_DRIVE_OPERATION,
) = block_13_upload_or_update_file(
    drive_service=
        NOTEBOOK_09_DRIVE_SERVICE,

    local_path=
        BLOCK_13_CHECKPOINT_LOCAL_PATH,

    filename=
        NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME,

    mimetype=
        "application/octet-stream",
)


BLOCK_13_CHECKPOINT_DRIVE_FILE_ID = (
    BLOCK_13_CHECKPOINT_DRIVE_INFO[
        "id"
    ]
)


# =============================================================================
# Metadata payload
# =============================================================================

NOTEBOOK_09_FINAL_METADATA = {
    "artifact_type":
        "media_ai_notebook_09_final_metadata",

    "metadata_schema_version":
        NOTEBOOK_09_FINAL_METADATA_SCHEMA_VERSION,

    "notebook":
        "09_transformative_weight_and_recurrent_state",

    "notebook_number":
        9,

    "completion_block":
        13,

    "block_version":
        NOTEBOOK_09_BLOCK_13_VERSION,

    "created_at_utc":
        BLOCK_13_CREATED_AT_UTC,

    # -------------------------------------------------------------------------
    # Checkpoint
    # -------------------------------------------------------------------------
    "checkpoint":
        {
            "filename":
                NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME,

            "drive_file_id":
                BLOCK_13_CHECKPOINT_DRIVE_FILE_ID,

            "byte_count":
                BLOCK_13_CHECKPOINT_BYTE_COUNT,

            "sha256":
                BLOCK_13_CHECKPOINT_SHA256,

            "checkpoint_schema_version":
                NOTEBOOK_09_FINAL_CHECKPOINT_SCHEMA_VERSION,
        },

    # -------------------------------------------------------------------------
    # Architecture
    # -------------------------------------------------------------------------
    "architecture":
        {
            "representation_parameters":
                BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT,

            "transformative_parameters":
                BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT,

            "transformative_weight_parameters":
                BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT,

            "total_parameters":
                BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT,

            "pathway_dimensions":
                deepcopy(
                    BLOCK_13_PATHWAY_DIMENSIONS
                ),

            "structural_weight_dimensions":
                BLOCK_13_STRUCTURAL_WEIGHT_DIMENSIONS,

            "active_weight_dimensions":
                BLOCK_13_ACTIVE_WEIGHT_DIMENSIONS,

            "deferred_weight_dimensions":
                BLOCK_13_DEFERRED_WEIGHT_DIMENSIONS,

            "active_dimension_indices":
                {
                    pathway_name:
                        list(
                            BLOCK_13_ACTIVE_DIMENSION_INDICES[
                                pathway_name
                            ]
                        )

                    for pathway_name
                    in (
                        "factual",
                        "psychological",
                        "social",
                    )
                },

            "deferred_dimension_indices":
                {
                    pathway_name:
                        list(
                            BLOCK_13_DEFERRED_DIMENSION_INDICES[
                                pathway_name
                            ]
                        )

                    for pathway_name
                    in (
                        "factual",
                        "psychological",
                        "social",
                    )
                },
        },

    # -------------------------------------------------------------------------
    # Learned Transformative Weights
    # -------------------------------------------------------------------------
    "effective_transformative_weights":
        {
            pathway_name:
                block_13_tensor_to_list(
                    BLOCK_13_FINAL_EFFECTIVE_WEIGHTS[
                        pathway_name
                    ]
                )

            for pathway_name
            in BLOCK_13_PATHWAY_DIMENSIONS
        },

    # -------------------------------------------------------------------------
    # Recurrent contract
    # -------------------------------------------------------------------------
    "recurrent_contract":
        {
            "update_type":
                BLOCK_13_RECURRENT_UPDATE_TYPE,

            "initial_state_policy":
                BLOCK_13_INITIAL_STATE_POLICY,

            "article_boundary_reset":
                BLOCK_13_ARTICLE_BOUNDARY_RESET,

            "cross_article_propagation":
                BLOCK_13_CROSS_ARTICLE_PROPAGATION,

            "cross_pathway_recurrence":
                BLOCK_13_CROSS_PATHWAY_RECURRENCE,

            "future_context_allowed":
                BLOCK_13_FUTURE_CONTEXT_ALLOWED,
        },

    # -------------------------------------------------------------------------
    # Pilot training
    # -------------------------------------------------------------------------
    "pilot_training":
        {
            "scope":
                "controlled_architectural_pilot",

            "article_count":
                BLOCK_13_CANONICAL_ARTICLE_COUNT,

            "sentence_count":
                BLOCK_13_CANONICAL_SENTENCE_COUNT,

            "final_training":
                False,

            "expanded_corpus_training_required":
                True,

            "pretraining_total_objective":
                float(
                    NOTEBOOK_09_TW_INITIAL_TOTAL_OBJECTIVE
                ),

            "posttraining_total_objective":
                float(
                    NOTEBOOK_09_TW_FINAL_TOTAL_OBJECTIVE
                ),

            "objective_delta":
                float(
                    NOTEBOOK_09_TW_OBJECTIVE_DELTA
                ),

            "final_pathway_losses":
                {
                    pathway_name:
                        float(
                            NOTEBOOK_09_TW_FINAL_PATHWAY_LOSSES[
                                pathway_name
                            ]
                        )

                    for pathway_name
                    in (
                        "factual",
                        "psychological",
                        "social",
                    )
                },
        },

    # -------------------------------------------------------------------------
    # Persistence state
    # -------------------------------------------------------------------------
    "persistence":
        {
            "all_parameters_frozen":
                BLOCK_13_ALL_PARAMETERS_FROZEN,

            "all_modules_eval":
                BLOCK_13_ALL_MODULES_EVAL,

            "runtime_recurrent_state_persisted_as_model_state":
                False,
        },

    # -------------------------------------------------------------------------
    # Notebook 10 bootstrap contract
    # -------------------------------------------------------------------------
    "notebook_10_bootstrap":
        {
            "source_notebook":
                9,

            "checkpoint_filename":
                NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME,

            "checkpoint_drive_file_id":
                BLOCK_13_CHECKPOINT_DRIVE_FILE_ID,

            "metadata_filename":
                NOTEBOOK_09_FINAL_METADATA_FILENAME,

            "metadata_drive_file_id":
                None,

            "architecture_complete":
                True,

            "expanded_corpus_training_required":
                True,

            "reconstruct_upstream_notebooks":
                False,
        },

    # -------------------------------------------------------------------------
    # Completion
    # -------------------------------------------------------------------------
    "completion":
        {
            "notebook_09_complete":
                True,

            "architecture_complete":
                True,

            "posttraining_validation_valid":
                True,

            "notebook_10_ready":
                True,
        },
}


# =============================================================================
# First metadata write
# =============================================================================
#
# The metadata Drive ID is not known until the first upload.
# =============================================================================

BLOCK_13_METADATA_LOCAL_PATH = (
    f"/tmp/{NOTEBOOK_09_FINAL_METADATA_FILENAME}"
)


with open(
    BLOCK_13_METADATA_LOCAL_PATH,
    "w",
    encoding=
        "utf-8",
) as file_handle:

    json.dump(
        NOTEBOOK_09_FINAL_METADATA,
        file_handle,
        indent=
            2,
        ensure_ascii=
            False,
        sort_keys=
            True,
    )


# =============================================================================
# Persist metadata to Google Drive
# =============================================================================

(
    BLOCK_13_METADATA_DRIVE_INFO,
    BLOCK_13_METADATA_DRIVE_OPERATION,
) = block_13_upload_or_update_file(
    drive_service=
        NOTEBOOK_09_DRIVE_SERVICE,

    local_path=
        BLOCK_13_METADATA_LOCAL_PATH,

    filename=
        NOTEBOOK_09_FINAL_METADATA_FILENAME,

    mimetype=
        "application/json",
)


BLOCK_13_METADATA_DRIVE_FILE_ID = (
    BLOCK_13_METADATA_DRIVE_INFO[
        "id"
    ]
)


# =============================================================================
# Insert direct metadata Drive ID into bootstrap contract
# =============================================================================

NOTEBOOK_09_FINAL_METADATA[
    "notebook_10_bootstrap"
][
    "metadata_drive_file_id"
] = (
    BLOCK_13_METADATA_DRIVE_FILE_ID
)


# =============================================================================
# Rewrite metadata locally with final bootstrap contract
# =============================================================================

with open(
    BLOCK_13_METADATA_LOCAL_PATH,
    "w",
    encoding=
        "utf-8",
) as file_handle:

    json.dump(
        NOTEBOOK_09_FINAL_METADATA,
        file_handle,
        indent=
            2,
        ensure_ascii=
            False,
        sort_keys=
            True,
    )


with open(
    BLOCK_13_METADATA_LOCAL_PATH,
    "rb",
) as file_handle:

    BLOCK_13_METADATA_BYTES = (
        file_handle.read()
    )


BLOCK_13_METADATA_BYTE_COUNT = len(
    BLOCK_13_METADATA_BYTES
)


BLOCK_13_METADATA_SHA256 = (
    block_13_sha256_bytes(
        BLOCK_13_METADATA_BYTES
    )
)


# =============================================================================
# Update metadata Drive file with final content
# =============================================================================

metadata_media = MediaFileUpload(
    BLOCK_13_METADATA_LOCAL_PATH,
    mimetype=
        "application/json",
    resumable=
        False,
)


BLOCK_13_METADATA_DRIVE_INFO = (
    NOTEBOOK_09_DRIVE_SERVICE.files()
    .update(
        fileId=
            BLOCK_13_METADATA_DRIVE_FILE_ID,

        media_body=
            metadata_media,

        fields=
            "id,name,size,modifiedTime",
    )
    .execute()
)


# =============================================================================
# Read checkpoint back from Drive
# =============================================================================

BLOCK_13_CHECKPOINT_READBACK_BYTES = (
    block_13_download_drive_bytes(
        drive_service=
            NOTEBOOK_09_DRIVE_SERVICE,

        file_id=
            BLOCK_13_CHECKPOINT_DRIVE_FILE_ID,
    )
)


BLOCK_13_CHECKPOINT_READBACK_SHA256 = (
    block_13_sha256_bytes(
        BLOCK_13_CHECKPOINT_READBACK_BYTES
    )
)


BLOCK_13_CHECKPOINT_SHA256_VALID = (
    BLOCK_13_CHECKPOINT_READBACK_SHA256
    ==
    BLOCK_13_CHECKPOINT_SHA256
)


BLOCK_13_CHECKPOINT_BYTE_COUNT_VALID = (
    len(
        BLOCK_13_CHECKPOINT_READBACK_BYTES
    )
    ==
    BLOCK_13_CHECKPOINT_BYTE_COUNT
)


# =============================================================================
# Parse read-back checkpoint
# =============================================================================

BLOCK_13_CHECKPOINT_READBACK_BUFFER = io.BytesIO(
    BLOCK_13_CHECKPOINT_READBACK_BYTES
)


BLOCK_13_CHECKPOINT_READBACK = torch.load(
    BLOCK_13_CHECKPOINT_READBACK_BUFFER,
    map_location=
        "cpu",
    weights_only=
        False,
)


BLOCK_13_CHECKPOINT_PARSED = isinstance(
    BLOCK_13_CHECKPOINT_READBACK,
    dict,
)


if not BLOCK_13_CHECKPOINT_PARSED:

    raise RuntimeError(
        "Notebook 09 checkpoint read-back did not produce "
        "a dictionary payload."
    )


# =============================================================================
# Read metadata back from Drive
# =============================================================================

BLOCK_13_METADATA_READBACK_BYTES = (
    block_13_download_drive_bytes(
        drive_service=
            NOTEBOOK_09_DRIVE_SERVICE,

        file_id=
            BLOCK_13_METADATA_DRIVE_FILE_ID,
    )
)


BLOCK_13_METADATA_READBACK_SHA256 = (
    block_13_sha256_bytes(
        BLOCK_13_METADATA_READBACK_BYTES
    )
)


BLOCK_13_METADATA_SHA256_VALID = (
    BLOCK_13_METADATA_READBACK_SHA256
    ==
    BLOCK_13_METADATA_SHA256
)


BLOCK_13_METADATA_BYTE_COUNT_VALID = (
    len(
        BLOCK_13_METADATA_READBACK_BYTES
    )
    ==
    BLOCK_13_METADATA_BYTE_COUNT
)


BLOCK_13_METADATA_READBACK = json.loads(
    BLOCK_13_METADATA_READBACK_BYTES.decode(
        "utf-8"
    )
)


BLOCK_13_METADATA_PARSED = isinstance(
    BLOCK_13_METADATA_READBACK,
    dict,
)


# =============================================================================
# Checkpoint contract validation
# =============================================================================

BLOCK_13_CHECKPOINT_IDENTITY_VALID = all(
    [
        (
            BLOCK_13_CHECKPOINT_READBACK.get(
                "artifact_type"
            )
            ==
            "media_ai_notebook_09_final_checkpoint"
        ),

        (
            BLOCK_13_CHECKPOINT_READBACK.get(
                "checkpoint_schema_version"
            )
            ==
            NOTEBOOK_09_FINAL_CHECKPOINT_SCHEMA_VERSION
        ),

        (
            BLOCK_13_CHECKPOINT_READBACK.get(
                "notebook_number"
            )
            ==
            9
        ),

        (
            BLOCK_13_CHECKPOINT_READBACK.get(
                "notebook_block"
            )
            ==
            13
        ),
    ]
)


BLOCK_13_CHECKPOINT_ARCHITECTURE = (
    BLOCK_13_CHECKPOINT_READBACK.get(
        "architecture",
        {},
    )
)


BLOCK_13_CHECKPOINT_PARAMETER_COUNTS_VALID = all(
    [
        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE.get(
                "representation_parameter_count"
            )
            ==
            BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE.get(
                "transformative_parameter_count"
            )
            ==
            BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE.get(
                "transformative_weight_parameter_count"
            )
            ==
            BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE.get(
                "total_parameter_count"
            )
            ==
            BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT
        ),
    ]
)


BLOCK_13_CHECKPOINT_PATHWAY_CONTRACT_VALID = all(
    [
        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE.get(
                "pathway_dimensions"
            )
            ==
            BLOCK_13_PATHWAY_DIMENSIONS
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE.get(
                "active_dimension_indices"
            )
            ==
            BLOCK_13_ACTIVE_DIMENSION_INDICES
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE.get(
                "deferred_dimension_indices"
            )
            ==
            BLOCK_13_DEFERRED_DIMENSION_INDICES
        ),
    ]
)


BLOCK_13_CHECKPOINT_RECURRENT_CONTRACT_VALID = (
    BLOCK_13_CHECKPOINT_READBACK.get(
        "recurrent_contract"
    )
    ==
    {
        "update_type":
            BLOCK_13_RECURRENT_UPDATE_TYPE,

        "initial_state_policy":
            BLOCK_13_INITIAL_STATE_POLICY,

        "article_boundary_reset":
            BLOCK_13_ARTICLE_BOUNDARY_RESET,

        "cross_article_propagation":
            BLOCK_13_CROSS_ARTICLE_PROPAGATION,

        "cross_pathway_recurrence":
            BLOCK_13_CROSS_PATHWAY_RECURRENCE,

        "future_context_allowed":
            BLOCK_13_FUTURE_CONTEXT_ALLOWED,
    }
)


# =============================================================================
# Exact checkpoint state read-back validation
# =============================================================================

BLOCK_13_REPRESENTATION_READBACK_EXACT = (
    block_13_nested_state_exact(
        BLOCK_13_REPRESENTATION_STATE_POST_FREEZE,
        BLOCK_13_CHECKPOINT_READBACK[
            "representation_state"
        ],
    )
)


BLOCK_13_TRANSFORMATIVE_READBACK_EXACT = (
    block_13_nested_state_exact(
        BLOCK_13_TRANSFORMATIVE_STATE_POST_FREEZE,
        BLOCK_13_CHECKPOINT_READBACK[
            "transformative_state"
        ],
    )
)


BLOCK_13_WEIGHT_READBACK_EXACT = (
    block_13_nested_state_exact(
        BLOCK_13_WEIGHT_STATE_POST_FREEZE,
        BLOCK_13_CHECKPOINT_READBACK[
            "transformative_weight_state"
        ],
    )
)


BLOCK_13_EFFECTIVE_WEIGHT_READBACK_EXACT = all(
    torch.equal(
        BLOCK_13_FINAL_EFFECTIVE_WEIGHTS[
            pathway_name
        ],
        BLOCK_13_CHECKPOINT_READBACK[
            "effective_transformative_weights"
        ][
            pathway_name
        ],
    )

    for pathway_name
    in (
        "factual",
        "psychological",
        "social",
    )
)


# =============================================================================
# Metadata contract validation
# =============================================================================

BLOCK_13_METADATA_EXPECTED_TOP_LEVEL_KEYS = {
    "artifact_type",
    "metadata_schema_version",
    "notebook",
    "notebook_number",
    "completion_block",
    "block_version",
    "created_at_utc",
    "checkpoint",
    "architecture",
    "effective_transformative_weights",
    "recurrent_contract",
    "pilot_training",
    "persistence",
    "notebook_10_bootstrap",
    "completion",
}


BLOCK_13_METADATA_TOP_LEVEL_KEYS_VALID = (
    set(
        BLOCK_13_METADATA_READBACK.keys()
    )
    ==
    BLOCK_13_METADATA_EXPECTED_TOP_LEVEL_KEYS
)


BLOCK_13_METADATA_IDENTITY_VALID = all(
    [
        (
            BLOCK_13_METADATA_READBACK.get(
                "artifact_type"
            )
            ==
            "media_ai_notebook_09_final_metadata"
        ),

        (
            BLOCK_13_METADATA_READBACK.get(
                "metadata_schema_version"
            )
            ==
            NOTEBOOK_09_FINAL_METADATA_SCHEMA_VERSION
        ),

        (
            BLOCK_13_METADATA_READBACK.get(
                "notebook_number"
            )
            ==
            9
        ),

        (
            BLOCK_13_METADATA_READBACK.get(
                "completion_block"
            )
            ==
            13
        ),
    ]
)


BLOCK_13_METADATA_CHECKPOINT_CONTRACT = (
    BLOCK_13_METADATA_READBACK.get(
        "checkpoint",
        {},
    )
)


BLOCK_13_METADATA_CHECKPOINT_IDENTITY_VALID = all(
    [
        (
            BLOCK_13_METADATA_CHECKPOINT_CONTRACT.get(
                "filename"
            )
            ==
            NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME
        ),

        (
            BLOCK_13_METADATA_CHECKPOINT_CONTRACT.get(
                "drive_file_id"
            )
            ==
            BLOCK_13_CHECKPOINT_DRIVE_FILE_ID
        ),

        (
            BLOCK_13_METADATA_CHECKPOINT_CONTRACT.get(
                "byte_count"
            )
            ==
            BLOCK_13_CHECKPOINT_BYTE_COUNT
        ),

        (
            BLOCK_13_METADATA_CHECKPOINT_CONTRACT.get(
                "sha256"
            )
            ==
            BLOCK_13_CHECKPOINT_SHA256
        ),
    ]
)


BLOCK_13_METADATA_ARCHITECTURE = (
    BLOCK_13_METADATA_READBACK.get(
        "architecture",
        {},
    )
)


BLOCK_13_METADATA_PARAMETER_COUNTS_VALID = all(
    [
        (
            BLOCK_13_METADATA_ARCHITECTURE.get(
                "representation_parameters"
            )
            ==
            BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT
        ),

        (
            BLOCK_13_METADATA_ARCHITECTURE.get(
                "transformative_parameters"
            )
            ==
            BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT
        ),

        (
            BLOCK_13_METADATA_ARCHITECTURE.get(
                "transformative_weight_parameters"
            )
            ==
            BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT
        ),

        (
            BLOCK_13_METADATA_ARCHITECTURE.get(
                "total_parameters"
            )
            ==
            BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT
        ),
    ]
)


BLOCK_13_METADATA_RECURRENT_CONTRACT_VALID = (
    BLOCK_13_METADATA_READBACK.get(
        "recurrent_contract"
    )
    ==
    {
        "update_type":
            BLOCK_13_RECURRENT_UPDATE_TYPE,

        "initial_state_policy":
            BLOCK_13_INITIAL_STATE_POLICY,

        "article_boundary_reset":
            BLOCK_13_ARTICLE_BOUNDARY_RESET,

        "cross_article_propagation":
            BLOCK_13_CROSS_ARTICLE_PROPAGATION,

        "cross_pathway_recurrence":
            BLOCK_13_CROSS_PATHWAY_RECURRENCE,

        "future_context_allowed":
            BLOCK_13_FUTURE_CONTEXT_ALLOWED,
    }
)


# =============================================================================
# Notebook 10 bootstrap validation
# =============================================================================

BLOCK_13_NOTEBOOK_10_BOOTSTRAP = (
    BLOCK_13_METADATA_READBACK.get(
        "notebook_10_bootstrap",
        {},
    )
)


BLOCK_13_NOTEBOOK_10_BOOTSTRAP_VALID = all(
    [
        (
            BLOCK_13_NOTEBOOK_10_BOOTSTRAP.get(
                "source_notebook"
            )
            ==
            9
        ),

        (
            BLOCK_13_NOTEBOOK_10_BOOTSTRAP.get(
                "checkpoint_filename"
            )
            ==
            NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME
        ),

        (
            BLOCK_13_NOTEBOOK_10_BOOTSTRAP.get(
                "checkpoint_drive_file_id"
            )
            ==
            BLOCK_13_CHECKPOINT_DRIVE_FILE_ID
        ),

        (
            BLOCK_13_NOTEBOOK_10_BOOTSTRAP.get(
                "metadata_filename"
            )
            ==
            NOTEBOOK_09_FINAL_METADATA_FILENAME
        ),

        (
            BLOCK_13_NOTEBOOK_10_BOOTSTRAP.get(
                "metadata_drive_file_id"
            )
            ==
            BLOCK_13_METADATA_DRIVE_FILE_ID
        ),

        (
            BLOCK_13_NOTEBOOK_10_BOOTSTRAP.get(
                "architecture_complete"
            )
            is True
        ),

        (
            BLOCK_13_NOTEBOOK_10_BOOTSTRAP.get(
                "expanded_corpus_training_required"
            )
            is True
        ),

        (
            BLOCK_13_NOTEBOOK_10_BOOTSTRAP.get(
                "reconstruct_upstream_notebooks"
            )
            is False
        ),
    ]
)


# =============================================================================
# Cross-artefact consistency
# =============================================================================

BLOCK_13_CROSS_ARTEFACT_PARAMETER_COUNTS_VALID = all(
    [
        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE[
                "representation_parameter_count"
            ]
            ==
            BLOCK_13_METADATA_ARCHITECTURE[
                "representation_parameters"
            ]
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE[
                "transformative_parameter_count"
            ]
            ==
            BLOCK_13_METADATA_ARCHITECTURE[
                "transformative_parameters"
            ]
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE[
                "transformative_weight_parameter_count"
            ]
            ==
            BLOCK_13_METADATA_ARCHITECTURE[
                "transformative_weight_parameters"
            ]
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE[
                "total_parameter_count"
            ]
            ==
            BLOCK_13_METADATA_ARCHITECTURE[
                "total_parameters"
            ]
        ),
    ]
)


BLOCK_13_METADATA_PATHWAY_DIMENSIONS_NORMALISED = {
    pathway_name:
        int(
            dimension
        )

    for (
        pathway_name,
        dimension,
    ) in BLOCK_13_METADATA_ARCHITECTURE[
        "pathway_dimensions"
    ].items()
}


BLOCK_13_METADATA_ACTIVE_INDICES_NORMALISED = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in BLOCK_13_METADATA_ARCHITECTURE[
        "active_dimension_indices"
    ].items()
}


BLOCK_13_METADATA_DEFERRED_INDICES_NORMALISED = {
    pathway_name:
        tuple(
            indices
        )

    for (
        pathway_name,
        indices,
    ) in BLOCK_13_METADATA_ARCHITECTURE[
        "deferred_dimension_indices"
    ].items()
}


BLOCK_13_CROSS_ARTEFACT_PATHWAY_CONTRACT_VALID = all(
    [
        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE[
                "pathway_dimensions"
            ]
            ==
            BLOCK_13_METADATA_PATHWAY_DIMENSIONS_NORMALISED
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE[
                "active_dimension_indices"
            ]
            ==
            BLOCK_13_METADATA_ACTIVE_INDICES_NORMALISED
        ),

        (
            BLOCK_13_CHECKPOINT_ARCHITECTURE[
                "deferred_dimension_indices"
            ]
            ==
            BLOCK_13_METADATA_DEFERRED_INDICES_NORMALISED
        ),
    ]
)


BLOCK_13_CROSS_ARTEFACT_RECURRENT_CONTRACT_VALID = (
    BLOCK_13_CHECKPOINT_READBACK[
        "recurrent_contract"
    ]
    ==
    BLOCK_13_METADATA_READBACK[
        "recurrent_contract"
    ]
)


BLOCK_13_CROSS_ARTEFACT_VALID = all(
    [
        BLOCK_13_CROSS_ARTEFACT_PARAMETER_COUNTS_VALID,
        BLOCK_13_CROSS_ARTEFACT_PATHWAY_CONTRACT_VALID,
        BLOCK_13_CROSS_ARTEFACT_RECURRENT_CONTRACT_VALID,
    ]
)


# =============================================================================
# Execution boundary
# =============================================================================

NOTEBOOK_09_BLOCK_13_TRAINING_EXECUTED = False

NOTEBOOK_09_BLOCK_13_OPTIMIZER_CREATED = False

NOTEBOOK_09_BLOCK_13_BACKWARD_PASS_EXECUTED = False

NOTEBOOK_09_BLOCK_13_GRADIENT_CLIPPING_EXECUTED = False

NOTEBOOK_09_BLOCK_13_PARAMETER_UPDATE_EXECUTED = False

NOTEBOOK_09_BLOCK_13_RUNTIME_RECURRENT_STATE_PERSISTED_AS_MODEL = False


# =============================================================================
# Final Block 13 validation
# =============================================================================

NOTEBOOK_09_BLOCK_13_ERRORS = []


BLOCK_13_VALIDATION_CHECKS = {
    "prerequisites_invalid":
        BLOCK_13_PREREQUISITES_VALID,

    "pathway_dimensions_invalid":
        BLOCK_13_PATHWAY_DIMENSIONS_VALID,

    "dimension_partitions_invalid":
        BLOCK_13_ALL_DIMENSION_PARTITIONS_VALID,

    "weight_dimension_accounting_invalid":
        BLOCK_13_WEIGHT_DIMENSION_ACCOUNTING_VALID,

    "trained_weight_state_mismatch":
        BLOCK_13_WEIGHT_STATE_MATCHES_TRAINED_STATE,

    "effective_weight_state_mismatch":
        BLOCK_13_EFFECTIVE_WEIGHT_STATE_MATCHES_BLOCK_11,

    "effective_weights_not_finite":
        BLOCK_13_ALL_EFFECTIVE_WEIGHTS_FINITE,

    "effective_weights_not_bounded":
        BLOCK_13_ALL_EFFECTIVE_WEIGHTS_BOUNDED,

    "deferred_weights_not_zero":
        BLOCK_13_ALL_DEFERRED_WEIGHTS_ZERO,

    "parameters_not_frozen":
        BLOCK_13_ALL_PARAMETERS_FROZEN,

    "modules_not_eval":
        BLOCK_13_ALL_MODULES_EVAL,

    "representation_changed_by_freeze":
        BLOCK_13_REPRESENTATION_UNCHANGED_BY_FREEZE,

    "transformative_changed_by_freeze":
        BLOCK_13_TRANSFORMATIVE_UNCHANGED_BY_FREEZE,

    "weight_changed_by_freeze":
        BLOCK_13_WEIGHT_UNCHANGED_BY_FREEZE,

    "parameter_accounting_invalid":
        BLOCK_13_PARAMETER_ACCOUNTING_VALID,

    "recurrent_policy_invalid":
        BLOCK_13_RECURRENT_POLICY_VALID,

    "checkpoint_sha256_invalid":
        BLOCK_13_CHECKPOINT_SHA256_VALID,

    "checkpoint_byte_count_invalid":
        BLOCK_13_CHECKPOINT_BYTE_COUNT_VALID,

    "checkpoint_parse_invalid":
        BLOCK_13_CHECKPOINT_PARSED,

    "checkpoint_identity_invalid":
        BLOCK_13_CHECKPOINT_IDENTITY_VALID,

    "checkpoint_parameter_counts_invalid":
        BLOCK_13_CHECKPOINT_PARAMETER_COUNTS_VALID,

    "checkpoint_pathway_contract_invalid":
        BLOCK_13_CHECKPOINT_PATHWAY_CONTRACT_VALID,

    "checkpoint_recurrent_contract_invalid":
        BLOCK_13_CHECKPOINT_RECURRENT_CONTRACT_VALID,

    "representation_readback_not_exact":
        BLOCK_13_REPRESENTATION_READBACK_EXACT,

    "transformative_readback_not_exact":
        BLOCK_13_TRANSFORMATIVE_READBACK_EXACT,

    "weight_readback_not_exact":
        BLOCK_13_WEIGHT_READBACK_EXACT,

    "effective_weight_readback_not_exact":
        BLOCK_13_EFFECTIVE_WEIGHT_READBACK_EXACT,

    "metadata_sha256_invalid":
        BLOCK_13_METADATA_SHA256_VALID,

    "metadata_byte_count_invalid":
        BLOCK_13_METADATA_BYTE_COUNT_VALID,

    "metadata_parse_invalid":
        BLOCK_13_METADATA_PARSED,

    "metadata_top_level_keys_invalid":
        BLOCK_13_METADATA_TOP_LEVEL_KEYS_VALID,

    "metadata_identity_invalid":
        BLOCK_13_METADATA_IDENTITY_VALID,

    "metadata_checkpoint_identity_invalid":
        BLOCK_13_METADATA_CHECKPOINT_IDENTITY_VALID,

    "metadata_parameter_counts_invalid":
        BLOCK_13_METADATA_PARAMETER_COUNTS_VALID,

    "metadata_recurrent_contract_invalid":
        BLOCK_13_METADATA_RECURRENT_CONTRACT_VALID,

    "notebook_10_bootstrap_invalid":
        BLOCK_13_NOTEBOOK_10_BOOTSTRAP_VALID,

    "cross_artefact_invalid":
        BLOCK_13_CROSS_ARTEFACT_VALID,
}


for (
    error_name,
    condition,
) in BLOCK_13_VALIDATION_CHECKS.items():

    if not condition:

        NOTEBOOK_09_BLOCK_13_ERRORS.append(
            error_name
        )


# =============================================================================
# Prohibited-operation audit
# =============================================================================

BLOCK_13_PROHIBITED_OPERATIONS = {
    "training_incorrectly_executed":
        NOTEBOOK_09_BLOCK_13_TRAINING_EXECUTED,

    "optimizer_incorrectly_created":
        NOTEBOOK_09_BLOCK_13_OPTIMIZER_CREATED,

    "backward_pass_incorrectly_executed":
        NOTEBOOK_09_BLOCK_13_BACKWARD_PASS_EXECUTED,

    "gradient_clipping_incorrectly_executed":
        NOTEBOOK_09_BLOCK_13_GRADIENT_CLIPPING_EXECUTED,

    "parameter_update_incorrectly_executed":
        NOTEBOOK_09_BLOCK_13_PARAMETER_UPDATE_EXECUTED,

    "runtime_recurrent_state_incorrectly_persisted":
        NOTEBOOK_09_BLOCK_13_RUNTIME_RECURRENT_STATE_PERSISTED_AS_MODEL,
}


for (
    error_name,
    operation_executed,
) in BLOCK_13_PROHIBITED_OPERATIONS.items():

    if operation_executed:

        NOTEBOOK_09_BLOCK_13_ERRORS.append(
            error_name
        )


# =============================================================================
# Final Notebook 09 state
# =============================================================================

NOTEBOOK_09_FINAL_ARCHITECTURE_VALID = (
    len(
        NOTEBOOK_09_BLOCK_13_ERRORS
    )
    ==
    0
)


NOTEBOOK_09_TO_10_HANDOVER_VALID = (
    NOTEBOOK_09_FINAL_ARCHITECTURE_VALID
)


NOTEBOOK_10_READY = (
    NOTEBOOK_09_TO_10_HANDOVER_VALID
)


NOTEBOOK_09_BLOCK_13_VALID = (
    NOTEBOOK_09_FINAL_ARCHITECTURE_VALID
)


if not NOTEBOOK_09_BLOCK_13_VALID:

    raise RuntimeError(
        "Notebook 09 Block 13 final architectural completion "
        "and persistent handover failed. "
        f"Errors: {NOTEBOOK_09_BLOCK_13_ERRORS}"
    )


NOTEBOOK_09_BLOCK_13_COMPLETE = True

NOTEBOOK_09_COMPLETE = True


# =============================================================================
# Expose direct Notebook 10 bootstrap contract
# =============================================================================

NOTEBOOK_09_TO_10_BOOTSTRAP = {
    "checkpoint_filename":
        NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME,

    "checkpoint_drive_file_id":
        BLOCK_13_CHECKPOINT_DRIVE_FILE_ID,

    "metadata_filename":
        NOTEBOOK_09_FINAL_METADATA_FILENAME,

    "metadata_drive_file_id":
        BLOCK_13_METADATA_DRIVE_FILE_ID,

    "checkpoint_sha256":
        BLOCK_13_CHECKPOINT_SHA256,

    "metadata_sha256":
        BLOCK_13_METADATA_SHA256,

    "architecture_parameter_count":
        BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT,

    "representation_parameter_count":
        BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT,

    "transformative_parameter_count":
        BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT,

    "transformative_weight_parameter_count":
        BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT,

    "expanded_corpus_training_required":
        True,

    "upstream_notebook_reconstruction_required":
        False,
}


# =============================================================================
# Block summary
# =============================================================================

NOTEBOOK_09_BLOCK_13_SUMMARY = {
    "block":
        NOTEBOOK_09_BLOCK_13,

    "block_name":
        NOTEBOOK_09_BLOCK_13_NAME,

    "block_version":
        NOTEBOOK_09_BLOCK_13_VERSION,

    "checkpoint_filename":
        NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME,

    "checkpoint_drive_file_id":
        BLOCK_13_CHECKPOINT_DRIVE_FILE_ID,

    "checkpoint_sha256":
        BLOCK_13_CHECKPOINT_SHA256,

    "metadata_filename":
        NOTEBOOK_09_FINAL_METADATA_FILENAME,

    "metadata_drive_file_id":
        BLOCK_13_METADATA_DRIVE_FILE_ID,

    "metadata_sha256":
        BLOCK_13_METADATA_SHA256,

    "representation_parameters":
        BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT,

    "transformative_parameters":
        BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT,

    "transformative_weight_parameters":
        BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT,

    "total_parameters":
        BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT,

    "all_parameters_frozen":
        BLOCK_13_ALL_PARAMETERS_FROZEN,

    "all_modules_eval":
        BLOCK_13_ALL_MODULES_EVAL,

    "checkpoint_contract_valid":
        all(
            [
                BLOCK_13_CHECKPOINT_IDENTITY_VALID,
                BLOCK_13_CHECKPOINT_PARAMETER_COUNTS_VALID,
                BLOCK_13_CHECKPOINT_PATHWAY_CONTRACT_VALID,
                BLOCK_13_CHECKPOINT_RECURRENT_CONTRACT_VALID,
            ]
        ),

    "metadata_contract_valid":
        all(
            [
                BLOCK_13_METADATA_IDENTITY_VALID,
                BLOCK_13_METADATA_CHECKPOINT_IDENTITY_VALID,
                BLOCK_13_METADATA_PARAMETER_COUNTS_VALID,
                BLOCK_13_METADATA_RECURRENT_CONTRACT_VALID,
            ]
        ),

    "cross_artefact_valid":
        BLOCK_13_CROSS_ARTEFACT_VALID,

    "notebook_09_to_10_valid":
        NOTEBOOK_09_TO_10_HANDOVER_VALID,

    "notebook_10_ready":
        NOTEBOOK_10_READY,

    "block_valid":
        NOTEBOOK_09_BLOCK_13_VALID,

    "block_complete":
        NOTEBOOK_09_BLOCK_13_COMPLETE,

    "notebook_09_complete":
        NOTEBOOK_09_COMPLETE,
}


# =============================================================================
# Final report
# =============================================================================

print("=" * 72)

print(
    "Media AI — Notebook 09, Block 13: "
    "Final Architectural Completion Audit and "
    "Persistent Notebook 09 → Notebook 10 Handover"
)

print("=" * 72)

print(
    f"Block version                : "
    f"{NOTEBOOK_09_BLOCK_13_VERSION}"
)

print("-" * 72)

print(
    "Final architecture"
)

print(
    f"Representation parameters    : "
    f"{BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT:,}"
)

print(
    f"Transformative parameters    : "
    f"{BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT:,}"
)

print(
    f"Transformative Weight params : "
    f"{BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT}"
)

print(
    f"Combined persisted params    : "
    f"{BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT:,}"
)

print("-" * 72)

print(
    "Transformative-Weight architecture"
)

for pathway_name in BLOCK_13_PATHWAY_DIMENSIONS:

    print(
        f"{pathway_name:<14} dimension          : "
        f"{BLOCK_13_PATHWAY_DIMENSIONS[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} active dimensions  : "
        f"{list(BLOCK_13_ACTIVE_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} deferred dimensions: "
        f"{list(BLOCK_13_DEFERRED_DIMENSION_INDICES[pathway_name])}"
    )

    print(
        f"{pathway_name:<14} finite             : "
        f"{BLOCK_13_EFFECTIVE_WEIGHTS_FINITE[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} bounded            : "
        f"{BLOCK_13_EFFECTIVE_WEIGHTS_BOUNDED[pathway_name]}"
    )

    print(
        f"{pathway_name:<14} deferred zero      : "
        f"{BLOCK_13_DEFERRED_WEIGHTS_ZERO[pathway_name]}"
    )

print("-" * 72)

print(
    "Final recurrent contract"
)

print(
    f"Update type                  : "
    f"{BLOCK_13_RECURRENT_UPDATE_TYPE}"
)

print(
    f"Initial-state policy         : "
    f"{BLOCK_13_INITIAL_STATE_POLICY}"
)

print(
    f"Article-boundary reset       : "
    f"{BLOCK_13_ARTICLE_BOUNDARY_RESET}"
)

print(
    f"Cross-article propagation    : "
    f"{BLOCK_13_CROSS_ARTICLE_PROPAGATION}"
)

print(
    f"Cross-pathway recurrence     : "
    f"{BLOCK_13_CROSS_PATHWAY_RECURRENCE}"
)

print(
    f"Future context allowed       : "
    f"{BLOCK_13_FUTURE_CONTEXT_ALLOWED}"
)

print("-" * 72)

print(
    "Final pilot training status"
)

print(
    f"Pretraining objective        : "
    f"{float(NOTEBOOK_09_TW_INITIAL_TOTAL_OBJECTIVE):.8f}"
)

print(
    f"Posttraining objective       : "
    f"{float(NOTEBOOK_09_TW_FINAL_TOTAL_OBJECTIVE):.8f}"
)

print(
    f"Objective delta              : "
    f"{float(NOTEBOOK_09_TW_OBJECTIVE_DELTA):.8f}"
)

print(
    f"Expanded training required   : True"
)

print("-" * 72)

print(
    "Final freeze state"
)

print(
    f"All parameters frozen        : "
    f"{BLOCK_13_ALL_PARAMETERS_FROZEN}"
)

print(
    f"All modules in eval mode     : "
    f"{BLOCK_13_ALL_MODULES_EVAL}"
)

print(
    f"Representation unchanged     : "
    f"{BLOCK_13_REPRESENTATION_UNCHANGED_BY_FREEZE}"
)

print(
    f"Transformative unchanged     : "
    f"{BLOCK_13_TRANSFORMATIVE_UNCHANGED_BY_FREEZE}"
)

print(
    f"Transform Weight unchanged   : "
    f"{BLOCK_13_WEIGHT_UNCHANGED_BY_FREEZE}"
)

print("-" * 72)

print(
    "Persistent handover"
)

print(
    f"Checkpoint filename          : "
    f"{NOTEBOOK_09_FINAL_CHECKPOINT_FILENAME}"
)

print(
    f"Checkpoint Drive operation   : "
    f"{BLOCK_13_CHECKPOINT_DRIVE_OPERATION}"
)

print(
    f"Checkpoint Drive file ID     : "
    f"{BLOCK_13_CHECKPOINT_DRIVE_FILE_ID}"
)

print(
    f"Checkpoint size              : "
    f"{BLOCK_13_CHECKPOINT_BYTE_COUNT / (1024 ** 2):.3f} MB"
)

print(
    f"Checkpoint SHA-256 valid     : "
    f"{BLOCK_13_CHECKPOINT_SHA256_VALID}"
)

print("-" * 72)

print(
    f"Metadata filename            : "
    f"{NOTEBOOK_09_FINAL_METADATA_FILENAME}"
)

print(
    f"Metadata Drive operation     : "
    f"{BLOCK_13_METADATA_DRIVE_OPERATION}"
)

print(
    f"Metadata Drive file ID       : "
    f"{BLOCK_13_METADATA_DRIVE_FILE_ID}"
)

print(
    f"Metadata size                : "
    f"{BLOCK_13_METADATA_BYTE_COUNT / 1024:.3f} KB"
)

print(
    f"Metadata SHA-256 valid       : "
    f"{BLOCK_13_METADATA_SHA256_VALID}"
)

print("-" * 72)

print(
    "Read-back validation"
)

print(
    f"Representation read-back     : "
    f"{BLOCK_13_REPRESENTATION_READBACK_EXACT}"
)

print(
    f"Transformative read-back     : "
    f"{BLOCK_13_TRANSFORMATIVE_READBACK_EXACT}"
)

print(
    f"Transform Weight read-back   : "
    f"{BLOCK_13_WEIGHT_READBACK_EXACT}"
)

print(
    f"Effective Weight read-back   : "
    f"{BLOCK_13_EFFECTIVE_WEIGHT_READBACK_EXACT}"
)

print(
    f"Checkpoint contract valid    : "
    f"{all([BLOCK_13_CHECKPOINT_IDENTITY_VALID, BLOCK_13_CHECKPOINT_PARAMETER_COUNTS_VALID, BLOCK_13_CHECKPOINT_PATHWAY_CONTRACT_VALID, BLOCK_13_CHECKPOINT_RECURRENT_CONTRACT_VALID])}"
)

print(
    f"Metadata contract valid      : "
    f"{all([BLOCK_13_METADATA_IDENTITY_VALID, BLOCK_13_METADATA_CHECKPOINT_IDENTITY_VALID, BLOCK_13_METADATA_PARAMETER_COUNTS_VALID, BLOCK_13_METADATA_RECURRENT_CONTRACT_VALID])}"
)

print(
    f"Cross-artefact valid         : "
    f"{BLOCK_13_CROSS_ARTEFACT_VALID}"
)

print(
    f"Notebook 10 bootstrap valid  : "
    f"{BLOCK_13_NOTEBOOK_10_BOOTSTRAP_VALID}"
)

print("-" * 72)

print(
    "Execution boundary"
)

print(
    f"Training executed            : "
    f"{NOTEBOOK_09_BLOCK_13_TRAINING_EXECUTED}"
)

print(
    f"Optimizer created            : "
    f"{NOTEBOOK_09_BLOCK_13_OPTIMIZER_CREATED}"
)

print(
    f"Backward pass executed       : "
    f"{NOTEBOOK_09_BLOCK_13_BACKWARD_PASS_EXECUTED}"
)

print(
    f"Parameter update executed    : "
    f"{NOTEBOOK_09_BLOCK_13_PARAMETER_UPDATE_EXECUTED}"
)

print(
    f"Runtime state persisted      : "
    f"{NOTEBOOK_09_BLOCK_13_RUNTIME_RECURRENT_STATE_PERSISTED_AS_MODEL}"
)

print("-" * 72)

print(
    f"Final architecture valid     : "
    f"{NOTEBOOK_09_FINAL_ARCHITECTURE_VALID}"
)

print(
    f"Notebook 09 → 10 valid       : "
    f"{NOTEBOOK_09_TO_10_HANDOVER_VALID}"
)

print(
    f"Notebook 10 ready            : "
    f"{NOTEBOOK_10_READY}"
)

print(
    f"Block valid                  : "
    f"{NOTEBOOK_09_BLOCK_13_VALID}"
)

print(
    f"Block complete               : "
    f"{NOTEBOOK_09_BLOCK_13_COMPLETE}"
)

print(
    f"Notebook 09 complete         : "
    f"{NOTEBOOK_09_COMPLETE}"
)

print("=" * 72)

print(
    "The complete Media AI representation, transformative, "
    "Transformative-Weight and recurrent architecture was validated "
    "successfully."
)

print(
    f"All {BLOCK_13_ACTUAL_REPRESENTATION_PARAMETER_COUNT:,} representation "
    f"parameters, {BLOCK_13_ACTUAL_TRANSFORMATIVE_PARAMETER_COUNT:,} "
    f"transformative parameters and {BLOCK_13_ACTUAL_WEIGHT_PARAMETER_COUNT} "
    f"learned Transformative-Weight parameter(s) were persisted as the final "
    f"{BLOCK_13_ACTUAL_TOTAL_PARAMETER_COUNT:,}-parameter Notebook 09 "
    "architecture."
)

print(
    "The learned Transformative-Weight state matches the validated "
    "post-training Block 12 state exactly."
)

print(
    "All deferred Transformative-Weight dimensions remain fixed at zero, "
    "and the causal additive recurrent contract remains unchanged."
)

print(
    "All persisted model parameters were frozen and all modules were placed "
    "in evaluation mode before checkpoint construction."
)

print(
    "The final checkpoint and metadata were persisted to Google Drive, "
    "read back independently and validated against the in-memory "
    "Notebook 09 completion state."
)

print(
    "Article-specific runtime recurrent states were not persisted as "
    "learned model state."
)

print(
    "The Notebook 10 bootstrap contract contains direct checkpoint and "
    "metadata Drive file IDs and does not require reconstruction of "
    "upstream notebooks."
)

print(
    "Notebook 09 is complete. Notebook 10 may now initialise directly "
    "from the validated persistent Notebook 09 handover."
)

print("=" * 72)

Media AI — Notebook 09, Block 13: Final Architectural Completion Audit and Persistent Notebook 09 → Notebook 10 Handover
Block version                : 1.1
------------------------------------------------------------------------
Final architecture
Representation parameters    : 894,003
Transformative parameters    : 8,187
Transformative Weight params : 49
Combined persisted params    : 902,239
------------------------------------------------------------------------
Transformative-Weight architecture
factual        dimension          : 10
factual        active dimensions  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
factual        deferred dimensions: []
factual        finite             : True
factual        bounded            : True
factual        deferred zero      : True
psychological  dimension          : 34
psychological  active dimensions  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
psychological  deferred d